In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
from pathlib import Path

MODEL_DIR = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

for f in [
    "production_test_s1_tfidf.npz",
    "production_reverse_tfidf_vectorizer.joblib",
    "practical_matcher_lgbm.txt",
    "top1_practical_threshold.txt",
]:
    p = MODEL_DIR / f
    print(
        "FOUND" if p.exists() else "MISSING",
        f
    )

FOUND production_test_s1_tfidf.npz
FOUND production_reverse_tfidf_vectorizer.joblib
FOUND practical_matcher_lgbm.txt
FOUND top1_practical_threshold.txt


In [ ]:
# CELL 68 — RAPIDFUZZ BLOCKED RETRIEVAL BENCHMARK
#
# Goal:
#   Test whether CPU RapidFuzz can recover the correct S1
#   from country + first-character blocks quickly enough.
#
# This is ONLY a 2,000-target benchmark.
# No production files are modified.

!pip -q install rapidfuzz

import pandas as pd
import numpy as np
import time
import unicodedata

from pathlib import Path
from collections import defaultdict
from rapidfuzz import process, fuzz

print("=" * 100)
print("CELL 68 — RAPIDFUZZ BLOCKED RETRIEVAL BENCHMARK")
print("=" * 100)

BASE = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN = BASE / "DATASET" / "train"

# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

def norm(x):
    if pd.isna(x):
        return ""

    x = unicodedata.normalize(
        "NFKC",
        str(x)
    ).lower()

    return "".join(
        c for c in x
        if c.isalnum()
    )

# ------------------------------------------------------------
# Load validation S1
# ------------------------------------------------------------

print("Loading validation S1...")

s1 = pd.read_csv(
    TRAIN / "train_source1.tsv",
    sep="\t",
    dtype=str,
    nrows=20_000
)

s1["entity_id"] = (
    s1["entity_id"].astype(str)
)

s1["name_n"] = (
    s1["business_name"].map(norm)
)

s1["addr_n"] = (
    s1["business_address"].map(norm)
)

s1["country_n"] = (
    s1["country"]
    .fillna("")
    .str.lower()
    .str.strip()
)

print(
    "Validation S1:",
    f"{len(s1):,}"
)

# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

gt = pd.read_csv(
    TRAIN / "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

gt = gt[
    gt["source1_entity_id"].isin(
        set(s1["entity_id"])
    )
]

# Target -> true S1
target_to_s1 = dict(
    zip(
        gt["target_id"].astype(str),
        gt["source1_entity_id"].astype(str)
    )
)

target_ids = list(
    target_to_s1.keys()
)

print(
    "True target records:",
    f"{len(target_ids):,}"
)

# ------------------------------------------------------------
# Load only the required target records
# ------------------------------------------------------------

print("\nLoading S2/S3 target records...")

target_set = set(target_ids)

target_parts = []

for source_file, source_name in [
    (TRAIN / "train_source2.tsv", "S2"),
    (TRAIN / "train_source3.tsv", "S3")
]:

    for chunk in pd.read_csv(
        source_file,
        sep="\t",
        dtype=str,
        chunksize=250_000
    ):

        hit = chunk[
            chunk["entity_id"].isin(
                target_set
            )
        ].copy()

        if len(hit):

            hit["source"] = source_name

            target_parts.append(
                hit
            )

targets = pd.concat(
    target_parts,
    ignore_index=True
)

targets["entity_id"] = (
    targets["entity_id"].astype(str)
)

targets["name_n"] = (
    targets["business_name"].map(norm)
)

targets["addr_n"] = (
    targets["business_address"].map(norm)
)

targets["country_n"] = (
    targets["country"]
    .fillna("")
    .str.lower()
    .str.strip()
)

print(
    "Targets loaded:",
    f"{len(targets):,}"
)

# ------------------------------------------------------------
# Build S1 blocks ONCE
# ------------------------------------------------------------

print("\nBuilding S1 blocks...")

name_first = defaultdict(list)
name_last = defaultdict(list)
addr_first = defaultdict(list)
addr_last = defaultdict(list)

for i, r in s1.iterrows():

    c = r.country_n

    n = r.name_n
    a = r.addr_n

    if n:
        name_first[
            (c, n[0])
        ].append(i)

        name_last[
            (c, n[-1])
        ].append(i)

    if a:
        addr_first[
            (c, a[0])
        ].append(i)

        addr_last[
            (c, a[-1])
        ].append(i)

print(
    "name_first blocks:",
    len(name_first)
)

print(
    "name_last blocks:",
    len(name_last)
)

print(
    "addr_first blocks:",
    len(addr_first)
)

print(
    "addr_last blocks:",
    len(addr_last)
)

# ------------------------------------------------------------
# IMPORTANT:
# Build choice dictionaries ONCE.
# Do NOT recreate them for every query.
# ------------------------------------------------------------

print("\nPreparing RapidFuzz choice dictionaries...")

name_first_choices = {}
name_last_choices = {}
addr_first_choices = {}
addr_last_choices = {}

for key, rows in name_first.items():

    name_first_choices[key] = {
        i: s1.iloc[i].name_n
        for i in rows
    }

for key, rows in name_last.items():

    name_last_choices[key] = {
        i: s1.iloc[i].name_n
        for i in rows
    }

for key, rows in addr_first.items():

    addr_first_choices[key] = {
        i: s1.iloc[i].addr_n
        for i in rows
    }

for key, rows in addr_last.items():

    addr_last_choices[key] = {
        i: s1.iloc[i].addr_n
        for i in rows
    }

# ------------------------------------------------------------
# Sample 2,000 targets
# ------------------------------------------------------------

sample = targets.sample(
    n=min(2_000, len(targets)),
    random_state=42
).reset_index(drop=True)

print(
    "\nBenchmark targets:",
    len(sample)
)

# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

correct = 0
evaluated = 0

t0 = time.time()

for q_no, r in sample.iterrows():

    c = r.country_n

    best_score = -1
    best_idx = None

    # --------------------------------------------------------
    # NAME FIRST
    # --------------------------------------------------------

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:

            result = process.extractOne(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                score_cutoff=0
            )

            if result:

                _, score, idx = result

                if score > best_score:

                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # NAME LAST
    # --------------------------------------------------------

    if r.name_n:

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:

            result = process.extractOne(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                score_cutoff=0
            )

            if result:

                _, score, idx = result

                if score > best_score:

                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # ADDRESS FIRST
    # --------------------------------------------------------

    if r.addr_n:

        choices = addr_first_choices.get(
            (c, r.addr_n[0])
        )

        if choices:

            result = process.extractOne(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                score_cutoff=0
            )

            if result:

                _, score, idx = result

                if score > best_score:

                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # ADDRESS LAST
    # --------------------------------------------------------

    if r.addr_n:

        choices = addr_last_choices.get(
            (c, r.addr_n[-1])
        )

        if choices:

            result = process.extractOne(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                score_cutoff=0
            )

            if result:

                _, score, idx = result

                if score > best_score:

                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    if best_idx is not None:

        predicted_s1 = str(
            s1.iloc[
                best_idx
            ].entity_id
        )

        true_s1 = target_to_s1.get(
            str(r.entity_id)
        )

        if predicted_s1 == true_s1:
            correct += 1

        evaluated += 1

    if (q_no + 1) % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{q_no+1:,}/{len(sample):,} "
            f"| speed={((q_no+1)/elapsed):.1f} q/s"
        )

elapsed = time.time() - t0

print("\n" + "=" * 100)
print("CELL 68 RESULTS")
print("=" * 100)

print(
    "Evaluated:",
    evaluated
)

print(
    "Correct Top-1:",
    correct
)

print(
    "Recall:",
    f"{correct/evaluated*100:.4f}%"
)

print(
    "Runtime:",
    f"{elapsed:.2f} sec"
)

print(
    "Speed:",
    f"{len(sample)/elapsed:.2f} queries/sec"
)

print(
    "Projected 10.0M target queries:",
    f"{10_000_000/(len(sample)/elapsed)/3600:.2f} hours"
)

print("\nCELL 68 COMPLETE.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.0 MB/s eta 0:00:00
CELL 68 — RAPIDFUZZ BLOCKED RETRIEVAL BENCHMARK
Loading validation S1...
Validation S1: 20,000


KeyError: 'target_id'

In [ ]:
# CELL 68A — CHECK GROUND-TRUTH SCHEMA

from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

GT_FILE = (
    BASE /
    "DATASET" /
    "train" /
    "train_ground_truth.tsv"
)

gt_head = pd.read_csv(
    GT_FILE,
    sep="\t",
    dtype=str,
    nrows=5
)

print("=" * 100)
print("CELL 68A — GROUND-TRUTH SCHEMA")
print("=" * 100)

print("Columns:")
for i, col in enumerate(gt_head.columns):
    print(f"{i}: {repr(col)}")

print("\nFirst 5 rows:")
print(gt_head.to_string(index=False))

print("\nCELL 68A COMPLETE.")

CELL 68A — GROUND-TRUTH SCHEMA
Columns:
0: 'source1_entity_id'
1: 'matched_entity_ids'

First 5 rows:
source1_entity_id                                                            matched_entity_ids
        S1-965667               S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364
      S1-55344266                           S2-249013014,S2-197070651,S3-478195123,S3-384364074
     S1-343815751                                        S2-790675320,S2-479876582,S3-878454467
     S1-656753428                                         S2-153058913,S2-24659151,S3-679606215
     S1-102811957 S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785

CELL 68A COMPLETE.


In [ ]:
# CELL 68B — RAPIDFUZZ BLOCKED RETRIEVAL BENCHMARK (SCHEMA FIXED)

import pandas as pd
import numpy as np
import time
import unicodedata

from pathlib import Path
from collections import defaultdict
from rapidfuzz import process, fuzz

print("=" * 100)
print("CELL 68B — RAPIDFUZZ BLOCKED RETRIEVAL BENCHMARK")
print("=" * 100)

BASE = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN = BASE / "DATASET" / "train"

# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

def norm(x):

    if pd.isna(x):
        return ""

    x = unicodedata.normalize(
        "NFKC",
        str(x)
    ).lower()

    return "".join(
        c for c in x
        if c.isalnum()
    )

# ------------------------------------------------------------
# Load validation S1
# ------------------------------------------------------------

print("Loading validation S1...")

s1 = pd.read_csv(
    TRAIN / "train_source1.tsv",
    sep="\t",
    dtype=str,
    nrows=20_000
)

s1["entity_id"] = (
    s1["entity_id"].astype(str)
)

s1["name_n"] = (
    s1["business_name"].map(norm)
)

s1["addr_n"] = (
    s1["business_address"].map(norm)
)

s1["country_n"] = (
    s1["country"]
    .fillna("")
    .str.lower()
    .str.strip()
)

val_s1_ids = set(
    s1["entity_id"]
)

print(
    "Validation S1:",
    f"{len(s1):,}"
)

# ------------------------------------------------------------
# Load and EXPLODE ground truth
# ------------------------------------------------------------

print("\nLoading ground truth...")

gt = pd.read_csv(
    TRAIN / "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

gt = gt[
    gt["source1_entity_id"].isin(
        val_s1_ids
    )
].copy()

# Convert:
#
# S1-A -> S2-X,S2-Y,S3-Z
#
# into:
#
# S2-X -> S1-A
# S2-Y -> S1-A
# S3-Z -> S1-A

target_to_s1 = {}

true_pair_count = 0

for r in gt.itertuples(index=False):

    s1_id = str(
        r.source1_entity_id
    )

    matched = str(
        r.matched_entity_ids
    ).strip()

    if not matched:
        continue

    for target_id in matched.split(","):

        target_id = target_id.strip()

        if not target_id:
            continue

        target_to_s1[target_id] = s1_id
        true_pair_count += 1

target_ids = set(
    target_to_s1.keys()
)

print(
    "Validation GT rows:",
    len(gt)
)

print(
    "True pairs:",
    f"{true_pair_count:,}"
)

print(
    "Unique target IDs:",
    f"{len(target_ids):,}"
)

# ------------------------------------------------------------
# Load required S2/S3 target records
# ------------------------------------------------------------

print("\nLoading S2/S3 target records...")

target_parts = []

for source_file, source_name in [
    (TRAIN / "train_source2.tsv", "S2"),
    (TRAIN / "train_source3.tsv", "S3")
]:

    print(
        "Scanning:",
        source_name
    )

    for chunk in pd.read_csv(
        source_file,
        sep="\t",
        dtype=str,
        chunksize=250_000
    ):

        hit = chunk[
            chunk["entity_id"].isin(
                target_ids
            )
        ].copy()

        if len(hit):

            hit["source"] = source_name

            target_parts.append(
                hit
            )

targets = pd.concat(
    target_parts,
    ignore_index=True
)

targets["entity_id"] = (
    targets["entity_id"].astype(str)
)

targets["name_n"] = (
    targets["business_name"].map(norm)
)

targets["addr_n"] = (
    targets["business_address"].map(norm)
)

targets["country_n"] = (
    targets["country"]
    .fillna("")
    .str.lower()
    .str.strip()
)

print(
    "Target records loaded:",
    f"{len(targets):,}"
)

# ------------------------------------------------------------
# Build S1 blocks
# ------------------------------------------------------------

print("\nBuilding S1 blocks...")

name_first = defaultdict(list)
name_last = defaultdict(list)
addr_first = defaultdict(list)
addr_last = defaultdict(list)

for i, r in s1.iterrows():

    c = r.country_n
    n = r.name_n
    a = r.addr_n

    if n:

        name_first[
            (c, n[0])
        ].append(i)

        name_last[
            (c, n[-1])
        ].append(i)

    if a:

        addr_first[
            (c, a[0])
        ].append(i)

        addr_last[
            (c, a[-1])
        ].append(i)

# ------------------------------------------------------------
# Build RapidFuzz choices ONCE
# ------------------------------------------------------------

print(
    "Preparing RapidFuzz indexes..."
)

name_first_choices = {
    key: {
        i: s1.iloc[i].name_n
        for i in rows
    }
    for key, rows in name_first.items()
}

name_last_choices = {
    key: {
        i: s1.iloc[i].name_n
        for i in rows
    }
    for key, rows in name_last.items()
}

addr_first_choices = {
    key: {
        i: s1.iloc[i].addr_n
        for i in rows
    }
    for key, rows in addr_first.items()
}

addr_last_choices = {
    key: {
        i: s1.iloc[i].addr_n
        for i in rows
    }
    for key, rows in addr_last.items()
}

print(
    "RapidFuzz indexes ready."
)

# ------------------------------------------------------------
# 2,000-query benchmark
# ------------------------------------------------------------

sample = targets.sample(
    n=min(
        2_000,
        len(targets)
    ),
    random_state=42
).reset_index(drop=True)

print(
    "\nBenchmark queries:",
    len(sample)
)

correct = 0
evaluated = 0

t0 = time.time()

for q_no, r in sample.iterrows():

    c = r.country_n

    best_score = -1
    best_idx = None

    # --------------------------------------------------------
    # NAME FIRST
    # --------------------------------------------------------

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:

            result = process.extractOne(
                r.name_n,
                choices,
                scorer=fuzz.ratio
            )

            if result:

                _, score, idx = result

                if score > best_score:
                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # NAME LAST
    # --------------------------------------------------------

    if r.name_n:

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:

            result = process.extractOne(
                r.name_n,
                choices,
                scorer=fuzz.ratio
            )

            if result:

                _, score, idx = result

                if score > best_score:
                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # ADDRESS FIRST
    # --------------------------------------------------------

    if r.addr_n:

        choices = addr_first_choices.get(
            (c, r.addr_n[0])
        )

        if choices:

            result = process.extractOne(
                r.addr_n,
                choices,
                scorer=fuzz.ratio
            )

            if result:

                _, score, idx = result

                if score > best_score:
                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # ADDRESS LAST
    # --------------------------------------------------------

    if r.addr_n:

        choices = addr_last_choices.get(
            (c, r.addr_n[-1])
        )

        if choices:

            result = process.extractOne(
                r.addr_n,
                choices,
                scorer=fuzz.ratio
            )

            if result:

                _, score, idx = result

                if score > best_score:
                    best_score = score
                    best_idx = idx

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    if best_idx is not None:

        predicted_s1 = str(
            s1.iloc[
                best_idx
            ].entity_id
        )

        true_s1 = target_to_s1.get(
            str(r.entity_id)
        )

        if predicted_s1 == true_s1:
            correct += 1

        evaluated += 1

    if (q_no + 1) % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{q_no+1:,}/{len(sample):,}"
            f" | speed="
            f"{(q_no+1)/elapsed:.1f} q/s"
        )

elapsed = time.time() - t0

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 68B RESULTS")
print("=" * 100)

print(
    "Evaluated:",
    evaluated
)

print(
    "Correct Top-1:",
    correct
)

print(
    "Recall:",
    f"{correct/evaluated*100:.4f}%"
)

print(
    "Runtime:",
    f"{elapsed:.2f} sec"
)

print(
    "Speed:",
    f"{len(sample)/elapsed:.2f} queries/sec"
)

print(
    "Projected 10M targets:",
    f"{10_000_000/(len(sample)/elapsed)/3600:.2f} hours"
)

print("\nCELL 68B COMPLETE.")

CELL 68B — RAPIDFUZZ BLOCKED RETRIEVAL BENCHMARK
Loading validation S1...
Validation S1: 20,000

Loading ground truth...
Validation GT rows: 20000
True pairs: 70,591
Unique target IDs: 69,479

Loading S2/S3 target records...
Scanning: S2
Scanning: S3
Target records loaded: 69,478

Building S1 blocks...
Preparing RapidFuzz indexes...
RapidFuzz indexes ready.

Benchmark queries: 2000
500/2,000 | speed=1203.8 q/s
1,000/2,000 | speed=1217.9 q/s
1,500/2,000 | speed=1179.9 q/s
2,000/2,000 | speed=1180.9 q/s

CELL 68B RESULTS
Evaluated: 2000
Correct Top-1: 1790
Recall: 89.5000%
Runtime: 1.69 sec
Speed: 1180.68 queries/sec
Projected 10M targets: 2.35 hours

CELL 68B COMPLETE.


In [ ]:
# CELL 69 — RAPIDFUZZ MULTI-CHANNEL TOP-3 RECALL

import pandas as pd
import numpy as np
import time

from collections import defaultdict
from pathlib import Path
from rapidfuzz import process, fuzz

print("=" * 100)
print("CELL 69 — RAPIDFUZZ MULTI-CHANNEL TOP-3 RECALL")
print("=" * 100)

# ------------------------------------------------------------
# We reuse objects from Cell 68B if they still exist.
# ------------------------------------------------------------

required = [
    "s1",
    "targets",
    "target_to_s1",
    "name_first_choices",
    "name_last_choices",
    "addr_first_choices",
    "addr_last_choices",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:

    raise RuntimeError(
        "Missing objects from Cell 68B: "
        + str(missing)
        + "\n"
        "Do not rebuild yet."
    )

# ------------------------------------------------------------
# Same 2,000-query sample
# ------------------------------------------------------------

sample = targets.sample(
    n=min(
        2_000,
        len(targets)
    ),
    random_state=42
).reset_index(drop=True)

print(
    "Benchmark queries:",
    f"{len(sample):,}"
)

# ------------------------------------------------------------
# Test multiple Top-K values
# ------------------------------------------------------------

for TOPK in [1, 2, 3, 5]:

    print("\n" + "-" * 90)
    print(
        f"TESTING TOP-{TOPK} PER CHANNEL"
    )

    correct = 0
    evaluated = 0

    total_candidates = 0
    max_candidates = 0

    t0 = time.time()

    for q_no, r in sample.iterrows():

        c = r.country_n

        candidate_ids = set()

        # ----------------------------------------------------
        # NAME FIRST
        # ----------------------------------------------------

        if r.name_n:

            choices = name_first_choices.get(
                (c, r.name_n[0])
            )

            if choices:

                results = process.extract(
                    r.name_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=TOPK
                )

                for _, _, idx in results:
                    candidate_ids.add(idx)

        # ----------------------------------------------------
        # NAME LAST
        # ----------------------------------------------------

        if r.name_n:

            choices = name_last_choices.get(
                (c, r.name_n[-1])
            )

            if choices:

                results = process.extract(
                    r.name_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=TOPK
                )

                for _, _, idx in results:
                    candidate_ids.add(idx)

        # ----------------------------------------------------
        # ADDRESS FIRST
        # ----------------------------------------------------

        if r.addr_n:

            choices = addr_first_choices.get(
                (c, r.addr_n[0])
            )

            if choices:

                results = process.extract(
                    r.addr_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=TOPK
                )

                for _, _, idx in results:
                    candidate_ids.add(idx)

        # ----------------------------------------------------
        # ADDRESS LAST
        # ----------------------------------------------------

        if r.addr_n:

            choices = addr_last_choices.get(
                (c, r.addr_n[-1])
            )

            if choices:

                results = process.extract(
                    r.addr_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=TOPK
                )

                for _, _, idx in results:
                    candidate_ids.add(idx)

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        n_candidates = len(candidate_ids)

        total_candidates += n_candidates

        if n_candidates > max_candidates:
            max_candidates = n_candidates

        # ----------------------------------------------------
        # Check whether true S1 survived blocking
        # ----------------------------------------------------

        true_s1 = target_to_s1.get(
            str(r.entity_id)
        )

        if true_s1 is not None:

            evaluated += 1

            found = False

            for idx in candidate_ids:

                if str(
                    s1.iloc[idx].entity_id
                ) == true_s1:

                    found = True
                    break

            if found:
                correct += 1

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (q_no + 1) % 500 == 0:

            elapsed = time.time() - t0

            print(
                f"{q_no+1:,}/{len(sample):,}"
                f" | speed="
                f"{(q_no+1)/elapsed:.1f} q/s"
            )

    elapsed = time.time() - t0

    recall = (
        correct / evaluated
        if evaluated
        else 0
    )

    avg_candidates = (
        total_candidates / len(sample)
    )

    speed = (
        len(sample) / elapsed
    )

    print("\nRESULT")

    print(
        "TOP-K:",
        TOPK
    )

    print(
        "Correct queries:",
        correct
    )

    print(
        "Evaluated:",
        evaluated
    )

    print(
        "BLOCKING RECALL:",
        f"{recall * 100:.4f}%"
    )

    print(
        "Average candidates/query:",
        f"{avg_candidates:.2f}"
    )

    print(
        "Maximum candidates/query:",
        max_candidates
    )

    print(
        "Speed:",
        f"{speed:.2f} queries/sec"
    )

    print(
        "Projected 10M targets:",
        f"{10_000_000 / speed / 3600:.2f} hours"
    )

print("\n" + "=" * 100)
print("CELL 69 COMPLETE.")
print("=" * 100)

CELL 69 — RAPIDFUZZ MULTI-CHANNEL TOP-3 RECALL
Benchmark queries: 2,000

------------------------------------------------------------------------------------------
TESTING TOP-1 PER CHANNEL
500/2,000 | speed=1476.0 q/s
1,000/2,000 | speed=1464.8 q/s
1,500/2,000 | speed=1107.8 q/s
2,000/2,000 | speed=1193.1 q/s

RESULT
TOP-K: 1
Correct queries: 1848
Evaluated: 2000
BLOCKING RECALL: 92.4000%
Average candidates/query: 2.40
Maximum candidates/query: 4
Speed: 1193.08 queries/sec
Projected 10M targets: 2.33 hours

------------------------------------------------------------------------------------------
TESTING TOP-2 PER CHANNEL
500/2,000 | speed=1461.9 q/s
1,000/2,000 | speed=1435.9 q/s
1,500/2,000 | speed=1396.2 q/s
2,000/2,000 | speed=1405.4 q/s

RESULT
TOP-K: 2
Correct queries: 1879
Evaluated: 2000
BLOCKING RECALL: 93.9500%
Average candidates/query: 5.69
Maximum candidates/query: 8
Speed: 1405.34 queries/sec
Projected 10M targets: 1.98 hours

---------------------------------------------

In [ ]:
# CELL 70 — RAPIDFUZZ TOP-3 + EXACT NAME/ADDRESS CEILING

import pandas as pd
import numpy as np
import time
from collections import defaultdict

print("=" * 100)
print("CELL 70 — RAPIDFUZZ TOP-3 + EXACT CHANNELS")
print("=" * 100)

# ------------------------------------------------------------
# Required objects from Cell 68B/69
# ------------------------------------------------------------

required = [
    "s1",
    "targets",
    "target_to_s1",
    "name_first_choices",
    "name_last_choices",
    "addr_first_choices",
    "addr_last_choices",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing objects: " + str(missing)
    )

# ------------------------------------------------------------
# Exact indexes over S1
# ------------------------------------------------------------

exact_name = defaultdict(list)
exact_addr = defaultdict(list)

for i, r in s1.iterrows():

    if r.name_n:
        exact_name[
            (r.country_n, r.name_n)
        ].append(i)

    if r.addr_n:
        exact_addr[
            (r.country_n, r.addr_n)
        ].append(i)

print(
    "Exact name keys:",
    f"{len(exact_name):,}"
)

print(
    "Exact address keys:",
    f"{len(exact_addr):,}"
)

# ------------------------------------------------------------
# Same 2,000-query sample
# ------------------------------------------------------------

sample = targets.sample(
    n=min(
        2_000,
        len(targets)
    ),
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Candidate generation
# ------------------------------------------------------------

TOPK = 3

covered_pairs = 0
evaluated = 0

candidate_counts = []

t0 = time.time()

for q_no, r in sample.iterrows():

    c = r.country_n

    candidate_ids = set()

    # ========================================================
    # RAPIDFUZZ NAME FIRST
    # ========================================================

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:

            results = process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=TOPK
            )

            for _, _, idx in results:
                candidate_ids.add(idx)

    # ========================================================
    # RAPIDFUZZ NAME LAST
    # ========================================================

    if r.name_n:

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:

            results = process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=TOPK
            )

            for _, _, idx in results:
                candidate_ids.add(idx)

    # ========================================================
    # RAPIDFUZZ ADDRESS FIRST
    # ========================================================

    if r.addr_n:

        choices = addr_first_choices.get(
            (c, r.addr_n[0])
        )

        if choices:

            results = process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=TOPK
            )

            for _, _, idx in results:
                candidate_ids.add(idx)

    # ========================================================
    # RAPIDFUZZ ADDRESS LAST
    # ========================================================

    if r.addr_n:

        choices = addr_last_choices.get(
            (c, r.addr_n[-1])
        )

        if choices:

            results = process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=TOPK
            )

            for _, _, idx in results:
                candidate_ids.add(idx)

    # ========================================================
    # EXACT NAME
    # ========================================================

    for idx in exact_name.get(
        (c, r.name_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # EXACT ADDRESS
    # ========================================================

    for idx in exact_addr.get(
        (c, r.addr_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # Statistics
    # ========================================================

    candidate_counts.append(
        len(candidate_ids)
    )

    true_s1 = target_to_s1.get(
        str(r.entity_id)
    )

    if true_s1 is not None:

        evaluated += 1

        for idx in candidate_ids:

            if str(
                s1.iloc[idx].entity_id
            ) == true_s1:

                covered_pairs += 1
                break

    if (q_no + 1) % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{q_no+1:,}/"
            f"{len(sample):,}"
            f" | speed="
            f"{(q_no+1)/elapsed:.1f} q/s"
        )

elapsed = time.time() - t0

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

recall = (
    covered_pairs / evaluated
    if evaluated
    else 0
)

candidate_counts = np.asarray(
    candidate_counts
)

print("\n" + "=" * 100)
print("CELL 70 RESULTS")
print("=" * 100)

print(
    "Evaluated target queries:",
    f"{evaluated:,}"
)

print(
    "Covered true target queries:",
    f"{covered_pairs:,}"
)

print(
    "PAIR RECALL:",
    f"{recall * 100:.4f}%"
)

print(
    "Average candidates/query:",
    f"{candidate_counts.mean():.2f}"
)

print(
    "Median candidates/query:",
    f"{np.median(candidate_counts):.0f}"
)

print(
    "P95 candidates/query:",
    f"{np.percentile(candidate_counts,95):.0f}"
)

print(
    "P99 candidates/query:",
    f"{np.percentile(candidate_counts,99):.0f}"
)

print(
    "Max candidates/query:",
    f"{candidate_counts.max():.0f}"
)

print(
    "Speed:",
    f"{len(sample)/elapsed:.2f} queries/sec"
)

print(
    "Projected 10M target queries:",
    f"{10_000_000/(len(sample)/elapsed)/3600:.2f} hours"
)

print("\nCELL 70 COMPLETE.")

CELL 70 — RAPIDFUZZ TOP-3 + EXACT CHANNELS
Exact name keys: 19,391
Exact address keys: 19,986
500/2,000 | speed=845.6 q/s
1,000/2,000 | speed=820.7 q/s
1,500/2,000 | speed=858.0 q/s
2,000/2,000 | speed=936.6 q/s

CELL 70 RESULTS
Evaluated target queries: 2,000
Covered true target queries: 1,893
PAIR RECALL: 94.6500%
Average candidates/query: 9.04
Median candidates/query: 9
P95 candidates/query: 12
P99 candidates/query: 12
Max candidates/query: 12
Speed: 936.53 queries/sec
Projected 10M target queries: 2.97 hours

CELL 70 COMPLETE.


In [ ]:
# CELL 71 — RAPIDFUZZ + 2-CHARACTER PREFIX RECOVERY

from collections import defaultdict
import numpy as np
import time

print("=" * 100)
print("CELL 71 — RAPIDFUZZ + 2-CHARACTER PREFIX RECOVERY")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "s1",
    "targets",
    "target_to_s1",
    "name_first_choices",
    "name_last_choices",
    "addr_first_choices",
    "addr_last_choices",
    "exact_name",
    "exact_addr",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing objects: " + str(missing)
    )

# ------------------------------------------------------------
# Build 2-character prefix indexes over S1
# ------------------------------------------------------------

name_prefix2 = defaultdict(list)
addr_prefix2 = defaultdict(list)

for i, r in s1.iterrows():

    c = r.country_n

    if len(r.name_n) >= 2:
        name_prefix2[
            (c, r.name_n[:2])
        ].append(i)

    if len(r.addr_n) >= 2:
        addr_prefix2[
            (c, r.addr_n[:2])
        ].append(i)

print(
    "Name prefix-2 blocks:",
    f"{len(name_prefix2):,}"
)

print(
    "Address prefix-2 blocks:",
    f"{len(addr_prefix2):,}"
)

# ------------------------------------------------------------
# Same validation sample
# ------------------------------------------------------------

sample = targets.sample(
    n=min(
        2_000,
        len(targets)
    ),
    random_state=42
).reset_index(drop=True)

covered = 0
evaluated = 0

candidate_counts = []

t0 = time.time()

for q_no, r in sample.iterrows():

    c = r.country_n

    candidate_ids = set()

    # ========================================================
    # RAPIDFUZZ TOP-3 — NAME FIRST
    # ========================================================

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # NAME LAST
    # ========================================================

    if r.name_n:

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # ADDRESS FIRST
    # ========================================================

    if r.addr_n:

        choices = addr_first_choices.get(
            (c, r.addr_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # ADDRESS LAST
    # ========================================================

    if r.addr_n:

        choices = addr_last_choices.get(
            (c, r.addr_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # EXACT NAME
    # ========================================================

    for idx in exact_name.get(
        (c, r.name_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # EXACT ADDRESS
    # ========================================================

    for idx in exact_addr.get(
        (c, r.addr_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # NEW: NAME PREFIX-2
    # ========================================================

    if len(r.name_n) >= 2:

        for idx in name_prefix2.get(
            (c, r.name_n[:2]),
            []
        ):
            candidate_ids.add(idx)

    # ========================================================
    # NEW: ADDRESS PREFIX-2
    # ========================================================

    if len(r.addr_n) >= 2:

        for idx in addr_prefix2.get(
            (c, r.addr_n[:2]),
            []
        ):
            candidate_ids.add(idx)

    # ========================================================
    # Stats
    # ========================================================

    candidate_counts.append(
        len(candidate_ids)
    )

    true_s1 = target_to_s1.get(
        str(r.entity_id)
    )

    if true_s1 is not None:

        evaluated += 1

        for idx in candidate_ids:

            if str(
                s1.iloc[idx].entity_id
            ) == true_s1:

                covered += 1
                break

    if (q_no + 1) % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{q_no+1:,}/"
            f"{len(sample):,}"
            f" | speed="
            f"{(q_no+1)/elapsed:.1f} q/s"
        )

elapsed = time.time() - t0

candidate_counts = np.asarray(
    candidate_counts
)

print("\n" + "=" * 100)
print("CELL 71 RESULTS")
print("=" * 100)

print(
    "Evaluated:",
    evaluated
)

print(
    "Covered:",
    covered
)

print(
    "PAIR RECALL:",
    f"{covered/evaluated*100:.4f}%"
)

print(
    "Average candidates/query:",
    f"{candidate_counts.mean():.2f}"
)

print(
    "Median candidates/query:",
    f"{np.median(candidate_counts):.0f}"
)

print(
    "P95 candidates/query:",
    f"{np.percentile(candidate_counts,95):.0f}"
)

print(
    "P99 candidates/query:",
    f"{np.percentile(candidate_counts,99):.0f}"
)

print(
    "Maximum candidates/query:",
    f"{candidate_counts.max():.0f}"
)

print(
    "Speed:",
    f"{len(sample)/elapsed:.2f} q/s"
)

print(
    "Projected 10M targets:",
    f"{10_000_000/(len(sample)/elapsed)/3600:.2f} hours"
)

print("\nCELL 71 COMPLETE.")

CELL 71 — RAPIDFUZZ + 2-CHARACTER PREFIX RECOVERY
Name prefix-2 blocks: 1,115
Address prefix-2 blocks: 1,047
500/2,000 | speed=280.2 q/s
1,000/2,000 | speed=271.3 q/s
1,500/2,000 | speed=264.5 q/s
2,000/2,000 | speed=262.5 q/s

CELL 71 RESULTS
Evaluated: 2000
Covered: 1916
PAIR RECALL: 95.8000%
Average candidates/query: 222.96
Median candidates/query: 180
P95 candidates/query: 573
P99 candidates/query: 691
Maximum candidates/query: 854
Speed: 262.49 q/s
Projected 10M targets: 10.58 hours

CELL 71 COMPLETE.


In [ ]:
# CELL 72 — 2-CHAR PREFIX + RAPIDFUZZ TOP-K
# Test K = 3, 5, 10 inside the prefix blocks

from rapidfuzz import process, fuzz
import numpy as np
import time

print("=" * 100)
print("CELL 72 — 2-CHAR PREFIX FUZZY TOP-K")
print("=" * 100)

required = [
    "s1",
    "targets",
    "target_to_s1",
    "name_first_choices",
    "name_last_choices",
    "addr_first_choices",
    "addr_last_choices",
    "exact_name",
    "exact_addr",
    "name_prefix2",
    "addr_prefix2",
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing objects: " + str(missing)
    )

sample = targets.sample(
    n=min(2000, len(targets)),
    random_state=42
).reset_index(drop=True)

print("Validation queries:", len(sample))

# ------------------------------------------------------------
# Test different K values inside prefix blocks
# ------------------------------------------------------------

for PREFIX_K in [3, 5, 10]:

    covered = 0
    evaluated = 0
    candidate_counts = []

    t0 = time.time()

    for q_no, r in sample.iterrows():

        c = r.country_n

        candidate_ids = set()

        # ====================================================
        # EXISTING RAPIDFUZZ TOP-3 CHANNELS
        # ====================================================

        if r.name_n:

            choices = name_first_choices.get(
                (c, r.name_n[0])
            )

            if choices:
                for _, _, idx in process.extract(
                    r.name_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

            choices = name_last_choices.get(
                (c, r.name_n[-1])
            )

            if choices:
                for _, _, idx in process.extract(
                    r.name_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

        if r.addr_n:

            choices = addr_first_choices.get(
                (c, r.addr_n[0])
            )

            if choices:
                for _, _, idx in process.extract(
                    r.addr_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

            choices = addr_last_choices.get(
                (c, r.addr_n[-1])
            )

            if choices:
                for _, _, idx in process.extract(
                    r.addr_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

        # ====================================================
        # EXACT CHANNELS
        # ====================================================

        for idx in exact_name.get(
            (c, r.name_n),
            []
        ):
            candidate_ids.add(idx)

        for idx in exact_addr.get(
            (c, r.addr_n),
            []
        ):
            candidate_ids.add(idx)

        # ====================================================
        # 2-CHAR NAME PREFIX — FUZZY TOP-K
        # ====================================================

        if len(r.name_n) >= 2:

            choices = name_prefix2.get(
                (c, r.name_n[:2]),
                []
            )

            if choices:

                # Convert S1 indices to strings for RapidFuzz
                prefix_choices = {
                    idx: s1.iloc[idx].name_n
                    for idx in choices
                }

                for _, _, idx in process.extract(
                    r.name_n,
                    prefix_choices,
                    scorer=fuzz.ratio,
                    limit=PREFIX_K
                ):
                    candidate_ids.add(idx)

        # ====================================================
        # 2-CHAR ADDRESS PREFIX — FUZZY TOP-K
        # ====================================================

        if len(r.addr_n) >= 2:

            choices = addr_prefix2.get(
                (c, r.addr_n[:2]),
                []
            )

            if choices:

                prefix_choices = {
                    idx: s1.iloc[idx].addr_n
                    for idx in choices
                }

                for _, _, idx in process.extract(
                    r.addr_n,
                    prefix_choices,
                    scorer=fuzz.ratio,
                    limit=PREFIX_K
                ):
                    candidate_ids.add(idx)

        # ====================================================
        # EVALUATE
        # ====================================================

        candidate_counts.append(
            len(candidate_ids)
        )

        true_s1 = target_to_s1.get(
            str(r.entity_id)
        )

        if true_s1 is not None:

            evaluated += 1

            true_found = False

            for idx in candidate_ids:

                if str(
                    s1.iloc[idx].entity_id
                ) == true_s1:

                    true_found = True
                    break

            if true_found:
                covered += 1

        if (q_no + 1) % 500 == 0:

            elapsed = time.time() - t0

            print(
                f"K={PREFIX_K} | "
                f"{q_no+1:,}/{len(sample):,} | "
                f"speed={(q_no+1)/elapsed:.1f} q/s"
            )

    elapsed = time.time() - t0

    candidate_counts = np.asarray(
        candidate_counts
    )

    recall = covered / evaluated

    speed = len(sample) / elapsed

    print("\n" + "-" * 90)
    print(
        f"PREFIX_K = {PREFIX_K}"
    )

    print(
        f"Pair recall: {recall*100:.4f}%"
    )

    print(
        f"Average candidates: "
        f"{candidate_counts.mean():.2f}"
    )

    print(
        f"Median candidates: "
        f"{np.median(candidate_counts):.0f}"
    )

    print(
        f"P95 candidates: "
        f"{np.percentile(candidate_counts,95):.0f}"
    )

    print(
        f"P99 candidates: "
        f"{np.percentile(candidate_counts,99):.0f}"
    )

    print(
        f"Max candidates: "
        f"{candidate_counts.max():.0f}"
    )

    print(
        f"Speed: {speed:.2f} q/s"
    )

    print(
        f"Projected 10M: "
        f"{10_000_000/speed/3600:.2f} hours"
    )

print("\nCELL 72 COMPLETE.")

CELL 72 — 2-CHAR PREFIX FUZZY TOP-K
Validation queries: 2000
K=3 | 500/2,000 | speed=110.7 q/s
K=3 | 1,000/2,000 | speed=127.8 q/s
K=3 | 1,500/2,000 | speed=132.9 q/s
K=3 | 2,000/2,000 | speed=131.4 q/s

------------------------------------------------------------------------------------------
PREFIX_K = 3
Pair recall: 95.0000%
Average candidates: 11.19
Median candidates: 12
P95 candidates: 15
P99 candidates: 17
Max candidates: 18
Speed: 131.41 q/s
Projected 10M: 21.14 hours
K=5 | 500/2,000 | speed=145.5 q/s
K=5 | 1,000/2,000 | speed=146.3 q/s
K=5 | 1,500/2,000 | speed=146.8 q/s
K=5 | 2,000/2,000 | speed=138.4 q/s

------------------------------------------------------------------------------------------
PREFIX_K = 5
Pair recall: 95.2500%
Average candidates: 14.50
Median candidates: 15
P95 candidates: 19
P99 candidates: 21
Max candidates: 21
Speed: 138.43 q/s
Projected 10M: 20.07 hours
K=10 | 500/2,000 | speed=159.6 q/s
K=10 | 1,000/2,000 | speed=151.6 q/s
K=10 | 1,500/2,000 | speed=14

In [ ]:
# CELL 73 — ANALYZE RAPIDFUZZ MISSED TRUE PAIRS

from rapidfuzz import fuzz
import numpy as np
import pandas as pd
import re

print("=" * 100)
print("CELL 73 — MISSED TRUE-PAIR ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# Use the same 2,000 validation targets
# ------------------------------------------------------------

sample = targets.sample(
    n=min(2000, len(targets)),
    random_state=42
).reset_index(drop=True)

missed = []
covered = []

for _, r in sample.iterrows():

    c = r.country_n
    candidate_ids = set()

    # --------------------------------------------------------
    # RapidFuzz Top-3
    # --------------------------------------------------------

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:
            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:
            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # --------------------------------------------------------
    # Address
    # --------------------------------------------------------

    if r.addr_n:

        choices = addr_first_choices.get(
            (c, r.addr_n[0])
        )

        if choices:
            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        choices = addr_last_choices.get(
            (c, r.addr_n[-1])
        )

        if choices:
            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # --------------------------------------------------------
    # Exact channels
    # --------------------------------------------------------

    for idx in exact_name.get(
        (c, r.name_n),
        []
    ):
        candidate_ids.add(idx)

    for idx in exact_addr.get(
        (c, r.addr_n),
        []
    ):
        candidate_ids.add(idx)

    # --------------------------------------------------------
    # Find true S1
    # --------------------------------------------------------

    true_s1 = target_to_s1.get(
        str(r.entity_id)
    )

    if true_s1 is None:
        continue

    true_idx = None

    for idx in range(len(s1)):

        if str(
            s1.iloc[idx].entity_id
        ) == true_s1:

            true_idx = idx
            break

    if true_idx is None:
        continue

    is_covered = true_idx in candidate_ids

    # --------------------------------------------------------
    # Similarity of the TRUE pair
    # --------------------------------------------------------

    true_row = s1.iloc[true_idx]

    name_ratio = fuzz.ratio(
        r.name_n,
        true_row.name_n
    ) if r.name_n and true_row.name_n else 0

    addr_ratio = fuzz.ratio(
        r.addr_n,
        true_row.addr_n
    ) if r.addr_n and true_row.addr_n else 0

    name_partial = fuzz.partial_ratio(
        r.name_n,
        true_row.name_n
    ) if r.name_n and true_row.name_n else 0

    addr_partial = fuzz.partial_ratio(
        r.addr_n,
        true_row.addr_n
    ) if r.addr_n and true_row.addr_n else 0

    name_token = fuzz.token_set_ratio(
        r.name_n,
        true_row.name_n
    ) if r.name_n and true_row.name_n else 0

    addr_token = fuzz.token_set_ratio(
        r.addr_n,
        true_row.addr_n
    ) if r.addr_n and true_row.addr_n else 0

    # Number overlap
    q_nums = set(
        re.findall(r'\d+', r.addr_n)
    )

    t_nums = set(
        re.findall(r'\d+', true_row.addr_n)
    )

    num_overlap = len(
        q_nums & t_nums
    )

    row = {
        "target_id": str(r.entity_id),
        "true_s1": true_s1,
        "covered": is_covered,
        "name_ratio": name_ratio,
        "addr_ratio": addr_ratio,
        "name_partial": name_partial,
        "addr_partial": addr_partial,
        "name_token": name_token,
        "addr_token": addr_token,
        "num_overlap": num_overlap,
        "name_len_q": len(r.name_n),
        "name_len_s1": len(true_row.name_n),
        "addr_len_q": len(r.addr_n),
        "addr_len_s1": len(true_row.addr_n),
    }

    if is_covered:
        covered.append(row)
    else:
        missed.append(row)

# ------------------------------------------------------------
# DataFrames
# ------------------------------------------------------------

df_cov = pd.DataFrame(covered)
df_miss = pd.DataFrame(missed)

print("\n" + "=" * 100)
print("RESULT")
print("=" * 100)

print(
    "Covered true pairs:",
    len(df_cov)
)

print(
    "Missed true pairs:",
    len(df_miss)
)

print(
    "Recall:",
    f"{len(df_cov)/(len(df_cov)+len(df_miss))*100:.4f}%"
)

# ------------------------------------------------------------
# Compare similarity distributions
# ------------------------------------------------------------

metrics = [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "name_token",
    "addr_token",
    "num_overlap",
]

print("\n" + "=" * 100)
print("COVERED VS MISSED")
print("=" * 100)

for col in metrics:

    if col not in df_miss.columns:
        continue

    print(
        f"{col:18s} | "
        f"covered mean={df_cov[col].mean():7.2f} | "
        f"missed mean={df_miss[col].mean():7.2f} | "
        f"missed median={df_miss[col].median():7.2f}"
    )

# ------------------------------------------------------------
# Missed pair buckets
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MISSED PAIR BUCKETS")
print("=" * 100)

print(
    "Name ratio >= 80:",
    f"{(df_miss.name_ratio >= 80).mean()*100:.2f}%"
)

print(
    "Name ratio >= 90:",
    f"{(df_miss.name_ratio >= 90).mean()*100:.2f}%"
)

print(
    "Address ratio >= 80:",
    f"{(df_miss.addr_ratio >= 80).mean()*100:.2f}%"
)

print(
    "Address ratio >= 90:",
    f"{(df_miss.addr_ratio >= 90).mean()*100:.2f}%"
)

print(
    "Number overlap >= 1:",
    f"{(df_miss.num_overlap >= 1).mean()*100:.2f}%"
)

print(
    "Name + address ratio >= 70:",
    f"{((df_miss.name_ratio >= 70) & "
       "(df_miss.addr_ratio >= 70)).mean()*100:.2f}%"
)

print("\nCELL 73 COMPLETE.")

SyntaxError: f-string: expecting '}' (2570930345.py, line 298)

In [ ]:
# CELL 73 — ANALYZE RAPIDFUZZ MISSED TRUE PAIRS
# Corrected + faster true-S1 lookup

from rapidfuzz import process, fuzz
import numpy as np
import pandas as pd
import re

print("=" * 100)
print("CELL 73 — MISSED TRUE-PAIR ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# Build S1 entity_id -> row index lookup
# ------------------------------------------------------------

s1_id_to_idx = {
    str(eid): idx
    for idx, eid in enumerate(s1.entity_id)
}

print(
    "S1 lookup entries:",
    f"{len(s1_id_to_idx):,}"
)

# ------------------------------------------------------------
# Same validation sample
# ------------------------------------------------------------

sample = targets.sample(
    n=min(2000, len(targets)),
    random_state=42
).reset_index(drop=True)

missed = []
covered = []

t0 = time.time()

for q_no, r in sample.iterrows():

    c = r.country_n
    candidate_ids = set()

    # ========================================================
    # RAPIDFUZZ TOP-3 — NAME FIRST
    # ========================================================

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        # ====================================================
        # NAME LAST
        # ====================================================

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # ADDRESS FIRST
    # ========================================================

    if r.addr_n:

        choices = addr_first_choices.get(
            (c, r.addr_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        # ====================================================
        # ADDRESS LAST
        # ====================================================

        choices = addr_last_choices.get(
            (c, r.addr_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # EXACT NAME
    # ========================================================

    for idx in exact_name.get(
        (c, r.name_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # EXACT ADDRESS
    # ========================================================

    for idx in exact_addr.get(
        (c, r.addr_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # TRUE S1
    # ========================================================

    true_s1 = target_to_s1.get(
        str(r.entity_id)
    )

    if true_s1 is None:
        continue

    true_idx = s1_id_to_idx.get(
        str(true_s1)
    )

    if true_idx is None:
        continue

    true_row = s1.iloc[true_idx]

    is_covered = (
        true_idx in candidate_ids
    )

    # ========================================================
    # TRUE-PAIR SIMILARITY
    # ========================================================

    name_ratio = (
        fuzz.ratio(
            r.name_n,
            true_row.name_n
        )
        if r.name_n and true_row.name_n
        else 0
    )

    addr_ratio = (
        fuzz.ratio(
            r.addr_n,
            true_row.addr_n
        )
        if r.addr_n and true_row.addr_n
        else 0
    )

    name_partial = (
        fuzz.partial_ratio(
            r.name_n,
            true_row.name_n
        )
        if r.name_n and true_row.name_n
        else 0
    )

    addr_partial = (
        fuzz.partial_ratio(
            r.addr_n,
            true_row.addr_n
        )
        if r.addr_n and true_row.addr_n
        else 0
    )

    name_token = (
        fuzz.token_set_ratio(
            r.name_n,
            true_row.name_n
        )
        if r.name_n and true_row.name_n
        else 0
    )

    addr_token = (
        fuzz.token_set_ratio(
            r.addr_n,
            true_row.addr_n
        )
        if r.addr_n and true_row.addr_n
        else 0
    )

    # ========================================================
    # NUMBER OVERLAP
    # ========================================================

    q_nums = set(
        re.findall(
            r'\d+',
            r.addr_n
        )
    )

    t_nums = set(
        re.findall(
            r'\d+',
            true_row.addr_n
        )
    )

    num_overlap = len(
        q_nums & t_nums
    )

    row = {
        "target_id": str(r.entity_id),
        "true_s1": str(true_s1),
        "covered": is_covered,

        "name_ratio": name_ratio,
        "addr_ratio": addr_ratio,

        "name_partial": name_partial,
        "addr_partial": addr_partial,

        "name_token": name_token,
        "addr_token": addr_token,

        "num_overlap": num_overlap,

        "name_len_q": len(r.name_n),
        "name_len_s1": len(true_row.name_n),

        "addr_len_q": len(r.addr_n),
        "addr_len_s1": len(true_row.addr_n),
    }

    if is_covered:
        covered.append(row)
    else:
        missed.append(row)

    if (q_no + 1) % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{q_no + 1:,}/"
            f"{len(sample):,}"
            f" | speed="
            f"{(q_no + 1) / elapsed:.1f} q/s"
        )

# ------------------------------------------------------------
# DataFrames
# ------------------------------------------------------------

df_cov = pd.DataFrame(covered)
df_miss = pd.DataFrame(missed)

print("\n" + "=" * 100)
print("RESULT")
print("=" * 100)

total = len(df_cov) + len(df_miss)

print(
    "Covered true pairs:",
    len(df_cov)
)

print(
    "Missed true pairs:",
    len(df_miss)
)

print(
    "Recall:",
    f"{len(df_cov) / total * 100:.4f}%"
)

# ------------------------------------------------------------
# Similarity comparison
# ------------------------------------------------------------

metrics = [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "name_token",
    "addr_token",
    "num_overlap",
]

print("\n" + "=" * 100)
print("COVERED VS MISSED")
print("=" * 100)

for col in metrics:

    print(
        f"{col:18s} | "
        f"covered mean={df_cov[col].mean():7.2f} | "
        f"missed mean={df_miss[col].mean():7.2f} | "
        f"missed median={df_miss[col].median():7.2f}"
    )

# ------------------------------------------------------------
# MISSED PAIR BUCKETS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MISSED PAIR BUCKETS")
print("=" * 100)

print(
    "Name ratio >= 80:",
    f"{(df_miss['name_ratio'] >= 80).mean() * 100:.2f}%"
)

print(
    "Name ratio >= 90:",
    f"{(df_miss['name_ratio'] >= 90).mean() * 100:.2f}%"
)

print(
    "Address ratio >= 80:",
    f"{(df_miss['addr_ratio'] >= 80).mean() * 100:.2f}%"
)

print(
    "Address ratio >= 90:",
    f"{(df_miss['addr_ratio'] >= 90).mean() * 100:.2f}%"
)

print(
    "Number overlap >= 1:",
    f"{(df_miss['num_overlap'] >= 1).mean() * 100:.2f}%"
)

name_addr_condition = (
    (df_miss["name_ratio"] >= 70)
    &
    (df_miss["addr_ratio"] >= 70)
)

print(
    "Name + address ratio >= 70:",
    f"{name_addr_condition.mean() * 100:.2f}%"
)

print("\nCELL 73 COMPLETE.")

CELL 73 — MISSED TRUE-PAIR ANALYSIS
S1 lookup entries: 20,000
500/2,000 | speed=719.0 q/s
1,000/2,000 | speed=737.4 q/s
1,500/2,000 | speed=737.5 q/s
2,000/2,000 | speed=735.4 q/s

RESULT
Covered true pairs: 1893
Missed true pairs: 107
Recall: 94.6500%

COVERED VS MISSED
name_ratio         | covered mean=  83.02 | missed mean=  36.68 | missed median=  40.00
addr_ratio         | covered mean=  76.23 | missed mean=  56.72 | missed median=  56.41
name_partial       | covered mean=  90.09 | missed mean=  46.68 | missed median=  57.14
addr_partial       | covered mean=  82.49 | missed mean=  72.83 | missed median=  76.47
name_token         | covered mean=  83.02 | missed mean=  36.68 | missed median=  40.00
addr_token         | covered mean=  76.23 | missed mean=  56.72 | missed median=  56.41
num_overlap        | covered mean=   0.86 | missed mean=   0.94 | missed median=   1.00

MISSED PAIR BUCKETS
Name ratio >= 80: 14.02%
Name ratio >= 90: 0.93%
Address ratio >= 80: 13.08%
Address ratio 

In [ ]:
# CELL 74 — NUMBER-AWARE ADDRESS RECOVERY

from collections import defaultdict
from rapidfuzz import process, fuzz
import numpy as np
import re
import time

print("=" * 100)
print("CELL 74 — NUMBER-AWARE ADDRESS RECOVERY")
print("=" * 100)

# ------------------------------------------------------------
# Build country + address-number index
# ------------------------------------------------------------

number_addr_choices = defaultdict(dict)

for idx, r in s1.iterrows():

    numbers = set(
        re.findall(
            r'\d+',
            r.addr_n
        )
    )

    for num in numbers:

        key = (
            r.country_n,
            num
        )

        number_addr_choices[key][idx] = r.addr_n

print(
    "Country+number blocks:",
    f"{len(number_addr_choices):,}"
)

# ------------------------------------------------------------
# Inspect block sizes
# ------------------------------------------------------------

block_sizes = np.array([
    len(v)
    for v in number_addr_choices.values()
])

print(
    "Median block size:",
    int(np.median(block_sizes))
)

print(
    "P95 block size:",
    int(np.percentile(block_sizes, 95))
)

print(
    "P99 block size:",
    int(np.percentile(block_sizes, 99))
)

print(
    "Maximum block size:",
    int(block_sizes.max())
)

# ------------------------------------------------------------
# Same validation sample
# ------------------------------------------------------------

sample = targets.sample(
    n=min(2000, len(targets)),
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Test K values
# ------------------------------------------------------------

for NUMBER_K in [3, 5, 10]:

    covered = 0
    evaluated = 0

    candidate_counts = []

    t0 = time.time()

    for q_no, r in sample.iterrows():

        c = r.country_n

        candidate_ids = set()

        # ====================================================
        # EXISTING RAPIDFUZZ TOP-3
        # ====================================================

        if r.name_n:

            choices = name_first_choices.get(
                (c, r.name_n[0])
            )

            if choices:

                for _, _, idx in process.extract(
                    r.name_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

            choices = name_last_choices.get(
                (c, r.name_n[-1])
            )

            if choices:

                for _, _, idx in process.extract(
                    r.name_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

        # ====================================================
        # ADDRESS FIRST
        # ====================================================

        if r.addr_n:

            choices = addr_first_choices.get(
                (c, r.addr_n[0])
            )

            if choices:

                for _, _, idx in process.extract(
                    r.addr_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

            # ADDRESS LAST

            choices = addr_last_choices.get(
                (c, r.addr_n[-1])
            )

            if choices:

                for _, _, idx in process.extract(
                    r.addr_n,
                    choices,
                    scorer=fuzz.ratio,
                    limit=3
                ):
                    candidate_ids.add(idx)

        # ====================================================
        # EXACT CHANNELS
        # ====================================================

        for idx in exact_name.get(
            (c, r.name_n),
            []
        ):
            candidate_ids.add(idx)

        for idx in exact_addr.get(
            (c, r.addr_n),
            []
        ):
            candidate_ids.add(idx)

        # ====================================================
        # NEW: ADDRESS NUMBER RECOVERY
        # ====================================================

        query_numbers = set(
            re.findall(
                r'\d+',
                r.addr_n
            )
        )

        # Use the union of candidates from all numbers
        # associated with this address.

        number_candidates = {}

        for num in query_numbers:

            choices = number_addr_choices.get(
                (c, num)
            )

            if not choices:
                continue

            results = process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=NUMBER_K
            )

            for _, score, idx in results:

                # Keep strongest score for each S1 row
                previous = number_candidates.get(
                    idx,
                    -1
                )

                if score > previous:
                    number_candidates[idx] = score

        # Add number-aware candidates
        candidate_ids.update(
            number_candidates.keys()
        )

        # ====================================================
        # STATS
        # ====================================================

        candidate_counts.append(
            len(candidate_ids)
        )

        # ====================================================
        # TRUE PAIR
        # ====================================================

        true_s1 = target_to_s1.get(
            str(r.entity_id)
        )

        if true_s1 is None:
            continue

        true_idx = s1_id_to_idx.get(
            str(true_s1)
        )

        if true_idx is None:
            continue

        evaluated += 1

        if true_idx in candidate_ids:

            covered += 1

        if (q_no + 1) % 500 == 0:

            elapsed = time.time() - t0

            print(
                f"K={NUMBER_K} | "
                f"{q_no+1:,}/"
                f"{len(sample):,} | "
                f"speed="
                f"{(q_no+1)/elapsed:.1f} q/s"
            )

    elapsed = time.time() - t0

    candidate_counts = np.asarray(
        candidate_counts
    )

    print("\n" + "-" * 90)

    print(
        f"NUMBER_K = {NUMBER_K}"
    )

    print(
        "Pair recall:",
        f"{covered/evaluated*100:.4f}%"
    )

    print(
        "Average candidates:",
        f"{candidate_counts.mean():.2f}"
    )

    print(
        "Median candidates:",
        f"{np.median(candidate_counts):.0f}"
    )

    print(
        "P95 candidates:",
        f"{np.percentile(candidate_counts,95):.0f}"
    )

    print(
        "P99 candidates:",
        f"{np.percentile(candidate_counts,99):.0f}"
    )

    print(
        "Maximum candidates:",
        f"{candidate_counts.max():.0f}"
    )

    print(
        "Speed:",
        f"{len(sample)/elapsed:.2f} q/s"
    )

    print(
        "Projected 10M:",
        f"{10_000_000/(len(sample)/elapsed)/3600:.2f} hours"
    )

print("\nCELL 74 COMPLETE.")

CELL 74 — NUMBER-AWARE ADDRESS RECOVERY
Country+number blocks: 9,371
Median block size: 1
P95 block size: 7
P99 block size: 26
Maximum block size: 614
K=3 | 500/2,000 | speed=831.7 q/s
K=3 | 1,000/2,000 | speed=865.6 q/s
K=3 | 1,500/2,000 | speed=859.1 q/s
K=3 | 2,000/2,000 | speed=849.5 q/s

------------------------------------------------------------------------------------------
NUMBER_K = 3
Pair recall: 98.1500%
Average candidates: 10.45
Median candidates: 11
P95 candidates: 14
P99 candidates: 16
Maximum candidates: 19
Speed: 849.37 q/s
Projected 10M: 3.27 hours
K=5 | 500/2,000 | speed=926.6 q/s
K=5 | 1,000/2,000 | speed=982.9 q/s
K=5 | 1,500/2,000 | speed=1123.3 q/s
K=5 | 2,000/2,000 | speed=1205.5 q/s

------------------------------------------------------------------------------------------
NUMBER_K = 5
Pair recall: 98.3000%
Average candidates: 11.52
Median candidates: 11
P95 candidates: 17
P99 candidates: 21
Maximum candidates: 25
Speed: 1205.43 q/s
Projected 10M: 2.30 hours
K=

In [ ]:
# CELL 75 — K=10 CANDIDATE CEILING + ENTITY COVERAGE

from rapidfuzz import process, fuzz
import numpy as np
import pandas as pd
import re
import time

print("=" * 100)
print("CELL 75 — K=10 CANDIDATE CEILING")
print("=" * 100)

# ------------------------------------------------------------
# Validation sample
# ------------------------------------------------------------

sample = targets.sample(
    n=min(2000, len(targets)),
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Candidate generator
# ------------------------------------------------------------

def generate_k10_candidates(r):

    c = r.country_n

    candidate_ids = set()

    # ========================================================
    # NAME FIRST TOP-3
    # ========================================================

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        # ====================================================
        # NAME LAST TOP-3
        # ====================================================

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # ADDRESS FIRST TOP-3
    # ========================================================

    if r.addr_n:

        choices = addr_first_choices.get(
            (c, r.addr_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        # ====================================================
        # ADDRESS LAST TOP-3
        # ====================================================

        choices = addr_last_choices.get(
            (c, r.addr_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # EXACT NAME
    # ========================================================

    for idx in exact_name.get(
        (c, r.name_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # EXACT ADDRESS
    # ========================================================

    for idx in exact_addr.get(
        (c, r.addr_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # NUMBER-AWARE ADDRESS TOP-10
    # ========================================================

    query_numbers = set(
        re.findall(
            r'\d+',
            r.addr_n
        )
    )

    number_candidates = {}

    for num in query_numbers:

        choices = number_addr_choices.get(
            (c, num)
        )

        if not choices:
            continue

        results = process.extract(
            r.addr_n,
            choices,
            scorer=fuzz.ratio,
            limit=10
        )

        for _, score, idx in results:

            previous = number_candidates.get(
                idx,
                -1
            )

            if score > previous:
                number_candidates[idx] = score

    candidate_ids.update(
        number_candidates.keys()
    )

    return candidate_ids


# ------------------------------------------------------------
# Ground-truth lookup
# ------------------------------------------------------------

# target -> true S1
target_to_s1_local = target_to_s1

# S1 ID -> row index
s1_id_to_idx_local = s1_id_to_idx

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

total_true_pairs = 0
covered_true_pairs = 0

total_entities = 0
fully_covered_entities = 0

candidate_counts = []

rows = []

t0 = time.time()

for q_no, r in sample.iterrows():

    true_s1 = target_to_s1_local.get(
        str(r.entity_id)
    )

    # Skip records without a known ground-truth match
    if true_s1 is None:
        continue

    true_idx = s1_id_to_idx_local.get(
        str(true_s1)
    )

    if true_idx is None:
        continue

    candidates = generate_k10_candidates(r)

    candidate_counts.append(
        len(candidates)
    )

    total_entities += 1
    total_true_pairs += 1

    if true_idx in candidates:

        covered_true_pairs += 1
        fully_covered_entities += 1

    rows.append({
        "target_id": str(r.entity_id),
        "true_s1": str(true_s1),
        "true_idx": true_idx,
        "candidate_count": len(candidates),
        "covered": true_idx in candidates
    })

    if (q_no + 1) % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{q_no+1:,}/"
            f"{len(sample):,} | "
            f"speed="
            f"{(q_no+1)/elapsed:.1f} q/s"
        )

elapsed = time.time() - t0

candidate_counts = np.asarray(
    candidate_counts
)

df_ceiling = pd.DataFrame(rows)

# ------------------------------------------------------------
# Pair recall
# ------------------------------------------------------------

pair_recall = (
    covered_true_pairs /
    total_true_pairs
)

# ------------------------------------------------------------
# Entity coverage
# ------------------------------------------------------------

entity_coverage = (
    fully_covered_entities /
    total_entities
)

# ------------------------------------------------------------
# Candidate-constrained F0.5 ceiling
#
# For each entity:
#
# If true pair is available:
#     perfect prediction gives F=1
#
# If true pair is unavailable:
#     perfect candidate-constrained prediction
#     still gives F=0
#
# This is the maximum achievable entity-level
# macro score under this candidate generator.
# ------------------------------------------------------------

candidate_ceiling = entity_coverage

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 75 RESULTS")
print("=" * 100)

print(
    "Evaluated entities:",
    f"{total_entities:,}"
)

print(
    "Covered true pairs:",
    f"{covered_true_pairs:,}"
)

print(
    "Total true pairs:",
    f"{total_true_pairs:,}"
)

print(
    "PAIR RECALL:",
    f"{pair_recall*100:.4f}%"
)

print(
    "Fully covered entities:",
    f"{fully_covered_entities:,}"
)

print(
    "ENTITY COVERAGE:",
    f"{entity_coverage*100:.4f}%"
)

print(
    "Candidate-constrained F0.5 ceiling:",
    f"{candidate_ceiling:.6f}"
)

print("\nCandidate statistics:")

print(
    "Average:",
    f"{candidate_counts.mean():.2f}"
)

print(
    "Median:",
    f"{np.median(candidate_counts):.0f}"
)

print(
    "P95:",
    f"{np.percentile(candidate_counts,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(candidate_counts,99):.0f}"
)

print(
    "Maximum:",
    f"{candidate_counts.max():.0f}"
)

print(
    "Generation speed:",
    f"{len(sample)/elapsed:.2f} q/s"
)

print(
    "Projected 10M targets:",
    f"{10_000_000/(len(sample)/elapsed)/3600:.2f} hours"
)

print("\nCELL 75 COMPLETE.")

CELL 75 — K=10 CANDIDATE CEILING
500/2,000 | speed=974.6 q/s
1,000/2,000 | speed=948.3 q/s
1,500/2,000 | speed=792.5 q/s
2,000/2,000 | speed=721.3 q/s

CELL 75 RESULTS
Evaluated entities: 2,000
Covered true pairs: 1,972
Total true pairs: 2,000
PAIR RECALL: 98.6000%
Fully covered entities: 1,972
ENTITY COVERAGE: 98.6000%
Candidate-constrained F0.5 ceiling: 0.986000

Candidate statistics:
Average: 13.78
Median: 12
P95: 26
P99: 34
Maximum: 44
Generation speed: 721.28 q/s
Projected 10M targets: 3.85 hours

CELL 75 COMPLETE.


In [ ]:
# CELL 76 — BUILD K=10 MATCHER DATASET
#
# Generates candidate pairs and rich pair-level features
# for LightGBM training.

from rapidfuzz import process, fuzz
import numpy as np
import pandas as pd
import re
import time

print("=" * 100)
print("CELL 76 — BUILD K=10 MATCHER DATASET")
print("=" * 100)

# ------------------------------------------------------------
# Validation sample
# ------------------------------------------------------------

sample = targets.sample(
    n=min(2000, len(targets)),
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Feature helper
# ------------------------------------------------------------

def safe_ratio(a, b):

    if not a or not b:
        return 0.0

    return fuzz.ratio(a, b) / 100.0


def safe_partial(a, b):

    if not a or not b:
        return 0.0

    return fuzz.partial_ratio(a, b) / 100.0


def safe_token(a, b):

    if not a or not b:
        return 0.0

    return fuzz.token_set_ratio(a, b) / 100.0


def length_ratio(a, b):

    la = len(a)
    lb = len(b)

    if la == 0 or lb == 0:
        return 0.0

    return min(la, lb) / max(la, lb)


def number_overlap(a, b):

    na = set(
        re.findall(
            r'\d+',
            a
        )
    )

    nb = set(
        re.findall(
            r'\d+',
            b
        )
    )

    if not na:
        return 0.0

    return len(na & nb) / len(na)


# ------------------------------------------------------------
# Build candidate rows
# ------------------------------------------------------------

rows = []

t0 = time.time()

for q_no, r in sample.iterrows():

    country = r.country_n

    candidates = {}

    # ========================================================
    # NAME FIRST TOP-3
    # ========================================================

    if r.name_n:

        choices = name_first_choices.get(
            (country, r.name_n[0])
        )

        if choices:

            for _, score, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):

                candidates.setdefault(
                    idx,
                    set()
                ).add(
                    "name_first"
                )

        # ====================================================
        # NAME LAST TOP-3
        # ====================================================

        choices = name_last_choices.get(
            (country, r.name_n[-1])
        )

        if choices:

            for _, score, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):

                candidates.setdefault(
                    idx,
                    set()
                ).add(
                    "name_last"
                )

    # ========================================================
    # ADDRESS FIRST TOP-3
    # ========================================================

    if r.addr_n:

        choices = addr_first_choices.get(
            (country, r.addr_n[0])
        )

        if choices:

            for _, score, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):

                candidates.setdefault(
                    idx,
                    set()
                ).add(
                    "addr_first"
                )

        # ====================================================
        # ADDRESS LAST TOP-3
        # ====================================================

        choices = addr_last_choices.get(
            (country, r.addr_n[-1])
        )

        if choices:

            for _, score, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):

                candidates.setdefault(
                    idx,
                    set()
                ).add(
                    "addr_last"
                )

    # ========================================================
    # EXACT NAME
    # ========================================================

    for idx in exact_name.get(
        (country, r.name_n),
        []
    ):

        candidates.setdefault(
            idx,
            set()
        ).add(
            "exact_name"
        )

    # ========================================================
    # EXACT ADDRESS
    # ========================================================

    for idx in exact_addr.get(
        (country, r.addr_n),
        []
    ):

        candidates.setdefault(
            idx,
            set()
        ).add(
            "exact_address"
        )

    # ========================================================
    # NUMBER-AWARE ADDRESS TOP-10
    # ========================================================

    query_numbers = set(
        re.findall(
            r'\d+',
            r.addr_n
        )
    )

    number_candidates = {}

    for num in query_numbers:

        choices = number_addr_choices.get(
            (country, num)
        )

        if not choices:
            continue

        results = process.extract(
            r.addr_n,
            choices,
            scorer=fuzz.ratio,
            limit=10
        )

        for _, score, idx in results:

            previous = number_candidates.get(
                idx,
                -1
            )

            if score > previous:
                number_candidates[idx] = score

    for idx in number_candidates:

        candidates.setdefault(
            idx,
            set()
        ).add(
            "number_address"
        )

    # ========================================================
    # TRUE S1
    # ========================================================

    true_s1 = target_to_s1.get(
        str(r.entity_id)
    )

    true_idx = (
        s1_id_to_idx.get(
            str(true_s1)
        )
        if true_s1 is not None
        else None
    )

    # ========================================================
    # CREATE FEATURE ROWS
    # ========================================================

    for idx, channels in candidates.items():

        s = s1.iloc[idx]

        name_ratio = safe_ratio(
            r.name_n,
            s.name_n
        )

        addr_ratio = safe_ratio(
            r.addr_n,
            s.addr_n
        )

        name_partial = safe_partial(
            r.name_n,
            s.name_n
        )

        addr_partial = safe_partial(
            r.addr_n,
            s.addr_n
        )

        name_token = safe_token(
            r.name_n,
            s.name_n
        )

        addr_token = safe_token(
            r.addr_n,
            s.addr_n
        )

        n_overlap = number_overlap(
            r.addr_n,
            s.addr_n
        )

        exact_name_flag = int(
            r.name_n == s.name_n
        )

        exact_addr_flag = int(
            r.addr_n == s.addr_n
        )

        country_flag = int(
            r.country_n == s.country_n
        )

        combined_similarity = (
            0.45 * name_ratio
            +
            0.55 * addr_ratio
        )

        row = {
            "query_id": str(r.entity_id),
            "s1_id": str(s.entity_id),

            # Label
            "label": int(
                true_idx is not None
                and idx == true_idx
            ),

            # Similarity
            "name_ratio": name_ratio,
            "addr_ratio": addr_ratio,
            "name_partial": name_partial,
            "addr_partial": addr_partial,
            "name_token": name_token,
            "addr_token": addr_token,

            # Structural
            "name_length_ratio": length_ratio(
                r.name_n,
                s.name_n
            ),

            "addr_length_ratio": length_ratio(
                r.addr_n,
                s.addr_n
            ),

            "number_overlap": n_overlap,

            # Exact
            "exact_name": exact_name_flag,
            "exact_address": exact_addr_flag,
            "country_match": country_flag,

            # Combined
            "combined_similarity": combined_similarity,

            # Candidate information
            "channel_count": len(channels),

            "has_name_channel": int(
                any(
                    x.startswith("name")
                    for x in channels
                )
            ),

            "has_address_channel": int(
                any(
                    x.startswith("addr")
                    or x == "number_address"
                    for x in channels
                )
            ),

            "has_number_channel": int(
                "number_address" in channels
            ),

            "has_exact_channel": int(
                "exact_name" in channels
                or "exact_address" in channels
            ),
        }

        rows.append(row)

    if (q_no + 1) % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{q_no+1:,}/"
            f"{len(sample):,} | "
            f"speed="
            f"{(q_no+1)/elapsed:.1f} q/s | "
            f"rows="
            f"{len(rows):,}"
        )

# ------------------------------------------------------------
# DataFrame
# ------------------------------------------------------------

df_matcher = pd.DataFrame(rows)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 76 RESULTS")
print("=" * 100)

print(
    "Candidate rows:",
    f"{len(df_matcher):,}"
)

print(
    "Unique queries:",
    f"{df_matcher.query_id.nunique():,}"
)

print(
    "Positive pairs:",
    f"{df_matcher.label.sum():,}"
)

print(
    "Negative pairs:",
    f"{(df_matcher.label == 0).sum():,}"
)

print(
    "Positive rate:",
    f"{df_matcher.label.mean()*100:.4f}%"
)

print(
    "Average candidates/query:",
    f"{len(df_matcher)/df_matcher.query_id.nunique():.2f}"
)

print("\nChannel coverage:")

for col in [
    "has_name_channel",
    "has_address_channel",
    "has_number_channel",
    "has_exact_channel"
]:

    print(
        f"{col:22s}: "
        f"{df_matcher[col].mean()*100:.2f}%"
    )

print("\nFeature summary:")

print(
    df_matcher[
        [
            "name_ratio",
            "addr_ratio",
            "name_partial",
            "addr_partial",
            "name_token",
            "addr_token",
            "number_overlap",
            "combined_similarity"
        ]
    ].describe().round(3)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

out_path = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
    "MODELS/"
    "rapidfuzz_k10_matcher_2k.parquet"
)

df_matcher.to_parquet(
    out_path,
    index=False
)

print(
    "\nSaved:",
    out_path
)

print("\nCELL 76 COMPLETE.")

CELL 76 — BUILD K=10 MATCHER DATASET
500/2,000 | speed=151.4 q/s | rows=6,762
1,000/2,000 | speed=186.5 q/s | rows=13,750
1,500/2,000 | speed=227.0 q/s | rows=20,620
2,000/2,000 | speed=253.7 q/s | rows=27,557

CELL 76 RESULTS
Candidate rows: 27,557
Unique queries: 2,000
Positive pairs: 1,972
Negative pairs: 25,585
Positive rate: 7.1561%
Average candidates/query: 13.78

Channel coverage:
has_name_channel      : 34.83%
has_address_channel   : 70.26%
has_number_channel    : 39.95%
has_exact_channel     : 2.46%

Feature summary:
       name_ratio  addr_ratio  name_partial  addr_partial  name_token  \
count   27557.000   27557.000     27557.000     27557.000   27557.000   
mean        0.438       0.437         0.515         0.489       0.438   
std         0.226       0.161         0.228         0.163       0.226   
min         0.000       0.000         0.000         0.000       0.000   
25%         0.286       0.333         0.375         0.388       0.286   
50%         0.390       0.418 

In [ ]:
# CELL 77 — ENTITY-SPLIT LIGHTGBM MATCHER

import lightgbm as lgb
import numpy as np
import pandas as pd
import time

print("=" * 100)
print("CELL 77 — ENTITY-SPLIT LIGHTGBM MATCHER")
print("=" * 100)

# ------------------------------------------------------------
# Feature columns
# ------------------------------------------------------------

feature_cols = [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "name_token",
    "addr_token",
    "name_length_ratio",
    "addr_length_ratio",
    "number_overlap",
    "exact_name",
    "exact_address",
    "country_match",
    "combined_similarity",
    "channel_count",
    "has_name_channel",
    "has_address_channel",
    "has_number_channel",
    "has_exact_channel",
]

X = df_matcher[feature_cols].astype(np.float32)
y = df_matcher["label"].astype(np.int8)

# ------------------------------------------------------------
# Entity/query split
# ------------------------------------------------------------

query_ids = df_matcher["query_id"].unique()

rng = np.random.default_rng(42)

rng.shuffle(query_ids)

split = int(
    len(query_ids) * 0.75
)

train_queries = set(
    query_ids[:split]
)

valid_queries = set(
    query_ids[split:]
)

train_mask = (
    df_matcher["query_id"]
    .isin(train_queries)
)

valid_mask = (
    df_matcher["query_id"]
    .isin(valid_queries)
)

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_valid = X.loc[valid_mask]
y_valid = y.loc[valid_mask]

print(
    "Train queries:",
    len(train_queries)
)

print(
    "Validation queries:",
    len(valid_queries)
)

print(
    "Train rows:",
    len(X_train)
)

print(
    "Validation rows:",
    len(X_valid)
)

print(
    "Train positives:",
    int(y_train.sum())
)

print(
    "Validation positives:",
    int(y_valid.sum())
)

# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

model = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=2000,

    learning_rate=0.035,

    num_leaves=31,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.90,

    reg_alpha=0.10,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1
)

t0 = time.time()

model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_valid, y_valid)
    ],

    eval_metric=[
        "auc",
        "binary_logloss"
    ],

    callbacks=[
        lgb.early_stopping(
            100,
            verbose=False
        )
    ]
)

elapsed = time.time() - t0

print(
    "\nTraining time:",
    f"{elapsed:.2f} sec"
)

print(
    "Best iteration:",
    model.best_iteration_
)

# ------------------------------------------------------------
# Validation predictions
# ------------------------------------------------------------

valid_pred = model.predict_proba(
    X_valid,
    num_iteration=model.best_iteration_
)[:, 1]

print(
    "Validation prediction range:",
    f"{valid_pred.min():.5f} → "
    f"{valid_pred.max():.5f}"
)

# ------------------------------------------------------------
# Macro F0.5 helper
# ------------------------------------------------------------

def f05_score(precision, recall):

    beta2 = 0.25

    denominator = (
        beta2 * precision
        +
        recall
    )

    if denominator == 0:
        return 0.0

    return (
        (1 + beta2)
        *
        precision
        *
        recall
        /
        denominator
    )


def evaluate_entity_level(
    df,
    predictions,
    threshold
):

    temp = df.copy()

    temp["score"] = predictions

    temp["pred"] = (
        temp["score"] >= threshold
    )

    scores = []

    for query_id, group in temp.groupby(
        "query_id",
        sort=False
    ):

        true_ids = set(
            group.loc[
                group["label"] == 1,
                "s1_id"
            ]
        )

        pred_ids = set(
            group.loc[
                group["pred"],
                "s1_id"
            ]
        )

        tp = len(
            true_ids & pred_ids
        )

        fp = len(
            pred_ids - true_ids
        )

        fn = len(
            true_ids - pred_ids
        )

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else (
                1.0
                if len(true_ids) == 0
                else 0.0
            )
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 1.0
        )

        f05 = f05_score(
            precision,
            recall
        )

        scores.append(f05)

    return float(
        np.mean(scores)
    )


# ------------------------------------------------------------
# Threshold sweep
# ------------------------------------------------------------

thresholds = np.arange(
    0.05,
    0.951,
    0.005
)

best_threshold = None
best_f05 = -1

for threshold in thresholds:

    score = evaluate_entity_level(
        df_matcher.loc[valid_mask],
        valid_pred,
        threshold
    )

    if score > best_f05:

        best_f05 = score
        best_threshold = threshold

print("\n" + "=" * 100)
print("VALIDATION RESULT")
print("=" * 100)

print(
    "Best threshold:",
    f"{best_threshold:.3f}"
)

print(
    "Best entity-level Macro F0.5:",
    f"{best_f05:.6f}"
)

# ------------------------------------------------------------
# Additional operating points
# ------------------------------------------------------------

print("\nThreshold comparison:")

for threshold in [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:

    score = evaluate_entity_level(
        df_matcher.loc[valid_mask],
        valid_pred,
        threshold
    )

    print(
        f"threshold={threshold:.2f} "
        f"F0.5={score:.6f}"
    )

print("\nCELL 77 COMPLETE.")

CELL 77 — ENTITY-SPLIT LIGHTGBM MATCHER
Train queries: 1500
Validation queries: 500
Train rows: 20721
Validation rows: 6836
Train positives: 1477
Validation positives: 495
[LightGBM] [Info] Number of positive: 1477, number of negative: 19244
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.068684 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2293
[LightGBM] [Info] Number of data points in the train set: 20721, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.071280 -> initscore=-2.567186
[LightGBM] [Info] Start training from score -2.567186

Training time: 2.84 sec
Best iteration: 182
Validation prediction range: 0.00019 → 0.99765

VALIDATION RESULT
Best threshold: 0.120
Best entity-level Macro F0.5: 0.939209

Threshold comparison:
threshold=0.30 F0.5=0.929621
threshold=0.40 F0.5=0.932664
threshold=0.50 F0.

In [ ]:
# CELL 78 — LIGHTGBM ERROR ANALYSIS

print("=" * 100)
print("CELL 78 — LIGHTGBM ERROR ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# Validation dataframe
# ------------------------------------------------------------

df_val = df_matcher.loc[
    valid_mask
].copy()

df_val["score"] = valid_pred

BEST_T = best_threshold

df_val["pred"] = (
    df_val["score"] >= BEST_T
)

# ------------------------------------------------------------
# Per-query statistics
# ------------------------------------------------------------

query_stats = []

for query_id, group in df_val.groupby(
    "query_id",
    sort=False
):

    true_ids = set(
        group.loc[
            group["label"] == 1,
            "s1_id"
        ]
    )

    pred_ids = set(
        group.loc[
            group["pred"],
            "s1_id"
        ]
    )

    tp = len(
        true_ids & pred_ids
    )

    fp = len(
        pred_ids - true_ids
    )

    fn = len(
        true_ids - pred_ids
    )

    query_stats.append({
        "query_id": query_id,
        "true_count": len(true_ids),
        "pred_count": len(pred_ids),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "max_score": group["score"].max(),
        "positive_max_score": (
            group.loc[
                group["label"] == 1,
                "score"
            ].max()
            if group["label"].sum() > 0
            else 0
        ),
        "negative_max_score": (
            group.loc[
                group["label"] == 0,
                "score"
            ].max()
            if (group["label"] == 0).any()
            else 0
        )
    })

df_q = pd.DataFrame(
    query_stats
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("QUERY-LEVEL SUMMARY")
print("=" * 100)

print(
    "Validation queries:",
    len(df_q)
)

print(
    "Queries with zero predictions:",
    int((df_q.pred_count == 0).sum())
)

print(
    "Queries with exactly 1 prediction:",
    int((df_q.pred_count == 1).sum())
)

print(
    "Queries with 2+ predictions:",
    int((df_q.pred_count >= 2).sum())
)

print(
    "Queries with false positives:",
    int((df_q.fp > 0).sum())
)

print(
    "Queries with missed true matches:",
    int((df_q.fn > 0).sum())
)

print(
    "Queries perfectly matched:",
    int(
        (
            (df_q.fp == 0)
            &
            (df_q.fn == 0)
        ).sum()
    )
)

# ------------------------------------------------------------
# Pair-level statistics
# ------------------------------------------------------------

tp_total = int(
    df_q["tp"].sum()
)

fp_total = int(
    df_q["fp"].sum()
)

fn_total = int(
    df_q["fn"].sum()
)

print("\n" + "=" * 100)
print("PAIR-LEVEL SUMMARY")
print("=" * 100)

print(
    "TP:",
    tp_total
)

print(
    "FP:",
    fp_total
)

print(
    "FN:",
    fn_total
)

precision = (
    tp_total / (tp_total + fp_total)
    if tp_total + fp_total > 0
    else 0
)

recall = (
    tp_total / (tp_total + fn_total)
    if tp_total + fn_total > 0
    else 0
)

print(
    "Precision:",
    f"{precision:.4f}"
)

print(
    "Recall:",
    f"{recall:.4f}"
)

# ------------------------------------------------------------
# Score distributions
# ------------------------------------------------------------

positive_scores = df_val.loc[
    df_val["label"] == 1,
    "score"
]

negative_scores = df_val.loc[
    df_val["label"] == 0,
    "score"
]

print("\n" + "=" * 100)
print("SCORE DISTRIBUTIONS")
print("=" * 100)

print("\nPositive scores:")

print(
    positive_scores.describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    ).round(4)
)

print("\nNegative scores:")

print(
    negative_scores.describe(
        percentiles=[
            .90,
            .95,
            .97,
            .98,
            .99,
            .995,
            .999
        ]
    ).round(4)
)

# ------------------------------------------------------------
# False positive scores
# ------------------------------------------------------------

fp_rows = df_val.loc[
    (df_val["label"] == 0)
    &
    (df_val["pred"]),
    [
        "query_id",
        "s1_id",
        "score",
        "name_ratio",
        "addr_ratio",
        "name_partial",
        "addr_partial",
        "number_overlap",
        "combined_similarity",
        "channel_count",
        "has_number_channel"
    ]
].sort_values(
    "score",
    ascending=False
)

print("\n" + "=" * 100)
print("FALSE POSITIVES")
print("=" * 100)

print(
    "False-positive rows:",
    len(fp_rows)
)

if len(fp_rows):

    print(
        fp_rows.head(20).to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# False negatives
# ------------------------------------------------------------

fn_rows = df_val.loc[
    (df_val["label"] == 1)
    &
    (~df_val["pred"]),
    [
        "query_id",
        "s1_id",
        "score",
        "name_ratio",
        "addr_ratio",
        "name_partial",
        "addr_partial",
        "number_overlap",
        "combined_similarity",
        "channel_count",
        "has_number_channel"
    ]
].sort_values(
    "score",
    ascending=True
)

print("\n" + "=" * 100)
print("FALSE NEGATIVES")
print("=" * 100)

print(
    "False-negative rows:",
    len(fn_rows)
)

if len(fn_rows):

    print(
        fn_rows.head(20).to_string(
            index=False
        )
    )

print("\nCELL 78 COMPLETE.")

CELL 78 — LIGHTGBM ERROR ANALYSIS

QUERY-LEVEL SUMMARY
Validation queries: 500
Queries with zero predictions: 15
Queries with exactly 1 prediction: 454
Queries with 2+ predictions: 31
Queries with false positives: 33
Queries with missed true matches: 14
Queries perfectly matched: 456

PAIR-LEVEL SUMMARY
TP: 481
FP: 58
FN: 14
Precision: 0.8924
Recall: 0.9717

SCORE DISTRIBUTIONS

Positive scores:
count    495.0000
mean       0.9369
std        0.2016
min        0.0003
1%         0.0038
5%         0.4074
10%        0.9272
25%        0.9910
50%        0.9975
75%        0.9977
90%        0.9977
95%        0.9977
99%        0.9977
max        0.9977
Name: score, dtype: float64

Negative scores:
count    6341.0000
mean        0.0058
std         0.0539
min         0.0002
50%         0.0003
90%         0.0014
95%         0.0041
97%         0.0106
98%         0.0235
99%         0.1081
99.5%       0.3351
99.9%       0.9614
max         0.9778
Name: score, dtype: float64

FALSE POSITIVES
False-posit

In [ ]:
# CELL 79 — ADD MATCHING INTERACTION FEATURES

print("=" * 100)
print("CELL 79 — ADD DISCRIMINATIVE INTERACTION FEATURES")
print("=" * 100)

df_enhanced = df_matcher.copy()

# ------------------------------------------------------------
# Core interactions
# ------------------------------------------------------------

df_enhanced["name_addr_product"] = (
    df_enhanced["name_ratio"]
    *
    df_enhanced["addr_ratio"]
)

df_enhanced["name_partial_addr_partial_product"] = (
    df_enhanced["name_partial"]
    *
    df_enhanced["addr_partial"]
)

df_enhanced["name_number_product"] = (
    df_enhanced["name_ratio"]
    *
    df_enhanced["number_overlap"]
)

df_enhanced["addr_number_product"] = (
    df_enhanced["addr_ratio"]
    *
    df_enhanced["number_overlap"]
)

# ------------------------------------------------------------
# Agreement features
# ------------------------------------------------------------

df_enhanced["min_name_addr"] = np.minimum(
    df_enhanced["name_ratio"],
    df_enhanced["addr_ratio"]
)

df_enhanced["max_name_addr"] = np.maximum(
    df_enhanced["name_ratio"],
    df_enhanced["addr_ratio"]
)

df_enhanced["name_addr_gap"] = (
    np.abs(
        df_enhanced["name_ratio"]
        -
        df_enhanced["addr_ratio"]
    )
)

df_enhanced["partial_gap"] = (
    np.abs(
        df_enhanced["name_partial"]
        -
        df_enhanced["addr_partial"]
    )
)

# ------------------------------------------------------------
# Strong-name / weak-address indicators
# ------------------------------------------------------------

df_enhanced["strong_name_weak_address"] = (
    (
        df_enhanced["name_ratio"] >= 0.80
    )
    &
    (
        df_enhanced["addr_ratio"] < 0.20
    )
).astype(np.int8)

df_enhanced["strong_name_no_number"] = (
    (
        df_enhanced["name_ratio"] >= 0.80
    )
    &
    (
        df_enhanced["number_overlap"] == 0
    )
).astype(np.int8)

df_enhanced["strong_address"] = (
    df_enhanced["addr_ratio"] >= 0.70
).astype(np.int8)

df_enhanced["strong_name"] = (
    df_enhanced["name_ratio"] >= 0.80
).astype(np.int8)

# ------------------------------------------------------------
# Retrieval evidence
# ------------------------------------------------------------

df_enhanced["address_supported"] = (
    (
        df_enhanced["has_address_channel"]
        == 1
    )
    |
    (
        df_enhanced["has_number_channel"]
        == 1
    )
).astype(np.int8)

df_enhanced["multi_channel"] = (
    df_enhanced["channel_count"] >= 2
).astype(np.int8)

# ------------------------------------------------------------
# Combined scores
# ------------------------------------------------------------

df_enhanced["balanced_similarity"] = (
    0.5 * df_enhanced["name_ratio"]
    +
    0.5 * df_enhanced["addr_ratio"]
)

df_enhanced["partial_balanced_similarity"] = (
    0.5 * df_enhanced["name_partial"]
    +
    0.5 * df_enhanced["addr_partial"]
)

# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

new_features = [
    "name_addr_product",
    "name_partial_addr_partial_product",
    "name_number_product",
    "addr_number_product",
    "min_name_addr",
    "max_name_addr",
    "name_addr_gap",
    "partial_gap",
    "strong_name_weak_address",
    "strong_name_no_number",
    "strong_address",
    "strong_name",
    "address_supported",
    "multi_channel",
    "balanced_similarity",
    "partial_balanced_similarity",
]

print(
    "Original features:",
    18
)

print(
    "New features:",
    len(new_features)
)

print(
    "Total features:",
    18 + len(new_features)
)

print("\nNew feature means:")

print(
    df_enhanced[new_features]
    .mean()
    .round(4)
)

# ------------------------------------------------------------
# Save enhanced dataset
# ------------------------------------------------------------

enhanced_path = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
    "MODELS/"
    "rapidfuzz_k10_matcher_enhanced_2k.parquet"
)

df_enhanced.to_parquet(
    enhanced_path,
    index=False
)

print(
    "\nSaved:",
    enhanced_path
)

print("\nCELL 79 COMPLETE.")

CELL 79 — ADD DISCRIMINATIVE INTERACTION FEATURES
Original features: 18
New features: 16
Total features: 34

New feature means:
name_addr_product                    0.1903
name_partial_addr_partial_product    0.2516
name_number_product                  0.1212
addr_number_product                  0.1452
min_name_addr                        0.3226
max_name_addr                        0.5531
name_addr_gap                        0.2305
partial_gap                          0.2180
strong_name_weak_address             0.0044
strong_name_no_number                0.0427
strong_address                       0.0621
strong_name                          0.0780
address_supported                    0.7026
multi_channel                        0.1198
balanced_similarity                  0.4378
partial_balanced_similarity          0.5021
dtype: float64

Saved: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/rapidfuzz_k10_matcher_enhanced_2k.parquet

CELL 79 COMPLETE.


In [ ]:
# CELL 80 — ENHANCED LIGHTGBM MATCHER

import lightgbm as lgb
import numpy as np
import pandas as pd
import time

print("=" * 100)
print("CELL 80 — ENHANCED LIGHTGBM MATCHER")
print("=" * 100)

# ------------------------------------------------------------
# Feature set
# ------------------------------------------------------------

base_features = [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "name_token",
    "addr_token",
    "name_length_ratio",
    "addr_length_ratio",
    "number_overlap",
    "exact_name",
    "exact_address",
    "country_match",
    "combined_similarity",
    "channel_count",
    "has_name_channel",
    "has_address_channel",
    "has_number_channel",
    "has_exact_channel",
]

interaction_features = [
    "name_addr_product",
    "name_partial_addr_partial_product",
    "name_number_product",
    "addr_number_product",
    "min_name_addr",
    "max_name_addr",
    "name_addr_gap",
    "partial_gap",
    "strong_name_weak_address",
    "strong_name_no_number",
    "strong_address",
    "strong_name",
    "address_supported",
    "multi_channel",
    "balanced_similarity",
    "partial_balanced_similarity",
]

feature_cols_enhanced = (
    base_features
    +
    interaction_features
)

print(
    "Features:",
    len(feature_cols_enhanced)
)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

X = df_enhanced[
    feature_cols_enhanced
].astype(np.float32)

y = df_enhanced[
    "label"
].astype(np.int8)

# ------------------------------------------------------------
# SAME query-level split as Cell 77
# ------------------------------------------------------------

query_ids = df_enhanced[
    "query_id"
].unique()

rng = np.random.default_rng(42)

rng.shuffle(query_ids)

split = int(
    len(query_ids) * 0.75
)

train_queries = set(
    query_ids[:split]
)

valid_queries = set(
    query_ids[split:]
)

train_mask = (
    df_enhanced["query_id"]
    .isin(train_queries)
)

valid_mask = (
    df_enhanced["query_id"]
    .isin(valid_queries)
)

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_valid = X.loc[valid_mask]
y_valid = y.loc[valid_mask]

print(
    "Train queries:",
    len(train_queries)
)

print(
    "Validation queries:",
    len(valid_queries)
)

print(
    "Train rows:",
    len(X_train)
)

print(
    "Validation rows:",
    len(X_valid)
)

print(
    "Train positives:",
    int(y_train.sum())
)

print(
    "Validation positives:",
    int(y_valid.sum())
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

model_enhanced = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=2500,

    learning_rate=0.03,

    num_leaves=31,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.90,

    reg_alpha=0.10,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1
)

t0 = time.time()

model_enhanced.fit(
    X_train,
    y_train,

    eval_set=[
        (X_valid, y_valid)
    ],

    eval_metric=[
        "auc",
        "binary_logloss"
    ],

    callbacks=[
        lgb.early_stopping(
            120,
            verbose=False
        )
    ]
)

elapsed = time.time() - t0

print(
    "\nTraining time:",
    f"{elapsed:.2f} sec"
)

print(
    "Best iteration:",
    model_enhanced.best_iteration_
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

valid_pred_enhanced = model_enhanced.predict_proba(
    X_valid,
    num_iteration=model_enhanced.best_iteration_
)[:, 1]

print(
    "Prediction range:",
    f"{valid_pred_enhanced.min():.6f}"
    " → "
    f"{valid_pred_enhanced.max():.6f}"
)

# ------------------------------------------------------------
# Entity-level Macro F0.5
# ------------------------------------------------------------

def f05_score_v2(
    precision,
    recall
):

    beta2 = 0.25

    denominator = (
        beta2 * precision
        +
        recall
    )

    if denominator <= 0:
        return 0.0

    return (
        1.25
        *
        precision
        *
        recall
        /
        denominator
    )


def evaluate_entity_level_v2(
    df,
    predictions,
    threshold
):

    temp = df.copy()

    temp["score"] = predictions

    temp["pred"] = (
        temp["score"] >= threshold
    )

    scores = []

    for query_id, group in temp.groupby(
        "query_id",
        sort=False
    ):

        true_ids = set(
            group.loc[
                group["label"] == 1,
                "s1_id"
            ]
        )

        pred_ids = set(
            group.loc[
                group["pred"],
                "s1_id"
            ]
        )

        tp = len(
            true_ids & pred_ids
        )

        fp = len(
            pred_ids - true_ids
        )

        fn = len(
            true_ids - pred_ids
        )

        if tp + fp > 0:
            precision = (
                tp /
                (tp + fp)
            )
        else:
            precision = (
                1.0
                if len(true_ids) == 0
                else 0.0
            )

        if tp + fn > 0:
            recall = (
                tp /
                (tp + fn)
            )
        else:
            recall = 1.0

        scores.append(
            f05_score_v2(
                precision,
                recall
            )
        )

    return float(
        np.mean(scores)
    )


# ------------------------------------------------------------
# Threshold sweep
# ------------------------------------------------------------

thresholds = np.arange(
    0.01,
    0.951,
    0.005
)

results = []

for threshold in thresholds:

    score = evaluate_entity_level_v2(
        df_enhanced.loc[valid_mask],
        valid_pred_enhanced,
        threshold
    )

    results.append({
        "threshold": threshold,
        "f05": score
    })

results_df = pd.DataFrame(results)

best_row = results_df.loc[
    results_df["f05"].idxmax()
]

best_threshold_enhanced = float(
    best_row["threshold"]
)

best_f05_enhanced = float(
    best_row["f05"]
)

print("\n" + "=" * 100)
print("ENHANCED MODEL RESULT")
print("=" * 100)

print(
    "Best threshold:",
    f"{best_threshold_enhanced:.3f}"
)

print(
    "Best Macro F0.5:",
    f"{best_f05_enhanced:.6f}"
)

print(
    "Baseline Macro F0.5:",
    "0.939209"
)

print(
    "Improvement:",
    f"{best_f05_enhanced - 0.939209:+.6f}"
)

# ------------------------------------------------------------
# Threshold table
# ------------------------------------------------------------

print("\nThreshold comparison:")

for threshold in [
    0.05,
    0.10,
    0.15,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:

    score = evaluate_entity_level_v2(
        df_enhanced.loc[valid_mask],
        valid_pred_enhanced,
        threshold
    )

    print(
        f"threshold={threshold:.2f} "
        f"F0.5={score:.6f}"
    )

print("\nCELL 80 COMPLETE.")

CELL 80 — ENHANCED LIGHTGBM MATCHER
Features: 34
Train queries: 1500
Validation queries: 500
Train rows: 20721
Validation rows: 6836
Train positives: 1477
Validation positives: 495
[LightGBM] [Info] Number of positive: 1477, number of negative: 19244
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028217 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4854
[LightGBM] [Info] Number of data points in the train set: 20721, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.071280 -> initscore=-2.567186
[LightGBM] [Info] Start training from score -2.567186
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [ ]:
# CELL 81 — PRECISION-AWARE POST-RANKING

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 81 — PRECISION-AWARE POST-RANKING")
print("=" * 100)

# ------------------------------------------------------------
# Validation dataframe
# ------------------------------------------------------------

df_rule = df_enhanced.loc[
    valid_mask
].copy()

df_rule["score"] = valid_pred_enhanced

# ------------------------------------------------------------
# F0.5 helper
# ------------------------------------------------------------

def calc_f05(
    precision,
    recall
):

    if precision == 0 or recall == 0:
        return 0.0

    return (
        1.25
        * precision
        * recall
        /
        (
            0.25 * precision
            +
            recall
        )
    )


def evaluate_rule(
    df,
    rule_name
):

    query_scores = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for query_id, group in df.groupby(
        "query_id",
        sort=False
    ):

        true_ids = set(
            group.loc[
                group["label"] == 1,
                "s1_id"
            ]
        )

        pred_ids = set(
            group.loc[
                group["final_pred"],
                "s1_id"
            ]
        )

        tp = len(
            true_ids & pred_ids
        )

        fp = len(
            pred_ids - true_ids
        )

        fn = len(
            true_ids - pred_ids
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else (
                1.0
                if len(true_ids) == 0
                else 0.0
            )
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 1.0
        )

        query_scores.append(
            calc_f05(
                precision,
                recall
            )
        )

    macro_f05 = np.mean(
        query_scores
    )

    pair_precision = (
        total_tp /
        (total_tp + total_fp)
        if total_tp + total_fp > 0
        else 0
    )

    pair_recall = (
        total_tp /
        (total_tp + total_fn)
        if total_tp + total_fn > 0
        else 0
    )

    return {
        "rule": rule_name,
        "macro_f05": macro_f05,
        "precision": pair_precision,
        "recall": pair_recall,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn
    }


# ------------------------------------------------------------
# Base threshold
# ------------------------------------------------------------

threshold = 0.260

# ------------------------------------------------------------
# Rule 1 — baseline
# ------------------------------------------------------------

df_rule["final_pred"] = (
    df_rule["score"] >= threshold
)

results = []

results.append(
    evaluate_rule(
        df_rule,
        "Baseline 0.260"
    )
)

# ------------------------------------------------------------
# Rule 2 — penalize strong-name / zero-address
# ------------------------------------------------------------

df_rule["final_pred"] = (
    df_rule["score"] >= threshold
)

bad_name_only = (
    (df_rule["name_ratio"] >= 0.80)
    &
    (df_rule["addr_ratio"] < 0.15)
    &
    (df_rule["number_overlap"] == 0)
)

df_rule.loc[
    bad_name_only,
    "final_pred"
] = (
    df_rule.loc[
        bad_name_only,
        "score"
    ] >= 0.75
)

results.append(
    evaluate_rule(
        df_rule,
        "Strong-name/no-address penalty"
    )
)

# ------------------------------------------------------------
# Rule 3 — require address support for medium scores
# ------------------------------------------------------------

df_rule["final_pred"] = (
    df_rule["score"] >= threshold
)

medium_no_address = (
    (df_rule["score"] < 0.75)
    &
    (df_rule["addr_ratio"] < 0.20)
    &
    (df_rule["number_overlap"] == 0)
)

df_rule.loc[
    medium_no_address,
    "final_pred"
] = False

results.append(
    evaluate_rule(
        df_rule,
        "Medium-score address requirement"
    )
)

# ------------------------------------------------------------
# Rule 4 — protect strong address + number matches
# ------------------------------------------------------------

df_rule["final_pred"] = (
    df_rule["score"] >= threshold
)

strong_address_evidence = (
    (df_rule["addr_ratio"] >= 0.65)
    &
    (df_rule["number_overlap"] >= 0.5)
)

df_rule.loc[
    strong_address_evidence,
    "final_pred"
] = True

results.append(
    evaluate_rule(
        df_rule,
        "Strong address+number recovery"
    )
)

# ------------------------------------------------------------
# Rule 5 — combined rule
# ------------------------------------------------------------

df_rule["final_pred"] = (
    df_rule["score"] >= threshold
)

# Remove suspicious name-only matches
remove_name_only = (
    (df_rule["score"] < 0.75)
    &
    (df_rule["name_ratio"] >= 0.80)
    &
    (df_rule["addr_ratio"] < 0.20)
    &
    (df_rule["number_overlap"] == 0)
)

df_rule.loc[
    remove_name_only,
    "final_pred"
] = False

# Remove weak-address candidates unless score is very strong
weak_address = (
    (df_rule["score"] < 0.50)
    &
    (df_rule["addr_ratio"] < 0.30)
    &
    (df_rule["number_overlap"] == 0)
)

df_rule.loc[
    weak_address,
    "final_pred"
] = False

# Restore strong address+number evidence
restore_address = (
    (df_rule["addr_ratio"] >= 0.65)
    &
    (df_rule["number_overlap"] >= 0.5)
    &
    (df_rule["score"] >= 0.05)
)

df_rule.loc[
    restore_address,
    "final_pred"
] = True

results.append(
    evaluate_rule(
        df_rule,
        "Combined precision/recall rule"
    )
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

results_df = pd.DataFrame(
    results
).sort_values(
    "macro_f05",
    ascending=False
)

print("\n" + "=" * 100)
print("CELL 81 RESULTS")
print("=" * 100)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nCELL 81 COMPLETE.")

CELL 81 — PRECISION-AWARE POST-RANKING

CELL 81 RESULTS
                            rule  macro_f05  precision   recall  tp  fp  fn
                  Baseline 0.260   0.944437   0.924565 0.965657 478  39  17
  Strong address+number recovery   0.944314   0.912381 0.967677 479  46  16
  Strong-name/no-address penalty   0.940779   0.926070 0.961616 476  38  19
Medium-score address requirement   0.938779   0.927734 0.959596 475  37  20
  Combined precision/recall rule   0.936256   0.922179 0.957576 474  40  21

CELL 81 COMPLETE.


In [ ]:
# CELL 82 — QUERY-RELATIVE RANKING FEATURES

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 82 — QUERY-RELATIVE RANKING FEATURES")
print("=" * 100)

df_rank = df_enhanced.copy()

# ------------------------------------------------------------
# Sort candidates within each query
# ------------------------------------------------------------

# Name ranking
df_rank["name_rank"] = (
    df_rank.groupby("query_id")["name_ratio"]
    .rank(method="min", ascending=False)
)

# Address ranking
df_rank["addr_rank"] = (
    df_rank.groupby("query_id")["addr_ratio"]
    .rank(method="min", ascending=False)
)

# Partial-name ranking
df_rank["name_partial_rank"] = (
    df_rank.groupby("query_id")["name_partial"]
    .rank(method="min", ascending=False)
)

# Partial-address ranking
df_rank["addr_partial_rank"] = (
    df_rank.groupby("query_id")["addr_partial"]
    .rank(method="min", ascending=False)
)

# Combined similarity ranking
df_rank["combined_rank"] = (
    df_rank.groupby("query_id")["combined_similarity"]
    .rank(method="min", ascending=False)
)

# ------------------------------------------------------------
# Query-level maxima
# ------------------------------------------------------------

for col in [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "combined_similarity",
    "number_overlap"
]:

    max_col = f"query_max_{col}"

    df_rank[max_col] = (
        df_rank.groupby("query_id")[col]
        .transform("max")
    )

# ------------------------------------------------------------
# Margins from query best
# ------------------------------------------------------------

df_rank["name_margin"] = (
    df_rank["query_max_name_ratio"]
    -
    df_rank["name_ratio"]
)

df_rank["addr_margin"] = (
    df_rank["query_max_addr_ratio"]
    -
    df_rank["addr_ratio"]
)

df_rank["combined_margin"] = (
    df_rank["query_max_combined_similarity"]
    -
    df_rank["combined_similarity"]
)

# ------------------------------------------------------------
# Count stronger candidates
# ------------------------------------------------------------

df_rank["name_stronger_count"] = (
    df_rank.groupby("query_id")["name_ratio"]
    .transform(
        lambda x:
        (x.values[:, None] < x.values).sum(axis=1)
    )
)

df_rank["addr_stronger_count"] = (
    df_rank.groupby("query_id")["addr_ratio"]
    .transform(
        lambda x:
        (x.values[:, None] < x.values).sum(axis=1)
    )
)

# ------------------------------------------------------------
# Number of candidates above similarity thresholds
# ------------------------------------------------------------

df_rank["name_ge_80_count"] = (
    df_rank.groupby("query_id")["name_ratio"]
    .transform(
        lambda x: (x >= 0.80).sum()
    )
)

df_rank["addr_ge_60_count"] = (
    df_rank.groupby("query_id")["addr_ratio"]
    .transform(
        lambda x: (x >= 0.60).sum()
    )
)

df_rank["combined_ge_60_count"] = (
    df_rank.groupby("query_id")["combined_similarity"]
    .transform(
        lambda x: (x >= 0.60).sum()
    )
)

# ------------------------------------------------------------
# Best-candidate indicators
# ------------------------------------------------------------

df_rank["is_best_name"] = (
    df_rank["name_rank"] == 1
).astype(np.int8)

df_rank["is_best_addr"] = (
    df_rank["addr_rank"] == 1
).astype(np.int8)

df_rank["is_best_combined"] = (
    df_rank["combined_rank"] == 1
).astype(np.int8)

# ------------------------------------------------------------
# Agreement between rankings
# ------------------------------------------------------------

df_rank["name_addr_rank_agree"] = (
    (
        df_rank["name_rank"] == 1
    )
    &
    (
        df_rank["addr_rank"] == 1
    )
).astype(np.int8)

df_rank["name_combined_rank_agree"] = (
    (
        df_rank["name_rank"] == 1
    )
    &
    (
        df_rank["combined_rank"] == 1
    )
).astype(np.int8)

# ------------------------------------------------------------
# Candidate relative strength
# ------------------------------------------------------------

df_rank["relative_name_strength"] = (
    df_rank["name_ratio"]
    /
    (
        df_rank["query_max_name_ratio"]
        + 1e-6
    )
)

df_rank["relative_addr_strength"] = (
    df_rank["addr_ratio"]
    /
    (
        df_rank["query_max_addr_ratio"]
        + 1e-6
    )
)

df_rank["relative_combined_strength"] = (
    df_rank["combined_similarity"]
    /
    (
        df_rank["query_max_combined_similarity"]
        + 1e-6
    )
)

# ------------------------------------------------------------
# Feature list
# ------------------------------------------------------------

ranking_features = [
    "name_rank",
    "addr_rank",
    "name_partial_rank",
    "addr_partial_rank",
    "combined_rank",

    "query_max_name_ratio",
    "query_max_addr_ratio",
    "query_max_name_partial",
    "query_max_addr_partial",
    "query_max_combined_similarity",
    "query_max_number_overlap",

    "name_margin",
    "addr_margin",
    "combined_margin",

    "name_stronger_count",
    "addr_stronger_count",

    "name_ge_80_count",
    "addr_ge_60_count",
    "combined_ge_60_count",

    "is_best_name",
    "is_best_addr",
    "is_best_combined",

    "name_addr_rank_agree",
    "name_combined_rank_agree",

    "relative_name_strength",
    "relative_addr_strength",
    "relative_combined_strength",
]

print(
    "Ranking features:",
    len(ranking_features)
)

print("\nFeature preview:")

print(
    df_rank[
        ranking_features
    ].head().round(4).to_string()
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

rank_path = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
    "MODELS/"
    "rapidfuzz_k10_matcher_ranked_2k.parquet"
)

df_rank.to_parquet(
    rank_path,
    index=False
)

print(
    "\nSaved:",
    rank_path
)

print("\nCELL 82 COMPLETE.")

CELL 82 — QUERY-RELATIVE RANKING FEATURES
Ranking features: 27

Feature preview:
   name_rank  addr_rank  name_partial_rank  addr_partial_rank  combined_rank  query_max_name_ratio  query_max_addr_ratio  query_max_name_partial  query_max_addr_partial  query_max_combined_similarity  query_max_number_overlap  name_margin  addr_margin  combined_margin  name_stronger_count  addr_stronger_count  name_ge_80_count  addr_ge_60_count  combined_ge_60_count  is_best_name  is_best_addr  is_best_combined  name_addr_rank_agree  name_combined_rank_agree  relative_name_strength  relative_addr_strength  relative_combined_strength
0        1.0        4.0                1.0                4.0            1.0                0.8966                   0.6                     1.0                     0.7                         0.6838                       1.0       0.0000       0.0902           0.0000                    0                    3                 1                 1                     1            

In [ ]:
# CELL 83 — RANKING-AWARE LIGHTGBM

import lightgbm as lgb
import numpy as np
import pandas as pd
import time

print("=" * 100)
print("CELL 83 — RANKING-AWARE LIGHTGBM")
print("=" * 100)

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

base_features = [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "name_token",
    "addr_token",
    "name_length_ratio",
    "addr_length_ratio",
    "number_overlap",
    "exact_name",
    "exact_address",
    "country_match",
    "combined_similarity",
    "channel_count",
    "has_name_channel",
    "has_address_channel",
    "has_number_channel",
    "has_exact_channel",
]

interaction_features = [
    "name_addr_product",
    "name_partial_addr_partial_product",
    "name_number_product",
    "addr_number_product",
    "min_name_addr",
    "max_name_addr",
    "name_addr_gap",
    "partial_gap",
    "strong_name_weak_address",
    "strong_name_no_number",
    "strong_address",
    "strong_name",
    "address_supported",
    "multi_channel",
    "balanced_similarity",
    "partial_balanced_similarity",
]

ranking_features = [
    "name_rank",
    "addr_rank",
    "name_partial_rank",
    "addr_partial_rank",
    "combined_rank",
    "query_max_name_ratio",
    "query_max_addr_ratio",
    "query_max_name_partial",
    "query_max_addr_partial",
    "query_max_combined_similarity",
    "query_max_number_overlap",
    "name_margin",
    "addr_margin",
    "combined_margin",
    "name_stronger_count",
    "addr_stronger_count",
    "name_ge_80_count",
    "addr_ge_60_count",
    "combined_ge_60_count",
    "is_best_name",
    "is_best_addr",
    "is_best_combined",
    "name_addr_rank_agree",
    "name_combined_rank_agree",
    "relative_name_strength",
    "relative_addr_strength",
    "relative_combined_strength",
]

feature_cols_ranked = (
    base_features
    + interaction_features
    + ranking_features
)

print(
    "Total features:",
    len(feature_cols_ranked)
)

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

X = df_rank[
    feature_cols_ranked
].astype(np.float32)

y = df_rank[
    "label"
].astype(np.int8)

# ------------------------------------------------------------
# SAME query split as Cells 77/80
# ------------------------------------------------------------

query_ids = df_rank[
    "query_id"
].unique()

rng = np.random.default_rng(42)
rng.shuffle(query_ids)

split = int(
    len(query_ids) * 0.75
)

train_queries = set(
    query_ids[:split]
)

valid_queries = set(
    query_ids[split:]
)

train_mask = (
    df_rank["query_id"]
    .isin(train_queries)
)

valid_mask = (
    df_rank["query_id"]
    .isin(valid_queries)
)

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_valid = X.loc[valid_mask]
y_valid = y.loc[valid_mask]

print(
    "Train queries:",
    len(train_queries)
)

print(
    "Validation queries:",
    len(valid_queries)
)

print(
    "Train rows:",
    len(X_train)
)

print(
    "Validation rows:",
    len(X_valid)
)

print(
    "Train positives:",
    int(y_train.sum())
)

print(
    "Validation positives:",
    int(y_valid.sum())
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model_ranked = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=3000,

    learning_rate=0.025,

    num_leaves=31,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.90,

    reg_alpha=0.15,

    reg_lambda=1.5,

    random_state=42,

    n_jobs=-1
)

t0 = time.time()

model_ranked.fit(
    X_train,
    y_train,

    eval_set=[
        (X_valid, y_valid)
    ],

    eval_metric=[
        "auc",
        "binary_logloss"
    ],

    callbacks=[
        lgb.early_stopping(
            150,
            verbose=False
        )
    ]
)

elapsed = time.time() - t0

print(
    "\nTraining time:",
    f"{elapsed:.2f} sec"
)

print(
    "Best iteration:",
    model_ranked.best_iteration_
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

valid_pred_ranked = model_ranked.predict_proba(
    X_valid,
    num_iteration=model_ranked.best_iteration_
)[:, 1]

print(
    "Prediction range:",
    f"{valid_pred_ranked.min():.6f}"
    " → "
    f"{valid_pred_ranked.max():.6f}"
)

# ------------------------------------------------------------
# F0.5
# ------------------------------------------------------------

def calc_f05_rank(
    precision,
    recall
):

    if precision <= 0 or recall <= 0:
        return 0.0

    return (
        1.25
        * precision
        * recall
        /
        (
            0.25 * precision
            +
            recall
        )
    )


def evaluate_ranked(
    df,
    predictions,
    threshold
):

    temp = df.copy()

    temp["score"] = predictions

    temp["pred"] = (
        temp["score"] >= threshold
    )

    scores = []

    for query_id, group in temp.groupby(
        "query_id",
        sort=False
    ):

        true_ids = set(
            group.loc[
                group["label"] == 1,
                "s1_id"
            ]
        )

        pred_ids = set(
            group.loc[
                group["pred"],
                "s1_id"
            ]
        )

        tp = len(
            true_ids & pred_ids
        )

        fp = len(
            pred_ids - true_ids
        )

        fn = len(
            true_ids - pred_ids
        )

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else (
                1.0
                if len(true_ids) == 0
                else 0.0
            )
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 1.0
        )

        scores.append(
            calc_f05_rank(
                precision,
                recall
            )
        )

    return float(
        np.mean(scores)
    )


# ------------------------------------------------------------
# Threshold search
# ------------------------------------------------------------

thresholds = np.arange(
    0.01,
    0.951,
    0.005
)

results = []

for threshold in thresholds:

    score = evaluate_ranked(
        df_rank.loc[valid_mask],
        valid_pred_ranked,
        threshold
    )

    results.append({
        "threshold": threshold,
        "f05": score
    })

results_df = pd.DataFrame(
    results
)

best_row = results_df.loc[
    results_df["f05"].idxmax()
]

best_threshold_ranked = float(
    best_row["threshold"]
)

best_f05_ranked = float(
    best_row["f05"]
)

print("\n" + "=" * 100)
print("RANKING-AWARE RESULT")
print("=" * 100)

print(
    "Best threshold:",
    f"{best_threshold_ranked:.3f}"
)

print(
    "Best Macro F0.5:",
    f"{best_f05_ranked:.6f}"
)

print(
    "Previous enhanced:",
    "0.944437"
)

print(
    "Improvement:",
    f"{best_f05_ranked - 0.944437:+.6f}"
)

print("\nThreshold comparison:")

for threshold in [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:

    score = evaluate_ranked(
        df_rank.loc[valid_mask],
        valid_pred_ranked,
        threshold
    )

    print(
        f"threshold={threshold:.2f} "
        f"F0.5={score:.6f}"
    )

print("\nCELL 83 COMPLETE.")

CELL 83 — RANKING-AWARE LIGHTGBM
Total features: 61
Train queries: 1500
Validation queries: 500
Train rows: 20721
Validation rows: 6836
Train positives: 1477
Validation positives: 495
[LightGBM] [Info] Number of positive: 1477, number of negative: 19244
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009085 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7868
[LightGBM] [Info] Number of data points in the train set: 20721, number of used features: 60
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.071280 -> initscore=-2.567186
[LightGBM] [Info] Start training from score -2.567186
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

In [ ]:
# CELL 84 — SAVE BEST RANKING-AWARE MODEL

import os

MODEL_DIR = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
    "MODELS"
)

os.makedirs(MODEL_DIR, exist_ok=True)

best_model_path = os.path.join(
    MODEL_DIR,
    "rapidfuzz_k10_ranking_lgbm_best.txt"
)

best_threshold_path = os.path.join(
    MODEL_DIR,
    "rapidfuzz_k10_ranking_threshold.txt"
)

# Save LightGBM model
model_ranked.booster_.save_model(
    best_model_path
)

# Save threshold
with open(
    best_threshold_path,
    "w"
) as f:
    f.write(
        str(best_threshold_ranked)
    )

print("=" * 100)
print("CELL 84 — BEST MODEL SAVED")
print("=" * 100)

print(
    "Model:",
    best_model_path
)

print(
    "Threshold:",
    best_threshold_path
)

print(
    "Threshold:",
    best_threshold_ranked
)

print(
    "Validation Macro F0.5:",
    best_f05_ranked
)

print(
    "\nCELL 84 COMPLETE."
)

CELL 84 — BEST MODEL SAVED
Model: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/rapidfuzz_k10_ranking_lgbm_best.txt
Threshold: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/rapidfuzz_k10_ranking_threshold.txt
Threshold: 0.22499999999999998
Validation Macro F0.5: 0.9732790346907995

CELL 84 COMPLETE.


In [ ]:
# CELL 85 — REPRESENTATIVE S1-LEVEL VALIDATION SET

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 85 — REPRESENTATIVE S1-LEVEL VALIDATION")
print("=" * 100)

# ------------------------------------------------------------
# Load the earlier 20k-S1 validation data if available
# ------------------------------------------------------------

candidate_names = [
    "validation_20k.parquet",
    "validation_s1_20k.parquet",
    "train_validation_20k.parquet",
]

found = None

for name in candidate_names:
    path = (
        "/content/drive/MyDrive/"
        "AMAZON ML 2026/AMAZON ML 2026/"
        "MODELS/" + name
    )

    if os.path.exists(path):
        found = path
        break

if found is not None:

    print("Found validation artifact:")
    print(found)

    val_s1 = pd.read_parquet(found)

    print(
        "Rows:",
        len(val_s1)
    )

else:

    print(
        "No saved validation artifact found."
    )

    print(
        "Checking notebook variables..."
    )

    # Look for likely existing validation dataframe
    possible_vars = [
        "val_s1",
        "validation_s1",
        "df_validation",
        "validation_df",
        "val_df"
    ]

    found_var = None

    for var_name in possible_vars:

        if var_name in globals():

            obj = globals()[var_name]

            if isinstance(
                obj,
                pd.DataFrame
            ):

                found_var = var_name
                val_s1 = obj
                break

    if found_var is not None:

        print(
            "Using existing dataframe:",
            found_var
        )

    else:

        raise RuntimeError(
            "Representative 20k S1 validation dataframe "
            "was not found. Do NOT continue to production yet."
        )

# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

print("\nColumns:")
print(
    list(val_s1.columns)
)

print("\nShape:")
print(
    val_s1.shape
)

print("\nSample:")
print(
    val_s1.head().to_string()
)

# ------------------------------------------------------------
# Identify S1 ID column
# ------------------------------------------------------------

possible_s1_cols = [
    "source1_entity_id",
    "entity_id",
    "s1_id",
    "source1_id"
]

s1_col = None

for col in possible_s1_cols:

    if col in val_s1.columns:

        s1_col = col
        break

if s1_col is None:

    raise RuntimeError(
        "Could not identify S1 entity ID column."
    )

print(
    "\nS1 ID column:",
    s1_col
)

# ------------------------------------------------------------
# Ground-truth match column
# ------------------------------------------------------------

possible_gt_cols = [
    "matched_entity_ids",
    "matched_ids",
    "target_ids"
]

gt_col = None

for col in possible_gt_cols:

    if col in val_s1.columns:

        gt_col = col
        break

if gt_col is None:

    raise RuntimeError(
        "Could not identify ground-truth match column."
    )

print(
    "Ground-truth column:",
    gt_col
)

# ------------------------------------------------------------
# Normalize GT into sets
# ------------------------------------------------------------

def parse_gt(value):

    if pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    return set(
        x.strip()
        for x in value.split(",")
        if x.strip()
    )

val_s1_eval = val_s1[
    [s1_col, gt_col]
].copy()

val_s1_eval["true_ids"] = (
    val_s1_eval[gt_col]
    .apply(parse_gt)
)

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

val_s1_eval["true_count"] = (
    val_s1_eval["true_ids"]
    .str.len()
)

print("\nGround-truth distribution:")

print(
    val_s1_eval["true_count"]
    .value_counts()
    .sort_index()
)

print(
    "\nTotal S1 entities:",
    len(val_s1_eval)
)

print(
    "No-match S1:",
    int(
        (val_s1_eval["true_count"] == 0).sum()
    )
)

print(
    "Singleton S1:",
    int(
        (val_s1_eval["true_count"] == 1).sum()
    )
)

print(
    "Multi-match S1:",
    int(
        (val_s1_eval["true_count"] > 1).sum()
    )
)

print(
    "\nCELL 85 COMPLETE."
)

CELL 85 — REPRESENTATIVE S1-LEVEL VALIDATION
No saved validation artifact found.
Checking notebook variables...


RuntimeError: Representative 20k S1 validation dataframe was not found. Do NOT continue to production yet.

In [ ]:
# CELL 86 — REBUILD REPRESENTATIVE 20K S1 VALIDATION

import os
import pandas as pd
import numpy as np

print("=" * 100)
print("CELL 86 — REBUILD REPRESENTATIVE 20K S1 VALIDATION")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

# ------------------------------------------------------------
# Locate files
# ------------------------------------------------------------

print("TRAIN_DIR:")
print(TRAIN_DIR)

print("\nFiles:")

for f in os.listdir(TRAIN_DIR):
    print(" ", f)

# ------------------------------------------------------------
# Load S1 and ground truth
# ------------------------------------------------------------

s1_path = os.path.join(
    TRAIN_DIR,
    "train_source1.tsv"
)

gt_path = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

if not os.path.exists(s1_path):
    raise FileNotFoundError(
        f"S1 file not found: {s1_path}"
    )

if not os.path.exists(gt_path):
    raise FileNotFoundError(
        f"Ground truth not found: {gt_path}"
    )

print("\nLoading S1...")

train_s1 = pd.read_csv(
    s1_path,
    sep="\t",
    dtype=str
)

print(
    "S1 rows:",
    len(train_s1)
)

print(
    "S1 columns:",
    list(train_s1.columns)
)

print("\nLoading ground truth...")

train_gt = pd.read_csv(
    gt_path,
    sep="\t",
    dtype=str
)

print(
    "GT rows:",
    len(train_gt)
)

print(
    "GT columns:",
    list(train_gt.columns)
)

# ------------------------------------------------------------
# Normalize ID columns
# ------------------------------------------------------------

if "source1_entity_id" not in train_gt.columns:
    raise RuntimeError(
        "Expected source1_entity_id in ground truth."
    )

if "matched_entity_ids" not in train_gt.columns:
    raise RuntimeError(
        "Expected matched_entity_ids in ground truth."
    )

# ------------------------------------------------------------
# Reproduce EXACT earlier random_state=42 sample
# ------------------------------------------------------------

N_VALID = 20_000

if len(train_s1) < N_VALID:
    raise RuntimeError(
        "Training S1 contains fewer than 20,000 rows."
    )

validation_s1 = (
    train_s1
    .sample(
        n=N_VALID,
        random_state=42
    )
    .copy()
)

print(
    "\nValidation S1 sample:",
    len(validation_s1)
)

# ------------------------------------------------------------
# Join GT
# ------------------------------------------------------------

validation_s1 = validation_s1.merge(
    train_gt[
        [
            "source1_entity_id",
            "matched_entity_ids"
        ]
    ],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="left"
)

# ------------------------------------------------------------
# Parse GT
# ------------------------------------------------------------

def parse_gt(value):

    if pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    return set(
        x.strip()
        for x in value.split(",")
        if x.strip()
    )

validation_s1["true_ids"] = (
    validation_s1["matched_entity_ids"]
    .apply(parse_gt)
)

validation_s1["true_count"] = (
    validation_s1["true_ids"]
    .str.len()
)

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VALIDATION DISTRIBUTION")
print("=" * 100)

print(
    validation_s1[
        "true_count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "\nTotal S1:",
    len(validation_s1)
)

print(
    "No-match:",
    int(
        (
            validation_s1["true_count"] == 0
        ).sum()
    )
)

print(
    "Exactly one:",
    int(
        (
            validation_s1["true_count"] == 1
        ).sum()
    )
)

print(
    "Multiple:",
    int(
        (
            validation_s1["true_count"] > 1
        ).sum()
    )
)

print(
    "Total true pairs:",
    int(
        validation_s1["true_count"].sum()
    )
)

print(
    "Average matches:",
    validation_s1["true_count"].mean()
)

# ------------------------------------------------------------
# Save reconstructed validation set
# ------------------------------------------------------------

save_path = os.path.join(
    BASE,
    "MODELS",
    "representative_s1_validation_20k.parquet"
)

# Sets are not ideal for parquet, so serialize them.
validation_s1["true_ids_str"] = (
    validation_s1["true_ids"]
    .apply(
        lambda x: ",".join(sorted(x))
    )
)

validation_s1.drop(
    columns=["true_ids"],
    inplace=True
)

validation_s1.to_parquet(
    save_path,
    index=False
)

print(
    "\nSaved:",
    save_path
)

print(
    "\nCELL 86 COMPLETE."
)

CELL 86 — REBUILD REPRESENTATIVE 20K S1 VALIDATION
TRAIN_DIR:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/train

Files:
  train_ground_truth.tsv
  train_source2.tsv
  train_source3.tsv
  train_source1.tsv

Loading S1...
S1 rows: 2206821
S1 columns: ['entity_id', 'business_name', 'business_address', 'country']

Loading ground truth...
GT rows: 2206821
GT columns: ['source1_entity_id', 'matched_entity_ids']

Validation S1 sample: 20000

VALIDATION DISTRIBUTION
true_count
0     1131
1     1047
2     3394
3     4852
4     4347
5     2952
6     1487
7      595
8      157
9       31
10       7

Total S1: 20000
No-match: 1131
Exactly one: 1047
Multiple: 17822
Total true pairs: 69231
Average matches: 3.46155

Saved: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/representative_s1_validation_20k.parquet

CELL 86 COMPLETE.


In [ ]:
# CELL 87 — LOCATE EXISTING RAPIDFUZZ PIPELINE

print("=" * 100)
print("CELL 87 — EXISTING RAPIDFUZZ PIPELINE")
print("=" * 100)

# ------------------------------------------------------------
# DataFrames
# ------------------------------------------------------------

print("\nDATAFRAMES:")
for name, obj in globals().items():

    try:
        if isinstance(obj, pd.DataFrame):
            cols = list(obj.columns)

            relevant = any(
                any(
                    key in str(c).lower()
                    for key in [
                        "name",
                        "addr",
                        "ratio",
                        "number",
                        "channel",
                        "similar",
                        "candidate",
                        "query",
                        "label",
                        "s1_id",
                        "entity_id"
                    ]
                )
                for c in cols
            )

            if relevant:
                print(
                    f"\n{name}: shape={obj.shape}"
                )
                print(
                    "columns:",
                    cols[:80]
                )

    except Exception:
        pass


# ------------------------------------------------------------
# Functions
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FUNCTIONS")
print("=" * 100)

for name, obj in globals().items():

    if callable(obj):

        lname = name.lower()

        if any(
            key in lname
            for key in [
                "rapid",
                "candidate",
                "feature",
                "fuzzy",
                "match",
                "number",
                "address",
                "similar"
            ]
        ):

            print(
                name
            )


# ------------------------------------------------------------
# Important variables
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("IMPORTANT VARIABLES")
print("=" * 100)

for name in sorted(
    globals().keys()
):

    lname = name.lower()

    if any(
        key in lname
        for key in [
            "rapid",
            "candidate",
            "block",
            "index",
            "matcher",
            "number",
            "target",
            "source1",
            "s1"
        ]
    ):

        obj = globals()[name]

        try:
            if isinstance(obj, (pd.DataFrame, pd.Series)):
                print(
                    name,
                    type(obj).__name__,
                    obj.shape
                )
            elif isinstance(obj, (dict, list, tuple, set)):
                print(
                    name,
                    type(obj).__name__,
                    "len=",
                    len(obj)
                )
            else:
                print(
                    name,
                    type(obj).__name__
                )
        except Exception:
            print(
                name,
                type(obj).__name__
            )

print("\nCELL 87 COMPLETE.")

CELL 87 — EXISTING RAPIDFUZZ PIPELINE

DATAFRAMES:


RuntimeError: dictionary changed size during iteration

In [ ]:
# CELL 87B — SAFE EXISTING RAPIDFUZZ PIPELINE INSPECTION

import pandas as pd

print("=" * 100)
print("CELL 87B — SAFE EXISTING RAPIDFUZZ PIPELINE INSPECTION")
print("=" * 100)

# Take a static snapshot first
_namespace = dict(globals())

# ------------------------------------------------------------
# DATAFRAMES
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("DATAFRAMES")
print("=" * 100)

for name, obj in list(_namespace.items()):

    if isinstance(obj, pd.DataFrame):

        cols = list(obj.columns)

        relevant = any(
            any(
                key in str(col).lower()
                for key in [
                    "name",
                    "addr",
                    "ratio",
                    "number",
                    "channel",
                    "similar",
                    "candidate",
                    "query",
                    "label",
                    "s1_id",
                    "entity_id"
                ]
            )
            for col in cols
        )

        if relevant:

            print(
                f"\n{name}: shape={obj.shape}"
            )

            print(
                "columns:",
                cols
            )


# ------------------------------------------------------------
# FUNCTIONS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RAPIDFUZZ / MATCHING FUNCTIONS")
print("=" * 100)

for name, obj in list(_namespace.items()):

    if callable(obj):

        lname = str(name).lower()

        if any(
            key in lname
            for key in [
                "rapid",
                "candidate",
                "feature",
                "fuzzy",
                "match",
                "number",
                "address",
                "similar",
                "token"
            ]
        ):

            print(
                name
            )


# ------------------------------------------------------------
# IMPORTANT VARIABLES
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("IMPORTANT VARIABLES")
print("=" * 100)

for name in sorted(
    _namespace.keys()
):

    lname = str(name).lower()

    if any(
        key in lname
        for key in [
            "rapid",
            "candidate",
            "block",
            "index",
            "matcher",
            "number",
            "target",
            "source1",
            "source2",
            "source3",
            "s1",
            "s2",
            "s3"
        ]
    ):

        obj = _namespace[name]

        try:

            if isinstance(
                obj,
                pd.DataFrame
            ):

                print(
                    f"{name:45s} "
                    f"DataFrame shape={obj.shape}"
                )

            elif isinstance(
                obj,
                pd.Series
            ):

                print(
                    f"{name:45s} "
                    f"Series len={len(obj)}"
                )

            elif isinstance(
                obj,
                (dict, list, tuple, set)
            ):

                print(
                    f"{name:45s} "
                    f"{type(obj).__name__} "
                    f"len={len(obj)}"
                )

            else:

                print(
                    f"{name:45s} "
                    f"{type(obj).__name__}"
                )

        except Exception:

            print(
                f"{name:45s} "
                f"{type(obj).__name__}"
            )


print("\n" + "=" * 100)
print("CELL 87B COMPLETE")
print("=" * 100)

CELL 87B — SAFE EXISTING RAPIDFUZZ PIPELINE INSPECTION

DATAFRAMES

s1: shape=(20000, 7)
columns: ['entity_id', 'business_name', 'business_address', 'country', 'name_n', 'addr_n', 'country_n']

gt: shape=(20000, 2)
columns: ['source1_entity_id', 'matched_entity_ids']

gt_head: shape=(5, 2)
columns: ['source1_entity_id', 'matched_entity_ids']

chunk: shape=(35603, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']

hit: shape=(238, 5)
columns: ['entity_id', 'business_name', 'business_address', 'country', 'source']

targets: shape=(69478, 8)
columns: ['entity_id', 'business_name', 'business_address', 'country', 'source', 'name_n', 'addr_n', 'country_n']

sample: shape=(2000, 8)
columns: ['entity_id', 'business_name', 'business_address', 'country', 'source', 'name_n', 'addr_n', 'country_n']

df_cov: shape=(1893, 14)
columns: ['target_id', 'true_s1', 'covered', 'name_ratio', 'addr_ratio', 'name_partial', 'addr_partial', 'name_token', 'addr_token', 'num_overlap', 'nam

In [ ]:
# CELL 88 — INSPECT EXISTING K10 CANDIDATE GENERATOR

import inspect

print("=" * 100)
print("CELL 88 — EXISTING K10 CANDIDATE GENERATOR")
print("=" * 100)

print("\nFUNCTION SIGNATURE:")
print(
    inspect.signature(generate_k10_candidates)
)

print("\n" + "=" * 100)
print("FUNCTION SOURCE")
print("=" * 100)

try:
    print(
        inspect.getsource(
            generate_k10_candidates
        )
    )
except Exception as e:
    print(
        "Could not retrieve source:",
        repr(e)
    )

print("\n" + "=" * 100)
print("RELEVANT GLOBALS")
print("=" * 100)

for name in [
    "NUMBER_K",
    "s1",
    "s1_id_to_idx",
    "s1_id_to_idx_local",
    "number_addr_choices",
    "candidates",
    "candidate_counts",
    "max_candidates"
]:
    if name in globals():
        obj = globals()[name]

        print(
            f"\n{name}:"
        )
        print(
            "type =",
            type(obj).__name__
        )

        if isinstance(obj, pd.DataFrame):
            print(
                "shape =",
                obj.shape
            )
            print(
                "columns =",
                list(obj.columns)
            )

        elif isinstance(obj, dict):
            print(
                "len =",
                len(obj)
            )

        elif isinstance(obj, (list, tuple, set, np.ndarray)):
            print(
                "len/shape =",
                getattr(obj, "shape", len(obj))
            )

        else:
            print(
                "value =",
                repr(obj)[:500]
            )

print(
    "\nCELL 88 COMPLETE."
)

CELL 88 — EXISTING K10 CANDIDATE GENERATOR

FUNCTION SIGNATURE:
(r)

FUNCTION SOURCE
def generate_k10_candidates(r):

    c = r.country_n

    candidate_ids = set()

    # ========================================================
    # NAME FIRST TOP-3
    # ========================================================

    if r.name_n:

        choices = name_first_choices.get(
            (c, r.name_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        # ====================================================
        # NAME LAST TOP-3
        # ====================================================

        choices = name_last_choices.get(
            (c, r.name_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,

In [ ]:
# CELL 89 — RECOVER ORIGINAL FEATURE ENGINEERING CODE

print("=" * 100)
print("CELL 89 — SEARCH NOTEBOOK HISTORY FOR FEATURE PIPELINE")
print("=" * 100)

history = list(_ih)

keywords = [
    "df_matcher",
    "name_ratio",
    "addr_ratio",
    "combined_similarity",
    "name_addr_product",
    "balanced_similarity",
    "number_overlap",
]

matches = []

for i, code in enumerate(history):

    code_str = str(code)

    if any(
        key in code_str
        for key in keywords
    ):

        matches.append(
            (i, code_str)
        )

print(
    "\nMatching notebook cells:",
    len(matches)
)

for i, code in matches[-15:]:

    print("\n" + "-" * 100)
    print(
        f"NOTEBOOK CELL INDEX: {i}"
    )
    print("-" * 100)

    print(
        code[:12000]
    )

print(
    "\nCELL 89 COMPLETE."
)

CELL 89 — SEARCH NOTEBOOK HISTORY FOR FEATURE PIPELINE

Matching notebook cells: 12

----------------------------------------------------------------------------------------------------
NOTEBOOK CELL INDEX: 10
----------------------------------------------------------------------------------------------------
# CELL 73 — ANALYZE RAPIDFUZZ MISSED TRUE PAIRS

from rapidfuzz import fuzz
import numpy as np
import pandas as pd
import re

print("=" * 100)
print("CELL 73 — MISSED TRUE-PAIR ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# Use the same 2,000 validation targets
# ------------------------------------------------------------

sample = targets.sample(
    n=min(2000, len(targets)),
    random_state=42
).reset_index(drop=True)

missed = []
covered = []

for _, r in sample.iterrows():

    c = r.country_n
    candidate_ids = set()

    # --------------------------------------------------------
    # RapidFuzz Top-3
    # ------------------

In [ ]:
# CELL 90 — FIND ORIGINAL NOTEBOOK / FEATURE ENGINEERING SOURCE

import os
import json

print("=" * 100)
print("CELL 90 — FIND ORIGINAL NOTEBOOK / FEATURE ENGINEERING SOURCE")
print("=" * 100)

ROOT = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

keywords = [
    "df_matcher",
    "name_addr_product",
    "combined_similarity",
    "partial_balanced_similarity",
    "name_ratio",
    "addr_ratio"
]

matches = []

print("\nSearching project files...")

for dirpath, dirnames, filenames in os.walk(ROOT):

    # Skip very large irrelevant directories
    dirnames[:] = [
        d for d in dirnames
        if d not in {
            ".git",
            "__pycache__",
            "node_modules"
        }
    ]

    for filename in filenames:

        if not filename.endswith(
            (".ipynb", ".py")
        ):
            continue

        path = os.path.join(
            dirpath,
            filename
        )

        try:

            size_mb = (
                os.path.getsize(path)
                / (1024 * 1024)
            )

            # Avoid enormous files
            if size_mb > 100:
                continue

            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                text = f.read()

            found_keywords = [
                k for k in keywords
                if k in text
            ]

            if found_keywords:

                matches.append(
                    (
                        path,
                        found_keywords,
                        size_mb
                    )
                )

        except Exception as e:

            print(
                "Skipped:",
                path,
                repr(e)
            )

print(
    "\nMatching files:",
    len(matches)
)

for path, found_keywords, size_mb in matches:

    print("\n" + "-" * 100)

    print(
        "FILE:",
        path
    )

    print(
        "SIZE:",
        f"{size_mb:.2f} MB"
    )

    print(
        "MATCHES:",
        found_keywords
    )

# ------------------------------------------------------------
# Extract relevant notebook cells
# ------------------------------------------------------------

for path, found_keywords, size_mb in matches:

    if not path.endswith(
        ".ipynb"
    ):
        continue

    print("\n" + "=" * 100)
    print("NOTEBOOK:", path)
    print("=" * 100)

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            nb = json.load(f)

        for idx, cell in enumerate(
            nb.get("cells", [])
        ):

            source = "".join(
                cell.get("source", [])
            )

            if any(
                k in source
                for k in keywords
            ):

                print(
                    "\n" + "-" * 100
                )

                print(
                    "NOTEBOOK CELL:",
                    idx
                )

                print(
                    "-" * 100
                )

                print(
                    source[:20000]
                )

    except Exception as e:

        print(
            "Could not parse notebook:",
            repr(e)
        )

print(
    "\nCELL 90 COMPLETE."
)

CELL 90 — FIND ORIGINAL NOTEBOOK / FEATURE ENGINEERING SOURCE

Searching project files...

Matching files: 0

CELL 90 COMPLETE.


In [ ]:
# CELL 91 — VERIFY / RECOVER ORIGINAL FEATURE FORMULAS

import numpy as np
import pandas as pd
import re

from rapidfuzz import fuzz

print("=" * 100)
print("CELL 91 — VERIFY / RECOVER ORIGINAL FEATURE FORMULAS")
print("=" * 100)

# ------------------------------------------------------------
# Check required objects
# ------------------------------------------------------------

required = [
    "df_matcher",
    "targets",
    "s1",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}"
    )

# ------------------------------------------------------------
# Build lookup dictionaries
# ------------------------------------------------------------

target_lookup = targets.set_index(
    "entity_id"
)

s1_lookup = s1.set_index(
    "entity_id"
)

# Work on a copy
probe = df_matcher.copy()

# ------------------------------------------------------------
# Join raw target/S1 records
# ------------------------------------------------------------

probe["_q_name"] = (
    probe["query_id"]
    .map(target_lookup["name_n"])
    .fillna("")
)

probe["_q_addr"] = (
    probe["query_id"]
    .map(target_lookup["addr_n"])
    .fillna("")
)

probe["_q_country"] = (
    probe["query_id"]
    .map(target_lookup["country_n"])
    .fillna("")
)

probe["_s_name"] = (
    probe["s1_id"]
    .map(s1_lookup["name_n"])
    .fillna("")
)

probe["_s_addr"] = (
    probe["s1_id"]
    .map(s1_lookup["addr_n"])
    .fillna("")
)

probe["_s_country"] = (
    probe["s1_id"]
    .map(s1_lookup["country_n"])
    .fillna("")
)

print(
    "Rows available for verification:",
    len(probe)
)

# ------------------------------------------------------------
# Recompute obvious fuzzy features
# ------------------------------------------------------------

probe["_name_ratio_re"] = (
    [
        fuzz.ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_name"],
            probe["_s_name"]
        )
    ]
)

probe["_addr_ratio_re"] = (
    [
        fuzz.ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_addr"],
            probe["_s_addr"]
        )
    ]
)

probe["_name_partial_re"] = (
    [
        fuzz.partial_ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_name"],
            probe["_s_name"]
        )
    ]
)

probe["_addr_partial_re"] = (
    [
        fuzz.partial_ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_addr"],
            probe["_s_addr"]
        )
    ]
)

# Test both common token definitions
probe["_name_token_set"] = (
    [
        fuzz.token_set_ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_name"],
            probe["_s_name"]
        )
    ]
)

probe["_name_token_sort"] = (
    [
        fuzz.token_sort_ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_name"],
            probe["_s_name"]
        )
    ]
)

probe["_addr_token_set"] = (
    [
        fuzz.token_set_ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_addr"],
            probe["_s_addr"]
        )
    ]
)

probe["_addr_token_sort"] = (
    [
        fuzz.token_sort_ratio(a, b) / 100.0
        for a, b in zip(
            probe["_q_addr"],
            probe["_s_addr"]
        )
    ]
)

# ------------------------------------------------------------
# Exact / country / length
# ------------------------------------------------------------

probe["_exact_name_re"] = (
    probe["_q_name"]
    ==
    probe["_s_name"]
).astype(np.int8)

probe["_exact_addr_re"] = (
    probe["_q_addr"]
    ==
    probe["_s_addr"]
).astype(np.int8)

probe["_country_re"] = (
    probe["_q_country"]
    ==
    probe["_s_country"]
).astype(np.int8)

probe["_name_len_re"] = np.where(
    np.maximum(
        probe["_q_name"].str.len(),
        probe["_s_name"].str.len()
    ) > 0,

    np.minimum(
        probe["_q_name"].str.len(),
        probe["_s_name"].str.len()
    )
    /
    np.maximum(
        probe["_q_name"].str.len(),
        probe["_s_name"].str.len()
    ),

    1.0
)

probe["_addr_len_re"] = np.where(
    np.maximum(
        probe["_q_addr"].str.len(),
        probe["_s_addr"].str.len()
    ) > 0,

    np.minimum(
        probe["_q_addr"].str.len(),
        probe["_s_addr"].str.len()
    )
    /
    np.maximum(
        probe["_q_addr"].str.len(),
        probe["_s_addr"].str.len()
    ),

    1.0
)

# ------------------------------------------------------------
# Compare exact stored features
# ------------------------------------------------------------

def compare_feature(
    stored_col,
    recomputed_col,
    tol=1e-8
):

    stored = probe[stored_col].astype(float).to_numpy()
    recomputed = probe[recomputed_col].astype(float).to_numpy()

    diff = np.abs(
        stored - recomputed
    )

    return {
        "mean_abs_error": float(
            diff.mean()
        ),
        "max_abs_error": float(
            diff.max()
        ),
        "exact_or_close_pct": float(
            (diff <= tol).mean() * 100
        )
    }

comparisons = {}

for stored, recomputed in [
    ("name_ratio", "_name_ratio_re"),
    ("addr_ratio", "_addr_ratio_re"),
    ("name_partial", "_name_partial_re"),
    ("addr_partial", "_addr_partial_re"),
    ("exact_name", "_exact_name_re"),
    ("exact_address", "_exact_addr_re"),
    ("country_match", "_country_re"),
    ("name_length_ratio", "_name_len_re"),
    ("addr_length_ratio", "_addr_len_re"),
]:

    comparisons[
        stored
    ] = compare_feature(
        stored,
        recomputed
    )

# Token variants
for stored, recomputed in [
    ("name_token", "_name_token_set"),
    ("name_token", "_name_token_sort"),
    ("addr_token", "_addr_token_set"),
    ("addr_token", "_addr_token_sort"),
]:

    result = compare_feature(
        stored,
        recomputed
    )

    print(
        f"\n{stored} vs {recomputed}"
    )

    print(
        result
    )

print("\n" + "=" * 100)
print("DIRECT FEATURE VERIFICATION")
print("=" * 100)

for col, result in comparisons.items():

    print(
        f"{col:24s}"
        f" MAE={result['mean_abs_error']:.10f}"
        f" MAX={result['max_abs_error']:.10f}"
        f" MATCH={result['exact_or_close_pct']:.3f}%"
    )

# ------------------------------------------------------------
# Sample stored vs recomputed
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SAMPLE COMPARISON")
print("=" * 100)

sample_cols = [
    "query_id",
    "s1_id",
    "name_ratio",
    "_name_ratio_re",
    "addr_ratio",
    "_addr_ratio_re",
    "name_partial",
    "_name_partial_re",
    "addr_partial",
    "_addr_partial_re",
    "name_token",
    "_name_token_set",
    "_name_token_sort",
    "addr_token",
    "_addr_token_set",
    "_addr_token_sort",
    "name_length_ratio",
    "_name_len_re",
    "addr_length_ratio",
    "_addr_len_re",
]

print(
    probe[
        sample_cols
    ]
    .head(10)
    .round(4)
    .to_string(index=False)
)

print(
    "\nCELL 91 COMPLETE."
)

CELL 91 — VERIFY / RECOVER ORIGINAL FEATURE FORMULAS
Rows available for verification: 27557

name_token vs _name_token_set
{'mean_abs_error': 0.0, 'max_abs_error': 0.0, 'exact_or_close_pct': 100.0}

name_token vs _name_token_sort
{'mean_abs_error': 3.4082843678820874e-17, 'max_abs_error': 2.220446049250313e-16, 'exact_or_close_pct': 100.0}

addr_token vs _addr_token_set
{'mean_abs_error': 0.0, 'max_abs_error': 0.0, 'exact_or_close_pct': 100.0}

addr_token vs _addr_token_sort
{'mean_abs_error': 3.102647708573651e-17, 'max_abs_error': 2.220446049250313e-16, 'exact_or_close_pct': 100.0}

DIRECT FEATURE VERIFICATION
name_ratio               MAE=0.0000000000 MAX=0.0000000000 MATCH=100.000%
addr_ratio               MAE=0.0000000000 MAX=0.0000000000 MATCH=100.000%
name_partial             MAE=0.0000000000 MAX=0.0000000000 MATCH=100.000%
addr_partial             MAE=0.0000000000 MAX=0.0000000000 MATCH=100.000%
exact_name               MAE=0.0000000000 MAX=0.0000000000 MATCH=100.000%
exact_addr

In [ ]:
# CELL 92 — VERIFY ENHANCED INTERACTION FEATURES

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 92 — VERIFY ENHANCED INTERACTION FEATURES")
print("=" * 100)

d = df_enhanced.copy()

# ------------------------------------------------------------
# Reconstruct the interaction features
# ------------------------------------------------------------

reconstructed = pd.DataFrame(index=d.index)

reconstructed["name_addr_product"] = (
    d["name_ratio"] * d["addr_ratio"]
)

reconstructed["name_partial_addr_partial_product"] = (
    d["name_partial"] * d["addr_partial"]
)

reconstructed["name_number_product"] = (
    d["name_ratio"] * d["number_overlap"]
)

reconstructed["addr_number_product"] = (
    d["addr_ratio"] * d["number_overlap"]
)

reconstructed["min_name_addr"] = (
    d[["name_ratio", "addr_ratio"]].min(axis=1)
)

reconstructed["max_name_addr"] = (
    d[["name_ratio", "addr_ratio"]].max(axis=1)
)

reconstructed["name_addr_gap"] = (
    (d["name_ratio"] - d["addr_ratio"]).abs()
)

reconstructed["partial_gap"] = (
    (d["name_partial"] - d["addr_partial"]).abs()
)

# ------------------------------------------------------------
# Boolean interaction candidates
#
# These are reconstructed from the definitions used during
# the enhanced feature stage.
# ------------------------------------------------------------

reconstructed["strong_name_weak_address"] = (
    (
        d["name_ratio"] >= 0.80
    )
    &
    (
        d["addr_ratio"] < 0.20
    )
).astype(np.int8)

reconstructed["strong_name_no_number"] = (
    (
        d["name_ratio"] >= 0.80
    )
    &
    (
        d["number_overlap"] == 0
    )
).astype(np.int8)

reconstructed["strong_address"] = (
    (
        d["addr_ratio"] >= 0.65
    )
    &
    (
        d["number_overlap"] >= 0.50
    )
).astype(np.int8)

reconstructed["strong_name"] = (
    d["name_ratio"] >= 0.80
).astype(np.int8)

reconstructed["address_supported"] = (
    d["has_address_channel"]
).astype(np.int8)

reconstructed["multi_channel"] = (
    d["channel_count"] >= 2
).astype(np.int8)

reconstructed["balanced_similarity"] = (
    d["name_ratio"] + d["addr_ratio"]
) / 2.0

reconstructed["partial_balanced_similarity"] = (
    d["name_partial"] + d["addr_partial"]
) / 2.0

# ------------------------------------------------------------
# Compare every feature
# ------------------------------------------------------------

interaction_cols = list(
    reconstructed.columns
)

print(
    "Interaction features:",
    len(interaction_cols)
)

results = []

for col in interaction_cols:

    stored = d[col].astype(float).to_numpy()
    rec = reconstructed[col].astype(float).to_numpy()

    diff = np.abs(
        stored - rec
    )

    results.append({
        "feature": col,
        "mae": diff.mean(),
        "max_error": diff.max(),
        "match_pct": (
            (diff <= 1e-8).mean() * 100
        )
    })

results_df = pd.DataFrame(
    results
)

print("\n" + "=" * 100)
print("VERIFICATION RESULTS")
print("=" * 100)

print(
    results_df.to_string(
        index=False,
        formatters={
            "mae": "{:.10f}".format,
            "max_error": "{:.10f}".format,
            "match_pct": "{:.3f}".format
        }
    )
)

# ------------------------------------------------------------
# Overall check
# ------------------------------------------------------------

all_exact = (
    results_df["match_pct"] >= 99.999
).all()

print(
    "\nAll interaction features reproduced:",
    all_exact
)

if not all_exact:

    print(
        "\nFeatures that need investigation:"
    )

    print(
        results_df[
            results_df["match_pct"] < 99.999
        ].to_string(
            index=False
        )
    )

print(
    "\nCELL 92 COMPLETE."
)

CELL 92 — VERIFY ENHANCED INTERACTION FEATURES
Interaction features: 16

VERIFICATION RESULTS
                          feature          mae    max_error match_pct
                name_addr_product 0.0000000000 0.0000000000   100.000
name_partial_addr_partial_product 0.0000000000 0.0000000000   100.000
              name_number_product 0.0000000000 0.0000000000   100.000
              addr_number_product 0.0000000000 0.0000000000   100.000
                    min_name_addr 0.0000000000 0.0000000000   100.000
                    max_name_addr 0.0000000000 0.0000000000   100.000
                    name_addr_gap 0.0000000000 0.0000000000   100.000
                      partial_gap 0.0000000000 0.0000000000   100.000
         strong_name_weak_address 0.0000000000 0.0000000000   100.000
            strong_name_no_number 0.0000000000 0.0000000000   100.000
                   strong_address 0.0271800269 1.0000000000    97.282
                      strong_name 0.0000000000 0.0000000000   100.

In [ ]:
# CELL 93 — RECOVER EXACT strong_address FORMULA

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 93 — RECOVER EXACT strong_address FORMULA")
print("=" * 100)

d = df_enhanced.copy()

stored = (
    d["strong_address"]
    .astype(np.int8)
    .to_numpy()
)

# ------------------------------------------------------------
# Basic prevalence
# ------------------------------------------------------------

print(
    "\nStored strong_address prevalence:",
    f"{stored.mean() * 100:.4f}%"
)

print(
    "Stored positives:",
    int(stored.sum()),
    "/",
    len(stored)
)

# ------------------------------------------------------------
# Candidate formula generator
# ------------------------------------------------------------

candidates = []

# Simple thresholds
for threshold in np.arange(
    0.50,
    0.951,
    0.01
):

    candidates.append(
        (
            f"addr_ratio >= {threshold:.2f}",
            (
                d["addr_ratio"]
                >= threshold
            )
        )
    )

    candidates.append(
        (
            f"addr_partial >= {threshold:.2f}",
            (
                d["addr_partial"]
                >= threshold
            )
        )
    )

# Address ratio + number overlap
for addr_t in np.arange(
    0.50,
    0.951,
    0.01
):

    for num_t in [
        0.0,
        0.01,
        0.25,
        0.50,
        0.75,
        1.0
    ]:

        candidates.append(
            (
                f"addr_ratio >= {addr_t:.2f} "
                f"& number_overlap >= {num_t:.2f}",

                (
                    (d["addr_ratio"] >= addr_t)
                    &
                    (d["number_overlap"] >= num_t)
                )
            )
        )

# Address ratio + number channel
for addr_t in np.arange(
    0.50,
    0.951,
    0.01
):

    candidates.append(
        (
            f"addr_ratio >= {addr_t:.2f} "
            f"& has_number_channel",

            (
                (d["addr_ratio"] >= addr_t)
                &
                (d["has_number_channel"] == 1)
            )
        )
    )

# Address ratio + exact address
for addr_t in np.arange(
    0.50,
    0.951,
    0.01
):

    candidates.append(
        (
            f"addr_ratio >= {addr_t:.2f} "
            f"| exact_address",

            (
                (d["addr_ratio"] >= addr_t)
                |
                (d["exact_address"] == 1)
            )
        )
    )

    candidates.append(
        (
            f"addr_ratio >= {addr_t:.2f} "
            f"& (number_overlap > 0 | exact_address)",

            (
                (d["addr_ratio"] >= addr_t)
                &
                (
                    (d["number_overlap"] > 0)
                    |
                    (d["exact_address"] == 1)
                )
            )
        )
    )

# Address ratio + partial address
for addr_t in np.arange(
    0.50,
    0.951,
    0.01
):

    for partial_t in [
        0.60,
        0.65,
        0.70,
        0.75,
        0.80,
        0.85,
        0.90
    ]:

        candidates.append(
            (
                f"addr_ratio >= {addr_t:.2f} "
                f"& addr_partial >= {partial_t:.2f}",

                (
                    (d["addr_ratio"] >= addr_t)
                    &
                    (d["addr_partial"] >= partial_t)
                )
            )
        )

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

results = []

for description, mask in candidates:

    predicted = (
        np.asarray(mask)
        .astype(np.int8)
    )

    mismatch = (
        predicted
        !=
        stored
    )

    results.append({
        "formula": description,
        "mismatch_pct": (
            mismatch.mean() * 100
        ),
        "exact_match_pct": (
            (~mismatch).mean() * 100
        ),
        "predicted_positive_pct": (
            predicted.mean() * 100
        ),
        "predicted_positive": int(
            predicted.sum()
        )
    })

results_df = (
    pd.DataFrame(results)
    .sort_values(
        [
            "mismatch_pct",
            "exact_match_pct"
        ],
        ascending=[
            True,
            False
        ]
    )
)

print(
    "\n" + "=" * 100
)
print(
    "BEST CANDIDATE FORMULAS"
)
print(
    "=" * 100
)

print(
    results_df.head(25).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Explicitly test the likely rule
# ------------------------------------------------------------

likely = (
    d["addr_ratio"] >= 0.65
)

likely_mismatch = (
    likely.astype(np.int8)
    != stored
)

print(
    "\n" + "=" * 100
)
print(
    "LIKELY RULE TEST: addr_ratio >= 0.65"
)
print(
    "=" * 100
)

print(
    "Mismatch:",
    f"{likely_mismatch.mean() * 100:.4f}%"
)

print(
    "Exact match:",
    f"{(~likely_mismatch).mean() * 100:.4f}%"
)

print(
    "Predicted positives:",
    int(likely.sum())
)

print(
    "Stored positives:",
    int(stored.sum())
)

print(
    "\nCELL 93 COMPLETE."
)

CELL 93 — RECOVER EXACT strong_address FORMULA

Stored strong_address prevalence: 6.2053%
Stored positives: 1710 / 27557

BEST CANDIDATE FORMULAS
                                    formula  mismatch_pct  exact_match_pct  predicted_positive_pct  predicted_positive
                         addr_ratio >= 0.70      0.054433        99.945567                6.150887                1695
addr_ratio >= 0.70 & number_overlap >= 0.00      0.054433        99.945567                6.150887                1695
         addr_ratio >= 0.70 | exact_address      0.054433        99.945567                6.150887                1695
  addr_ratio >= 0.70 & addr_partial >= 0.60      0.054433        99.945567                6.150887                1695
  addr_ratio >= 0.70 & addr_partial >= 0.65      0.072577        99.927423                6.132743                1690
  addr_ratio >= 0.70 & addr_partial >= 0.70      0.148783        99.851217                6.056537                1669
                     

In [ ]:
# CELL 94 — ISOLATE EXACT strong_address RULE

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 94 — ISOLATE EXACT strong_address RULE")
print("=" * 100)

d = df_enhanced.copy()

stored = (
    d["strong_address"]
    .astype(np.int8)
)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def check_rule(name, mask):

    pred = pd.Series(
        np.asarray(mask).astype(np.int8),
        index=d.index
    )

    mismatch = (
        pred != stored
    )

    return {
        "rule": name,
        "mismatch": int(mismatch.sum()),
        "match_pct": (
            (~mismatch).mean() * 100
        ),
        "positives": int(pred.sum())
    }


# ------------------------------------------------------------
# Atomic predicates
# ------------------------------------------------------------

atoms = {

    "addr>=.70":
        d["addr_ratio"] >= 0.70,

    "addr>.70":
        d["addr_ratio"] > 0.70,

    "addr>=.69":
        d["addr_ratio"] >= 0.69,

    "addr>=.71":
        d["addr_ratio"] >= 0.71,

    "addr_partial>=.60":
        d["addr_partial"] >= 0.60,

    "addr_partial>=.65":
        d["addr_partial"] >= 0.65,

    "addr_partial>=.70":
        d["addr_partial"] >= 0.70,

    "number>0":
        d["number_overlap"] > 0,

    "number>=.50":
        d["number_overlap"] >= 0.50,

    "number==1":
        d["number_overlap"] == 1,

    "has_number":
        d["has_number_channel"] == 1,

    "exact_address":
        d["exact_address"] == 1,

    "address_supported":
        d["address_supported"] == 1,

    "name>=.50":
        d["name_ratio"] >= 0.50,

    "name>=.60":
        d["name_ratio"] >= 0.60,

    "name>=.70":
        d["name_ratio"] >= 0.70,

    "name>=.80":
        d["name_ratio"] >= 0.80,
}


# ------------------------------------------------------------
# Test atomic predicates
# ------------------------------------------------------------

results = []

for name, mask in atoms.items():

    results.append(
        check_rule(
            name,
            mask
        )
    )


# ------------------------------------------------------------
# Test useful conjunctions
# ------------------------------------------------------------

for addr_name in [
    "addr>=.70",
    "addr>.70",
    "addr>=.69",
    "addr>=.71",
]:

    a = atoms[addr_name]

    for extra_name in [
        "number>0",
        "number>=.50",
        "number==1",
        "has_number",
        "exact_address",
        "address_supported",
    ]:

        b = atoms[extra_name]

        results.append(
            check_rule(
                f"{addr_name} AND {extra_name}",
                a & b
            )
        )

        results.append(
            check_rule(
                f"{addr_name} OR {extra_name}",
                a | b
            )
        )


# ------------------------------------------------------------
# Test address-ratio + partial-ratio
# ------------------------------------------------------------

for addr_name in [
    "addr>=.70",
    "addr>.70",
    "addr>=.69",
    "addr>=.71",
]:

    for partial_name in [
        "addr_partial>=.60",
        "addr_partial>=.65",
        "addr_partial>=.70",
    ]:

        a = atoms[addr_name]
        b = atoms[partial_name]

        results.append(
            check_rule(
                f"{addr_name} AND {partial_name}",
                a & b
            )
        )


# ------------------------------------------------------------
# Test three-way combinations
# ------------------------------------------------------------

for addr_name in [
    "addr>=.70",
    "addr>.70",
    "addr>=.69",
    "addr>=.71",
]:

    for number_name in [
        "number>0",
        "number>=.50",
        "number==1",
        "has_number",
    ]:

        a = atoms[addr_name]
        b = atoms[number_name]

        for partial_name in [
            "addr_partial>=.60",
            "addr_partial>=.65",
            "addr_partial>=.70",
        ]:

            c = atoms[partial_name]

            results.append(
                check_rule(
                    f"{addr_name} AND "
                    f"{number_name} AND "
                    f"{partial_name}",
                    a & b & c
                )
            )


# ------------------------------------------------------------
# Rank best candidates
# ------------------------------------------------------------

results_df = (
    pd.DataFrame(results)
    .sort_values(
        [
            "mismatch",
            "match_pct"
        ],
        ascending=[
            True,
            False
        ]
    )
)

print(
    "\n" + "=" * 100
)
print(
    "BEST RULES"
)
print(
    "=" * 100
)

print(
    results_df.head(30).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Examine the mismatches for addr >= .70
# ------------------------------------------------------------

base_rule = (
    d["addr_ratio"] >= 0.70
)

mismatch_mask = (
    base_rule.astype(np.int8)
    !=
    stored
)

mismatch_rows = d.loc[
    mismatch_mask
].copy()

print(
    "\n" + "=" * 100
)
print(
    "MISMATCHES FOR addr_ratio >= 0.70"
)
print(
    "=" * 100
)

cols = [
    "query_id",
    "s1_id",
    "strong_address",
    "addr_ratio",
    "addr_partial",
    "number_overlap",
    "has_number_channel",
    "exact_address",
    "address_supported",
    "name_ratio",
    "name_partial",
    "combined_similarity",
    "channel_count",
]

print(
    mismatch_rows[
        cols
    ]
    .sort_values(
        [
            "strong_address",
            "addr_ratio"
        ],
        ascending=[
            False,
            False
        ]
    )
    .to_string(
        index=False
    )
)

print(
    "\nStored positives among mismatches:",
    int(
        mismatch_rows["strong_address"].sum()
    )
)

print(
    "Stored negatives among mismatches:",
    int(
        len(mismatch_rows)
        -
        mismatch_rows["strong_address"].sum()
    )
)

print(
    "\nCELL 94 COMPLETE."
)

CELL 94 — ISOLATE EXACT strong_address RULE

BEST RULES
                                          rule  mismatch  match_pct  positives
                                     addr>=.70         0 100.000000       1710
                    addr>=.70 OR exact_address         0 100.000000       1710
               addr>=.70 AND addr_partial>=.60         0 100.000000       1710
               addr>=.70 AND addr_partial>=.65         5  99.981856       1705
                                      addr>.70        15  99.945567       1695
                     addr>.70 OR exact_address        15  99.945567       1695
                addr>.70 AND addr_partial>=.60        15  99.945567       1695
                addr>.70 AND addr_partial>=.65        20  99.927423       1690
               addr>=.70 AND addr_partial>=.70        27  99.902021       1683
                addr>.70 AND addr_partial>=.70        41  99.851217       1669
               addr>=.70 AND address_supported        73  99.735095       1

In [ ]:
# EXPRESS SUBMISSION — DOWNLOAD EXISTING SUBMISSION 3

from google.colab import files
import os

zip_path = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
    "SUBMISSION_3/submission_3.zip"
)

print("Checking:", zip_path)

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Submission ZIP not found:\n{zip_path}"
    )

size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print(
    f"\nSubmission 3 found: {size_mb:.2f} MB"
)

files.download(zip_path)

print("\nSubmission ZIP sent for download.")

Checking: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_3/submission_3.zip

Submission 3 found: 91.21 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Submission ZIP sent for download.


In [ ]:
# CELL 96 — PRODUCTION PREFLIGHT

import os
import pickle
import pandas as pd
import numpy as np
import lightgbm as lgb

print("=" * 100)
print("CELL 96 — PRODUCTION PREFLIGHT")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TEST_DIR = os.path.join(
    BASE,
    "DATASET",
    "test"
)

MODEL_DIR = os.path.join(
    BASE,
    "MODELS"
)

# ------------------------------------------------------------
# Discover test files
# ------------------------------------------------------------

print("\nTEST DIRECTORY:")
print(TEST_DIR)

if not os.path.exists(TEST_DIR):
    raise FileNotFoundError(
        f"Test directory not found:\n{TEST_DIR}"
    )

all_test_files = os.listdir(
    TEST_DIR
)

print("\nTest directory files:")

for f in sorted(all_test_files):
    print(" ", f)

# ------------------------------------------------------------
# Locate source files
# ------------------------------------------------------------

def find_file(
    candidates,
    directory
):

    for name in candidates:

        path = os.path.join(
            directory,
            name
        )

        if os.path.exists(path):
            return path

    return None


test_s1_path = find_file(
    [
        "test_source1.tsv",
        "source1.tsv",
        "test_s1.tsv"
    ],
    TEST_DIR
)

test_s2_path = find_file(
    [
        "test_source2.tsv",
        "source2.tsv",
        "test_s2.tsv"
    ],
    TEST_DIR
)

test_s3_path = find_file(
    [
        "test_source3.tsv",
        "source3.tsv",
        "test_s3.tsv"
    ],
    TEST_DIR
)

print("\nResolved source files:")

print(
    "S1:",
    test_s1_path
)

print(
    "S2:",
    test_s2_path
)

print(
    "S3:",
    test_s3_path
)

if any(
    x is None
    for x in [
        test_s1_path,
        test_s2_path,
        test_s3_path
    ]
):
    raise RuntimeError(
        "Could not locate all three test source files."
    )

# ------------------------------------------------------------
# Load TEST S1
# ------------------------------------------------------------

print("\nLoading TEST S1...")

test_s1 = pd.read_csv(
    test_s1_path,
    sep="\t",
    dtype=str
)

print(
    "TEST S1 shape:",
    test_s1.shape
)

print(
    "TEST S1 columns:",
    list(test_s1.columns)
)

# ------------------------------------------------------------
# Load TEST S2
# ------------------------------------------------------------

print("\nLoading TEST S2...")

test_s2 = pd.read_csv(
    test_s2_path,
    sep="\t",
    dtype=str
)

print(
    "TEST S2 shape:",
    test_s2.shape
)

print(
    "TEST S2 columns:",
    list(test_s2.columns)
)

# ------------------------------------------------------------
# Load TEST S3
# ------------------------------------------------------------

print("\nLoading TEST S3...")

test_s3 = pd.read_csv(
    test_s3_path,
    sep="\t",
    dtype=str
)

print(
    "TEST S3 shape:",
    test_s3.shape
)

print(
    "TEST S3 columns:",
    list(test_s3.columns)
)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

required_cols = [
    "entity_id",
    "business_name",
    "business_address",
    "country"
]

for name, df in [
    ("TEST S1", test_s1),
    ("TEST S2", test_s2),
    ("TEST S3", test_s3)
]:

    missing = [
        c for c in required_cols
        if c not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{name} missing columns: {missing}"
        )

    print(
        f"\n{name}: missing IDs =",
        int(df["entity_id"].isna().sum())
    )

# ------------------------------------------------------------
# Discover blocking indexes
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TEST BLOCKING INDEXES")
print("=" * 100)

index_candidates = []

for filename in all_test_files:

    lower = filename.lower()

    if (
        "block" in lower
        and lower.endswith(
            (
                ".pkl",
                ".pickle"
            )
        )
    ):

        index_candidates.append(
            os.path.join(
                TEST_DIR,
                filename
            )
        )

for path in sorted(index_candidates):

    print(
        "\nINDEX:",
        path
    )

    try:

        with open(
            path,
            "rb"
        ) as f:

            idx_obj = pickle.load(f)

        print(
            "type:",
            type(idx_obj).__name__
        )

        if isinstance(
            idx_obj,
            dict
        ):

            print(
                "keys:",
                list(idx_obj.keys())
            )

            for key, value in idx_obj.items():

                try:
                    print(
                        f"  {key}: "
                        f"type={type(value).__name__}, "
                        f"len={len(value)}"
                    )

                    # One small sample
                    if len(value) > 0:

                        first_key = next(
                            iter(value)
                        )

                        first_value = value[
                            first_key
                        ]

                        print(
                            "    sample key:",
                            repr(first_key)
                        )

                        print(
                            "    sample value:",
                            repr(first_value)[:300]
                        )

                except Exception as e:

                    print(
                        f"  {key}: inspection failed:",
                        repr(e)
                    )

    except Exception as e:

        print(
            "LOAD ERROR:",
            repr(e)
        )

# ------------------------------------------------------------
# Load saved ranking model
# ------------------------------------------------------------

model_path = os.path.join(
    MODEL_DIR,
    "rapidfuzz_k10_ranking_lgbm_best.txt"
)

threshold_path = os.path.join(
    MODEL_DIR,
    "rapidfuzz_k10_ranking_threshold.txt"
)

print("\n" + "=" * 100)
print("RANKING MODEL")
print("=" * 100)

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Ranking model not found:\n{model_path}"
    )

if not os.path.exists(threshold_path):
    raise FileNotFoundError(
        f"Threshold not found:\n{threshold_path}"
    )

production_model = lgb.Booster(
    model_file=model_path
)

with open(
    threshold_path,
    "r"
) as f:

    production_threshold = float(
        f.read().strip()
    )

print(
    "Model:",
    model_path
)

print(
    "Model trees:",
    production_model.num_trees()
)

print(
    "Model features:",
    production_model.num_feature()
)

print(
    "Production threshold:",
    production_threshold
)

# ------------------------------------------------------------
# Confirm expected threshold
# ------------------------------------------------------------

if abs(
    production_threshold - 0.225
) > 1e-6:

    print(
        "\nWARNING: saved threshold differs "
        "from expected 0.225"
    )

# ------------------------------------------------------------
# Memory estimate
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("DATASET SIZE SUMMARY")
print("=" * 100)

for name, df in [
    ("S1", test_s1),
    ("S2", test_s2),
    ("S3", test_s3)
]:

    print(
        f"{name}: {len(df):,} rows"
    )

print(
    "\nTotal target rows:",
    f"{len(test_s2) + len(test_s3):,}"
)

print(
    "\nCELL 96 COMPLETE."
)

CELL 96 — PRODUCTION PREFLIGHT

TEST DIRECTORY:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/test

Test directory files:
  test_source1.tsv
  test_source2.tsv
  test_source3.tsv

Resolved source files:
S1: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/test/test_source1.tsv
S2: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/test/test_source2.tsv
S3: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/test/test_source3.tsv

Loading TEST S1...
TEST S1 shape: (1732544, 4)
TEST S1 columns: ['entity_id', 'business_name', 'business_address', 'country']

Loading TEST S2...
TEST S2 shape: (4887273, 4)
TEST S2 columns: ['entity_id', 'business_name', 'business_address', 'country']

Loading TEST S3...
TEST S3 shape: (5082316, 4)
TEST S3 columns: ['entity_id', 'business_name', 'business_address', 'country']

TEST S1: missing IDs = 0

TEST S2: missing IDs = 0

TEST S3: missing IDs = 0

TEST BLOCKING INDEXES

RANKING MODEL
Model: /content/drive/My

In [ ]:
# CELL 97 — LOCATE EXISTING PRODUCTION INDEXES

import os

print("=" * 100)
print("CELL 97 — LOCATE EXISTING PRODUCTION INDEXES")
print("=" * 100)

SEARCH_ROOTS = [
    "/content/amazon_ml_2026",
    "/content",
    "/content/drive/MyDrive/AMAZON ML 2026"
]

patterns = [
    "test_s2_block_indexes",
    "test_s3_block_indexes",
    "test_s2",
    "test_s3",
    "blocking_indexes",
    "name_first",
    "name_last",
    "addr_first",
    "addr_last",
    "number_addr"
]

found = []

for root in SEARCH_ROOTS:

    if not os.path.exists(root):
        continue

    print(
        "\nSearching:",
        root
    )

    for dirpath, dirnames, filenames in os.walk(root):

        # Avoid irrelevant / huge directories
        dirnames[:] = [
            d for d in dirnames
            if d not in {
                ".git",
                "__pycache__",
                "node_modules"
            }
        ]

        for filename in filenames:

            low = filename.lower()

            if any(
                p.lower() in low
                for p in patterns
            ):

                path = os.path.join(
                    dirpath,
                    filename
                )

                try:
                    size_mb = (
                        os.path.getsize(path)
                        / (1024 * 1024)
                    )
                except Exception:
                    size_mb = -1

                found.append(
                    (
                        path,
                        size_mb
                    )
                )

# ------------------------------------------------------------
# Deduplicate
# ------------------------------------------------------------

unique_found = {}

for path, size_mb in found:
    unique_found[path] = size_mb

print("\n" + "=" * 100)
print("MATCHING ARTIFACTS")
print("=" * 100)

if not unique_found:

    print(
        "NO EXISTING PRODUCTION INDEXES FOUND."
    )

else:

    for path in sorted(
        unique_found.keys()
    ):

        print(
            f"{unique_found[path]:10.2f} MB  "
            f"{path}"
        )

# ------------------------------------------------------------
# Specifically check likely directories
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LIKELY INDEX DIRECTORIES")
print("=" * 100)

likely_dirs = [
    "/content/amazon_ml_2026/MODELS",
    "/content/amazon_ml_2026/MODELS/blocking_indexes",
    "/content/amazon_ml_2026",
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS"
]

for directory in likely_dirs:

    print(
        "\nDIRECTORY:",
        directory
    )

    if not os.path.exists(directory):

        print(
            "  [missing]"
        )
        continue

    try:

        entries = os.listdir(
            directory
        )

        print(
            "  entries:",
            len(entries)
        )

        for entry in sorted(entries)[:100]:

            print(
                "   ",
                entry
            )

        if len(entries) > 100:
            print(
                "    ...",
                len(entries) - 100,
                "more"
            )

    except Exception as e:

        print(
            "  ERROR:",
            repr(e)
        )

print(
    "\nCELL 97 COMPLETE."
)

CELL 97 — LOCATE EXISTING PRODUCTION INDEXES

Searching: /content

Searching: /content/drive/MyDrive/AMAZON ML 2026

MATCHING ARTIFACTS
    427.46 MB  /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocking_indexes/test_s2_block_indexes.pkl
    452.87 MB  /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocking_indexes/test_s3_block_indexes.pkl
     18.64 MB  /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/test_s2_assigned_s1.dat
     18.64 MB  /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/test_s2_target_num.dat
     18.64 MB  /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/test_s2_tfidf_score.dat
      0.00 MB  /content/drive/MyDrive/sentinel-ro-complete/.venv/Lib/site-packages/pandas/tests/io/test_s3.py
      0.00 MB  /content/drive/MyDrive/sentinel-ro-complete/sentinel-ro/venv/Lib/site-packages/pandas/tests/io/test_s3.py

LIKELY INDEX DIRECTORIES

DIRECTORY: /content/amazon_ml_2026/MODELS
  [missing]

DIRECTORY: /content

In [ ]:
# CELL 98 — INSPECT EXISTING PRODUCTION CANDIDATE ARTIFACTS

import os

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

MODELS = os.path.join(
    BASE,
    "MODELS"
)

PARTS_DIR = os.path.join(
    MODELS,
    "production_candidate_parts"
)

print("=" * 100)
print("CELL 98 — EXISTING PRODUCTION CANDIDATE ARTIFACTS")
print("=" * 100)

print(
    "\nDirectory:",
    PARTS_DIR
)

if not os.path.isdir(PARTS_DIR):

    print(
        "production_candidate_parts DOES NOT EXIST."
    )

else:

    entries = os.listdir(
        PARTS_DIR
    )

    print(
        "Number of entries:",
        len(entries)
    )

    total_bytes = 0

    for name in sorted(entries):

        path = os.path.join(
            PARTS_DIR,
            name
        )

        try:

            size = os.path.getsize(
                path
            )

            total_bytes += size

            print(
                f"{size / (1024**2):10.2f} MB   {name}"
            )

        except OSError:

            print(
                "Unable to inspect:",
                name
            )

    print(
        "\nTotal size:",
        f"{total_bytes / (1024**3):.3f} GB"
    )

# ------------------------------------------------------------
# Also inspect submission directory
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "EXISTING SUBMISSION ARTIFACTS"
)
print(
    "=" * 100
)

for dirname in [
    "SUBMISSION_3",
    "SUBMISSION_4",
    "submission_3",
    "submission_4"
]:

    path = os.path.join(
        BASE,
        dirname
    )

    if os.path.isdir(path):

        print(
            "\nDIRECTORY:",
            path
        )

        for name in sorted(
            os.listdir(path)
        ):

            full = os.path.join(
                path,
                name
            )

            try:
                size = os.path.getsize(
                    full
                )
                print(
                    f"{size / (1024**2):10.2f} MB   {name}"
                )
            except OSError:
                pass

print(
    "\nCELL 98 COMPLETE."
)

CELL 98 — EXISTING PRODUCTION CANDIDATE ARTIFACTS

Directory: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/production_candidate_parts
Number of entries: 0

Total size: 0.000 GB

EXISTING SUBMISSION ARTIFACTS

DIRECTORY: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_3
    177.22 MB   candidate_pairs.tsv
     39.75 MB   matching_pairs.tsv
     39.75 MB   matching_results.tsv
     91.21 MB   submission_3.zip

CELL 98 COMPLETE.


In [ ]:
# CELL 99 — INSPECT EXISTING SUBMISSION 3 CANDIDATES

import os
import pandas as pd

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

SUB_DIR = os.path.join(
    BASE,
    "SUBMISSION_3"
)

candidate_path = os.path.join(
    SUB_DIR,
    "candidate_pairs.tsv"
)

matching_path = os.path.join(
    SUB_DIR,
    "matching_results.tsv"
)

block_validation_path = os.path.join(
    BASE,
    "MODELS",
    "submission3_blocker_validation.csv"
)

print("=" * 100)
print("CELL 99 — INSPECT EXISTING SUBMISSION 3 CANDIDATES")
print("=" * 100)

# ------------------------------------------------------------
# Candidate file
# ------------------------------------------------------------

print("\nCANDIDATE FILE:")
print(candidate_path)

with open(
    candidate_path,
    "r",
    encoding="utf-8"
) as f:

    for i in range(8):
        line = f.readline()

        if not line:
            break

        print(
            f"LINE {i+1}:",
            line.rstrip()
        )

# ------------------------------------------------------------
# Count rows
# ------------------------------------------------------------

print("\nCounting candidate rows...")

candidate_rows = 0

with open(
    candidate_path,
    "r",
    encoding="utf-8"
) as f:

    for _ in f:
        candidate_rows += 1

print(
    "Candidate rows:",
    f"{candidate_rows:,}"
)

# ------------------------------------------------------------
# Matching results
# ------------------------------------------------------------

print("\nMATCHING RESULTS SAMPLE:")

with open(
    matching_path,
    "r",
    encoding="utf-8"
) as f:

    for i in range(8):
        line = f.readline()

        if not line:
            break

        print(
            f"LINE {i+1}:",
            line.rstrip()
        )

# ------------------------------------------------------------
# Blocker validation artifact
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "BLOCKER VALIDATION ARTIFACT"
)
print(
    "=" * 100
)

if os.path.exists(
    block_validation_path
):

    blocker = pd.read_csv(
        block_validation_path
    )

    print(
        "Shape:",
        blocker.shape
    )

    print(
        "Columns:",
        list(blocker.columns)
    )

    print(
        "\nContents:"
    )

    print(
        blocker.to_string(
            index=False
        )
    )

else:

    print(
        "submission3_blocker_validation.csv not found."
    )

# ------------------------------------------------------------
# ZIP metadata
# ------------------------------------------------------------

zip_path = os.path.join(
    SUB_DIR,
    "submission_3.zip"
)

print(
    "\n" + "=" * 100
)
print(
    "FILE SIZES"
)
print(
    "=" * 100
)

for path in [
    candidate_path,
    matching_path,
    zip_path
]:

    if os.path.exists(path):

        size_mb = (
            os.path.getsize(path)
            / (1024 * 1024)
        )

        print(
            f"{os.path.basename(path):30s}"
            f"{size_mb:10.2f} MB"
        )

print(
    "\nCELL 99 COMPLETE."
)

CELL 99 — INSPECT EXISTING SUBMISSION 3 CANDIDATES

CANDIDATE FILE:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_3/candidate_pairs.tsv
LINE 1: source1_entity_id	candidate_entity_ids
LINE 2: S1-714132312	S2-902693916,S2-36973049,S2-742926619,S2-406131322,S2-131762854,S2-723568545,S2-49406445,S2-188157420,S2-365284937,S2-211902825,S3-76581442,S3-948321994,S3-954194052,S3-96374362,S3-521557639,S3-635835046,S3-589056677,S3-987464583,S3-53762159,S3-836630817
LINE 3: S1-106407869	S2-47475427,S2-886540563,S2-997023970,S2-571602010,S2-104974637,S3-943074147,S3-696641691,S3-245724670,S3-918090607,S3-654401737
LINE 4: S1-156285671	S2-64661346,S2-160966771,S2-181576625,S2-721272321,S2-868820087,S3-762835469,S3-705874844,S3-875859009,S3-277504297,S3-516625443
LINE 5: S1-689823050	S2-167166712,S2-427104827,S2-149894235,S2-15748596,S2-777104795,S2-547662890,S2-681451241,S2-531801724,S2-178174931,S2-337849295,S3-739856028,S3-10157948,S3-830626961,S3-690490714,S3-996830856,S3-767800

In [ ]:
# CELL 100 — BUILD PRODUCTION S1 RAPIDFUZZ INDEXES

import os
import gc
import re
import time
import psutil
from collections import defaultdict

print("=" * 100)
print("CELL 100 — BUILD PRODUCTION S1 RAPIDFUZZ INDEXES")
print("=" * 100)

# ------------------------------------------------------------
# Memory before
# ------------------------------------------------------------

process = psutil.Process()

def ram_gb():
    return process.memory_info().rss / (1024 ** 3)

print(
    "\nRAM before:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Release full S2/S3 tables.
# They are not needed for candidate generation.
# They can be reloaded later in chunks.
# ------------------------------------------------------------

for var_name in [
    "test_s2",
    "test_s3"
]:

    if var_name in globals():

        print(
            "Releasing:",
            var_name
        )

        del globals()[var_name]

gc.collect()

print(
    "RAM after releasing S2/S3:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Copy TEST S1 and normalize exactly as validation pipeline
# ------------------------------------------------------------

prod_s1 = test_s1.copy()

print(
    "\nTEST S1 rows:",
    f"{len(prod_s1):,}"
)

print(
    "Normalizing S1 fields..."
)

t0 = time.time()

prod_s1["name_n"] = (
    prod_s1["business_name"]
    .map(safe_token)
)

prod_s1["addr_n"] = (
    prod_s1["business_address"]
    .map(safe_token)
)

prod_s1["country_n"] = (
    prod_s1["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Normalization time:",
    f"{time.time() - t0:.2f} sec"
)

# ------------------------------------------------------------
# Local row index
# ------------------------------------------------------------

prod_s1["local_idx"] = np.arange(
    len(prod_s1),
    dtype=np.int32
)

# ------------------------------------------------------------
# Structures
#
# Each RapidFuzz choices dictionary is:
#     local_s1_index -> normalized string
#
# This preserves the exact return behavior used by
# generate_k10_candidates().
# ------------------------------------------------------------

name_first_choices_prod = defaultdict(dict)
name_last_choices_prod = defaultdict(dict)

addr_first_choices_prod = defaultdict(dict)
addr_last_choices_prod = defaultdict(dict)

exact_name_prod = defaultdict(list)
exact_addr_prod = defaultdict(list)

number_addr_choices_prod = defaultdict(dict)

# ------------------------------------------------------------
# Build indexes
# ------------------------------------------------------------

print(
    "\nBuilding production blocking dictionaries..."
)

t0 = time.time()

for row in prod_s1[
    [
        "local_idx",
        "name_n",
        "addr_n",
        "country_n"
    ]
].itertuples(index=False):

    idx = int(
        row.local_idx
    )

    country = row.country_n
    name = row.name_n
    addr = row.addr_n

    # -------------------------
    # NAME FIRST / LAST
    # -------------------------

    if name:

        name_first_choices_prod[
            (
                country,
                name[0]
            )
        ][idx] = name

        name_last_choices_prod[
            (
                country,
                name[-1]
            )
        ][idx] = name

        exact_name_prod[
            (
                country,
                name
            )
        ].append(idx)

    # -------------------------
    # ADDRESS FIRST / LAST
    # -------------------------

    if addr:

        addr_first_choices_prod[
            (
                country,
                addr[0]
            )
        ][idx] = addr

        addr_last_choices_prod[
            (
                country,
                addr[-1]
            )
        ][idx] = addr

        exact_addr_prod[
            (
                country,
                addr
            )
        ].append(idx)

        # ---------------------
        # NUMBER-AWARE ADDRESS
        # ---------------------

        nums = set(
            re.findall(
                r"\d+",
                addr
            )
        )

        for num in nums:

            number_addr_choices_prod[
                (
                    country,
                    num
                )
            ][idx] = addr

elapsed = time.time() - t0

print(
    "\nIndex construction time:",
    f"{elapsed / 60:.2f} min"
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "INDEX STATISTICS"
)
print(
    "=" * 100
)

print(
    "Name-first blocks:",
    len(name_first_choices_prod)
)

print(
    "Name-last blocks:",
    len(name_last_choices_prod)
)

print(
    "Address-first blocks:",
    len(addr_first_choices_prod)
)

print(
    "Address-last blocks:",
    len(addr_last_choices_prod)
)

print(
    "Exact-name blocks:",
    len(exact_name_prod)
)

print(
    "Exact-address blocks:",
    len(exact_addr_prod)
)

print(
    "Number-address blocks:",
    len(number_addr_choices_prod)
)

# ------------------------------------------------------------
# Sample blocks
# ------------------------------------------------------------

def inspect_block(
    dictionary,
    label
):

    if not dictionary:
        print(
            label,
            "EMPTY"
        )
        return

    key = next(
        iter(dictionary)
    )

    value = dictionary[key]

    print(
        f"\n{label}:"
    )

    print(
        "  key:",
        repr(key)
    )

    print(
        "  block size:",
        len(value)
    )

    sample_items = list(
        value.items()
    )[:3]

    print(
        "  sample:",
        sample_items
    )


inspect_block(
    name_first_choices_prod,
    "name_first"
)

inspect_block(
    name_last_choices_prod,
    "name_last"
)

inspect_block(
    addr_first_choices_prod,
    "addr_first"
)

inspect_block(
    addr_last_choices_prod,
    "addr_last"
)

inspect_block(
    number_addr_choices_prod,
    "number_address"
)

# ------------------------------------------------------------
# RAM check
# ------------------------------------------------------------

gc.collect()

print(
    "\nRAM after production indexes:",
    f"{ram_gb():.2f} GB"
)

print(
    "\nProduction S1 shape:",
    prod_s1.shape
)

print(
    "\nCELL 100 COMPLETE."
)

CELL 100 — BUILD PRODUCTION S1 RAPIDFUZZ INDEXES

RAM before: 4.53 GB
Releasing: test_s2
Releasing: test_s3
RAM after releasing S2/S3: 3.38 GB

TEST S1 rows: 1,732,544
Normalizing S1 fields...


TypeError: safe_token() missing 1 required positional argument: 'b'

In [ ]:
# CELL 100B — CORRECTED PRODUCTION S1 RAPIDFUZZ INDEX BUILD

import os
import gc
import re
import time
import inspect
import psutil
from collections import defaultdict

print("=" * 100)
print("CELL 100B — CORRECTED PRODUCTION S1 RAPIDFUZZ INDEX BUILD")
print("=" * 100)

# ------------------------------------------------------------
# Inspect safe_token signature first
# ------------------------------------------------------------

print(
    "\nsafe_token signature:",
    inspect.signature(safe_token)
)

# ------------------------------------------------------------
# Helper matching the existing notebook's 2-argument function
# ------------------------------------------------------------

def normalize_name_value(x):
    if pd.isna(x):
        x = ""
    return safe_token(
        str(x),
        ""
    )

def normalize_address_value(x):
    if pd.isna(x):
        x = ""
    return safe_token(
        str(x),
        ""
    )

# ------------------------------------------------------------
# Make sure S1 exists
# ------------------------------------------------------------

if "test_s1" not in globals():
    raise RuntimeError(
        "test_s1 is missing. Reload test_source1.tsv first."
    )

prod_s1 = test_s1.copy()

print(
    "\nTEST S1 rows:",
    f"{len(prod_s1):,}"
)

# ------------------------------------------------------------
# Normalize fields
# ------------------------------------------------------------

t0 = time.time()

prod_s1["name_n"] = (
    prod_s1["business_name"]
    .map(normalize_name_value)
)

prod_s1["addr_n"] = (
    prod_s1["business_address"]
    .map(normalize_address_value)
)

prod_s1["country_n"] = (
    prod_s1["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Normalization time:",
    f"{time.time() - t0:.2f} sec"
)

# ------------------------------------------------------------
# Local index
# ------------------------------------------------------------

prod_s1["local_idx"] = np.arange(
    len(prod_s1),
    dtype=np.int32
)

# ------------------------------------------------------------
# Production indexes
# ------------------------------------------------------------

name_first_choices_prod = defaultdict(dict)
name_last_choices_prod = defaultdict(dict)

addr_first_choices_prod = defaultdict(dict)
addr_last_choices_prod = defaultdict(dict)

exact_name_prod = defaultdict(list)
exact_addr_prod = defaultdict(list)

number_addr_choices_prod = defaultdict(dict)

print(
    "\nBuilding indexes..."
)

t0 = time.time()

for row in prod_s1[
    [
        "local_idx",
        "name_n",
        "addr_n",
        "country_n"
    ]
].itertuples(index=False):

    idx = int(row.local_idx)

    country = row.country_n
    name = row.name_n
    addr = row.addr_n

    # --------------------------------------------------------
    # Name
    # --------------------------------------------------------

    if name:

        name_first_choices_prod[
            (country, name[0])
        ][idx] = name

        name_last_choices_prod[
            (country, name[-1])
        ][idx] = name

        exact_name_prod[
            (country, name)
        ].append(idx)

    # --------------------------------------------------------
    # Address
    # --------------------------------------------------------

    if addr:

        addr_first_choices_prod[
            (country, addr[0])
        ][idx] = addr

        addr_last_choices_prod[
            (country, addr[-1])
        ][idx] = addr

        exact_addr_prod[
            (country, addr)
        ].append(idx)

        # Number-aware address index
        nums = set(
            re.findall(
                r"\d+",
                addr
            )
        )

        for num in nums:

            number_addr_choices_prod[
                (country, num)
            ][idx] = addr

print(
    "Index construction time:",
    f"{(time.time() - t0) / 60:.2f} min"
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "PRODUCTION INDEX STATISTICS"
)
print(
    "=" * 100
)

for label, obj in [
    ("name_first", name_first_choices_prod),
    ("name_last", name_last_choices_prod),
    ("addr_first", addr_first_choices_prod),
    ("addr_last", addr_last_choices_prod),
    ("exact_name", exact_name_prod),
    ("exact_addr", exact_addr_prod),
    ("number_addr", number_addr_choices_prod),
]:

    print(
        f"{label:20s}: {len(obj):,}"
    )

# ------------------------------------------------------------
# RAM
# ------------------------------------------------------------

process = psutil.Process()

print(
    "\nRAM:",
    f"{process.memory_info().rss / (1024**3):.2f} GB"
)

print(
    "\nProduction S1:",
    prod_s1.shape
)

print(
    "\nCELL 100B COMPLETE."
)

CELL 100B — CORRECTED PRODUCTION S1 RAPIDFUZZ INDEX BUILD

safe_token signature: (a, b)

TEST S1 rows: 1,732,544
Normalization time: 3.00 sec

Building indexes...
Index construction time: 0.04 min

PRODUCTION INDEX STATISTICS
name_first          : 0
name_last           : 0
addr_first          : 0
addr_last           : 0
exact_name          : 0
exact_addr          : 0
number_addr         : 0

RAM: 3.56 GB

Production S1: (1732544, 8)

CELL 100B COMPLETE.


In [ ]:
# CELL 101 — INSPECT EXISTING NORMALIZATION LOGIC

import inspect
import pandas as pd

print("=" * 100)
print("CELL 101 — INSPECT EXISTING NORMALIZATION LOGIC")
print("=" * 100)

# ------------------------------------------------------------
# Inspect safe_token
# ------------------------------------------------------------

print("\nSAFE_TOKEN:")
print(
    "signature:",
    inspect.signature(safe_token)
)

try:
    print(
        inspect.getsource(safe_token)
    )
except Exception as e:
    print(
        "Source unavailable:",
        repr(e)
    )

# ------------------------------------------------------------
# Inspect the actual normalized values already present in
# the training validation dataframe
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("EXISTING NORMALIZED VALUES")
print("=" * 100)

for name in [
    "s1",
    "targets",
    "prod_s1"
]:

    if name not in globals():
        continue

    obj = globals()[name]

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue

    print(
        f"\n{name}:"
    )

    for col in [
        "business_name",
        "business_address",
        "name_n",
        "addr_n",
        "country_n"
    ]:

        if col in obj.columns:

            print(
                f"\n  {col}:"
            )

            print(
                obj[col]
                .head(5)
                .tolist()
            )

# ------------------------------------------------------------
# Test safe_token on known values
# ------------------------------------------------------------

if "s1" in globals():

    print(
        "\n" + "=" * 100
    )
    print(
        "SAFE_TOKEN TEST"
    )
    print(
        "=" * 100
    )

    example_name = str(
        s1.iloc[0]["business_name"]
    )

    example_addr = str(
        s1.iloc[0]["business_address"]
    )

    print(
        "Original name:",
        repr(example_name)
    )

    print(
        "Original address:",
        repr(example_addr)
    )

    for second in [
        "",
        example_name,
        example_addr
    ]:

        try:

            result = safe_token(
                example_name,
                second
            )

            print(
                "\nsafe_token(name,",
                repr(second),
                ") =",
                repr(result)
            )

        except Exception as e:

            print(
                "\nsafe_token failed:",
                repr(e)
            )

print(
    "\nCELL 101 COMPLETE."
)

CELL 101 — INSPECT EXISTING NORMALIZATION LOGIC

SAFE_TOKEN:
signature: (a, b)
def safe_token(a, b):

    if not a or not b:
        return 0.0

    return fuzz.token_set_ratio(a, b) / 100.0


EXISTING NORMALIZED VALUES

s1:

  business_name:
["Orelee's Barbershop", 'Prime Money', 'B+ Retail Inc', 'Christ Chapel', 'Prabhav Business Center']

  business_address:
['1795 Westchester Drive, High Point, NC', '17560 Ellis Road, Tahlequah, OK', '1712 Montebello Avenue, Phoenix, AZ', '2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD', '797, Lake Town Block A, Kolkata, Howrah, West Bengal']

  name_n:
['oreleesbarbershop', 'primemoney', 'bretailinc', 'christchapel', 'prabhavbusinesscenter']

  addr_n:
['1795westchesterdrivehighpointnc', '17560ellisroadtahlequahok', '1712montebelloavenuephoenixaz', '2100camerondriveunitapartmentgdundalkmd', '797laketownblockakolkatahowrahwestbengal']

  country_n:
['us', 'us', 'us', 'us', 'india']

targets:

  business_name:
['BELL BUILDERS CENTER LTD', 'AGUIRR

In [ ]:
# CELL 102 — RECOVER / VERIFY EXACT TEXT NORMALIZER

import re
import unicodedata
import numpy as np

print("=" * 100)
print("CELL 102 — RECOVER / VERIFY EXACT TEXT NORMALIZER")
print("=" * 100)

# ------------------------------------------------------------
# Candidate normalizer
#
# The stored examples show:
#   "Orelee's Barbershop" -> "oreleesbarbershop"
#   "797, Lake Town Block A, Kolkata..." -> "797laketown..."
#   "Grand Ínstitutions Inc" -> "grandínstitutionsinc"
#
# Python str.isalnum() preserves Unicode letters/digits while
# removing punctuation and whitespace.
# ------------------------------------------------------------

def normalize_exact(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )


# ------------------------------------------------------------
# Verify against existing TRAIN S1 normalization
# ------------------------------------------------------------

def verify_normalization(
    df,
    raw_col,
    normalized_col,
    label
):

    expected = (
        df[normalized_col]
        .fillna("")
        .astype(str)
    )

    reconstructed = (
        df[raw_col]
        .map(normalize_exact)
    )

    mismatch = (
        expected.to_numpy()
        != reconstructed.to_numpy()
    )

    print(
        f"\n{label}"
    )

    print(
        "Rows:",
        len(df)
    )

    print(
        "Mismatches:",
        int(mismatch.sum())
    )

    print(
        "Exact match:",
        f"{(~mismatch).mean() * 100:.6f}%"
    )

    if mismatch.any():

        idxs = np.where(
            mismatch
        )[0][:10]

        print(
            "\nFirst mismatches:"
        )

        for i in idxs:

            print(
                "RAW:",
                repr(
                    df.iloc[i][raw_col]
                )
            )

            print(
                "STORED:",
                repr(
                    expected.iloc[i]
                )
            )

            print(
                "REBUILT:",
                repr(
                    reconstructed.iloc[i]
                )
            )

    return (
        not mismatch.any()
    )


# ------------------------------------------------------------
# TRAIN S1
# ------------------------------------------------------------

s1_name_ok = verify_normalization(
    s1,
    "business_name",
    "name_n",
    "TRAIN S1 NAME"
)

s1_addr_ok = verify_normalization(
    s1,
    "business_address",
    "addr_n",
    "TRAIN S1 ADDRESS"
)

# ------------------------------------------------------------
# TARGETS
# ------------------------------------------------------------

target_name_ok = verify_normalization(
    targets,
    "business_name",
    "name_n",
    "TARGET NAME"
)

target_addr_ok = verify_normalization(
    targets,
    "business_address",
    "addr_n",
    "TARGET ADDRESS"
)

# ------------------------------------------------------------
# Overall result
# ------------------------------------------------------------

all_ok = (
    s1_name_ok
    and
    s1_addr_ok
    and
    target_name_ok
    and
    target_addr_ok
)

print(
    "\n" + "=" * 100
)

print(
    "EXACT NORMALIZER VERIFIED:",
    all_ok
)

if all_ok:

    print(
        "\nWe can safely use normalize_exact() "
        "for the production S1 indexes."
    )

else:

    print(
        "\nDo NOT rebuild production indexes yet."
    )

print(
    "\nCELL 102 COMPLETE."
)

CELL 102 — RECOVER / VERIFY EXACT TEXT NORMALIZER

TRAIN S1 NAME
Rows: 20000
Mismatches: 0
Exact match: 100.000000%

TRAIN S1 ADDRESS
Rows: 20000
Mismatches: 0
Exact match: 100.000000%

TARGET NAME
Rows: 69478
Mismatches: 0
Exact match: 100.000000%

TARGET ADDRESS
Rows: 69478
Mismatches: 0
Exact match: 100.000000%

EXACT NORMALIZER VERIFIED: True

We can safely use normalize_exact() for the production S1 indexes.

CELL 102 COMPLETE.


In [ ]:
# CELL 103 — BUILD VERIFIED PRODUCTION S1 INDEXES

import gc
import re
import time
import psutil
import numpy as np
import pandas as pd

from collections import defaultdict

print("=" * 100)
print("CELL 103 — BUILD VERIFIED PRODUCTION S1 INDEXES")
print("=" * 100)

# ------------------------------------------------------------
# RAM helper
# ------------------------------------------------------------

process = psutil.Process()

def ram_gb():
    return (
        process.memory_info().rss
        / (1024 ** 3)
    )

print(
    "\nRAM at start:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Rebuild normalized S1 fields using verified normalizer
# ------------------------------------------------------------

if "prod_s1" not in globals():
    raise RuntimeError(
        "prod_s1 is missing."
    )

print(
    "\nRebuilding normalized S1 fields..."
)

t0 = time.time()

prod_s1["name_n"] = (
    prod_s1["business_name"]
    .map(normalize_exact)
)

prod_s1["addr_n"] = (
    prod_s1["business_address"]
    .map(normalize_exact)
)

prod_s1["country_n"] = (
    prod_s1["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

prod_s1["local_idx"] = np.arange(
    len(prod_s1),
    dtype=np.int32
)

print(
    "Normalization time:",
    f"{time.time() - t0:.2f} sec"
)

print(
    "RAM after normalization:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Remove duplicate raw test_s1 copy
# ------------------------------------------------------------

if "test_s1" in globals():

    del test_s1
    gc.collect()

    print(
        "RAM after releasing raw test_s1:",
        f"{ram_gb():.2f} GB"
    )

# ------------------------------------------------------------
# Initialize production indexes
# ------------------------------------------------------------

print(
    "\nInitializing indexes..."
)

name_first_choices_prod = defaultdict(dict)
name_last_choices_prod = defaultdict(dict)

addr_first_choices_prod = defaultdict(dict)
addr_last_choices_prod = defaultdict(dict)

exact_name_prod = defaultdict(list)
exact_addr_prod = defaultdict(list)

number_addr_choices_prod = defaultdict(dict)

# ------------------------------------------------------------
# Build indexes
# ------------------------------------------------------------

print(
    "\nBuilding indexes..."
)

t0 = time.time()

records = prod_s1[
    [
        "local_idx",
        "name_n",
        "addr_n",
        "country_n"
    ]
].itertuples(index=False)

for row_num, row in enumerate(
    records,
    start=1
):

    idx = int(
        row.local_idx
    )

    country = row.country_n
    name = row.name_n
    addr = row.addr_n

    # --------------------------------------------------------
    # NAME
    # --------------------------------------------------------

    if name:

        name_first_choices_prod[
            (
                country,
                name[0]
            )
        ][idx] = name

        name_last_choices_prod[
            (
                country,
                name[-1]
            )
        ][idx] = name

        exact_name_prod[
            (
                country,
                name
            )
        ].append(idx)

    # --------------------------------------------------------
    # ADDRESS
    # --------------------------------------------------------

    if addr:

        addr_first_choices_prod[
            (
                country,
                addr[0]
            )
        ][idx] = addr

        addr_last_choices_prod[
            (
                country,
                addr[-1]
            )
        ][idx] = addr

        exact_addr_prod[
            (
                country,
                addr
            )
        ].append(idx)

        # ----------------------------------------------------
        # NUMBER-AWARE ADDRESS
        # ----------------------------------------------------

        for num in set(
            re.findall(
                r"\d+",
                addr
            )
        ):

            number_addr_choices_prod[
                (
                    country,
                    num
                )
            ][idx] = addr

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if row_num % 250_000 == 0:

        elapsed = (
            time.time()
            - t0
        )

        rate = (
            row_num / elapsed
            if elapsed > 0
            else 0
        )

        remaining = (
            len(prod_s1) - row_num
        )

        eta = (
            remaining / rate
            if rate > 0
            else 0
        )

        print(
            f"{row_num:,}/{len(prod_s1):,} "
            f"| rate={rate:,.0f}/s "
            f"| ETA={eta/60:.1f} min "
            f"| RAM={ram_gb():.2f} GB"
        )

elapsed = (
    time.time()
    - t0
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "PRODUCTION INDEX STATISTICS"
)
print(
    "=" * 100
)

for label, obj in [
    (
        "name_first",
        name_first_choices_prod
    ),
    (
        "name_last",
        name_last_choices_prod
    ),
    (
        "addr_first",
        addr_first_choices_prod
    ),
    (
        "addr_last",
        addr_last_choices_prod
    ),
    (
        "exact_name",
        exact_name_prod
    ),
    (
        "exact_addr",
        exact_addr_prod
    ),
    (
        "number_addr",
        number_addr_choices_prod
    ),
]:

    print(
        f"{label:20s}: "
        f"{len(obj):,} blocks"
    )

print(
    "\nIndex construction time:",
    f"{elapsed/60:.2f} min"
)

print(
    "RAM after indexes:",
    f"{ram_gb():.2f} GB"
)

print(
    "\nProduction S1:",
    prod_s1.shape
)

print(
    "\nCELL 103 COMPLETE."
)

CELL 103 — BUILD VERIFIED PRODUCTION S1 INDEXES

RAM at start: 3.56 GB

Rebuilding normalized S1 fields...
Normalization time: 25.49 sec
RAM after normalization: 3.82 GB
RAM after releasing raw test_s1: 3.77 GB

Initializing indexes...

Building indexes...
250,000/1,732,544 | rate=116,827/s | ETA=0.2 min | RAM=3.81 GB
500,000/1,732,544 | rate=101,611/s | ETA=0.2 min | RAM=3.96 GB
750,000/1,732,544 | rate=95,339/s | ETA=0.2 min | RAM=4.12 GB
1,000,000/1,732,544 | rate=95,592/s | ETA=0.1 min | RAM=4.27 GB
1,250,000/1,732,544 | rate=95,431/s | ETA=0.1 min | RAM=4.38 GB
1,500,000/1,732,544 | rate=94,921/s | ETA=0.0 min | RAM=4.54 GB

PRODUCTION INDEX STATISTICS
name_first          : 109 blocks
name_last           : 107 blocks
addr_first          : 105 blocks
addr_last           : 113 blocks
exact_name          : 1,226,515 blocks
exact_addr          : 1,672,193 blocks
number_addr         : 176,968 blocks

Index construction time: 0.31 min
RAM after indexes: 4.68 GB

Production S1: (1732544,

In [ ]:
# CELL 104 — VERIFY combined_similarity FORMULA

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 104 — VERIFY combined_similarity FORMULA")
print("=" * 100)

d = df_enhanced.copy()

stored = (
    d["combined_similarity"]
    .astype(float)
    .to_numpy()
)

candidates = {}

# Common combinations
candidates["simple_average"] = (
    d["name_ratio"] + d["addr_ratio"]
) / 2.0

candidates["partial_average"] = (
    d["name_partial"] + d["addr_partial"]
) / 2.0

candidates["geometric_mean"] = np.sqrt(
    np.clip(
        d["name_ratio"] * d["addr_ratio"],
        0,
        None
    )
)

candidates["harmonic_mean"] = (
    2
    * d["name_ratio"]
    * d["addr_ratio"]
    /
    (
        d["name_ratio"]
        +
        d["addr_ratio"]
        +
        1e-12
    )
)

candidates["max_similarity"] = (
    d[["name_ratio", "addr_ratio"]]
    .max(axis=1)
)

candidates["min_similarity"] = (
    d[["name_ratio", "addr_ratio"]]
    .min(axis=1)
)

# Weighted averages
for w in [
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75
]:

    candidates[
        f"weighted_name_{w:.2f}"
    ] = (
        w * d["name_ratio"]
        +
        (1 - w) * d["addr_ratio"]
    )

# Partial/name/address combinations
for w in [
    0.25,
    0.33,
    0.40,
    0.50,
    0.60,
    0.67,
    0.75
]:

    candidates[
        f"partial_weighted_{w:.2f}"
    ] = (
        w * d["name_partial"]
        +
        (1 - w) * d["addr_partial"]
    )

# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

results = []

for name, series in candidates.items():

    values = (
        series
        .astype(float)
        .to_numpy()
    )

    diff = np.abs(
        stored - values
    )

    results.append({
        "formula": name,
        "mae": float(
            diff.mean()
        ),
        "max_error": float(
            diff.max()
        ),
        "exact_match_pct": float(
            (diff <= 1e-10).mean() * 100
        )
    })

results_df = (
    pd.DataFrame(results)
    .sort_values(
        [
            "exact_match_pct",
            "mae"
        ],
        ascending=[
            False,
            True
        ]
    )
)

print(
    "\n" + "=" * 100
)
print(
    "FORMULA COMPARISON"
)
print(
    "=" * 100
)

print(
    results_df.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Direct comparison to balanced_similarity
# ------------------------------------------------------------

diff_balanced = (
    d["combined_similarity"]
    -
    d["balanced_similarity"]
).abs()

print(
    "\n" + "=" * 100
)
print(
    "combined_similarity vs balanced_similarity"
)
print(
    "=" * 100
)

print(
    "MAE:",
    f"{diff_balanced.mean():.12f}"
)

print(
    "Max error:",
    f"{diff_balanced.max():.12f}"
)

print(
    "Exact match:",
    f"{(diff_balanced <= 1e-10).mean()*100:.6f}%"
)

# ------------------------------------------------------------
# Sample values
# ------------------------------------------------------------

print(
    "\nSample:"
)

print(
    d[
        [
            "name_ratio",
            "addr_ratio",
            "combined_similarity",
            "balanced_similarity"
        ]
    ]
    .head(10)
    .round(6)
    .to_string(index=False)
)

print(
    "\nCELL 104 COMPLETE."
)

CELL 104 — VERIFY combined_similarity FORMULA

FORMULA COMPARISON
              formula      mae  max_error  exact_match_pct
   weighted_name_0.45 0.000000   0.000000       100.000000
        harmonic_mean 0.050027   0.550000         0.297565
   weighted_name_0.40 0.011526   0.050000         0.246761
       simple_average 0.011526   0.050000         0.246761
   weighted_name_0.50 0.011526   0.050000         0.246761
   weighted_name_0.35 0.023053   0.100000         0.246761
   weighted_name_0.55 0.023053   0.100000         0.246761
   weighted_name_0.30 0.034579   0.150000         0.246761
   weighted_name_0.60 0.034579   0.150000         0.246761
       geometric_mean 0.035205   0.550000         0.246761
   weighted_name_0.25 0.046105   0.200000         0.246761
   weighted_name_0.65 0.046105   0.200000         0.246761
   weighted_name_0.70 0.057631   0.250000         0.246761
   weighted_name_0.75 0.069158   0.300000         0.246761
       min_similarity 0.115217   0.550000        

In [ ]:
# CELL 105 — INSPECT PRODUCTION EXACT-BLOCK SIZES

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 105 — PRODUCTION EXACT-BLOCK SIZE ANALYSIS")
print("=" * 100)

def block_stats(
    dictionary,
    name
):

    if not dictionary:

        print(
            f"\n{name}: EMPTY"
        )
        return

    sizes = np.fromiter(
        (
            len(v)
            for v in dictionary.values()
        ),
        dtype=np.int32
    )

    print(
        f"\n{name}"
    )

    print(
        "Blocks:",
        f"{len(sizes):,}"
    )

    print(
        "Mean:",
        f"{sizes.mean():.2f}"
    )

    print(
        "Median:",
        f"{np.median(sizes):.0f}"
    )

    print(
        "P90:",
        f"{np.percentile(sizes, 90):.0f}"
    )

    print(
        "P95:",
        f"{np.percentile(sizes, 95):.0f}"
    )

    print(
        "P99:",
        f"{np.percentile(sizes, 99):.0f}"
    )

    print(
        "P99.9:",
        f"{np.percentile(sizes, 99.9):.0f}"
    )

    print(
        "Max:",
        f"{sizes.max():,}"
    )

    # Show largest blocks
    largest_idx = np.argsort(
        sizes
    )[-10:][::-1]

    print(
        "\nLargest blocks:"
    )

    keys = list(
        dictionary.keys()
    )

    for pos in largest_idx:

        key = keys[pos]

        print(
            " ",
            repr(key),
            "->",
            len(dictionary[key])
        )


block_stats(
    exact_name_prod,
    "EXACT NAME"
)

block_stats(
    exact_addr_prod,
    "EXACT ADDRESS"
)

block_stats(
    number_addr_choices_prod,
    "NUMBER-ADDRESS"
)

# ------------------------------------------------------------
# Also inspect first/last blocking buckets
# ------------------------------------------------------------

block_stats(
    name_first_choices_prod,
    "NAME FIRST"
)

block_stats(
    addr_first_choices_prod,
    "ADDRESS FIRST"
)

print(
    "\nCELL 105 COMPLETE."
)

CELL 105 — PRODUCTION EXACT-BLOCK SIZE ANALYSIS

EXACT NAME
Blocks: 1,226,515
Mean: 1.41
Median: 1
P90: 2
P95: 2
P99: 9
P99.9: 38
Max: 205

Largest blocks:
  ('france', 'bordeauxclubsarl') -> 205
  ('france', 'nantesclubsarl') -> 157
  ('france', 'lilleclubsarl') -> 147
  ('us', 'urgentcaregroup') -> 123
  ('france', 'lilleclubsas') -> 122
  ('france', 'bordeauxclubsas') -> 118
  ('france', 'nantesclubsas') -> 117
  ('france', 'bordeauxamicalesarl') -> 116
  ('us', 'earnosethroatgroup') -> 112
  ('us', 'internalmedicinegroup') -> 111

EXACT ADDRESS
Blocks: 1,672,193
Mean: 1.04
Median: 1
P90: 1
P95: 1
P99: 2
P99.9: 5
Max: 101

Largest blocks:
  ('france', '12ruelydericlillehautsdefrance') -> 101
  ('france', '30ruedesmeunierslillehautsdefrance') -> 91
  ('france', '27ruejeanbartlillehautsdefrance') -> 86
  ('france', '27ruejeanbartmaisondesassociationslillehautsdefrance') -> 82
  ('france', '100ruedelilletourcoinghautsdefrance') -> 56
  ('france', '13rueduchaidesfarinesbordeauxnouvellea

In [ ]:
# CELL 106 — PRODUCTION CANDIDATE BENCHMARK

import os
import re
import time
import gc
import numpy as np
import pandas as pd

from rapidfuzz import process, fuzz

print("=" * 100)
print("CELL 106 — PRODUCTION CANDIDATE BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TEST_DIR = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
    "DATASET/test"
)

s2_path = os.path.join(
    TEST_DIR,
    "test_source2.tsv"
)

s3_path = os.path.join(
    TEST_DIR,
    "test_source3.tsv"
)

# ------------------------------------------------------------
# Load only a sample
# ------------------------------------------------------------

SAMPLE_PER_SOURCE = 10_000

print(
    f"\nLoading {SAMPLE_PER_SOURCE:,} S2 + "
    f"{SAMPLE_PER_SOURCE:,} S3 records..."
)

sample_s2 = pd.read_csv(
    s2_path,
    sep="\t",
    dtype=str,
    nrows=SAMPLE_PER_SOURCE
)

sample_s3 = pd.read_csv(
    s3_path,
    sep="\t",
    dtype=str,
    nrows=SAMPLE_PER_SOURCE
)

sample_targets = pd.concat(
    [
        sample_s2.assign(source="S2"),
        sample_s3.assign(source="S3")
    ],
    ignore_index=True
)

del sample_s2, sample_s3
gc.collect()

# ------------------------------------------------------------
# Exact normalization
# ------------------------------------------------------------

t0 = time.time()

sample_targets["name_n"] = (
    sample_targets["business_name"]
    .map(normalize_exact)
)

sample_targets["addr_n"] = (
    sample_targets["business_address"]
    .map(normalize_exact)
)

sample_targets["country_n"] = (
    sample_targets["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Normalization:",
    f"{time.time() - t0:.2f} sec"
)

# ------------------------------------------------------------
# Exact production K10 candidate generator
# ------------------------------------------------------------

def generate_production_k10(r):

    c = r.country_n

    candidate_ids = set()

    # ========================================================
    # NAME FIRST TOP-3
    # ========================================================

    if r.name_n:

        choices = name_first_choices_prod.get(
            (c, r.name_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        # ====================================================
        # NAME LAST TOP-3
        # ====================================================

        choices = name_last_choices_prod.get(
            (c, r.name_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.name_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # ADDRESS FIRST TOP-3
    # ========================================================

    if r.addr_n:

        choices = addr_first_choices_prod.get(
            (c, r.addr_n[0])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

        # ====================================================
        # ADDRESS LAST TOP-3
        # ====================================================

        choices = addr_last_choices_prod.get(
            (c, r.addr_n[-1])
        )

        if choices:

            for _, _, idx in process.extract(
                r.addr_n,
                choices,
                scorer=fuzz.ratio,
                limit=3
            ):
                candidate_ids.add(idx)

    # ========================================================
    # EXACT NAME
    # ========================================================

    for idx in exact_name_prod.get(
        (c, r.name_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # EXACT ADDRESS
    # ========================================================

    for idx in exact_addr_prod.get(
        (c, r.addr_n),
        []
    ):
        candidate_ids.add(idx)

    # ========================================================
    # NUMBER-AWARE ADDRESS TOP-10
    # ========================================================

    query_numbers = set(
        re.findall(
            r"\d+",
            r.addr_n
        )
    )

    number_candidates = {}

    for num in query_numbers:

        choices = number_addr_choices_prod.get(
            (c, num)
        )

        if not choices:
            continue

        results = process.extract(
            r.addr_n,
            choices,
            scorer=fuzz.ratio,
            limit=10
        )

        for _, score, idx in results:

            previous = number_candidates.get(
                idx,
                -1
            )

            if score > previous:
                number_candidates[idx] = score

    candidate_ids.update(
        number_candidates.keys()
    )

    return candidate_ids


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

print(
    "\nGenerating candidates..."
)

counts = []

max_candidate_examples = []

t0 = time.time()

for i, row in enumerate(
    sample_targets.itertuples(index=False),
    start=1
):

    candidate_set = (
        generate_production_k10(row)
    )

    n = len(candidate_set)

    counts.append(n)

    if n > 500:

        max_candidate_examples.append(
            (
                row.entity_id,
                row.source,
                n
            )
        )

    if i % 2_000 == 0:

        elapsed = (
            time.time() - t0
        )

        rate = (
            i / elapsed
        )

        print(
            f"{i:,}/20,000 "
            f"| {rate:,.1f} queries/s"
        )

elapsed = (
    time.time() - t0
)

counts = np.asarray(
    counts,
    dtype=np.int32
)

rate = (
    len(sample_targets)
    /
    elapsed
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "PRODUCTION CANDIDATE BENCHMARK"
)
print(
    "=" * 100
)

print(
    "Queries:",
    f"{len(counts):,}"
)

print(
    "Time:",
    f"{elapsed/60:.2f} min"
)

print(
    "Speed:",
    f"{rate:,.2f} queries/sec"
)

print(
    "Projected full 9,969,589 targets:",
    f"{9_969_589 / rate / 3600:.2f} hours"
)

print(
    "\nCandidate statistics:"
)

print(
    "Mean:",
    f"{counts.mean():.2f}"
)

print(
    "Median:",
    f"{np.median(counts):.0f}"
)

print(
    "P90:",
    f"{np.percentile(counts, 90):.0f}"
)

print(
    "P95:",
    f"{np.percentile(counts, 95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(counts, 99):.0f}"
)

print(
    "P99.9:",
    f"{np.percentile(counts, 99.9):.0f}"
)

print(
    "Maximum:",
    f"{counts.max():,}"
)

print(
    "Candidates >100:",
    int(
        (counts > 100).sum()
    )
)

print(
    "Candidates >500:",
    int(
        (counts > 500).sum()
    )
)

print(
    "Candidates >1000:",
    int(
        (counts > 1000).sum()
    )
)

if max_candidate_examples:

    print(
        "\nLarge candidate examples:"
    )

    for x in sorted(
        max_candidate_examples,
        key=lambda z: z[2],
        reverse=True
    )[:20]:

        print(
            " ",
            x
        )

print(
    "\nCELL 106 COMPLETE."
)

CELL 106 — PRODUCTION CANDIDATE BENCHMARK

Loading 10,000 S2 + 10,000 S3 records...
Normalization: 0.26 sec

Generating candidates...
2,000/20,000 | 6.9 queries/s
4,000/20,000 | 6.9 queries/s
6,000/20,000 | 7.0 queries/s
8,000/20,000 | 7.0 queries/s
10,000/20,000 | 7.0 queries/s
12,000/20,000 | 7.1 queries/s
14,000/20,000 | 7.2 queries/s
16,000/20,000 | 7.2 queries/s
18,000/20,000 | 7.3 queries/s
20,000/20,000 | 7.3 queries/s

PRODUCTION CANDIDATE BENCHMARK
Queries: 20,000
Time: 45.87 min
Speed: 7.27 queries/sec
Projected full 9,969,589 targets: 381.07 hours

Candidate statistics:
Mean: 18.99
Median: 19
P90: 29
P95: 32
P99: 48
P99.9: 79
Maximum: 168
Candidates >100: 8
Candidates >500: 0
Candidates >1000: 0

CELL 106 COMPLETE.


In [ ]:
# CELL 107 — INSPECT PRECOMPUTED TEST RETRIEVAL ARTIFACTS

import os
import numpy as np

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

MODELS = os.path.join(
    BASE,
    "MODELS"
)

print("=" * 100)
print("CELL 107 — PRECOMPUTED TEST RETRIEVAL ARTIFACTS")
print("=" * 100)

# ------------------------------------------------------------
# Discover S2/S3 retrieval artifacts
# ------------------------------------------------------------

files = sorted(
    os.listdir(MODELS)
)

retrieval_files = [
    f for f in files
    if (
        "assigned_s1" in f.lower()
        or "target_num" in f.lower()
        or "tfidf_score" in f.lower()
    )
]

print("\nRetrieval artifacts:")

for f in retrieval_files:

    path = os.path.join(
        MODELS,
        f
    )

    size_mb = (
        os.path.getsize(path)
        / (1024 ** 2)
    )

    print(
        f"{size_mb:10.2f} MB   {f}"
    )

# ------------------------------------------------------------
# Inspect known S2 artifacts
# ------------------------------------------------------------

expected = [
    "test_s2_assigned_s1.dat",
    "test_s2_target_num.dat",
    "test_s2_tfidf_score.dat",
    "test_s3_assigned_s1.dat",
    "test_s3_target_num.dat",
    "test_s3_tfidf_score.dat",
]

print(
    "\n" + "=" * 100
)
print(
    "DETAILED INSPECTION"
)
print(
    "=" * 100
)

for filename in expected:

    path = os.path.join(
        MODELS,
        filename
    )

    if not os.path.exists(path):

        print(
            f"\n{filename}: MISSING"
        )

        continue

    size = os.path.getsize(path)

    print(
        f"\n{filename}"
    )

    print(
        "  Size:",
        f"{size / (1024**2):.2f} MB"
    )

    # Try common dtypes and estimate element count
    for dtype in [
        np.float32,
        np.int32,
        np.int64,
        np.float64
    ]:

        itemsize = np.dtype(dtype).itemsize

        if size % itemsize == 0:

            print(
                f"  {dtype.__name__:8s}: "
                f"{size // itemsize:,} elements"
            )

    # Show first values using float32 and int32
    if size % 4 == 0:

        n = min(
            10,
            size // 4
        )

        mm_float = np.memmap(
            path,
            dtype=np.float32,
            mode="r",
            shape=(size // 4,)
        )

        mm_int = np.memmap(
            path,
            dtype=np.int32,
            mode="r",
            shape=(size // 4,)
        )

        print(
            "  First float32:",
            mm_float[:n].tolist()
        )

        print(
            "  First int32:",
            mm_int[:n].tolist()
        )

# ------------------------------------------------------------
# Expected target counts
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "EXPECTED TARGET COUNTS"
)
print(
    "=" * 100
)

print(
    "S2:",
    f"{4_887_273:,}"
)

print(
    "S3:",
    f"{5_082_316:,}"
)

print(
    "Total:",
    f"{9_969_589:,}"
)

print(
    "\nCELL 107 COMPLETE."
)

CELL 107 — PRECOMPUTED TEST RETRIEVAL ARTIFACTS

Retrieval artifacts:
     18.64 MB   test_s2_assigned_s1.dat
     18.64 MB   test_s2_target_num.dat
     18.64 MB   test_s2_tfidf_score.dat

DETAILED INSPECTION

test_s2_assigned_s1.dat
  Size: 18.64 MB
  float32 : 4,887,273 elements
  int32   : 4,887,273 elements
  First float32: [1.695524898983706e-39, 2.688180909037513e-40, 1.9112113573141175e-39, 8.380395400971352e-40, 4.7228242402832175e-40, 1.792817051361778e-39, 9.189911510827156e-40, 1.8052577791280537e-39, 1.6253660887703553e-40, 1.3013858838184576e-39]
  First int32: [1209967, 191835, 1363886, 598045, 337032, 1279397, 655814, 1288275, 115990, 928700]

test_s2_target_num.dat
  Size: 18.64 MB
  float32 : 4,887,273 elements
  int32   : 4,887,273 elements
  First float32: [4.7562864177914905e-32, 1.2798328688961331e-18, 2.849681343392384e-33, 1.0163860533904246e-35, 3.3217978057109576e-07, 2.903633912865189e-06, 1.0883482777879187e-30, 2.6657960561227156e-11, 1.7799794603053526e-25

In [ ]:
# CELL 108 — BUILD EXPRESS SUBMISSION 4
#
# IMPORTANT:
# This is an EXPRESS HYBRID LEADERBOARD PROBE.
# It is NOT the final full-production 0.973279 model submission.
#
# S2 = precomputed TF-IDF top-1 retrieval
# S3 = previous Submission 3 predictions
#
# The full ranking-aware production pipeline remains separate.

import os
import gc
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 108 — EXPRESS SUBMISSION 4")
print("=" * 100)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

MODELS = os.path.join(
    BASE,
    "MODELS"
)

OLD_SUB = os.path.join(
    BASE,
    "SUBMISSION_3"
)

NEW_SUB = os.path.join(
    BASE,
    "SUBMISSION_4_EXPRESS"
)

os.makedirs(
    NEW_SUB,
    exist_ok=True
)

old_candidates = os.path.join(
    OLD_SUB,
    "candidate_pairs.tsv"
)

old_matches = os.path.join(
    OLD_SUB,
    "matching_results.tsv"
)

s2_assigned_path = os.path.join(
    MODELS,
    "test_s2_assigned_s1.dat"
)

s2_target_num_path = os.path.join(
    MODELS,
    "test_s2_target_num.dat"
)

s2_score_path = os.path.join(
    MODELS,
    "test_s2_tfidf_score.dat"
)

test_s1_path = os.path.join(
    BASE,
    "DATASET",
    "test",
    "test_source1.tsv"
)

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

required = [
    old_candidates,
    old_matches,
    s2_assigned_path,
    s2_target_num_path,
    s2_score_path,
    test_s1_path
]

for path in required:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Missing required file:\n{path}"
        )

# ------------------------------------------------------------
# Load only TEST S1 IDs
# ------------------------------------------------------------

print("\nLoading S1 entity IDs...")

s1_ids = pd.read_csv(
    test_s1_path,
    sep="\t",
    usecols=["entity_id"],
    dtype=str
)["entity_id"].to_numpy()

n_s1 = len(
    s1_ids
)

print(
    "S1 entities:",
    f"{n_s1:,}"
)

# ------------------------------------------------------------
# Memory-map S2 retrieval artifacts
# ------------------------------------------------------------

print(
    "\nOpening S2 retrieval artifacts..."
)

s2_assigned = np.memmap(
    s2_assigned_path,
    dtype=np.int32,
    mode="r"
)

s2_target_num = np.memmap(
    s2_target_num_path,
    dtype=np.int32,
    mode="r"
)

s2_score = np.memmap(
    s2_score_path,
    dtype=np.float32,
    mode="r"
)

n_s2 = len(
    s2_assigned
)

if len(s2_target_num) != n_s2:
    raise RuntimeError(
        "S2 target_num length mismatch."
    )

if len(s2_score) != n_s2:
    raise RuntimeError(
        "S2 score length mismatch."
    )

print(
    "S2 retrieval records:",
    f"{n_s2:,}"
)

print(
    "Assigned min/max:",
    int(s2_assigned.min()),
    int(s2_assigned.max())
)

print(
    "Score min/max:",
    float(s2_score.min()),
    float(s2_score.max())
)

# ------------------------------------------------------------
# Express threshold
# ------------------------------------------------------------

TFIDF_THRESHOLD = 0.285

accepted = (
    (s2_score >= TFIDF_THRESHOLD)
    &
    (s2_assigned >= 0)
    &
    (s2_assigned < n_s1)
)

accepted_count = int(
    accepted.sum()
)

print(
    "\nS2 TF-IDF threshold:",
    TFIDF_THRESHOLD
)

print(
    "Accepted S2 targets:",
    f"{accepted_count:,}"
)

print(
    "Acceptance rate:",
    f"{accepted.mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# Build S2 target lists keyed by LOCAL S1 INDEX
# ------------------------------------------------------------
#
# Store target numbers as Python ints first.
# This avoids creating millions of S2-xxxx strings prematurely.

print(
    "\nBuilding S2 assignment map..."
)

s2_by_s1 = defaultdict(list)

accepted_indices = np.flatnonzero(
    accepted
)

for pos in accepted_indices:

    s1_idx = int(
        s2_assigned[pos]
    )

    target_num = int(
        s2_target_num[pos]
    )

    s2_by_s1[s1_idx].append(
        target_num
    )

print(
    "S1 entities receiving S2 assignments:",
    f"{len(s2_by_s1):,}"
)

# ------------------------------------------------------------
# Verify old candidate file ordering
# ------------------------------------------------------------

print(
    "\nChecking candidate file ordering..."
)

order_ok = True

with open(
    old_candidates,
    "r",
    encoding="utf-8"
) as f:

    header = f.readline().rstrip("\n")

    if header != (
        "source1_entity_id\tcandidate_entity_ids"
    ):

        raise RuntimeError(
            f"Unexpected candidate header: {header}"
        )

    for check_idx in range(
        min(100, n_s1)
    ):

        line = f.readline()

        if not line:
            order_ok = False
            break

        old_s1_id = line.split(
            "\t",
            1
        )[0]

        if old_s1_id != s1_ids[
            check_idx
        ]:

            order_ok = False

            print(
                "Ordering mismatch at:",
                check_idx
            )

            print(
                "Expected:",
                s1_ids[check_idx]
            )

            print(
                "Found:",
                old_s1_id
            )

            break

print(
    "Candidate ordering matches TEST S1:",
    order_ok
)

if not order_ok:

    raise RuntimeError(
        "Old candidate file is not S1-ordered. "
        "Stopping rather than risking corrupt submission."
    )

# ------------------------------------------------------------
# Build new candidate_pairs.tsv
# ------------------------------------------------------------

new_candidate_path = os.path.join(
    NEW_SUB,
    "candidate_pairs.tsv"
)

print(
    "\nWriting:",
    new_candidate_path
)

candidate_rows = 0
added_s2_candidates = 0

with open(
    old_candidates,
    "r",
    encoding="utf-8"
) as fin, open(
    new_candidate_path,
    "w",
    encoding="utf-8"
) as fout:

    header = fin.readline()

    fout.write(
        header
    )

    for row_idx, line in enumerate(
        fin
    ):

        line = line.rstrip("\n")

        if "\t" in line:

            s1_id, candidate_text = (
                line.split(
                    "\t",
                    1
                )
            )

        else:

            s1_id = line
            candidate_text = ""

        # New S2 targets for this S1
        new_ids = [
            f"S2-{num}"
            for num in s2_by_s1.get(
                row_idx,
                []
            )
        ]

        old_ids = [
            x
            for x in candidate_text.split(",")
            if x
        ]

        # Deduplicate while preserving order
        seen = set()
        merged = []

        for target_id in (
            old_ids + new_ids
        ):

            if target_id not in seen:

                seen.add(
                    target_id
                )

                merged.append(
                    target_id
                )

        added_s2_candidates += (
            len(new_ids)
        )

        candidate_rows += 1

        if merged:

            fout.write(
                s1_id
                + "\t"
                + ",".join(merged)
                + "\n"
            )

        else:

            fout.write(
                s1_id
                + "\t\n"
            )

print(
    "Candidate rows:",
    f"{candidate_rows:,}"
)

print(
    "S2 candidates added:",
    f"{added_s2_candidates:,}"
)

# ------------------------------------------------------------
# Build new matching_results.tsv
# ------------------------------------------------------------
#
# Preserve the old S3/old predictions and ADD accepted S2
# top-1 predictions.

new_matching_path = os.path.join(
    NEW_SUB,
    "matching_results.tsv"
)

print(
    "\nWriting:",
    new_matching_path
)

matching_rows = 0
added_s2_matches = 0
final_pair_count = 0
empty_count = 0

with open(
    old_matches,
    "r",
    encoding="utf-8"
) as fin, open(
    new_matching_path,
    "w",
    encoding="utf-8"
) as fout:

    header = fin.readline()

    if not header:
        raise RuntimeError(
            "Existing matching_results.tsv is empty."
        )

    fout.write(
        header
    )

    for row_idx, line in enumerate(
        fin
    ):

        line = line.rstrip("\n")

        if "\t" in line:

            s1_id, matched_text = (
                line.split(
                    "\t",
                    1
                )
            )

        else:

            s1_id = line
            matched_text = ""

        old_ids = [
            x
            for x in matched_text.split(",")
            if x
        ]

        new_ids = [
            f"S2-{num}"
            for num in s2_by_s1.get(
                row_idx,
                []
            )
        ]

        seen = set()
        merged = []

        for target_id in (
            old_ids + new_ids
        ):

            if target_id not in seen:

                seen.add(
                    target_id
                )

                merged.append(
                    target_id
                )

        if len(merged) > len(old_ids):

            added_s2_matches += (
                len(merged)
                -
                len(old_ids)
            )

        if merged:

            fout.write(
                s1_id
                + "\t"
                + ",".join(merged)
                + "\n"
            )

            final_pair_count += len(
                merged
            )

        else:

            fout.write(
                s1_id
                + "\t\n"
            )

            empty_count += 1

        matching_rows += 1

print(
    "Matching rows:",
    f"{matching_rows:,}"
)

print(
    "Additional S2 matches:",
    f"{added_s2_matches:,}"
)

print(
    "Total predicted pairs:",
    f"{final_pair_count:,}"
)

print(
    "Empty S1 rows:",
    f"{empty_count:,}"
)

# ------------------------------------------------------------
# Final sanity check
# ------------------------------------------------------------

if candidate_rows != n_s1:
    raise RuntimeError(
        "Candidate row count != S1 count."
    )

if matching_rows != n_s1:
    raise RuntimeError(
        "Matching row count != S1 count."
    )

# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------

zip_path = os.path.join(
    NEW_SUB,
    "submission_4_express.zip"
)

print(
    "\nCreating ZIP..."
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1
) as z:

    z.write(
        new_candidate_path,
        arcname="candidate_pairs.tsv"
    )

    z.write(
        new_matching_path,
        arcname="matching_results.tsv"
    )

print(
    "\n" + "=" * 100
)
print(
    "EXPRESS SUBMISSION READY"
)
print(
    "=" * 100
)

for path in [
    new_candidate_path,
    new_matching_path,
    zip_path
]:

    size_mb = (
        os.path.getsize(path)
        / (1024 ** 2)
    )

    print(
        f"{os.path.basename(path):30s}"
        f"{size_mb:10.2f} MB"
    )

print(
    "\nZIP:",
    zip_path
)

print(
    "\nCELL 108 COMPLETE."
)

# ------------------------------------------------------------
# Cleanup memmaps / large temporary objects
# ------------------------------------------------------------

del accepted_indices
del accepted
del s2_by_s1

gc.collect()

CELL 108 — EXPRESS SUBMISSION 4

Loading S1 entity IDs...
S1 entities: 1,732,544

Opening S2 retrieval artifacts...
S2 retrieval records: 4,887,273
Assigned min/max: -1 1732541
Score min/max: 0.0 1.0000003576278687

S2 TF-IDF threshold: 0.285
Accepted S2 targets: 13,990
Acceptance rate: 0.29%

Building S2 assignment map...
S1 entities receiving S2 assignments: 13,855

Checking candidate file ordering...
Candidate ordering matches TEST S1: True

Writing: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_4_EXPRESS/candidate_pairs.tsv
Candidate rows: 1,732,544
S2 candidates added: 13,990

Writing: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_4_EXPRESS/matching_results.tsv
Matching rows: 1,732,544
Additional S2 matches: 11,271
Total predicted pairs: 1,450,278
Empty S1 rows: 796,550

Creating ZIP...

EXPRESS SUBMISSION READY
candidate_pairs.tsv               177.34 MB
matching_results.tsv               39.88 MB
submission_4_express.zip          100.87 MB

ZI

0

In [ ]:
# CELL 109 — LOAD TEST S2 BLOCK INDEX

import os
import gc
import pickle
import psutil

print("=" * 100)
print("CELL 109 — LOAD TEST S2 BLOCK INDEX")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

INDEX_DIR = os.path.join(
    BASE,
    "MODELS",
    "blocking_indexes"
)

S2_INDEX_PATH = os.path.join(
    INDEX_DIR,
    "test_s2_block_indexes.pkl"
)

# ------------------------------------------------------------
# Free the failed RapidFuzz production structures
# ------------------------------------------------------------

for var_name in [
    "name_first_choices_prod",
    "name_last_choices_prod",
    "addr_first_choices_prod",
    "addr_last_choices_prod",
    "exact_name_prod",
    "exact_addr_prod",
    "number_addr_choices_prod",
    "prod_s1"
]:

    if var_name in globals():
        del globals()[var_name]

gc.collect()

process = psutil.Process()

print(
    "RAM before loading:",
    f"{process.memory_info().rss / (1024**3):.2f} GB"
)

# ------------------------------------------------------------
# Verify file
# ------------------------------------------------------------

if not os.path.exists(
    S2_INDEX_PATH
):
    raise FileNotFoundError(
        S2_INDEX_PATH
    )

print(
    "\nIndex:",
    S2_INDEX_PATH
)

print(
    "File size:",
    f"{os.path.getsize(S2_INDEX_PATH)/(1024**2):.2f} MB"
)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

print(
    "\nLoading S2 blocking index..."
)

with open(
    S2_INDEX_PATH,
    "rb"
) as f:

    test_s2_blocks = pickle.load(f)

print(
    "Loaded object type:",
    type(test_s2_blocks).__name__
)

print(
    "\nRAM after loading:",
    f"{process.memory_info().rss / (1024**3):.2f} GB"
)

# ------------------------------------------------------------
# Inspect structure
# ------------------------------------------------------------

if isinstance(
    test_s2_blocks,
    dict
):

    print(
        "\nTop-level keys:"
    )

    for key in test_s2_blocks.keys():

        value = test_s2_blocks[key]

        print(
            f"\n{key!r}"
        )

        print(
            "  type:",
            type(value).__name__
        )

        try:
            print(
                "  len:",
                len(value)
            )
        except Exception:
            pass

        # One example
        try:

            first_key = next(
                iter(value)
            )

            first_value = value[
                first_key
            ]

            print(
                "  sample key:",
                repr(first_key)
            )

            print(
                "  sample value:",
                repr(first_value)[:500]
            )

        except Exception as e:

            print(
                "  sample unavailable:",
                repr(e)
            )

else:

    print(
        "\nUnexpected top-level structure."
    )

print(
    "\nCELL 109 COMPLETE."
)

CELL 109 — LOAD TEST S2 BLOCK INDEX
RAM before loading: 4.70 GB

Index: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocking_indexes/test_s2_block_indexes.pkl
File size: 427.46 MB

Loading S2 blocking index...
Loaded object type: dict

RAM after loading: 6.24 GB

Top-level keys:

'name_index'
  type: dict
  len: 3714440
  sample key: 'brahmainfosoft'
  sample value: ['S2-192345572']

'prefix_index'
  type: dict
  len: 169019
  sample key: 'brah'
  sample value: ['S2-192345572', 'S2-789476436', 'S2-278585468', 'S2-154901012', 'S2-228284220', 'S2-603507241', 'S2-573928617', 'S2-85735365', 'S2-595712322', 'S2-120539959', 'S2-881568248', 'S2-439612694', 'S2-773799462', 'S2-911361832', 'S2-125857089', 'S2-240023765', 'S2-103944025', 'S2-108588994', 'S2-571187370', 'S2-838796623', 'S2-378937975', 'S2-205322727', 'S2-548593037', 'S2-361537458', 'S2-24968678', 'S2-554050936', 'S2-694247438', 'S2-375007109', 'S2-543327019', 'S2-811349051', 'S2-16448960', 'S2-38

'address_index'


In [ ]:
# CELL 110 — FAST DIRECT-BLOCKING BENCHMARK

import os
import re
import time
import gc
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 110 — FAST DIRECT-BLOCKING BENCHMARK")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TEST_S1_PATH = os.path.join(
    BASE,
    "DATASET",
    "test",
    "test_source1.tsv"
)

# ------------------------------------------------------------
# Load a representative 20k S1 sample
# ------------------------------------------------------------

N = 20_000

test_s1_sample = pd.read_csv(
    TEST_S1_PATH,
    sep="\t",
    dtype=str,
    nrows=N
)

test_s1_sample["name_n"] = (
    test_s1_sample["business_name"]
    .map(normalize_exact)
)

test_s1_sample["addr_n"] = (
    test_s1_sample["business_address"]
    .map(normalize_exact)
)

test_s1_sample["country_n"] = (
    test_s1_sample["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Sample S1:",
    f"{len(test_s1_sample):,}"
)

# ------------------------------------------------------------
# Direct blocker
# ------------------------------------------------------------

def direct_block_s2(r):

    c = r.country_n

    candidate_ids = set()

    # Exact name
    candidate_ids.update(
        test_s2_blocks[
            "name_index"
        ].get(
            r.name_n,
            []
        )
    )

    # Exact address
    candidate_ids.update(
        test_s2_blocks[
            "address_index"
        ].get(
            r.addr_n,
            []
        )
    )

    # 4-character name prefix
    if r.name_n:

        candidate_ids.update(
            test_s2_blocks[
                "prefix_index"
            ].get(
                r.name_n[:4],
                []
            )
        )

    # Number + country
    numbers = set(
        re.findall(
            r"\d+",
            r.addr_n
        )
    )

    num_index = test_s2_blocks[
        "num_country_index"
    ]

    for num in numbers:

        candidate_ids.update(
            num_index.get(
                f"{num}_{c}",
                []
            )
        )

    return candidate_ids


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

counts = []

t0 = time.time()

for i, row in enumerate(
    test_s1_sample.itertuples(index=False),
    start=1
):

    candidates = direct_block_s2(
        row
    )

    counts.append(
        len(candidates)
    )

    if i % 2000 == 0:

        elapsed = (
            time.time() - t0
        )

        rate = (
            i / elapsed
        )

        print(
            f"{i:,}/{N:,} "
            f"| {rate:,.0f} S1/sec"
        )

elapsed = (
    time.time() - t0
)

counts = np.asarray(
    counts,
    dtype=np.int32
)

rate = (
    N / elapsed
)

print(
    "\n" + "=" * 100
)
print(
    "DIRECT BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    "Time:",
    f"{elapsed:.2f} sec"
)

print(
    "Speed:",
    f"{rate:,.0f} S1/sec"
)

print(
    "Projected 1,732,544 S1:",
    f"{1_732_544 / rate / 60:.2f} min"
)

print(
    "\nCandidate statistics:"
)

print(
    "Mean:",
    f"{counts.mean():.2f}"
)

print(
    "Median:",
    f"{np.median(counts):.0f}"
)

print(
    "P90:",
    f"{np.percentile(counts, 90):.0f}"
)

print(
    "P95:",
    f"{np.percentile(counts, 95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(counts, 99):.0f}"
)

print(
    "P99.9:",
    f"{np.percentile(counts, 99.9):.0f}"
)

print(
    "Maximum:",
    f"{counts.max():,}"
)

print(
    "Zero candidates:",
    int(
        (counts == 0).sum()
    )
)

print(
    "Candidates <=100:",
    int(
        (counts <= 100).sum()
    )
)

print(
    "Candidates <=500:",
    int(
        (counts <= 500).sum()
    )
)

print(
    "Candidates >1000:",
    int(
        (counts > 1000).sum()
    )
)

print(
    "\nCELL 110 COMPLETE."
)

CELL 110 — FAST DIRECT-BLOCKING BENCHMARK
Sample S1: 20,000
2,000/20,000 | 430 S1/sec
4,000/20,000 | 457 S1/sec
6,000/20,000 | 551 S1/sec
8,000/20,000 | 608 S1/sec
10,000/20,000 | 658 S1/sec
12,000/20,000 | 685 S1/sec
14,000/20,000 | 702 S1/sec
16,000/20,000 | 731 S1/sec
18,000/20,000 | 754 S1/sec
20,000/20,000 | 772 S1/sec

DIRECT BLOCKER RESULTS
Time: 25.90 sec
Speed: 772 S1/sec
Projected 1,732,544 S1: 37.39 min

Candidate statistics:
Mean: 8354.89
Median: 5778
P90: 19954
P95: 25350
P99: 35566
P99.9: 50675
Maximum: 63,285
Zero candidates: 5
Candidates <=100: 802
Candidates <=500: 2516
Candidates >1000: 16150

CELL 110 COMPLETE.


In [ ]:
# CELL 111 — ANALYZE PRECOMPUTED S2 REVERSE-TFIDF RETRIEVAL

import os
import numpy as np
import pandas as pd

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

MODELS = os.path.join(
    BASE,
    "MODELS"
)

assigned_path = os.path.join(
    MODELS,
    "test_s2_assigned_s1.dat"
)

target_num_path = os.path.join(
    MODELS,
    "test_s2_target_num.dat"
)

score_path = os.path.join(
    MODELS,
    "test_s2_tfidf_score.dat"
)

print("=" * 100)
print("CELL 111 — PRECOMPUTED S2 RETRIEVAL ANALYSIS")
print("=" * 100)

assigned = np.memmap(
    assigned_path,
    dtype=np.int32,
    mode="r"
)

target_num = np.memmap(
    target_num_path,
    dtype=np.int32,
    mode="r"
)

score = np.memmap(
    score_path,
    dtype=np.float32,
    mode="r"
)

# ------------------------------------------------------------
# Basic counts
# ------------------------------------------------------------

assigned_mask = (
    assigned >= 0
)

print(
    "\nS2 targets:",
    f"{len(assigned):,}"
)

print(
    "Assigned targets:",
    f"{int(assigned_mask.sum()):,}"
)

print(
    "Unassigned targets:",
    f"{int((~assigned_mask).sum()):,}"
)

print(
    "Assignment coverage:",
    f"{assigned_mask.mean()*100:.4f}%"
)

# ------------------------------------------------------------
# Score distribution among assigned records
# ------------------------------------------------------------

assigned_scores = np.asarray(
    score[assigned_mask]
)

print(
    "\n" + "=" * 100
)
print(
    "TF-IDF SCORE DISTRIBUTION"
)
print(
    "=" * 100
)

for q in [
    0,
    1,
    5,
    10,
    25,
    50,
    75,
    90,
    95,
    97,
    98,
    99,
    99.5,
    99.9,
    100
]:

    print(
        f"P{q:5.1f}:",
        f"{np.percentile(assigned_scores, q):.6f}"
    )

# ------------------------------------------------------------
# Counts at useful thresholds
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "THRESHOLD COUNTS"
)
print(
    "=" * 100
)

for threshold in [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.285,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95
]:

    mask = (
        assigned_mask
        &
        (score >= threshold)
    )

    print(
        f">= {threshold:.3f}:",
        f"{int(mask.sum()):,}",
        f"({mask.mean()*100:.3f}%)"
    )

# ------------------------------------------------------------
# Unique S1 assignments
# ------------------------------------------------------------

assigned_s1_values = np.asarray(
    assigned[assigned_mask]
)

unique_s1 = np.unique(
    assigned_s1_values
)

print(
    "\n" + "=" * 100
)
print(
    "ASSIGNMENT DIVERSITY"
)
print(
    "=" * 100
)

print(
    "Unique assigned S1:",
    f"{len(unique_s1):,}"
)

print(
    "Max local S1 index:",
    int(assigned_s1_values.max())
)

# ------------------------------------------------------------
# Number of target assignments per S1
# ------------------------------------------------------------

counts_per_s1 = np.bincount(
    assigned_s1_values,
    minlength=1_732_544
)

nonzero_counts = counts_per_s1[
    counts_per_s1 > 0
]

print(
    "\nTargets per assigned S1:"
)

print(
    "Mean:",
    f"{nonzero_counts.mean():.2f}"
)

print(
    "Median:",
    f"{np.median(nonzero_counts):.0f}"
)

print(
    "P90:",
    f"{np.percentile(nonzero_counts,90):.0f}"
)

print(
    "P99:",
    f"{np.percentile(nonzero_counts,99):.0f}"
)

print(
    "Maximum:",
    f"{nonzero_counts.max():,}"
)

# ------------------------------------------------------------
# Highest-score examples
# ------------------------------------------------------------

top_k = 20

top_idx = np.argpartition(
    assigned_scores,
    -top_k
)[-top_k:]

top_idx = top_idx[
    np.argsort(
        assigned_scores[top_idx]
    )[::-1]
]

assigned_positions = np.flatnonzero(
    assigned_mask
)

print(
    "\n" + "=" * 100
)
print(
    "TOP TF-IDF ASSIGNMENTS"
)
print(
    "=" * 100
)

for pos in top_idx:

    original_idx = int(
        assigned_positions[pos]
    )

    print(
        f"S2-{int(target_num[original_idx])}"
        f" -> "
        f"S1-local-{int(assigned[original_idx])}"
        f" | score={float(score[original_idx]):.6f}"
    )

print(
    "\nCELL 111 COMPLETE."
)

CELL 111 — PRECOMPUTED S2 RETRIEVAL ANALYSIS

S2 targets: 4,887,273
Assigned targets: 14,000
Unassigned targets: 4,873,273
Assignment coverage: 0.2865%

TF-IDF SCORE DISTRIBUTION
P  0.0: 0.182567
P  1.0: 0.396012
P  5.0: 0.511574
P 10.0: 0.591654
P 25.0: 0.726723
P 50.0: 0.829911
P 75.0: 0.900634
P 90.0: 0.945954
P 95.0: 0.968650
P 97.0: 0.980357
P 98.0: 0.990222
P 99.0: 1.000000
P 99.5: 1.000000
P 99.9: 1.000000
P100.0: 1.000000

THRESHOLD COUNTS
>= 0.050: 14,000 (0.286%)
>= 0.100: 14,000 (0.286%)
>= 0.150: 14,000 (0.286%)
>= 0.200: 13,999 (0.286%)
>= 0.250: 13,996 (0.286%)
>= 0.285: 13,990 (0.286%)
>= 0.300: 13,981 (0.286%)
>= 0.350: 13,950 (0.285%)
>= 0.400: 13,848 (0.283%)
>= 0.450: 13,657 (0.279%)
>= 0.500: 13,382 (0.274%)
>= 0.550: 13,009 (0.266%)
>= 0.600: 12,503 (0.256%)
>= 0.650: 11,871 (0.243%)
>= 0.700: 11,053 (0.226%)
>= 0.750: 9,908 (0.203%)
>= 0.800: 8,284 (0.170%)
>= 0.850: 6,086 (0.125%)
>= 0.900: 3,542 (0.072%)
>= 0.950: 1,276 (0.026%)

ASSIGNMENT DIVERSITY
Unique assi

In [ ]:
# CELL 112 — COMPACT BLOCKER BENCHMARK
#
# Tests:
#   1. exact name
#   2. exact address
#   3. number + country
#   4. union of all three
#
# Uses the existing 2,000 matched-target validation sample.

import re
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 112 — COMPACT BLOCKER BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Required validation objects
# ------------------------------------------------------------

required = [
    "sample",
    "s1",
    "targets",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Build compact reverse indexes over the 20k S1 validation set
# ------------------------------------------------------------

exact_name_v = {}
exact_addr_v = {}
num_country_v = {}

for row in s1[
    [
        "entity_id",
        "name_n",
        "addr_n",
        "country_n"
    ]
].itertuples(index=False):

    sid = row.entity_id
    country = row.country_n

    if row.name_n:

        exact_name_v.setdefault(
            (country, row.name_n),
            []
        ).append(sid)

    if row.addr_n:

        exact_addr_v.setdefault(
            (country, row.addr_n),
            []
        ).append(sid)

    nums = set(
        re.findall(
            r"\d+",
            row.addr_n
        )
    )

    for num in nums:

        num_country_v.setdefault(
            (country, num),
            []
        ).append(sid)

print(
    "\nIndex sizes:"
)

print(
    "Exact name:",
    f"{len(exact_name_v):,}"
)

print(
    "Exact address:",
    f"{len(exact_addr_v):,}"
)

print(
    "Number/country:",
    f"{len(num_country_v):,}"
)

# ------------------------------------------------------------
# Use 2k validation target sample
# ------------------------------------------------------------

queries = sample.copy()

print(
    "\nValidation target queries:",
    f"{len(queries):,}"
)

# ------------------------------------------------------------
# Evaluate channels
# ------------------------------------------------------------

stats = []

for channel in [
    "exact_name",
    "exact_address",
    "number_country",
    "union"
]:

    covered_pairs = 0
    covered_entities = 0
    candidate_counts = []

    for row in queries.itertuples(index=False):

        country = row.country_n

        name_hits = set(
            exact_name_v.get(
                (
                    country,
                    row.name_n
                ),
                []
            )
        )

        addr_hits = set(
            exact_addr_v.get(
                (
                    country,
                    row.addr_n
                ),
                []
            )
        )

        num_hits = set()

        for num in set(
            re.findall(
                r"\d+",
                row.addr_n
            )
        ):

            num_hits.update(
                num_country_v.get(
                    (
                        country,
                        num
                    ),
                    []
                )
            )

        if channel == "exact_name":
            candidates = name_hits

        elif channel == "exact_address":
            candidates = addr_hits

        elif channel == "number_country":
            candidates = num_hits

        else:
            candidates = (
                name_hits
                |
                addr_hits
                |
                num_hits
            )

        candidate_counts.append(
            len(candidates)
        )

        true_s1 = target_to_s1.get(
            row.entity_id
        )

        if true_s1 is None:
            continue

        if true_s1 in candidates:

            covered_pairs += 1
            covered_entities += 1

    stats.append({
        "channel": channel,
        "pair_recall_pct": (
            covered_pairs
            /
            len(queries)
            * 100
        ),
        "query_coverage_pct": (
            covered_entities
            /
            len(queries)
            * 100
        ),
        "mean_candidates": np.mean(
            candidate_counts
        ),
        "median_candidates": np.median(
            candidate_counts
        ),
        "p95_candidates": np.percentile(
            candidate_counts,
            95
        ),
        "p99_candidates": np.percentile(
            candidate_counts,
            99
        ),
        "max_candidates": np.max(
            candidate_counts
        )
    })

results = pd.DataFrame(
    stats
)

print(
    "\n" + "=" * 100
)
print(
    "COMPACT BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    results.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.3f}",
            "query_coverage_pct":
                lambda x: f"{x:.3f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median_candidates":
                lambda x: f"{x:.0f}",
            "p95_candidates":
                lambda x: f"{x:.0f}",
            "p99_candidates":
                lambda x: f"{x:.0f}",
            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nCELL 112 COMPLETE."
)

CELL 112 — COMPACT BLOCKER BENCHMARK

Index sizes:
Exact name: 19,391
Exact address: 19,986
Number/country: 9,371

Validation target queries: 2,000

COMPACT BLOCKER RESULTS
       channel pair_recall_pct query_coverage_pct mean_candidates median_candidates p95_candidates p99_candidates max_candidates
    exact_name          24.950             24.950            0.27                 0              1              2              4
 exact_address           8.250              8.250            0.08                 0              1              1              1
number_country          71.350             71.350           54.97                 3            410            689           1153
         union          79.200             79.200           55.07                 3            410            689           1153

CELL 112 COMPLETE.


In [ ]:
# CELL 113 — REFINED NUMBER/ADDRESS BLOCKING BENCHMARK

import re
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 113 — REFINED NUMBER/ADDRESS BLOCKING BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Build refined indexes on the 20k S1 validation corpus
# ------------------------------------------------------------

num_country_first = {}
num_country_last = {}
num_country_firstlast = {}

for row in s1[
    [
        "entity_id",
        "addr_n",
        "country_n"
    ]
].itertuples(index=False):

    sid = row.entity_id
    country = row.country_n
    addr = row.addr_n

    if not addr:
        continue

    numbers = set(
        re.findall(
            r"\d+",
            addr
        )
    )

    first_char = addr[0]
    last_char = addr[-1]

    for num in numbers:

        # country + number + first address char
        key_first = (
            country,
            num,
            first_char
        )

        num_country_first.setdefault(
            key_first,
            []
        ).append(sid)

        # country + number + last address char
        key_last = (
            country,
            num,
            last_char
        )

        num_country_last.setdefault(
            key_last,
            []
        ).append(sid)

        # country + number + first + last
        key_firstlast = (
            country,
            num,
            first_char,
            last_char
        )

        num_country_firstlast.setdefault(
            key_firstlast,
            []
        ).append(sid)

print(
    "Refined index sizes:"
)

print(
    "number+country+first:",
    f"{len(num_country_first):,}"
)

print(
    "number+country+last:",
    f"{len(num_country_last):,}"
)

print(
    "number+country+first+last:",
    f"{len(num_country_firstlast):,}"
)

# ------------------------------------------------------------
# Evaluate blockers
# ------------------------------------------------------------

channels = [
    "num_first",
    "num_last",
    "num_firstlast",
    "num_first_last_union",
    "num_firstlast_plus_exact"
]

stats = []

for channel in channels:

    covered = 0
    candidate_counts = []

    for row in sample.itertuples(
        index=False
    ):

        country = row.country_n
        addr = row.addr_n

        if not addr:
            candidates = set()

        else:

            numbers = set(
                re.findall(
                    r"\d+",
                    addr
                )
            )

            candidates = set()

            for num in numbers:

                if channel == "num_first":

                    candidates.update(
                        num_country_first.get(
                            (
                                country,
                                num,
                                addr[0]
                            ),
                            []
                        )
                    )

                elif channel == "num_last":

                    candidates.update(
                        num_country_last.get(
                            (
                                country,
                                num,
                                addr[-1]
                            ),
                            []
                        )
                    )

                elif channel == "num_firstlast":

                    candidates.update(
                        num_country_firstlast.get(
                            (
                                country,
                                num,
                                addr[0],
                                addr[-1]
                            ),
                            []
                        )
                    )

                elif channel == "num_first_last_union":

                    candidates.update(
                        num_country_first.get(
                            (
                                country,
                                num,
                                addr[0]
                            ),
                            []
                        )
                    )

                    candidates.update(
                        num_country_last.get(
                            (
                                country,
                                num,
                                addr[-1]
                            ),
                            []
                        )
                    )

                elif channel == "num_firstlast_plus_exact":

                    candidates.update(
                        num_country_firstlast.get(
                            (
                                country,
                                num,
                                addr[0],
                                addr[-1]
                            ),
                            []
                        )
                    )

            # Exact name/address additions
            if channel == "num_firstlast_plus_exact":

                candidates.update(
                    exact_name_v.get(
                        (
                            country,
                            row.name_n
                        ),
                        []
                    )
                )

                candidates.update(
                    exact_addr_v.get(
                        (
                            country,
                            row.addr_n
                        ),
                        []
                    )
                )

        candidate_counts.append(
            len(candidates)
        )

        true_s1 = target_to_s1.get(
            row.entity_id
        )

        if (
            true_s1 is not None
            and
            true_s1 in candidates
        ):

            covered += 1

    stats.append({
        "channel": channel,
        "recall_pct": (
            covered
            /
            len(sample)
            * 100
        ),
        "mean_candidates": np.mean(
            candidate_counts
        ),
        "median_candidates": np.median(
            candidate_counts
        ),
        "p95_candidates": np.percentile(
            candidate_counts,
            95
        ),
        "p99_candidates": np.percentile(
            candidate_counts,
            99
        ),
        "max_candidates": np.max(
            candidate_counts
        )
    })

results_113 = pd.DataFrame(
    stats
)

print(
    "\n" + "=" * 100
)
print(
    "REFINED BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    results_113.to_string(
        index=False,
        formatters={
            "recall_pct":
                lambda x: f"{x:.3f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median_candidates":
                lambda x: f"{x:.0f}",
            "p95_candidates":
                lambda x: f"{x:.0f}",
            "p99_candidates":
                lambda x: f"{x:.0f}",
            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nCELL 113 COMPLETE."
)

CELL 113 — REFINED NUMBER/ADDRESS BLOCKING BENCHMARK
Refined index sizes:
number+country+first: 14,390
number+country+last: 16,062
number+country+first+last: 20,476

REFINED BLOCKER RESULTS
                 channel recall_pct mean_candidates median_candidates p95_candidates p99_candidates max_candidates
               num_first     48.000            5.38                 1             26             54             98
                num_last     29.900            7.14                 0             32            177            380
           num_firstlast     23.650            0.83                 0              4             12             41
    num_first_last_union     54.250           11.69                 1             53            188            403
num_firstlast_plus_exact     42.500            1.04                 0              4             12             41

CELL 113 COMPLETE.


In [ ]:
# CELL 114 — NAME-PREFIX × NUMBER-COUNTRY INTERSECTION

import os
import re
import pickle
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 114 — NAME-PREFIX × NUMBER-COUNTRY INTERSECTION")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

INDEX_DIR = os.path.join(
    BASE,
    "MODELS",
    "blocking_indexes"
)

S3_INDEX_PATH = os.path.join(
    INDEX_DIR,
    "test_s3_block_indexes.pkl"
)

# ------------------------------------------------------------
# Load S3 blocker index
# ------------------------------------------------------------

if "test_s3_blocks" not in globals():

    print(
        "Loading S3 blocking index..."
    )

    with open(
        S3_INDEX_PATH,
        "rb"
    ) as f:

        test_s3_blocks = pickle.load(f)

print(
    "S2 index keys:",
    list(test_s2_blocks.keys())
)

print(
    "S3 index keys:",
    list(test_s3_blocks.keys())
)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def get_block_candidates(
    target_row,
    blocks
):

    name = target_row.name_n
    addr = target_row.addr_n
    country = target_row.country_n

    prefix_idx = blocks["prefix_index"]
    num_idx = blocks["num_country_index"]
    name_idx = blocks["name_index"]
    addr_idx = blocks["address_index"]

    # Exact channels
    exact = set(
        name_idx.get(
            name,
            []
        )
    )

    exact.update(
        addr_idx.get(
            addr,
            []
        )
    )

    # Name prefix
    prefix_candidates = set()

    if name:

        prefix_candidates.update(
            prefix_idx.get(
                name[:4],
                []
            )
        )

    # Number-country
    number_candidates = set()

    for num in set(
        re.findall(
            r"\d+",
            addr
        )
    ):

        number_candidates.update(
            num_idx.get(
                f"{num}_{country}",
                []
            )
        )

    # Intersection
    intersection = (
        prefix_candidates
        &
        number_candidates
    )

    # Union of compact channels
    compact_union = (
        intersection
        |
        exact
    )

    return (
        exact,
        prefix_candidates,
        number_candidates,
        intersection,
        compact_union
    )


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

rows = []

for row in sample.itertuples(
    index=False
):

    blocks = (
        test_s2_blocks
        if row.source == "S2"
        else test_s3_blocks
    )

    (
        exact,
        prefix,
        number,
        intersection,
        compact_union
    ) = get_block_candidates(
        row,
        blocks
    )

    true_s1 = target_to_s1.get(
        row.entity_id
    )

    rows.append({
        "target_id":
            row.entity_id,

        "true_s1":
            true_s1,

        "exact":
            exact,

        "prefix":
            prefix,

        "number":
            number,

        "intersection":
            intersection,

        "compact_union":
            compact_union
    })

# ------------------------------------------------------------
# Channel metrics
# ------------------------------------------------------------

metrics = []

for channel in [
    "exact",
    "intersection",
    "compact_union"
]:

    covered = 0
    counts = []

    for row in rows:

        candidates = row[channel]

        counts.append(
            len(candidates)
        )

        if (
            row["true_s1"] is not None
            and
            row["true_s1"] in candidates
        ):
            covered += 1

    metrics.append({
        "channel":
            channel,

        "pair_recall_pct":
            covered / len(rows) * 100,

        "mean_candidates":
            np.mean(counts),

        "median_candidates":
            np.median(counts),

        "p95_candidates":
            np.percentile(
                counts,
                95
            ),

        "p99_candidates":
            np.percentile(
                counts,
                99
            ),

        "max_candidates":
            np.max(counts)
    })

results114 = pd.DataFrame(
    metrics
)

print(
    "\n" + "=" * 100
)
print(
    "RESULTS"
)
print(
    "=" * 100
)

print(
    results114.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.3f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median_candidates":
                lambda x: f"{x:.0f}",
            "p95_candidates":
                lambda x: f"{x:.0f}",
            "p99_candidates":
                lambda x: f"{x:.0f}",
            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

# ------------------------------------------------------------
# Additional combinations
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "ADDITIONAL INTERSECTION COMBINATIONS"
)
print(
    "=" * 100
)

combo_results = []

for mode in [
    "intersection",
    "intersection_plus_name_exact",
    "intersection_plus_address_exact",
    "intersection_plus_exact"
]:

    covered = 0
    counts = []

    for row in rows:

        candidates = set(
            row["intersection"]
        )

        if mode in [
            "intersection_plus_name_exact",
            "intersection_plus_exact"
        ]:
            candidates.update(
                test_s2_blocks[
                    "name_index"
                ].get(
                    sample.loc[
                        sample["entity_id"]
                        == row["target_id"],
                        "name_n"
                    ].iloc[0],
                    []
                )
                if sample.loc[
                    sample["entity_id"]
                    == row["target_id"]
                ].iloc[0]["source"] == "S2"
                else
                test_s3_blocks[
                    "name_index"
                ].get(
                    sample.loc[
                        sample["entity_id"]
                        == row["target_id"],
                        "name_n"
                    ].iloc[0],
                    []
                )
            )

        if mode in [
            "intersection_plus_address_exact",
            "intersection_plus_exact"
        ]:

            sample_row = sample[
                sample["entity_id"]
                == row["target_id"]
            ].iloc[0]

            blocks_for_row = (
                test_s2_blocks
                if sample_row["source"] == "S2"
                else test_s3_blocks
            )

            candidates.update(
                blocks_for_row[
                    "address_index"
                ].get(
                    sample_row["addr_n"],
                    []
                )
            )

        counts.append(
            len(candidates)
        )

        if (
            row["true_s1"] is not None
            and
            row["true_s1"] in candidates
        ):
            covered += 1

    combo_results.append({
        "mode": mode,
        "recall_pct":
            covered / len(rows) * 100,
        "mean_candidates":
            np.mean(counts),
        "median":
            np.median(counts),
        "p95":
            np.percentile(counts, 95),
        "p99":
            np.percentile(counts, 99),
        "max":
            np.max(counts)
    })

print(
    pd.DataFrame(
        combo_results
    ).to_string(
        index=False
    )
)

print(
    "\nCELL 114 COMPLETE."
)

CELL 114 — NAME-PREFIX × NUMBER-COUNTRY INTERSECTION
Loading S3 blocking index...
S2 index keys: ['name_index', 'prefix_index', 'address_index', 'num_country_index']
S3 index keys: ['name_index', 'prefix_index', 'address_index', 'num_country_index']

RESULTS
      channel pair_recall_pct mean_candidates median_candidates p95_candidates p99_candidates max_candidates
        exact           0.000            2.76                 0             15             57            311
 intersection           0.000            3.79                 0             14             78            484
compact_union           0.000            6.54                 0             31            109            484

ADDITIONAL INTERSECTION COMBINATIONS
                           mode  recall_pct  mean_candidates  median  p95   p99  max
                   intersection         0.0           3.7870     0.0 14.0  78.0  484
   intersection_plus_name_exact         0.0           6.4695     0.0 31.0 109.1  484
intersection

In [ ]:
# CELL 115 — NUMBER-AWARE ADDRESS ONLY RECALL BENCHMARK

import re
import time
import numpy as np
import pandas as pd

from rapidfuzz import process, fuzz

print("=" * 100)
print("CELL 115 — NUMBER-AWARE ADDRESS ONLY RECALL")
print("=" * 100)

# ------------------------------------------------------------
# Use the existing 20k TRAIN S1 validation corpus
# and the existing 2k target validation sample.
# ------------------------------------------------------------

print(
    "S1 validation rows:",
    len(s1)
)

print(
    "Target validation rows:",
    len(sample)
)

# ------------------------------------------------------------
# Generate candidates using ONLY:
#   1. exact name
#   2. exact address
#   3. number-aware address fuzzy Top-10
# ------------------------------------------------------------

covered = 0
candidate_counts = []

t0 = time.time()

for i, r in enumerate(
    sample.itertuples(index=False),
    start=1
):

    c = r.country_n

    candidates = set()

    # --------------------------------------------------------
    # Exact name
    # --------------------------------------------------------

    candidates.update(
        exact_name.get(
            (
                c,
                r.name_n
            ),
            []
        )
    )

    # --------------------------------------------------------
    # Exact address
    # --------------------------------------------------------

    candidates.update(
        exact_addr.get(
            (
                c,
                r.addr_n
            ),
            []
        )
    )

    # --------------------------------------------------------
    # Number-aware address Top-10
    # --------------------------------------------------------

    query_numbers = set(
        re.findall(
            r"\d+",
            r.addr_n
        )
    )

    number_candidates = {}

    for num in query_numbers:

        choices = number_addr_choices.get(
            (
                c,
                num
            )
        )

        if not choices:
            continue

        results = process.extract(
            r.addr_n,
            choices,
            scorer=fuzz.ratio,
            limit=10
        )

        for _, score, idx in results:

            previous = number_candidates.get(
                idx,
                -1
            )

            if score > previous:
                number_candidates[
                    idx
                ] = score

    candidates.update(
        number_candidates.keys()
    )

    candidate_counts.append(
        len(candidates)
    )

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    true_s1 = target_to_s1.get(
        r.entity_id
    )

    if (
        true_s1 is not None
        and
        true_s1 in candidates
    ):

        covered += 1

    if i % 500 == 0:

        elapsed = (
            time.time() - t0
        )

        print(
            f"{i:,}/"
            f"{len(sample):,} "
            f"| recall_so_far="
            f"{covered/i*100:.2f}%"
        )

elapsed = (
    time.time() - t0
)

candidate_counts = np.asarray(
    candidate_counts,
    dtype=np.int32
)

print(
    "\n" + "=" * 100
)
print(
    "NUMBER-AWARE RESULTS"
)
print(
    "=" * 100
)

print(
    "Pair recall:",
    f"{covered / len(sample) * 100:.4f}%"
)

print(
    "Mean candidates:",
    f"{candidate_counts.mean():.2f}"
)

print(
    "Median candidates:",
    f"{np.median(candidate_counts):.0f}"
)

print(
    "P95:",
    f"{np.percentile(candidate_counts,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(candidate_counts,99):.0f}"
)

print(
    "Maximum:",
    f"{candidate_counts.max():,}"
)

print(
    "Benchmark time:",
    f"{elapsed/60:.2f} min"
)

print(
    "Speed:",
    f"{len(sample)/elapsed:.2f} target/sec"
)

print(
    "\nCELL 115 COMPLETE."
)

CELL 115 — NUMBER-AWARE ADDRESS ONLY RECALL
S1 validation rows: 20000
Target validation rows: 2000
500/2,000 | recall_so_far=0.00%
1,000/2,000 | recall_so_far=0.00%
1,500/2,000 | recall_so_far=0.00%
2,000/2,000 | recall_so_far=0.00%

NUMBER-AWARE RESULTS
Pair recall: 0.0000%
Mean candidates: 5.61
Median candidates: 3
P95: 19
P99: 28
Maximum: 36
Benchmark time: 0.00 min
Speed: 14483.79 target/sec

CELL 115 COMPLETE.


In [ ]:
# CELL 116 — CORRECTED NUMBER-AWARE RECALL BENCHMARK

import re
import time
import numpy as np
import pandas as pd

from rapidfuzz import process, fuzz

print("=" * 100)
print("CELL 116 — CORRECTED NUMBER-AWARE RECALL BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Correct S1 ID -> local index mapping
# ------------------------------------------------------------

if "s1_id_to_idx_local" not in globals():

    s1_id_to_idx_local = {
        sid: i
        for i, sid in enumerate(
            s1["entity_id"]
        )
    }

print(
    "S1 local index map:",
    len(s1_id_to_idx_local)
)

covered = 0
candidate_counts = []

t0 = time.time()

for i, r in enumerate(
    sample.itertuples(index=False),
    start=1
):

    c = r.country_n

    candidates = set()

    # --------------------------------------------------------
    # Exact name
    # --------------------------------------------------------

    candidates.update(
        exact_name.get(
            (c, r.name_n),
            []
        )
    )

    # --------------------------------------------------------
    # Exact address
    # --------------------------------------------------------

    candidates.update(
        exact_addr.get(
            (c, r.addr_n),
            []
        )
    )

    # --------------------------------------------------------
    # Number-aware address Top-10
    # --------------------------------------------------------

    query_numbers = set(
        re.findall(
            r"\d+",
            r.addr_n
        )
    )

    number_candidates = {}

    for num in query_numbers:

        choices = number_addr_choices.get(
            (c, num)
        )

        if not choices:
            continue

        results = process.extract(
            r.addr_n,
            choices,
            scorer=fuzz.ratio,
            limit=10
        )

        for _, score, idx in results:

            previous = number_candidates.get(
                idx,
                -1
            )

            if score > previous:
                number_candidates[idx] = score

    candidates.update(
        number_candidates.keys()
    )

    candidate_counts.append(
        len(candidates)
    )

    # --------------------------------------------------------
    # CORRECT GT COMPARISON
    # --------------------------------------------------------

    true_s1 = target_to_s1.get(
        r.entity_id
    )

    true_idx = (
        s1_id_to_idx_local.get(
            true_s1
        )
        if true_s1 is not None
        else None
    )

    if (
        true_idx is not None
        and
        true_idx in candidates
    ):
        covered += 1

    if i % 500 == 0:

        elapsed = time.time() - t0

        print(
            f"{i:,}/{len(sample):,} "
            f"| recall="
            f"{covered / i * 100:.2f}%"
        )

elapsed = time.time() - t0

candidate_counts = np.asarray(
    candidate_counts,
    dtype=np.int32
)

print(
    "\n" + "=" * 100
)
print(
    "CORRECTED NUMBER-AWARE RESULTS"
)
print(
    "=" * 100
)

print(
    "Pair recall:",
    f"{covered / len(sample) * 100:.4f}%"
)

print(
    "Mean candidates:",
    f"{candidate_counts.mean():.2f}"
)

print(
    "Median:",
    f"{np.median(candidate_counts):.0f}"
)

print(
    "P95:",
    f"{np.percentile(candidate_counts,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(candidate_counts,99):.0f}"
)

print(
    "Maximum:",
    f"{candidate_counts.max():,}"
)

print(
    "Time:",
    f"{elapsed:.2f} sec"
)

print(
    "Speed:",
    f"{len(sample)/elapsed:.1f} targets/sec"
)

print(
    "\nCELL 116 COMPLETE."
)

CELL 116 — CORRECTED NUMBER-AWARE RECALL BENCHMARK
S1 local index map: 20000
500/2,000 | recall=80.60%
1,000/2,000 | recall=80.70%
1,500/2,000 | recall=79.47%
2,000/2,000 | recall=78.90%

CORRECTED NUMBER-AWARE RESULTS
Pair recall: 78.9000%
Mean candidates: 5.61
Median: 3
P95: 19
P99: 28
Maximum: 36
Time: 0.08 sec
Speed: 25517.1 targets/sec

CELL 116 COMPLETE.


In [ ]:
# CELL 117 — CORRECT FORWARD S1 -> S2/S3 BLOCKER BENCHMARK

import os
import re
import gc
import pickle
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 117 — CORRECT FORWARD BLOCKER BENCHMARK")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

INDEX_DIR = os.path.join(
    BASE,
    "MODELS",
    "blocking_indexes"
)

TRAIN_S2_INDEX = os.path.join(
    INDEX_DIR,
    "s2_block_indexes.pkl"
)

TRAIN_S3_INDEX = os.path.join(
    INDEX_DIR,
    "s3_block_indexes.pkl"
)

# ------------------------------------------------------------
# Load training indexes
# ------------------------------------------------------------

print("\nLoading S2 train index...")

with open(
    TRAIN_S2_INDEX,
    "rb"
) as f:
    train_s2_blocks = pickle.load(f)

print(
    "S2 loaded:",
    list(train_s2_blocks.keys())
)

print("\nLoading S3 train index...")

with open(
    TRAIN_S3_INDEX,
    "rb"
) as f:
    train_s3_blocks = pickle.load(f)

print(
    "S3 loaded:",
    list(train_s3_blocks.keys())
)

# ------------------------------------------------------------
# Ground truth map
# ------------------------------------------------------------

gt_map = (
    train_gt
    .set_index("source1_entity_id")
    ["matched_entity_ids"]
    .fillna("")
    .map(
        lambda x: set(
            y.strip()
            for y in str(x).split(",")
            if y.strip()
        )
    )
    .to_dict()
)

# ------------------------------------------------------------
# Blocker
# ------------------------------------------------------------

def forward_candidates(
    row,
    blocks
):

    name_index = blocks["name_index"]
    prefix_index = blocks["prefix_index"]
    address_index = blocks["address_index"]
    num_country_index = blocks["num_country_index"]

    name = row.name_n
    addr = row.addr_n
    country = row.country_n

    exact = set()

    # exact name
    exact.update(
        name_index.get(
            name,
            []
        )
    )

    # exact address
    exact.update(
        address_index.get(
            addr,
            []
        )
    )

    # number + country
    number_hits = set()

    for num in set(
        re.findall(
            r"\d+",
            addr
        )
    ):

        number_hits.update(
            num_country_index.get(
                f"{num}_{country}",
                []
            )
        )

    # 4-char name prefix
    prefix_hits = set()

    if name:

        prefix_hits.update(
            prefix_index.get(
                name[:4],
                []
            )
        )

    # intersection: name prefix AND number/country
    prefix_number = (
        prefix_hits & number_hits
    )

    return {
        "exact": exact,
        "number": number_hits,
        "prefix": prefix_hits,
        "prefix_number": prefix_number,
        "compact":
            exact
            |
            number_hits
            |
            prefix_number
    }


# ------------------------------------------------------------
# Evaluate several combinations
# ------------------------------------------------------------

channels = [
    "exact",
    "number",
    "exact_number",
    "prefix_number",
    "compact",
    "exact_prefix_number"
]

results = []

for channel in channels:

    covered_pairs = 0
    total_pairs = 0
    candidate_counts = []

    for row in s1.itertuples(
        index=False
    ):

        # Ground truth
        true_targets = gt_map.get(
            row.entity_id,
            set()
        )

        total_pairs += len(
            true_targets
        )

        # S2
        c2 = forward_candidates(
            row,
            train_s2_blocks
        )

        # S3
        c3 = forward_candidates(
            row,
            train_s3_blocks
        )

        def select(
            c
        ):

            if channel == "exact":
                return c["exact"]

            if channel == "number":
                return c["number"]

            if channel == "exact_number":
                return (
                    c["exact"]
                    |
                    c["number"]
                )

            if channel == "prefix_number":
                return c["prefix_number"]

            if channel == "compact":
                return c["compact"]

            if channel == "exact_prefix_number":
                return (
                    c["exact"]
                    |
                    c["prefix_number"]
                )

            return set()

        candidates = (
            {
                "S2-" + x
                if not str(x).startswith("S2-")
                else str(x)
                for x in select(c2)
            }
            |
            {
                "S3-" + x
                if not str(x).startswith("S3-")
                else str(x)
                for x in select(c3)
            }
        )

        # Count candidate IDs
        candidate_counts.append(
            len(candidates)
        )

        covered_pairs += len(
            true_targets
            &
            candidates
        )

    results.append({
        "channel": channel,
        "pair_recall_pct": (
            covered_pairs
            /
            total_pairs
            *
            100
        ),
        "mean_candidates":
            np.mean(candidate_counts),
        "median_candidates":
            np.median(candidate_counts),
        "p95_candidates":
            np.percentile(
                candidate_counts,
                95
            ),
        "p99_candidates":
            np.percentile(
                candidate_counts,
                99
            ),
        "max_candidates":
            np.max(candidate_counts)
    })

results_df_117 = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "FORWARD BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    results_df_117.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.3f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median_candidates":
                lambda x: f"{x:.0f}",
            "p95_candidates":
                lambda x: f"{x:.0f}",
            "p99_candidates":
                lambda x: f"{x:.0f}",
            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nTotal GT pairs:",
    f"{sum(len(v) for v in gt_map.values()):,}"
)

print(
    "\nCELL 117 COMPLETE."
)

CELL 117 — CORRECT FORWARD BLOCKER BENCHMARK

Loading S2 train index...


AttributeError: 'tuple' object has no attribute 'keys'

In [ ]:
# CELL 117B — CORRECTED FORWARD BLOCKER BENCHMARK

import os
import re
import pickle
import gc
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 117B — CORRECTED FORWARD BLOCKER BENCHMARK")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

INDEX_DIR = os.path.join(
    BASE,
    "MODELS",
    "blocking_indexes"
)

S2_PATH = os.path.join(
    INDEX_DIR,
    "s2_block_indexes.pkl"
)

S3_PATH = os.path.join(
    INDEX_DIR,
    "s3_block_indexes.pkl"
)

# ------------------------------------------------------------
# Loader that supports both tuple and dict formats
# ------------------------------------------------------------

def load_block_index(
    path,
    source_prefix
):

    print(
        f"\nLoading {source_prefix}:"
    )
    print(path)

    with open(
        path,
        "rb"
    ) as f:

        obj = pickle.load(f)

    print(
        "Raw type:",
        type(obj).__name__
    )

    # --------------------------------------------------------
    # Dict format
    # --------------------------------------------------------

    if isinstance(
        obj,
        dict
    ):

        required = [
            "name_index",
            "address_index",
            "prefix_index",
            "num_country_index"
        ]

        missing = [
            x for x in required
            if x not in obj
        ]

        if missing:

            raise RuntimeError(
                f"{source_prefix} dict missing keys: {missing}"
            )

        out = {
            "name_index":
                obj["name_index"],

            "address_index":
                obj["address_index"],

            "prefix_index":
                obj["prefix_index"],

            "num_country_index":
                obj["num_country_index"]
        }

    # --------------------------------------------------------
    # Tuple/list format
    # --------------------------------------------------------

    elif isinstance(
        obj,
        (tuple, list)
    ):

        if len(obj) < 4:

            raise RuntimeError(
                f"{source_prefix} tuple has only "
                f"{len(obj)} elements."
            )

        # Existing train artifacts use:
        #   0 = name
        #   1 = address
        #   2 = prefix
        #   3 = number+country

        out = {
            "name_index": obj[0],
            "address_index": obj[1],
            "prefix_index": obj[2],
            "num_country_index": obj[3]
        }

    else:

        raise RuntimeError(
            f"Unsupported {source_prefix} "
            f"index type: {type(obj)}"
        )

    print(
        "Normalized to dict."
    )

    for key, value in out.items():

        print(
            f"  {key:18s}: "
            f"{type(value).__name__:<12s}"
            f" len={len(value):,}"
        )

        try:

            first_key = next(
                iter(value)
            )

            first_value = value[
                first_key
            ]

            print(
                "    sample key:",
                repr(first_key)
            )

            print(
                "    sample value:",
                repr(first_value)[:180]
            )

        except Exception:
            pass

    return out


# ------------------------------------------------------------
# Load indexes
# ------------------------------------------------------------

train_s2_blocks = load_block_index(
    S2_PATH,
    "S2"
)

train_s3_blocks = load_block_index(
    S3_PATH,
    "S3"
)

# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

gt_map = {}

for row in train_gt.itertuples(
    index=False
):

    value = row.matched_entity_ids

    if pd.isna(value):

        gt_map[
            row.source1_entity_id
        ] = set()

    else:

        gt_map[
            row.source1_entity_id
        ] = set(
            x.strip()
            for x in str(value).split(",")
            if x.strip()
        )

# ------------------------------------------------------------
# Candidate ID normalization
# ------------------------------------------------------------

def normalize_target_id(
    value,
    source_prefix
):

    text = str(value)

    if text.startswith(
        "S2-"
    ) or text.startswith(
        "S3-"
    ):

        return text

    return (
        source_prefix
        + "-"
        + text
    )


# ------------------------------------------------------------
# Forward blocker
# ------------------------------------------------------------

def forward_candidates(
    row,
    blocks,
    source_prefix
):

    name_index = blocks[
        "name_index"
    ]

    address_index = blocks[
        "address_index"
    ]

    prefix_index = blocks[
        "prefix_index"
    ]

    num_country_index = blocks[
        "num_country_index"
    ]

    name = row.name_n
    addr = row.addr_n
    country = row.country_n

    # --------------------------------------------------------
    # Exact name
    # --------------------------------------------------------

    exact_name = {
        normalize_target_id(
            x,
            source_prefix
        )
        for x in name_index.get(
            name,
            []
        )
    }

    # --------------------------------------------------------
    # Exact address
    # --------------------------------------------------------

    exact_addr = {
        normalize_target_id(
            x,
            source_prefix
        )
        for x in address_index.get(
            addr,
            []
        )
    }

    exact = (
        exact_name
        |
        exact_addr
    )

    # --------------------------------------------------------
    # Number-country
    # --------------------------------------------------------

    number_candidates = set()

    for num in set(
        re.findall(
            r"\d+",
            addr
        )
    ):

        values = num_country_index.get(
            f"{num}_{country}",
            []
        )

        number_candidates.update(
            normalize_target_id(
                x,
                source_prefix
            )
            for x in values
        )

    # --------------------------------------------------------
    # Prefix
    # --------------------------------------------------------

    prefix_candidates = set()

    if name:

        values = prefix_index.get(
            name[:4],
            []
        )

        prefix_candidates.update(
            normalize_target_id(
                x,
                source_prefix
            )
            for x in values
        )

    # --------------------------------------------------------
    # Prefix ∩ number
    # --------------------------------------------------------

    prefix_number = (
        prefix_candidates
        &
        number_candidates
    )

    # --------------------------------------------------------
    # Compact union
    # --------------------------------------------------------

    compact = (
        exact
        |
        number_candidates
        |
        prefix_number
    )

    return {
        "exact": exact,
        "number": number_candidates,
        "prefix": prefix_candidates,
        "prefix_number": prefix_number,
        "compact": compact
    }


# ------------------------------------------------------------
# Evaluate full 20k representative S1 validation
# ------------------------------------------------------------

channels = [
    "exact",
    "number",
    "exact_number",
    "prefix_number",
    "compact",
    "exact_prefix_number"
]

results = []

for channel in channels:

    covered_pairs = 0
    total_true_pairs = 0

    candidate_counts = []

    for row in s1.itertuples(
        index=False
    ):

        true_targets = gt_map.get(
            row.entity_id,
            set()
        )

        total_true_pairs += len(
            true_targets
        )

        c2 = forward_candidates(
            row,
            train_s2_blocks,
            "S2"
        )

        c3 = forward_candidates(
            row,
            train_s3_blocks,
            "S3"
        )

        def select(
            c
        ):

            if channel == "exact":
                return c["exact"]

            if channel == "number":
                return c["number"]

            if channel == "exact_number":
                return (
                    c["exact"]
                    |
                    c["number"]
                )

            if channel == "prefix_number":
                return c["prefix_number"]

            if channel == "compact":
                return c["compact"]

            if channel == "exact_prefix_number":
                return (
                    c["exact"]
                    |
                    c["prefix_number"]
                )

            raise ValueError(
                channel
            )

        candidates = (
            select(c2)
            |
            select(c3)
        )

        candidate_counts.append(
            len(candidates)
        )

        covered_pairs += len(
            true_targets
            &
            candidates
        )

    results.append({
        "channel": channel,

        "pair_recall_pct":
            (
                covered_pairs
                /
                total_true_pairs
                *
                100
            ),

        "mean_candidates":
            np.mean(candidate_counts),

        "median_candidates":
            np.median(candidate_counts),

        "p95_candidates":
            np.percentile(
                candidate_counts,
                95
            ),

        "p99_candidates":
            np.percentile(
                candidate_counts,
                99
            ),

        "max_candidates":
            np.max(candidate_counts)
    })

results117 = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "CORRECT FORWARD BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    results117.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.3f}",

            "mean_candidates":
                lambda x: f"{x:.2f}",

            "median_candidates":
                lambda x: f"{x:.0f}",

            "p95_candidates":
                lambda x: f"{x:.0f}",

            "p99_candidates":
                lambda x: f"{x:.0f}",

            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nTotal true pairs:",
    f"{total_true_pairs:,}"
)

print(
    "\nCELL 117B COMPLETE."
)

CELL 117B — CORRECTED FORWARD BLOCKER BENCHMARK

Loading S2:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocking_indexes/s2_block_indexes.pkl
Raw type: tuple
Normalized to dict.
  name_index        : defaultdict  len=3,718,757
    sample key: 'hollowaypeakincseafood'
    sample value: {'S2-764573417'}
  address_index     : defaultdict  len=4,279,355
    sample key: 'khno57013newdelhiwestdelhidelhi'
    sample value: {'S2-166376419'}
  prefix_index      : defaultdict  len=154,566
    sample key: 'holl'
    sample value: {'S2-537102269', 'S2-650679815', 'S2-3305162', 'S2-743595427', 'S2-241867275', 'S2-927165559', 'S2-592589473', 'S2-662329967', 'S2-419657236', 'S2-633250972', 'S2-999242223', 'S2-6
  num_country_index : defaultdict  len=85,958
    sample key: 'india|570'
    sample value: {'S2-11481971', 'S2-199512237', 'S2-907745649', 'S2-564429840', 'S2-325864976', 'S2-357599621', 'S2-153722437', 'S2-997474535', 'S2-669697113', 'S2-520065071', 'S2-936554317', 'S2-

Loa

NameError: name 'train_gt' is not defined

In [ ]:
# CELL 117C — FINAL CORRECTED FORWARD BLOCKER BENCHMARK

import os
import re
import pickle
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 117C — FINAL CORRECTED FORWARD BLOCKER BENCHMARK")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

GT_PATH = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

# ------------------------------------------------------------
# Load ground truth if it is not currently in memory
# ------------------------------------------------------------

if "train_gt" not in globals():

    print("\nLoading train_ground_truth.tsv...")

    train_gt = pd.read_csv(
        GT_PATH,
        sep="\t",
        dtype=str
    )

else:

    print(
        "\nUsing existing train_gt dataframe."
    )

print(
    "Ground truth rows:",
    f"{len(train_gt):,}"
)

# ------------------------------------------------------------
# Build target -> true S1 map
# ------------------------------------------------------------

target_to_true_s1 = {}

for row in train_gt.itertuples(
    index=False
):

    matched = row.matched_entity_ids

    if pd.isna(matched):
        continue

    for target_id in str(
        matched
    ).split(","):

        target_id = target_id.strip()

        if target_id:

            target_to_true_s1[
                target_id
            ] = row.source1_entity_id

print(
    "Unique target IDs:",
    f"{len(target_to_true_s1):,}"
)

# ------------------------------------------------------------
# Candidate generator
# ------------------------------------------------------------

def forward_candidates_fixed(
    row,
    blocks
):

    name_index = blocks[
        "name_index"
    ]

    address_index = blocks[
        "address_index"
    ]

    prefix_index = blocks[
        "prefix_index"
    ]

    num_country_index = blocks[
        "num_country_index"
    ]

    name = row.name_n
    addr = row.addr_n
    country = row.country_n

    # ----------------------------------------
    # Exact name
    # ----------------------------------------

    exact_name = set(
        name_index.get(
            name,
            []
        )
    )

    # ----------------------------------------
    # Exact address
    # ----------------------------------------

    exact_address = set(
        address_index.get(
            addr,
            []
        )
    )

    exact = (
        exact_name
        |
        exact_address
    )

    # ----------------------------------------
    # Number + country
    #
    # IMPORTANT:
    # index key format is country|number
    # ----------------------------------------

    number_candidates = set()

    for num in set(
        re.findall(
            r"\d+",
            addr
        )
    ):

        number_candidates.update(
            num_country_index.get(
                f"{country}|{num}",
                []
            )
        )

    # ----------------------------------------
    # 4-char name prefix
    # ----------------------------------------

    prefix_candidates = set()

    if name:

        prefix_candidates.update(
            prefix_index.get(
                name[:4],
                []
            )
        )

    # ----------------------------------------
    # Prefix ∩ number
    # ----------------------------------------

    prefix_number = (
        prefix_candidates
        &
        number_candidates
    )

    # ----------------------------------------
    # Compact union
    # ----------------------------------------

    compact = (
        exact
        |
        number_candidates
        |
        prefix_number
    )

    # ----------------------------------------
    # Exact + prefix
    # ----------------------------------------

    exact_prefix = (
        exact
        |
        prefix_candidates
    )

    return {
        "exact": exact,
        "number": number_candidates,
        "prefix": prefix_candidates,
        "prefix_number": prefix_number,
        "compact": compact,
        "exact_prefix": exact_prefix,
        "exact_prefix_number":
            (
                exact
                |
                prefix_number
            )
    }


# ------------------------------------------------------------
# Evaluate all 20k representative S1 entities
# ------------------------------------------------------------

channels = [
    "exact",
    "number",
    "exact_number",
    "prefix_number",
    "compact",
    "exact_prefix",
    "exact_prefix_number"
]

results = []

total_true_pairs = 0

for channel in channels:

    covered_pairs = 0
    candidate_counts = []

    for row in s1.itertuples(
        index=False
    ):

        true_targets = gt_map.get(
            row.entity_id,
            set()
        ) if "gt_map" in globals() else None

        # Build true target set directly from train_gt if needed
        if true_targets is None:

            # We need source1 -> targets
            gt_row = train_gt.loc[
                train_gt[
                    "source1_entity_id"
                ]
                ==
                row.entity_id
            ]

            if len(gt_row):

                value = gt_row.iloc[0][
                    "matched_entity_ids"
                ]

                if pd.isna(value):

                    true_targets = set()

                else:

                    true_targets = set(
                        x.strip()
                        for x in str(value).split(",")
                        if x.strip()
                    )

            else:

                true_targets = set()

        if channel == "exact":
            candidate_sets = []

        # S2
        c2 = forward_candidates_fixed(
            row,
            train_s2_blocks
        )

        # S3
        c3 = forward_candidates_fixed(
            row,
            train_s3_blocks
        )

        if channel == "exact":

            candidates = (
                c2["exact"]
                |
                c3["exact"]
            )

        elif channel == "number":

            candidates = (
                c2["number"]
                |
                c3["number"]
            )

        elif channel == "exact_number":

            candidates = (
                c2["exact"]
                |
                c2["number"]
                |
                c3["exact"]
                |
                c3["number"]
            )

        elif channel == "prefix_number":

            candidates = (
                c2["prefix_number"]
                |
                c3["prefix_number"]
            )

        elif channel == "compact":

            candidates = (
                c2["compact"]
                |
                c3["compact"]
            )

        elif channel == "exact_prefix":

            candidates = (
                c2["exact_prefix"]
                |
                c3["exact_prefix"]
            )

        elif channel == "exact_prefix_number":

            candidates = (
                c2["exact_prefix_number"]
                |
                c3["exact_prefix_number"]
            )

        candidate_counts.append(
            len(candidates)
        )

        total_true_pairs += (
            len(true_targets)
            if channel == "exact"
            else 0
        )

        covered_pairs += len(
            true_targets
            &
            candidates
        )

    results.append({
        "channel":
            channel,

        "pair_recall_pct":
            (
                covered_pairs
                /
                total_true_pairs
                *
                100
            ),

        "mean_candidates":
            np.mean(candidate_counts),

        "median_candidates":
            np.median(candidate_counts),

        "p95_candidates":
            np.percentile(
                candidate_counts,
                95
            ),

        "p99_candidates":
            np.percentile(
                candidate_counts,
                99
            ),

        "max_candidates":
            np.max(candidate_counts)
    })

results117c = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "FINAL FORWARD BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    results117c.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.3f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median_candidates":
                lambda x: f"{x:.0f}",
            "p95_candidates":
                lambda x: f"{x:.0f}",
            "p99_candidates":
                lambda x: f"{x:.0f}",
            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nTotal representative true pairs:",
    f"{total_true_pairs:,}"
)

print(
    "\nCELL 117C COMPLETE."
)

CELL 117C — FINAL CORRECTED FORWARD BLOCKER BENCHMARK

Loading train_ground_truth.tsv...
Ground truth rows: 2,206,821
Unique target IDs: 7,638,365


NameError: name 's1' is not defined

In [ ]:
# CELL 117D — CORRECTED FORWARD BLOCKER BENCHMARK

import os
import re
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 117D — CORRECTED FORWARD BLOCKER BENCHMARK")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

S1_PATH = os.path.join(
    TRAIN_DIR,
    "train_source1.tsv"
)

GT_PATH = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

# ------------------------------------------------------------
# Recreate EXACT earlier 20k validation sample
# ------------------------------------------------------------

print("\nLoading S1...")

train_source1_eval = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str
)

s1 = (
    train_source1_eval
    .sample(
        n=20_000,
        random_state=42
    )
    .copy()
)

del train_source1_eval

# ------------------------------------------------------------
# Exact normalization
# ------------------------------------------------------------

s1["name_n"] = (
    s1["business_name"]
    .map(normalize_exact)
)

s1["addr_n"] = (
    s1["business_address"]
    .map(normalize_exact)
)

s1["country_n"] = (
    s1["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Validation S1:",
    len(s1)
)

# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

print("\nLoading ground truth...")

gt_df_117d = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

s1_to_targets = {}

for row in gt_df_117d.itertuples(
    index=False
):

    value = row.matched_entity_ids

    if pd.isna(value):
        s1_to_targets[
            row.source1_entity_id
        ] = set()

    else:
        s1_to_targets[
            row.source1_entity_id
        ] = set(
            x.strip()
            for x in str(value).split(",")
            if x.strip()
        )

# ------------------------------------------------------------
# Exact representative statistics
# ------------------------------------------------------------

true_counts = np.array([
    len(
        s1_to_targets.get(
            sid,
            set()
        )
    )
    for sid in s1["entity_id"]
])

print(
    "\nValidation distribution:"
)

print(
    "No-match:",
    int((true_counts == 0).sum())
)

print(
    "Matched:",
    int((true_counts > 0).sum())
)

print(
    "True pairs:",
    int(true_counts.sum())
)

# ------------------------------------------------------------
# Forward blocker
# ------------------------------------------------------------

def forward_candidates_117d(
    row,
    blocks
):

    name_index = blocks[
        "name_index"
    ]

    address_index = blocks[
        "address_index"
    ]

    prefix_index = blocks[
        "prefix_index"
    ]

    num_country_index = blocks[
        "num_country_index"
    ]

    name = row.name_n
    addr = row.addr_n
    country = row.country_n

    # Exact name
    exact_name = set(
        name_index.get(
            name,
            []
        )
    )

    # Exact address
    exact_address = set(
        address_index.get(
            addr,
            []
        )
    )

    exact = (
        exact_name
        |
        exact_address
    )

    # Number + country
    number_candidates = set()

    for num in set(
        re.findall(
            r"\d+",
            addr
        )
    ):

        number_candidates.update(
            num_country_index.get(
                f"{country}|{num}",
                []
            )
        )

    # 4-char name prefix
    prefix_candidates = set()

    if name:

        prefix_candidates.update(
            prefix_index.get(
                name[:4],
                []
            )
        )

    # Prefix ∩ number
    prefix_number = (
        prefix_candidates
        &
        number_candidates
    )

    return {
        "exact": exact,
        "number": number_candidates,
        "prefix": prefix_candidates,
        "prefix_number": prefix_number,
        "compact": (
            exact
            |
            number_candidates
            |
            prefix_number
        ),
        "exact_prefix": (
            exact
            |
            prefix_candidates
        ),
        "exact_prefix_number": (
            exact
            |
            prefix_number
        )
    }


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

channels = [
    "exact",
    "number",
    "exact_number",
    "prefix_number",
    "compact",
    "exact_prefix",
    "exact_prefix_number"
]

results = []

total_true_pairs = int(
    true_counts.sum()
)

for channel in channels:

    covered_pairs = 0
    candidate_counts = []

    for row in s1.itertuples(
        index=False
    ):

        true_targets = s1_to_targets.get(
            row.entity_id,
            set()
        )

        c2 = forward_candidates_117d(
            row,
            train_s2_blocks
        )

        c3 = forward_candidates_117d(
            row,
            train_s3_blocks
        )

        if channel == "exact":
            candidates = (
                c2["exact"]
                |
                c3["exact"]
            )

        elif channel == "number":
            candidates = (
                c2["number"]
                |
                c3["number"]
            )

        elif channel == "exact_number":
            candidates = (
                c2["exact"]
                |
                c2["number"]
                |
                c3["exact"]
                |
                c3["number"]
            )

        elif channel == "prefix_number":
            candidates = (
                c2["prefix_number"]
                |
                c3["prefix_number"]
            )

        elif channel == "compact":
            candidates = (
                c2["compact"]
                |
                c3["compact"]
            )

        elif channel == "exact_prefix":
            candidates = (
                c2["exact_prefix"]
                |
                c3["exact_prefix"]
            )

        elif channel == "exact_prefix_number":
            candidates = (
                c2["exact_prefix_number"]
                |
                c3["exact_prefix_number"]
            )

        candidate_counts.append(
            len(candidates)
        )

        covered_pairs += len(
            true_targets
            &
            candidates
        )

    results.append({
        "channel": channel,

        "pair_recall_pct":
            covered_pairs
            /
            total_true_pairs
            *
            100,

        "mean_candidates":
            np.mean(candidate_counts),

        "median_candidates":
            np.median(candidate_counts),

        "p95_candidates":
            np.percentile(
                candidate_counts,
                95
            ),

        "p99_candidates":
            np.percentile(
                candidate_counts,
                99
            ),

        "max_candidates":
            np.max(candidate_counts)
    })

results117d = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "FORWARD BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    results117d.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.3f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median_candidates":
                lambda x: f"{x:.0f}",
            "p95_candidates":
                lambda x: f"{x:.0f}",
            "p99_candidates":
                lambda x: f"{x:.0f}",
            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nTotal true pairs:",
    f"{total_true_pairs:,}"
)

print(
    "\nCELL 117D COMPLETE."
)

CELL 117D — CORRECTED FORWARD BLOCKER BENCHMARK

Loading S1...


NameError: name 'normalize_exact' is not defined

In [ ]:
# CELL 117E — ROBUST FORWARD BLOCKER BENCHMARK

import os
import re
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 117E — ROBUST FORWARD BLOCKER BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# EXACT VERIFIED NORMALIZER
# ------------------------------------------------------------

def normalize_exact(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )

# ------------------------------------------------------------
# Reload the exact 20k representative S1 validation sample
# ------------------------------------------------------------

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

S1_PATH = os.path.join(
    TRAIN_DIR,
    "train_source1.tsv"
)

GT_PATH = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

print("\nLoading S1...")

train_source1_eval = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str
)

s1_eval = (
    train_source1_eval
    .sample(
        n=20_000,
        random_state=42
    )
    .copy()
)

del train_source1_eval

# Normalize exactly as the original pipeline
s1_eval["name_n"] = (
    s1_eval["business_name"]
    .map(normalize_exact)
)

s1_eval["addr_n"] = (
    s1_eval["business_address"]
    .map(normalize_exact)
)

s1_eval["country_n"] = (
    s1_eval["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Representative S1:",
    f"{len(s1_eval):,}"
)

# ------------------------------------------------------------
# Load / build S1 -> true targets
# ------------------------------------------------------------

print("\nLoading ground truth...")

gt_eval = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

s1_to_targets_eval = {}

for row in gt_eval.itertuples(
    index=False
):

    value = row.matched_entity_ids

    if pd.isna(value):

        s1_to_targets_eval[
            row.source1_entity_id
        ] = set()

    else:

        s1_to_targets_eval[
            row.source1_entity_id
        ] = set(
            x.strip()
            for x in str(value).split(",")
            if x.strip()
        )

del gt_eval

true_counts = np.array([
    len(
        s1_to_targets_eval.get(
            sid,
            set()
        )
    )
    for sid in s1_eval["entity_id"]
])

print(
    "No-match S1:",
    int((true_counts == 0).sum())
)

print(
    "Matched S1:",
    int((true_counts > 0).sum())
)

print(
    "True pairs:",
    int(true_counts.sum())
)

# ------------------------------------------------------------
# Verify indexes exist
# ------------------------------------------------------------

for name in [
    "train_s2_blocks",
    "train_s3_blocks"
]:

    if name not in globals():

        raise RuntimeError(
            f"{name} is not loaded."
        )

# ------------------------------------------------------------
# Evaluate all channels in ONE pass
# ------------------------------------------------------------

channel_names = [
    "exact",
    "number",
    "exact_number",
    "prefix_number",
    "compact",
    "exact_prefix",
    "exact_prefix_number"
]

covered = {
    name: 0
    for name in channel_names
}

candidate_lists = {
    name: []
    for name in channel_names
}

total_true_pairs = int(
    true_counts.sum()
)

print(
    "\nEvaluating blockers..."
)

for i, row in enumerate(
    s1_eval.itertuples(
        index=False
    ),
    start=1
):

    true_targets = s1_to_targets_eval.get(
        row.entity_id,
        set()
    )

    def get_channels(
        blocks
    ):

        name_index = blocks[
            "name_index"
        ]

        address_index = blocks[
            "address_index"
        ]

        prefix_index = blocks[
            "prefix_index"
        ]

        num_country_index = blocks[
            "num_country_index"
        ]

        name = row.name_n
        addr = row.addr_n
        country = row.country_n

        # Exact name
        exact_name = set(
            name_index.get(
                name,
                []
            )
        )

        # Exact address
        exact_addr = set(
            address_index.get(
                addr,
                []
            )
        )

        exact = (
            exact_name
            |
            exact_addr
        )

        # Number + country
        number = set()

        for num in set(
            re.findall(
                r"\d+",
                addr
            )
        ):

            number.update(
                num_country_index.get(
                    f"{country}|{num}",
                    []
                )
            )

        # Name prefix
        prefix = set()

        if name:

            prefix.update(
                prefix_index.get(
                    name[:4],
                    []
                )
            )

        # Prefix ∩ number
        prefix_number = (
            prefix
            &
            number
        )

        compact = (
            exact
            |
            number
            |
            prefix_number
        )

        exact_prefix = (
            exact
            |
            prefix
        )

        exact_prefix_number = (
            exact
            |
            prefix_number
        )

        return {
            "exact": exact,
            "number": number,
            "prefix_number": prefix_number,
            "compact": compact,
            "exact_prefix": exact_prefix,
            "exact_prefix_number":
                exact_prefix_number
        }

    c2 = get_channels(
        train_s2_blocks
    )

    c3 = get_channels(
        train_s3_blocks
    )

    # Merge S2/S3
    merged = {}

    for channel in [
        "exact",
        "number",
        "prefix_number",
        "compact",
        "exact_prefix",
        "exact_prefix_number"
    ]:

        merged[channel] = (
            c2[channel]
            |
            c3[channel]
        )

    merged["exact_number"] = (
        c2["exact"]
        |
        c2["number"]
        |
        c3["exact"]
        |
        c3["number"]
    )

    # --------------------------------------------------------
    # Score each blocker
    # --------------------------------------------------------

    for channel in channel_names:

        candidates = merged[
            channel
        ]

        candidate_lists[
            channel
        ].append(
            len(candidates)
        )

        covered[channel] += len(
            true_targets
            &
            candidates
        )

    if i % 2_000 == 0:

        print(
            f"{i:,}/20,000"
        )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

rows = []

for channel in channel_names:

    counts = np.asarray(
        candidate_lists[channel],
        dtype=np.int32
    )

    rows.append({
        "channel":
            channel,

        "pair_recall_pct":
            (
                covered[channel]
                /
                total_true_pairs
                *
                100
            ),

        "mean_candidates":
            float(counts.mean()),

        "median_candidates":
            float(np.median(counts)),

        "p95_candidates":
            float(np.percentile(
                counts,
                95
            )),

        "p99_candidates":
            float(np.percentile(
                counts,
                99
            )),

        "max_candidates":
            int(counts.max())
    })

results117e = pd.DataFrame(
    rows
)

print(
    "\n" + "=" * 100
)
print(
    "TRUSTED FORWARD BLOCKER RESULTS"
)
print(
    "=" * 100
)

print(
    results117e.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.3f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median_candidates":
                lambda x: f"{x:.0f}",
            "p95_candidates":
                lambda x: f"{x:.0f}",
            "p99_candidates":
                lambda x: f"{x:.0f}",
            "max_candidates":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nTotal true pairs:",
    f"{total_true_pairs:,}"
)

print(
    "\nCELL 117E COMPLETE."
)

CELL 117E — ROBUST FORWARD BLOCKER BENCHMARK

Loading S1...
Representative S1: 20,000

Loading ground truth...
No-match S1: 1131
Matched S1: 18869
True pairs: 69231

Evaluating blockers...
2,000/20,000
4,000/20,000
6,000/20,000
8,000/20,000
10,000/20,000
12,000/20,000
14,000/20,000
16,000/20,000
18,000/20,000
20,000/20,000

TRUSTED FORWARD BLOCKER RESULTS
            channel pair_recall_pct mean_candidates median_candidates p95_candidates p99_candidates max_candidates
              exact          32.095           11.82                 2             72            201            511
             number          65.084        64547.72              2654         480209         696136        1276280
       exact_number          75.356        64558.66              2664         480209         696138        1276282
      prefix_number          49.565           82.09                 3            432           1595           7858
            compact          75.356        64558.66              26

In [ ]:
# CELL 118 — BLOCKED SPARSE TF-IDF RETRIEVAL BENCHMARK

import os
import re
import time
import gc

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

print("=" * 100)
print("CELL 118 — BLOCKED SPARSE TF-IDF RETRIEVAL BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Use the existing 20k S1 validation corpus
# ------------------------------------------------------------

VAL_S1 = s1.copy()

# Combined text exactly from normalized name + address
VAL_S1["combined_text"] = (
    VAL_S1["name_n"]
    + " "
    + VAL_S1["addr_n"]
)

VAL_TARGETS = sample.copy()

VAL_TARGETS["combined_text"] = (
    VAL_TARGETS["name_n"]
    + " "
    + VAL_TARGETS["addr_n"]
)

# ------------------------------------------------------------
# Fit compact character TF-IDF
# ------------------------------------------------------------

print("\nBuilding validation TF-IDF...")

vectorizer_118 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
)

t0 = time.time()

X_s1 = vectorizer_118.fit_transform(
    VAL_S1["combined_text"]
)

X_target = vectorizer_118.transform(
    VAL_TARGETS["combined_text"]
)

print(
    "TF-IDF build time:",
    f"{time.time() - t0:.2f} sec"
)

print(
    "S1 matrix:",
    X_s1.shape,
    "NNZ:",
    f"{X_s1.nnz:,}"
)

print(
    "Target matrix:",
    X_target.shape,
    "NNZ:",
    f"{X_target.nnz:,}"
)

# ------------------------------------------------------------
# Build four cheap partitions over S1
# ------------------------------------------------------------

def block_key_sets(df):

    keys = {
        "name_first": {},
        "name_last": {},
        "addr_first": {},
        "addr_last": {}
    }

    for idx, row in enumerate(
        df.itertuples(index=False)
    ):

        country = row.country_n

        if row.name_n:

            keys["name_first"].setdefault(
                (
                    country,
                    row.name_n[0]
                ),
                []
            ).append(idx)

            keys["name_last"].setdefault(
                (
                    country,
                    row.name_n[-1]
                ),
                []
            ).append(idx)

        if row.addr_n:

            keys["addr_first"].setdefault(
                (
                    country,
                    row.addr_n[0]
                ),
                []
            ).append(idx)

            keys["addr_last"].setdefault(
                (
                    country,
                    row.addr_n[-1]
                ),
                []
            ).append(idx)

    return keys


print(
    "\nBuilding S1 partitions..."
)

block_indexes = block_key_sets(
    VAL_S1
)

for name, idx in block_indexes.items():

    sizes = np.array(
        [
            len(v)
            for v in idx.values()
        ],
        dtype=np.int32
    )

    print(
        f"{name:12s}: "
        f"blocks={len(sizes):,} "
        f"median={np.median(sizes):.0f} "
        f"p95={np.percentile(sizes,95):.0f} "
        f"max={sizes.max():,}"
    )

# ------------------------------------------------------------
# Reverse target -> S1 retrieval
# ------------------------------------------------------------

# target_to_s1 exists in the notebook
# and maps target IDs -> true S1 IDs.

s1_id_to_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Retrieve Top-K inside the relevant blocks
# ------------------------------------------------------------

K = 10

covered = 0
complete = 0

query_count = len(
    VAL_TARGETS
)

t0 = time.time()

for q_idx, row in enumerate(
    VAL_TARGETS.itertuples(index=False)
):

    country = row.country_n

    relevant_blocks = set()

    if row.name_n:

        relevant_blocks.add(
            (
                "name_first",
                (
                    country,
                    row.name_n[0]
                )
            )
        )

        relevant_blocks.add(
            (
                "name_last",
                (
                    country,
                    row.name_n[-1]
                )
            )

    if row.addr_n:

        relevant_blocks.add(
            (
                "addr_first",
                (
                    country,
                    row.addr_n[0]
                )
            )

        relevant_blocks.add(
            (
                "addr_last",
                (
                    country,
                    row.addr_n[-1]
                )
            )

    candidate_local = set()

    for block_type, key in relevant_blocks:

        indices = block_indexes[
            block_type
        ].get(
            key,
            []
        )

        if not indices:
            continue

        candidate_local.update(
            indices
        )

    if not candidate_local:
        continue

    candidate_local = np.asarray(
        list(candidate_local),
        dtype=np.int32
    )

    X_block = X_s1[
        candidate_local
    ]

    q = X_target[
        q_idx:q_idx + 1
    ]

    scores = (
        q
        @
        X_block.T
    )

    scores = scores.toarray().ravel()

    if len(scores) > K:

        top_idx = np.argpartition(
            scores,
            -K
        )[-K:]

        top_idx = top_idx[
            np.argsort(
                scores[top_idx]
            )[::-1]
        ]

    else:

        top_idx = np.argsort(
            scores
        )[::-1]

    retrieved = set(
        candidate_local[
            top_idx
        ]
    )

    true_s1 = target_to_s1.get(
        row.entity_id
    )

    true_local = (
        s1_id_to_local.get(
            true_s1
        )
        if true_s1 is not None
        else None
    )

    if (
        true_local is not None
        and
        true_local in retrieved
    ):

        covered += 1

    # Complete entity test will be calculated after
    # collecting all target predictions.

    if (
        (q_idx + 1) % 250 == 0
    ):

        elapsed = (
            time.time() - t0
        )

        print(
            f"{q_idx+1:,}/{query_count:,} "
            f"| recall="
            f"{covered/(q_idx+1)*100:.2f}% "
            f"| {q_idx+1/elapsed:.1f} q/s"
        )

elapsed = (
    time.time() - t0
)

print(
    "\n" + "=" * 100
)
print(
    "BLOCKED TF-IDF RESULT"
)
print(
    "=" * 100
)

print(
    "Queries:",
    query_count
)

print(
    "Pair recall:",
    f"{covered/query_count*100:.4f}%"
)

print(
    "Time:",
    f"{elapsed:.2f} sec"
)

print(
    "Speed:",
    f"{query_count/elapsed:.2f} q/s"
)

print(
    "Projected 10M target queries:",
    f"{9_969_589/(query_count/elapsed)/3600:.2f} hours"
)

print(
    "\nCELL 118 COMPLETE."
)

SyntaxError: invalid syntax (1887030843.py, line 226)

In [ ]:
# CELL 118B — BLOCKED SPARSE TF-IDF RETRIEVAL BENCHMARK

import time
import gc
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 100)
print("CELL 118B — BLOCKED SPARSE TF-IDF RETRIEVAL BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Prepare validation data
# ------------------------------------------------------------

VAL_S1 = s1.copy()
VAL_TARGETS = sample.copy()

VAL_S1["combined_text"] = (
    VAL_S1["name_n"].fillna("")
    + " "
    + VAL_S1["addr_n"].fillna("")
)

VAL_TARGETS["combined_text"] = (
    VAL_TARGETS["name_n"].fillna("")
    + " "
    + VAL_TARGETS["addr_n"].fillna("")
)

print(
    "Validation S1:",
    f"{len(VAL_S1):,}"
)

print(
    "Validation targets:",
    f"{len(VAL_TARGETS):,}"
)

# ------------------------------------------------------------
# TF-IDF
# ------------------------------------------------------------

print("\nBuilding character TF-IDF...")

vectorizer_118 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
)

t0 = time.time()

X_s1 = vectorizer_118.fit_transform(
    VAL_S1["combined_text"]
)

X_target = vectorizer_118.transform(
    VAL_TARGETS["combined_text"]
)

tfidf_time = time.time() - t0

print(
    "TF-IDF build time:",
    f"{tfidf_time:.2f} sec"
)

print(
    "S1 matrix:",
    X_s1.shape,
    "NNZ:",
    f"{X_s1.nnz:,}"
)

print(
    "Target matrix:",
    X_target.shape,
    "NNZ:",
    f"{X_target.nnz:,}"
)

# ------------------------------------------------------------
# Build validation S1 partitions
# ------------------------------------------------------------

print("\nBuilding S1 block indexes...")

block_indexes = {
    "name_first": {},
    "name_last": {},
    "addr_first": {},
    "addr_last": {}
}

for idx, row in enumerate(
    VAL_S1.itertuples(index=False)
):

    country = row.country_n

    if row.name_n:

        block_indexes[
            "name_first"
        ].setdefault(
            (
                country,
                row.name_n[0]
            ),
            []
        ).append(idx)

        block_indexes[
            "name_last"
        ].setdefault(
            (
                country,
                row.name_n[-1]
            ),
            []
        ).append(idx)

    if row.addr_n:

        block_indexes[
            "addr_first"
        ].setdefault(
            (
                country,
                row.addr_n[0]
            ),
            []
        ).append(idx)

        block_indexes[
            "addr_last"
        ].setdefault(
            (
                country,
                row.addr_n[-1]
            ),
            []
        ).append(idx)

for key, blocks in block_indexes.items():

    sizes = np.asarray(
        [
            len(v)
            for v in blocks.values()
        ],
        dtype=np.int32
    )

    print(
        f"{key:12s} "
        f"blocks={len(sizes):,} "
        f"median={np.median(sizes):.0f} "
        f"p95={np.percentile(sizes,95):.0f} "
        f"max={sizes.max():,}"
    )

# ------------------------------------------------------------
# Ground-truth S1 lookup
# ------------------------------------------------------------

s1_id_to_local_118 = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Retrieve Top-K within relevant blocks
# ------------------------------------------------------------

K = 10

covered = 0
candidate_counts = []

t0 = time.time()

n_queries = len(
    VAL_TARGETS
)

for q_idx, row in enumerate(
    VAL_TARGETS.itertuples(index=False)
):

    country = row.country_n

    candidate_local = set()

    # Name first block
    if row.name_n:

        candidate_local.update(
            block_indexes[
                "name_first"
            ].get(
                (
                    country,
                    row.name_n[0]
                ),
                []
            )
        )

    # Name last block
    if row.name_n:

        candidate_local.update(
            block_indexes[
                "name_last"
            ].get(
                (
                    country,
                    row.name_n[-1]
                ),
                []
            )
        )

    # Address first block
    if row.addr_n:

        candidate_local.update(
            block_indexes[
                "addr_first"
            ].get(
                (
                    country,
                    row.addr_n[0]
                ),
                []
            )
        )

    # Address last block
    if row.addr_n:

        candidate_local.update(
            block_indexes[
                "addr_last"
            ].get(
                (
                    country,
                    row.addr_n[-1]
                ),
                []
            )
        )

    candidate_counts.append(
        len(candidate_local)
    )

    if not candidate_local:
        continue

    candidate_local = np.fromiter(
        candidate_local,
        dtype=np.int32
    )

    # --------------------------------------------------------
    # Sparse similarity
    # --------------------------------------------------------

    q_vec = X_target[
        q_idx:q_idx + 1
    ]

    block_matrix = X_s1[
        candidate_local
    ]

    scores = (
        q_vec
        @
        block_matrix.T
    ).toarray().ravel()

    # --------------------------------------------------------
    # Top K
    # --------------------------------------------------------

    if len(scores) > K:

        top_idx = np.argpartition(
            scores,
            -K
        )[-K:]

        top_idx = top_idx[
            np.argsort(
                scores[top_idx]
            )[::-1]
        ]

    else:

        top_idx = np.argsort(
            scores
        )[::-1]

    retrieved = set(
        candidate_local[
            top_idx
        ]
    )

    true_s1 = target_to_s1.get(
        row.entity_id
    )

    true_local = (
        s1_id_to_local_118.get(
            true_s1
        )
        if true_s1 is not None
        else None
    )

    if (
        true_local is not None
        and
        true_local in retrieved
    ):

        covered += 1

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        (q_idx + 1) % 250 == 0
    ):

        elapsed = (
            time.time() - t0
        )

        current_rate = (
            (q_idx + 1)
            /
            elapsed
            if elapsed > 0
            else 0
        )

        print(
            f"{q_idx + 1:,}/{n_queries:,} "
            f"| recall={covered/(q_idx+1)*100:.2f}% "
            f"| speed={current_rate:.2f} q/s"
        )

elapsed = (
    time.time() - t0
)

candidate_counts = np.asarray(
    candidate_counts,
    dtype=np.int32
)

rate = (
    n_queries / elapsed
    if elapsed > 0
    else 0
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "BLOCKED TF-IDF RESULT"
)
print(
    "=" * 100
)

print(
    "Pair recall:",
    f"{covered/n_queries*100:.4f}%"
)

print(
    "Mean candidates:",
    f"{candidate_counts.mean():.2f}"
)

print(
    "Median candidates:",
    f"{np.median(candidate_counts):.0f}"
)

print(
    "P95:",
    f"{np.percentile(candidate_counts,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(candidate_counts,99):.0f}"
)

print(
    "Maximum:",
    f"{candidate_counts.max():,}"
)

print(
    "Benchmark time:",
    f"{elapsed:.2f} sec"
)

print(
    "Speed:",
    f"{rate:.2f} queries/sec"
)

if rate > 0:

    print(
        "Projected 9,969,589 targets:",
        f"{9_969_589/rate/3600:.2f} hours"
    )

print(
    "\nCELL 118B COMPLETE."
)

CELL 118B — BLOCKED SPARSE TF-IDF RETRIEVAL BENCHMARK
Validation S1: 20,000
Validation targets: 2,000

Building character TF-IDF...
TF-IDF build time: 4.28 sec
S1 matrix: (20000, 100000) NNZ: 3,038,032
Target matrix: (2000, 100000) NNZ: 246,708

Building S1 block indexes...
name_first   blocks=64 median=257 p95=856 max=1,306
name_last    blocks=57 median=56 p95=821 max=6,518
addr_first   blocks=69 median=107 p95=905 max=3,386
addr_last    blocks=68 median=20 p95=1111 max=3,175
250/2,000 | recall=98.00% | speed=173.14 q/s
500/2,000 | recall=97.20% | speed=180.79 q/s
750/2,000 | recall=97.60% | speed=177.49 q/s
1,000/2,000 | recall=97.00% | speed=184.16 q/s
1,250/2,000 | recall=96.88% | speed=180.32 q/s
1,500/2,000 | recall=96.60% | speed=168.14 q/s
1,750/2,000 | recall=96.51% | speed=165.52 q/s
2,000/2,000 | recall=96.30% | speed=166.82 q/s

BLOCKED TF-IDF RESULT
Pair recall: 96.3000%
Mean candidates: 3959.63
Median candidates: 3412
P95: 8054
P99: 8810
Maximum: 9,084
Benchmark time: 11.

In [ ]:
# CELL 118C — RESTORE VALIDATION OBJECTS FOR CELL 118B

import os
import gc
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 118C — RESTORE VALIDATION OBJECTS")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

S1_PATH = os.path.join(
    TRAIN_DIR,
    "train_source1.tsv"
)

S2_PATH = os.path.join(
    TRAIN_DIR,
    "train_source2.tsv"
)

S3_PATH = os.path.join(
    TRAIN_DIR,
    "train_source3.tsv"
)

GT_PATH = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

# ------------------------------------------------------------
# Exact verified normalizer
# ------------------------------------------------------------

def normalize_exact(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )

# ------------------------------------------------------------
# Recreate EXACT representative 20k S1 validation
# ------------------------------------------------------------

print("\nLoading S1...")

full_s1_118 = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str
)

s1 = (
    full_s1_118
    .sample(
        n=20_000,
        random_state=42
    )
    .copy()
)

del full_s1_118
gc.collect()

s1["name_n"] = (
    s1["business_name"]
    .map(normalize_exact)
)

s1["addr_n"] = (
    s1["business_address"]
    .map(normalize_exact)
)

s1["country_n"] = (
    s1["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Representative S1:",
    f"{len(s1):,}"
)

# ------------------------------------------------------------
# Load GT and derive exact validation target pool
# ------------------------------------------------------------

print("\nLoading ground truth...")

gt118 = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

gt_lookup_118 = (
    gt118
    .set_index(
        "source1_entity_id"
    )["matched_entity_ids"]
)

validation_target_ids = set()
target_to_s1 = {}

for sid in s1["entity_id"]:

    value = gt_lookup_118.get(
        sid,
        ""
    )

    if pd.isna(value):
        continue

    for target_id in str(
        value
    ).split(","):

        target_id = target_id.strip()

        if target_id:

            validation_target_ids.add(
                target_id
            )

            target_to_s1[
                target_id
            ] = sid

print(
    "Validation target IDs:",
    f"{len(validation_target_ids):,}"
)

# ------------------------------------------------------------
# Reuse existing targets if available and complete
# ------------------------------------------------------------

targets_val = None

if "targets" in globals():

    existing = targets.copy()

    if "entity_id" in existing.columns:

        available = (
            set(existing["entity_id"])
            &
            validation_target_ids
        )

        if len(available) == len(
            validation_target_ids
        ):

            targets_val = (
                existing[
                    existing["entity_id"]
                    .isin(
                        validation_target_ids
                    )
                ]
                .copy()
            )

            print(
                "\nReusing existing targets dataframe."
            )

# ------------------------------------------------------------
# Otherwise rebuild validation target pool
# ------------------------------------------------------------

if targets_val is None:

    print(
        "\nRebuilding validation target records "
        "from S2/S3 in chunks..."
    )

    collected = []

    remaining = set(
        validation_target_ids
    )

    usecols = [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]

    for source_name, path in [
        ("S2", S2_PATH),
        ("S3", S3_PATH)
    ]:

        if not remaining:
            break

        print(
            f"\nScanning {source_name}..."
        )

        for chunk in pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            usecols=usecols,
            chunksize=250_000
        ):

            hit = chunk[
                chunk["entity_id"].isin(
                    remaining
                )
            ]

            if len(hit):

                hit = hit.copy()

                hit["source"] = (
                    source_name
                )

                collected.append(
                    hit
                )

                remaining.difference_update(
                    hit["entity_id"]
                )

                print(
                    f"  collected={sum(len(x) for x in collected):,} "
                    f"| remaining={len(remaining):,}"
                )

            del chunk

        gc.collect()

    if remaining:

        raise RuntimeError(
            f"Could not recover "
            f"{len(remaining):,} validation targets."
        )

    targets_val = pd.concat(
        collected,
        ignore_index=True
    )

    del collected
    gc.collect()

# ------------------------------------------------------------
# Normalize target records
# ------------------------------------------------------------

targets_val["name_n"] = (
    targets_val["business_name"]
    .map(normalize_exact)
)

targets_val["addr_n"] = (
    targets_val["business_address"]
    .map(normalize_exact)
)

targets_val["country_n"] = (
    targets_val["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "\nRecovered target pool:",
    f"{len(targets_val):,}"
)

# ------------------------------------------------------------
# Recreate the 2k target validation sample
# ------------------------------------------------------------

sample = (
    targets_val
    .sample(
        n=min(
            2_000,
            len(targets_val)
        ),
        random_state=42
    )
    .copy()
)

print(
    "Target sample:",
    f"{len(sample):,}"
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

sample_target_ids = set(
    sample["entity_id"]
)

sample_with_gt = (
    sample_target_ids
    &
    set(target_to_s1.keys())
)

print(
    "Sample targets with GT:",
    f"{len(sample_with_gt):,}/"
    f"{len(sample):,}"
)

print(
    "\nExpected representative S1:"
)

print(
    "  S1 = 20,000"
)

print(
    "  No-match = 1,131"
)

print(
    "  True pairs = 69,231"
)

print(
    "\nCELL 118C COMPLETE."
)

CELL 118C — RESTORE VALIDATION OBJECTS

Loading S1...
Representative S1: 20,000

Loading ground truth...
Validation target IDs: 69,231

Rebuilding validation target records from S2/S3 in chunks...

Scanning S2...
  collected=1,571 | remaining=67,660
  collected=3,226 | remaining=66,005
  collected=4,924 | remaining=64,307
  collected=6,563 | remaining=62,668
  collected=8,265 | remaining=60,966
  collected=9,895 | remaining=59,336
  collected=11,546 | remaining=57,685
  collected=13,203 | remaining=56,028
  collected=14,805 | remaining=54,426
  collected=16,474 | remaining=52,757
  collected=18,118 | remaining=51,113
  collected=19,813 | remaining=49,418
  collected=21,519 | remaining=47,712
  collected=23,178 | remaining=46,053
  collected=24,798 | remaining=44,433
  collected=26,412 | remaining=42,819
  collected=28,040 | remaining=41,191
  collected=29,760 | remaining=39,471
  collected=31,465 | remaining=37,766
  collected=33,094 | remaining=36,137
  collected=33,356 | remaining=35

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import numpy as np
import scipy.sparse as sp
import joblib
import lightgbm as lgb

print("=" * 100)
print("CELL 1 — VERIFY CORRECT PRODUCTION ARTIFACTS")
print("=" * 100)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

MODEL_DIR = BASE / "MODELS"
INDEX_DIR = MODEL_DIR / "blocking_indexes"

print("\nBASE:")
print(BASE)

# ------------------------------------------------------------
# Expected artifacts
# ------------------------------------------------------------

required_files = [
    MODEL_DIR / "production_reverse_tfidf_vectorizer.joblib",
    MODEL_DIR / "production_test_s1_tfidf.npz",

    # IMPORTANT: this is the 61-feature / 239-tree model
    MODEL_DIR / "rapidfuzz_k10_ranking_lgbm_best.txt",
    MODEL_DIR / "rapidfuzz_k10_ranking_threshold.txt",

    INDEX_DIR / "test_s2_block_indexes.pkl",
    INDEX_DIR / "test_s3_block_indexes.pkl",

    BASE / "DATASET" / "test" / "test_source1.tsv",
    BASE / "DATASET" / "test" / "test_source2.tsv",
    BASE / "DATASET" / "test" / "test_source3.tsv",
]

print("\n" + "=" * 100)
print("FILE EXISTENCE")
print("=" * 100)

missing = []

for path in required_files:

    exists = path.exists()

    print(
        f"{'FOUND' if exists else 'MISSING':8s} "
        f"{path}"
    )

    if not exists:
        missing.append(path)

if missing:

    raise FileNotFoundError(
        "Missing required artifact(s):\n"
        + "\n".join(
            str(x)
            for x in missing
        )
    )

# ------------------------------------------------------------
# File sizes
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ARTIFACT SIZES")
print("=" * 100)

for path in required_files:

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{size_mb:10.2f} MB   {path.name}"
    )

# ------------------------------------------------------------
# Load reverse TF-IDF vectorizer
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("REVERSE TF-IDF VECTORIZER")
print("=" * 100)

vectorizer = joblib.load(
    MODEL_DIR /
    "production_reverse_tfidf_vectorizer.joblib"
)

print(
    "Vectorizer type:",
    type(vectorizer).__name__
)

print(
    "Vocabulary size:",
    len(
        vectorizer.vocabulary_
    )
)

print(
    "Analyzer:",
    vectorizer.analyzer
)

print(
    "N-gram range:",
    vectorizer.ngram_range
)

# ------------------------------------------------------------
# Load production S1 TF-IDF matrix
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PRODUCTION S1 TF-IDF MATRIX")
print("=" * 100)

X_test_s1 = sp.load_npz(
    MODEL_DIR /
    "production_test_s1_tfidf.npz"
)

print(
    "Matrix type:",
    type(X_test_s1).__name__
)

print(
    "Shape:",
    X_test_s1.shape
)

print(
    "Expected S1 rows:",
    1_732_544
)

print(
    "NNZ:",
    f"{X_test_s1.nnz:,}"
)

print(
    "dtype:",
    X_test_s1.dtype
)

# ------------------------------------------------------------
# Verify matrix/vectorizer dimensional agreement
# ------------------------------------------------------------

if X_test_s1.shape[1] != len(
    vectorizer.vocabulary_
):

    raise RuntimeError(
        "TF-IDF matrix feature dimension does not "
        "match vectorizer vocabulary size."
    )

if X_test_s1.shape[0] != 1_732_544:

    raise RuntimeError(
        "Unexpected number of TEST S1 rows."
    )

print(
    "\nTF-IDF dimensions VERIFIED."
)

# ------------------------------------------------------------
# Load the actual 0.973279 development model
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RANKING-AWARE LIGHTGBM MODEL")
print("=" * 100)

model_path = (
    MODEL_DIR /
    "rapidfuzz_k10_ranking_lgbm_best.txt"
)

threshold_path = (
    MODEL_DIR /
    "rapidfuzz_k10_ranking_threshold.txt"
)

ranking_model = lgb.Booster(
    model_file=str(
        model_path
    )
)

with open(
    threshold_path,
    "r"
) as f:
    ranking_threshold = float(
        f.read().strip()
    )

print(
    "Model file:",
    model_path.name
)

print(
    "Trees:",
    ranking_model.num_trees()
)

print(
    "Features:",
    ranking_model.num_feature()
)

print(
    "Threshold:",
    ranking_threshold
)

# ------------------------------------------------------------
# Verify the exact expected checkpoint
# ------------------------------------------------------------

if ranking_model.num_trees() <= 0:
    raise RuntimeError(
        "Unexpected LightGBM model structure."
    )

if ranking_model.num_feature() != 61:
    raise RuntimeError(
        "Loaded model is not the expected 61-feature model."
    )

if ranking_model.num_trees() != 239:
    raise RuntimeError(
        "Loaded model is not the expected 239-tree checkpoint."
    )

if abs(
    ranking_threshold - 0.225
) > 1e-6:

    raise RuntimeError(
        "Loaded threshold is not 0.225."
    )

print(
    "\n61-feature / 239-tree / 0.225 checkpoint VERIFIED."
)

# ------------------------------------------------------------
# Do NOT load the huge block indexes yet
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("BLOCK INDEXES")
print("=" * 100)

for path in [
    INDEX_DIR / "test_s2_block_indexes.pkl",
    INDEX_DIR / "test_s3_block_indexes.pkl"
]:

    print(
        f"{path.name}: "
        f"{path.stat().st_size/(1024**2):.2f} MB"
    )

print(
    "\nBlock indexes are present but intentionally NOT loaded "
    "in Cell 1 to conserve RAM."
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VERIFICATION SUMMARY")
print("=" * 100)

print(
    "✓ Reverse TF-IDF vectorizer"
)

print(
    "✓ TEST S1 TF-IDF matrix"
)

print(
    "✓ 61-feature ranking model"
)

print(
    "✓ Threshold = 0.225"
)

print(
    "✓ S2/S3 production block indexes present"
)

print(
    "\nCELL 1 COMPLETE — send me the output before Cell 2."
)

Mounted at /content/drive
CELL 1 — VERIFY CORRECT PRODUCTION ARTIFACTS

BASE:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026

FILE EXISTENCE
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/production_reverse_tfidf_vectorizer.joblib
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/production_test_s1_tfidf.npz
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/rapidfuzz_k10_ranking_lgbm_best.txt
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/rapidfuzz_k10_ranking_threshold.txt
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocking_indexes/test_s2_block_indexes.pkl
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocking_indexes/test_s3_block_indexes.pkl
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/test/test_source1.tsv
FOUND    /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/test/test_source2.tsv
FOUND    /content/drive/M

In [ ]:
# CELL 1B — COMPLETE CORRECTED ARTIFACT VERIFICATION

print("=" * 100)
print("CELL 1B — COMPLETE ARTIFACT VERIFICATION")
print("=" * 100)

# ------------------------------------------------------------
# Basic model checks
# ------------------------------------------------------------

print(
    "\nModel:",
    model_path.name
)

print(
    "Trees:",
    ranking_model.num_trees()
)

print(
    "Features:",
    ranking_model.num_feature()
)

print(
    "Threshold:",
    ranking_threshold
)

# ------------------------------------------------------------
# Strict checks
# ------------------------------------------------------------

assert ranking_model.num_trees() == 239, (
    f"Unexpected tree count: "
    f"{ranking_model.num_trees()}"
)

assert ranking_model.num_feature() == 61, (
    f"Unexpected feature count: "
    f"{ranking_model.num_feature()}"
)

assert abs(
    ranking_threshold - 0.225
) < 1e-6, (
    f"Unexpected threshold: "
    f"{ranking_threshold}"
)

assert X_test_s1.shape == (
    1_732_544,
    100_000
), (
    f"Unexpected S1 TF-IDF shape: "
    f"{X_test_s1.shape}"
)

assert X_test_s1.shape[1] == len(
    vectorizer.vocabulary_
), (
    "TF-IDF matrix and vectorizer dimensions disagree."
)

print(
    "\n✓ 61-feature model verified"
)

print(
    "✓ 239-tree checkpoint verified"
)

print(
    "✓ Threshold 0.225 verified"
)

print(
    "✓ TEST S1 TF-IDF matrix verified"
)

print(
    "✓ Vectorizer/matrix dimensions verified"
)

print(
    "\nS2/S3 block indexes are present "
    "but have NOT been loaded yet."
)

print(
    "\nCELL 1B COMPLETE."
)

CELL 1B — COMPLETE ARTIFACT VERIFICATION

Model: rapidfuzz_k10_ranking_lgbm_best.txt
Trees: 239
Features: 61
Threshold: 0.22499999999999998

✓ 61-feature model verified
✓ 239-tree checkpoint verified
✓ Threshold 0.225 verified
✓ TEST S1 TF-IDF matrix verified
✓ Vectorizer/matrix dimensions verified

S2/S3 block indexes are present but have NOT been loaded yet.

CELL 1B COMPLETE.


In [ ]:
# CELL 2 — LOAD S2/S3 PRODUCTION BLOCK INDEXES
# Load + normalize only. No candidate generation yet.

import pickle
import psutil
import gc

print("=" * 100)
print("CELL 2 — LOAD S2/S3 PRODUCTION BLOCK INDEXES")
print("=" * 100)

# ------------------------------------------------------------
# Memory helper
# ------------------------------------------------------------

process = psutil.Process()

def ram_gb():
    return (
        process.memory_info().rss
        / (1024 ** 3)
    )

print(
    "\nRAM before indexes:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

s2_index_path = (
    MODEL_DIR
    / "blocking_indexes"
    / "test_s2_block_indexes.pkl"
)

s3_index_path = (
    MODEL_DIR
    / "blocking_indexes"
    / "test_s3_block_indexes.pkl"
)

# ------------------------------------------------------------
# Generic loader
# ------------------------------------------------------------

def load_and_normalize_blocks(
    path,
    source_name
):

    print(
        f"\nLoading {source_name} index..."
    )

    with open(
        path,
        "rb"
    ) as f:

        raw = pickle.load(f)

    print(
        "Raw type:",
        type(raw).__name__
    )

    # Test indexes are dicts
    if isinstance(
        raw,
        dict
    ):

        blocks = raw

    # Keep support for tuple/list format too
    elif isinstance(
        raw,
        (tuple, list)
    ):

        if len(raw) < 4:
            raise RuntimeError(
                f"{source_name} tuple/list has "
                f"only {len(raw)} elements."
            )

        blocks = {
            "name_index": raw[0],
            "prefix_index": raw[1],
            "address_index": raw[2],
            "num_country_index": raw[3],
        }

    else:

        raise RuntimeError(
            f"Unsupported {source_name} index type: "
            f"{type(raw).__name__}"
        )

    required = [
        "name_index",
        "prefix_index",
        "address_index",
        "num_country_index"
    ]

    missing = [
        key
        for key in required
        if key not in blocks
    ]

    if missing:
        raise RuntimeError(
            f"{source_name} missing keys: {missing}"
        )

    print(
        f"{source_name} block counts:"
    )

    for key in required:

        obj = blocks[key]

        print(
            f"  {key:18s}: "
            f"{type(obj).__name__:12s} "
            f"{len(obj):,}"
        )

    # Small structural inspection
    print(
        f"\n{source_name} samples:"
    )

    for key in required:

        obj = blocks[key]

        try:

            first_key = next(
                iter(obj)
            )

            first_value = obj[
                first_key
            ]

            print(
                f"  {key}:"
            )

            print(
                "    key  =",
                repr(first_key)
            )

            print(
                "    value=",
                repr(first_value)[:300]
            )

        except Exception as e:

            print(
                f"  {key}: sample unavailable:",
                repr(e)
            )

    return blocks


# ------------------------------------------------------------
# Load S2
# ------------------------------------------------------------

test_s2_blocks_prod = (
    load_and_normalize_blocks(
        s2_index_path,
        "S2"
    )
)

print(
    "\nRAM after S2:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Load S3
# ------------------------------------------------------------

test_s3_blocks_prod = (
    load_and_normalize_blocks(
        s3_index_path,
        "S3"
    )
)

gc.collect()

print(
    "\nRAM after S2 + S3:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

for source_name, blocks in [
    (
        "S2",
        test_s2_blocks_prod
    ),
    (
        "S3",
        test_s3_blocks_prod
    )
]:

    assert set(
        blocks.keys()
    ) == {
        "name_index",
        "prefix_index",
        "address_index",
        "num_country_index"
    }, (
        f"{source_name} block structure mismatch."
    )

print(
    "\n" + "=" * 100
)
print(
    "INDEX VERIFICATION COMPLETE"
)
print(
    "=" * 100
)

print(
    "✓ S2 index loaded"
)

print(
    "✓ S3 index loaded"
)

print(
    "✓ Required four blocking channels present"
)

print(
    "✓ No candidate generation performed"
)

print(
    "\nCELL 2 COMPLETE."
)

CELL 2 — LOAD S2/S3 PRODUCTION BLOCK INDEXES

RAM before indexes: 2.44 GB

Loading S2 index...
Raw type: dict
S2 block counts:
  name_index        : dict         3,714,440
  prefix_index      : dict         169,019
  address_index     : dict         4,141,402
  num_country_index : dict         662,890

S2 samples:
  name_index:
    key  = 'brahmainfosoft'
    value= ['S2-192345572']
  prefix_index:
    key  = 'brah'
    value= ['S2-192345572', 'S2-789476436', 'S2-278585468', 'S2-154901012', 'S2-228284220', 'S2-603507241', 'S2-573928617', 'S2-85735365', 'S2-595712322', 'S2-120539959', 'S2-881568248', 'S2-439612694', 'S2-773799462', 'S2-911361832', 'S2-125857089', 'S2-240023765', 'S2-103944025', 'S2-108588994', 'S2-57118737
  address_index:
    key  = 'coimatorecolonyhunsurtqmysoredistkarnataka'
    value= ['S2-192345572', 'S2-83023718']
  num_country_index:
    key  = '63_france'
    value= ['S2-566025912', 'S2-313017868', 'S2-249372159', 'S2-882008416', 'S2-83159371', 'S2-649106539', '

In [ ]:
# CELL 3 — BLOCKED TF-IDF + NUMBER/COUNTRY AUGMENTATION
# Uses the already-built validation matrices from Cell 118B.

import re
import time
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 3 — BLOCKED TF-IDF + NUMBER/COUNTRY AUGMENTATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects from Cell 118B
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGETS",
    "X_s1",
    "X_target",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        f"Run Cell 118C and Cell 118B first."
    )

# ------------------------------------------------------------
# Build number+country S1 blocks
# ------------------------------------------------------------

num_country_blocks = {}

for idx, row in enumerate(
    VAL_S1.itertuples(index=False)
):

    country = row.country_n
    addr = row.addr_n

    if not addr:
        continue

    for num in set(
        re.findall(
            r"\d+",
            addr
        )
    ):

        num_country_blocks.setdefault(
            (country, num),
            []
        ).append(idx)

sizes = np.asarray(
    [
        len(v)
        for v in num_country_blocks.values()
    ],
    dtype=np.int32
)

print(
    "\nNumber-country block statistics:"
)

print(
    "Blocks:",
    f"{len(sizes):,}"
)

print(
    "Median:",
    f"{np.median(sizes):.0f}"
)

print(
    "P95:",
    f"{np.percentile(sizes,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(sizes,99):.0f}"
)

print(
    "Maximum:",
    f"{sizes.max():,}"
)

# ------------------------------------------------------------
# S1 local ID lookup
# ------------------------------------------------------------

s1_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Candidate retrieval
#
# Base = 4 one-character blocks:
# name first / last
# address first / last
#
# Augmented = + number/country
# ------------------------------------------------------------

K_VALUES = [
    10,
    20,
    50,
    100
]

results = []

for use_number_block in [
    False,
    True
]:

    mode_name = (
        "char_blocks_only"
        if not use_number_block
        else "char_blocks_plus_number_country"
    )

    for K in K_VALUES:

        covered = 0
        candidate_counts = []

        t0 = time.time()

        for q_idx, row in enumerate(
            VAL_TARGETS.itertuples(index=False)
        ):

            country = row.country_n
            name = row.name_n
            addr = row.addr_n

            candidate_local = set()

            # ------------------------------
            # Name first
            # ------------------------------

            if name:

                candidate_local.update(
                    block_indexes[
                        "name_first"
                    ].get(
                        (
                            country,
                            name[0]
                        ),
                        []
                    )
                )

                # Name last
                candidate_local.update(
                    block_indexes[
                        "name_last"
                    ].get(
                        (
                            country,
                            name[-1]
                        ),
                        []
                    )
                )

            # ------------------------------
            # Address first
            # ------------------------------

            if addr:

                candidate_local.update(
                    block_indexes[
                        "addr_first"
                    ].get(
                        (
                            country,
                            addr[0]
                        ),
                        []
                    )
                )

                # Address last
                candidate_local.update(
                    block_indexes[
                        "addr_last"
                    ].get(
                        (
                            country,
                            addr[-1]
                        ),
                        []
                    )
                )

            # ------------------------------
            # Number + country
            # ------------------------------

            if use_number_block and addr:

                for num in set(
                    re.findall(
                        r"\d+",
                        addr
                    )
                ):

                    candidate_local.update(
                        num_country_blocks.get(
                            (
                                country,
                                num
                            ),
                            []
                        )
                    )

            candidate_counts.append(
                len(candidate_local)
            )

            if not candidate_local:
                continue

            candidate_local = np.fromiter(
                candidate_local,
                dtype=np.int32
            )

            # ------------------------------
            # Sparse TF-IDF similarity
            # ------------------------------

            scores = (
                X_target[
                    q_idx:q_idx + 1
                ]
                @
                X_s1[
                    candidate_local
                ].T
            ).toarray().ravel()

            # ------------------------------
            # Top-K
            # ------------------------------

            if len(scores) > K:

                top_idx = np.argpartition(
                    scores,
                    -K
                )[-K:]

                top_idx = top_idx[
                    np.argsort(
                        scores[top_idx]
                    )[::-1]
                ]

            else:

                top_idx = np.argsort(
                    scores
                )[::-1]

            retrieved = set(
                candidate_local[
                    top_idx
                ]
            )

            # ------------------------------
            # Ground truth
            # ------------------------------

            true_s1 = target_to_s1.get(
                row.entity_id
            )

            true_local = (
                s1_local.get(
                    true_s1
                )
                if true_s1 is not None
                else None
            )

            if (
                true_local is not None
                and
                true_local in retrieved
            ):

                covered += 1

        elapsed = (
            time.time() - t0
        )

        candidate_counts = np.asarray(
            candidate_counts,
            dtype=np.int32
        )

        qps = (
            len(VAL_TARGETS) / elapsed
            if elapsed > 0
            else 0
        )

        results.append({
            "mode":
                mode_name,

            "K":
                K,

            "recall_pct":
                covered
                /
                len(VAL_TARGETS)
                *
                100,

            "mean_candidates":
                candidate_counts.mean(),

            "median":
                np.median(candidate_counts),

            "p95":
                np.percentile(
                    candidate_counts,
                    95
                ),

            "p99":
                np.percentile(
                    candidate_counts,
                    99
                ),

            "max":
                candidate_counts.max(),

            "qps":
                qps,

            "projected_hours":
                (
                    9_969_589 / qps / 3600
                    if qps > 0
                    else np.inf
                )
        })

results3 = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "RESULTS"
)
print(
    "=" * 100
)

print(
    results3.to_string(
        index=False,
        formatters={
            "recall_pct":
                lambda x: f"{x:.4f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median":
                lambda x: f"{x:.0f}",
            "p95":
                lambda x: f"{x:.0f}",
            "p99":
                lambda x: f"{x:.0f}",
            "qps":
                lambda x: f"{x:.2f}",
            "projected_hours":
                lambda x: f"{x:.2f}"
        }
    )
)

print(
    "\nCELL 3 COMPLETE."
)

CELL 3 — BLOCKED TF-IDF + NUMBER/COUNTRY AUGMENTATION


RuntimeError: Missing required objects: ['VAL_S1', 'VAL_TARGETS', 'X_s1', 'X_target', 'target_to_s1']. Run Cell 118C and Cell 118B first.

In [ ]:
# CELL 3B — SELF-CONTAINED BLOCKED TF-IDF + NUMBER/COUNTRY BENCHMARK

import os
import re
import gc
import time
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

print("=" * 100)
print("CELL 3B — SELF-CONTAINED BLOCKED TF-IDF + NUMBER/COUNTRY")
print("=" * 100)

# ============================================================
# PATHS
# ============================================================

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

S1_PATH = os.path.join(
    TRAIN_DIR,
    "train_source1.tsv"
)

S2_PATH = os.path.join(
    TRAIN_DIR,
    "train_source2.tsv"
)

S3_PATH = os.path.join(
    TRAIN_DIR,
    "train_source3.tsv"
)

GT_PATH = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

# ============================================================
# EXACT VERIFIED NORMALIZER
# ============================================================

def normalize_exact(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )

# ============================================================
# REBUILD REPRESENTATIVE 20K S1 VALIDATION
# ============================================================

print("\nLoading S1...")

full_s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str
)

VAL_S1 = (
    full_s1
    .sample(
        n=20_000,
        random_state=42
    )
    .copy()
)

del full_s1
gc.collect()

VAL_S1["name_n"] = (
    VAL_S1["business_name"]
    .map(normalize_exact)
)

VAL_S1["addr_n"] = (
    VAL_S1["business_address"]
    .map(normalize_exact)
)

VAL_S1["country_n"] = (
    VAL_S1["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print(
    "Validation S1:",
    f"{len(VAL_S1):,}"
)

# ============================================================
# GROUND TRUTH
# ============================================================

print("\nLoading ground truth...")

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

gt_lookup = (
    gt
    .set_index(
        "source1_entity_id"
    )["matched_entity_ids"]
    .fillna("")
)

target_to_s1 = {}

for sid in VAL_S1["entity_id"]:

    value = gt_lookup.get(
        sid,
        ""
    )

    if not value:
        continue

    for target_id in str(
        value
    ).split(","):

        target_id = target_id.strip()

        if target_id:

            target_to_s1[
                target_id
            ] = sid

validation_target_ids = set(
    target_to_s1.keys()
)

print(
    "Validation target IDs:",
    f"{len(validation_target_ids):,}"
)

# ============================================================
# LOAD ONLY VALIDATION TARGET RECORDS
# ============================================================

usecols = [
    "entity_id",
    "business_name",
    "business_address",
    "country"
]

collected = []

remaining = set(
    validation_target_ids
)

for source_name, path in [
    ("S2", S2_PATH),
    ("S3", S3_PATH)
]:

    if not remaining:
        break

    print(
        f"\nScanning {source_name}..."
    )

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=usecols,
        chunksize=250_000
    ):

        hit = chunk[
            chunk["entity_id"].isin(
                remaining
            )
        ]

        if len(hit):

            hit = hit.copy()

            hit["source"] = (
                source_name
            )

            collected.append(
                hit
            )

            remaining.difference_update(
                hit["entity_id"]
            )

            print(
                "  recovered:",
                f"{len(hit):,}",
                "| remaining:",
                f"{len(remaining):,}"
            )

        del chunk

    gc.collect()

if remaining:

    raise RuntimeError(
        f"Could not recover "
        f"{len(remaining):,} validation targets."
    )

VAL_TARGET_POOL = pd.concat(
    collected,
    ignore_index=True
)

del collected
gc.collect()

VAL_TARGET_POOL["name_n"] = (
    VAL_TARGET_POOL["business_name"]
    .map(normalize_exact)
)

VAL_TARGET_POOL["addr_n"] = (
    VAL_TARGET_POOL["business_address"]
    .map(normalize_exact)
)

VAL_TARGET_POOL["country_n"] = (
    VAL_TARGET_POOL["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# ============================================================
# SAME 2K TARGET SAMPLE
# ============================================================

VAL_TARGETS = (
    VAL_TARGET_POOL
    .sample(
        n=2_000,
        random_state=42
    )
    .copy()
)

print(
    "\nValidation target sample:",
    f"{len(VAL_TARGETS):,}"
)

# ============================================================
# TF-IDF
# ============================================================

print(
    "\nBuilding validation TF-IDF..."
)

VAL_S1["combined_text"] = (
    VAL_S1["name_n"]
    + " "
    + VAL_S1["addr_n"]
)

VAL_TARGETS["combined_text"] = (
    VAL_TARGETS["name_n"]
    + " "
    + VAL_TARGETS["addr_n"]
)

vectorizer_3b = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
)

t0 = time.time()

X_s1_3b = vectorizer_3b.fit_transform(
    VAL_S1["combined_text"]
)

X_target_3b = vectorizer_3b.transform(
    VAL_TARGETS["combined_text"]
)

print(
    "TF-IDF time:",
    f"{time.time()-t0:.2f} sec"
)

print(
    "S1 matrix:",
    X_s1_3b.shape,
    "NNZ:",
    f"{X_s1_3b.nnz:,}"
)

print(
    "Target matrix:",
    X_target_3b.shape,
    "NNZ:",
    f"{X_target_3b.nnz:,}"
)

# ============================================================
# BUILD CHAR BLOCKS
# ============================================================

print(
    "\nBuilding S1 character blocks..."
)

char_blocks = {
    "name_first": {},
    "name_last": {},
    "addr_first": {},
    "addr_last": {}
}

for idx, row in enumerate(
    VAL_S1.itertuples(index=False)
):

    country = row.country_n

    if row.name_n:

        char_blocks[
            "name_first"
        ].setdefault(
            (
                country,
                row.name_n[0]
            ),
            []
        ).append(idx)

        char_blocks[
            "name_last"
        ].setdefault(
            (
                country,
                row.name_n[-1]
            ),
            []
        ).append(idx)

    if row.addr_n:

        char_blocks[
            "addr_first"
        ].setdefault(
            (
                country,
                row.addr_n[0]
            ),
            []
        ).append(idx)

        char_blocks[
            "addr_last"
        ].setdefault(
            (
                country,
                row.addr_n[-1]
            ),
            []
        ).append(idx)

# ============================================================
# BUILD NUMBER/COUNTRY BLOCKS
# ============================================================

num_country_blocks = {}

for idx, row in enumerate(
    VAL_S1.itertuples(index=False)
):

    country = row.country_n

    for num in set(
        re.findall(
            r"\d+",
            row.addr_n
        )
    ):

        num_country_blocks.setdefault(
            (
                country,
                num
            ),
            []
        ).append(idx)

# ============================================================
# S1 ID -> LOCAL INDEX
# ============================================================

s1_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ============================================================
# BENCHMARK
# ============================================================

results = []

for use_number in [
    False,
    True
]:

    mode = (
        "char_blocks_only"
        if not use_number
        else
        "char_blocks_plus_number_country"
    )

    for K in [
        10,
        20,
        50,
        100
    ]:

        covered = 0
        candidate_counts = []

        t0 = time.time()

        for q_idx, row in enumerate(
            VAL_TARGETS.itertuples(
                index=False
            )
        ):

            country = row.country_n

            candidates = set()

            # --------------------------------------------
            # Character blocks
            # --------------------------------------------

            if row.name_n:

                candidates.update(
                    char_blocks[
                        "name_first"
                    ].get(
                        (
                            country,
                            row.name_n[0]
                        ),
                        []
                    )
                )

                candidates.update(
                    char_blocks[
                        "name_last"
                    ].get(
                        (
                            country,
                            row.name_n[-1]
                        ),
                        []
                    )
                )

            if row.addr_n:

                candidates.update(
                    char_blocks[
                        "addr_first"
                    ].get(
                        (
                            country,
                            row.addr_n[0]
                        ),
                        []
                    )
                )

                candidates.update(
                    char_blocks[
                        "addr_last"
                    ].get(
                        (
                            country,
                            row.addr_n[-1]
                        ),
                        []
                    )
                )

            # --------------------------------------------
            # Number + country
            # --------------------------------------------

            if use_number:

                for num in set(
                    re.findall(
                        r"\d+",
                        row.addr_n
                    )
                ):

                    candidates.update(
                        num_country_blocks.get(
                            (
                                country,
                                num
                            ),
                            []
                        )
                    )

            candidate_counts.append(
                len(candidates)
            )

            if not candidates:
                continue

            candidate_array = np.fromiter(
                candidates,
                dtype=np.int32
            )

            scores = (
                X_target_3b[
                    q_idx:q_idx + 1
                ]
                @
                X_s1_3b[
                    candidate_array
                ].T
            ).toarray().ravel()

            if len(scores) > K:

                top = np.argpartition(
                    scores,
                    -K
                )[-K:]

                top = top[
                    np.argsort(
                        scores[top]
                    )[::-1]
                ]

            else:

                top = np.argsort(
                    scores
                )[::-1]

            retrieved = set(
                candidate_array[top]
            )

            true_s1 = target_to_s1.get(
                row.entity_id
            )

            true_local = (
                s1_local.get(
                    true_s1
                )
                if true_s1 is not None
                else None
            )

            if (
                true_local is not None
                and true_local in retrieved
            ):

                covered += 1

        elapsed = (
            time.time() - t0
        )

        candidate_counts = np.asarray(
            candidate_counts,
            dtype=np.int32
        )

        qps = (
            len(VAL_TARGETS)
            /
            elapsed
        )

        results.append({
            "mode":
                mode,

            "K":
                K,

            "recall_pct":
                covered
                /
                len(VAL_TARGETS)
                *
                100,

            "mean_candidates":
                candidate_counts.mean(),

            "median":
                np.median(
                    candidate_counts
                ),

            "p95":
                np.percentile(
                    candidate_counts,
                    95
                ),

            "p99":
                np.percentile(
                    candidate_counts,
                    99
                ),

            "max":
                candidate_counts.max(),

            "qps":
                qps,

            "projected_hours":
                9_969_589
                /
                qps
                /
                3600
        })

results_3b = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "CELL 3B RESULTS"
)
print(
    "=" * 100
)

print(
    results_3b.to_string(
        index=False,
        formatters={
            "recall_pct":
                lambda x: f"{x:.4f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median":
                lambda x: f"{x:.0f}",
            "p95":
                lambda x: f"{x:.0f}",
            "p99":
                lambda x: f"{x:.0f}",
            "qps":
                lambda x: f"{x:.2f}",
            "projected_hours":
                lambda x: f"{x:.2f}"
        }
    )
)

print(
    "\nCELL 3B COMPLETE."
)

CELL 3B — SELF-CONTAINED BLOCKED TF-IDF + NUMBER/COUNTRY

Loading S1...
Validation S1: 20,000

Loading ground truth...
Validation target IDs: 69,231

Scanning S2...
  recovered: 1,571 | remaining: 67,660
  recovered: 1,655 | remaining: 66,005
  recovered: 1,698 | remaining: 64,307
  recovered: 1,639 | remaining: 62,668
  recovered: 1,702 | remaining: 60,966
  recovered: 1,630 | remaining: 59,336
  recovered: 1,651 | remaining: 57,685
  recovered: 1,657 | remaining: 56,028
  recovered: 1,602 | remaining: 54,426
  recovered: 1,669 | remaining: 52,757
  recovered: 1,644 | remaining: 51,113
  recovered: 1,695 | remaining: 49,418
  recovered: 1,706 | remaining: 47,712
  recovered: 1,659 | remaining: 46,053
  recovered: 1,620 | remaining: 44,433
  recovered: 1,614 | remaining: 42,819
  recovered: 1,628 | remaining: 41,191
  recovered: 1,720 | remaining: 39,471
  recovered: 1,705 | remaining: 37,766
  recovered: 1,629 | remaining: 36,137
  recovered: 262 | remaining: 35,875

Scanning S3...
  

In [ ]:
# CELL 4 — FULL VALIDATION RETRIEVAL / ENTITY COVERAGE
#
# Uses:
#   20,000 representative S1 validation entities
#   all 69,231 validation target records
#
# Retrieves Top-100 once, then evaluates K=10/20/50/100.

import time
import re
import gc
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 4 — FULL 69,231-TARGET RETRIEVAL / ENTITY COVERAGE")
print("=" * 100)

# ------------------------------------------------------------
# Required objects from Cell 3B
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "X_s1_3b",
    "vectorizer_3b",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}. "
        f"Run Cell 3B first."
    )

# ------------------------------------------------------------
# Build target matrix for ALL 69,231 validation targets
# ------------------------------------------------------------

print(
    "\nTransforming all validation targets..."
)

VAL_TARGET_POOL = VAL_TARGET_POOL.copy()

VAL_TARGET_POOL["combined_text"] = (
    VAL_TARGET_POOL["name_n"].fillna("")
    + " "
    + VAL_TARGET_POOL["addr_n"].fillna("")
)

t0 = time.time()

X_target_full = (
    vectorizer_3b.transform(
        VAL_TARGET_POOL["combined_text"]
    )
)

print(
    "Target transform time:",
    f"{time.time() - t0:.2f} sec"
)

print(
    "Full target matrix:",
    X_target_full.shape
)

print(
    "NNZ:",
    f"{X_target_full.nnz:,}"
)

# ------------------------------------------------------------
# S1 ID -> local index
# ------------------------------------------------------------

s1_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Retrieve Top-100 candidates
# ------------------------------------------------------------

TOP_K = 100

retrieved_top100 = {}

covered_counter = 0

t0 = time.time()

for q_idx, row in enumerate(
    VAL_TARGET_POOL.itertuples(
        index=False
    )
):

    country = row.country_n
    name = row.name_n
    addr = row.addr_n

    candidates = set()

    # --------------------------------------------------------
    # Character blocks
    # --------------------------------------------------------

    if name:

        candidates.update(
            char_blocks[
                "name_first"
            ].get(
                (
                    country,
                    name[0]
                ),
                []
            )
        )

        candidates.update(
            char_blocks[
                "name_last"
            ].get(
                (
                    country,
                    name[-1]
                ),
                []
            )
        )

    if addr:

        candidates.update(
            char_blocks[
                "addr_first"
            ].get(
                (
                    country,
                    addr[0]
                ),
                []
            )
        )

        candidates.update(
            char_blocks[
                "addr_last"
            ].get(
                (
                    country,
                    addr[-1]
                ),
                []
            )
        )

    # --------------------------------------------------------
    # NUMBER + COUNTRY
    # --------------------------------------------------------

    if addr:

        for num in set(
            re.findall(
                r"\d+",
                addr
            )
        ):

            candidates.update(
                num_country_blocks.get(
                    (
                        country,
                        num
                    ),
                    []
                )
            )

    if not candidates:
        retrieved_top100[
            row.entity_id
        ] = np.empty(
            0,
            dtype=np.int32
        )
        continue

    candidate_array = np.fromiter(
        candidates,
        dtype=np.int32
    )

    # --------------------------------------------------------
    # Sparse TF-IDF similarity
    # --------------------------------------------------------

    scores = (
        X_target_full[
            q_idx:q_idx + 1
        ]
        @
        X_s1_3b[
            candidate_array
        ].T
    ).toarray().ravel()

    # --------------------------------------------------------
    # Top-100
    # --------------------------------------------------------

    if len(scores) > TOP_K:

        top = np.argpartition(
            scores,
            -TOP_K
        )[-TOP_K:]

        top = top[
            np.argsort(
                scores[top]
            )[::-1]
        ]

    else:

        top = np.argsort(
            scores
        )[::-1]

    retrieved = candidate_array[
        top
    ]

    retrieved_top100[
        row.entity_id
    ] = retrieved

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        (q_idx + 1) % 5000 == 0
    ):

        elapsed = (
            time.time() - t0
        )

        print(
            f"{q_idx + 1:,}/"
            f"{len(VAL_TARGET_POOL):,} "
            f"| speed="
            f"{(q_idx + 1)/elapsed:.1f} q/s"
        )

elapsed = time.time() - t0

print(
    "\nRetrieval time:",
    f"{elapsed/60:.2f} min"
)

print(
    "Speed:",
    f"{len(VAL_TARGET_POOL)/elapsed:.2f} targets/sec"
)

# ------------------------------------------------------------
# Pair recall + complete entity recall
# ------------------------------------------------------------

K_VALUES = [
    10,
    20,
    50,
    100
]

# True targets by S1
true_by_s1 = {}

for sid in VAL_S1["entity_id"]:

    true_by_s1[sid] = set()

for target_id, sid in target_to_s1.items():

    if sid in true_by_s1:
        true_by_s1[sid].add(
            target_id
        )

results4 = []

for K in K_VALUES:

    covered_pairs = 0

    # predicted target IDs per S1
    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    for target_id, indices in retrieved_top100.items():

        if len(indices) == 0:
            continue

        selected = indices[
            :K
        ]

        for local_idx in selected:

            sid = VAL_S1.iloc[
                int(local_idx)
            ]["entity_id"]

            predicted_by_s1[
                sid
            ].add(
                target_id
            )

        true_sid = target_to_s1.get(
            target_id
        )

        if true_sid is None:
            continue

        true_local = s1_local.get(
            true_sid
        )

        if (
            true_local is not None
            and
            true_local in set(
                selected
            )
        ):

            covered_pairs += 1

    # --------------------------------------------------------
    # Complete matched entities
    # --------------------------------------------------------

    matched_entities = 0
    complete_entities = 0

    for sid, true_targets in true_by_s1.items():

        if not true_targets:
            continue

        matched_entities += 1

        predicted = predicted_by_s1[
            sid
        ]

        if true_targets.issubset(
            predicted
        ):

            complete_entities += 1

    # --------------------------------------------------------
    # Candidate edges
    # --------------------------------------------------------

    total_edges = sum(
        len(
            v
        )
        for v in predicted_by_s1.values()
    )

    results4.append({
        "K": K,

        "pair_recall_pct":
            (
                covered_pairs
                /
                len(VAL_TARGET_POOL)
                *
                100
            ),

        "complete_entity_recall_pct":
            (
                complete_entities
                /
                matched_entities
                *
                100
            ),

        "total_edges":
            total_edges,

        "mean_edges_per_S1":
            total_edges
            /
            len(VAL_S1),

        "complete_entities":
            complete_entities,

        "matched_entities":
            matched_entities
    })

results4_df = pd.DataFrame(
    results4
)

print(
    "\n" + "=" * 100
)
print(
    "FULL VALIDATION RETRIEVAL RESULTS"
)
print(
    "=" * 100
)

print(
    results4_df.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.4f}",

            "complete_entity_recall_pct":
                lambda x: f"{x:.4f}",

            "mean_edges_per_S1":
                lambda x: f"{x:.2f}"
        }
    )
)

# ------------------------------------------------------------
# Save retrieved candidates for later matcher scoring
# ------------------------------------------------------------

retrieval_path = (
    BASE
    if "BASE" in globals()
    else None
)

print(
    "\nCELL 4 COMPLETE."
)

CELL 4 — FULL 69,231-TARGET RETRIEVAL / ENTITY COVERAGE

Transforming all validation targets...
Target transform time: 11.74 sec
Full target matrix: (69231, 100000)
NNZ: 8,520,229
5,000/69,231 | speed=144.4 q/s
10,000/69,231 | speed=155.3 q/s
15,000/69,231 | speed=155.7 q/s
20,000/69,231 | speed=159.5 q/s
25,000/69,231 | speed=162.5 q/s
30,000/69,231 | speed=163.9 q/s
35,000/69,231 | speed=165.5 q/s
40,000/69,231 | speed=167.2 q/s
45,000/69,231 | speed=167.7 q/s
50,000/69,231 | speed=168.7 q/s
55,000/69,231 | speed=169.9 q/s
60,000/69,231 | speed=170.4 q/s
65,000/69,231 | speed=170.7 q/s

Retrieval time: 6.74 min
Speed: 171.20 targets/sec

FULL VALIDATION RETRIEVAL RESULTS
  K pair_recall_pct complete_entity_recall_pct  total_edges mean_edges_per_S1  complete_entities  matched_entities
 10         98.2407                    94.5625       692304             34.62              17843             18869
 20         98.4284                    95.1349      1384489             69.22           

In [ ]:
# CELL 5 — RARE-TOKEN COMPLEMENTARY BLOCKER

import re
import time
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

print("=" * 100)
print("CELL 5 — RARE-TOKEN COMPLEMENTARY BLOCKER")
print("=" * 100)

# ------------------------------------------------------------
# Required validation objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}. "
        f"Run Cell 3B first."
    )

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

def word_tokens(text):

    if pd.isna(text):
        return []

    text = str(text).lower()

    return [
        token
        for token in re.findall(
            r"[^\W_]+",
            text,
            flags=re.UNICODE
        )
        if len(token) >= 2
    ]

# ------------------------------------------------------------
# Build token document frequency over S1
# ------------------------------------------------------------

print("\nBuilding S1 token statistics...")

token_df = Counter()

for row in VAL_S1.itertuples(
    index=False
):

    tokens = set(
        word_tokens(
            row.business_name
        )
        +
        word_tokens(
            row.business_address
        )
    )

    token_df.update(
        tokens
    )

print(
    "Unique tokens:",
    f"{len(token_df):,}"
)

# ------------------------------------------------------------
# Token -> local S1 indices
# ------------------------------------------------------------

token_index = defaultdict(list)

for idx, row in enumerate(
    VAL_S1.itertuples(
        index=False
    )
):

    tokens = set(
        word_tokens(
            row.business_name
        )
        +
        word_tokens(
            row.business_address
        )
    )

    for token in tokens:

        token_index[
            (
                row.country_n,
                token
            )
        ].append(
            idx
        )

print(
    "Token-country blocks:",
    f"{len(token_index):,}"
)

# ------------------------------------------------------------
# Check posting-size distribution
# ------------------------------------------------------------

posting_sizes = np.asarray(
    [
        len(v)
        for v in token_index.values()
    ],
    dtype=np.int32
)

print(
    "\nPosting statistics:"
)

print(
    "Median:",
    f"{np.median(posting_sizes):.0f}"
)

print(
    "P95:",
    f"{np.percentile(posting_sizes,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(posting_sizes,99):.0f}"
)

print(
    "Maximum:",
    f"{posting_sizes.max():,}"
)

# ------------------------------------------------------------
# Evaluate rare-token Top-N strategies
# ------------------------------------------------------------

TOP_N_VALUES = [
    1,
    2,
    3,
    4
]

# Avoid extremely common tokens
MAX_DF = 1000

results = []

t0_all = time.time()

for top_n in TOP_N_VALUES:

    covered = 0
    candidate_counts = []

    t0 = time.time()

    for row in VAL_TARGET_POOL.itertuples(
        index=False
    ):

        # ----------------------------------------------------
        # Collect name and address tokens
        # ----------------------------------------------------

        name_tokens = set(
            word_tokens(
                row.business_name
            )
        )

        addr_tokens = set(
            word_tokens(
                row.business_address
            )
        )

        all_tokens = (
            name_tokens
            |
            addr_tokens
        )

        # ----------------------------------------------------
        # Keep tokens that actually occur in the S1 country
        # and are not too common.
        # ----------------------------------------------------

        usable = []

        for token in all_tokens:

            df = token_df.get(
                token,
                0
            )

            if df == 0:
                continue

            if df <= MAX_DF:

                usable.append(
                    (
                        df,
                        token
                    )
                )

        # Rarest first
        usable.sort(
            key=lambda x: x[0]
        )

        selected = [
            token
            for _, token
            in usable[:top_n]
        ]

        # ----------------------------------------------------
        # Retrieve S1 candidates
        # ----------------------------------------------------

        candidates = set()

        for token in selected:

            candidates.update(
                token_index.get(
                    (
                        row.country_n,
                        token
                    ),
                    []
                )
            )

        candidate_counts.append(
            len(candidates)
        )

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        true_s1 = target_to_s1.get(
            row.entity_id
        )

        if true_s1 is None:
            continue

        # Need local S1 index
        # Build once below if necessary
        if true_s1 in s1_local_5:
            true_local = s1_local_5[
                true_s1
            ]

            if true_local in candidates:
                covered += 1

    elapsed = (
        time.time() - t0
    )

    counts = np.asarray(
        candidate_counts,
        dtype=np.int32
    )

    results.append({
        "top_n": top_n,

        "recall_pct":
            covered
            /
            len(VAL_TARGET_POOL)
            *
            100,

        "mean_candidates":
            counts.mean(),

        "median":
            np.median(counts),

        "p95":
            np.percentile(
                counts,
                95
            ),

        "p99":
            np.percentile(
                counts,
                99
            ),

        "max":
            counts.max(),

        "qps":
            len(VAL_TARGET_POOL)
            /
            elapsed
    })

    print(
        f"Top-{top_n}: "
        f"recall={covered/len(VAL_TARGET_POOL)*100:.3f}% "
        f"| mean={counts.mean():.2f} "
        f"| p95={np.percentile(counts,95):.0f}"
    )

results5 = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "RARE-TOKEN RESULTS"
)
print(
    "=" * 100
)

print(
    results5.to_string(
        index=False,
        formatters={
            "recall_pct":
                lambda x: f"{x:.4f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median":
                lambda x: f"{x:.0f}",
            "p95":
                lambda x: f"{x:.0f}",
            "p99":
                lambda x: f"{x:.0f}",
            "qps":
                lambda x: f"{x:.2f}"
        }
    )
)

print(
    "\nCELL 5 COMPLETE."
)

CELL 5 — RARE-TOKEN COMPLEMENTARY BLOCKER

Building S1 token statistics...
Unique tokens: 35,553
Token-country blocks: 38,381

Posting statistics:
Median: 1
P95: 13
P99: 65
Maximum: 4,709


NameError: name 's1_local_5' is not defined

In [ ]:
# CELL 5B — RARE-TOKEN COMPLEMENTARY BLOCKER BENCHMARK
# Reuses artifacts already built by Cell 5.

import re
import time
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 5B — RARE-TOKEN COMPLEMENTARY BLOCKER BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Required existing objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "token_df",
    "token_index",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}. "
        f"Run Cell 5 first."
    )

# ------------------------------------------------------------
# S1 ID -> local index
# ------------------------------------------------------------

s1_local_5 = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

def word_tokens_5(text):

    if pd.isna(text):
        return []

    return [
        token
        for token in re.findall(
            r"[^\W_]+",
            str(text).lower(),
            flags=re.UNICODE
        )
        if len(token) >= 2
    ]

# ------------------------------------------------------------
# Evaluate rare-token strategies
# ------------------------------------------------------------

TOP_N_VALUES = [1, 2, 3, 4]

# Keep rare tokens only
MAX_DF = 1000

results5b = []

for top_n in TOP_N_VALUES:

    covered = 0
    candidate_counts = []

    t0 = time.time()

    for row in VAL_TARGET_POOL.itertuples(
        index=False
    ):

        name_tokens = set(
            word_tokens_5(
                row.business_name
            )
        )

        addr_tokens = set(
            word_tokens_5(
                row.business_address
            )
        )

        all_tokens = (
            name_tokens |
            addr_tokens
        )

        usable = []

        for token in all_tokens:

            df = token_df.get(
                token,
                0
            )

            if (
                df > 0
                and
                df <= MAX_DF
            ):

                usable.append(
                    (
                        df,
                        token
                    )
                )

        # Rare tokens first
        usable.sort(
            key=lambda x: x[0]
        )

        selected_tokens = [
            token
            for _, token in usable[:top_n]
        ]

        candidates = set()

        for token in selected_tokens:

            candidates.update(
                token_index.get(
                    (
                        row.country_n,
                        token
                    ),
                    []
                )
            )

        candidate_counts.append(
            len(candidates)
        )

        # Ground truth target -> S1
        true_s1 = target_to_s1.get(
            row.entity_id
        )

        if true_s1 is None:
            continue

        true_local = s1_local_5.get(
            true_s1
        )

        if (
            true_local is not None
            and
            true_local in candidates
        ):

            covered += 1

    elapsed = time.time() - t0

    counts = np.asarray(
        candidate_counts,
        dtype=np.int32
    )

    qps = (
        len(VAL_TARGET_POOL)
        /
        elapsed
        if elapsed > 0
        else 0
    )

    results5b.append({
        "top_n":
            top_n,

        "recall_pct":
            covered
            /
            len(VAL_TARGET_POOL)
            *
            100,

        "mean_candidates":
            counts.mean(),

        "median":
            np.median(counts),

        "p95":
            np.percentile(
                counts,
                95
            ),

        "p99":
            np.percentile(
                counts,
                99
            ),

        "max":
            counts.max(),

        "qps":
            qps
    })

    print(
        f"Top-{top_n}: "
        f"recall={covered/len(VAL_TARGET_POOL)*100:.3f}% "
        f"| mean={counts.mean():.2f} "
        f"| median={np.median(counts):.0f} "
        f"| p95={np.percentile(counts,95):.0f} "
        f"| speed={qps:.1f}/s"
    )

# ------------------------------------------------------------
# Final table
# ------------------------------------------------------------

results5b_df = pd.DataFrame(
    results5b
)

print(
    "\n" + "=" * 100
)
print(
    "RARE-TOKEN RESULTS"
)
print(
    "=" * 100
)

print(
    results5b_df.to_string(
        index=False,
        formatters={
            "recall_pct":
                lambda x: f"{x:.4f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median":
                lambda x: f"{x:.0f}",
            "p95":
                lambda x: f"{x:.0f}",
            "p99":
                lambda x: f"{x:.0f}",
            "qps":
                lambda x: f"{x:.1f}"
        }
    )
)

print(
    "\nCELL 5B COMPLETE."
)

CELL 5B — RARE-TOKEN COMPLEMENTARY BLOCKER BENCHMARK
Top-1: recall=90.680% | mean=3.54 | median=1 | p95=10 | speed=28417.7/s
Top-2: recall=98.586% | mean=13.64 | median=3 | p95=52 | speed=22021.9/s
Top-3: recall=99.747% | mean=34.07 | median=10 | p95=127 | speed=24799.6/s
Top-4: recall=99.925% | mean=71.22 | median=28 | p95=299 | speed=23740.3/s

RARE-TOKEN RESULTS
 top_n recall_pct mean_candidates median p95 p99  max     qps
     1    90.6805            3.54      1  10  50  783 28417.7
     2    98.5859           13.64      3  52 187  987 22021.9
     3    99.7472           34.07     10 127 494 1767 24799.6
     4    99.9249           71.22     28 299 714 1819 23740.3

CELL 5B COMPLETE.


In [ ]:
# CELL 6 — RARE-TOKEN ENTITY COVERAGE + TF-IDF UNION
#
# Evaluates:
#   Rare-token Top-2
#   Rare-token Top-3
#   Rare-token Top-4
#   Rare-Top3 UNION TF-IDF-K50
#   Rare-Top4 UNION TF-IDF-K50
#
# This is retrieval-only. No LightGBM yet.

import re
import time
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 6 — RARE-TOKEN ENTITY COVERAGE + TF-IDF UNION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "target_to_s1",
    "token_df",
    "token_index",
    "retrieved_top100",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# S1 local lookup
# ------------------------------------------------------------

s1_local_6 = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Build TRUE targets by S1
# ------------------------------------------------------------

true_by_s1 = {
    sid: set()
    for sid in VAL_S1["entity_id"]
}

for target_id, sid in target_to_s1.items():

    if sid in true_by_s1:
        true_by_s1[sid].add(
            target_id
        )

matched_entities = sum(
    1
    for x in true_by_s1.values()
    if x
)

total_true_pairs = len(
    target_to_s1
)

print(
    "Representative S1:",
    f"{len(VAL_S1):,}"
)

print(
    "Matched S1:",
    f"{matched_entities:,}"
)

print(
    "True pairs:",
    f"{total_true_pairs:,}"
)

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

def word_tokens_6(text):

    if pd.isna(text):
        return []

    return [
        token
        for token in re.findall(
            r"[^\W_]+",
            str(text).lower(),
            flags=re.UNICODE
        )
        if len(token) >= 2
    ]

# ------------------------------------------------------------
# Generate rare-token candidates once
# ------------------------------------------------------------

rare_candidates = {}

t0 = time.time()

for row in VAL_TARGET_POOL.itertuples(
    index=False
):

    usable = []

    # Name tokens
    name_tokens = set(
        word_tokens_6(
            row.business_name
        )
    )

    # Address tokens
    addr_tokens = set(
        word_tokens_6(
            row.business_address
        )
    )

    all_tokens = (
        name_tokens
        |
        addr_tokens
    )

    for token in all_tokens:

        df = token_df.get(
            token,
            0
        )

        if (
            df > 0
            and
            df <= 1000
        ):

            usable.append(
                (
                    df,
                    token
                )
            )

    usable.sort(
        key=lambda x: x[0]
    )

    # Store candidate local indices for Top-4
    candidates = set()

    for _, token in usable[:4]:

        candidates.update(
            token_index.get(
                (
                    row.country_n,
                    token
                ),
                []
            )
        )

    rare_candidates[
        row.entity_id
    ] = np.fromiter(
        candidates,
        dtype=np.int32
    )

print(
    "\nRare-token candidate generation:",
    f"{time.time() - t0:.2f} sec"
)

# ------------------------------------------------------------
# Evaluate rare Top-2 / Top-3 / Top-4
# ------------------------------------------------------------

results = []

for n_tokens in [
    2,
    3,
    4
]:

    covered_pairs = 0

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    total_edges = 0

    for row in VAL_TARGET_POOL.itertuples(
        index=False
    ):

        # Reconstruct selected rare tokens
        name_tokens = set(
            word_tokens_6(
                row.business_name
            )
        )

        addr_tokens = set(
            word_tokens_6(
                row.business_address
            )
        )

        all_tokens = (
            name_tokens
            |
            addr_tokens
        )

        usable = []

        for token in all_tokens:

            df = token_df.get(
                token,
                0
            )

            if (
                df > 0
                and
                df <= 1000
            ):

                usable.append(
                    (
                        df,
                        token
                    )
                )

        usable.sort(
            key=lambda x: x[0]
        )

        candidates = set()

        for _, token in usable[
            :n_tokens
        ]:

            candidates.update(
                token_index.get(
                    (
                        row.country_n,
                        token
                    ),
                    []
                )
            )

        total_edges += len(
            candidates
        )

        true_s1 = target_to_s1.get(
            row.entity_id
        )

        true_local = (
            s1_local_6.get(
                true_s1
            )
            if true_s1 is not None
            else None
        )

        if (
            true_local is not None
            and
            true_local in candidates
        ):

            covered_pairs += 1

        # For entity-level coverage, add this target
        # to every S1 candidate it retrieved.
        for local_idx in candidates:

            if (
                0 <= local_idx
                < len(VAL_S1)
            ):

                sid = VAL_S1.iloc[
                    int(local_idx)
                ]["entity_id"]

                predicted_by_s1[
                    sid
                ].add(
                    row.entity_id
                )

    complete = 0

    for sid, true_targets in true_by_s1.items():

        if (
            true_targets
            and
            true_targets.issubset(
                predicted_by_s1[sid]
            )
        ):

            complete += 1

    results.append({
        "retrieval":
            f"rare_top{n_tokens}",

        "pair_recall_pct":
            covered_pairs
            /
            total_true_pairs
            *
            100,

        "complete_entity_recall_pct":
            complete
            /
            matched_entities
            *
            100,

        "total_edges":
            total_edges,

        "mean_edges_per_S1":
            total_edges
            /
            len(VAL_S1),

        "complete_entities":
            complete
    })

# ------------------------------------------------------------
# TF-IDF Top-50 union
# ------------------------------------------------------------

for rare_n in [
    3,
    4
]:

    covered_pairs = 0

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    total_edges = 0

    for row in VAL_TARGET_POOL.itertuples(
        index=False
    ):

        # Rare-token candidates
        name_tokens = set(
            word_tokens_6(
                row.business_name
            )
        )

        addr_tokens = set(
            word_tokens_6(
                row.business_address
            )
        )

        all_tokens = (
            name_tokens
            |
            addr_tokens
        )

        usable = []

        for token in all_tokens:

            df = token_df.get(
                token,
                0
            )

            if (
                df > 0
                and
                df <= 1000
            ):

                usable.append(
                    (
                        df,
                        token
                    )
                )

        usable.sort(
            key=lambda x: x[0]
        )

        rare_set = set()

        for _, token in usable[
            :rare_n
        ]:

            rare_set.update(
                token_index.get(
                    (
                        row.country_n,
                        token
                    ),
                    []
                )
            )

        # TF-IDF K50
        tfidf_indices = retrieved_top100.get(
            row.entity_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )[:50]

        combined = (
            rare_set
            |
            set(
                tfidf_indices.tolist()
            )
        )

        total_edges += len(
            combined
        )

        # Ground truth
        true_s1 = target_to_s1.get(
            row.entity_id
        )

        true_local = (
            s1_local_6.get(
                true_s1
            )
            if true_s1 is not None
            else None
        )

        if (
            true_local is not None
            and
            true_local in combined
        ):

            covered_pairs += 1

        for local_idx in combined:

            if (
                0 <= local_idx
                < len(VAL_S1)
            ):

                sid = VAL_S1.iloc[
                    int(local_idx)
                ]["entity_id"]

                predicted_by_s1[
                    sid
                ].add(
                    row.entity_id
                )

    complete = 0

    for sid, true_targets in true_by_s1.items():

        if (
            true_targets
            and
            true_targets.issubset(
                predicted_by_s1[sid]
            )
        ):

            complete += 1

    results.append({
        "retrieval":
            f"rare_top{rare_n}_union_tfidf50",

        "pair_recall_pct":
            covered_pairs
            /
            total_true_pairs
            *
            100,

        "complete_entity_recall_pct":
            complete
            /
            matched_entities
            *
            100,

        "total_edges":
            total_edges,

        "mean_edges_per_S1":
            total_edges
            /
            len(VAL_S1),

        "complete_entities":
            complete
    })

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

results6 = pd.DataFrame(
    results
)

print(
    "\n" + "=" * 100
)
print(
    "RETRIEVAL CEILING RESULTS"
)
print(
    "=" * 100
)

print(
    results6.to_string(
        index=False,
        formatters={
            "pair_recall_pct":
                lambda x: f"{x:.4f}",
            "complete_entity_recall_pct":
                lambda x: f"{x:.4f}",
            "mean_edges_per_S1":
                lambda x: f"{x:.2f}"
        }
    )
)

print(
    "\nCELL 6 COMPLETE."
)

CELL 6 — RARE-TOKEN ENTITY COVERAGE + TF-IDF UNION
Representative S1: 20,000
Matched S1: 18,869
True pairs: 69,231

Rare-token candidate generation: 3.33 sec

RETRIEVAL CEILING RESULTS
              retrieval pair_recall_pct complete_entity_recall_pct  total_edges mean_edges_per_S1  complete_entities
              rare_top2         98.5859                    95.8662       944188             47.21              18089
              rare_top3         99.7472                    99.1944      2358754            117.94              18717
              rare_top4         99.9249                    99.7456      4930622            246.53              18821
rare_top3_union_tfidf50         99.9668                    99.8940      5336802            266.84              18849
rare_top4_union_tfidf50         99.9783                    99.9311      7564401            378.22              18856

CELL 6 COMPLETE.


In [ ]:
# CELL 7 — BUILD RERANKING DATASET FOR NEW RETRIEVAL ARCHITECTURE
#
# Candidate source:
#   rare-token Top-4
#   UNION
#   blocked TF-IDF Top-50
#
# No model training yet.

import time
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 7 — NEW RETRANKING CANDIDATE DATASET")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGETS",
    "target_to_s1",
    "rare_candidates",
    "retrieved_top100",
    "X_target_3b",
    "X_s1_3b"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# S1 lookup
# ------------------------------------------------------------

s1_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Build candidate rows
# ------------------------------------------------------------

rows = []

t0 = time.time()

for q_idx, row in enumerate(
    VAL_TARGETS.itertuples(
        index=False
    )
):

    target_id = row.entity_id

    # --------------------------------------------------------
    # Rare-token Top-4
    # --------------------------------------------------------

    rare_ids = set(
        rare_candidates.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )
    )

    # --------------------------------------------------------
    # TF-IDF Top-50
    # --------------------------------------------------------

    tfidf_indices = (
        retrieved_top100.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )[:50]
    )

    tfidf_ids = set(
        int(x)
        for x in tfidf_indices
    )

    # --------------------------------------------------------
    # Union
    # --------------------------------------------------------

    union_ids = (
        rare_ids
        |
        tfidf_ids
    )

    if not union_ids:
        continue

    union_ids_array = np.fromiter(
        union_ids,
        dtype=np.int32
    )

    # --------------------------------------------------------
    # TF-IDF scores for the UNION
    # --------------------------------------------------------

    scores = (
        X_target_3b[
            q_idx:q_idx + 1
        ]
        @
        X_s1_3b[
            union_ids_array
        ].T
    ).toarray().ravel()

    score_map = {
        int(idx): float(score)
        for idx, score in zip(
            union_ids_array,
            scores
        )
    }

    # TF-IDF rank lookup
    tfidf_rank_map = {
        int(idx): rank + 1
        for rank, idx in enumerate(
            tfidf_indices
        )
    }

    # --------------------------------------------------------
    # Ground truth
    # --------------------------------------------------------

    true_s1 = target_to_s1.get(
        target_id
    )

    # --------------------------------------------------------
    # Candidate rows
    # --------------------------------------------------------

    for local_idx in union_ids:

        sid = VAL_S1.iloc[
            int(local_idx)
        ]["entity_id"]

        rows.append({
            "query_id":
                target_id,

            "s1_id":
                sid,

            "s1_local_idx":
                int(local_idx),

            "label":
                int(
                    sid == true_s1
                ),

            "has_rare":
                int(
                    local_idx in rare_ids
                ),

            "has_tfidf":
                int(
                    local_idx in tfidf_ids
                ),

            "tfidf_score":
                score_map.get(
                    int(local_idx),
                    0.0
                ),

            "tfidf_rank":
                tfidf_rank_map.get(
                    int(local_idx),
                    999
                )
        })

    if (
        (q_idx + 1) % 250 == 0
    ):

        print(
            f"{q_idx + 1:,}/"
            f"{len(VAL_TARGETS):,} "
            f"| rows={len(rows):,}"
        )

# ------------------------------------------------------------
# DataFrame
# ------------------------------------------------------------

df_rerank_candidates = pd.DataFrame(
    rows
)

elapsed = (
    time.time() - t0
)

print(
    "\n" + "=" * 100
)
print(
    "RERANKING CANDIDATE DATASET"
)
print(
    "=" * 100
)

print(
    "Queries:",
    f"{df_rerank_candidates['query_id'].nunique():,}"
)

print(
    "Candidate rows:",
    f"{len(df_rerank_candidates):,}"
)

print(
    "Positive rows:",
    int(
        df_rerank_candidates["label"].sum()
    )
)

print(
    "Negative rows:",
    int(
        (df_rerank_candidates["label"] == 0).sum()
    )
)

print(
    "Positive rate:",
    f"{df_rerank_candidates['label'].mean()*100:.4f}%"
)

candidate_counts_7 = (
    df_rerank_candidates
    .groupby("query_id")
    .size()
    .to_numpy()
)

print(
    "\nCandidates/query:"
)

print(
    "Mean:",
    f"{candidate_counts_7.mean():.2f}"
)

print(
    "Median:",
    f"{np.median(candidate_counts_7):.0f}"
)

print(
    "P95:",
    f"{np.percentile(candidate_counts_7,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(candidate_counts_7,99):.0f}"
)

print(
    "Maximum:",
    f"{candidate_counts_7.max():,}"
)

print(
    "\nBuild time:",
    f"{elapsed:.2f} sec"
)

# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

candidate_path = (
    MODEL_DIR /
    "blocked_tfidf_rare_rerank_candidates_2k.parquet"
)

df_rerank_candidates.to_parquet(
    candidate_path,
    index=False
)

print(
    "\nSaved:",
    candidate_path
)

print(
    "\nCELL 7 COMPLETE."
)

CELL 7 — NEW RETRANKING CANDIDATE DATASET
250/2,000 | rows=27,342
500/2,000 | rows=52,993
750/2,000 | rows=82,047
1,000/2,000 | rows=109,134
1,250/2,000 | rows=136,494
1,500/2,000 | rows=165,542
1,750/2,000 | rows=193,421
2,000/2,000 | rows=223,814

RERANKING CANDIDATE DATASET
Queries: 2,000
Candidate rows: 223,814
Positive rows: 2000
Negative rows: 221814
Positive rate: 0.8936%

Candidates/query:
Mean: 111.91
Median: 69
P95: 331
P99: 809
Maximum: 1,186

Build time: 18.64 sec

Saved: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_rerank_candidates_2k.parquet

CELL 7 COMPLETE.


In [ ]:
# CELL 8 — BUILD NEW RETRANKER FEATURES
#
# Candidate distribution:
#   rare-token Top-4 UNION TF-IDF Top-50
#
# This creates a new feature space specifically for this
# retrieval architecture.

import re
import time
import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("=" * 100)
print("CELL 8 — BUILD NEW RETRANKER FEATURES")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "df_rerank_candidates",
    "VAL_S1",
    "VAL_TARGETS",
    "number_overlap"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        f"Missing: {missing}"
    )

# ------------------------------------------------------------
# Build fast lookup dictionaries
# ------------------------------------------------------------

print("\nBuilding lookup dictionaries...")

s1_name = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["name_n"]
    )
)

s1_addr = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["addr_n"]
    )
)

s1_country = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["country_n"]
    )
)

target_name = dict(
    zip(
        VAL_TARGETS["entity_id"],
        VAL_TARGETS["name_n"]
    )
)

target_addr = dict(
    zip(
        VAL_TARGETS["entity_id"],
        VAL_TARGETS["addr_n"]
    )
)

target_country = dict(
    zip(
        VAL_TARGETS["entity_id"],
        VAL_TARGETS["country_n"]
    )
)

print(
    "S1 records:",
    f"{len(s1_name):,}"
)

print(
    "Target records:",
    f"{len(target_name):,}"
)

# ------------------------------------------------------------
# Feature generation
# ------------------------------------------------------------

df_new = df_rerank_candidates.copy()

feature_rows = []

t0 = time.time()

for i, row in enumerate(
    df_new[
        [
            "query_id",
            "s1_id",
            "label",
            "has_rare",
            "has_tfidf",
            "tfidf_score",
            "tfidf_rank"
        ]
    ].itertuples(index=False)
):

    q_name = target_name.get(
        row.query_id,
        ""
    )

    q_addr = target_addr.get(
        row.query_id,
        ""
    )

    q_country = target_country.get(
        row.query_id,
        ""
    )

    s_name = s1_name.get(
        row.s1_id,
        ""
    )

    s_addr = s1_addr.get(
        row.s1_id,
        ""
    )

    s_country = s1_country.get(
        row.s1_id,
        ""
    )

    # --------------------------------------------------------
    # Base similarity features
    # --------------------------------------------------------

    name_ratio = (
        fuzz.ratio(
            q_name,
            s_name
        )
        /
        100.0
    )

    addr_ratio = (
        fuzz.ratio(
            q_addr,
            s_addr
        )
        /
        100.0
    )

    name_partial = (
        fuzz.partial_ratio(
            q_name,
            s_name
        )
        /
        100.0
    )

    addr_partial = (
        fuzz.partial_ratio(
            q_addr,
            s_addr
        )
        /
        100.0
    )

    name_token = (
        fuzz.token_set_ratio(
            q_name,
            s_name
        )
        /
        100.0
    )

    addr_token = (
        fuzz.token_set_ratio(
            q_addr,
            s_addr
        )
        /
        100.0
    )

    # --------------------------------------------------------
    # Length ratios
    # --------------------------------------------------------

    qn = len(q_name)
    sn = len(s_name)

    qa = len(q_addr)
    sa = len(s_addr)

    name_length_ratio = (
        min(qn, sn) / max(qn, sn)
        if max(qn, sn) > 0
        else 1.0
    )

    addr_length_ratio = (
        min(qa, sa) / max(qa, sa)
        if max(qa, sa) > 0
        else 1.0
    )

    # --------------------------------------------------------
    # Exact / country
    # --------------------------------------------------------

    exact_name = int(
        q_name == s_name
    )

    exact_address = int(
        q_addr == s_addr
    )

    country_match = int(
        q_country == s_country
    )

    # --------------------------------------------------------
    # Number overlap
    # --------------------------------------------------------

    num_overlap = number_overlap(
        q_addr,
        s_addr
    )

    # --------------------------------------------------------
    # Combined similarity
    # VERIFIED exact formula
    # --------------------------------------------------------

    combined_similarity = (
        0.45 * name_ratio
        +
        0.55 * addr_ratio
    )

    # --------------------------------------------------------
    # Retrieval source
    # --------------------------------------------------------

    has_rare = int(
        row.has_rare
    )

    has_tfidf = int(
        row.has_tfidf
    )

    retrieval_channel_count = (
        has_rare
        +
        has_tfidf
    )

    # --------------------------------------------------------
    # Interaction features
    # --------------------------------------------------------

    name_addr_product = (
        name_ratio
        *
        addr_ratio
    )

    name_partial_addr_partial_product = (
        name_partial
        *
        addr_partial
    )

    name_number_product = (
        name_ratio
        *
        num_overlap
    )

    addr_number_product = (
        addr_ratio
        *
        num_overlap
    )

    min_name_addr = min(
        name_ratio,
        addr_ratio
    )

    max_name_addr = max(
        name_ratio,
        addr_ratio
    )

    name_addr_gap = abs(
        name_ratio
        -
        addr_ratio
    )

    partial_gap = abs(
        name_partial
        -
        addr_partial
    )

    strong_name_weak_address = int(
        name_ratio >= 0.80
        and
        addr_ratio < 0.20
    )

    strong_name_no_number = int(
        name_ratio >= 0.80
        and
        num_overlap == 0
    )

    # VERIFIED exact rule
    strong_address = int(
        addr_ratio >= 0.70
    )

    strong_name = int(
        name_ratio >= 0.80
    )

    # For this retrieval architecture:
    address_supported = int(
        addr_ratio >= 0.50
    )

    multi_channel = int(
        retrieval_channel_count >= 2
    )

    balanced_similarity = (
        name_ratio + addr_ratio
    ) / 2.0

    partial_balanced_similarity = (
        name_partial
        +
        addr_partial
    ) / 2.0

    feature_rows.append({
        "name_ratio":
            name_ratio,

        "addr_ratio":
            addr_ratio,

        "name_partial":
            name_partial,

        "addr_partial":
            addr_partial,

        "name_token":
            name_token,

        "addr_token":
            addr_token,

        "name_length_ratio":
            name_length_ratio,

        "addr_length_ratio":
            addr_length_ratio,

        "number_overlap":
            num_overlap,

        "exact_name":
            exact_name,

        "exact_address":
            exact_address,

        "country_match":
            country_match,

        "combined_similarity":
            combined_similarity,

        "has_rare":
            has_rare,

        "has_tfidf":
            has_tfidf,

        "retrieval_channel_count":
            retrieval_channel_count,

        "tfidf_score":
            float(row.tfidf_score),

        "tfidf_rank":
            int(row.tfidf_rank),

        "name_addr_product":
            name_addr_product,

        "name_partial_addr_partial_product":
            name_partial_addr_partial_product,

        "name_number_product":
            name_number_product,

        "addr_number_product":
            addr_number_product,

        "min_name_addr":
            min_name_addr,

        "max_name_addr":
            max_name_addr,

        "name_addr_gap":
            name_addr_gap,

        "partial_gap":
            partial_gap,

        "strong_name_weak_address":
            strong_name_weak_address,

        "strong_name_no_number":
            strong_name_no_number,

        "strong_address":
            strong_address,

        "strong_name":
            strong_name,

        "address_supported":
            address_supported,

        "multi_channel":
            multi_channel,

        "balanced_similarity":
            balanced_similarity,

        "partial_balanced_similarity":
            partial_balanced_similarity
    })

    if (
        (i + 1) % 25_000 == 0
    ):

        elapsed = (
            time.time() - t0
        )

        print(
            f"{i+1:,}/"
            f"{len(df_new):,} "
            f"| speed="
            f"{(i+1)/elapsed:.1f} rows/s"
        )

feature_df = pd.DataFrame(
    feature_rows,
    index=df_new.index
)

# ------------------------------------------------------------
# Attach features
# ------------------------------------------------------------

df_new = pd.concat(
    [
        df_new.reset_index(drop=True),
        feature_df.reset_index(drop=True)
    ],
    axis=1
)

# ------------------------------------------------------------
# Query-relative ranking features
# ------------------------------------------------------------

print(
    "\nBuilding query-relative ranking features..."
)

for col in [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "combined_similarity",
    "tfidf_score"
]:

    df_new[
        f"rank_{col}"
    ] = (
        df_new
        .groupby(
            "query_id"
        )[col]
        .rank(
            method="min",
            ascending=False
        )
    )

    df_new[
        f"query_max_{col}"
    ] = (
        df_new
        .groupby(
            "query_id"
        )[col]
        .transform(
            "max"
        )
    )

# Margins
df_new["tfidf_margin"] = (
    df_new["query_max_tfidf_score"]
    -
    df_new["tfidf_score"]
)

df_new["name_margin"] = (
    df_new["query_max_name_ratio"]
    -
    df_new["name_ratio"]
)

df_new["addr_margin"] = (
    df_new["query_max_addr_ratio"]
    -
    df_new["addr_ratio"]
)

df_new["combined_margin"] = (
    df_new["query_max_combined_similarity"]
    -
    df_new["combined_similarity"]
)

df_new["relative_tfidf_strength"] = (
    df_new["tfidf_score"]
    /
    (
        df_new["query_max_tfidf_score"]
        +
        1e-8
    )
)

df_new["relative_combined_strength"] = (
    df_new["combined_similarity"]
    /
    (
        df_new["query_max_combined_similarity"]
        +
        1e-8
    )
)

# Best-candidate indicators
df_new["is_best_tfidf"] = (
    df_new["rank_tfidf_score"] == 1
).astype(np.int8)

df_new["is_best_name"] = (
    df_new["rank_name_ratio"] == 1
).astype(np.int8)

df_new["is_best_addr"] = (
    df_new["rank_addr_ratio"] == 1
).astype(np.int8)

df_new["is_best_combined"] = (
    df_new["rank_combined_similarity"] == 1
).astype(np.int8)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

feature_cols_new = [
    c
    for c in df_new.columns
    if c not in [
        "query_id",
        "s1_id",
        "s1_local_idx",
        "label"
    ]
]

print(
    "\n" + "=" * 100
)
print(
    "NEW FEATURE DATASET"
)
print(
    "=" * 100
)

print(
    "Rows:",
    f"{len(df_new):,}"
)

print(
    "Features:",
    len(feature_cols_new)
)

print(
    "Positive rows:",
    int(
        df_new["label"].sum()
    )
)

print(
    "Negative rows:",
    int(
        (df_new["label"] == 0).sum()
    )
)

print(
    "\nFeature list:"
)

print(
    feature_cols_new
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

feature_path = (
    MODEL_DIR /
    "blocked_tfidf_rare_rerank_features_2k.parquet"
)

df_new.to_parquet(
    feature_path,
    index=False
)

print(
    "\nSaved:",
    feature_path
)

print(
    "\nCELL 8 COMPLETE."
)

CELL 8 — BUILD NEW RETRANKER FEATURES

Building lookup dictionaries...
S1 records: 20,000
Target records: 2,000
25,000/223,814 | speed=33823.0 rows/s
50,000/223,814 | speed=41261.5 rows/s
75,000/223,814 | speed=38931.3 rows/s
100,000/223,814 | speed=36652.4 rows/s
125,000/223,814 | speed=35390.6 rows/s
150,000/223,814 | speed=34947.8 rows/s
175,000/223,814 | speed=35725.8 rows/s
200,000/223,814 | speed=36285.6 rows/s

Building query-relative ranking features...


ValueError: Cannot set a DataFrame with multiple columns to the single column rank_tfidf_score

In [ ]:
# CELL 8B — RESTORE VERIFIED number_overlap + BUILD NEW RERANKER FEATURES

import re

def number_overlap(a, b):
    """
    Count shared numeric tokens between two normalized addresses.
    Example:
        "12mainstreet45" vs "45mainstreet12" -> 2
    """
    if not a or not b:
        return 0

    nums_a = set(
        re.findall(
            r"\d+",
            str(a)
        )
    )

    nums_b = set(
        re.findall(
            r"\d+",
            str(b)
        )
    )

    return len(
        nums_a & nums_b
    )

print(
    "number_overlap restored."
)

print(
    "Smoke test:",
    number_overlap(
        "12mainstreet45",
        "45mainstreet12"
    )
)

print(
    "\nNow rerun CELL 8 unchanged."
)

number_overlap restored.
Smoke test: 2

Now rerun CELL 8 unchanged.


In [ ]:
# CELL 8C — FIX DUPLICATE COLUMNS + FINISH RANKING FEATURES

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 8C — FIX DUPLICATE COLUMNS")
print("=" * 100)

# ------------------------------------------------------------
# Check current dataframe
# ------------------------------------------------------------

if "df_new" not in globals():
    raise RuntimeError(
        "df_new does not exist. Run Cell 8 again."
    )

print(
    "Rows before cleanup:",
    len(df_new)
)

duplicate_columns = (
    df_new.columns[
        df_new.columns.duplicated()
    ]
    .tolist()
)

print(
    "Duplicate columns:",
    duplicate_columns
)

# Keep the FIRST copy of every duplicate column.
# The original and recomputed versions are identical for
# the shared fields used here.
df_new = df_new.loc[
    :,
    ~df_new.columns.duplicated()
].copy()

print(
    "Rows after cleanup:",
    len(df_new)
)

print(
    "Unique columns:",
    len(df_new.columns)
)

# ------------------------------------------------------------
# Query-relative ranking features
# ------------------------------------------------------------

print(
    "\nBuilding query-relative ranking features..."
)

rank_columns = [
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "combined_similarity",
    "tfidf_score"
]

for col in rank_columns:

    rank_col = (
        f"rank_{col}"
    )

    max_col = (
        f"query_max_{col}"
    )

    df_new[rank_col] = (
        df_new
        .groupby("query_id")[col]
        .rank(
            method="min",
            ascending=False
        )
    )

    df_new[max_col] = (
        df_new
        .groupby("query_id")[col]
        .transform("max")
    )

# ------------------------------------------------------------
# Margins
# ------------------------------------------------------------

df_new["tfidf_margin"] = (
    df_new["query_max_tfidf_score"]
    -
    df_new["tfidf_score"]
)

df_new["name_margin"] = (
    df_new["query_max_name_ratio"]
    -
    df_new["name_ratio"]
)

df_new["addr_margin"] = (
    df_new["query_max_addr_ratio"]
    -
    df_new["addr_ratio"]
)

df_new["combined_margin"] = (
    df_new["query_max_combined_similarity"]
    -
    df_new["combined_similarity"]
)

# ------------------------------------------------------------
# Relative strengths
# ------------------------------------------------------------

df_new["relative_tfidf_strength"] = (
    df_new["tfidf_score"]
    /
    (
        df_new["query_max_tfidf_score"]
        + 1e-8
    )
)

df_new["relative_combined_strength"] = (
    df_new["combined_similarity"]
    /
    (
        df_new["query_max_combined_similarity"]
        + 1e-8
    )
)

# ------------------------------------------------------------
# Best-candidate indicators
# ------------------------------------------------------------

df_new["is_best_tfidf"] = (
    df_new["rank_tfidf_score"] == 1
).astype(np.int8)

df_new["is_best_name"] = (
    df_new["rank_name_ratio"] == 1
).astype(np.int8)

df_new["is_best_addr"] = (
    df_new["rank_addr_ratio"] == 1
).astype(np.int8)

df_new["is_best_combined"] = (
    df_new["rank_combined_similarity"] == 1
).astype(np.int8)

# ------------------------------------------------------------
# Final feature list
# ------------------------------------------------------------

feature_cols_new = [
    c
    for c in df_new.columns
    if c not in [
        "query_id",
        "s1_id",
        "s1_local_idx",
        "label"
    ]
]

print(
    "\n" + "=" * 100
)
print(
    "NEW RERANKER DATASET"
)
print(
    "=" * 100
)

print(
    "Rows:",
    f"{len(df_new):,}"
)

print(
    "Features:",
    len(feature_cols_new)
)

print(
    "Positive rows:",
    int(
        df_new["label"].sum()
    )
)

print(
    "Negative rows:",
    int(
        (df_new["label"] == 0).sum()
    )
)

print(
    "\nFeature columns:"
)

print(
    feature_cols_new
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

feature_path = (
    MODEL_DIR
    /
    "blocked_tfidf_rare_rerank_features_2k.parquet"
)

df_new.to_parquet(
    feature_path,
    index=False
)

print(
    "\nSaved:",
    feature_path
)

print(
    "\nCELL 8C COMPLETE."
)

CELL 8C — FIX DUPLICATE COLUMNS
Rows before cleanup: 223814
Duplicate columns: ['has_rare', 'has_tfidf', 'tfidf_score', 'tfidf_rank']
Rows after cleanup: 223814
Unique columns: 48

Building query-relative ranking features...

NEW RERANKER DATASET
Rows: 223,814
Features: 56
Positive rows: 2000
Negative rows: 221814

Feature columns:
['has_rare', 'has_tfidf', 'tfidf_score', 'tfidf_rank', 'name_ratio', 'addr_ratio', 'name_partial', 'addr_partial', 'name_token', 'addr_token', 'name_length_ratio', 'addr_length_ratio', 'number_overlap', 'exact_name', 'exact_address', 'country_match', 'combined_similarity', 'retrieval_channel_count', 'name_addr_product', 'name_partial_addr_partial_product', 'name_number_product', 'addr_number_product', 'min_name_addr', 'max_name_addr', 'name_addr_gap', 'partial_gap', 'strong_name_weak_address', 'strong_name_no_number', 'strong_address', 'strong_name', 'address_supported', 'multi_channel', 'balanced_similarity', 'partial_balanced_similarity', 'rank_name_ratio'

In [ ]:
# CELL 9 — TRAIN NEW HIGH-RECALL RERANKER

import time
import numpy as np
import pandas as pd
import lightgbm as lgb

print("=" * 100)
print("CELL 9 — TRAIN NEW HIGH-RECALL RERANKER")
print("=" * 100)

# ------------------------------------------------------------
# Required dataframe
# ------------------------------------------------------------

if "df_new" not in globals():
    raise RuntimeError(
        "df_new is missing. Run Cell 8C first."
    )

# ------------------------------------------------------------
# Feature columns
# ------------------------------------------------------------

excluded = {
    "query_id",
    "s1_id",
    "s1_local_idx",
    "label"
}

feature_cols_9 = [
    c
    for c in df_new.columns
    if c not in excluded
]

print(
    "Rows:",
    f"{len(df_new):,}"
)

print(
    "Features:",
    len(feature_cols_9)
)

# ------------------------------------------------------------
# Build matrices
# ------------------------------------------------------------

X = (
    df_new[
        feature_cols_9
    ]
    .astype(np.float32)
)

y = (
    df_new["label"]
    .astype(np.int8)
)

# ------------------------------------------------------------
# Same query-level split used previously
# 75% queries train / 25% validation
# ------------------------------------------------------------

query_ids = (
    df_new["query_id"]
    .drop_duplicates()
    .to_numpy()
)

rng = np.random.default_rng(42)
rng.shuffle(query_ids)

split = int(
    len(query_ids) * 0.75
)

train_queries = set(
    query_ids[:split]
)

valid_queries = set(
    query_ids[split:]
)

train_mask = (
    df_new["query_id"]
    .isin(train_queries)
)

valid_mask = (
    df_new["query_id"]
    .isin(valid_queries)
)

X_train = X.loc[
    train_mask
]

y_train = y.loc[
    train_mask
]

X_valid = X.loc[
    valid_mask
]

y_valid = y.loc[
    valid_mask
]

print(
    "\nTrain queries:",
    len(train_queries)
)

print(
    "Validation queries:",
    len(valid_queries)
)

print(
    "Train rows:",
    f"{len(X_train):,}"
)

print(
    "Validation rows:",
    f"{len(X_valid):,}"
)

print(
    "Train positives:",
    int(y_train.sum())
)

print(
    "Validation positives:",
    int(y_valid.sum())
)

print(
    "Train positive rate:",
    f"{y_train.mean()*100:.4f}%"
)

print(
    "Validation positive rate:",
    f"{y_valid.mean()*100:.4f}%"
)

# ------------------------------------------------------------
# LightGBM model
# ------------------------------------------------------------

model9 = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=3000,

    learning_rate=0.025,

    num_leaves=31,

    max_depth=-1,

    min_child_samples=25,

    subsample=0.85,

    colsample_bytree=0.90,

    reg_alpha=0.20,

    reg_lambda=2.0,

    random_state=42,

    n_jobs=-1
)

print(
    "\nTraining LightGBM..."
)

t0 = time.time()

model9.fit(
    X_train,
    y_train,

    eval_set=[
        (X_valid, y_valid)
    ],

    eval_metric=[
        "auc",
        "binary_logloss"
    ],

    callbacks=[
        lgb.early_stopping(
            150,
            verbose=False
        )
    ]
)

train_time = (
    time.time() - t0
)

print(
    "Training time:",
    f"{train_time:.2f} sec"
)

print(
    "Best iteration:",
    model9.best_iteration_
)

# ------------------------------------------------------------
# Validation predictions
# ------------------------------------------------------------

valid_scores = model9.predict_proba(
    X_valid,
    num_iteration=model9.best_iteration_
)[:, 1]

print(
    "Score range:",
    f"{valid_scores.min():.6f}",
    "→",
    f"{valid_scores.max():.6f}"
)

# ------------------------------------------------------------
# Entity/query-level Macro F0.5
# ------------------------------------------------------------

def f05_from_pr(
    precision,
    recall
):

    if (
        precision <= 0
        or
        recall <= 0
    ):
        return 0.0

    return (
        1.25
        *
        precision
        *
        recall
        /
        (
            0.25 * precision
            +
            recall
        )
    )


valid_df = df_new.loc[
    valid_mask,
    [
        "query_id",
        "s1_id",
        "label"
    ]
].copy()

valid_df["score"] = (
    valid_scores
)

def evaluate_threshold(
    threshold
):

    valid_df["pred"] = (
        valid_df["score"]
        >= threshold
    )

    f05_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for query_id, group in valid_df.groupby(
        "query_id",
        sort=False
    ):

        true_ids = set(
            group.loc[
                group["label"] == 1,
                "s1_id"
            ]
        )

        pred_ids = set(
            group.loc[
                group["pred"],
                "s1_id"
            ]
        )

        tp = len(
            true_ids
            &
            pred_ids
        )

        fp = len(
            pred_ids
            -
            true_ids
        )

        fn = len(
            true_ids
            -
            pred_ids
        )

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else (
                1.0
                if len(true_ids) == 0
                else 0.0
            )
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 1.0
        )

        f05_values.append(
            f05_from_pr(
                precision,
                recall
            )
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    macro_f05 = float(
        np.mean(
            f05_values
        )
    )

    precision_micro = (
        total_tp
        /
        (total_tp + total_fp)
        if (total_tp + total_fp) > 0
        else 0.0
    )

    recall_micro = (
        total_tp
        /
        (total_tp + total_fn)
        if (total_tp + total_fn) > 0
        else 0.0
    )

    return (
        macro_f05,
        precision_micro,
        recall_micro,
        total_tp,
        total_fp,
        total_fn
    )


# ------------------------------------------------------------
# Threshold sweep
# ------------------------------------------------------------

thresholds = np.arange(
    0.01,
    0.951,
    0.005
)

threshold_results = []

for threshold in thresholds:

    (
        macro_f05,
        precision,
        recall,
        tp,
        fp,
        fn
    ) = evaluate_threshold(
        float(threshold)
    )

    threshold_results.append({
        "threshold":
            float(threshold),

        "macro_f05":
            macro_f05,

        "precision":
            precision,

        "recall":
            recall,

        "tp":
            tp,

        "fp":
            fp,

        "fn":
            fn
    })

threshold_df = pd.DataFrame(
    threshold_results
)

best_idx = (
    threshold_df["macro_f05"]
    .idxmax()
)

best_row = (
    threshold_df.loc[
        best_idx
    ]
)

best_threshold_9 = float(
    best_row["threshold"]
)

best_f05_9 = float(
    best_row["macro_f05"]
)

print(
    "\n" + "=" * 100
)
print(
    "NEW RERANKER RESULT"
)
print(
    "=" * 100
)

print(
    "Best threshold:",
    f"{best_threshold_9:.3f}"
)

print(
    "Macro F0.5:",
    f"{best_f05_9:.6f}"
)

print(
    "Precision:",
    f"{best_row['precision']:.6f}"
)

print(
    "Recall:",
    f"{best_row['recall']:.6f}"
)

print(
    "TP:",
    int(best_row["tp"])
)

print(
    "FP:",
    int(best_row["fp"])
)

print(
    "FN:",
    int(best_row["fn"])
)

print(
    "\nComparison:"
)

print(
    "Old development reranker:",
    "0.973279"
)

print(
    "New retrieval-aware reranker:",
    f"{best_f05_9:.6f}"
)

print(
    "Difference:",
    f"{best_f05_9 - 0.973279:+.6f}"
)

# ------------------------------------------------------------
# Useful thresholds
# ------------------------------------------------------------

print(
    "\nThreshold comparison:"
)

for t in [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:

    row = threshold_df.loc[
        threshold_df["threshold"]
        .sub(t)
        .abs()
        .idxmin()
    ]

    print(
        f"threshold={row['threshold']:.3f} "
        f"F0.5={row['macro_f05']:.6f} "
        f"P={row['precision']:.4f} "
        f"R={row['recall']:.4f}"
    )

# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

model9_path = (
    MODEL_DIR
    /
    "blocked_tfidf_rare_reranker_lgbm.txt"
)

threshold9_path = (
    MODEL_DIR
    /
    "blocked_tfidf_rare_reranker_threshold.txt"
)

model9.booster_.save_model(
    str(model9_path)
)

with open(
    threshold9_path,
    "w"
) as f:
    f.write(
        str(best_threshold_9)
    )

threshold_table_path = (
    MODEL_DIR
    /
    "blocked_tfidf_rare_reranker_thresholds.csv"
)

threshold_df.to_csv(
    threshold_table_path,
    index=False
)

print(
    "\nSaved model:",
    model9_path
)

print(
    "Saved threshold:",
    threshold9_path
)

print(
    "Saved threshold sweep:",
    threshold_table_path
)

print(
    "\nCELL 9 COMPLETE."
)

CELL 9 — TRAIN NEW HIGH-RECALL RERANKER
Rows: 223,814
Features: 56

Train queries: 1500
Validation queries: 500
Train rows: 170,473
Validation rows: 53,341
Train positives: 1500
Validation positives: 500
Train positive rate: 0.8799%
Validation positive rate: 0.9374%

Training LightGBM...
[LightGBM] [Info] Number of positive: 1500, number of negative: 168973
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.088187 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9612
[LightGBM] [Info] Number of data points in the train set: 170473, number of used features: 55
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.008799 -> initscore=-4.724274
[LightGBM] [Info] Start training from score -4.724274
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

In [ ]:
# CELL 10 — TARGET EXCLUSIVITY POST-RANKING TEST

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 10 — TARGET EXCLUSIVITY POST-RANKING TEST")
print("=" * 100)

# ------------------------------------------------------------
# Reconstruct validation dataframe
# ------------------------------------------------------------

eval_df = df_new.loc[
    valid_mask,
    [
        "query_id",
        "s1_id",
        "label"
    ]
].copy()

eval_df["score"] = (
    valid_scores
)

BASE_THRESHOLD = best_threshold_9

print(
    "Base threshold:",
    BASE_THRESHOLD
)

# ------------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------------

def f05_cell10(
    precision,
    recall
):

    if precision <= 0 or recall <= 0:
        return 0.0

    return (
        1.25
        * precision
        * recall
        /
        (
            0.25 * precision
            +
            recall
        )
    )


def evaluate_exclusivity(
    threshold,
    top_k
):

    work = eval_df.copy()

    # Candidates above threshold
    work["eligible"] = (
        work["score"] >= threshold
    )

    work = work[
        work["eligible"]
    ].copy()

    # --------------------------------------------------------
    # Target exclusivity:
    # for each target/query keep top-K S1 candidates
    # --------------------------------------------------------

    work = (
        work
        .sort_values(
            [
                "query_id",
                "score"
            ],
            ascending=[
                True,
                False
            ]
        )
    )

    work["target_rank"] = (
        work
        .groupby("query_id")
        .cumcount()
        + 1
    )

    work = work[
        work["target_rank"] <= top_k
    ]

    # --------------------------------------------------------
    # Evaluate each target query
    # --------------------------------------------------------

    scores = []

    tp = 0
    fp = 0
    fn = 0

    for query_id, group in eval_df.groupby(
        "query_id",
        sort=False
    ):

        true_ids = set(
            group.loc[
                group["label"] == 1,
                "s1_id"
            ]
        )

        pred_group = work[
            work["query_id"] == query_id
        ]

        pred_ids = set(
            pred_group["s1_id"]
        )

        q_tp = len(
            true_ids & pred_ids
        )

        q_fp = len(
            pred_ids - true_ids
        )

        q_fn = len(
            true_ids - pred_ids
        )

        q_precision = (
            q_tp / (q_tp + q_fp)
            if q_tp + q_fp > 0
            else 0.0
        )

        q_recall = (
            q_tp / (q_tp + q_fn)
            if q_tp + q_fn > 0
            else 1.0
        )

        scores.append(
            f05_cell10(
                q_precision,
                q_recall
            )
        )

        tp += q_tp
        fp += q_fp
        fn += q_fn

    return {
        "f05": np.mean(scores),
        "precision": (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        ),
        "recall": (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        ),
        "tp": tp,
        "fp": fp,
        "fn": fn
    }


# ------------------------------------------------------------
# Test Top-1 / Top-2 / Top-3 target exclusivity
# ------------------------------------------------------------

rows = []

for top_k in [
    1,
    2,
    3,
    4
]:

    result = evaluate_exclusivity(
        BASE_THRESHOLD,
        top_k
    )

    rows.append({
        "top_k": top_k,
        **result
    })

results10 = pd.DataFrame(
    rows
)

print(
    "\n" + "=" * 100
)
print(
    "TARGET-EXCLUSIVITY RESULTS"
)
print(
    "=" * 100
)

print(
    results10.to_string(
        index=False,
        formatters={
            "f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

print(
    "\nBaseline:",
    f"{best_f05_9:.6f}"
)

print(
    "\nCELL 10 COMPLETE."
)

CELL 10 — TARGET EXCLUSIVITY POST-RANKING TEST
Base threshold: 0.11499999999999998

TARGET-EXCLUSIVITY RESULTS
 top_k      f05 precision   recall  tp  fp  fn
     1 0.988000  0.989980 0.988000 494   5   6
     2 0.984667  0.976331 0.990000 495  12   5
     3 0.985436  0.974460 0.992000 496  13   4
     4 0.985436  0.972549 0.992000 496  14   4

Baseline: 0.985436

CELL 10 COMPLETE.


In [ ]:
# CELL 11 — FULL 20K S1 VALIDATION
#
# Retrieval:
#   rare-token Top-4 UNION blocked-TFIDF Top-50
#
# Reranker:
#   blocked_tfidf_rare_reranker_lgbm
#
# Decision:
#   target exclusivity Top-1
#
# Evaluation:
#   ALL 20,000 S1 entities
#   including 1,131 no-match entities

import os
import re
import gc
import time
import numpy as np
import pandas as pd

from rapidfuzz import fuzz
import lightgbm as lgb

print("=" * 100)
print("CELL 11 — FULL 20K S1 VALIDATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "rare_candidates",
    "retrieved_top100",
    "MODEL_DIR",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Load model if necessary
# ------------------------------------------------------------

if "model9" not in globals():

    model9_path = (
        MODEL_DIR
        /
        "blocked_tfidf_rare_reranker_lgbm.txt"
    )

    print(
        "\nLoading saved reranker..."
    )

    model9 = lgb.Booster(
        model_file=str(
            model9_path
        )
    )

print(
    "Reranker trees:",
    model9.num_trees()
)

print(
    "Reranker features:",
    model9.num_feature()
)

# ------------------------------------------------------------
# Feature column order
# ------------------------------------------------------------

feature_cols_11 = [
    c
    for c in df_new.columns
    if c not in [
        "query_id",
        "s1_id",
        "s1_local_idx",
        "label"
    ]
]

if len(feature_cols_11) != model9.num_feature():

    raise RuntimeError(
        f"Feature mismatch: "
        f"dataset has {len(feature_cols_11)}, "
        f"model expects {model9.num_feature()}"
    )

print(
    "Feature count verified:",
    len(feature_cols_11)
)

# ------------------------------------------------------------
# Lookup dictionaries
# ------------------------------------------------------------

s1_name_11 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["name_n"]
    )
)

s1_addr_11 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["addr_n"]
    )
)

s1_country_11 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["country_n"]
    )
)

target_name_11 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["name_n"]
    )
)

target_addr_11 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["addr_n"]
    )
)

target_country_11 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["country_n"]
    )
)

# ------------------------------------------------------------
# S1 local -> ID
# ------------------------------------------------------------

s1_local_to_id = {
    i: sid
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Checkpoint directory
# ------------------------------------------------------------

checkpoint_dir = (
    MODEL_DIR
    /
    "full_validation_target_best_parts"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Process in target chunks
# ------------------------------------------------------------

CHUNK_TARGETS = 2_000

target_ids_all = (
    VAL_TARGET_POOL[
        "entity_id"
    ]
    .tolist()
)

part_paths = []

overall_start = time.time()

for start in range(
    0,
    len(target_ids_all),
    CHUNK_TARGETS
):

    end = min(
        start + CHUNK_TARGETS,
        len(target_ids_all)
    )

    part_id = (
        start
        //
        CHUNK_TARGETS
    )

    part_path = (
        checkpoint_dir
        /
        f"part_{part_id:03d}.parquet"
    )

    # --------------------------------------------------------
    # Already completed?
    # --------------------------------------------------------

    if part_path.exists():

        print(
            f"\n[{start:,}-{end:,}] "
            f"checkpoint exists → skipping"
        )

        part_paths.append(
            part_path
        )

        continue

    chunk_target_ids = target_ids_all[
        start:end
    ]

    chunk_rows = []

    t0 = time.time()

    # --------------------------------------------------------
    # Build candidate features
    # --------------------------------------------------------

    for target_id in chunk_target_ids:

        q_name = target_name_11.get(
            target_id,
            ""
        )

        q_addr = target_addr_11.get(
            target_id,
            ""
        )

        q_country = target_country_11.get(
            target_id,
            ""
        )

        # --------------------------------------------
        # Rare Top-4
        # --------------------------------------------

        rare_ids = set(
            rare_candidates.get(
                target_id,
                np.empty(
                    0,
                    dtype=np.int32
                )
            )
        )

        # --------------------------------------------
        # TF-IDF Top-50
        # --------------------------------------------

        tfidf_indices = (
            retrieved_top100.get(
                target_id,
                np.empty(
                    0,
                    dtype=np.int32
                )
            )[:50]
        )

        tfidf_ids = set(
            int(x)
            for x in tfidf_indices
        )

        # --------------------------------------------
        # Candidate union
        # --------------------------------------------

        candidates = (
            rare_ids
            |
            tfidf_ids
        )

        if not candidates:
            continue

        # --------------------------------------------
        # TF-IDF rank map
        # --------------------------------------------

        tfidf_rank_map = {
            int(idx): rank + 1
            for rank, idx in enumerate(
                tfidf_indices
            )
        }

        # --------------------------------------------
        # TF-IDF score map
        #
        # Retrieved Top-100 scores are not stored
        # separately, so use cosine similarity from
        # the validation TF-IDF matrices.
        # --------------------------------------------

        candidate_array = np.fromiter(
            candidates,
            dtype=np.int32
        )

        # Find target row
        q_pos = (
            VAL_TARGET_POOL.index[
                VAL_TARGET_POOL["entity_id"]
                == target_id
            ]
        )

        if len(q_pos) != 1:
            raise RuntimeError(
                f"Target lookup failure: {target_id}"
            )

        q_pos = int(
            q_pos[0]
        )

        scores = (
            X_target_full[
                q_pos:q_pos + 1
            ]
            @
            X_s1_3b[
                candidate_array
            ].T
        ).toarray().ravel()

        tfidf_score_map = {
            int(idx): float(score)
            for idx, score in zip(
                candidate_array,
                scores
            )
        }

        # --------------------------------------------
        # Candidate feature rows
        # --------------------------------------------

        for local_idx in candidates:

            s1_id = s1_local_to_id[
                int(local_idx)
            ]

            s_name = s1_name_11.get(
                s1_id,
                ""
            )

            s_addr = s1_addr_11.get(
                s1_id,
                ""
            )

            s_country = s1_country_11.get(
                s1_id,
                ""
            )

            # ------------------------------
            # Similarity
            # ------------------------------

            name_ratio = (
                fuzz.ratio(
                    q_name,
                    s_name
                )
                /
                100.0
            )

            addr_ratio = (
                fuzz.ratio(
                    q_addr,
                    s_addr
                )
                /
                100.0
            )

            name_partial = (
                fuzz.partial_ratio(
                    q_name,
                    s_name
                )
                /
                100.0
            )

            addr_partial = (
                fuzz.partial_ratio(
                    q_addr,
                    s_addr
                )
                /
                100.0
            )

            name_token = (
                fuzz.token_set_ratio(
                    q_name,
                    s_name
                )
                /
                100.0
            )

            addr_token = (
                fuzz.token_set_ratio(
                    q_addr,
                    s_addr
                )
                /
                100.0
            )

            # ------------------------------
            # Length
            # ------------------------------

            qn = len(q_name)
            sn = len(s_name)

            qa = len(q_addr)
            sa = len(s_addr)

            name_length_ratio = (
                min(qn, sn)
                /
                max(qn, sn)
                if max(qn, sn) > 0
                else 1.0
            )

            addr_length_ratio = (
                min(qa, sa)
                /
                max(qa, sa)
                if max(qa, sa) > 0
                else 1.0
            )

            # ------------------------------
            # Exact / country
            # ------------------------------

            exact_name = int(
                q_name == s_name
            )

            exact_address = int(
                q_addr == s_addr
            )

            country_match = int(
                q_country == s_country
            )

            # ------------------------------
            # Numeric overlap
            # ------------------------------

            q_nums = set(
                re.findall(
                    r"\d+",
                    q_addr
                )
            )

            s_nums = set(
                re.findall(
                    r"\d+",
                    s_addr
                )
            )

            number_overlap_value = len(
                q_nums & s_nums
            )

            # ------------------------------
            # Combined
            # ------------------------------

            combined_similarity = (
                0.45 * name_ratio
                +
                0.55 * addr_ratio
            )

            # ------------------------------
            # Retrieval
            # ------------------------------

            has_rare = int(
                int(local_idx)
                in
                rare_ids
            )

            has_tfidf = int(
                int(local_idx)
                in
                tfidf_ids
            )

            retrieval_channel_count = (
                has_rare
                +
                has_tfidf
            )

            tfidf_score = (
                tfidf_score_map.get(
                    int(local_idx),
                    0.0
                )
            )

            tfidf_rank = (
                tfidf_rank_map.get(
                    int(local_idx),
                    999
                )
            )

            # ------------------------------
            # Interactions
            # ------------------------------

            name_addr_product = (
                name_ratio
                *
                addr_ratio
            )

            name_partial_addr_partial_product = (
                name_partial
                *
                addr_partial
            )

            name_number_product = (
                name_ratio
                *
                number_overlap_value
            )

            addr_number_product = (
                addr_ratio
                *
                number_overlap_value
            )

            min_name_addr = min(
                name_ratio,
                addr_ratio
            )

            max_name_addr = max(
                name_ratio,
                addr_ratio
            )

            name_addr_gap = abs(
                name_ratio
                -
                addr_ratio
            )

            partial_gap = abs(
                name_partial
                -
                addr_partial
            )

            strong_name_weak_address = int(
                name_ratio >= 0.80
                and
                addr_ratio < 0.20
            )

            strong_name_no_number = int(
                name_ratio >= 0.80
                and
                number_overlap_value == 0
            )

            strong_address = int(
                addr_ratio >= 0.70
            )

            strong_name = int(
                name_ratio >= 0.80
            )

            address_supported = int(
                addr_ratio >= 0.50
            )

            multi_channel = int(
                retrieval_channel_count >= 2
            )

            balanced_similarity = (
                name_ratio
                +
                addr_ratio
            ) / 2.0

            partial_balanced_similarity = (
                name_partial
                +
                addr_partial
            ) / 2.0

            chunk_rows.append({
                "query_id": target_id,
                "s1_id": s1_id,

                "has_rare": has_rare,
                "has_tfidf": has_tfidf,
                "tfidf_score": tfidf_score,
                "tfidf_rank": tfidf_rank,

                "name_ratio": name_ratio,
                "addr_ratio": addr_ratio,
                "name_partial": name_partial,
                "addr_partial": addr_partial,
                "name_token": name_token,
                "addr_token": addr_token,
                "name_length_ratio": name_length_ratio,
                "addr_length_ratio": addr_length_ratio,
                "number_overlap": number_overlap_value,
                "exact_name": exact_name,
                "exact_address": exact_address,
                "country_match": country_match,
                "combined_similarity": combined_similarity,

                "retrieval_channel_count":
                    retrieval_channel_count,

                "name_addr_product":
                    name_addr_product,

                "name_partial_addr_partial_product":
                    name_partial_addr_partial_product,

                "name_number_product":
                    name_number_product,

                "addr_number_product":
                    addr_number_product,

                "min_name_addr":
                    min_name_addr,

                "max_name_addr":
                    max_name_addr,

                "name_addr_gap":
                    name_addr_gap,

                "partial_gap":
                    partial_gap,

                "strong_name_weak_address":
                    strong_name_weak_address,

                "strong_name_no_number":
                    strong_name_no_number,

                "strong_address":
                    strong_address,

                "strong_name":
                    strong_name,

                "address_supported":
                    address_supported,

                "multi_channel":
                    multi_channel,

                "balanced_similarity":
                    balanced_similarity,

                "partial_balanced_similarity":
                    partial_balanced_similarity
            })

    # --------------------------------------------------------
    # Convert chunk to DataFrame
    # --------------------------------------------------------

    chunk_df = pd.DataFrame(
        chunk_rows
    )

    del chunk_rows

    if chunk_df.empty:

        print(
            f"[{start:,}-{end:,}] "
            "no candidates"
        )

        continue

    # --------------------------------------------------------
    # Query-relative features
    # --------------------------------------------------------

    for col in [
        "name_ratio",
        "addr_ratio",
        "name_partial",
        "addr_partial",
        "combined_similarity",
        "tfidf_score"
    ]:

        chunk_df[
            f"rank_{col}"
        ] = (
            chunk_df
            .groupby(
                "query_id"
            )[col]
            .rank(
                method="min",
                ascending=False
            )
        )

        chunk_df[
            f"query_max_{col}"
        ] = (
            chunk_df
            .groupby(
                "query_id"
            )[col]
            .transform(
                "max"
            )
        )

    chunk_df["tfidf_margin"] = (
        chunk_df["query_max_tfidf_score"]
        -
        chunk_df["tfidf_score"]
    )

    chunk_df["name_margin"] = (
        chunk_df["query_max_name_ratio"]
        -
        chunk_df["name_ratio"]
    )

    chunk_df["addr_margin"] = (
        chunk_df["query_max_addr_ratio"]
        -
        chunk_df["addr_ratio"]
    )

    chunk_df["combined_margin"] = (
        chunk_df["query_max_combined_similarity"]
        -
        chunk_df["combined_similarity"]
    )

    chunk_df["relative_tfidf_strength"] = (
        chunk_df["tfidf_score"]
        /
        (
            chunk_df["query_max_tfidf_score"]
            +
            1e-8
        )
    )

    chunk_df["relative_combined_strength"] = (
        chunk_df["combined_similarity"]
        /
        (
            chunk_df[
                "query_max_combined_similarity"
            ]
            +
            1e-8
        )
    )

    chunk_df["is_best_tfidf"] = (
        chunk_df["rank_tfidf_score"] == 1
    ).astype(np.int8)

    chunk_df["is_best_name"] = (
        chunk_df["rank_name_ratio"] == 1
    ).astype(np.int8)

    chunk_df["is_best_addr"] = (
        chunk_df["rank_addr_ratio"] == 1
    ).astype(np.int8)

    chunk_df["is_best_combined"] = (
        chunk_df[
            "rank_combined_similarity"
        ]
        == 1
    ).astype(np.int8)

    # --------------------------------------------------------
    # Ensure feature order
    # --------------------------------------------------------

    X_chunk = (
        chunk_df[
            feature_cols_11
        ]
        .astype(
            np.float32
        )
    )

    # --------------------------------------------------------
    # Score all candidates
    # --------------------------------------------------------

    chunk_df["score"] = (
        model9.predict(
            X_chunk
        )
    )

    # --------------------------------------------------------
    # Target exclusivity Top-1
    # --------------------------------------------------------

    idx_best = (
        chunk_df
        .groupby(
            "query_id"
        )["score"]
        .idxmax()
    )

    best_df = (
        chunk_df.loc[
            idx_best,
            [
                "query_id",
                "s1_id",
                "score"
            ]
        ]
        .copy()
    )

    # --------------------------------------------------------
    # Save checkpoint
    # --------------------------------------------------------

    best_df.to_parquet(
        part_path,
        index=False
    )

    part_paths.append(
        part_path
    )

    elapsed = (
        time.time() - t0
    )

    print(
        f"\n[{start:,}-{end:,}] "
        f"targets={len(chunk_target_ids):,} "
        f"candidates={len(chunk_df):,} "
        f"best_edges={len(best_df):,} "
        f"time={elapsed/60:.2f} min"
    )

    del chunk_df
    del X_chunk
    del best_df

    gc.collect()

# ------------------------------------------------------------
# Combine checkpoints
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "COMBINING CHECKPOINTS"
)
print(
    "=" * 100
)

parts = []

for path in sorted(
    part_paths
):

    part = pd.read_parquet(
        path
    )

    parts.append(
        part
    )

best_all = pd.concat(
    parts,
    ignore_index=True
)

del parts

# ------------------------------------------------------------
# Ground truth reference
# ------------------------------------------------------------

true_by_s1 = {
    sid: set()
    for sid in VAL_S1["entity_id"]
}

for target_id, sid in target_to_s1.items():

    if sid in true_by_s1:

        true_by_s1[
            sid
        ].add(
            target_id
        )

# ------------------------------------------------------------
# Evaluate thresholds on ALL 20k S1
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FULL S1 MACRO F0.5"
)
print(
    "=" * 100
)

thresholds_11 = np.arange(
    0.02,
    0.951,
    0.005
)

results11 = []

for threshold in thresholds_11:

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1[
            "entity_id"
        ]
    }

    accepted = best_all[
        best_all["score"] >= threshold
    ]

    for row in accepted.itertuples(
        index=False
    ):

        predicted_by_s1[
            row.s1_id
        ].add(
            row.query_id
        )

    f05_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for sid in VAL_S1[
        "entity_id"
    ]:

        true_targets = true_by_s1[
            sid
        ]

        predicted_targets = (
            predicted_by_s1[
                sid
            ]
        )

        tp = len(
            true_targets
            &
            predicted_targets
        )

        fp = len(
            predicted_targets
            -
            true_targets
        )

        fn = len(
            true_targets
            -
            predicted_targets
        )

        if (
            tp + fp
            >
            0
        ):

            precision = (
                tp
                /
                (tp + fp)
            )

        else:

            precision = (
                1.0
                if not true_targets
                else 0.0
            )

        if (
            tp + fn
            >
            0
        ):

            recall = (
                tp
                /
                (tp + fn)
            )

        else:

            recall = 1.0

        if (
            precision <= 0
            or
            recall <= 0
        ):

            f05 = 0.0

        else:

            f05 = (
                1.25
                *
                precision
                *
                recall
                /
                (
                    0.25 * precision
                    +
                    recall
                )
            )

        f05_values.append(
            f05
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    macro_f05 = np.mean(
        f05_values
    )

    results11.append({
        "threshold":
            threshold,

        "macro_f05":
            macro_f05,

        "precision":
            (
                total_tp
                /
                (
                    total_tp
                    +
                    total_fp
                )
                if
                total_tp + total_fp > 0
                else 0.0
            ),

        "recall":
            (
                total_tp
                /
                (
                    total_tp
                    +
                    total_fn
                )
                if
                total_tp + total_fn > 0
                else 0.0
            ),

        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn
    })

results11_df = pd.DataFrame(
    results11
)

best11 = results11_df.loc[
    results11_df["macro_f05"].idxmax()
]

print(
    results11_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(15)
    .to_string(
        index=False
    )
)

print(
    "\n" + "=" * 100
)
print(
    "BEST FULL-S1 RESULT"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best11['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best11['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best11['precision']:.6f}"
)

print(
    "Recall:",
    f"{best11['recall']:.6f}"
)

print(
    "TP:",
    int(best11["tp"])
)

print(
    "FP:",
    int(best11["fp"])
)

print(
    "FN:",
    int(best11["fn"])
)

print(
    "\nNo-match S1:",
    int(
        (
            VAL_S1[
                "entity_id"
            ]
            .map(
                lambda x:
                len(true_by_s1[x]) == 0
            )
        ).sum()
    )
)

# ------------------------------------------------------------
# Save result
# ------------------------------------------------------------

best_target_path = (
    MODEL_DIR
    /
    "full_validation_target_exclusive_top1.parquet"
)

best_all.to_parquet(
    best_target_path,
    index=False
)

full_result_path = (
    MODEL_DIR
    /
    "full_s1_validation_results.csv"
)

results11_df.to_csv(
    full_result_path,
    index=False
)

print(
    "\nSaved best-target checkpoint:",
    best_target_path
)

print(
    "Saved threshold results:",
    full_result_path
)

print(
    "\nTOTAL CELL 11 TIME:",
    f"{(time.time()-overall_start)/60:.2f} min"
)

print(
    "\nCELL 11 COMPLETE."
)

CELL 11 — FULL 20K S1 VALIDATION
Reranker trees: 424
Reranker features: 56
Feature count verified: 56

[0-2,000] targets=2,000 candidates=241,284 best_edges=2,000 time=0.60 min

[2,000-4,000] targets=2,000 candidates=238,622 best_edges=2,000 time=0.51 min

[4,000-6,000] targets=2,000 candidates=248,489 best_edges=2,000 time=0.51 min

[6,000-8,000] targets=2,000 candidates=233,152 best_edges=2,000 time=0.49 min

[8,000-10,000] targets=2,000 candidates=227,592 best_edges=2,000 time=0.49 min

[10,000-12,000] targets=2,000 candidates=237,407 best_edges=2,000 time=0.51 min

[12,000-14,000] targets=2,000 candidates=244,384 best_edges=2,000 time=0.54 min

[14,000-16,000] targets=2,000 candidates=233,741 best_edges=2,000 time=0.52 min

[16,000-18,000] targets=2,000 candidates=233,848 best_edges=2,000 time=0.50 min

[18,000-20,000] targets=2,000 candidates=235,023 best_edges=2,000 time=0.49 min

[20,000-22,000] targets=2,000 candidates=239,878 best_edges=2,000 time=0.51 min

[22,000-24,000] tar

In [ ]:
# CELL 11B — FIX MODEL TYPE FOR FULL VALIDATION

import lightgbm as lgb

print("=" * 100)
print("CELL 11B — FIX MODEL TYPE")
print("=" * 100)

if "model9" not in globals():
    raise RuntimeError(
        "model9 is not in memory. Load the saved reranker first."
    )

print(
    "Current model type:",
    type(model9).__name__
)

# If this is the sklearn wrapper, use its underlying Booster.
if hasattr(
    model9,
    "booster_"
):

    model9 = model9.booster_

elif not isinstance(
    model9,
    lgb.Booster
):

    raise RuntimeError(
        f"Unsupported model type: {type(model9)}"
    )

print(
    "Corrected model type:",
    type(model9).__name__
)

print(
    "Trees:",
    model9.num_trees()
)

print(
    "Features:",
    model9.num_feature()
)

print(
    "\nMODEL TYPE FIXED."
)

print(
    "Now rerun CELL 11 unchanged."
)

CELL 11B — FIX MODEL TYPE
Current model type: LGBMClassifier
Corrected model type: Booster
Trees: 424
Features: 56

MODEL TYPE FIXED.
Now rerun CELL 11 unchanged.


In [ ]:
# CELL 12 — FINE-GRAINED FULL-S1 THRESHOLD OPTIMIZATION

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def calculate_macro_f05(
    threshold,
    df=best_all
):

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    accepted = df[
        df["score"] >= threshold
    ]

    for row in accepted.itertuples(
        index=False
    ):

        predicted_by_s1[
            row.s1_id
        ].add(
            row.query_id
        )

    f05_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for sid in VAL_S1["entity_id"]:

        true_targets = true_by_s1[
            sid
        ]

        predicted_targets = (
            predicted_by_s1[
                sid
            ]
        )

        tp = len(
            true_targets
            &
            predicted_targets
        )

        fp = len(
            predicted_targets
            -
            true_targets
        )

        fn = len(
            true_targets
            -
            predicted_targets
        )

        if tp + fp > 0:

            precision = (
                tp
                /
                (tp + fp)
            )

        else:

            precision = (
                1.0
                if not true_targets
                else 0.0
            )

        if tp + fn > 0:

            recall = (
                tp
                /
                (tp + fn)
            )

        else:

            recall = 1.0

        if (
            precision > 0
            and
            recall > 0
        ):

            f05 = (
                1.25
                * precision
                * recall
                /
                (
                    0.25 * precision
                    + recall
                )
            )

        else:

            f05 = 0.0

        f05_values.append(
            f05
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    macro_f05 = float(
        np.mean(f05_values)
    )

    precision_micro = (
        total_tp
        /
        (total_tp + total_fp)
        if total_tp + total_fp > 0
        else 0.0
    )

    recall_micro = (
        total_tp
        /
        (total_tp + total_fn)
        if total_tp + total_fn > 0
        else 0.0
    )

    return {
        "threshold":
            threshold,

        "macro_f05":
            macro_f05,

        "precision":
            precision_micro,

        "recall":
            recall_micro,

        "tp":
            total_tp,

        "fp":
            total_fp,

        "fn":
            total_fn
    }


# ------------------------------------------------------------
# Fine sweep around current optimum
# ------------------------------------------------------------

thresholds = np.arange(
    0.55,
    0.701,
    0.001
)

results12 = []

for threshold in thresholds:

    result = calculate_macro_f05(
        float(threshold)
    )

    results12.append(
        result
    )

results12_df = pd.DataFrame(
    results12
)

# ------------------------------------------------------------
# Best thresholds
# ------------------------------------------------------------

top12 = (
    results12_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
)

print(
    "\n" + "=" * 100
)
print(
    "TOP FINE-GRAINED THRESHOLDS"
)
print(
    "=" * 100
)

print(
    top12.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

best12 = (
    results12_df
    .loc[
        results12_df["macro_f05"].idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "BEST FINE-GRAINED THRESHOLD"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best12['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best12['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best12['precision']:.6f}"
)

print(
    "Recall:",
    f"{best12['recall']:.6f}"
)

print(
    "TP:",
    int(best12["tp"])
)

print(
    "FP:",
    int(best12["fp"])
)

print(
    "FN:",
    int(best12["fn"])
)

print(
    "\nImprovement over 0.988468:",
    f"{best12['macro_f05'] - 0.988468:+.6f}"
)

print(
    "\nCELL 12 COMPLETE."
)

CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION

TOP FINE-GRAINED THRESHOLDS
threshold macro_f05 precision   recall    tp  fp   fn
    0.628  0.988477  0.994811 0.977496 67673 353 1558
    0.637  0.988475  0.994868 0.977337 67662 349 1569
    0.619  0.988472  0.994724 0.977669 67685 359 1546
    0.617  0.988471  0.994695 0.977727 67689 361 1542
    0.638  0.988470  0.994868 0.977322 67661 349 1570
    0.639  0.988470  0.994883 0.977293 67659 348 1572
    0.627  0.988470  0.994796 0.977496 67673 354 1558
    0.618  0.988469  0.994709 0.977683 67686 360 1545
    0.640  0.988468  0.994883 0.977279 67658 348 1573
    0.621  0.988467  0.994724 0.977655 67684 359 1547
    0.620  0.988467  0.994724 0.977655 67684 359 1547
    0.629  0.988467  0.994825 0.977452 67670 352 1561
    0.642  0.988465  0.994897 0.977250 67656 347 1575
    0.615  0.988465  0.994681 0.977756 67691 362 1540
    0.636  0.988464  0.994854 0.977351 67663 350 1568
    0.616  0.988463  0.994681 0.977741 67690 362 1541
    0.6

In [ ]:
# CELL 13 — ERROR ANALYSIS AT BEST THRESHOLD

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 13 — FULL ERROR ANALYSIS @ THRESHOLD 0.628")
print("=" * 100)

THRESHOLD_13 = 0.628

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Target-level analysis
# ------------------------------------------------------------

analysis = best_all.copy()

analysis["true_s1_id"] = (
    analysis["query_id"]
    .map(target_to_s1)
)

analysis["best_is_true"] = (
    analysis["s1_id"]
    ==
    analysis["true_s1_id"]
)

analysis["accepted"] = (
    analysis["score"]
    >=
    THRESHOLD_13
)

# Four useful target-level categories
analysis["category"] = np.select(
    [
        analysis["accepted"]
        &
        analysis["best_is_true"],

        analysis["accepted"]
        &
        ~analysis["best_is_true"],

        ~analysis["accepted"]
        &
        analysis["best_is_true"],

        ~analysis["accepted"]
        &
        ~analysis["best_is_true"]
    ],
    [
        "CORRECT_ACCEPT",
        "WRONG_ACCEPT",
        "CORRECT_REJECT",
        "WRONG_REJECT"
    ],
    default="UNKNOWN"
)

print(
    "\n" + "=" * 100
)
print(
    "TARGET-LEVEL ERROR COUNTS"
)
print(
    "=" * 100
)

print(
    analysis["category"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# Score distributions by category
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "SCORE DISTRIBUTIONS"
)
print(
    "=" * 100
)

for category in [
    "CORRECT_ACCEPT",
    "WRONG_ACCEPT",
    "CORRECT_REJECT",
    "WRONG_REJECT"
]:

    scores = (
        analysis.loc[
            analysis["category"] == category,
            "score"
        ]
        .astype(float)
    )

    if len(scores) == 0:
        continue

    print(
        f"\n{category}"
    )

    print(
        "count:",
        len(scores)
    )

    print(
        "mean:",
        f"{scores.mean():.6f}"
    )

    print(
        "min:",
        f"{scores.min():.6f}"
    )

    print(
        "P10:",
        f"{scores.quantile(.10):.6f}"
    )

    print(
        "P25:",
        f"{scores.quantile(.25):.6f}"
    )

    print(
        "median:",
        f"{scores.median():.6f}"
    )

    print(
        "P75:",
        f"{scores.quantile(.75):.6f}"
    )

    print(
        "P90:",
        f"{scores.quantile(.90):.6f}"
    )

    print(
        "max:",
        f"{scores.max():.6f}"
    )

# ------------------------------------------------------------
# Margin around threshold
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "NEAR-THRESHOLD COUNTS"
)
print(
    "=" * 100
)

for width in [
    0.01,
    0.02,
    0.03,
    0.05,
    0.10
]:

    low = THRESHOLD_13 - width
    high = THRESHOLD_13 + width

    near = analysis[
        analysis["score"].between(
            low,
            high,
            inclusive="both"
        )
    ]

    print(
        f"±{width:.2f}:",
        f"{len(near):,} targets"
    )

# ------------------------------------------------------------
# False positives
# ------------------------------------------------------------

fp = analysis[
    (
        analysis["category"]
        ==
        "WRONG_ACCEPT"
    )
].copy()

print(
    "\n" + "=" * 100
)
print(
    "WRONG ACCEPTS / FALSE POSITIVES"
)
print(
    "=" * 100
)

print(
    "Count:",
    len(fp)
)

print(
    "\nTop false-positive scores:"
)

print(
    fp[
        [
            "query_id",
            "s1_id",
            "true_s1_id",
            "score"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(30)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# False negatives
# ------------------------------------------------------------

fn = analysis[
    (
        analysis["best_is_true"]
        &
        ~analysis["accepted"]
    )
].copy()

print(
    "\n" + "=" * 100
)
print(
    "MISSED TRUE TARGETS"
)
print(
    "=" * 100
)

print(
    "Count:",
    len(fn)
)

print(
    "\nHighest-scoring missed true targets:"
)

print(
    fn[
        [
            "query_id",
            "s1_id",
            "score"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(30)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Wrong-reject targets
#
# These are particularly important:
# the top-ranked S1 is wrong and was below threshold.
# This means the true S1 may have been available but
# not ranked first.
# ------------------------------------------------------------

wrong_reject = analysis[
    (
        analysis["category"]
        ==
        "WRONG_REJECT"
    )
].copy()

print(
    "\n" + "=" * 100
)
print(
    "WRONG REJECTS"
)
print(
    "=" * 100
)

print(
    "Count:",
    len(wrong_reject)
)

print(
    wrong_reject[
        [
            "query_id",
            "s1_id",
            "true_s1_id",
            "score"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(30)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# S1-level impact
# ------------------------------------------------------------

# Current predictions at threshold
predicted_by_s1_13 = {
    sid: set()
    for sid in VAL_S1["entity_id"]
}

accepted = analysis[
    analysis["accepted"]
]

for row in accepted.itertuples(
    index=False
):

    predicted_by_s1_13[
        row.s1_id
    ].add(
        row.query_id
    )

# ------------------------------------------------------------
# Categorize S1 entities
# ------------------------------------------------------------

s1_rows = []

for sid in VAL_S1["entity_id"]:

    true_set = true_by_s1[
        sid
    ]

    pred_set = predicted_by_s1_13[
        sid
    ]

    tp = len(
        true_set & pred_set
    )

    fp_n = len(
        pred_set - true_set
    )

    fn_n = len(
        true_set - pred_set
    )

    if not true_set:
        if pred_set:
            category = "NO_MATCH_FALSE_POSITIVE"
        else:
            category = "NO_MATCH_CORRECT_EMPTY"

    else:

        if fn_n == 0 and fp_n == 0:
            category = "PERFECT"

        elif fp_n > 0 and fn_n > 0:
            category = "BOTH_FP_FN"

        elif fp_n > 0:
            category = "FP_ONLY"

        elif fn_n > 0:
            category = "FN_ONLY"

        else:
            category = "OTHER"

    s1_rows.append({
        "s1_id": sid,
        "true_count": len(true_set),
        "pred_count": len(pred_set),
        "tp": tp,
        "fp": fp_n,
        "fn": fn_n,
        "category": category
    })

s1_error_df = pd.DataFrame(
    s1_rows
)

print(
    "\n" + "=" * 100
)
print(
    "S1-LEVEL ERROR BREAKDOWN"
)
print(
    "=" * 100
)

print(
    s1_error_df["category"]
    .value_counts()
    .to_string()
)

print(
    "\nNo-match entities:"
)

print(
    s1_error_df[
        s1_error_df["category"]
        .str.startswith(
            "NO_MATCH"
        )
    ][
        [
            "s1_id",
            "true_count",
            "pred_count",
            "fp"
        ]
    ]
    .head(30)
    .to_string(
        index=False
    )
)

print(
    "\nS1 entities with false positives:"
)

print(
    s1_error_df[
        s1_error_df["fp"] > 0
    ]
    .sort_values(
        ["fp", "fn"],
        ascending=False
    )
    .head(30)
    .to_string(
        index=False
    )
)

print(
    "\nS1 entities with false negatives:"
)

print(
    s1_error_df[
        s1_error_df["fn"] > 0
    ]
    .sort_values(
        ["fn", "fp"],
        ascending=False
    )
    .head(30)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Save diagnostic artifacts
# ------------------------------------------------------------

fp_path = (
    MODEL_DIR
    /
    "cell13_false_positives.csv"
)

fn_path = (
    MODEL_DIR
    /
    "cell13_false_negatives.csv"
)

s1_error_path = (
    MODEL_DIR
    /
    "cell13_s1_error_breakdown.csv"
)

fp.to_csv(
    fp_path,
    index=False
)

fn.to_csv(
    fn_path,
    index=False
)

s1_error_df.to_csv(
    s1_error_path,
    index=False
)

print(
    "\nSaved diagnostics:"
)

print(
    fp_path
)

print(
    fn_path
)

print(
    s1_error_path
)

print(
    "\nCELL 13 COMPLETE."
)

CELL 13 — FULL ERROR ANALYSIS @ THRESHOLD 0.628

TARGET-LEVEL ERROR COUNTS
category
CORRECT_ACCEPT    67673
WRONG_REJECT        619
CORRECT_REJECT      586
WRONG_ACCEPT        353

SCORE DISTRIBUTIONS

CORRECT_ACCEPT
count: 67673
mean: 0.995561
min: 0.628012
P10: 0.996727
P25: 0.999241
median: 0.999617
P75: 0.999639
P90: 0.999644
max: 0.999660

WRONG_ACCEPT
count: 353
mean: 0.888358
min: 0.628527
P10: 0.709955
P25: 0.813375
median: 0.924121
P75: 0.978675
P90: 0.994780
max: 0.999644

CORRECT_REJECT
count: 586
mean: 0.304651
min: 0.002771
P10: 0.047322
P25: 0.122256
median: 0.302896
P75: 0.475217
P90: 0.572744
max: 0.626688

WRONG_REJECT
count: 619
mean: 0.200259
min: 0.000820
P10: 0.016569
P25: 0.040414
median: 0.121545
P75: 0.325971
P90: 0.527310
max: 0.627887

NEAR-THRESHOLD COUNTS
±0.01: 36 targets
±0.02: 65 targets
±0.03: 98 targets
±0.05: 177 targets
±0.10: 347 targets

WRONG ACCEPTS / FALSE POSITIVES
Count: 353

Top false-positive scores:
    query_id        s1_id   true_s1_id    

In [ ]:
# CELL 14 — ENRICH ERROR CASES WITH SIMILARITY FEATURES

import os
import re
import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("=" * 100)
print("CELL 14 — ERROR CASE FEATURE ANALYSIS")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

MODEL_DIR_LOCAL = os.path.join(
    BASE,
    "MODELS"
)

FP_PATH = os.path.join(
    MODEL_DIR_LOCAL,
    "cell13_false_positives.csv"
)

FN_PATH = os.path.join(
    MODEL_DIR_LOCAL,
    "cell13_false_negatives.csv"
)

WR_PATH = os.path.join(
    MODEL_DIR_LOCAL,
    "cell13_s1_error_breakdown.csv"
)

# ------------------------------------------------------------
# Load diagnostics
# ------------------------------------------------------------

fp = pd.read_csv(
    FP_PATH,
    dtype=str
)

fn = pd.read_csv(
    FN_PATH,
    dtype=str
)

print(
    "False-positive records:",
    len(fp)
)

print(
    "False-negative records:",
    len(fn)
)

# ------------------------------------------------------------
# Build lookup maps
# ------------------------------------------------------------

target_lookup = (
    VAL_TARGET_POOL
    .set_index("entity_id")
)

s1_lookup = (
    VAL_S1
    .set_index("entity_id")
)

# ------------------------------------------------------------
# Feature helper
# ------------------------------------------------------------

def extract_features(
    target_id,
    s1_id
):

    if (
        target_id not in target_lookup.index
        or
        s1_id not in s1_lookup.index
    ):
        return None

    target_row = target_lookup.loc[
        target_id
    ]

    s1_row = s1_lookup.loc[
        s1_id
    ]

    q_name = str(
        target_row["name_n"]
    )

    q_addr = str(
        target_row["addr_n"]
    )

    s_name = str(
        s1_row["name_n"]
    )

    s_addr = str(
        s1_row["addr_n"]
    )

    q_nums = set(
        re.findall(
            r"\d+",
            q_addr
        )
    )

    s_nums = set(
        re.findall(
            r"\d+",
            s_addr
        )
    )

    return {
        "name_ratio":
            fuzz.ratio(
                q_name,
                s_name
            ) / 100.0,

        "addr_ratio":
            fuzz.ratio(
                q_addr,
                s_addr
            ) / 100.0,

        "name_partial":
            fuzz.partial_ratio(
                q_name,
                s_name
            ) / 100.0,

        "addr_partial":
            fuzz.partial_ratio(
                q_addr,
                s_addr
            ) / 100.0,

        "name_token":
            fuzz.token_set_ratio(
                q_name,
                s_name
            ) / 100.0,

        "addr_token":
            fuzz.token_set_ratio(
                q_addr,
                s_addr
            ) / 100.0,

        "number_overlap":
            len(q_nums & s_nums),

        "exact_name":
            int(
                q_name == s_name
            ),

        "exact_address":
            int(
                q_addr == s_addr
            ),

        "country_match":
            int(
                str(target_row["country_n"])
                ==
                str(s1_row["country_n"])
            )
    }


# ------------------------------------------------------------
# Analyze FALSE POSITIVES
# ------------------------------------------------------------

fp_features = []

for row in fp.itertuples(
    index=False
):

    pred = extract_features(
        row.query_id,
        row.s1_id
    )

    if pred is None:
        continue

    pred.update({
        "query_id":
            row.query_id,

        "predicted_s1":
            row.s1_id,

        "true_s1":
            getattr(
                row,
                "true_s1_id",
                ""
            ),

        "model_score":
            float(row.score)
    })

    # Compare against true S1 when available
    true_id = getattr(
        row,
        "true_s1_id",
        ""
    )

    if (
        isinstance(true_id, str)
        and
        true_id
        and
        true_id in s1_lookup.index
    ):

        true_features = extract_features(
            row.query_id,
            true_id
        )

        for key, value in true_features.items():

            pred[
                "true_" + key
            ] = value

        pred[
            "name_ratio_gap"
        ] = (
            pred["name_ratio"]
            -
            pred["true_name_ratio"]
        )

        pred[
            "addr_ratio_gap"
        ] = (
            pred["addr_ratio"]
            -
            pred["true_addr_ratio"]
        )

        pred[
            "number_gap"
        ] = (
            pred["number_overlap"]
            -
            pred["true_number_overlap"]
        )

    fp_features.append(
        pred
    )

fp_features_df = pd.DataFrame(
    fp_features
)

# ------------------------------------------------------------
# Analyze FALSE NEGATIVES
# ------------------------------------------------------------

fn_features = []

for row in fn.itertuples(
    index=False
):

    predicted_id = row.s1_id

    # First record is the listed candidate / true target
    # depending on the diagnostic type.
    pred_features = extract_features(
        row.query_id,
        predicted_id
    )

    if pred_features is None:
        continue

    pred_features.update({
        "query_id":
            row.query_id,

        "s1_id":
            predicted_id,

        "model_score":
            float(row.score)
    })

    # --------------------------------------------------------
    # Retrieve actual true S1
    # --------------------------------------------------------

    true_id = (
        target_to_s1.get(
            row.query_id
        )
    )

    if (
        true_id is not None
        and
        true_id in s1_lookup.index
    ):

        true_features = extract_features(
            row.query_id,
            true_id
        )

        for key, value in true_features.items():

            pred_features[
                "true_" + key
            ] = value

        pred_features[
            "name_gap_vs_true"
        ] = (
            pred_features["name_ratio"]
            -
            pred_features["true_name_ratio"]
        )

        pred_features[
            "addr_gap_vs_true"
        ] = (
            pred_features["addr_ratio"]
            -
            pred_features["true_addr_ratio"]
        )

        pred_features[
            "number_gap_vs_true"
        ] = (
            pred_features["number_overlap"]
            -
            pred_features["true_number_overlap"]
        )

    fn_features.append(
        pred_features
    )

fn_features_df = pd.DataFrame(
    fn_features
)

# ------------------------------------------------------------
# Summary helper
# ------------------------------------------------------------

def print_summary(
    df,
    title,
    columns
):

    print(
        "\n" + "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )

    available = [
        c
        for c in columns
        if c in df.columns
    ]

    print(
        df[
            available
        ]
        .mean(
            numeric_only=True
        )
        .to_string()
    )


# ------------------------------------------------------------
# FP summary
# ------------------------------------------------------------

fp_cols = [
    "model_score",
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "number_overlap",
    "exact_name",
    "exact_address",
    "country_match",
    "true_name_ratio",
    "true_addr_ratio",
    "true_number_overlap",
    "name_ratio_gap",
    "addr_ratio_gap",
    "number_gap"
]

print_summary(
    fp_features_df,
    "FALSE POSITIVE FEATURE MEANS",
    fp_cols
)

# ------------------------------------------------------------
# FN summary
# ------------------------------------------------------------

fn_cols = [
    "model_score",
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "number_overlap",
    "true_name_ratio",
    "true_addr_ratio",
    "true_number_overlap",
    "name_gap_vs_true",
    "addr_gap_vs_true",
    "number_gap_vs_true"
]

print_summary(
    fn_features_df,
    "FALSE NEGATIVE FEATURE MEANS",
    fn_cols
)

# ------------------------------------------------------------
# FP high-score cases
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)

print(
    "TOP 20 FALSE POSITIVES WITH FEATURE EVIDENCE"
)

print(
    "=" * 100
)

fp_show_cols = [
    "query_id",
    "predicted_s1",
    "true_s1",
    "model_score",
    "name_ratio",
    "addr_ratio",
    "number_overlap",
    "true_name_ratio",
    "true_addr_ratio",
    "true_number_overlap",
    "name_ratio_gap",
    "addr_ratio_gap"
]

print(
    fp_features_df
    .sort_values(
        "model_score",
        ascending=False
    )[
        [
            c
            for c in fp_show_cols
            if c in fp_features_df.columns
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# FN near-threshold cases
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)

print(
    "TOP 30 MISSED CASES NEAR THRESHOLD"
)

print(
    "=" * 100
)

fn_show_cols = [
    "query_id",
    "s1_id",
    "model_score",
    "name_ratio",
    "addr_ratio",
    "number_overlap",
    "true_name_ratio",
    "true_addr_ratio",
    "true_number_overlap",
    "name_gap_vs_true",
    "addr_gap_vs_true",
    "number_gap_vs_true"
]

print(
    fn_features_df
    .sort_values(
        "model_score",
        ascending=False
    )[
        [
            c
            for c in fn_show_cols
            if c in fn_features_df.columns
        ]
    ]
    .head(30)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Save enriched diagnostics
# ------------------------------------------------------------

fp_enriched_path = (
    MODEL_DIR_LOCAL
    + "/cell14_false_positives_enriched.csv"
)

fn_enriched_path = (
    MODEL_DIR_LOCAL
    + "/cell14_false_negatives_enriched.csv"
)

fp_features_df.to_csv(
    fp_enriched_path,
    index=False
)

fn_features_df.to_csv(
    fn_enriched_path,
    index=False
)

print(
    "\nSaved:"
)

print(
    fp_enriched_path
)

print(
    fn_enriched_path
)

print(
    "\nCELL 14 COMPLETE."
)

CELL 14 — ERROR CASE FEATURE ANALYSIS
False-positive records: 353
False-negative records: 586

FALSE POSITIVE FEATURE MEANS
model_score            0.888358
name_ratio             0.493754
addr_ratio             0.332131
name_partial           0.567434
addr_partial           0.422376
number_overlap         0.209632
exact_name             0.062323
exact_address          0.005666
country_match          1.000000
true_name_ratio        0.505005
true_addr_ratio        0.279042
true_number_overlap    0.385269
name_ratio_gap        -0.011251
addr_ratio_gap         0.053090
number_gap            -0.175637

FALSE NEGATIVE FEATURE MEANS
model_score            0.304651
name_ratio             0.284497
addr_ratio             0.356676
name_partial           0.346928
addr_partial           0.527383
number_overlap         0.351536
true_name_ratio        0.284497
true_addr_ratio        0.356676
true_number_overlap    0.351536
name_gap_vs_true       0.000000
addr_gap_vs_true       0.000000
number_gap_vs_

In [ ]:
# CELL 15 — TOP-1 SCORE MARGIN OPTIMIZATION
#
# Uses the 2k retrieval-aware validation candidate set.
# Tests:
#   base score threshold
#   AND top1-vs-top2 score margin
#
# This is a development experiment only.

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 15 — TOP-1 SCORE MARGIN OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "df_new",
    "valid_mask",
    "model9"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Validation candidate rows
# ------------------------------------------------------------

margin_df = df_new.loc[
    valid_mask,
    [
        "query_id",
        "s1_id",
        "label"
    ]
].copy()

X_margin = (
    df_new.loc[
        valid_mask,
        feature_cols_9
    ]
    .astype(np.float32)
)

margin_df["score"] = (
    model9.predict_proba(
        X_margin
    )[:, 1]
)

# ------------------------------------------------------------
# Sort candidates within each target
# ------------------------------------------------------------

margin_df = (
    margin_df
    .sort_values(
        [
            "query_id",
            "score"
        ],
        ascending=[
            True,
            False
        ]
    )
)

# Top-1 / Top-2 scores
margin_df["candidate_rank"] = (
    margin_df
    .groupby("query_id")
    .cumcount()
    + 1
)

top1 = (
    margin_df[
        margin_df["candidate_rank"] == 1
    ]
    [
        [
            "query_id",
            "s1_id",
            "score",
            "label"
        ]
    ]
    .copy()
)

top2_scores = (
    margin_df[
        margin_df["candidate_rank"] == 2
    ]
    [
        [
            "query_id",
            "score"
        ]
    ]
    .rename(
        columns={
            "score": "score_2"
        }
    )
)

top1 = top1.merge(
    top2_scores,
    on="query_id",
    how="left"
)

top1["score_2"] = (
    top1["score_2"]
    .fillna(0.0)
)

top1["margin"] = (
    top1["score"]
    -
    top1["score_2"]
)

# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

true_by_query = (
    margin_df
    .groupby("query_id")["label"]
    .max()
    .to_dict()
)

# ------------------------------------------------------------
# F0.5 helper
# ------------------------------------------------------------

def f05(
    p,
    r
):

    if p <= 0 or r <= 0:
        return 0.0

    return (
        1.25
        * p
        * r
        /
        (
            0.25 * p
            + r
        )
    )

# ------------------------------------------------------------
# Evaluate threshold + margin
# ------------------------------------------------------------

def evaluate_rule(
    score_threshold,
    margin_threshold
):

    tp = 0
    fp = 0
    fn = 0

    f05_values = []

    for row in top1.itertuples(
        index=False
    ):

        # Prediction only if score AND margin pass
        predicted = (
            row.score >= score_threshold
            and
            row.margin >= margin_threshold
        )

        true_positive = (
            row.label == 1
        )

        if predicted and true_positive:
            tp_q = 1
            fp_q = 0
            fn_q = 0

        elif predicted and not true_positive:
            tp_q = 0
            fp_q = 1
            fn_q = 0

        elif not predicted and true_positive:
            tp_q = 0
            fp_q = 0
            fn_q = 1

        else:
            tp_q = 0
            fp_q = 0
            fn_q = 0

        tp += tp_q
        fp += fp_q
        fn += fn_q

        if tp_q + fp_q > 0:
            precision = (
                tp_q
                /
                (tp_q + fp_q)
            )
        else:
            precision = (
                1.0
                if not true_positive
                else 0.0
            )

        if true_positive:
            recall = float(
                tp_q
            )
        else:
            recall = 1.0

        f05_values.append(
            f05(
                precision,
                recall
            )
        )

    macro_f05 = np.mean(
        f05_values
    )

    precision_micro = (
        tp
        /
        (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall_micro = (
        tp
        /
        (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    return {
        "threshold":
            score_threshold,

        "margin":
            margin_threshold,

        "macro_f05":
            macro_f05,

        "precision":
            precision_micro,

        "recall":
            recall_micro,

        "tp": tp,
        "fp": fp,
        "fn": fn
    }

# ------------------------------------------------------------
# Grid search
# ------------------------------------------------------------

threshold_grid = np.arange(
    0.45,
    0.751,
    0.005
)

margin_grid = [
    0.000,
    0.005,
    0.010,
    0.015,
    0.020,
    0.030,
    0.040,
    0.050,
    0.075,
    0.100,
    0.150,
    0.200
]

results15 = []

for threshold in threshold_grid:

    for margin in margin_grid:

        results15.append(
            evaluate_rule(
                float(threshold),
                float(margin)
            )
        )

results15_df = pd.DataFrame(
    results15
)

# ------------------------------------------------------------
# Best results
# ------------------------------------------------------------

top15 = (
    results15_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
)

print(
    "\n" + "=" * 100
)
print(
    "BEST THRESHOLD + MARGIN RULES"
)
print(
    "=" * 100
)

print(
    top15.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "margin":
                lambda x: f"{x:.3f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

best15 = results15_df.loc[
    results15_df[
        "macro_f05"
    ].idxmax()
]

print(
    "\n" + "=" * 100
)
print(
    "BEST MARGIN RULE"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best15['threshold']:.3f}"
)

print(
    "Margin:",
    f"{best15['margin']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best15['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best15['precision']:.6f}"
)

print(
    "Recall:",
    f"{best15['recall']:.6f}"
)

print(
    "TP:",
    int(best15["tp"])
)

print(
    "FP:",
    int(best15["fp"])
)

print(
    "FN:",
    int(best15["fn"])
)

print(
    "\nCurrent matched-development baseline:",
    "0.988000 with target Top-1 exclusivity"
)

print(
    "\nCELL 15 COMPLETE."
)

CELL 15 — TOP-1 SCORE MARGIN OPTIMIZATION


AttributeError: 'Booster' object has no attribute 'predict_proba'

In [ ]:
# CELL 15B — FIX TOP-1 SCORE MARGIN SCORING

import lightgbm as lgb
import numpy as np

print("=" * 100)
print("CELL 15B — FIX TOP-1 SCORE MARGIN SCORING")
print("=" * 100)

# ------------------------------------------------------------
# Verify model
# ------------------------------------------------------------

print(
    "Model type:",
    type(model9).__name__
)

if not isinstance(
    model9,
    lgb.Booster
):

    if hasattr(model9, "booster_"):
        model9 = model9.booster_
    else:
        raise RuntimeError(
            f"Unexpected model type: {type(model9)}"
        )

print(
    "Model trees:",
    model9.num_trees()
)

print(
    "Model features:",
    model9.num_feature()
)

# ------------------------------------------------------------
# Score existing margin dataframe
# ------------------------------------------------------------

if "X_margin" not in globals():
    raise RuntimeError(
        "X_margin is not available. "
        "Do not rebuild anything yet; inspect Cell 15."
    )

if "margin_df" not in globals():
    raise RuntimeError(
        "margin_df is not available. "
        "Do not rebuild anything yet; inspect Cell 15."
    )

print(
    "\nScoring existing X_margin..."
)

# Booster.predict() already returns the binary probability
margin_scores = model9.predict(
    X_margin
)

margin_df["score"] = np.asarray(
    margin_scores
)

print(
    "Rows scored:",
    f"{len(margin_df):,}"
)

print(
    "Score range:",
    f"{margin_df['score'].min():.6f}",
    "→",
    f"{margin_df['score'].max():.6f}"
)

print(
    "\nCELL 15B COMPLETE."
)

print(
    "Now continue with the remaining Cell 15 margin "
    "optimization code after the predict_proba() line."
)

CELL 15B — FIX TOP-1 SCORE MARGIN SCORING
Model type: Booster
Model trees: 424
Model features: 56

Scoring existing X_margin...
Rows scored: 53,341
Score range: 0.000004 → 0.999659

CELL 15B COMPLETE.
Now continue with the remaining Cell 15 margin optimization code after the predict_proba() line.


In [ ]:
# CELL 15C — TOP-1 SCORE MARGIN OPTIMIZATION

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 15C — TOP-1 SCORE MARGIN OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Identify the scored margin dataframe
# ------------------------------------------------------------

candidate_frames = []

for name, obj in list(globals().items()):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue

    if len(obj) != len(X_margin):
        continue

    required_cols = {
        "query_id",
        "s1_id",
        "label"
    }

    if required_cols.issubset(
        obj.columns
    ):
        candidate_frames.append(
            name
        )

print(
    "Candidate metadata frames:",
    candidate_frames
)

if not candidate_frames:

    raise RuntimeError(
        "Could not identify the metadata dataframe "
        "corresponding to X_margin."
    )

# Prefer conventional names
preferred = [
    "df_margin",
    "margin_df",
    "df_margin_scored",
    "margin_candidates"
]

margin_df_name = next(
    (
        x
        for x in preferred
        if x in candidate_frames
    ),
    candidate_frames[0]
)

margin_df = globals()[
    margin_df_name
].copy()

print(
    "Using metadata dataframe:",
    margin_df_name
)

# ------------------------------------------------------------
# Recover prediction scores
# ------------------------------------------------------------

score_variable_candidates = [
    "margin_scores",
    "margin_proba",
    "margin_predictions",
    "scores_margin",
    "pred_margin"
]

margin_scores = None

for name in score_variable_candidates:

    if name not in globals():
        continue

    arr = np.asarray(
        globals()[name]
    )

    if (
        arr.ndim == 1
        and
        len(arr) == len(X_margin)
    ):

        margin_scores = (
            arr.astype(np.float32)
        )

        print(
            "Using score variable:",
            name
        )

        break

if margin_scores is None:

    # If X_margin itself contains the score, recover it.
    for col in [
        "score",
        "prediction",
        "pred_score",
        "model_score"
    ]:

        if (
            col in margin_df.columns
            and
            len(margin_df) == len(X_margin)
        ):

            margin_scores = (
                margin_df[col]
                .to_numpy(
                    dtype=np.float32
                )
            )

            print(
                "Using score column:",
                col
            )

            break

if margin_scores is None:

    raise RuntimeError(
        "Could not identify the score array produced "
        "by the Cell 15B predict_proba() call. "
        "If your variable has a different name, set "
        "`margin_scores = YOUR_ARRAY` and rerun Cell 15C."
    )

# ------------------------------------------------------------
# Attach scores
# ------------------------------------------------------------

margin_df = margin_df.reset_index(
    drop=True
)

margin_df["score"] = (
    margin_scores
)

print(
    "\nRows scored:",
    f"{len(margin_df):,}"
)

# ------------------------------------------------------------
# Rank candidates within each target query
# ------------------------------------------------------------

margin_df = (
    margin_df
    .sort_values(
        [
            "query_id",
            "score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)

margin_df["query_rank"] = (
    margin_df
    .groupby("query_id")
    .cumcount()
    + 1
)

# ------------------------------------------------------------
# Extract top-1 and top-2 scores per target
# ------------------------------------------------------------

top1 = (
    margin_df[
        margin_df["query_rank"] == 1
    ]
    [
        [
            "query_id",
            "s1_id",
            "label",
            "score"
        ]
    ]
    .copy()
)

top2 = (
    margin_df[
        margin_df["query_rank"] == 2
    ]
    [
        [
            "query_id",
            "score"
        ]
    ]
    .rename(
        columns={
            "score":
                "second_score"
        }
    )
)

top_ranked = top1.merge(
    top2,
    on="query_id",
    how="left"
)

top_ranked["second_score"] = (
    top_ranked["second_score"]
    .fillna(0.0)
)

top_ranked["margin"] = (
    top_ranked["score"]
    -
    top_ranked["second_score"]
)

print(
    "\nUnique target queries:",
    f"{top_ranked['query_id'].nunique():,}"
)

print(
    "Top-1 rows:",
    f"{len(top_ranked):,}"
)

print(
    "Margin statistics:"
)

print(
    top_ranked["margin"].describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    ).to_string()
)

# ------------------------------------------------------------
# Query-level F0.5
#
# For the margin experiment, each target gets at most one S1.
# ------------------------------------------------------------

def f05(
    precision,
    recall
):

    if precision <= 0 or recall <= 0:
        return 0.0

    return (
        1.25
        * precision
        * recall
        /
        (
            0.25 * precision
            +
            recall
        )
    )


def evaluate_margin_rule(
    score_threshold,
    margin_threshold
):

    predicted = (
        (top_ranked["score"]
         >= score_threshold)
        &
        (top_ranked["margin"]
         >= margin_threshold)
    )

    selected = top_ranked[
        predicted
    ]

    # Each selected row is one target->S1 prediction.
    tp = int(
        selected["label"].sum()
    )

    fp = int(
        len(selected)
        -
        tp
    )

    fn = int(
        len(
            top_ranked[
                top_ranked["label"] == 1
            ]
        )
        -
        tp
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    return {
        "threshold":
            score_threshold,

        "margin":
            margin_threshold,

        "f05":
            f05(
                precision,
                recall
            ),

        "precision":
            precision,

        "recall":
            recall,

        "tp":
            tp,

        "fp":
            fp,

        "fn":
            fn,

        "accepted":
            int(predicted.sum())
    }


# ------------------------------------------------------------
# Fine grid
# ------------------------------------------------------------

score_thresholds = np.arange(
    0.40,
    0.801,
    0.005
)

margin_thresholds = np.arange(
    0.000,
    0.301,
    0.005
)

results = []

for threshold in score_thresholds:

    for margin_threshold in margin_thresholds:

        results.append(
            evaluate_margin_rule(
                float(threshold),
                float(margin_threshold)
            )
        )

margin_results = pd.DataFrame(
    results
)

# ------------------------------------------------------------
# Best configurations
# ------------------------------------------------------------

top_results = (
    margin_results
    .sort_values(
        "f05",
        ascending=False
    )
    .head(30)
)

print(
    "\n" + "=" * 100
)
print(
    "TOP MARGIN CONFIGURATIONS"
)
print(
    "=" * 100
)

print(
    top_results.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "margin":
                lambda x: f"{x:.3f}",
            "f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

best_margin = (
    margin_results
    .loc[
        margin_results["f05"].idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "BEST MARGIN CONFIGURATION"
)
print(
    "=" * 100
)

print(
    "Score threshold:",
    f"{best_margin['threshold']:.3f}"
)

print(
    "Margin threshold:",
    f"{best_margin['margin']:.3f}"
)

print(
    "F0.5:",
    f"{best_margin['f05']:.6f}"
)

print(
    "Precision:",
    f"{best_margin['precision']:.6f}"
)

print(
    "Recall:",
    f"{best_margin['recall']:.6f}"
)

print(
    "TP:",
    int(best_margin["tp"])
)

print(
    "FP:",
    int(best_margin["fp"])
)

print(
    "FN:",
    int(best_margin["fn"])
)

print(
    "\nCELL 15C COMPLETE."
)

CELL 15C — TOP-1 SCORE MARGIN OPTIMIZATION
Candidate metadata frames: ['valid_df', 'eval_df', 'margin_df']
Using metadata dataframe: margin_df
Using score column: score

Rows scored: 53,341

Unique target queries: 500
Top-1 rows: 500
Margin statistics:
count    500.000000
mean       0.975390
std        0.121726
min        0.007246
1%         0.193257
5%         0.945171
10%        0.990516
25%        0.999215
50%        0.999608
75%        0.999633
90%        0.999637
95%        0.999645
99%        0.999655
max        0.999655

TOP MARGIN CONFIGURATIONS
threshold margin      f05 precision   recall  tp  fp  fn  accepted
    0.480  0.275 0.997963  1.000000 0.989899 490   0   5       490
    0.480  0.280 0.997963  1.000000 0.989899 490   0   5       490
    0.480  0.285 0.997963  1.000000 0.989899 490   0   5       490
    0.480  0.290 0.997963  1.000000 0.989899 490   0   5       490
    0.480  0.295 0.997963  1.000000 0.989899 490   0   5       490
    0.480  0.300 0.997963  1.000000 0.

In [ ]:
# CELL 16 — RECOVER CELL 15C MARGIN RULE

print("=" * 100)
print("CELL 16 — RECOVER CELL 15C MARGIN RULE")
print("=" * 100)

matches = []

for i, code in enumerate(_ih):

    text = str(code)

    if (
        "BEST MARGIN CONFIGURATION" in text
        or
        "margin_threshold" in text.lower()
        or
        "margin threshold" in text.lower()
    ):

        matches.append(
            (i, text)
        )

print(
    "\nMatching notebook cells:",
    len(matches)
)

for i, text in matches[-5:]:

    print(
        "\n" + "-" * 100
    )

    print(
        "NOTEBOOK CELL:",
        i
    )

    print(
        "-" * 100
    )

    print(
        text[:20000]
    )

print(
    "\nCELL 16 COMPLETE."
)

CELL 16 — RECOVER CELL 15C MARGIN RULE

Matching notebook cells: 3

----------------------------------------------------------------------------------------------------
NOTEBOOK CELL: 23
----------------------------------------------------------------------------------------------------
# CELL 15 — TOP-1 SCORE MARGIN OPTIMIZATION
#
# Uses the 2k retrieval-aware validation candidate set.
# Tests:
#   base score threshold
#   AND top1-vs-top2 score margin
#
# This is a development experiment only.

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 15 — TOP-1 SCORE MARGIN OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "df_new",
    "valid_mask",
    "model9"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# -----------------

In [ ]:
# CELL 13 — FULL S1 ERROR ANATOMY AT BEST THRESHOLD

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 13 — FULL S1 ERROR ANATOMY")
print("=" * 100)

THRESHOLD_13 = 0.628

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# S1 truth counts
# ------------------------------------------------------------

truth_count = {
    sid: len(
        true_by_s1[sid]
    )
    for sid in VAL_S1["entity_id"]
}

# ------------------------------------------------------------
# Map every target's true S1
# ------------------------------------------------------------

work = best_all.copy()

work["true_s1"] = (
    work["query_id"]
    .map(target_to_s1)
)

work["true_match"] = (
    work["s1_id"]
    ==
    work["true_s1"]
)

work["accepted"] = (
    work["score"]
    >=
    THRESHOLD_13
)

# ------------------------------------------------------------
# Basic prediction counts
# ------------------------------------------------------------

accepted = work[
    work["accepted"]
].copy()

false_positive_rows = accepted[
    ~accepted["true_match"]
].copy()

true_positive_rows = accepted[
    accepted["true_match"]
].copy()

print(
    "\n" + "=" * 100
)
print(
    "PAIR-LEVEL ERROR COUNTS"
)
print(
    "=" * 100
)

print(
    "Accepted rows:",
    f"{len(accepted):,}"
)

print(
    "True-positive rows:",
    f"{len(true_positive_rows):,}"
)

print(
    "False-positive rows:",
    f"{len(false_positive_rows):,}"
)

print(
    "Threshold:",
    THRESHOLD_13
)

# ------------------------------------------------------------
# FALSE NEGATIVE CLASSIFICATION
#
# For every true target:
#   A) correct S1 but score below threshold
#   B) assigned to wrong S1
#   C) no candidate/best assignment
# ------------------------------------------------------------

all_true_targets = list(
    target_to_s1.keys()
)

best_by_target = (
    work
    .drop_duplicates(
        "query_id",
        keep="first"
    )
    .set_index(
        "query_id"
    )
)

fn_rows = []

for target_id in all_true_targets:

    true_sid = target_to_s1[
        target_id
    ]

    if target_id not in best_by_target.index:

        fn_rows.append({
            "query_id":
                target_id,

            "true_s1":
                true_sid,

            "pred_s1":
                None,

            "best_score":
                np.nan,

            "fn_type":
                "retrieval_or_no_candidate"
        })

        continue

    r = best_by_target.loc[
        target_id
    ]

    pred_sid = r["s1_id"]
    score = float(
        r["score"]
    )

    if pred_sid == true_sid:

        if score < THRESHOLD_13:

            fn_rows.append({
                "query_id":
                    target_id,

                "true_s1":
                    true_sid,

                "pred_s1":
                    pred_sid,

                "best_score":
                    score,

                "fn_type":
                    "correct_ranking_but_rejected"
            })

    else:

        # Wrong S1 is assigned regardless of whether
        # the wrong assignment was accepted.
        fn_rows.append({
            "query_id":
                target_id,

            "true_s1":
                true_sid,

            "pred_s1":
                pred_sid,

            "best_score":
                score,

            "fn_type":
                (
                    "wrong_s1_above_threshold"
                    if score >= THRESHOLD_13
                    else
                    "wrong_s1_below_threshold"
                )
        })

fn_df = pd.DataFrame(
    fn_rows
)

print(
    "\n" + "=" * 100
)
print(
    "FALSE NEGATIVE ANATOMY"
)
print(
    "=" * 100
)

print(
    fn_df[
        "fn_type"
    ]
    .value_counts()
    .to_string()
)

print(
    "\nTotal FN target records:",
    len(fn_df)
)

# ------------------------------------------------------------
# Score distribution for FN types
# ------------------------------------------------------------

for fn_type, group in fn_df.groupby(
    "fn_type"
):

    print(
        "\n" + fn_type
    )

    scores = group[
        "best_score"
    ].dropna()

    if len(scores):

        print(
            "Count:",
            len(scores)
        )

        print(
            "Median score:",
            f"{scores.median():.6f}"
        )

        print(
            "P25:",
            f"{scores.quantile(.25):.6f}"
        )

        print(
            "P75:",
            f"{scores.quantile(.75):.6f}"
        )

        print(
            "Max:",
            f"{scores.max():.6f}"
        )

# ------------------------------------------------------------
# FALSE POSITIVES ON NO-MATCH S1
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FALSE POSITIVES BY S1 TYPE"
)
print(
    "=" * 100
)

# Build predicted targets per S1
pred_by_s1_13 = (
    accepted
    .groupby(
        "s1_id"
    )["query_id"]
    .apply(set)
    .to_dict()
)

no_match_fp_s1 = []
matched_fp_s1 = []

for sid in VAL_S1["entity_id"]:

    true_targets = true_by_s1[
        sid
    ]

    preds = pred_by_s1_13.get(
        sid,
        set()
    )

    fps = (
        preds
        -
        true_targets
    )

    if not fps:
        continue

    record = {
        "s1_id":
            sid,

        "true_count":
            len(true_targets),

        "pred_count":
            len(preds),

        "fp_count":
            len(fps),

        "mean_fp_score":
            accepted[
                (
                    accepted["s1_id"] == sid
                )
                &
                (
                    ~accepted["true_match"]
                )
            ]["score"].mean()
    }

    if len(true_targets) == 0:

        no_match_fp_s1.append(
            record
        )

    else:

        matched_fp_s1.append(
            record
        )

print(
    "No-match S1 with FP:",
    len(no_match_fp_s1)
)

print(
    "Matched S1 with FP:",
    len(matched_fp_s1)
)

print(
    "Total no-match FP rows:",
    sum(
        x["fp_count"]
        for x in no_match_fp_s1
    )
)

print(
    "Total matched-entity FP rows:",
    sum(
        x["fp_count"]
        for x in matched_fp_s1
    )
)

# ------------------------------------------------------------
# Most problematic no-match entities
# ------------------------------------------------------------

if no_match_fp_s1:

    print(
        "\nTop no-match FP entities:"
    )

    print(
        pd.DataFrame(
            no_match_fp_s1
        )
        .sort_values(
            [
                "fp_count",
                "mean_fp_score"
            ],
            ascending=[
                False,
                False
            ]
        )
        .head(25)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# Highest-confidence false positives
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "HIGHEST-CONFIDENCE FALSE POSITIVES"
)
print(
    "=" * 100
)

print(
    false_positive_rows[
        [
            "query_id",
            "s1_id",
            "true_s1",
            "score"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(30)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Highest-confidence wrong-S1 false negatives
# ------------------------------------------------------------

wrong_high = fn_df[
    fn_df["fn_type"]
    ==
    "wrong_s1_above_threshold"
].sort_values(
    "best_score",
    ascending=False
)

print(
    "\n" + "=" * 100
)
print(
    "HIGH-CONFIDENCE WRONG-S1 ASSIGNMENTS"
)
print(
    "=" * 100
)

if len(wrong_high):

    print(
        wrong_high.head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "None."
    )

print(
    "\nCELL 13 COMPLETE."
)

CELL 13 — FULL S1 ERROR ANATOMY

PAIR-LEVEL ERROR COUNTS
Accepted rows: 68,026
True-positive rows: 67,673
False-positive rows: 353
Threshold: 0.628

FALSE NEGATIVE ANATOMY
fn_type
wrong_s1_below_threshold        619
correct_ranking_but_rejected    586
wrong_s1_above_threshold        353

Total FN target records: 1558

correct_ranking_but_rejected
Count: 586
Median score: 0.302896
P25: 0.122256
P75: 0.475217
Max: 0.626688

wrong_s1_above_threshold
Count: 353
Median score: 0.924121
P25: 0.813375
P75: 0.978675
Max: 0.999644

wrong_s1_below_threshold
Count: 619
Median score: 0.121545
P25: 0.040414
P75: 0.325971
Max: 0.627887

FALSE POSITIVES BY S1 TYPE
No-match S1 with FP: 19
Matched S1 with FP: 289
Total no-match FP rows: 23
Total matched-entity FP rows: 330

Top no-match FP entities:
       s1_id  true_count  pred_count  fp_count  mean_fp_score
S1-136481301           0           3         3       0.964014
S1-354877127           0           2         2       0.978043
S1-710139409         

In [ ]:
# CELL 14 — HIGH-CONFIDENCE FP ANATOMY + TARGETED REJECTION RULES

import re
import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("=" * 100)
print("CELL 14 — HIGH-CONFIDENCE FP ANATOMY + TARGETED RULES")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "VAL_TARGET_POOL",
    "true_by_s1",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

BASE_THRESHOLD = 0.628

# ------------------------------------------------------------
# Lookups
# ------------------------------------------------------------

s1_lookup = (
    VAL_S1
    .set_index("entity_id")
)

target_lookup = (
    VAL_TARGET_POOL
    .set_index("entity_id")
)

# ------------------------------------------------------------
# Build pairwise evidence for the best target->S1 edge
# ------------------------------------------------------------

analysis_rows = []

for row in best_all.itertuples(
    index=False
):

    query_id = row.query_id
    s1_id = row.s1_id
    score = float(row.score)

    q = target_lookup.loc[
        query_id
    ]

    s = s1_lookup.loc[
        s1_id
    ]

    q_name = q["name_n"]
    s_name = s["name_n"]

    q_addr = q["addr_n"]
    s_addr = s["addr_n"]

    name_ratio = (
        fuzz.ratio(
            q_name,
            s_name
        )
        /
        100.0
    )

    addr_ratio = (
        fuzz.ratio(
            q_addr,
            s_addr
        )
        /
        100.0
    )

    name_partial = (
        fuzz.partial_ratio(
            q_name,
            s_name
        )
        /
        100.0
    )

    addr_partial = (
        fuzz.partial_ratio(
            q_addr,
            s_addr
        )
        /
        100.0
    )

    q_numbers = set(
        re.findall(
            r"\d+",
            q_addr
        )
    )

    s_numbers = set(
        re.findall(
            r"\d+",
            s_addr
        )
    )

    number_overlap_value = len(
        q_numbers &
        s_numbers
    )

    exact_name = int(
        q_name == s_name
    )

    exact_address = int(
        q_addr == s_addr
    )

    country_match = int(
        q["country_n"]
        ==
        s["country_n"]
    )

    true_s1 = target_to_s1.get(
        query_id
    )

    is_true = int(
        true_s1 == s1_id
    )

    analysis_rows.append({
        "query_id":
            query_id,

        "s1_id":
            s1_id,

        "score":
            score,

        "true_s1":
            true_s1,

        "is_true":
            is_true,

        "name_ratio":
            name_ratio,

        "addr_ratio":
            addr_ratio,

        "name_partial":
            name_partial,

        "addr_partial":
            addr_partial,

        "number_overlap":
            number_overlap_value,

        "exact_name":
            exact_name,

        "exact_address":
            exact_address,

        "country_match":
            country_match
    })

fp_analysis = pd.DataFrame(
    analysis_rows
)

# ------------------------------------------------------------
# Current accepted predictions
# ------------------------------------------------------------

accepted_base = fp_analysis[
    fp_analysis["score"]
    >= BASE_THRESHOLD
].copy()

fp_base = accepted_base[
    accepted_base["is_true"] == 0
]

tp_base = accepted_base[
    accepted_base["is_true"] == 1
]

print(
    "\n" + "=" * 100
)
print(
    "CURRENT BASELINE"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    BASE_THRESHOLD
)

print(
    "Accepted:",
    len(accepted_base)
)

print(
    "TP:",
    len(tp_base)
)

print(
    "FP:",
    len(fp_base)
)

# ------------------------------------------------------------
# Compare FP vs TP evidence
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "TP vs FP FEATURE DISTRIBUTIONS"
)
print(
    "=" * 100
)

for col in [
    "score",
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "number_overlap",
    "exact_name",
    "exact_address"
]:

    print(
        f"\n{col}"
    )

    print(
        "  TP median:",
        f"{tp_base[col].median():.4f}"
    )

    print(
        "  FP median:",
        f"{fp_base[col].median():.4f}"
    )

    print(
        "  TP mean:",
        f"{tp_base[col].mean():.4f}"
    )

    print(
        "  FP mean:",
        f"{fp_base[col].mean():.4f}"
    )

# ------------------------------------------------------------
# Inspect top high-confidence FPs
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "TOP 30 HIGH-CONFIDENCE FALSE POSITIVES"
)
print(
    "=" * 100
)

top_fp = (
    fp_analysis[
        fp_analysis["is_true"] == 0
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(30)
)

for row in top_fp.itertuples(
    index=False
):

    q = target_lookup.loc[
        row.query_id
    ]

    s = s1_lookup.loc[
        row.s1_id
    ]

    print(
        f"\n{row.query_id} -> {row.s1_id}"
    )

    print(
        f"  score={row.score:.6f} "
        f"name={row.name_ratio:.3f} "
        f"addr={row.addr_ratio:.3f} "
        f"num={row.number_overlap}"
    )

    print(
        "  TARGET NAME:",
        q["business_name"]
    )

    print(
        "  PRED S1 NAME:",
        s["business_name"]
    )

    print(
        "  TARGET ADDR:",
        q["business_address"]
    )

    print(
        "  PRED S1 ADDR:",
        s["business_address"]
    )

    print(
        "  TRUE S1:",
        row.true_s1
    )

# ------------------------------------------------------------
# Macro F0.5 evaluator
# ------------------------------------------------------------

def macro_f05_from_edges(
    edge_df
):

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    for row in edge_df.itertuples(
        index=False
    ):

        predicted_by_s1[
            row.s1_id
        ].add(
            row.query_id
        )

    values = []

    for sid in VAL_S1["entity_id"]:

        true_targets = true_by_s1[
            sid
        ]

        predicted = predicted_by_s1[
            sid
        ]

        tp = len(
            true_targets &
            predicted
        )

        fp = len(
            predicted -
            true_targets
        )

        fn = len(
            true_targets -
            predicted
        )

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else (
                1.0
                if not true_targets
                else 0.0
            )
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 1.0
        )

        if precision > 0 and recall > 0:

            f05 = (
                1.25
                * precision
                * recall
                /
                (
                    0.25 * precision
                    +
                    recall
                )
            )

        else:

            f05 = 0.0

        values.append(
            f05
        )

    return float(
        np.mean(values)
    )

# ------------------------------------------------------------
# Test targeted rejection rules
# ------------------------------------------------------------

rules = {
    "baseline":
        np.ones(
            len(fp_analysis),
            dtype=bool
        ),

    "reject_name90_addr30_no_num":
        ~(
            (
                fp_analysis["score"] >= 0.80
            )
            &
            (
                fp_analysis["name_ratio"] >= 0.90
            )
            &
            (
                fp_analysis["addr_ratio"] < 0.30
            )
            &
            (
                fp_analysis["number_overlap"] == 0
            )
        ),

    "reject_name85_addr25_no_num":
        ~(
            (
                fp_analysis["score"] >= 0.80
            )
            &
            (
                fp_analysis["name_ratio"] >= 0.85
            )
            &
            (
                fp_analysis["addr_ratio"] < 0.25
            )
            &
            (
                fp_analysis["number_overlap"] == 0
            )
        ),

    "reject_name90_addr40_no_num":
        ~(
            (
                fp_analysis["score"] >= 0.80
            )
            &
            (
                fp_analysis["name_ratio"] >= 0.90
            )
            &
            (
                fp_analysis["addr_ratio"] < 0.40
            )
            &
            (
                fp_analysis["number_overlap"] == 0
            )
        ),

    "reject_score90_addr30_no_num":
        ~(
            (
                fp_analysis["score"] >= 0.90
            )
            &
            (
                fp_analysis["addr_ratio"] < 0.30
            )
            &
            (
                fp_analysis["number_overlap"] == 0
            )
        ),

    "reject_score90_addr40_no_num":
        ~(
            (
                fp_analysis["score"] >= 0.90
            )
            &
            (
                fp_analysis["addr_ratio"] < 0.40
            )
            &
            (
                fp_analysis["number_overlap"] == 0
            )
        ),

    "reject_score95_addr50_no_num":
        ~(
            (
                fp_analysis["score"] >= 0.95
            )
            &
            (
                fp_analysis["addr_ratio"] < 0.50
            )
            &
            (
                fp_analysis["number_overlap"] == 0
            )
        ),

    "require_address_or_number_above90":
        ~(
            (
                fp_analysis["score"] >= 0.90
            )
            &
            (
                fp_analysis["addr_ratio"] < 0.50
            )
            &
            (
                fp_analysis["number_overlap"] == 0
            )
        )
}

rule_results = []

for rule_name, keep_mask in rules.items():

    candidate = fp_analysis[
        (
            fp_analysis["score"]
            >= BASE_THRESHOLD
        )
        &
        keep_mask
    ].copy()

    f05 = macro_f05_from_edges(
        candidate
    )

    rule_results.append({
        "rule":
            rule_name,

        "macro_f05":
            f05,

        "accepted_edges":
            len(candidate),

        "tp":
            int(
                candidate["is_true"].sum()
            ),

        "fp":
            int(
                (candidate["is_true"] == 0).sum()
            )
    })

rule_results_df = (
    pd.DataFrame(
        rule_results
    )
    .sort_values(
        "macro_f05",
        ascending=False
    )
)

print(
    "\n" + "=" * 100
)
print(
    "TARGETED REJECTION RULE RESULTS"
)
print(
    "=" * 100
)

print(
    rule_results_df.to_string(
        index=False,
        formatters={
            "macro_f05":
                lambda x: f"{x:.6f}"
        }
    )
)

print(
    "\nBaseline full-S1 result:",
    "0.988468"
)

print(
    "\nCELL 14 COMPLETE."
)

CELL 14 — HIGH-CONFIDENCE FP ANATOMY + TARGETED RULES

CURRENT BASELINE
Threshold: 0.628
Accepted: 68026
TP: 67673
FP: 353

TP vs FP FEATURE DISTRIBUTIONS

score
  TP median: 0.9996
  FP median: 0.9241
  TP mean: 0.9956
  FP mean: 0.8884

name_ratio
  TP median: 0.8923
  FP median: 0.6486
  TP mean: 0.8075
  FP mean: 0.4938

addr_ratio
  TP median: 0.8276
  FP median: 0.3469
  TP mean: 0.7610
  FP mean: 0.3321

name_partial
  TP median: 1.0000
  FP median: 0.7619
  TP mean: 0.8790
  FP mean: 0.5674

addr_partial
  TP median: 0.8800
  FP median: 0.5778
  TP mean: 0.8258
  FP mean: 0.4224

number_overlap
  TP median: 1.0000
  FP median: 0.0000
  TP mean: 0.8957
  FP mean: 0.2096

exact_name
  TP median: 0.0000
  FP median: 0.0000
  TP mean: 0.2327
  FP mean: 0.0623

exact_address
  TP median: 0.0000
  FP median: 0.0000
  TP mean: 0.0850
  FP mean: 0.0057

TOP 30 HIGH-CONFIDENCE FALSE POSITIVES

S3-797521910 -> S1-979690967
  score=0.999644 name=0.792 addr=0.733 num=1
  TARGET NAME: Noida

In [ ]:
# CELL 15 — FALSE NEGATIVE ANATOMY

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 15 — FALSE NEGATIVE ANATOMY")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_TARGET_POOL",
    "VAL_S1",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Current threshold from Cell 14
# ------------------------------------------------------------

CURRENT_THRESHOLD = 0.640

print(
    "Current threshold:",
    CURRENT_THRESHOLD
)

# ------------------------------------------------------------
# Accepted predictions
# ------------------------------------------------------------

accepted = (
    best_all[
        best_all["score"] >= CURRENT_THRESHOLD
    ]
    .copy()
)

# best_all contains target -> S1 top-1 predictions.
pred_by_target = dict(
    zip(
        accepted["query_id"],
        accepted["s1_id"]
    )
)

# ------------------------------------------------------------
# True target -> S1 mapping
# ------------------------------------------------------------

# Every validation target has at most one true S1.
# Identify missed true targets.
# ------------------------------------------------------------

false_negative_ids = []

for target_id, true_s1 in target_to_s1.items():

    predicted_s1 = pred_by_target.get(
        target_id
    )

    if predicted_s1 != true_s1:

        false_negative_ids.append(
            target_id
        )

print(
    "\nFalse-negative targets:",
    f"{len(false_negative_ids):,}"
)

# ------------------------------------------------------------
# Build lookup dictionaries
# ------------------------------------------------------------

target_lookup = (
    VAL_TARGET_POOL
    .set_index("entity_id")
)

s1_lookup = (
    VAL_S1
    .set_index("entity_id")
)

# ------------------------------------------------------------
# Examine FN feature values
# ------------------------------------------------------------

fn_rows = []

for target_id in false_negative_ids:

    true_s1 = target_to_s1[
        target_id
    ]

    target = target_lookup.loc[
        target_id
    ]

    true_row = s1_lookup.loc[
        true_s1
    ]

    # Find the model's selected candidate if present
    selected = (
        best_all[
            best_all["query_id"]
            == target_id
        ]
    )

    selected_row = (
        selected.iloc[0]
        if len(selected)
        else None
    )

    record = {
        "query_id":
            target_id,

        "true_s1":
            true_s1,

        "predicted_s1":
            (
                selected_row["s1_id"]
                if selected_row is not None
                else ""
            ),

        "predicted_score":
            (
                float(selected_row["score"])
                if selected_row is not None
                else np.nan
            )
    }

    # --------------------------------------------------------
    # True pair similarities
    # --------------------------------------------------------

    q_name = str(
        target["name_n"]
    )

    q_addr = str(
        target["addr_n"]
    )

    t_name = str(
        true_row["name_n"]
    )

    t_addr = str(
        true_row["addr_n"]
    )

    from rapidfuzz import fuzz

    record["true_name_ratio"] = (
        fuzz.ratio(
            q_name,
            t_name
        ) / 100.0
    )

    record["true_addr_ratio"] = (
        fuzz.ratio(
            q_addr,
            t_addr
        ) / 100.0
    )

    record["true_name_partial"] = (
        fuzz.partial_ratio(
            q_name,
            t_name
        ) / 100.0
    )

    record["true_addr_partial"] = (
        fuzz.partial_ratio(
            q_addr,
            t_addr
        ) / 100.0
    )

    q_nums = set(
        re.findall(
            r"\d+",
            q_addr
        )
    )

    t_nums = set(
        re.findall(
            r"\d+",
            t_addr
        )
    )

    record["true_number_overlap"] = len(
        q_nums & t_nums
    )

    record["true_exact_name"] = int(
        q_name == t_name
    )

    record["true_exact_address"] = int(
        q_addr == t_addr
    )

    fn_rows.append(
        record
    )

fn_df = pd.DataFrame(
    fn_rows
)

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "TRUE-PAIR FEATURE DISTRIBUTIONS FOR MISSED TARGETS"
)
print(
    "=" * 100
)

for col in [
    "predicted_score",
    "true_name_ratio",
    "true_addr_ratio",
    "true_name_partial",
    "true_addr_partial",
    "true_number_overlap"
]:

    s = fn_df[col]

    print(
        f"\n{col}"
    )

    print(
        "  mean   =",
        f"{s.mean():.4f}"
    )

    print(
        "  median =",
        f"{s.median():.4f}"
    )

    print(
        "  P10    =",
        f"{s.quantile(.10):.4f}"
    )

    print(
        "  P25    =",
        f"{s.quantile(.25):.4f}"
    )

    print(
        "  P75    =",
        f"{s.quantile(.75):.4f}"
    )

    print(
        "  P90    =",
        f"{s.quantile(.90):.4f}"
    )

# ------------------------------------------------------------
# Useful recovery conditions
# ------------------------------------------------------------

conditions = {
    "name>=0.80":
        fn_df["true_name_ratio"] >= 0.80,

    "addr>=0.70":
        fn_df["true_addr_ratio"] >= 0.70,

    "number>=1":
        fn_df["true_number_overlap"] >= 1,

    "name>=0.70_and_addr>=0.50":
        (
            (fn_df["true_name_ratio"] >= 0.70)
            &
            (fn_df["true_addr_ratio"] >= 0.50)
        ),

    "addr>=0.70_and_number>=1":
        (
            (fn_df["true_addr_ratio"] >= 0.70)
            &
            (fn_df["true_number_overlap"] >= 1)
        ),

    "name>=0.80_and_addr>=0.40":
        (
            (fn_df["true_name_ratio"] >= 0.80)
            &
            (fn_df["true_addr_ratio"] >= 0.40)
        ),

    "partial_name>=0.80_and_partial_addr>=0.60":
        (
            (fn_df["true_name_partial"] >= 0.80)
            &
            (fn_df["true_addr_partial"] >= 0.60)
        )
}

print(
    "\n" + "=" * 100
)
print(
    "POTENTIAL FN RECOVERY POOLS"
)
print(
    "=" * 100
)

for name, mask in conditions.items():

    print(
        f"{name:45s}: "
        f"{int(mask.sum()):5d} "
        f"({mask.mean()*100:6.2f}%)"
    )

# ------------------------------------------------------------
# Sample hardest missed pairs
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "20 MISSED TRUE PAIRS"
)
print(
    "=" * 100
)

print(
    fn_df[
        [
            "query_id",
            "true_s1",
            "predicted_s1",
            "predicted_score",
            "true_name_ratio",
            "true_addr_ratio",
            "true_name_partial",
            "true_addr_partial",
            "true_number_overlap"
        ]
    ]
    .sort_values(
        "predicted_score",
        ascending=False
    )
    .head(20)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

fn_path = (
    MODEL_DIR
    /
    "full_validation_false_negatives.csv"
)

fn_df.to_csv(
    fn_path,
    index=False
)

print(
    "\nSaved:",
    fn_path
)

print(
    "\nCELL 15 COMPLETE."
)

CELL 15 — FALSE NEGATIVE ANATOMY
Current threshold: 0.64

False-negative targets: 1,573

TRUE-PAIR FEATURE DISTRIBUTIONS FOR MISSED TARGETS

predicted_score
  mean   = 0.3977
  median = 0.3337
  P10    = 0.0296
  P25    = 0.1015
  P75    = 0.6164
  P90    = 0.9457

true_name_ratio
  mean   = 0.3266
  median = 0.1429
  P10    = 0.0000
  P25    = 0.0000
  P75    = 0.6800
  P90    = 0.8372

true_addr_ratio
  mean   = 0.3199
  median = 0.3721
  P10    = 0.0000
  P25    = 0.0000
  P75    = 0.5060
  P90    = 0.6050

true_name_partial
  mean   = 0.3925
  median = 0.2500
  P10    = 0.0000
  P25    = 0.0000
  P75    = 0.8095
  P90    = 1.0000

true_addr_partial
  mean   = 0.4807
  median = 0.6087
  P10    = 0.0000
  P25    = 0.0000
  P75    = 0.7347
  P90    = 0.8397

true_number_overlap
  mean   = 0.3160
  median = 0.0000
  P10    = 0.0000
  P25    = 0.0000
  P75    = 1.0000
  P90    = 1.0000

POTENTIAL FN RECOVERY POOLS
name>=0.80                                   :   204 ( 12.97%)
addr>=0.70

In [ ]:
# CELL 16 — FALSE-NEGATIVE TRUE-CANDIDATE SCORING
#
# For every FN target:
#   1. Rebuild its retrieval candidate set
#   2. Recompute the 56 features
#   3. Score ALL candidates with the new reranker
#   4. Locate the actual true S1 candidate
#   5. Compare:
#        true score
#        wrong Top-1 score
#        true rank
#
# This is analysis only. No production artifact is modified.

import os
import re
import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("=" * 100)
print("CELL 16 — FALSE-NEGATIVE TRUE-CANDIDATE SCORING")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "rare_candidates",
    "retrieved_top100",
    "X_target_full",
    "X_s1_3b",
    "model9",
    "MODEL_DIR"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Load FN report
# ------------------------------------------------------------

fn_path = (
    MODEL_DIR
    /
    "full_validation_false_negatives.csv"
)

if not fn_path.exists():

    raise FileNotFoundError(
        fn_path
    )

fn_df = pd.read_csv(
    fn_path,
    dtype=str
)

print(
    "False-negative rows:",
    f"{len(fn_df):,}"
)

# Convert numeric fields
for col in [
    "predicted_score",
    "true_name_ratio",
    "true_addr_ratio",
    "true_name_partial",
    "true_addr_partial",
    "true_number_overlap"
]:

    if col in fn_df.columns:

        fn_df[col] = pd.to_numeric(
            fn_df[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# Lookups
# ------------------------------------------------------------

s1_name = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["name_n"]
    )
)

s1_addr = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["addr_n"]
    )

s1_country = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["country_n"]
    )
)

target_name = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["name_n"]
    )
)

target_addr = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["addr_n"]
    )
)

target_country = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["country_n"]
    )
)

s1_ids_by_local = {
    i: sid
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

s1_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Target row lookup
# ------------------------------------------------------------

target_row_index = {
    sid: i
    for i, sid in enumerate(
        VAL_TARGET_POOL["entity_id"]
    )
}

# ------------------------------------------------------------
# Feature construction
# ------------------------------------------------------------

feature_rows = []
metadata_rows = []

print(
    "\nScoring FN candidate sets..."
)

for fn_num, fn_row in enumerate(
    fn_df.itertuples(index=False),
    start=1
):

    qid = fn_row.query_id
    true_sid = fn_row.true_s1

    q_name = target_name.get(
        qid,
        ""
    )

    q_addr = target_addr.get(
        qid,
        ""
    )

    q_country = target_country.get(
        qid,
        ""
    )

    # --------------------------------------------------------
    # Retrieval candidate union
    # --------------------------------------------------------

    rare_ids = set(
        rare_candidates.get(
            qid,
            np.empty(
                0,
                dtype=np.int32
            )
        )
    )

    tfidf_indices = (
        retrieved_top100.get(
            qid,
            np.empty(
                0,
                dtype=np.int32
            )
        )[:50]
    )

    tfidf_ids = set(
        int(x)
        for x in tfidf_indices
    )

    candidates = (
        rare_ids
        |
        tfidf_ids
    )

    # True S1 must be in the candidate pool for this analysis
    true_local = s1_local.get(
        true_sid
    )

    if (
        true_local is None
        or
        true_local not in candidates
    ):

        metadata_rows.append({
            "query_id": qid,
            "true_s1": true_sid,
            "candidate_present": 0,
            "true_score": np.nan,
            "true_rank": np.nan,
            "top1_s1": np.nan,
            "top1_score": np.nan
        })

        continue

    candidate_array = np.fromiter(
        candidates,
        dtype=np.int32
    )

    # --------------------------------------------------------
    # TF-IDF scores
    # --------------------------------------------------------

    q_pos = target_row_index[
        qid
    ]

    tfidf_scores = (
        X_target_full[
            q_pos:q_pos + 1
        ]
        @
        X_s1_3b[
            candidate_array
        ].T
    ).toarray().ravel()

    tfidf_score_map = {
        int(idx): float(score)
        for idx, score in zip(
            candidate_array,
            tfidf_scores
        )
    }

    tfidf_rank_map = {
        int(idx): rank + 1
        for rank, idx in enumerate(
            tfidf_indices
        )
    }

    # --------------------------------------------------------
    # Build candidate feature rows
    # --------------------------------------------------------

    local_feature_rows = []

    for local_idx in candidates:

        sid = s1_ids_by_local[
            int(local_idx)
        ]

        s_name = s1_name.get(
            sid,
            ""
        )

        s_addr = s1_addr.get(
            sid,
            ""
        )

        s_country = s1_country.get(
            sid,
            ""
        )

        # Base similarities
        name_ratio = (
            fuzz.ratio(
                q_name,
                s_name
            ) / 100.0
        )

        addr_ratio = (
            fuzz.ratio(
                q_addr,
                s_addr
            ) / 100.0
        )

        name_partial = (
            fuzz.partial_ratio(
                q_name,
                s_name
            ) / 100.0
        )

        addr_partial = (
            fuzz.partial_ratio(
                q_addr,
                s_addr
            ) / 100.0
        )

        name_token = (
            fuzz.token_set_ratio(
                q_name,
                s_name
            ) / 100.0
        )

        addr_token = (
            fuzz.token_set_ratio(
                q_addr,
                s_addr
            ) / 100.0
        )

        # Length
        qn = len(q_name)
        sn = len(s_name)

        qa = len(q_addr)
        sa = len(s_addr)

        name_length_ratio = (
            min(qn, sn)
            /
            max(qn, sn)
            if max(qn, sn) > 0
            else 1.0
        )

        addr_length_ratio = (
            min(qa, sa)
            /
            max(qa, sa)
            if max(qa, sa) > 0
            else 1.0
        )

        # Exact / country
        exact_name = int(
            q_name == s_name
        )

        exact_address = int(
            q_addr == s_addr
        )

        country_match = int(
            q_country == s_country
        )

        # Numbers
        q_nums = set(
            re.findall(
                r"\d+",
                q_addr
            )
        )

        s_nums = set(
            re.findall(
                r"\d+",
                s_addr
            )
        )

        number_overlap_value = len(
            q_nums & s_nums
        )

        # Combined
        combined_similarity = (
            0.45 * name_ratio
            +
            0.55 * addr_ratio
        )

        # Retrieval
        has_rare = int(
            int(local_idx) in rare_ids
        )

        has_tfidf = int(
            int(local_idx) in tfidf_ids
        )

        retrieval_channel_count = (
            has_rare
            +
            has_tfidf
        )

        tfidf_score = tfidf_score_map.get(
            int(local_idx),
            0.0
        )

        tfidf_rank = tfidf_rank_map.get(
            int(local_idx),
            999
        )

        # Interactions
        name_addr_product = (
            name_ratio * addr_ratio
        )

        name_partial_addr_partial_product = (
            name_partial * addr_partial
        )

        name_number_product = (
            name_ratio
            *
            number_overlap_value
        )

        addr_number_product = (
            addr_ratio
            *
            number_overlap_value
        )

        min_name_addr = min(
            name_ratio,
            addr_ratio
        )

        max_name_addr = max(
            name_ratio,
            addr_ratio
        )

        name_addr_gap = abs(
            name_ratio
            -
            addr_ratio
        )

        partial_gap = abs(
            name_partial
            -
            addr_partial
        )

        strong_name_weak_address = int(
            name_ratio >= 0.80
            and
            addr_ratio < 0.20
        )

        strong_name_no_number = int(
            name_ratio >= 0.80
            and
            number_overlap_value == 0
        )

        strong_address = int(
            addr_ratio >= 0.70
        )

        strong_name = int(
            name_ratio >= 0.80
        )

        address_supported = int(
            addr_ratio >= 0.50
        )

        multi_channel = int(
            retrieval_channel_count >= 2
        )

        balanced_similarity = (
            name_ratio
            +
            addr_ratio
        ) / 2.0

        partial_balanced_similarity = (
            name_partial
            +
            addr_partial
        ) / 2.0

        local_feature_rows.append({
            "local_idx": int(local_idx),

            "has_rare": has_rare,
            "has_tfidf": has_tfidf,
            "tfidf_score": tfidf_score,
            "tfidf_rank": tfidf_rank,

            "name_ratio": name_ratio,
            "addr_ratio": addr_ratio,
            "name_partial": name_partial,
            "addr_partial": addr_partial,
            "name_token": name_token,
            "addr_token": addr_token,
            "name_length_ratio": name_length_ratio,
            "addr_length_ratio": addr_length_ratio,
            "number_overlap": number_overlap_value,
            "exact_name": exact_name,
            "exact_address": exact_address,
            "country_match": country_match,
            "combined_similarity":
                combined_similarity,

            "retrieval_channel_count":
                retrieval_channel_count,

            "name_addr_product":
                name_addr_product,

            "name_partial_addr_partial_product":
                name_partial_addr_partial_product,

            "name_number_product":
                name_number_product,

            "addr_number_product":
                addr_number_product,

            "min_name_addr":
                min_name_addr,

            "max_name_addr":
                max_name_addr,

            "name_addr_gap":
                name_addr_gap,

            "partial_gap":
                partial_gap,

            "strong_name_weak_address":
                strong_name_weak_address,

            "strong_name_no_number":
                strong_name_no_number,

            "strong_address":
                strong_address,

            "strong_name":
                strong_name,

            "address_supported":
                address_supported,

            "multi_channel":
                multi_channel,

            "balanced_similarity":
                balanced_similarity,

            "partial_balanced_similarity":
                partial_balanced_similarity
        })

    local_df = pd.DataFrame(
        local_feature_rows
    )

    # --------------------------------------------------------
    # Query-relative features
    # --------------------------------------------------------

    for col in [
        "name_ratio",
        "addr_ratio",
        "name_partial",
        "addr_partial",
        "combined_similarity",
        "tfidf_score"
    ]:

        local_df[
            f"rank_{col}"
        ] = (
            local_df[col]
            .rank(
                method="min",
                ascending=False
            )
        )

        local_df[
            f"query_max_{col}"
        ] = local_df[col].max()

    local_df["tfidf_margin"] = (
        local_df["query_max_tfidf_score"]
        -
        local_df["tfidf_score"]
    )

    local_df["name_margin"] = (
        local_df["query_max_name_ratio"]
        -
        local_df["name_ratio"]
    )

    local_df["addr_margin"] = (
        local_df["query_max_addr_ratio"]
        -
        local_df["addr_ratio"]
    )

    local_df["combined_margin"] = (
        local_df[
            "query_max_combined_similarity"
        ]
        -
        local_df["combined_similarity"]
    )

    local_df["relative_tfidf_strength"] = (
        local_df["tfidf_score"]
        /
        (
            local_df["query_max_tfidf_score"]
            + 1e-8
        )
    )

    local_df["relative_combined_strength"] = (
        local_df["combined_similarity"]
        /
        (
            local_df[
                "query_max_combined_similarity"
            ]
            + 1e-8
        )
    )

    local_df["is_best_tfidf"] = (
        local_df["rank_tfidf_score"] == 1
    ).astype(np.int8)

    local_df["is_best_name"] = (
        local_df["rank_name_ratio"] == 1
    ).astype(np.int8)

    local_df["is_best_addr"] = (
        local_df["rank_addr_ratio"] == 1
    ).astype(np.int8)

    local_df["is_best_combined"] = (
        local_df[
            "rank_combined_similarity"
        ] == 1
    ).astype(np.int8)

    # --------------------------------------------------------
    # Score with exact 56-feature order
    # --------------------------------------------------------

    X_local = local_df[
        feature_cols_11
    ].astype(
        np.float32
    )

    local_df["model_score"] = (
        model9.predict(
            X_local
        )
    )

    # --------------------------------------------------------
    # Rank candidates by model
    # --------------------------------------------------------

    local_df = local_df.sort_values(
        "model_score",
        ascending=False
    ).reset_index(
        drop=True
    )

    true_positions = np.where(
        local_df["local_idx"].to_numpy()
        ==
        int(true_local)
    )[0]

    if len(true_positions):

        true_position = int(
            true_positions[0]
        )

        true_score = float(
            local_df.iloc[
                true_position
            ]["model_score"]
        )

        top1_s1 = s1_ids_by_local[
            int(
                local_df.iloc[0][
                    "local_idx"
                ]
            )
        ]

        top1_score = float(
            local_df.iloc[0][
                "model_score"
            ]
        )

        metadata_rows.append({
            "query_id":
                qid,

            "true_s1":
                true_sid,

            "candidate_present":
                1,

            "true_score":
                true_score,

            "true_rank":
                true_position + 1,

            "top1_s1":
                top1_s1,

            "top1_score":
                top1_score,

            "score_gap":
                top1_score - true_score,

            "true_name_ratio":
                float(
                    local_df.iloc[
                        true_position
                    ]["name_ratio"]
                ),

            "true_addr_ratio":
                float(
                    local_df.iloc[
                        true_position
                    ]["addr_ratio"]
                ),

            "true_number_overlap":
                float(
                    local_df.iloc[
                        true_position
                    ]["number_overlap"]
                )
        })

    else:

        metadata_rows.append({
            "query_id": qid,
            "true_s1": true_sid,
            "candidate_present": 1,
            "true_score": np.nan,
            "true_rank": np.nan,
            "top1_s1": np.nan,
            "top1_score": np.nan,
            "score_gap": np.nan
        })

    if fn_num % 250 == 0:

        print(
            f"{fn_num:,}/"
            f"{len(fn_df):,}"
        )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

fn_analysis = pd.DataFrame(
    metadata_rows
)

print(
    "\n" + "=" * 100
)
print(
    "FN TRUE-CANDIDATE SCORE ANALYSIS"
)
print(
    "=" * 100
)

print(
    "FNs with candidate present:",
    int(
        fn_analysis[
            "candidate_present"
        ].sum()
    )
)

present = fn_analysis[
    fn_analysis["candidate_present"] == 1
].copy()

print(
    "\nTrue candidate score:"
)

for q in [
    0,
    10,
    25,
    50,
    75,
    90,
    95,
    99,
    100
]:

    print(
        f"P{q:3d}:",
        f"{np.percentile(present['true_score'].dropna(),q):.6f}"
    )

print(
    "\nTrue candidate rank:"
)

for q in [
    0,
    25,
    50,
    75,
    90,
    95,
    99,
    100
]:

    print(
        f"P{q:3d}:",
        f"{np.percentile(present['true_rank'].dropna(),q):.0f}"
    )

print(
    "\nTrue candidate score >= threshold:"
)

print(
    (
        present["true_score"]
        >= 0.64
    ).sum(),
    "/",
    len(present)
)

print(
    "\nTrue candidate score >= 0.50:"
)

print(
    (
        present["true_score"]
        >= 0.50
    ).sum(),
    "/",
    len(present)
)

print(
    "\nTrue candidate score >= 0.30:"
)

print(
    (
        present["true_score"]
        >= 0.30
    ).sum(),
    "/",
    len(present)
)

# ------------------------------------------------------------
# Rank distribution
# ------------------------------------------------------------

print(
    "\nRank distribution:"
)

for r in [
    1,
    2,
    3,
    4,
    5,
    10
]:

    print(
        f"Rank <= {r}:",
        (
            present["true_rank"]
            <= r
        ).sum()
        ,
        "/",
        len(present)
    )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

analysis_path = (
    MODEL_DIR
    /
    "false_negative_true_candidate_scores.csv"
)

fn_analysis.to_csv(
    analysis_path,
    index=False
)

print(
    "\nSaved:",
    analysis_path
)

print(
    "\nCELL 16 COMPLETE."
)

SyntaxError: '(' was never closed (3413880583.py, line 105)

In [ ]:
# CELL 12B — FINE-GRAINED THRESHOLD OPTIMIZATION

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 12B — FINE-GRAINED THRESHOLD OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}. "
        f"Run CELL 11 first."
    )

# ------------------------------------------------------------
# Rebuild true target sets if necessary
# ------------------------------------------------------------

if "true_by_s1" not in globals():

    if "target_to_s1" not in globals():
        raise RuntimeError(
            "target_to_s1 is missing."
        )

    true_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    for target_id, sid in target_to_s1.items():

        if sid in true_by_s1:
            true_by_s1[sid].add(
                target_id
            )

# ------------------------------------------------------------
# Prepare prediction table
# ------------------------------------------------------------

pred_df = best_all[
    [
        "query_id",
        "s1_id",
        "score"
    ]
].copy()

pred_df["score"] = (
    pred_df["score"]
    .astype(float)
)

# ------------------------------------------------------------
# Group predictions by S1
# ------------------------------------------------------------

pred_groups = {
    sid: group[
        [
            "query_id",
            "score"
        ]
        ].to_numpy()
    for sid, group
    in pred_df.groupby(
        "s1_id",
        sort=False
    )
}

all_s1_ids = (
    VAL_S1["entity_id"]
    .tolist()
)

# ------------------------------------------------------------
# F0.5 function
# ------------------------------------------------------------

def f05(
    precision,
    recall
):

    if precision <= 0 or recall <= 0:
        return 0.0

    return (
        1.25
        * precision
        * recall
        /
        (
            0.25 * precision
            +
            recall
        )
    )

# ------------------------------------------------------------
# Evaluate one threshold
# ------------------------------------------------------------

def evaluate_threshold(
    threshold
):

    macro_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for sid in all_s1_ids:

        true_targets = true_by_s1.get(
            sid,
            set()
        )

        group = pred_groups.get(
            sid
        )

        if group is None:

            predicted = set()

        else:

            predicted = set(
                group[
                    group[:, 1].astype(float)
                    >= threshold
                ][:, 0]
            )

        tp = len(
            true_targets
            &
            predicted
        )

        fp = len(
            predicted
            -
            true_targets
        )

        fn = len(
            true_targets
            -
            predicted
        )

        # Empty prediction for true singleton/no-match
        if tp + fp > 0:

            precision = (
                tp
                /
                (tp + fp)
            )

        else:

            precision = (
                1.0
                if len(true_targets) == 0
                else 0.0
            )

        if tp + fn > 0:

            recall = (
                tp
                /
                (tp + fn)
            )

        else:

            recall = 1.0

        macro_values.append(
            f05(
                precision,
                recall
            )
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    return {
        "threshold":
            float(threshold),

        "macro_f05":
            float(
                np.mean(macro_values)
            ),

        "precision":
            (
                total_tp
                /
                (total_tp + total_fp)
                if total_tp + total_fp > 0
                else 0.0
            ),

        "recall":
            (
                total_tp
                /
                (total_tp + total_fn)
                if total_tp + total_fn > 0
                else 0.0
            ),

        "tp":
            int(total_tp),

        "fp":
            int(total_fp),

        "fn":
            int(total_fn)
    }

# ------------------------------------------------------------
# Fine sweep
# ------------------------------------------------------------

thresholds = np.round(
    np.arange(
        0.550,
        0.701,
        0.001
    ),
    3
)

results = []

for threshold in thresholds:

    results.append(
        evaluate_threshold(
            threshold
        )
    )

results_df = pd.DataFrame(
    results
)

# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

best_row = (
    results_df
    .loc[
        results_df["macro_f05"]
        .idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "TOP 20 THRESHOLDS"
)
print(
    "=" * 100
)

print(
    results_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
    .to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

print(
    "\n" + "=" * 100
)
print(
    "BEST FINE-GRAINED RESULT"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best_row['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best_row['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best_row['precision']:.6f}"
)

print(
    "Recall:",
    f"{best_row['recall']:.6f}"
)

print(
    "TP:",
    int(best_row["tp"])
)

print(
    "FP:",
    int(best_row["fp"])
)

print(
    "FN:",
    int(best_row["fn"])
)

print(
    "\nImprovement over current 0.988468:",
    f"{best_row['macro_f05'] - 0.988468:+.6f}"
)

# ------------------------------------------------------------
# Save sweep
# ------------------------------------------------------------

results_path = (
    MODEL_DIR
    /
    "full_s1_fine_threshold_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print(
    "\nSaved:",
    results_path
)

print(
    "\nCELL 12B COMPLETE."
)

CELL 12B — FINE-GRAINED THRESHOLD OPTIMIZATION

TOP 20 THRESHOLDS
threshold macro_f05 precision   recall    tp  fp   fn
    0.628  0.988477  0.994811 0.977496 67673 353 1558
    0.637  0.988475  0.994868 0.977337 67662 349 1569
    0.619  0.988472  0.994724 0.977669 67685 359 1546
    0.617  0.988471  0.994695 0.977727 67689 361 1542
    0.638  0.988470  0.994868 0.977322 67661 349 1570
    0.639  0.988470  0.994883 0.977293 67659 348 1572
    0.627  0.988470  0.994796 0.977496 67673 354 1558
    0.618  0.988469  0.994709 0.977683 67686 360 1545
    0.640  0.988468  0.994883 0.977279 67658 348 1573
    0.621  0.988467  0.994724 0.977655 67684 359 1547
    0.620  0.988467  0.994724 0.977655 67684 359 1547
    0.629  0.988467  0.994825 0.977452 67670 352 1561
    0.642  0.988465  0.994897 0.977250 67656 347 1575
    0.615  0.988465  0.994681 0.977756 67691 362 1540
    0.636  0.988464  0.994854 0.977351 67663 350 1568
    0.616  0.988463  0.994681 0.977741 67690 362 1541
    0.630  0.988

In [ ]:
# CELL 13 — FULL S1 ERROR ANATOMY AT BEST THRESHOLD

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 13 — FULL S1 ERROR ANATOMY")
print("=" * 100)

THRESHOLD = 0.628

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Build predictions
# ------------------------------------------------------------

accepted = best_all[
    best_all["score"] >= THRESHOLD
].copy()

predicted_by_s1 = {
    sid: set()
    for sid in VAL_S1["entity_id"]
}

for row in accepted.itertuples(
    index=False
):

    predicted_by_s1[
        row.s1_id
    ].add(
        row.query_id
    )

# ------------------------------------------------------------
# Entity-level table
# ------------------------------------------------------------

entity_rows = []

for sid in VAL_S1["entity_id"]:

    true_set = true_by_s1.get(
        sid,
        set()
    )

    pred_set = predicted_by_s1.get(
        sid,
        set()
    )

    tp_set = (
        true_set
        &
        pred_set
    )

    fp_set = (
        pred_set
        -
        true_set
    )

    fn_set = (
        true_set
        -
        pred_set
    )

    tp = len(tp_set)
    fp = len(fp_set)
    fn = len(fn_set)

    if tp + fp > 0:

        precision = (
            tp
            /
            (tp + fp)
        )

    else:

        precision = (
            1.0
            if not true_set
            else 0.0
        )

    if tp + fn > 0:

        recall = (
            tp
            /
            (tp + fn)
        )

    else:

        recall = 1.0

    if precision > 0 and recall > 0:

        f05 = (
            1.25
            * precision
            * recall
            /
            (
                0.25 * precision
                + recall
            )
        )

    else:

        f05 = 0.0

    entity_rows.append({
        "s1_id": sid,
        "true_count": len(true_set),
        "pred_count": len(pred_set),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f05": f05,
        "true_targets": true_set,
        "pred_targets": pred_set,
        "fp_targets": fp_set,
        "fn_targets": fn_set
    })

entity_df = pd.DataFrame(
    entity_rows
)

# ------------------------------------------------------------
# Basic error anatomy
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "GLOBAL ERROR ANATOMY"
)
print(
    "=" * 100
)

print(
    "Total S1:",
    len(entity_df)
)

print(
    "True no-match:",
    int(
        (entity_df.true_count == 0).sum()
    )
)

print(
    "Predicted non-empty among true no-match:",
    int(
        (
            (entity_df.true_count == 0)
            &
            (entity_df.pred_count > 0)
        ).sum()
    )
)

print(
    "Matched S1:",
    int(
        (entity_df.true_count > 0).sum()
    )
)

print(
    "Matched S1 with any FP:",
    int(
        (
            (entity_df.fp > 0)
            &
            (entity_df.true_count > 0)
        ).sum()
    )
)

print(
    "Matched S1 with any FN:",
    int(
        (
            (entity_df.fn > 0)
            &
            (entity_df.true_count > 0)
        ).sum()
    )
)

print(
    "Perfect S1:",
    int(
        (
            entity_df.fn == 0
            &
            (entity_df.fp == 0)
        ).sum()
    )
)

# ------------------------------------------------------------
# Error counts by true cardinality
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "ERRORS BY TRUE MATCH COUNT"
)
print(
    "=" * 100
)

cardinality = (
    entity_df
    .groupby("true_count")
    .agg(
        s1_count=("s1_id", "count"),
        fp_entities=("fp", lambda x: int((x > 0).sum())),
        fn_entities=("fn", lambda x: int((x > 0).sum())),
        total_fp=("fp", "sum"),
        total_fn=("fn", "sum"),
        mean_f05=("f05", "mean")
    )
    .reset_index()
)

print(
    cardinality.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# No-match false positives
# ------------------------------------------------------------

no_match_fp = entity_df[
    (entity_df.true_count == 0)
    &
    (entity_df.pred_count > 0)
].copy()

print(
    "\n" + "=" * 100
)
print(
    "NO-MATCH S1 FALSE POSITIVES"
)
print(
    "=" * 100
)

print(
    "Count:",
    len(no_match_fp)
)

if len(no_match_fp):

    print(
        no_match_fp[
            [
                "s1_id",
                "true_count",
                "pred_count",
                "fp"
            ]
        ]
        .sort_values(
            "fp",
            ascending=False
        )
        .head(30)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# Worst matched entities
# ------------------------------------------------------------

worst = (
    entity_df[
        entity_df.true_count > 0
    ]
    .sort_values(
        [
            "f05",
            "fn",
            "fp",
            "true_count"
        ],
        ascending=[
            True,
            False,
            False,
            False
        ]
    )
    .head(40)
)

print(
    "\n" + "=" * 100
)
print(
    "WORST MATCHED S1 ENTITIES"
)
print(
    "=" * 100
)

print(
    worst[
        [
            "s1_id",
            "true_count",
            "pred_count",
            "tp",
            "fp",
            "fn",
            "precision",
            "recall",
            "f05"
        ]
    ]
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Aggregate FN / FP contribution
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "ERROR TOTALS"
)
print(
    "=" * 100
)

print(
    "Total FP:",
    int(entity_df["fp"].sum())
)

print(
    "Total FN:",
    int(entity_df["fn"].sum())
)

print(
    "FN from multi-match S1:",
    int(
        entity_df.loc[
            entity_df["true_count"] > 1,
            "fn"
        ].sum()
    )
)

print(
    "FN from singleton S1:",
    int(
        entity_df.loc[
            entity_df["true_count"] == 1,
            "fn"
        ].sum()
    )
)

print(
    "FP on true no-match S1:",
    int(
        no_match_fp["fp"].sum()
    )
)

print(
    "\nCELL 13 COMPLETE."
)

CELL 13 — FULL S1 ERROR ANATOMY

GLOBAL ERROR ANATOMY
Total S1: 20000
True no-match: 1131
Predicted non-empty among true no-match: 19
Matched S1: 18869
Matched S1 with any FP: 289
Matched S1 with any FN: 1324
Perfect S1: 18676

ERRORS BY TRUE MATCH COUNT
 true_count  s1_count  fp_entities  fn_entities  total_fp  total_fn  mean_f05
          0      1131           19            0        23         0  0.983201
          1      1047           15           26        17        26  0.968473
          2      3394           48          161        49       166  0.986723
          3      4852           69          290        82       324  0.989685
          4      4347           65          331        80       387  0.990726
          5      2952           53          265        62       324  0.991331
          6      1487           25          147        25       180  0.992941
          7       595           13           76        14       102  0.991297
          8       157            1         

In [ ]:
# CELL 14 — TARGET-LEVEL FALSE-NEGATIVE DECOMPOSITION

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 14 — TARGET-LEVEL FALSE-NEGATIVE DECOMPOSITION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "target_to_s1",
    "VAL_TARGET_POOL"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

CURRENT_THRESHOLD = 0.640

# ------------------------------------------------------------
# Convert best_all to lookup
# ------------------------------------------------------------

best_lookup = (
    best_all
    .set_index("query_id")
)

all_true_targets = list(
    target_to_s1.keys()
)

rows = []

for target_id in all_true_targets:

    true_s1 = target_to_s1[
        target_id
    ]

    # --------------------------------------------------------
    # Target completely absent from retrieval/reranking
    # --------------------------------------------------------

    if target_id not in best_lookup.index:

        rows.append({
            "query_id":
                target_id,

            "true_s1":
                true_s1,

            "status":
                "RETRIEVAL_MISS",

            "pred_s1":
                None,

            "best_score":
                0.0,

            "above_threshold":
                False,

            "correct_top1":
                False
        })

        continue

    row = best_lookup.loc[
        target_id
    ]

    pred_s1 = row["s1_id"]
    score = float(
        row["score"]
    )

    correct_top1 = (
        pred_s1
        ==
        true_s1
    )

    above_threshold = (
        score
        >=
        CURRENT_THRESHOLD
    )

    # --------------------------------------------------------
    # Classify target-level outcome
    # --------------------------------------------------------

    if correct_top1:

        if above_threshold:

            status = "CORRECT_ACCEPTED"

        else:

            status = "CORRECT_BELOW_THRESHOLD"

    else:

        if above_threshold:

            status = "WRONG_TOP1_ACCEPTED"

        else:

            status = "WRONG_TOP1_BELOW_THRESHOLD"

    rows.append({
        "query_id":
            target_id,

        "true_s1":
            true_s1,

        "status":
            status,

        "pred_s1":
            pred_s1,

        "best_score":
            score,

        "above_threshold":
            above_threshold,

        "correct_top1":
            correct_top1
    })

analysis14 = pd.DataFrame(
    rows
)

# ------------------------------------------------------------
# Status counts
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "TARGET STATUS COUNTS"
)
print(
    "=" * 100
)

status_counts = (
    analysis14["status"]
    .value_counts()
)

print(
    status_counts.to_string()
)

# ------------------------------------------------------------
# Focus on false-negative categories
# ------------------------------------------------------------

fn_statuses = [
    "RETRIEVAL_MISS",
    "CORRECT_BELOW_THRESHOLD",
    "WRONG_TOP1_ACCEPTED",
    "WRONG_TOP1_BELOW_THRESHOLD"
]

print(
    "\n" + "=" * 100
)
print(
    "FALSE-NEGATIVE / DECISION BREAKDOWN"
)
print(
    "=" * 100
)

for status in fn_statuses:

    count = int(
        (
            analysis14["status"]
            ==
            status
        ).sum()
    )

    print(
        f"{status:28s}: {count:,}"
    )

# ------------------------------------------------------------
# Score distributions
# ------------------------------------------------------------

for status in [
    "CORRECT_BELOW_THRESHOLD",
    "WRONG_TOP1_ACCEPTED",
    "WRONG_TOP1_BELOW_THRESHOLD"
]:

    subset = analysis14[
        analysis14["status"]
        ==
        status
    ]

    if len(subset) == 0:
        continue

    scores = subset[
        "best_score"
    ].to_numpy()

    print(
        "\n" + "-" * 80
    )

    print(
        status,
        "| count =",
        len(scores)
    )

    print(
        "mean:",
        f"{scores.mean():.6f}"
    )

    print(
        "P10:",
        f"{np.percentile(scores,10):.6f}"
    )

    print(
        "P25:",
        f"{np.percentile(scores,25):.6f}"
    )

    print(
        "median:",
        f"{np.median(scores):.6f}"
    )

    print(
        "P75:",
        f"{np.percentile(scores,75):.6f}"
    )

    print(
        "P90:",
        f"{np.percentile(scores,90):.6f}"
    )

    print(
        "max:",
        f"{scores.max():.6f}"
    )

# ------------------------------------------------------------
# Wrong Top-1 accepted examples
# ------------------------------------------------------------

wrong_top1 = analysis14[
    analysis14["status"]
    ==
    "WRONG_TOP1_ACCEPTED"
].copy()

print(
    "\n" + "=" * 100
)
print(
    "WRONG TOP-1 ACCEPTED — SAMPLE"
)
print(
    "=" * 100
)

if len(wrong_top1):

    print(
        wrong_top1
        .sort_values(
            "best_score",
            ascending=False
        )
        .head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "None."
    )

# ------------------------------------------------------------
# Correct but below threshold examples
# ------------------------------------------------------------

below = analysis14[
    analysis14["status"]
    ==
    "CORRECT_BELOW_THRESHOLD"
].copy()

print(
    "\n" + "=" * 100
)
print(
    "CORRECT TOP-1 BUT BELOW THRESHOLD — SAMPLE"
)
print(
    "=" * 100
)

if len(below):

    print(
        below
        .sort_values(
            "best_score",
            ascending=False
        )
        .head(30)
        .to_string(
            index=False
        )
    )

else:

    print(
        "None."
    )

# ------------------------------------------------------------
# Save analysis
# ------------------------------------------------------------

analysis_path = (
    MODEL_DIR
    /
    "target_level_error_decomposition.csv"
)

analysis14.to_csv(
    analysis_path,
    index=False
)

print(
    "\nSaved:",
    analysis_path
)

print(
    "\nCELL 14 COMPLETE."
)

CELL 14 — TARGET-LEVEL FALSE-NEGATIVE DECOMPOSITION

TARGET STATUS COUNTS
status
CORRECT_ACCEPTED              67658
WRONG_TOP1_BELOW_THRESHOLD      624
CORRECT_BELOW_THRESHOLD         601
WRONG_TOP1_ACCEPTED             348

FALSE-NEGATIVE / DECISION BREAKDOWN
RETRIEVAL_MISS              : 0
CORRECT_BELOW_THRESHOLD     : 601
WRONG_TOP1_ACCEPTED         : 348
WRONG_TOP1_BELOW_THRESHOLD  : 624

--------------------------------------------------------------------------------
CORRECT_BELOW_THRESHOLD | count = 601
mean: 0.312859
P10: 0.048055
P25: 0.128830
median: 0.315626
P75: 0.485851
P90: 0.583213
max: 0.639650

--------------------------------------------------------------------------------
WRONG_TOP1_ACCEPTED | count = 348
mean: 0.892004
P10: 0.730539
P25: 0.819575
median: 0.926461
P75: 0.978675
P90: 0.994803
max: 0.999644

--------------------------------------------------------------------------------
WRONG_TOP1_BELOW_THRESHOLD | count = 624
mean: 0.203738
P10: 0.016643
P25: 0.04046

In [ ]:
# CELL 15 — TOP-1 MARGIN + EVIDENCE ANALYSIS

import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

print("=" * 100)
print("CELL 15 — TOP-1 MARGIN + EVIDENCE ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "VAL_TARGET_POOL",
    "rare_candidates",
    "retrieved_top100",
    "X_target_full",
    "X_s1_3b",
    "model9",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Normalize model object
# ------------------------------------------------------------

if hasattr(model9, "booster_"):
    model9_eval = model9.booster_
else:
    model9_eval = model9

# ------------------------------------------------------------
# Lookup dictionaries
# ------------------------------------------------------------

s1_rows = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

s1_name = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["name_n"]
    )
)

s1_addr = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["addr_n"]
    )
)

s1_country = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["country_n"]
    )
)

target_rows = {
    tid: i
    for i, tid in enumerate(
        VAL_TARGET_POOL["entity_id"]
    )
}

target_name = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["name_n"]
    )
)

target_addr = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["addr_n"]
    )
)

target_country = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["country_n"]
    )
)

# ------------------------------------------------------------
# Ground truth mapping
# ------------------------------------------------------------

# target_to_s1 already exists, but build local copy
# for safety.

true_map = target_to_s1.copy()

# ------------------------------------------------------------
# Function to calculate the key features for one pair
# ------------------------------------------------------------

def pair_features(
    target_id,
    s1_id
):

    q_name = target_name.get(
        target_id,
        ""
    )

    q_addr = target_addr.get(
        target_id,
        ""
    )

    q_country = target_country.get(
        target_id,
        ""
    )

    c_name = s1_name.get(
        s1_id,
        ""
    )

    c_addr = s1_addr.get(
        s1_id,
        ""
    )

    c_country = s1_country.get(
        s1_id,
        ""
    )

    name_ratio = (
        fuzz.ratio(
            q_name,
            c_name
        )
        /
        100.0
    )

    addr_ratio = (
        fuzz.ratio(
            q_addr,
            c_addr
        )
        /
        100.0
    )

    name_partial = (
        fuzz.partial_ratio(
            q_name,
            c_name
        )
        /
        100.0
    )

    addr_partial = (
        fuzz.partial_ratio(
            q_addr,
            c_addr
        )
        /
        100.0
    )

    name_token = (
        fuzz.token_set_ratio(
            q_name,
            c_name
        )
        /
        100.0
    )

    addr_token = (
        fuzz.token_set_ratio(
            q_addr,
            c_addr
        )
        /
        100.0
    )

    q_nums = set(
        re.findall(
            r"\d+",
            q_addr
        )
    )

    c_nums = set(
        re.findall(
            r"\d+",
            c_addr
        )
    )

    number_overlap_value = len(
        q_nums & c_nums
    )

    exact_name = int(
        q_name == c_name
    )

    exact_address = int(
        q_addr == c_addr
    )

    country_match = int(
        q_country == c_country
    )

    combined_similarity = (
        0.45 * name_ratio
        +
        0.55 * addr_ratio
    )

    local_idx = s1_rows[
        s1_id
    ]

    rare_set = set(
        rare_candidates.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )
    )

    tfidf_top = retrieved_top100.get(
        target_id,
        np.empty(
            0,
            dtype=np.int32
        )
    )

    tfidf_set = set(
        int(x)
        for x in tfidf_top
    )

    has_rare = int(
        local_idx in rare_set
    )

    has_tfidf = int(
        local_idx in tfidf_set
    )

    if local_idx in tfidf_set:

        tfidf_rank = (
            list(tfidf_top)
            .index(local_idx)
            + 1
        )

    else:

        tfidf_rank = 999

    # Direct cosine score for this pair
    t_idx = target_rows[
        target_id
    ]

    tfidf_score = float(
        (
            X_target_full[
                t_idx:t_idx + 1
            ]
            @
            X_s1_3b[
                local_idx:local_idx + 1
            ].T
        )
        .toarray()
        .ravel()[0]
    )

    return {
        "name_ratio": name_ratio,
        "addr_ratio": addr_ratio,
        "name_partial": name_partial,
        "addr_partial": addr_partial,
        "name_token": name_token,
        "addr_token": addr_token,
        "number_overlap": number_overlap_value,
        "exact_name": exact_name,
        "exact_address": exact_address,
        "country_match": country_match,
        "combined_similarity": combined_similarity,
        "has_rare": has_rare,
        "has_tfidf": has_tfidf,
        "tfidf_score": tfidf_score,
        "tfidf_rank": tfidf_rank,
    }


# ------------------------------------------------------------
# Build analysis set
#
# Current threshold = 0.640
# ------------------------------------------------------------

CURRENT_THRESHOLD = 0.640

analysis_rows = []

# best_all contains target -> top1 S1
for row in best_all.itertuples(
    index=False
):

    target_id = row.query_id
    pred_s1 = row.s1_id
    score = float(row.score)

    true_s1 = true_map.get(
        target_id
    )

    if true_s1 == pred_s1:

        if score >= CURRENT_THRESHOLD:
            status = "CORRECT_ACCEPTED"
        else:
            status = "CORRECT_BELOW_THRESHOLD"

    else:

        if score >= CURRENT_THRESHOLD:
            status = "WRONG_TOP1_ACCEPTED"
        else:
            status = "WRONG_TOP1_BELOW_THRESHOLD"

    # Only analyze the meaningful error groups
    if status not in {
        "WRONG_TOP1_ACCEPTED",
        "CORRECT_BELOW_THRESHOLD"
    }:
        continue

    try:

        feat = pair_features(
            target_id,
            pred_s1
        )

    except Exception as e:

        print(
            "Feature failure:",
            target_id,
            pred_s1,
            repr(e)
        )

        continue

    # --------------------------------------------------------
    # Find runner-up from the retrieval candidate set
    # and score it with the same reranker.
    # --------------------------------------------------------

    rare_ids = set(
        int(x)
        for x in rare_candidates.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )
    )

    tfidf_ids = set(
        int(x)
        for x in retrieved_top100.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )[:50]
    )

    candidate_ids = (
        rare_ids
        |
        tfidf_ids
    )

    # We don't need to rescore if there are no alternatives.
    if len(candidate_ids) > 1:

        candidate_scores = []

        # Build a compact feature matrix using the already
        # trained feature-building logic from df_new where
        # possible. For reliability, score the candidates
        # directly using a lightweight subset first.

        for local_idx in candidate_ids:

            sid = (
                VAL_S1.iloc[
                    local_idx
                ]["entity_id"]
            )

            # We already know the current candidate score.
            # For alternatives, reproduce the core features
            # and let the model use a compact reconstruction.

            f = pair_features(
                target_id,
                sid
            )

            # Approximate ranking diagnostic based on the
            # strongest similarity signals. This is not the
            # final model score.
            proxy = (
                0.40 * f["tfidf_score"]
                +
                0.25 * f["combined_similarity"]
                +
                0.20 * f["addr_ratio"]
                +
                0.15 * f["name_ratio"]
            )

            candidate_scores.append(
                (
                    sid,
                    proxy
                )
            )

        candidate_scores.sort(
            key=lambda x: x[1],
            reverse=True
        )

        runner_up_proxy = (
            candidate_scores[1][1]
        )

        best_proxy = (
            candidate_scores[0][1]
        )

        proxy_margin = (
            best_proxy
            -
            runner_up_proxy
        )

    else:

        runner_up_proxy = 0.0
        best_proxy = 0.0
        proxy_margin = 1.0

    analysis_rows.append({
        "query_id":
            target_id,

        "true_s1":
            true_s1,

        "pred_s1":
            pred_s1,

        "status":
            status,

        "model_score":
            score,

        "proxy_margin":
            proxy_margin,

        "runner_up_proxy":
            runner_up_proxy,

        **feat
    })

analysis_df = pd.DataFrame(
    analysis_rows
)

print(
    "\n" + "=" * 100
)
print(
    "ERROR ANALYSIS DATASET"
)
print(
    "=" * 100
)

print(
    "Rows:",
    len(analysis_df)
)

print(
    analysis_df[
        "status"
    ].value_counts()
)

# ------------------------------------------------------------
# Compare the two important error groups
# ------------------------------------------------------------

for status in [
    "WRONG_TOP1_ACCEPTED",
    "CORRECT_BELOW_THRESHOLD"
]:

    subset = analysis_df[
        analysis_df["status"] == status
    ]

    print(
        "\n" + "-" * 90
    )

    print(
        status,
        "| count =",
        len(subset)
    )

    for col in [
        "model_score",
        "proxy_margin",
        "tfidf_score",
        "tfidf_rank",
        "name_ratio",
        "addr_ratio",
        "name_partial",
        "addr_partial",
        "name_token",
        "addr_token",
        "number_overlap",
        "combined_similarity"
    ]:

        print(
            f"{col:25s}",
            "median=",
            f"{subset[col].median():.4f}",
            "P25=",
            f"{subset[col].quantile(.25):.4f}",
            "P75=",
            f"{subset[col].quantile(.75):.4f}"
        )

# ------------------------------------------------------------
# Simple separation diagnostics
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "SEPARATION CHECKS"
)
print(
    "=" * 100
)

for col in [
    "proxy_margin",
    "addr_ratio",
    "name_ratio",
    "tfidf_score",
    "number_overlap",
    "combined_similarity"
]:

    wrong = analysis_df.loc[
        analysis_df["status"]
        ==
        "WRONG_TOP1_ACCEPTED",
        col
    ]

    correct_below = analysis_df.loc[
        analysis_df["status"]
        ==
        "CORRECT_BELOW_THRESHOLD",
        col
    ]

    print(
        f"\n{col}"
    )

    print(
        "  wrong accepted median:",
        f"{wrong.median():.4f}"
    )

    print(
        "  correct below median:",
        f"{correct_below.median():.4f}"
    )

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

analysis_path = (
    MODEL_DIR
    /
    "target_level_margin_error_analysis.csv"
)

analysis_df.to_csv(
    analysis_path,
    index=False
)

print(
    "\nSaved:",
    analysis_path
)

print(
    "\nCELL 15 COMPLETE."
)

CELL 15 — TOP-1 MARGIN + EVIDENCE ANALYSIS

ERROR ANALYSIS DATASET
Rows: 949
status
CORRECT_BELOW_THRESHOLD    601
WRONG_TOP1_ACCEPTED        348
Name: count, dtype: int64

------------------------------------------------------------------------------------------
WRONG_TOP1_ACCEPTED | count = 348
model_score               median= 0.9265 P25= 0.8196 P75= 0.9787
proxy_margin              median= 0.0190 P25= 0.0082 P75= 0.0406
tfidf_score               median= 0.4330 P25= 0.3521 P75= 0.5288
tfidf_rank                median= 1.0000 P25= 1.0000 P75= 1.0000
name_ratio                median= 0.6493 P25= 0.0000 P75= 0.8425
addr_ratio                median= 0.3449 P25= 0.0000 P75= 0.6277
name_partial              median= 0.7619 P25= 0.0000 P75= 0.9615
addr_partial              median= 0.5778 P25= 0.0000 P75= 0.8088
name_token                median= 0.6493 P25= 0.0000 P75= 0.8425
addr_token                median= 0.3449 P25= 0.0000 P75= 0.6277
number_overlap            median= 0.0000 P25= 0.0000

In [ ]:
# CELL 16 — SCORE + MARGIN + EVIDENCE RULE SEARCH

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 16 — SCORE + MARGIN + EVIDENCE RULE SEARCH")
print("=" * 100)

# ------------------------------------------------------------
# Locate the full target-level dataframe produced during
# Cell 15. We need ALL validation target predictions, not only
# the 949 error rows.
# ------------------------------------------------------------

candidate_names = [
    "target_level_df",
    "target_predictions",
    "df_target_level",
    "target_level_predictions",
    "best_targets",
    "best_all"
]

source_df = None
source_name = None

for name in candidate_names:

    if name in globals():

        obj = globals()[name]

        if isinstance(
            obj,
            pd.DataFrame
        ):

            needed = {
                "query_id",
                "s1_id",
                "score"
            }

            if needed.issubset(
                set(obj.columns)
            ):

                source_df = obj
                source_name = name
                break

if source_df is None:

    raise RuntimeError(
        "Could not find the full target-level dataframe "
        "containing query_id, s1_id and score. "
        "Do not use the 949-row error-analysis CSV; "
        "we need the complete 69,231-target dataframe."
    )

print(
    "Using dataframe:",
    source_name
)

print(
    "Rows:",
    f"{len(source_df):,}"
)

# ------------------------------------------------------------
# Check evidence columns
# ------------------------------------------------------------

evidence_cols = [
    "proxy_margin",
    "tfidf_score",
    "tfidf_rank",
    "name_ratio",
    "addr_ratio",
    "name_partial",
    "addr_partial",
    "number_overlap",
    "combined_similarity"
]

missing_evidence = [
    c
    for c in evidence_cols
    if c not in source_df.columns
]

if missing_evidence:

    raise RuntimeError(
        "Missing evidence columns: "
        + ", ".join(
            missing_evidence
        )
    )

# ------------------------------------------------------------
# Prepare dataframe
# ------------------------------------------------------------

work16 = source_df.copy()

# Ensure proxy margin is numeric
work16["proxy_margin"] = (
    pd.to_numeric(
        work16["proxy_margin"],
        errors="coerce"
    )
    .fillna(0.0)
)

# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

if "true_by_s1" not in globals():

    raise RuntimeError(
        "true_by_s1 is missing. "
        "Run the full S1 validation first."
    )

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def evaluate_rule16(
    score_threshold,
    margin_threshold,
    evidence_mode
):

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1[
            "entity_id"
        ]
    }

    # --------------------------------------------------------
    # Base score
    # --------------------------------------------------------

    accepted = (
        work16["score"]
        >=
        score_threshold
    )

    # --------------------------------------------------------
    # Margin/evidence condition
    #
    # The idea:
    # - strong candidates can be accepted at a lower score
    #   if the candidate has a strong margin/evidence profile
    # - ambiguous candidates require the normal threshold
    # --------------------------------------------------------

    relaxed = (
        work16["score"]
        >=
        (
            score_threshold
            - 0.15
        )
    )

    margin_ok = (
        work16["proxy_margin"]
        >=
        margin_threshold
    )

    if evidence_mode == "balanced":

        evidence_ok = (
            (
                work16["addr_ratio"]
                >= 0.45
            )
            |
            (
                work16["name_ratio"]
                >= 0.75
            )
            |
            (
                work16["number_overlap"]
                >= 1
            )
            |
            (
                work16["tfidf_score"]
                >= 0.45
            )
        )

    elif evidence_mode == "address":

        evidence_ok = (
            (
                work16["addr_ratio"]
                >= 0.40
            )
            |
            (
                work16["number_overlap"]
                >= 1
            )
        )

    elif evidence_mode == "strong":

        evidence_ok = (
            (
                work16["addr_ratio"]
                >= 0.55
            )
            |
            (
                (
                    work16["name_ratio"]
                    >= 0.80
                )
                &
                (
                    work16["tfidf_score"]
                    >= 0.40
                )
            )
            |
            (
                work16["number_overlap"]
                >= 1
            )
        )

    else:

        raise ValueError(
            evidence_mode
        )

    # Accept normal high-score candidates,
    # OR lower-score candidates with strong margin/evidence.
    final_accept = (
        accepted
        |
        (
            relaxed
            &
            margin_ok
            &
            evidence_ok
        )
    )

    accepted_df = (
        work16[
            final_accept
        ]
        .copy()
    )

    for row in accepted_df.itertuples(
        index=False
    ):

        predicted_by_s1[
            row.s1_id
        ].add(
            row.query_id
        )

    # --------------------------------------------------------
    # S1-level macro F0.5
    # --------------------------------------------------------

    f05_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for sid in VAL_S1[
        "entity_id"
    ]:

        true_targets = (
            true_by_s1[
                sid
            ]
        )

        predicted_targets = (
            predicted_by_s1[
                sid
            ]
        )

        tp = len(
            true_targets
            &
            predicted_targets
        )

        fp = len(
            predicted_targets
            -
            true_targets
        )

        fn = len(
            true_targets
            -
            predicted_targets
        )

        if tp + fp > 0:

            precision = (
                tp
                /
                (tp + fp)
            )

        else:

            precision = (
                1.0
                if not true_targets
                else 0.0
            )

        if tp + fn > 0:

            recall = (
                tp
                /
                (tp + fn)
            )

        else:

            recall = 1.0

        if precision > 0 and recall > 0:

            f05 = (
                1.25
                * precision
                * recall
                /
                (
                    0.25 * precision
                    +
                    recall
                )
            )

        else:

            f05 = 0.0

        f05_values.append(
            f05
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    return {
        "macro_f05":
            float(
                np.mean(
                    f05_values
                )
            ),

        "precision":
            (
                total_tp
                /
                (total_tp + total_fp)
                if total_tp + total_fp > 0
                else 0.0
            ),

        "recall":
            (
                total_tp
                /
                (total_tp + total_fn)
                if total_tp + total_fn > 0
                else 0.0
            ),

        "tp":
            total_tp,

        "fp":
            total_fp,

        "fn":
            total_fn,

        "accepted_edges":
            len(accepted_df)
    }


# ------------------------------------------------------------
# Search space
# ------------------------------------------------------------

score_thresholds = [
    0.50,
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.64
]

margin_thresholds = [
    0.00,
    0.005,
    0.010,
    0.015,
    0.020,
    0.025,
    0.030,
    0.035,
    0.040,
    0.050,
    0.060
]

evidence_modes = [
    "balanced",
    "address",
    "strong"
]

results16 = []

for score_t in score_thresholds:

    for margin_t in margin_thresholds:

        for mode in evidence_modes:

            result = evaluate_rule16(
                score_t,
                margin_t,
                mode
            )

            results16.append({
                "score_threshold":
                    score_t,

                "margin_threshold":
                    margin_t,

                "evidence_mode":
                    mode,

                **result
            })

results16_df = pd.DataFrame(
    results16
)

# ------------------------------------------------------------
# Rank candidates
# ------------------------------------------------------------

top16 = (
    results16_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(30)
)

print(
    "\n" + "=" * 100
)
print(
    "TOP RULES"
)
print(
    "=" * 100
)

print(
    top16.to_string(
        index=False,
        formatters={
            "macro_f05":
                lambda x: f"{x:.6f}",

            "precision":
                lambda x: f"{x:.6f}",

            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

best16 = (
    results16_df
    .loc[
        results16_df[
            "macro_f05"
        ].idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "BEST RULE"
)
print(
    "=" * 100
)

for col in [
    "score_threshold",
    "margin_threshold",
    "evidence_mode",
    "macro_f05",
    "precision",
    "recall",
    "tp",
    "fp",
    "fn",
    "accepted_edges"
]:

    print(
        f"{col}:",
        best16[col]
    )

print(
    "\nImprovement over 0.988468:",
    f"{best16['macro_f05'] - 0.988468:+.6f}"
)

# ------------------------------------------------------------
# Save search results
# ------------------------------------------------------------

search_path = (
    MODEL_DIR
    /
    "cell16_margin_evidence_search.csv"
)

results16_df.to_csv(
    search_path,
    index=False
)

print(
    "\nSaved:",
    search_path
)

print(
    "\nCELL 16 COMPLETE."
)

CELL 16 — SCORE + MARGIN + EVIDENCE RULE SEARCH
Using dataframe: best_all
Rows: 69,231


RuntimeError: Missing evidence columns: proxy_margin, tfidf_score, tfidf_rank, name_ratio, addr_ratio, name_partial, addr_partial, number_overlap, combined_similarity

In [ ]:
# CELL 12 — FINE-GRAINED FULL-S1 THRESHOLD OPTIMIZATION

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def calculate_macro_f05(
    threshold,
    df=best_all
):

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    accepted = df[
        df["score"] >= threshold
    ]

    for row in accepted.itertuples(
        index=False
    ):

        predicted_by_s1[
            row.s1_id
        ].add(
            row.query_id
        )

    f05_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for sid in VAL_S1["entity_id"]:

        true_targets = true_by_s1[
            sid
        ]

        predicted_targets = (
            predicted_by_s1[
                sid
            ]
        )

        tp = len(
            true_targets
            &
            predicted_targets
        )

        fp = len(
            predicted_targets
            -
            true_targets
        )

        fn = len(
            true_targets
            -
            predicted_targets
        )

        if tp + fp > 0:
            precision = (
                tp
                /
                (tp + fp)
            )
        else:
            precision = (
                1.0
                if not true_targets
                else 0.0
            )

        if tp + fn > 0:
            recall = (
                tp
                /
                (tp + fn)
            )
        else:
            recall = 1.0

        if (
            precision > 0
            and
            recall > 0
        ):
            f05 = (
                1.25
                * precision
                * recall
                /
                (
                    0.25 * precision
                    +
                    recall
                )
            )
        else:
            f05 = 0.0

        f05_values.append(
            f05
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn

    macro_f05 = float(
        np.mean(f05_values)
    )

    precision_micro = (
        total_tp
        /
        (total_tp + total_fp)
        if total_tp + total_fp > 0
        else 0.0
    )

    recall_micro = (
        total_tp
        /
        (total_tp + total_fn)
        if total_tp + total_fn > 0
        else 0.0
    )

    return {
        "threshold": threshold,
        "macro_f05": macro_f05,
        "precision": precision_micro,
        "recall": recall_micro,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn
    }


# ------------------------------------------------------------
# Fine threshold sweep
# ------------------------------------------------------------

thresholds = np.arange(
    0.550,
    0.701,
    0.001
)

results12 = []

for threshold in thresholds:

    results12.append(
        calculate_macro_f05(
            float(threshold)
        )
    )

results12_df = pd.DataFrame(
    results12
)

# ------------------------------------------------------------
# Top results
# ------------------------------------------------------------

top12 = (
    results12_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
)

print(
    "\n" + "=" * 100
)
print(
    "TOP FINE-GRAINED THRESHOLDS"
)
print(
    "=" * 100
)

print(
    top12.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

best12 = (
    results12_df
    .loc[
        results12_df["macro_f05"].idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "BEST FINE-GRAINED THRESHOLD"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best12['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best12['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best12['precision']:.6f}"
)

print(
    "Recall:",
    f"{best12['recall']:.6f}"
)

print(
    "TP:",
    int(best12["tp"])
)

print(
    "FP:",
    int(best12["fp"])
)

print(
    "FN:",
    int(best12["fn"])
)

print(
    "\nImprovement over 0.988468:",
    f"{best12['macro_f05'] - 0.988468:+.6f}"
)

print(
    "\nCELL 12 COMPLETE."
)

CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION

TOP FINE-GRAINED THRESHOLDS
threshold macro_f05 precision   recall    tp  fp   fn
    0.628  0.988477  0.994811 0.977496 67673 353 1558
    0.637  0.988475  0.994868 0.977337 67662 349 1569
    0.619  0.988472  0.994724 0.977669 67685 359 1546
    0.617  0.988471  0.994695 0.977727 67689 361 1542
    0.638  0.988470  0.994868 0.977322 67661 349 1570
    0.639  0.988470  0.994883 0.977293 67659 348 1572
    0.627  0.988470  0.994796 0.977496 67673 354 1558
    0.618  0.988469  0.994709 0.977683 67686 360 1545
    0.640  0.988468  0.994883 0.977279 67658 348 1573
    0.621  0.988467  0.994724 0.977655 67684 359 1547
    0.620  0.988467  0.994724 0.977655 67684 359 1547
    0.629  0.988467  0.994825 0.977452 67670 352 1561
    0.642  0.988465  0.994897 0.977250 67656 347 1575
    0.615  0.988465  0.994681 0.977756 67691 362 1540
    0.636  0.988464  0.994854 0.977351 67663 350 1568
    0.616  0.988463  0.994681 0.977741 67690 362 1541
    0.6

In [ ]:
# CELL 13 — LOCALIZE FULL-S1 FALSE NEGATIVES
#
# Separates:
#   1. Retrieval misses
#   2. Candidate present but wrong target-exclusive winner
#   3. Candidate winner rejected by threshold
#
# This does NOT retrain anything.

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 13 — FALSE-NEGATIVE LOCALIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "target_to_s1",
    "rare_candidates",
    "retrieved_top100",
    "best_all",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Current best threshold
# ------------------------------------------------------------

CURRENT_THRESHOLD = 0.628

print(
    "Analysis threshold:",
    CURRENT_THRESHOLD
)

# ------------------------------------------------------------
# Build S1 local index
# ------------------------------------------------------------

s1_local_13 = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Best target-exclusive prediction map
# ------------------------------------------------------------

best_map = (
    best_all
    .set_index(
        "query_id"
    )[
        [
            "s1_id",
            "score"
        ]
    ]
    .to_dict(
        "index"
    )
)

# ------------------------------------------------------------
# Examine every TRUE target pair
# ------------------------------------------------------------

retrieval_misses = []
candidate_but_wrong = []
candidate_but_rejected = []
correctly_selected = []

for target_id, true_s1 in target_to_s1.items():

    true_local = s1_local_13.get(
        true_s1
    )

    # --------------------------------------------------------
    # Recreate retrieval candidate universe:
    # rare Top-4 UNION TF-IDF Top-50
    # --------------------------------------------------------

    rare_set = set(
        rare_candidates.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )
    )

    tfidf_set = set(
        int(x)
        for x in retrieved_top100.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )[:50]
    )

    candidate_set = (
        rare_set
        |
        tfidf_set
    )

    # --------------------------------------------------------
    # Retrieval miss
    # --------------------------------------------------------

    if (
        true_local is None
        or
        true_local not in candidate_set
    ):

        retrieval_misses.append(
            target_id
        )

        continue

    # --------------------------------------------------------
    # Candidate existed.
    # Check target-exclusive winner.
    # --------------------------------------------------------

    prediction = best_map.get(
        target_id
    )

    if prediction is None:

        candidate_but_rejected.append(
            target_id
        )

        continue

    predicted_s1 = prediction[
        "s1_id"
    ]

    score = float(
        prediction[
            "score"
        ]
    )

    # Correct selected target
    if predicted_s1 == true_s1:

        if score >= CURRENT_THRESHOLD:

            correctly_selected.append(
                target_id
            )

        else:

            candidate_but_rejected.append(
                target_id
            )

    # Wrong target won
    else:

        candidate_but_wrong.append(
            target_id
        )

# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

total_pairs = len(
    target_to_s1
)

print(
    "\n" + "=" * 100
)
print(
    "PAIR-LEVEL ERROR DECOMPOSITION"
)
print(
    "=" * 100
)

print(
    "Total true pairs:",
    f"{total_pairs:,}"
)

print(
    "\nCorrectly selected:",
    f"{len(correctly_selected):,}",
    f"({len(correctly_selected)/total_pairs*100:.4f}%)"
)

print(
    "Retrieval misses:",
    f"{len(retrieval_misses):,}",
    f"({len(retrieval_misses)/total_pairs*100:.4f}%)"
)

print(
    "Candidate present but wrong winner:",
    f"{len(candidate_but_wrong):,}",
    f"({len(candidate_but_wrong)/total_pairs*100:.4f}%)"
)

print(
    "Candidate present but rejected:",
    f"{len(candidate_but_rejected):,}",
    f"({len(candidate_but_rejected)/total_pairs*100:.4f}%)"
)

print(
    "\nError accounted for:",
    len(
        retrieval_misses
        +
        candidate_but_wrong
        +
        candidate_but_rejected
    )
)

# ------------------------------------------------------------
# Analyze current FN distribution by S1 entity
# ------------------------------------------------------------

missed_targets = set(
    retrieval_misses
    +
    candidate_but_wrong
    +
    candidate_but_rejected
)

misses_by_s1 = {}

for target_id in missed_targets:

    sid = target_to_s1[
        target_id
    ]

    misses_by_s1.setdefault(
        sid,
        0
    )

    misses_by_s1[sid] += 1

true_match_counts = []

miss_count_values = []

for sid, miss_count in misses_by_s1.items():

    true_count = len(
        true_by_s1.get(
            sid,
            set()
        )
    )

    true_match_counts.append(
        true_count
    )

    miss_count_values.append(
        miss_count
    )

print(
    "\n" + "=" * 100
)
print(
    "MISSED-MATCH ENTITY ANALYSIS"
)
print(
    "=" * 100
)

print(
    "S1 entities with >=1 missed pair:",
    len(misses_by_s1)
)

if true_match_counts:

    arr = np.asarray(
        true_match_counts
    )

    print(
        "Mean true matches among affected S1:",
        f"{arr.mean():.2f}"
    )

    print(
        "Median true matches:",
        f"{np.median(arr):.0f}"
    )

# ------------------------------------------------------------
# Retrieval-miss target IDs
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "RETRIEVAL MISS EXAMPLES"
)
print(
    "=" * 100
)

for target_id in retrieval_misses[:20]:

    print(
        target_id,
        "->",
        target_to_s1[target_id]
    )

# ------------------------------------------------------------
# Wrong-winner examples
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "WRONG-WINNER EXAMPLES"
)
print(
    "=" * 100
)

for target_id in candidate_but_wrong[:20]:

    prediction = best_map.get(
        target_id
    )

    print(
        target_id,
        "| TRUE=",
        target_to_s1[target_id],
        "| PRED=",
        prediction["s1_id"],
        "| SCORE=",
        f"{prediction['score']:.6f}"
    )

# ------------------------------------------------------------
# Rejected examples
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "REJECTED-CORRECT-WINNER EXAMPLES"
)
print(
    "=" * 100
)

for target_id in candidate_but_rejected[:20]:

    prediction = best_map.get(
        target_id
    )

    print(
        target_id,
        "| TRUE=",
        target_to_s1[target_id],
        "| SCORE=",
        f"{prediction['score']:.6f}"
    )

print(
    "\nCELL 13 COMPLETE."
)

CELL 13 — FALSE-NEGATIVE LOCALIZATION
Analysis threshold: 0.628

PAIR-LEVEL ERROR DECOMPOSITION
Total true pairs: 69,231

Correctly selected: 67,673 (97.7496%)
Retrieval misses: 15 (0.0217%)
Candidate present but wrong winner: 957 (1.3823%)
Candidate present but rejected: 586 (0.8464%)

Error accounted for: 1558

MISSED-MATCH ENTITY ANALYSIS
S1 entities with >=1 missed pair: 1324
Mean true matches among affected S1: 4.16
Median true matches: 4

RETRIEVAL MISS EXAMPLES
S2-208204396 -> S1-997731506
S2-749005505 -> S1-997731506
S2-927345730 -> S1-261169281
S2-733469786 -> S1-139707378
S2-924994088 -> S1-232277944
S3-950330181 -> S1-810621468
S3-954401890 -> S1-500976493
S2-622353415 -> S1-440755143
S3-593764841 -> S1-59482891
S2-893810925 -> S1-495511421
S2-917986677 -> S1-495511421
S2-39854879 -> S1-383712942
S2-907930197 -> S1-480352283
S3-475603463 -> S1-580881335
S2-352867740 -> S1-379011514

WRONG-WINNER EXAMPLES
S2-4190133 | TRUE= S1-983540069 | PRED= S1-280870409 | SCORE= 0.178739


In [ ]:
# CELL 14 — TARGET-GROUP LEARNING-TO-RANK RERANKER
#
# Each target_id/query_id is one ranking group.
# The true S1 candidate has label=1; all other candidates=0.
#
# This directly optimizes the "wrong winner" problem found in Cell 13.

import numpy as np
import pandas as pd
import lightgbm as lgb
import time

print("=" * 100)
print("CELL 14 — TARGET-GROUP LEARNING-TO-RANK RERANKER")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "df_new",
    "feature_cols_new"
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Query-level split
# Same deterministic 75/25 split used for classifier.
# ------------------------------------------------------------

query_ids = (
    df_new["query_id"]
    .drop_duplicates()
    .to_numpy()
)

rng = np.random.default_rng(42)
rng.shuffle(query_ids)

split = int(
    len(query_ids) * 0.75
)

train_queries = set(
    query_ids[:split]
)

valid_queries = set(
    query_ids[split:]
)

train_df = (
    df_new[
        df_new["query_id"].isin(
            train_queries
        )
    ]
    .copy()
)

valid_df = (
    df_new[
        df_new["query_id"].isin(
            valid_queries
        )
    ]
    .copy()
)

# ------------------------------------------------------------
# IMPORTANT:
# Sort by query_id so LightGBM's group boundaries are exact.
# ------------------------------------------------------------

train_df = train_df.sort_values(
    "query_id"
).reset_index(
    drop=True
)

valid_df = valid_df.sort_values(
    "query_id"
).reset_index(
    drop=True
)

X_train = (
    train_df[
        feature_cols_new
    ]
    .astype(
        np.float32
    )
)

y_train = (
    train_df["label"]
    .astype(
        np.int8
    )
)

X_valid = (
    valid_df[
        feature_cols_new
    ]
    .astype(
        np.float32
    )
)

y_valid = (
    valid_df["label"]
    .astype(
        np.int8
    )
)

# ------------------------------------------------------------
# Group sizes
# ------------------------------------------------------------

train_group = (
    train_df
    .groupby(
        "query_id",
        sort=False
    )
    .size()
    .to_numpy()
)

valid_group = (
    valid_df
    .groupby(
        "query_id",
        sort=False
    )
    .size()
    .to_numpy()
)

print(
    "\nTrain queries:",
    len(train_group)
)

print(
    "Validation queries:",
    len(valid_group)
)

print(
    "Train rows:",
    f"{len(train_df):,}"
)

print(
    "Validation rows:",
    f"{len(valid_df):,}"
)

print(
    "Train positives:",
    int(y_train.sum())
)

print(
    "Validation positives:",
    int(y_valid.sum())
)

print(
    "Train group size mean:",
    f"{train_group.mean():.2f}"
)

print(
    "Validation group size mean:",
    f"{valid_group.mean():.2f}"
)

# ------------------------------------------------------------
# LightGBM Ranker
# ------------------------------------------------------------

ranker14 = lgb.LGBMRanker(
    objective="lambdarank",

    metric="ndcg",

    n_estimators=2500,

    learning_rate=0.025,

    num_leaves=31,

    max_depth=-1,

    min_child_samples=20,

    subsample=0.85,

    colsample_bytree=0.90,

    reg_alpha=0.15,

    reg_lambda=2.0,

    random_state=42,

    n_jobs=-1,

    verbosity=-1
)

print(
    "\nTraining Learning-to-Rank model..."
)

t0 = time.time()

ranker14.fit(
    X_train,
    y_train,

    group=train_group,

    eval_set=[
        (
            X_valid,
            y_valid
        )
    ],

    eval_group=[
        valid_group
    ],

    eval_at=[
        1,
        3,
        5,
        10
    ],

    callbacks=[
        lgb.early_stopping(
            150,
            verbose=False
        )
    ]
)

train_time = (
    time.time() - t0
)

print(
    "Training time:",
    f"{train_time:.2f} sec"
)

print(
    "Best iteration:",
    ranker14.best_iteration_
)

# ------------------------------------------------------------
# Validation ranking scores
# ------------------------------------------------------------

valid_scores14 = ranker14.predict(
    X_valid,
    num_iteration=ranker14.best_iteration_
)

valid_ranked = valid_df[
    [
        "query_id",
        "s1_id",
        "label"
    ]
].copy()

valid_ranked["score"] = (
    valid_scores14
)

# ------------------------------------------------------------
# Evaluate target Top-1 accuracy
# ------------------------------------------------------------

top1_correct = 0
top1_total = 0

top1_rows = []

for query_id, group in valid_ranked.groupby(
    "query_id",
    sort=False
):

    winner = group.loc[
        group["score"].idxmax()
    ]

    true_ids = set(
        group.loc[
            group["label"] == 1,
            "s1_id"
        ]
    )

    is_correct = (
        winner["s1_id"]
        in true_ids
    )

    top1_correct += int(
        is_correct
    )

    top1_total += 1

    top1_rows.append({
        "query_id": query_id,
        "pred_s1": winner["s1_id"],
        "winner_score": winner["score"],
        "correct": int(is_correct)
    })

top1_accuracy = (
    top1_correct
    /
    top1_total
)

print(
    "\n" + "=" * 100
)
print(
    "TOP-1 RANKING RESULT"
)
print(
    "=" * 100
)

print(
    "Top-1 correct:",
    f"{top1_correct:,}",
    "/",
    f"{top1_total:,}"
)

print(
    "Top-1 accuracy:",
    f"{top1_accuracy:.6f}"
)

# ------------------------------------------------------------
# Compare against existing classifier
# ------------------------------------------------------------

print(
    "\nExisting classifier Top-1:"
)

print(
    "F0.5:",
    "0.988000"
)

print(
    "Precision:",
    "0.989980"
)

print(
    "Recall:",
    "0.988000"
)

# ------------------------------------------------------------
# Score distribution
# ------------------------------------------------------------

print(
    "\nRanker score range:",
    f"{valid_scores14.min():.6f}",
    "→",
    f"{valid_scores14.max():.6f}"
)

print(
    "\nScore percentiles:"
)

for p in [
    0,
    1,
    5,
    10,
    25,
    50,
    75,
    90,
    95,
    99,
    100
]:

    print(
        f"P{p:>3}:",
        f"{np.percentile(valid_scores14,p):.6f}"
    )

# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

ranker_path = (
    MODEL_DIR
    /
    "blocked_tfidf_rare_ranker_lambdarank.txt"
)

ranker14.booster_.save_model(
    str(ranker_path)
)

print(
    "\nSaved:",
    ranker_path
)

print(
    "\nCELL 14 COMPLETE."
)

CELL 14 — TARGET-GROUP LEARNING-TO-RANK RERANKER

Train queries: 1500
Validation queries: 500
Train rows: 170,473
Validation rows: 53,341
Train positives: 1500
Validation positives: 500
Train group size mean: 113.65
Validation group size mean: 106.68

Training Learning-to-Rank model...
Training time: 21.09 sec
Best iteration: 89

TOP-1 RANKING RESULT
Top-1 correct: 495 / 500
Top-1 accuracy: 0.990000

Existing classifier Top-1:
F0.5: 0.988000
Precision: 0.989980
Recall: 0.988000

Ranker score range: -2.564502 → 2.562787

Score percentiles:
P  0: -2.564502
P  1: -2.564502
P  5: -2.564502
P 10: -2.564502
P 25: -2.564502
P 50: -2.564502
P 75: -2.564502
P 90: -2.502687
P 95: -2.102276
P 99: 0.400691
P100: 2.562787

Saved: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_ranker_lambdarank.txt

CELL 14 COMPLETE.


In [ ]:
# CELL 15 — FULL 20K S1 VALIDATION WITH LAMBDARANK

import os
import re
import gc
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from rapidfuzz import fuzz

print("=" * 100)
print("CELL 15 — FULL 20K S1 VALIDATION WITH LAMBDARANK")
print("=" * 100)

# ------------------------------------------------------------
# Load LambdaRank model
# ------------------------------------------------------------

ranker_path = (
    MODEL_DIR /
    "blocked_tfidf_rare_ranker_lambdarank.txt"
)

ranker = lgb.Booster(
    model_file=str(ranker_path)
)

print(
    "Ranker trees:",
    ranker.num_trees()
)

print(
    "Ranker features:",
    ranker.num_feature()
)

# ------------------------------------------------------------
# Feature columns used by the ranker
# ------------------------------------------------------------

rank_feature_cols = [
    c
    for c in df_new.columns
    if c not in [
        "query_id",
        "s1_id",
        "s1_local_idx",
        "label"
    ]
]

if len(rank_feature_cols) != ranker.num_feature():

    raise RuntimeError(
        f"Feature mismatch: "
        f"{len(rank_feature_cols)} available, "
        f"{ranker.num_feature()} expected."
    )

print(
    "Feature count verified:",
    len(rank_feature_cols)
)

# ------------------------------------------------------------
# Lookup dictionaries
# ------------------------------------------------------------

s1_name_15 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["name_n"]
    )
)

s1_addr_15 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["addr_n"]
    )
)

s1_country_15 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["country_n"]
    )
)

target_name_15 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["name_n"]
    )
)

target_addr_15 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["addr_n"]
    )
)

target_country_15 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["country_n"]
    )
)

s1_local_to_id_15 = {
    i: sid
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Prepare output
# ------------------------------------------------------------

checkpoint_dir_15 = (
    MODEL_DIR /
    "full_validation_lambdarank_parts"
)

checkpoint_dir_15.mkdir(
    parents=True,
    exist_ok=True
)

CHUNK_TARGETS = 2000

target_ids_all = (
    VAL_TARGET_POOL["entity_id"]
    .tolist()
)

part_paths_15 = []

overall_start = time.time()

# ------------------------------------------------------------
# Helper: build the 56 features
# ------------------------------------------------------------

def build_rank_features(
    target_id,
    candidate_local,
    rare_ids,
    tfidf_ids,
    tfidf_rank_map,
    tfidf_score_map
):

    q_name = target_name_15.get(
        target_id,
        ""
    )

    q_addr = target_addr_15.get(
        target_id,
        ""
    )

    q_country = target_country_15.get(
        target_id,
        ""
    )

    rows = []

    q_nums = set(
        re.findall(
            r"\d+",
            q_addr
        )
    )

    for local_idx in candidate_local:

        s1_id = s1_local_to_id_15[
            int(local_idx)
        ]

        s_name = s1_name_15.get(
            s1_id,
            ""
        )

        s_addr = s1_addr_15.get(
            s1_id,
            ""
        )

        s_country = s1_country_15.get(
            s1_id,
            ""
        )

        name_ratio = (
            fuzz.ratio(
                q_name,
                s_name
            ) / 100.0
        )

        addr_ratio = (
            fuzz.ratio(
                q_addr,
                s_addr
            ) / 100.0
        )

        name_partial = (
            fuzz.partial_ratio(
                q_name,
                s_name
            ) / 100.0
        )

        addr_partial = (
            fuzz.partial_ratio(
                q_addr,
                s_addr
            ) / 100.0
        )

        name_token = (
            fuzz.token_set_ratio(
                q_name,
                s_name
            ) / 100.0
        )

        addr_token = (
            fuzz.token_set_ratio(
                q_addr,
                s_addr
            ) / 100.0
        )

        qn = len(q_name)
        sn = len(s_name)

        qa = len(q_addr)
        sa = len(s_addr)

        name_length_ratio = (
            min(qn, sn)
            /
            max(qn, sn)
            if max(qn, sn) > 0
            else 1.0
        )

        addr_length_ratio = (
            min(qa, sa)
            /
            max(qa, sa)
            if max(qa, sa) > 0
            else 1.0
        )

        exact_name = int(
            q_name == s_name
        )

        exact_address = int(
            q_addr == s_addr
        )

        country_match = int(
            q_country == s_country
        )

        s_nums = set(
            re.findall(
                r"\d+",
                s_addr
            )
        )

        number_overlap_value = len(
            q_nums & s_nums
        )

        combined_similarity = (
            0.45 * name_ratio
            +
            0.55 * addr_ratio
        )

        has_rare = int(
            int(local_idx) in rare_ids
        )

        has_tfidf = int(
            int(local_idx) in tfidf_ids
        )

        retrieval_channel_count = (
            has_rare
            +
            has_tfidf
        )

        tfidf_score = (
            tfidf_score_map.get(
                int(local_idx),
                0.0
            )
        )

        tfidf_rank = (
            tfidf_rank_map.get(
                int(local_idx),
                999
            )
        )

        name_addr_product = (
            name_ratio *
            addr_ratio
        )

        name_partial_addr_partial_product = (
            name_partial *
            addr_partial
        )

        name_number_product = (
            name_ratio *
            number_overlap_value
        )

        addr_number_product = (
            addr_ratio *
            number_overlap_value
        )

        min_name_addr = min(
            name_ratio,
            addr_ratio
        )

        max_name_addr = max(
            name_ratio,
            addr_ratio
        )

        name_addr_gap = abs(
            name_ratio -
            addr_ratio
        )

        partial_gap = abs(
            name_partial -
            addr_partial
        )

        strong_name_weak_address = int(
            name_ratio >= 0.80
            and
            addr_ratio < 0.20
        )

        strong_name_no_number = int(
            name_ratio >= 0.80
            and
            number_overlap_value == 0
        )

        strong_address = int(
            addr_ratio >= 0.70
        )

        strong_name = int(
            name_ratio >= 0.80
        )

        address_supported = int(
            addr_ratio >= 0.50
        )

        multi_channel = int(
            retrieval_channel_count >= 2
        )

        balanced_similarity = (
            name_ratio +
            addr_ratio
        ) / 2.0

        partial_balanced_similarity = (
            name_partial +
            addr_partial
        ) / 2.0

        rows.append({
            "has_rare": has_rare,
            "has_tfidf": has_tfidf,
            "tfidf_score": tfidf_score,
            "tfidf_rank": tfidf_rank,

            "name_ratio": name_ratio,
            "addr_ratio": addr_ratio,
            "name_partial": name_partial,
            "addr_partial": addr_partial,
            "name_token": name_token,
            "addr_token": addr_token,
            "name_length_ratio":
                name_length_ratio,
            "addr_length_ratio":
                addr_length_ratio,
            "number_overlap":
                number_overlap_value,
            "exact_name": exact_name,
            "exact_address":
                exact_address,
            "country_match":
                country_match,
            "combined_similarity":
                combined_similarity,
            "retrieval_channel_count":
                retrieval_channel_count,

            "name_addr_product":
                name_addr_product,
            "name_partial_addr_partial_product":
                name_partial_addr_partial_product,
            "name_number_product":
                name_number_product,
            "addr_number_product":
                addr_number_product,
            "min_name_addr":
                min_name_addr,
            "max_name_addr":
                max_name_addr,
            "name_addr_gap":
                name_addr_gap,
            "partial_gap":
                partial_gap,
            "strong_name_weak_address":
                strong_name_weak_address,
            "strong_name_no_number":
                strong_name_no_number,
            "strong_address":
                strong_address,
            "strong_name":
                strong_name,
            "address_supported":
                address_supported,
            "multi_channel":
                multi_channel,
            "balanced_similarity":
                balanced_similarity,
            "partial_balanced_similarity":
                partial_balanced_similarity
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Process targets
# ------------------------------------------------------------

for start in range(
    0,
    len(target_ids_all),
    CHUNK_TARGETS
):

    end = min(
        start + CHUNK_TARGETS,
        len(target_ids_all)
    )

    part_id = (
        start //
        CHUNK_TARGETS
    )

    part_path = (
        checkpoint_dir_15 /
        f"part_{part_id:03d}.parquet"
    )

    if part_path.exists():

        print(
            f"[{start:,}-{end:,}] "
            "checkpoint exists → skipping"
        )

        part_paths_15.append(
            part_path
        )

        continue

    chunk_rows = []

    t0 = time.time()

    for target_id in target_ids_all[
        start:end
    ]:

        rare_ids = set(
            rare_candidates.get(
                target_id,
                np.empty(
                    0,
                    dtype=np.int32
                )
            )
        )

        tfidf_indices = (
            retrieved_top100.get(
                target_id,
                np.empty(
                    0,
                    dtype=np.int32
                )
            )[:50]
        )

        tfidf_ids = set(
            int(x)
            for x in tfidf_indices
        )

        candidates = (
            rare_ids |
            tfidf_ids
        )

        if not candidates:
            continue

        tfidf_rank_map = {
            int(idx): rank + 1
            for rank, idx in enumerate(
                tfidf_indices
            )
        }

        # Lookup target position
        q_pos = VAL_TARGET_POOL.index[
            VAL_TARGET_POOL["entity_id"]
            == target_id
        ]

        if len(q_pos) != 1:
            raise RuntimeError(
                f"Target lookup failed: {target_id}"
            )

        q_pos = int(
            q_pos[0]
        )

        candidate_array = np.fromiter(
            candidates,
            dtype=np.int32
        )

        scores = (
            X_target_full[
                q_pos:q_pos + 1
            ]
            @
            X_s1_3b[
                candidate_array
            ].T
        ).toarray().ravel()

        tfidf_score_map = {
            int(idx): float(score)
            for idx, score in zip(
                candidate_array,
                scores
            )
        }

        feature_chunk = build_rank_features(
            target_id,
            candidate_array,
            rare_ids,
            tfidf_ids,
            tfidf_rank_map,
            tfidf_score_map
        )

        # ----------------------------------------------------
        # Query-relative features
        # ----------------------------------------------------

        for col in [
            "name_ratio",
            "addr_ratio",
            "name_partial",
            "addr_partial",
            "combined_similarity",
            "tfidf_score"
        ]:

            feature_chunk[
                f"rank_{col}"
            ] = (
                feature_chunk
                .groupby(
                    lambda _: target_id
                )[col]
                .rank(
                    method="min",
                    ascending=False
                )
                .to_numpy()
            )

            feature_chunk[
                f"query_max_{col}"
            ] = feature_chunk[
                col
            ].max()

        feature_chunk["tfidf_margin"] = (
            feature_chunk[
                "query_max_tfidf_score"
            ]
            -
            feature_chunk[
                "tfidf_score"
            ]
        )

        feature_chunk["name_margin"] = (
            feature_chunk[
                "query_max_name_ratio"
            ]
            -
            feature_chunk[
                "name_ratio"
            ]
        )

        feature_chunk["addr_margin"] = (
            feature_chunk[
                "query_max_addr_ratio"
            ]
            -
            feature_chunk[
                "addr_ratio"
            ]
        )

        feature_chunk[
            "combined_margin"
        ] = (
            feature_chunk[
                "query_max_combined_similarity"
            ]
            -
            feature_chunk[
                "combined_similarity"
            ]
        )

        feature_chunk[
            "relative_tfidf_strength"
        ] = (
            feature_chunk[
                "tfidf_score"
            ]
            /
            (
                feature_chunk[
                    "query_max_tfidf_score"
                ]
                +
                1e-8
            )
        )

        feature_chunk[
            "relative_combined_strength"
        ] = (
            feature_chunk[
                "combined_similarity"
            ]
            /
            (
                feature_chunk[
                    "query_max_combined_similarity"
                ]
                +
                1e-8
            )
        )

        feature_chunk[
            "is_best_tfidf"
        ] = (
            feature_chunk[
                "rank_tfidf_score"
            ] == 1
        ).astype(np.int8)

        feature_chunk[
            "is_best_name"
        ] = (
            feature_chunk[
                "rank_name_ratio"
            ] == 1
        ).astype(np.int8)

        feature_chunk[
            "is_best_addr"
        ] = (
            feature_chunk[
                "rank_addr_ratio"
            ] == 1
        ).astype(np.int8)

        feature_chunk[
            "is_best_combined"
        ] = (
            feature_chunk[
                "rank_combined_similarity"
            ] == 1
        ).astype(np.int8)

        # ----------------------------------------------------
        # Ranker score
        # ----------------------------------------------------

        scores_ranker = ranker.predict(
            feature_chunk[
                rank_feature_cols
            ].astype(np.float32)
        )

        # ----------------------------------------------------
        # Target exclusivity: top-1
        # ----------------------------------------------------

        best_idx = int(
            np.argmax(
                scores_ranker
            )
        )

        best_s1 = (
            s1_local_to_id_15[
                int(candidate_array[best_idx])
            ]
        )

        chunk_rows.append({
            "query_id":
                target_id,

            "s1_id":
                best_s1,

            "ranker_score":
                float(
                    scores_ranker[
                        best_idx
                    ]
                )
        })

    chunk_df = pd.DataFrame(
        chunk_rows
    )

    chunk_df.to_parquet(
        part_path,
        index=False
    )

    part_paths_15.append(
        part_path
    )

    elapsed = (
        time.time() - t0
    )

    print(
        f"[{start:,}-{end:,}] "
        f"targets={end-start:,} "
        f"top1_edges={len(chunk_df):,} "
        f"time={elapsed/60:.2f} min"
    )

    del chunk_df
    del chunk_rows

    gc.collect()

# ------------------------------------------------------------
# Combine all target→S1 Top-1 edges
# ------------------------------------------------------------

parts = [
    pd.read_parquet(
        p
    )
    for p in sorted(
        part_paths_15
    )
]

ranked_edges = pd.concat(
    parts,
    ignore_index=True
)

# ------------------------------------------------------------
# Ground truth by S1
# ------------------------------------------------------------

true_by_s1_15 = {
    sid: set()
    for sid in VAL_S1["entity_id"]
}

for target_id, sid in target_to_s1.items():

    if sid in true_by_s1_15:

        true_by_s1_15[
            sid
        ].add(
            target_id
        )

# ------------------------------------------------------------
# Score as a function of ranker-score threshold
# ------------------------------------------------------------

thresholds = np.arange(
    -2.5,
    2.501,
    0.01
)

results15 = []

for threshold in thresholds:

    predictions = (
        ranked_edges[
            ranked_edges[
                "ranker_score"
            ] >= threshold
        ]
        .groupby(
            "s1_id"
        )["query_id"]
        .apply(
            set
        )
        .to_dict()
    )

    f05_values = []

    tp_total = 0
    fp_total = 0
    fn_total = 0

    for sid in VAL_S1[
        "entity_id"
    ]:

        true_targets = (
            true_by_s1_15[
                sid
            ]
        )

        pred_targets = (
            predictions.get(
                sid,
                set()
            )
        )

        tp = len(
            true_targets &
            pred_targets
        )

        fp = len(
            pred_targets -
            true_targets
        )

        fn = len(
            true_targets -
            pred_targets
        )

        if tp + fp > 0:

            precision = (
                tp /
                (tp + fp)
            )

        else:

            precision = (
                1.0
                if not true_targets
                else 0.0
            )

        recall = (
            tp /
            (tp + fn)
            if tp + fn > 0
            else 1.0
        )

        if precision > 0 and recall > 0:

            f05 = (
                1.25 *
                precision *
                recall
                /
                (
                    0.25 * precision
                    +
                    recall
                )
            )

        else:

            f05 = 0.0

        f05_values.append(
            f05
        )

        tp_total += tp
        fp_total += fp
        fn_total += fn

    results15.append({
        "threshold": threshold,

        "macro_f05":
            np.mean(f05_values),

        "precision":
            (
                tp_total /
                (tp_total + fp_total)
                if tp_total + fp_total > 0
                else 0
            ),

        "recall":
            (
                tp_total /
                (tp_total + fn_total)
                if tp_total + fn_total > 0
                else 0
            ),

        "tp": tp_total,
        "fp": fp_total,
        "fn": fn_total
    })

results15_df = pd.DataFrame(
    results15
)

best15 = results15_df.loc[
    results15_df[
        "macro_f05"
    ].idxmax()
]

print(
    "\n" + "=" * 100
)
print(
    "FULL S1 LAMBDARANK RESULT"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best15['threshold']:.2f}"
)

print(
    "Macro F0.5:",
    f"{best15['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best15['precision']:.6f}"
)

print(
    "Recall:",
    f"{best15['recall']:.6f}"
)

print(
    "TP:",
    int(best15["tp"])
)

print(
    "FP:",
    int(best15["fp"])
)

print(
    "FN:",
    int(best15["fn"])
)

print(
    "\nPrevious full-S1 baseline:",
    "0.988468"
)

print(
    "Difference:",
    f"{best15['macro_f05'] - 0.988468:+.6f}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

ranked_edges_path = (
    MODEL_DIR /
    "full_validation_lambdarank_top1.parquet"
)

ranked_edges.to_parquet(
    ranked_edges_path,
    index=False
)

rank_results_path = (
    MODEL_DIR /
    "full_s1_lambdarank_results.csv"
)

results15_df.to_csv(
    rank_results_path,
    index=False
)

print(
    "\nSaved:",
    ranked_edges_path
)

print(
    "Saved:",
    rank_results_path
)

print(
    "\nCELL 15 COMPLETE."
)

CELL 15 — FULL 20K S1 VALIDATION WITH LAMBDARANK
Ranker trees: 89
Ranker features: 56
Feature count verified: 56
[0-2,000] targets=2,000 top1_edges=2,000 time=1.07 min
[2,000-4,000] targets=2,000 top1_edges=2,000 time=0.92 min
[4,000-6,000] targets=2,000 top1_edges=2,000 time=0.94 min
[6,000-8,000] targets=2,000 top1_edges=2,000 time=0.94 min
[8,000-10,000] targets=2,000 top1_edges=2,000 time=0.93 min
[10,000-12,000] targets=2,000 top1_edges=2,000 time=0.93 min
[12,000-14,000] targets=2,000 top1_edges=2,000 time=0.95 min
[14,000-16,000] targets=2,000 top1_edges=2,000 time=0.94 min
[16,000-18,000] targets=2,000 top1_edges=2,000 time=0.95 min
[18,000-20,000] targets=2,000 top1_edges=2,000 time=0.95 min
[20,000-22,000] targets=2,000 top1_edges=2,000 time=0.93 min
[22,000-24,000] targets=2,000 top1_edges=2,000 time=0.96 min
[24,000-26,000] targets=2,000 top1_edges=2,000 time=0.94 min
[26,000-28,000] targets=2,000 top1_edges=2,000 time=0.93 min
[28,000-30,000] targets=2,000 top1_edges=2,000

In [ ]:
# CELL 12B — FINE THRESHOLD SWEEP FOR CURRENT 0.988468 PIPELINE

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 12B — FINE THRESHOLD SWEEP")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Evaluate threshold
# ------------------------------------------------------------

def evaluate_threshold_12b(threshold):

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    accepted = best_all[
        best_all["score"] >= threshold
    ]

    for row in accepted.itertuples(index=False):

        predicted_by_s1[
            row.s1_id
        ].add(
            row.query_id
        )

    f05_values = []

    TP = FP = FN = 0

    for sid in VAL_S1["entity_id"]:

        true_ids = true_by_s1[sid]
        pred_ids = predicted_by_s1[sid]

        tp = len(true_ids & pred_ids)
        fp = len(pred_ids - true_ids)
        fn = len(true_ids - pred_ids)

        if tp + fp > 0:
            precision = tp / (tp + fp)
        else:
            precision = (
                1.0
                if len(true_ids) == 0
                else 0.0
            )

        if tp + fn > 0:
            recall = tp / (tp + fn)
        else:
            recall = 1.0

        if precision > 0 and recall > 0:
            f05 = (
                1.25 * precision * recall
                /
                (0.25 * precision + recall)
            )
        else:
            f05 = 0.0

        f05_values.append(f05)

        TP += tp
        FP += fp
        FN += fn

    return {
        "threshold": threshold,
        "macro_f05": np.mean(f05_values),
        "precision": (
            TP / (TP + FP)
            if TP + FP > 0
            else 0.0
        ),
        "recall": (
            TP / (TP + FN)
            if TP + FN > 0
            else 0.0
        ),
        "TP": TP,
        "FP": FP,
        "FN": FN
    }


# ------------------------------------------------------------
# Fine sweep
# ------------------------------------------------------------

thresholds = np.arange(
    0.600,
    0.681,
    0.0005
)

results12b = []

for threshold in thresholds:

    results12b.append(
        evaluate_threshold_12b(
            float(threshold)
        )
    )

results12b = pd.DataFrame(
    results12b
)

# ------------------------------------------------------------
# Best results
# ------------------------------------------------------------

top = (
    results12b
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
)

print(
    "\n" + "=" * 100
)
print(
    "TOP THRESHOLDS"
)
print(
    "=" * 100
)

print(
    top.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.4f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

best = (
    results12b
    .loc[
        results12b["macro_f05"].idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "BEST THRESHOLD"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best['threshold']:.4f}"
)

print(
    "Macro F0.5:",
    f"{best['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best['precision']:.6f}"
)

print(
    "Recall:",
    f"{best['recall']:.6f}"
)

print(
    "TP:",
    int(best["TP"])
)

print(
    "FP:",
    int(best["FP"])
)

print(
    "FN:",
    int(best["FN"])
)

print(
    "Improvement over 0.988468:",
    f"{best['macro_f05'] - 0.988468:+.6f}"
)

print(
    "\nCELL 12B COMPLETE."
)

CELL 12B — FINE THRESHOLD SWEEP

TOP THRESHOLDS
threshold macro_f05 precision   recall    TP  FP   FN
   0.6185  0.988480  0.994724 0.977683 67686 359 1545
   0.6365  0.988478  0.994869 0.977351 67663 349 1568
   0.6280  0.988477  0.994811 0.977496 67673 353 1558
   0.6370  0.988475  0.994868 0.977337 67662 349 1569
   0.6195  0.988472  0.994724 0.977669 67685 359 1546
   0.6190  0.988472  0.994724 0.977669 67685 359 1546
   0.6265  0.988471  0.994796 0.977510 67674 354 1557
   0.6170  0.988471  0.994695 0.977727 67689 361 1542
   0.6380  0.988470  0.994868 0.977322 67661 349 1570
   0.6375  0.988470  0.994868 0.977322 67661 349 1570
   0.6395  0.988470  0.994883 0.977293 67659 348 1572
   0.6390  0.988470  0.994883 0.977293 67659 348 1572
   0.6385  0.988470  0.994883 0.977293 67659 348 1572
   0.6275  0.988470  0.994796 0.977496 67673 354 1558
   0.6270  0.988470  0.994796 0.977496 67673 354 1558
   0.6180  0.988469  0.994709 0.977683 67686 360 1545
   0.6400  0.988468  0.994883 0.97

In [ ]:
# CELL 13 — FULL-S1 ERROR ANALYSIS AT BEST THRESHOLD

import re
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 13 — FULL-S1 ERROR ANALYSIS")
print("=" * 100)

BEST_THRESHOLD = 0.6185

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "VAL_TARGET_POOL",
    "rare_candidates",
    "retrieved_top100",
    "target_to_s1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# S1 local index
# ------------------------------------------------------------

s1_id_to_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

s1_local_to_id = {
    i: sid
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

# ------------------------------------------------------------
# Prediction map at threshold
# ------------------------------------------------------------

pred = best_all[
    best_all["score"] >= BEST_THRESHOLD
].copy()

pred_by_target = dict(
    zip(
        pred["query_id"],
        zip(
            pred["s1_id"],
            pred["score"]
        )
    )
)

# ------------------------------------------------------------
# True target list
# ------------------------------------------------------------

true_target_ids = set(
    target_to_s1.keys()
)

# ------------------------------------------------------------
# ERROR CATEGORY COUNTS
# ------------------------------------------------------------

retrieval_miss = 0
ranking_miss = 0
threshold_miss = 0
correct_positive = 0

fn_records = []

for target_id in true_target_ids:

    true_s1 = target_to_s1[
        target_id
    ]

    true_local = s1_id_to_local.get(
        true_s1
    )

    # --------------------------------------------
    # Candidate union
    # --------------------------------------------

    rare = set(
        rare_candidates.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )
    )

    tfidf = set(
        retrieved_top100.get(
            target_id,
            np.empty(
                0,
                dtype=np.int32
            )
        )[:50]
    )

    union = (
        rare
        |
        tfidf
    )

    true_in_retrieval = (
        true_local is not None
        and
        true_local in union
    )

    # --------------------------------------------
    # Best predicted candidate
    # --------------------------------------------

    selected = pred_by_target.get(
        target_id
    )

    # --------------------------------------------
    # Correct prediction
    # --------------------------------------------

    if (
        selected is not None
        and
        selected[0] == true_s1
    ):

        correct_positive += 1
        continue

    # --------------------------------------------
    # Retrieval miss
    # --------------------------------------------

    if not true_in_retrieval:

        retrieval_miss += 1

        fn_records.append({
            "query_id":
                target_id,

            "true_s1":
                true_s1,

            "category":
                "retrieval_miss",

            "selected_s1":
                (
                    selected[0]
                    if selected
                    else None
                ),

            "selected_score":
                (
                    selected[1]
                    if selected
                    else np.nan
                ),

            "candidate_count":
                len(union)
        })

        continue

    # --------------------------------------------
    # True candidate was retrieved, but not accepted
    # --------------------------------------------

    if selected is None:

        threshold_miss += 1

        fn_records.append({
            "query_id":
                target_id,

            "true_s1":
                true_s1,

            "category":
                "threshold_miss",

            "selected_s1":
                None,

            "selected_score":
                np.nan,

            "candidate_count":
                len(union)
        })

        continue

    # True candidate existed but Top-1 selected another S1
    ranking_miss += 1

    fn_records.append({
        "query_id":
            target_id,

        "true_s1":
            true_s1,

        "category":
            "ranking_or_exclusivity_miss",

        "selected_s1":
            selected[0],

        "selected_score":
            selected[1],

        "candidate_count":
            len(union)
    })

fn_df = pd.DataFrame(
    fn_records
)

# ------------------------------------------------------------
# FALSE POSITIVES
# ------------------------------------------------------------

fp_rows = []

for row in pred.itertuples(
    index=False
):

    true_s1 = target_to_s1.get(
        row.query_id
    )

    if (
        true_s1 is not None
        and
        row.s1_id != true_s1
    ):

        fp_rows.append({
            "query_id":
                row.query_id,

            "predicted_s1":
                row.s1_id,

            "true_s1":
                true_s1,

            "score":
                row.score
        })

fp_df = pd.DataFrame(
    fp_rows
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FALSE NEGATIVE BREAKDOWN"
)
print(
    "=" * 100
)

print(
    "Correct true-target predictions:",
    correct_positive
)

print(
    "Retrieval misses:",
    retrieval_miss
)

print(
    "Ranking/exclusivity misses:",
    ranking_miss
)

print(
    "Threshold misses:",
    threshold_miss
)

print(
    "Total FN:",
    len(fn_df)
)

if len(fn_df) > 0:

    print(
        "\nFN category distribution:"
    )

    print(
        fn_df[
            "category"
        ]
        .value_counts()
        .to_string()
    )

# ------------------------------------------------------------
# FP statistics
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FALSE POSITIVE SUMMARY"
)
print(
    "=" * 100
)

print(
    "False positives:",
    len(fp_df)
)

if len(fp_df):

    print(
        "FP score mean:",
        f"{fp_df['score'].mean():.6f}"
    )

    print(
        "FP score median:",
        f"{fp_df['score'].median():.6f}"
    )

    print(
        "FP score minimum:",
        f"{fp_df['score'].min():.6f}"
    )

    print(
        "\nHighest-scoring false positives:"
    )

    print(
        fp_df
        .sort_values(
            "score",
            ascending=False
        )
        .head(20)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# FN score distribution
# ------------------------------------------------------------

if len(fn_df):

    ranking_fns = fn_df[
        fn_df["category"]
        == "ranking_or_exclusivity_miss"
    ]

    print(
        "\n" + "=" * 100
    )
    print(
        "RANKING MISS SCORE DISTRIBUTION"
    )
    print(
        "=" * 100
    )

    if len(ranking_fns):

        print(
            ranking_fns[
                "selected_score"
            ]
            .describe(
                percentiles=[
                    .01,
                    .05,
                    .10,
                    .25,
                    .50,
                    .75,
                    .90,
                    .95,
                    .99
                ]
            )
            .to_string()
        )

# ------------------------------------------------------------
# Save error tables
# ------------------------------------------------------------

fn_path = (
    MODEL_DIR
    /
    "full_validation_false_negatives.csv"
)

fp_path = (
    MODEL_DIR
    /
    "full_validation_false_positives.csv"
)

fn_df.to_csv(
    fn_path,
    index=False
)

fp_df.to_csv(
    fp_path,
    index=False
)

print(
    "\nSaved:",
    fn_path
)

print(
    "Saved:",
    fp_path
)

print(
    "\nCELL 13 COMPLETE."
)

CELL 13 — FULL-S1 ERROR ANALYSIS

FALSE NEGATIVE BREAKDOWN
Correct true-target predictions: 67686
Retrieval misses: 15
Ranking/exclusivity misses: 357
Threshold misses: 1173
Total FN: 1545

FN category distribution:
category
threshold_miss                 1173
ranking_or_exclusivity_miss     357
retrieval_miss                   15

FALSE POSITIVE SUMMARY
False positives: 359
FP score mean: 0.883963
FP score median: 0.916081
FP score minimum: 0.623258

Highest-scoring false positives:
    query_id predicted_s1      true_s1    score
S3-797521910 S1-979690967 S1-383911575 0.999644
S2-640825269 S1-491584669 S1-111270002 0.999479
S3-791170196 S1-597550527 S1-330267378 0.999391
S2-692390669 S1-289217668 S1-613731683 0.998502
S3-564841594 S1-550886545 S1-438115387 0.998449
S2-786265004 S1-360579271 S1-321361130 0.998398
S3-727062069 S1-123636921 S1-437766158 0.998378
S3-181916057 S1-387795523 S1-797677711 0.998322
S3-712542961 S1-617334573  S1-88664201 0.998206
S3-241720778 S1-720777938  S1-3

In [ ]:
# CELL 14 — FULL-S1 ERROR CASE ANALYSIS

import os
import re
import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("=" * 100)
print("CELL 14 — FULL-S1 ERROR CASE ANALYSIS")
print("=" * 100)

MODEL_DIR = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

FN_PATH = (
    MODEL_DIR /
    "full_validation_false_negatives.csv"
)

FP_PATH = (
    MODEL_DIR /
    "full_validation_false_positives.csv"
)

if not FN_PATH.exists():
    raise FileNotFoundError(FN_PATH)

if not FP_PATH.exists():
    raise FileNotFoundError(FP_PATH)

# ------------------------------------------------------------
# Load saved errors
# ------------------------------------------------------------

fn = pd.read_csv(
    FN_PATH
)

fp = pd.read_csv(
    FP_PATH
)

print(
    "\nFalse-negative rows:",
    f"{len(fn):,}"
)

print(
    "False-positive rows:",
    f"{len(fp):,}"
)

print(
    "\nFN columns:"
)

print(
    fn.columns.tolist()
)

print(
    "\nFP columns:"
)

print(
    fp.columns.tolist()
)

# ------------------------------------------------------------
# Category counts
# ------------------------------------------------------------

if "category" in fn.columns:

    print(
        "\n" + "=" * 100
    )

    print(
        "FN CATEGORIES"
    )

    print(
        fn["category"]
        .value_counts()
        .to_string()
    )

# ------------------------------------------------------------
# Build lookup dictionaries
# ------------------------------------------------------------

s1_name14 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["name_n"]
    )
)

s1_addr14 = dict(
    zip(
        VAL_S1["entity_id"],
        VAL_S1["addr_n"]
    )
)

target_name14 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["name_n"]
    )
)

target_addr14 = dict(
    zip(
        VAL_TARGET_POOL["entity_id"],
        VAL_TARGET_POOL["addr_n"]
    )
)

# ------------------------------------------------------------
# Feature calculator
# ------------------------------------------------------------

def pair_evidence(
    target_id,
    s1_id
):

    q_name = target_name14.get(
        target_id,
        ""
    )

    q_addr = target_addr14.get(
        target_id,
        ""
    )

    s_name = s1_name14.get(
        s1_id,
        ""
    )

    s_addr = s1_addr14.get(
        s1_id,
        ""
    )

    q_nums = set(
        re.findall(
            r"\d+",
            q_addr
        )
    )

    s_nums = set(
        re.findall(
            r"\d+",
            s_addr
        )
    )

    number_overlap_value = len(
        q_nums & s_nums
    )

    name_ratio = (
        fuzz.ratio(
            q_name,
            s_name
        )
        / 100.0
    )

    addr_ratio = (
        fuzz.ratio(
            q_addr,
            s_addr
        )
        / 100.0
    )

    name_partial = (
        fuzz.partial_ratio(
            q_name,
            s_name
        )
        / 100.0
    )

    addr_partial = (
        fuzz.partial_ratio(
            q_addr,
            s_addr
        )
        / 100.0
    )

    combined = (
        0.45 * name_ratio
        +
        0.55 * addr_ratio
    )

    return {
        "name_ratio":
            name_ratio,

        "addr_ratio":
            addr_ratio,

        "name_partial":
            name_partial,

        "addr_partial":
            addr_partial,

        "number_overlap":
            number_overlap_value,

        "exact_name":
            int(q_name == s_name),

        "exact_address":
            int(q_addr == s_addr),

        "combined_similarity":
            combined
    }


# ------------------------------------------------------------
# Analyze ranking/exclusivity misses
# ------------------------------------------------------------

if "category" in fn.columns:

    rank_fn = fn[
        fn["category"]
        ==
        "ranking_or_exclusivity_miss"
    ].copy()

else:

    rank_fn = fn.iloc[
        0:0
    ].copy()

print(
    "\n" + "=" * 100
)

print(
    "RANKING / EXCLUSIVITY MISS ANALYSIS"
)

print(
    "Rows:",
    f"{len(rank_fn):,}"
)

if len(rank_fn) > 0:

    comparison_rows = []

    for row in rank_fn.itertuples(
        index=False
    ):

        query_id = getattr(
            row,
            "query_id"
        )

        predicted_s1 = getattr(
            row,
            "predicted_s1"
        )

        true_s1 = getattr(
            row,
            "true_s1"
        )

        pred_ev = pair_evidence(
            query_id,
            predicted_s1
        )

        true_ev = pair_evidence(
            query_id,
            true_s1
        )

        comparison_rows.append({
            "query_id":
                query_id,

            "predicted_s1":
                predicted_s1,

            "true_s1":
                true_s1,

            "model_score":
                getattr(
                    row,
                    "score",
                    np.nan
                ),

            "pred_name_ratio":
                pred_ev["name_ratio"],

            "true_name_ratio":
                true_ev["name_ratio"],

            "pred_addr_ratio":
                pred_ev["addr_ratio"],

            "true_addr_ratio":
                true_ev["addr_ratio"],

            "pred_number_overlap":
                pred_ev["number_overlap"],

            "true_number_overlap":
                true_ev["number_overlap"],

            "pred_exact_address":
                pred_ev["exact_address"],

            "true_exact_address":
                true_ev["exact_address"],

            "pred_combined":
                pred_ev["combined_similarity"],

            "true_combined":
                true_ev["combined_similarity"]
        })

    rank_compare = pd.DataFrame(
        comparison_rows
    )

    print(
        "\nMean evidence:"
    )

    print(
        rank_compare[
            [
                "pred_name_ratio",
                "true_name_ratio",
                "pred_addr_ratio",
                "true_addr_ratio",
                "pred_number_overlap",
                "true_number_overlap",
                "pred_combined",
                "true_combined"
            ]
        ]
        .mean()
        .to_string()
    )

    print(
        "\nStrong true-target evidence:"
    )

    checks = {
        "true_exact_address":
            rank_compare[
                "true_exact_address"
            ].sum(),

        "true_addr>=0.70":
            (
                rank_compare[
                    "true_addr_ratio"
                ] >= 0.70
            ).sum(),

        "true_addr>=0.80":
            (
                rank_compare[
                    "true_addr_ratio"
                ] >= 0.80
            ).sum(),

        "true_number>=1":
            (
                rank_compare[
                    "true_number_overlap"
                ] >= 1
            ).sum(),

        "true_combined>=0.70":
            (
                rank_compare[
                    "true_combined"
                ] >= 0.70
            ).sum(),

        "true_name>=0.80":
            (
                rank_compare[
                    "true_name_ratio"
                ] >= 0.80
            ).sum()
    }

    for key, value in checks.items():

        print(
            f"{key:28s}: "
            f"{int(value):,} "
            f"({value/len(rank_compare)*100:.2f}%)"
        )

    print(
        "\nTop ranking misses:"
    )

    print(
        rank_compare
        .sort_values(
            "model_score",
            ascending=False
        )
        .head(25)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# Analyze threshold misses
# ------------------------------------------------------------

if "category" in fn.columns:

    threshold_fn = fn[
        fn["category"]
        ==
        "threshold_miss"
    ].copy()

else:

    threshold_fn = fn.iloc[
        0:0
    ].copy()

print(
    "\n" + "=" * 100
)

print(
    "THRESHOLD MISS ANALYSIS"
)

print(
    "Rows:",
    f"{len(threshold_fn):,}"
)

if len(threshold_fn) > 0:

    print(
        "\nThreshold-miss score distribution:"
    )

    print(
        threshold_fn["score"]
        .describe(
            percentiles=[
                0.01,
                0.05,
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
        .to_string()
    )

    threshold_ev = []

    for row in threshold_fn.itertuples(
        index=False
    ):

        ev = pair_evidence(
            getattr(
                row,
                "query_id"
            ),
            getattr(
                row,
                "true_s1"
            )
        )

        threshold_ev.append(
            ev
        )

    threshold_ev_df = pd.DataFrame(
        threshold_ev
    )

    print(
        "\nThreshold-miss evidence means:"
    )

    print(
        threshold_ev_df.mean()
        .to_string()
    )

    print(
        "\nThreshold misses with strong evidence:"
    )

    for col, condition in [
        (
            "addr>=0.70",
            threshold_ev_df[
                "addr_ratio"
            ] >= 0.70
        ),
        (
            "addr>=0.80",
            threshold_ev_df[
                "addr_ratio"
            ] >= 0.80
        ),
        (
            "number>=1",
            threshold_ev_df[
                "number_overlap"
            ] >= 1
        ),
        (
            "exact_address",
            threshold_ev_df[
                "exact_address"
            ] == 1
        ),
        (
            "combined>=0.70",
            threshold_ev_df[
                "combined_similarity"
            ] >= 0.70
        ),
        (
            "name>=0.80",
            threshold_ev_df[
                "name_ratio"
            ] >= 0.80
        )
    ]:

        count = int(
            condition.sum()
        )

        print(
            f"{col:22s}: "
            f"{count:,} "
            f"({count/len(threshold_ev_df)*100:.2f}%)"
        )

# ------------------------------------------------------------
# False-positive evidence
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)

print(
    "FALSE-POSITIVE EVIDENCE"
)

print(
    "Rows:",
    f"{len(fp):,}"
)

fp_ev = []

for row in fp.itertuples(
    index=False
):

    fp_ev.append(
        pair_evidence(
            getattr(
                row,
                "query_id"
            ),
            getattr(
                row,
                "predicted_s1"
            )
        )
    )

if fp_ev:

    fp_ev_df = pd.DataFrame(
        fp_ev
    )

    print(
        "\nFP evidence means:"
    )

    print(
        fp_ev_df.mean()
        .to_string()
    )

    print(
        "\nFP strong-evidence counts:"
    )

    for col, condition in [
        (
            "addr>=0.70",
            fp_ev_df[
                "addr_ratio"
            ] >= 0.70
        ),
        (
            "addr>=0.80",
            fp_ev_df[
                "addr_ratio"
            ] >= 0.80
        ),
        (
            "number>=1",
            fp_ev_df[
                "number_overlap"
            ] >= 1
        ),
        (
            "exact_address",
            fp_ev_df[
                "exact_address"
            ] == 1
        ),
        (
            "combined>=0.70",
            fp_ev_df[
                "combined_similarity"
            ] >= 0.70
        )
    ]:

        count = int(
            condition.sum()
        )

        print(
            f"{col:22s}: "
            f"{count:,} "
            f"({count/len(fp_ev_df)*100:.2f}%)"
        )

print(
    "\nCELL 14 COMPLETE."
)

CELL 14 — FULL-S1 ERROR CASE ANALYSIS

False-negative rows: 1,545
False-positive rows: 359

FN columns:
['query_id', 'true_s1', 'category', 'selected_s1', 'selected_score', 'candidate_count']

FP columns:
['query_id', 'predicted_s1', 'true_s1', 'score']

FN CATEGORIES
category
threshold_miss                 1173
ranking_or_exclusivity_miss     357
retrieval_miss                   15

RANKING / EXCLUSIVITY MISS ANALYSIS
Rows: 357


AttributeError: 'Pandas' object has no attribute 'predicted_s1'

In [ ]:
# CELL 14B — CORRECTED FULL-S1 ERROR ANALYSIS

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 14B — CORRECTED FULL-S1 ERROR ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# Locate the FN / FP dataframes already present
# ------------------------------------------------------------

fn_df = None
fp_df = None

for name, obj in list(globals().items()):

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue

    cols = set(
        obj.columns
    )

    # FN dataframe signature from Cell 14
    if {
        "query_id",
        "true_s1",
        "category",
        "selected_s1",
        "selected_score",
        "candidate_count"
    }.issubset(cols):

        fn_df = obj

        print(
            "Found FN dataframe:",
            name,
            obj.shape
        )

    # FP dataframe signature from Cell 14
    if {
        "query_id",
        "predicted_s1",
        "true_s1",
        "score"
    }.issubset(cols):

        fp_df = obj

        print(
            "Found FP dataframe:",
            name,
            obj.shape
        )

if fn_df is None:
    raise RuntimeError(
        "Could not find the FN dataframe."
    )

if fp_df is None:
    raise RuntimeError(
        "Could not find the FP dataframe."
    )

# ------------------------------------------------------------
# Basic summary
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "ERROR SUMMARY"
)
print(
    "=" * 100
)

print(
    "FN rows:",
    len(fn_df)
)

print(
    "FP rows:",
    len(fp_df)
)

print(
    "\nFN categories:"
)

print(
    fn_df["category"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# Ranking / exclusivity FN analysis
# ------------------------------------------------------------

rank_fn = fn_df[
    fn_df["category"]
    ==
    "ranking_or_exclusivity_miss"
].copy()

print(
    "\n" + "=" * 100
)
print(
    "RANKING / EXCLUSIVITY MISS ANALYSIS"
)
print(
    "=" * 100
)

print(
    "Rows:",
    len(rank_fn)
)

if len(rank_fn):

    print(
        "\nSelected-score distribution:"
    )

    print(
        rank_fn[
            "selected_score"
        ]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
        .to_string()
    )

    print(
        "\nCandidate-count distribution:"
    )

    print(
        rank_fn[
            "candidate_count"
        ]
        .describe(
            percentiles=[
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
        .to_string()
    )

    # Top cases by score
    print(
        "\nHighest-score ranking/exclusivity misses:"
    )

    print(
        rank_fn[
            [
                "query_id",
                "true_s1",
                "selected_s1",
                "selected_score",
                "candidate_count"
            ]
        ]
        .sort_values(
            "selected_score",
            ascending=False
        )
        .head(30)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# Threshold-miss FN analysis
# ------------------------------------------------------------

threshold_fn = fn_df[
    fn_df["category"]
    ==
    "threshold_miss"
].copy()

print(
    "\n" + "=" * 100
)
print(
    "THRESHOLD MISS ANALYSIS"
)
print(
    "=" * 100
)

print(
    "Rows:",
    len(threshold_fn)
)

if len(threshold_fn):

    print(
        "\nSelected-score distribution:"
    )

    print(
        threshold_fn[
            "selected_score"
        ]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
        .to_string()
    )

    print(
        "\nHighest threshold-miss scores:"
    )

    print(
        threshold_fn[
            [
                "query_id",
                "true_s1",
                "selected_s1",
                "selected_score",
                "candidate_count"
            ]
        ]
        .sort_values(
            "selected_score",
            ascending=False
        )
        .head(30)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# Retrieval misses
# ------------------------------------------------------------

retrieval_fn = fn_df[
    fn_df["category"]
    ==
    "retrieval_miss"
].copy()

print(
    "\n" + "=" * 100
)
print(
    "RETRIEVAL MISS ANALYSIS"
)
print(
    "=" * 100
)

print(
    "Rows:",
    len(retrieval_fn)
)

if len(retrieval_fn):

    print(
        retrieval_fn.to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# False positive analysis
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FALSE-POSITIVE ANALYSIS"
)
print(
    "=" * 100
)

print(
    "Score distribution:"
)

print(
    fp_df[
        "score"
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
    .to_string()
)

print(
    "\nHighest-confidence false positives:"
)

print(
    fp_df[
        [
            "query_id",
            "predicted_s1",
            "true_s1",
            "score"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(40)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Score bands for FN / FP
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "SCORE-BAND ANALYSIS"
)
print(
    "=" * 100
)

bands = [
    (0.00, 0.10),
    (0.10, 0.20),
    (0.20, 0.30),
    (0.30, 0.40),
    (0.40, 0.50),
    (0.50, 0.60),
    (0.60, 0.70),
    (0.70, 0.80),
    (0.80, 0.90),
    (0.90, 1.01)
]

band_rows = []

for lo, hi in bands:

    fn_n = int(
        (
            fn_df["selected_score"]
            >= lo
        ).__and__(
            fn_df["selected_score"] < hi
        ).sum()
    )

    fp_n = int(
        (
            fp_df["score"]
            >= lo
        ).__and__(
            fp_df["score"] < hi
        ).sum()
    )

    band_rows.append({
        "score_range":
            f"{lo:.2f}-{hi:.2f}",

        "FN":
            fn_n,

        "FP":
            fp_n
    })

print(
    pd.DataFrame(
        band_rows
    ).to_string(
        index=False
    )
)

print(
    "\nCELL 14B COMPLETE."
)

CELL 14B — CORRECTED FULL-S1 ERROR ANALYSIS
Found FP dataframe: fp (359, 4)
Found FN dataframe: fn (1545, 6)
Found FN dataframe: ranking_fns (357, 6)
Found FN dataframe: rank_fn (357, 6)

ERROR SUMMARY
FN rows: 357
FP rows: 359

FN categories:
category
ranking_or_exclusivity_miss    357

RANKING / EXCLUSIVITY MISS ANALYSIS
Rows: 357

Selected-score distribution:
count    357.000000
mean       0.884214
std        0.110176
min        0.623258
1%         0.626095
5%         0.662086
10%        0.705211
25%        0.811730
50%        0.916254
75%        0.978675
90%        0.994733
95%        0.997682
99%        0.998473
max        0.999644

Candidate-count distribution:
count     357.000000
mean      321.336134
std       269.756703
min        50.000000
50%       202.000000
75%       482.000000
90%       725.000000
95%       905.200000
99%      1097.000000
max      1306.000000

Highest-score ranking/exclusivity misses:
    query_id      true_s1  selected_s1  selected_score  candidate_count

In [ ]:
# CELL 12 — FINE-GRAINED FULL-S1 THRESHOLD OPTIMIZATION

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def calculate_macro_f05(
    threshold,
    df=best_all
):

    predicted_by_s1 = {
        sid: set()
        for sid in VAL_S1["entity_id"]
    }

    accepted = df[
        df["score"] >= threshold
    ]

    for row in accepted.itertuples(
        index=False
    ):

        predicted_by_s1[
            row.s1_id
        ].add(
            row.query_id
        )

    f05_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for sid in VAL_S1["entity_id"]:

        true_targets = true_by_s1[sid]
        predicted_targets = predicted_by_s1[sid]

        tp = len(
            true_targets & predicted_targets
        )

        fp = len(
            predicted_targets - true_targets
        )

        fn = len(
            true_targets - predicted_targets
        )

        if tp + fp > 0:
            precision = tp / (tp + fp)
        else:
            precision = (
                1.0
                if not true_targets
                else 0.0
            )

        if tp + fn > 0:
            recall = tp / (tp + fn)
        else:
            recall = 1.0

        if precision > 0 and recall > 0:
            f05 = (
                1.25
                * precision
                * recall
                /
                (
                    0.25 * precision
                    + recall
                )
            )
        else:
            f05 = 0.0

        f05_values.append(f05)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    return {
        "threshold": threshold,
        "macro_f05": float(np.mean(f05_values)),
        "precision": (
            total_tp / (total_tp + total_fp)
            if total_tp + total_fp > 0
            else 0.0
        ),
        "recall": (
            total_tp / (total_tp + total_fn)
            if total_tp + total_fn > 0
            else 0.0
        ),
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn
    }


# ------------------------------------------------------------
# Fine sweep
# ------------------------------------------------------------

thresholds = np.arange(
    0.55,
    0.701,
    0.001
)

results12 = []

for threshold in thresholds:

    results12.append(
        calculate_macro_f05(
            float(threshold)
        )
    )

results12_df = pd.DataFrame(
    results12
)

# ------------------------------------------------------------
# Best thresholds
# ------------------------------------------------------------

top12 = (
    results12_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
)

print(
    "\n" + "=" * 100
)
print(
    "TOP FINE-GRAINED THRESHOLDS"
)
print(
    "=" * 100
)

print(
    top12.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

best12 = (
    results12_df
    .loc[
        results12_df["macro_f05"].idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "BEST FINE-GRAINED THRESHOLD"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best12['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best12['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best12['precision']:.6f}"
)

print(
    "Recall:",
    f"{best12['recall']:.6f}"
)

print(
    "TP:",
    int(best12["tp"])
)

print(
    "FP:",
    int(best12["fp"])
)

print(
    "FN:",
    int(best12["fn"])
)

print(
    "\nImprovement over 0.988468:",
    f"{best12['macro_f05'] - 0.988468:+.6f}"
)

print(
    "\nCELL 12 COMPLETE."
)

CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION

TOP FINE-GRAINED THRESHOLDS
threshold macro_f05 precision   recall    tp  fp   fn
    0.628  0.988477  0.994811 0.977496 67673 353 1558
    0.637  0.988475  0.994868 0.977337 67662 349 1569
    0.619  0.988472  0.994724 0.977669 67685 359 1546
    0.617  0.988471  0.994695 0.977727 67689 361 1542
    0.638  0.988470  0.994868 0.977322 67661 349 1570
    0.639  0.988470  0.994883 0.977293 67659 348 1572
    0.627  0.988470  0.994796 0.977496 67673 354 1558
    0.618  0.988469  0.994709 0.977683 67686 360 1545
    0.640  0.988468  0.994883 0.977279 67658 348 1573
    0.621  0.988467  0.994724 0.977655 67684 359 1547
    0.620  0.988467  0.994724 0.977655 67684 359 1547
    0.629  0.988467  0.994825 0.977452 67670 352 1561
    0.642  0.988465  0.994897 0.977250 67656 347 1575
    0.615  0.988465  0.994681 0.977756 67691 362 1540
    0.636  0.988464  0.994854 0.977351 67663 350 1568
    0.616  0.988463  0.994681 0.977741 67690 362 1541
    0.6

In [ ]:
# CREATE matching_results.tsv FOR THE 0.988468 VALIDATION RESULT

import os
import pandas as pd

THRESHOLD = 0.640

OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/"
    "MODELS/matching_results_f05_0.988468.tsv"
)

# best_all came from Cell 11:
# one best S1 candidate per target after target exclusivity Top-1
if "best_all" not in globals():
    raise RuntimeError(
        "best_all is not in memory. "
        "Run Cell 11 first."
    )

if "VAL_S1" not in globals():
    raise RuntimeError(
        "VAL_S1 is not in memory."
    )

# Keep predictions at the threshold that produced 0.988468
accepted = best_all[
    best_all["score"] >= THRESHOLD
].copy()

# Aggregate accepted target IDs by S1
predicted = (
    accepted
    .groupby("s1_id")["query_id"]
    .apply(list)
    .to_dict()
)

# Write exactly one row per validation S1
with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "source1_entity_id\tmatched_entity_ids\n"
    )

    for sid in VAL_S1["entity_id"]:

        matched = predicted.get(
            sid,
            []
        )

        # deterministic ordering
        matched = sorted(
            set(matched)
        )

        if matched:

            f.write(
                sid
                + "\t"
                + ",".join(matched)
                + "\n"
            )

        else:

            f.write(
                sid
                + "\t\n"
            )

print("=" * 100)
print("VALIDATION MATCHING RESULTS CREATED")
print("=" * 100)

print(
    "Threshold:",
    THRESHOLD
)

print(
    "Rows:",
    len(VAL_S1)
)

print(
    "Accepted target→S1 edges:",
    len(accepted)
)

print(
    "Output:",
    OUTPUT_PATH
)

print(
    "Size:",
    f"{os.path.getsize(OUTPUT_PATH)/(1024**2):.2f} MB"
)

# Download
from google.colab import files
files.download(OUTPUT_PATH)

VALIDATION MATCHING RESULTS CREATED
Threshold: 0.64
Rows: 20000
Accepted target→S1 edges: 68006
Output: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/matching_results_f05_0.988468.tsv
Size: 1.08 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# CELL 13 — FINAL TEST INFERENCE @ FULL-S1 VALIDATION OPTIMUM
#
# IMPORTANT:
# The actual TEST F0.5 cannot be measured because TEST GT is hidden.
#
# This generates:
#   matching_results.tsv
#   candidate_pairs.tsv
#
# using:
#   rare-token Top-4 UNION blocked TF-IDF Top-50
#   56-feature reranker
#   target exclusivity Top-1
#   threshold 0.640
#
# This is expected to be a LONG CPU run.

import os
import re
import gc
import time
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import lightgbm as lgb
from rapidfuzz import fuzz
from collections import defaultdict

print("=" * 100)
print("CELL 13 — FINAL TEST INFERENCE")
print("=" * 100)

BASE = "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
MODEL_DIR = os.path.join(BASE, "MODELS")
TEST_DIR = os.path.join(BASE, "DATASET", "test")

OUT_DIR = os.path.join(
    BASE,
    "FINAL_SUBMISSION_0988468"
)

CHECKPOINT_DIR = os.path.join(
    OUT_DIR,
    "target_checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

THRESHOLD = 0.640
TOP_TFIDF = 50
RARE_TOP = 4

CHUNK_SIZE = 10_000

# ============================================================
# 1. LOAD TEST S1
# ============================================================

print("\nLoading TEST S1...")

test_s1 = pd.read_csv(
    os.path.join(
        TEST_DIR,
        "test_source1.tsv"
    ),
    sep="\t",
    dtype=str
)

print(
    "TEST S1:",
    f"{len(test_s1):,}"
)

assert len(test_s1) == 1_732_544

# ============================================================
# 2. EXACT NORMALIZER
# ============================================================

def normalize_exact(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )

test_s1["name_n"] = (
    test_s1["business_name"]
    .map(normalize_exact)
)

test_s1["addr_n"] = (
    test_s1["business_address"]
    .map(normalize_exact)
)

test_s1["country_n"] = (
    test_s1["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

test_s1["combined_text"] = (
    test_s1["name_n"]
    + " "
    + test_s1["addr_n"]
)

# ============================================================
# 3. LOAD S1 TF-IDF MATRIX
# ============================================================

print("\nLoading TEST S1 TF-IDF...")

X_test_s1 = sp.load_npz(
    os.path.join(
        MODEL_DIR,
        "production_test_s1_tfidf.npz"
    )
)

print(
    "TF-IDF shape:",
    X_test_s1.shape
)

assert X_test_s1.shape == (
    1_732_544,
    100_000
)

# ============================================================
# 4. LOAD VECTORIZER
# ============================================================

import joblib

vectorizer = joblib.load(
    os.path.join(
        MODEL_DIR,
        "production_reverse_tfidf_vectorizer.joblib"
    )
)

print(
    "Vocabulary:",
    len(
        vectorizer.vocabulary_
    )
)

# ============================================================
# 5. LOAD FINAL RERANKER
# ============================================================

model = lgb.Booster(
    model_file=os.path.join(
        MODEL_DIR,
        "blocked_tfidf_rare_reranker_lgbm.txt"
    )
)

print(
    "Reranker trees:",
    model.num_trees()
)

print(
    "Reranker features:",
    model.num_feature()
)

assert model.num_feature() == 56

# ============================================================
# 6. BUILD FULL TEST-S1 RARE-TOKEN INDEX
# ============================================================

print(
    "\nBuilding TEST S1 rare-token index..."
)

def word_tokens(text):

    if not text:
        return []

    return [
        token
        for token in re.findall(
            r"[^\W_]+",
            str(text).lower(),
            flags=re.UNICODE
        )
        if len(token) >= 2
    ]


token_df = defaultdict(int)

for row in test_s1[
    [
        "business_name",
        "business_address"
    ]
].itertuples(index=False):

    tokens = set(
        word_tokens(
            row.business_name
        )
        +
        word_tokens(
            row.business_address
        )
    )

    for token in tokens:
        token_df[token] += 1

print(
    "Unique tokens:",
    f"{len(token_df):,}"
)

# Country-aware token index
token_index = defaultdict(list)

for idx, row in enumerate(
    test_s1[
        [
            "business_name",
            "business_address",
            "country_n"
        ]
    ].itertuples(index=False)
):

    tokens = set(
        word_tokens(
            row.business_name
        )
        +
        word_tokens(
            row.business_address
        )
    )

    for token in tokens:

        if token_df[token] <= 1000:

            token_index[
                (
                    row.country_n,
                    token
                )
            ].append(
                idx
            )

print(
    "Token-country blocks:",
    f"{len(token_index):,}"
)

# ============================================================
# 7. FAST S1 LOOKUPS
# ============================================================

s1_ids = test_s1[
    "entity_id"
].to_numpy()

s1_names = test_s1[
    "name_n"
].to_numpy()

s1_addrs = test_s1[
    "addr_n"
].to_numpy()

s1_countries = test_s1[
    "country_n"
].to_numpy()

# ============================================================
# 8. CHECKPOINT MANIFEST
# ============================================================

manifest_path = os.path.join(
    OUT_DIR,
    "manifest.json"
)

completed_parts = set()

if os.path.exists(
    manifest_path
):

    with open(
        manifest_path,
        "r"
    ) as f:

        manifest = json.load(f)

    completed_parts = set(
        manifest.get(
            "completed_parts",
            []
        )
    )

print(
    "\nAlready completed parts:",
    len(completed_parts)
)

# ============================================================
# 9. PROCESS TEST S2 + S3
# ============================================================

source_paths = [
    (
        "S2",
        os.path.join(
            TEST_DIR,
            "test_source2.tsv"
        )
    ),
    (
        "S3",
        os.path.join(
            TEST_DIR,
            "test_source3.tsv"
        )
    )
]

part_number = 0

for source_name, source_path in source_paths:

    print(
        "\n" + "=" * 100
    )

    print(
        "PROCESSING:",
        source_name
    )

    print(
        "=" * 100
    )

    for target_chunk in pd.read_csv(
        source_path,
        sep="\t",
        dtype=str,
        chunksize=CHUNK_SIZE
    ):

        current_part = (
            f"{source_name}_{part_number:05d}"
        )

        checkpoint_file = os.path.join(
            CHECKPOINT_DIR,
            current_part + ".parquet"
        )

        # Move on if checkpoint already exists
        if current_part in completed_parts:

            print(
                current_part,
                "already complete."
            )

            part_number += 1
            continue

        start_time = time.time()

        target_chunk["name_n"] = (
            target_chunk["business_name"]
            .map(normalize_exact)
        )

        target_chunk["addr_n"] = (
            target_chunk["business_address"]
            .map(normalize_exact)
        )

        target_chunk["country_n"] = (
            target_chunk["country"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )

        target_chunk["combined_text"] = (
            target_chunk["name_n"]
            + " "
            + target_chunk["addr_n"]
        )

        # ----------------------------------------------------
        # Transform entire chunk to TF-IDF
        # ----------------------------------------------------

        X_target_chunk = (
            vectorizer.transform(
                target_chunk[
                    "combined_text"
                ]
            )
        )

        best_rows = []

        # ----------------------------------------------------
        # Each target retrieves S1 candidates
        # ----------------------------------------------------

        for q_idx, row in enumerate(
            target_chunk.itertuples(
                index=False
            )
        ):

            country = row.country_n
            name = row.name_n
            addr = row.addr_n

            # -----------------------------------------------
            # Rare-token Top-4
            # -----------------------------------------------

            usable_tokens = []

            tokens = set(
                word_tokens(
                    row.business_name
                )
                +
                word_tokens(
                    row.business_address
                )
            )

            for token in tokens:

                df = token_df.get(
                    token,
                    0
                )

                if (
                    df > 0
                    and
                    df <= 1000
                ):

                    usable_tokens.append(
                        (
                            df,
                            token
                        )
                    )

            usable_tokens.sort(
                key=lambda x: x[0]
            )

            rare_candidates = set()

            for _, token in usable_tokens[
                :RARE_TOP
            ]:

                rare_candidates.update(
                    token_index.get(
                        (
                            country,
                            token
                        ),
                        []
                    )
                )

            # -----------------------------------------------
            # Character block candidates
            #
            # We use the same block-union logic measured
            # during validation.
            # -----------------------------------------------

            char_candidates = set()

            # These blocks are derived directly from S1.
            if name:

                first_char = name[0]
                last_char = name[-1]

                # Build lazily from full S1 the first time
                # the production loop reaches this point.
                #
                # These dictionaries are created outside the
                # main loop below if absent.

            # -----------------------------------------------
            # Candidate union is assembled below after
            # production character dictionaries exist.
            # -----------------------------------------------

            # Placeholder: filled in after lazy indexes.
            # This block will never be reached until indexes
            # have been initialized.

        # ----------------------------------------------------
        # This loop intentionally breaks here because the
        # production char indexes must be built only once,
        # outside every target chunk.
        # ----------------------------------------------------

        del X_target_chunk
        break

    # Stop here; Cell 13A builds the production char indexes
    # before the actual inference loop.
    break

print(
    "\nCELL 13 PREPARATION COMPLETE."
)

print(
    "Run CELL 13A next to build the remaining "
    "production retrieval indexes before inference."
)

CELL 13 — FINAL TEST INFERENCE

Loading TEST S1...
TEST S1: 1,732,544

Loading TEST S1 TF-IDF...
TF-IDF shape: (1732544, 100000)
Vocabulary: 100000
Reranker trees: 424
Reranker features: 56

Building TEST S1 rare-token index...
Unique tokens: 463,036
Token-country blocks: 508,089

Already completed parts: 0

PROCESSING: S2

CELL 13 PREPARATION COMPLETE.
Run CELL 13A next to build the remaining production retrieval indexes before inference.


In [ ]:
# CELL 13A — BUILD REMAINING PRODUCTION S1 RETRIEVAL INDEXES

import re
import gc
import time
import psutil
from collections import defaultdict

print("=" * 100)
print("CELL 13A — BUILD REMAINING PRODUCTION S1 RETRIEVAL INDEXES")
print("=" * 100)

# ------------------------------------------------------------
# Memory helper
# ------------------------------------------------------------

process = psutil.Process()

def ram_gb():
    return (
        process.memory_info().rss
        / (1024 ** 3)
    )

print(
    "\nRAM before:",
    f"{ram_gb():.2f} GB"
)

# ------------------------------------------------------------
# Exact normalization helper
# ------------------------------------------------------------

def normalize_exact_13a(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )

# ------------------------------------------------------------
# Make sure TEST S1 exists
# ------------------------------------------------------------

if "test_s1" not in globals():

    raise RuntimeError(
        "test_s1 is not loaded. "
        "Cell 13 preparation must be rerun first."
    )

# ------------------------------------------------------------
# Verify / create normalized S1 fields
# ------------------------------------------------------------

if "name_n" not in test_s1.columns:

    print(
        "\nNormalizing TEST S1..."
    )

    test_s1["name_n"] = (
        test_s1["business_name"]
        .map(normalize_exact_13a)
    )

    test_s1["addr_n"] = (
        test_s1["business_address"]
        .map(normalize_exact_13a)
    )

if "country_n" not in test_s1.columns:

    test_s1["country_n"] = (
        test_s1["country"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

# ------------------------------------------------------------
# Local row indices
# ------------------------------------------------------------

if "local_idx" not in test_s1.columns:

    test_s1["local_idx"] = np.arange(
        len(test_s1),
        dtype=np.int32
    )

print(
    "\nTEST S1 rows:",
    f"{len(test_s1):,}"
)

# ------------------------------------------------------------
# Initialize indexes
# ------------------------------------------------------------

test_s1_name_first_blocks = defaultdict(list)
test_s1_name_last_blocks = defaultdict(list)

test_s1_addr_first_blocks = defaultdict(list)
test_s1_addr_last_blocks = defaultdict(list)

test_s1_num_country_blocks = defaultdict(list)

# ------------------------------------------------------------
# Build
# ------------------------------------------------------------

print(
    "\nBuilding S1 retrieval indexes..."
)

t0 = time.time()

for row_num, row in enumerate(
    test_s1[
        [
            "local_idx",
            "name_n",
            "addr_n",
            "country_n"
        ]
    ].itertuples(index=False),
    start=1
):

    idx = int(
        row.local_idx
    )

    country = row.country_n
    name = row.name_n
    addr = row.addr_n

    # --------------------------------------------------------
    # Name first / last
    # --------------------------------------------------------

    if name:

        test_s1_name_first_blocks[
            (
                country,
                name[0]
            )
        ].append(idx)

        test_s1_name_last_blocks[
            (
                country,
                name[-1]
            )
        ].append(idx)

    # --------------------------------------------------------
    # Address first / last
    # --------------------------------------------------------

    if addr:

        test_s1_addr_first_blocks[
            (
                country,
                addr[0]
            )
        ].append(idx)

        test_s1_addr_last_blocks[
            (
                country,
                addr[-1]
            )
        ].append(idx)

        # ----------------------------------------------------
        # Number + country
        # ----------------------------------------------------

        for num in set(
            re.findall(
                r"\d+",
                addr
            )
        ):

            test_s1_num_country_blocks[
                (
                    country,
                    num
                )
            ].append(idx)

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if row_num % 250_000 == 0:

        elapsed = (
            time.time() - t0
        )

        rate = (
            row_num / elapsed
            if elapsed > 0
            else 0
        )

        print(
            f"{row_num:,}/"
            f"{len(test_s1):,} "
            f"| {rate:,.0f} rows/s "
            f"| RAM={ram_gb():.2f} GB"
        )

elapsed = (
    time.time() - t0
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "PRODUCTION S1 RETRIEVAL INDEX STATISTICS"
)
print(
    "=" * 100
)

indexes_13a = [
    (
        "name_first",
        test_s1_name_first_blocks
    ),
    (
        "name_last",
        test_s1_name_last_blocks
    ),
    (
        "addr_first",
        test_s1_addr_first_blocks
    ),
    (
        "addr_last",
        test_s1_addr_last_blocks
    ),
    (
        "num_country",
        test_s1_num_country_blocks
    )
]

for name, index in indexes_13a:

    sizes = np.asarray(
        [
            len(v)
            for v in index.values()
        ],
        dtype=np.int32
    )

    print(
        f"\n{name}:"
    )

    print(
        "  blocks:",
        f"{len(index):,}"
    )

    print(
        "  median:",
        f"{np.median(sizes):.0f}"
    )

    print(
        "  p95:",
        f"{np.percentile(sizes,95):.0f}"
    )

    print(
        "  p99:",
        f"{np.percentile(sizes,99):.0f}"
    )

    print(
        "  max:",
        f"{sizes.max():,}"
    )

# ------------------------------------------------------------
# RAM
# ------------------------------------------------------------

gc.collect()

print(
    "\nIndex build time:",
    f"{elapsed/60:.2f} min"
)

print(
    "RAM after:",
    f"{ram_gb():.2f} GB"
)

print(
    "\n✓ Name blocks ready"
)

print(
    "✓ Address blocks ready"
)

print(
    "✓ Number-country blocks ready"
)

print(
    "\nCELL 13A COMPLETE."
)

CELL 13A — BUILD REMAINING PRODUCTION S1 RETRIEVAL INDEXES

RAM before: 3.72 GB

TEST S1 rows: 1,732,544

Building S1 retrieval indexes...
250,000/1,732,544 | 89,320 rows/s | RAM=3.76 GB
500,000/1,732,544 | 88,034 rows/s | RAM=3.78 GB
750,000/1,732,544 | 97,814 rows/s | RAM=3.81 GB
1,000,000/1,732,544 | 104,168 rows/s | RAM=3.83 GB
1,250,000/1,732,544 | 109,534 rows/s | RAM=3.86 GB
1,500,000/1,732,544 | 112,698 rows/s | RAM=3.88 GB

PRODUCTION S1 RETRIEVAL INDEX STATISTICS

name_first:
  blocks: 109
  median: 9879
  p95: 47994
  p99: 69876
  max: 136,193

name_last:
  blocks: 107
  median: 637
  p95: 45756
  p99: 336692
  max: 613,148

addr_first:
  blocks: 105
  median: 6683
  p95: 61520
  p99: 96492
  max: 180,957

addr_last:
  blocks: 113
  median: 784
  p95: 66208
  p99: 219078
  max: 324,662

num_country:
  blocks: 176,968
  median: 1
  p95: 20
  p99: 130
  max: 68,761

Index build time: 0.25 min
RAM after: 3.91 GB

✓ Name blocks ready
✓ Address blocks ready
✓ Number-country block

In [ ]:
# CELL 13B — BUILD PRODUCTION RARE-TOKEN INDEX
#
# Reproduces the validated rare-token retrieval strategy:
#   combined business_name + business_address tokens
#   global token document frequency
#   country-specific posting lists
#   retain tokens with df <= 1000
#
# No production target retrieval yet.

import os
import re
import gc
import time
import numpy as np
import pandas as pd

from collections import Counter, defaultdict

print("=" * 100)
print("CELL 13B — BUILD PRODUCTION RARE-TOKEN INDEX")
print("=" * 100)

BASE_13B = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TEST_S1_PATH_13B = os.path.join(
    BASE_13B,
    "DATASET",
    "test",
    "test_source1.tsv"
)

# ------------------------------------------------------------
# Exact tokenization used by validated rare-token experiment
# ------------------------------------------------------------

def word_tokens_13b(text):

    if pd.isna(text):
        return []

    return [
        token
        for token in re.findall(
            r"[^\W_]+",
            str(text).lower(),
            flags=re.UNICODE
        )
        if len(token) >= 2
    ]

# ------------------------------------------------------------
# Load only required TEST S1 columns
# ------------------------------------------------------------

print("\nLoading TEST S1...")

s1_rare_prod = pd.read_csv(
    TEST_S1_PATH_13B,
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

print(
    "TEST S1 rows:",
    f"{len(s1_rare_prod):,}"
)

# ------------------------------------------------------------
# Normalize country
# ------------------------------------------------------------

s1_rare_prod["country_n"] = (
    s1_rare_prod["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# ------------------------------------------------------------
# PASS 1 — global token document frequency
# ------------------------------------------------------------

print(
    "\nPASS 1 — computing global token frequencies..."
)

token_df_prod = Counter()

t0 = time.time()

for i, row in enumerate(
    s1_rare_prod[
        [
            "business_name",
            "business_address"
        ]
    ].itertuples(index=False),
    start=1
):

    tokens = set(
        word_tokens_13b(
            row.business_name
        )
        +
        word_tokens_13b(
            row.business_address
        )
    )

    token_df_prod.update(
        tokens
    )

    if i % 250_000 == 0:

        print(
            f"{i:,}/{len(s1_rare_prod):,}"
        )

print(
    "PASS 1 time:",
    f"{(time.time()-t0)/60:.2f} min"
)

print(
    "Unique tokens:",
    f"{len(token_df_prod):,}"
)

# ------------------------------------------------------------
# Rare token selection
# ------------------------------------------------------------

MAX_TOKEN_DF = 1000

rare_tokens_prod = {
    token
    for token, df in token_df_prod.items()
    if df <= MAX_TOKEN_DF
}

print(
    "\nRare-token threshold:",
    MAX_TOKEN_DF
)

print(
    "Rare tokens:",
    f"{len(rare_tokens_prod):,}"
)

# ------------------------------------------------------------
# PASS 2 — country-specific posting lists
# ------------------------------------------------------------

print(
    "\nPASS 2 — building country-specific postings..."
)

rare_token_index_prod = defaultdict(list)

t0 = time.time()

for i, row in enumerate(
    s1_rare_prod[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country_n"
        ]
    ].itertuples(index=False),
    start=1
):

    tokens = set(
        word_tokens_13b(
            row.business_name
        )
        +
        word_tokens_13b(
            row.business_address
        )
    )

    for token in tokens:

        if token not in rare_tokens_prod:
            continue

        rare_token_index_prod[
            (
                row.country_n,
                token
            )
        ].append(
            row.entity_id
        )

    if i % 250_000 == 0:

        elapsed = time.time() - t0

        print(
            f"{i:,}/{len(s1_rare_prod):,} "
            f"| {i/elapsed:,.0f} rows/s"
        )

print(
    "PASS 2 time:",
    f"{(time.time()-t0)/60:.2f} min"
)

# ------------------------------------------------------------
# Posting statistics
# ------------------------------------------------------------

posting_sizes_13b = np.asarray(
    [
        len(v)
        for v in rare_token_index_prod.values()
    ],
    dtype=np.int32
)

print(
    "\n" + "=" * 100
)
print(
    "PRODUCTION RARE-TOKEN INDEX STATISTICS"
)
print(
    "=" * 100
)

print(
    "Token-country postings:",
    f"{len(rare_token_index_prod):,}"
)

print(
    "Median posting:",
    f"{np.median(posting_sizes_13b):.0f}"
)

print(
    "P95 posting:",
    f"{np.percentile(posting_sizes_13b,95):.0f}"
)

print(
    "P99 posting:",
    f"{np.percentile(posting_sizes_13b,99):.0f}"
)

print(
    "Maximum posting:",
    f"{posting_sizes_13b.max():,}"
)

# ------------------------------------------------------------
# Sample postings
# ------------------------------------------------------------

print(
    "\nSample postings:"
)

for key in list(
    rare_token_index_prod.keys()
)[:10]:

    print(
        " ",
        repr(key),
        "->",
        len(
            rare_token_index_prod[key]
        ),
        "S1 records"
    )

# ------------------------------------------------------------
# Free temporary large columns
# ------------------------------------------------------------

gc.collect()

print(
    "\nRAM/index build complete."
)

print(
    "\nCELL 13B COMPLETE."
)

CELL 13B — BUILD PRODUCTION RARE-TOKEN INDEX

Loading TEST S1...
TEST S1 rows: 1,732,544

PASS 1 — computing global token frequencies...
250,000/1,732,544
500,000/1,732,544
750,000/1,732,544
1,000,000/1,732,544
1,250,000/1,732,544
1,500,000/1,732,544
PASS 1 time: 0.43 min
Unique tokens: 463,036

Rare-token threshold: 1000
Rare tokens: 461,160

PASS 2 — building country-specific postings...
250,000/1,732,544 | 74,747 rows/s
500,000/1,732,544 | 61,988 rows/s
750,000/1,732,544 | 66,803 rows/s
1,000,000/1,732,544 | 69,378 rows/s
1,250,000/1,732,544 | 69,860 rows/s
1,500,000/1,732,544 | 68,983 rows/s
PASS 2 time: 0.41 min

PRODUCTION RARE-TOKEN INDEX STATISTICS
Token-country postings: 508,089
Median posting: 1
P95 posting: 39
P99 posting: 214
Maximum posting: 999

Sample postings:
  ('us', '2621') -> 88 S1 records
  ('us', 'zephay') -> 20 S1 records
  ('us', 'cotten') -> 12 S1 records
  ('us', 'tyler') -> 866 S1 records
  ('us', '1064') -> 63 S1 records
  ('us', 'iowa') -> 536 S1 records
  

In [ ]:
# CELL 14 — PRODUCTION RARE-TOKEN RETRIEVAL BENCHMARK

import os
import re
import time
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 14 — PRODUCTION RARE-TOKEN RETRIEVAL BENCHMARK")
print("=" * 100)

# ------------------------------------------------------------
# Locate the production token index automatically
# ------------------------------------------------------------

prod_token_index = None
prod_token_df = None

for name, obj in list(globals().items()):

    if not isinstance(obj, dict):
        continue

    if len(obj) < 100_000:
        continue

    try:
        first_key = next(iter(obj))
    except Exception:
        continue

    # Production index keys are:
    # (country, token)
    if (
        isinstance(first_key, tuple)
        and len(first_key) == 2
    ):

        # Values should be lists/sets of S1 local indices
        try:
            first_value = obj[first_key]

            if (
                isinstance(
                    first_value,
                    (list, set, tuple, np.ndarray)
                )
            ):

                prod_token_index = obj

                print(
                    "Detected production token index:",
                    name
                )

                print(
                    "Postings:",
                    f"{len(obj):,}"
                )

                break

        except Exception:
            pass

if prod_token_index is None:

    raise RuntimeError(
        "Could not automatically locate the production "
        "token-country index in notebook memory."
    )

# ------------------------------------------------------------
# Exact production normalizer
# ------------------------------------------------------------

def normalize_exact_14(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )

# ------------------------------------------------------------
# Tokenizer matching validation
# ------------------------------------------------------------

def word_tokens_14(text):

    if pd.isna(text):
        return []

    return [
        token
        for token in re.findall(
            r"[^\W_]+",
            str(text).lower(),
            flags=re.UNICODE
        )
        if len(token) >= 2
    ]

# ------------------------------------------------------------
# Load 10k S2 + 10k S3
# ------------------------------------------------------------

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TEST_DIR = os.path.join(
    BASE,
    "DATASET",
    "test"
)

SAMPLE_N = 10_000

sample_s2_14 = pd.read_csv(
    os.path.join(
        TEST_DIR,
        "test_source2.tsv"
    ),
    sep="\t",
    dtype=str,
    nrows=SAMPLE_N
)

sample_s3_14 = pd.read_csv(
    os.path.join(
        TEST_DIR,
        "test_source3.tsv"
    ),
    sep="\t",
    dtype=str,
    nrows=SAMPLE_N
)

targets14 = pd.concat(
    [
        sample_s2_14.assign(source="S2"),
        sample_s3_14.assign(source="S3")
    ],
    ignore_index=True
)

del sample_s2_14, sample_s3_14

# ------------------------------------------------------------
# Normalize
# ------------------------------------------------------------

targets14["name_n"] = (
    targets14["business_name"]
    .map(normalize_exact_14)
)

targets14["addr_n"] = (
    targets14["business_address"]
    .map(normalize_exact_14)
)

targets14["country_n"] = (
    targets14["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# ------------------------------------------------------------
# Rare-token Top-4
# ------------------------------------------------------------

counts = []

t0 = time.time()

for row in targets14.itertuples(
    index=False
):

    tokens = set(
        word_tokens_14(
            row.business_name
        )
        +
        word_tokens_14(
            row.business_address
        )
    )

    # We already filtered the S1 index to DF <= 1000,
    # so we only need to look up the rarest usable tokens.
    #
    # The posting length provides a proxy for token frequency.
    candidates_ranked = []

    for token in tokens:

        posting = prod_token_index.get(
            (
                row.country_n,
                token
            )
        )

        if not posting:
            continue

        candidates_ranked.append(
            (
                len(posting),
                token
            )
        )

    candidates_ranked.sort(
        key=lambda x: x[0]
    )

    candidates = set()

    for _, token in candidates_ranked[:4]:

        candidates.update(
            prod_token_index.get(
                (
                    row.country_n,
                    token
                ),
                []
            )
        )

    counts.append(
        len(candidates)
    )

elapsed = time.time() - t0

counts = np.asarray(
    counts,
    dtype=np.int32
)

qps = (
    len(targets14)
    /
    elapsed
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "PRODUCTION RARE-TOKEN RESULTS"
)
print(
    "=" * 100
)

print(
    "Queries:",
    f"{len(targets14):,}"
)

print(
    "Runtime:",
    f"{elapsed:.2f} sec"
)

print(
    "Speed:",
    f"{qps:,.1f} targets/sec"
)

print(
    "Projected 9,969,589 targets:",
    f"{9_969_589/qps/3600:.2f} hours"
)

print(
    "\nCandidate statistics:"
)

print(
    "Mean:",
    f"{counts.mean():.2f}"
)

print(
    "Median:",
    f"{np.median(counts):.0f}"
)

print(
    "P90:",
    f"{np.percentile(counts,90):.0f}"
)

print(
    "P95:",
    f"{np.percentile(counts,95):.0f}"
)

print(
    "P99:",
    f"{np.percentile(counts,99):.0f}"
)

print(
    "P99.9:",
    f"{np.percentile(counts,99.9):.0f}"
)

print(
    "Maximum:",
    f"{counts.max():,}"
)

print(
    "Zero candidates:",
    int(
        (counts == 0).sum()
    )
)

print(
    "Candidates <=100:",
    int(
        (counts <= 100).sum()
    )
)

print(
    "Candidates >1000:",
    int(
        (counts > 1000).sum()
    )
)

print(
    "\nCELL 14 COMPLETE."
)

CELL 14 — PRODUCTION RARE-TOKEN RETRIEVAL BENCHMARK
Detected production token index: token_index
Postings: 508,089

PRODUCTION RARE-TOKEN RESULTS
Queries: 20,000
Runtime: 2.19 sec
Speed: 9,115.9 targets/sec
Projected 9,969,589 targets: 0.30 hours

Candidate statistics:
Mean: 496.86
Median: 400
P90: 1080
P95: 1303
P99: 1736
P99.9: 2204
Maximum: 2,562
Zero candidates: 767
Candidates <=100: 3605
Candidates >1000: 2528

CELL 14 COMPLETE.


In [ ]:
# CELL 15 — RARITY-WEIGHTED RARE-TOKEN CANDIDATE PRUNING

import re
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 15 — RARITY-WEIGHTED RARE-TOKEN CANDIDATE PRUNING")
print("=" * 100)

# ------------------------------------------------------------
# Required validation objects
# ------------------------------------------------------------

required = [
    "VAL_S1",
    "VAL_TARGET_POOL",
    "target_to_s1",
    "token_df",
    "token_index"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# S1 local index
# ------------------------------------------------------------

s1_local_15 = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

N_S1 = len(
    VAL_S1
)

# ------------------------------------------------------------
# IDF
# ------------------------------------------------------------

def idf_weight(df):
    return np.log(
        (N_S1 + 1)
        /
        (df + 1)
    ) + 1.0

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

def tokens_15(text):

    if pd.isna(text):
        return []

    return [
        x
        for x in re.findall(
            r"[^\W_]+",
            str(text).lower(),
            flags=re.UNICODE
        )
        if len(x) >= 2
    ]

# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

MAX_DF = 1000

TOP_N = [
    10,
    25,
    50,
    75,
    100,
    150
]

results15 = []

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

for top_n in TOP_N:

    covered = 0
    candidate_counts = []

    for row in VAL_TARGET_POOL.itertuples(
        index=False
    ):

        # ----------------------------------------------------
        # Query tokens
        # ----------------------------------------------------

        name_tokens = set(
            tokens_15(
                row.business_name
            )
        )

        addr_tokens = set(
            tokens_15(
                row.business_address
            )
        )

        query_tokens = (
            name_tokens
            |
            addr_tokens
        )

        usable_tokens = []

        for token in query_tokens:

            df = token_df.get(
                token,
                0
            )

            if (
                df > 0
                and
                df <= MAX_DF
            ):

                usable_tokens.append(
                    token
                )

        # ----------------------------------------------------
        # Candidate scoring
        #
        # Score = sum IDF of shared rare tokens
        # + modest bonus for multiple shared tokens
        # ----------------------------------------------------

        candidate_scores = {}

        for token in usable_tokens:

            posting = token_index.get(
                (
                    row.country_n,
                    token
                ),
                []
            )

            weight = idf_weight(
                token_df[token]
            )

            for local_idx in posting:

                candidate_scores[
                    local_idx
                ] = (
                    candidate_scores.get(
                        local_idx,
                        0.0
                    )
                    +
                    weight
                )

        if not candidate_scores:

            candidate_counts.append(
                0
            )

            continue

        # ----------------------------------------------------
        # Rank candidates
        # ----------------------------------------------------

        ranked = sorted(
            candidate_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        selected = {
            local_idx
            for local_idx, _ in ranked[
                :top_n
            ]
        }

        candidate_counts.append(
            len(selected)
        )

        # ----------------------------------------------------
        # Ground truth
        # ----------------------------------------------------

        true_s1 = target_to_s1.get(
            row.entity_id
        )

        true_local = (
            s1_local_15.get(
                true_s1
            )
            if true_s1 is not None
            else None
        )

        if (
            true_local is not None
            and
            true_local in selected
        ):

            covered += 1

    counts = np.asarray(
        candidate_counts,
        dtype=np.int32
    )

    results15.append({
        "top_n": top_n,

        "recall_pct":
            covered
            /
            len(VAL_TARGET_POOL)
            *
            100,

        "mean_candidates":
            counts.mean(),

        "median":
            np.median(counts),

        "p95":
            np.percentile(
                counts,
                95
            ),

        "p99":
            np.percentile(
                counts,
                99
            ),

        "max":
            counts.max()
    })

results15_df = pd.DataFrame(
    results15
)

print(
    "\n" + "=" * 100
)
print(
    "RARITY-WEIGHTED PRUNING RESULTS"
)
print(
    "=" * 100
)

print(
    results15_df.to_string(
        index=False,
        formatters={
            "recall_pct":
                lambda x: f"{x:.4f}",
            "mean_candidates":
                lambda x: f"{x:.2f}",
            "median":
                lambda x: f"{x:.0f}",
            "p95":
                lambda x: f"{x:.0f}",
            "p99":
                lambda x: f"{x:.0f}",
            "max":
                lambda x: f"{x:.0f}"
        }
    )
)

print(
    "\nCELL 15 COMPLETE."
)

CELL 15 — RARITY-WEIGHTED RARE-TOKEN CANDIDATE PRUNING


RuntimeError: Missing objects: ['VAL_S1', 'VAL_TARGET_POOL', 'target_to_s1']

In [ ]:
# CELL 15B — RESTORE VALIDATION STATE FOR CELL 15

import os
import gc
import pandas as pd

print("=" * 100)
print("CELL 15B — RESTORE VALIDATION STATE")
print("=" * 100)

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

MODEL_DIR = os.path.join(
    BASE,
    "MODELS"
)

S1_PATH = os.path.join(
    TRAIN_DIR,
    "train_source1.tsv"
)

S2_PATH = os.path.join(
    TRAIN_DIR,
    "train_source2.tsv"
)

S3_PATH = os.path.join(
    TRAIN_DIR,
    "train_source3.tsv"
)

GT_PATH = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

# ------------------------------------------------------------
# Exact verified normalizer
# ------------------------------------------------------------

def normalize_exact(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )

# ------------------------------------------------------------
# Restore the exact 20k representative S1 sample
# ------------------------------------------------------------

print("\nLoading representative S1 validation...")

saved_s1_path = os.path.join(
    MODEL_DIR,
    "representative_s1_validation_20k.parquet"
)

if os.path.exists(saved_s1_path):

    VAL_S1 = pd.read_parquet(
        saved_s1_path
    )

    print(
        "Loaded saved validation S1."
    )

else:

    print(
        "Saved validation file not found; "
        "rebuilding from train_source1.tsv..."
    )

    full_s1 = pd.read_csv(
        S1_PATH,
        sep="\t",
        dtype=str
    )

    VAL_S1 = (
        full_s1
        .sample(
            n=20_000,
            random_state=42
        )
        .copy()
    )

    del full_s1
    gc.collect()

# ------------------------------------------------------------
# Ensure normalized columns exist
# ------------------------------------------------------------

if "name_n" not in VAL_S1.columns:

    VAL_S1["name_n"] = (
        VAL_S1["business_name"]
        .map(normalize_exact)
    )

if "addr_n" not in VAL_S1.columns:

    VAL_S1["addr_n"] = (
        VAL_S1["business_address"]
        .map(normalize_exact)
    )

if "country_n" not in VAL_S1.columns:

    VAL_S1["country_n"] = (
        VAL_S1["country"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

print(
    "VAL_S1 rows:",
    f"{len(VAL_S1):,}"
)

# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

print("\nLoading ground truth...")

gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

target_to_s1 = {}

for row in gt.itertuples(
    index=False
):

    value = row.matched_entity_ids

    if pd.isna(value):
        continue

    for target_id in str(
        value
    ).split(","):

        target_id = target_id.strip()

        if target_id:

            # Only retain targets belonging to our
            # representative 20k S1 validation.
            if row.source1_entity_id in set(
                VAL_S1["entity_id"]
            ):

                target_to_s1[
                    target_id
                ] = row.source1_entity_id

validation_target_ids = set(
    target_to_s1.keys()
)

print(
    "Validation target IDs:",
    f"{len(validation_target_ids):,}"
)

# ------------------------------------------------------------
# Recover the complete validation target records
# ------------------------------------------------------------

print(
    "\nRecovering target records from S2/S3..."
)

existing_target_pool = None

if "VAL_TARGET_POOL" in globals():

    existing_ids = set(
        VAL_TARGET_POOL["entity_id"]
    )

    if validation_target_ids.issubset(
        existing_ids
    ):

        existing_target_pool = (
            VAL_TARGET_POOL[
                VAL_TARGET_POOL["entity_id"]
                .isin(validation_target_ids)
            ]
            .copy()
        )

if existing_target_pool is not None:

    VAL_TARGET_POOL = (
        existing_target_pool
    )

    print(
        "Reused existing VAL_TARGET_POOL."
    )

else:

    collected = []

    remaining = set(
        validation_target_ids
    )

    usecols = [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]

    for source_name, path in [
        ("S2", S2_PATH),
        ("S3", S3_PATH)
    ]:

        if not remaining:
            break

        print(
            f"\nScanning {source_name}..."
        )

        for chunk in pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            usecols=usecols,
            chunksize=250_000
        ):

            hit = chunk[
                chunk["entity_id"].isin(
                    remaining
                )
            ]

            if len(hit):

                hit = hit.copy()

                hit["source"] = (
                    source_name
                )

                collected.append(
                    hit
                )

                remaining.difference_update(
                    hit["entity_id"]
                )

                print(
                    "  recovered:",
                    f"{len(hit):,}",
                    "| remaining:",
                    f"{len(remaining):,}"
                )

            del chunk

        gc.collect()

    if remaining:

        raise RuntimeError(
            f"Could not recover "
            f"{len(remaining):,} validation targets."
        )

    VAL_TARGET_POOL = pd.concat(
        collected,
        ignore_index=True
    )

    del collected
    gc.collect()

# ------------------------------------------------------------
# Normalize target fields
# ------------------------------------------------------------

VAL_TARGET_POOL["name_n"] = (
    VAL_TARGET_POOL["business_name"]
    .map(normalize_exact)
)

VAL_TARGET_POOL["addr_n"] = (
    VAL_TARGET_POOL["business_address"]
    .map(normalize_exact)
)

VAL_TARGET_POOL["country_n"] = (
    VAL_TARGET_POOL["country"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# ------------------------------------------------------------
# Validation statistics
# ------------------------------------------------------------

true_counts = (
    VAL_S1["entity_id"]
    .map(
        lambda sid:
        sum(
            1
            for x in target_to_s1.values()
            if x == sid
        )
    )
)

print(
    "\n" + "=" * 100
)
print(
    "RESTORED VALIDATION STATE"
)
print(
    "=" * 100
)

print(
    "VAL_S1:",
    f"{len(VAL_S1):,}"
)

print(
    "VAL_TARGET_POOL:",
    f"{len(VAL_TARGET_POOL):,}"
)

print(
    "Unique target→S1 mappings:",
    f"{len(target_to_s1):,}"
)

print(
    "No-match S1:",
    int(
        (true_counts == 0).sum()
    )
)

print(
    "Matched S1:",
    int(
        (true_counts > 0).sum()
    )
)

print(
    "True pairs:",
    len(target_to_s1)
)

print(
    "\nCELL 15B COMPLETE."
)

print(
    "\nNow rerun CELL 15 unchanged."
)

CELL 15B — RESTORE VALIDATION STATE

Loading representative S1 validation...
Loaded saved validation S1.
VAL_S1 rows: 20,000

Loading ground truth...


KeyboardInterrupt: 

In [ ]:
# CELL 12B — FAST FINE-GRAINED THRESHOLD OPTIMIZATION

import numpy as np
import pandas as pd
import time

print("=" * 100)
print("CELL 12B — FAST FINE-GRAINED THRESHOLD OPTIMIZATION")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# Build S1 local index
# ------------------------------------------------------------

s1_ids = VAL_S1["entity_id"].to_numpy()

s1_to_local = {
    sid: i
    for i, sid in enumerate(s1_ids)
}

n_s1 = len(s1_ids)

# ------------------------------------------------------------
# Convert accepted-edge table to NumPy
# ------------------------------------------------------------

work = best_all[
    [
        "query_id",
        "s1_id",
        "score"
    ]
].copy()

work["s1_local"] = (
    work["s1_id"]
    .map(s1_to_local)
    .astype(np.int32)
)

# True edge = target actually belongs to this S1
work["is_true"] = (
    work["query_id"]
    .map(target_to_s1)
    ==
    work["s1_id"]
).astype(np.int8)

scores = work["score"].to_numpy(
    dtype=np.float32
)

s1_local_arr = work["s1_local"].to_numpy(
    dtype=np.int32
)

is_true_arr = work["is_true"].to_numpy(
    dtype=np.int8
)

# ------------------------------------------------------------
# True pair count for each S1
# ------------------------------------------------------------

true_counts = np.zeros(
    n_s1,
    dtype=np.int32
)

for i, sid in enumerate(
    s1_ids
):

    true_counts[i] = len(
        true_by_s1[sid]
    )

print(
    "S1 entities:",
    n_s1
)

print(
    "Candidate edges:",
    len(work)
)

print(
    "True edges:",
    int(is_true_arr.sum())
)

# ------------------------------------------------------------
# Fast threshold evaluation
# ------------------------------------------------------------

thresholds = np.arange(
    0.550,
    0.701,
    0.001,
    dtype=np.float32
)

results = []

t0 = time.time()

for threshold in thresholds:

    mask = (
        scores >= threshold
    )

    selected_s1 = (
        s1_local_arr[mask]
    )

    selected_true = (
        is_true_arr[mask]
    )

    # Number predicted for each S1
    pred_counts = np.bincount(
        selected_s1,
        minlength=n_s1
    )

    # Number of true pairs recovered for each S1
    tp_counts = np.bincount(
        selected_s1[
            selected_true.astype(bool)
        ],
        minlength=n_s1
    )

    fp_counts = (
        pred_counts
        -
        tp_counts
    )

    fn_counts = (
        true_counts
        -
        tp_counts
    )

    # --------------------------------------------------------
    # Per-S1 precision / recall
    # --------------------------------------------------------

    precision = np.empty(
        n_s1,
        dtype=np.float64
    )

    recall = np.empty(
        n_s1,
        dtype=np.float64
    )

    # No prediction
    no_prediction = (
        pred_counts == 0
    )

    # Predicted entities
    predicted = (
        ~no_prediction
    )

    precision[
        predicted
    ] = (
        tp_counts[predicted]
        /
        pred_counts[predicted]
    )

    # Empty prediction:
    # perfect only when true set is also empty
    precision[
        no_prediction
    ] = np.where(
        true_counts[no_prediction] == 0,
        1.0,
        0.0
    )

    # Matched S1
    has_true = (
        true_counts > 0
    )

    recall[
        has_true
    ] = (
        tp_counts[has_true]
        /
        true_counts[has_true]
    )

    # No-match S1
    recall[
        ~has_true
    ] = 1.0

    # --------------------------------------------------------
    # Macro F0.5
    # --------------------------------------------------------

    valid_pr = (
        (precision > 0)
        &
        (recall > 0)
    )

    f05 = np.zeros(
        n_s1,
        dtype=np.float64
    )

    f05[valid_pr] = (
        1.25
        *
        precision[valid_pr]
        *
        recall[valid_pr]
        /
        (
            0.25 * precision[valid_pr]
            +
            recall[valid_pr]
        )
    )

    results.append({
        "threshold":
            float(threshold),

        "macro_f05":
            float(f05.mean()),

        "precision":
            float(
                tp_counts.sum()
                /
                pred_counts.sum()
                if pred_counts.sum() > 0
                else 0.0
            ),

        "recall":
            float(
                tp_counts.sum()
                /
                true_counts.sum()
            ),

        "tp":
            int(tp_counts.sum()),

        "fp":
            int(fp_counts.sum()),

        "fn":
            int(fn_counts.sum())
    })

elapsed = (
    time.time() - t0
)

results12b = pd.DataFrame(
    results
)

# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

best = results12b.loc[
    results12b["macro_f05"].idxmax()
]

print(
    "\n" + "=" * 100
)
print(
    "TOP 20 THRESHOLDS"
)
print(
    "=" * 100
)

print(
    results12b
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
    .to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

print(
    "\n" + "=" * 100
)
print(
    "BEST FINE-GRAINED THRESHOLD"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best['precision']:.6f}"
)

print(
    "Recall:",
    f"{best['recall']:.6f}"
)

print(
    "TP:",
    int(best["tp"])
)

print(
    "FP:",
    int(best["fp"])
)

print(
    "FN:",
    int(best["fn"])
)

print(
    "\nRuntime:",
    f"{elapsed:.2f} sec"
)

print(
    "Improvement vs 0.988468:",
    f"{best['macro_f05'] - 0.988468:+.6f}"
)

print(
    "\nCELL 12B COMPLETE."
)

CELL 12B — FAST FINE-GRAINED THRESHOLD OPTIMIZATION


RuntimeError: Missing objects: ['best_all', 'true_by_s1']

In [ ]:
# CELL 12B — RESTORE FULL VALIDATION + FINE THRESHOLD SEARCH

import os
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 12B — RESTORE FULL VALIDATION + FINE THRESHOLD SEARCH")
print("=" * 100)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

MODEL_DIR = os.path.join(
    BASE,
    "MODELS"
)

BEST_ALL_PATH = os.path.join(
    MODEL_DIR,
    "full_validation_target_exclusive_top1.parquet"
)

S1_PATH = os.path.join(
    BASE,
    "DATASET",
    "train",
    "train_source1.tsv"
)

GT_PATH = os.path.join(
    BASE,
    "DATASET",
    "train",
    "train_ground_truth.tsv"
)

# ------------------------------------------------------------
# Restore target->S1 scored edges
# ------------------------------------------------------------

if not os.path.exists(BEST_ALL_PATH):
    raise FileNotFoundError(
        f"Validation checkpoint not found:\n{BEST_ALL_PATH}"
    )

print("\nLoading saved validation checkpoint...")

best_all = pd.read_parquet(
    BEST_ALL_PATH
)

print(
    "Rows:",
    f"{len(best_all):,}"
)

print(
    "Columns:",
    list(best_all.columns)
)

required_best = {
    "query_id",
    "s1_id",
    "score"
}

missing_best = (
    required_best
    -
    set(best_all.columns)
)

if missing_best:
    raise RuntimeError(
        f"Checkpoint missing columns: {missing_best}"
    )

# ------------------------------------------------------------
# Rebuild EXACT representative 20k S1 sample
# ------------------------------------------------------------

print("\nRebuilding representative 20k S1 set...")

full_s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    dtype=str
)

VAL_S1_12 = (
    full_s1
    .sample(
        n=20_000,
        random_state=42
    )
    .copy()
)

del full_s1

print(
    "Validation S1:",
    f"{len(VAL_S1_12):,}"
)

# ------------------------------------------------------------
# Rebuild S1 -> true target map
# ------------------------------------------------------------

print("\nLoading ground truth...")

gt12 = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

gt_lookup_12 = (
    gt12
    .set_index(
        "source1_entity_id"
    )["matched_entity_ids"]
    .fillna("")
)

true_by_s1 = {}

for sid in VAL_S1_12["entity_id"]:

    value = gt_lookup_12.get(
        sid,
        ""
    )

    if not value:

        true_by_s1[sid] = set()

    else:

        true_by_s1[sid] = set(
            x.strip()
            for x in str(value).split(",")
            if x.strip()
        )

del gt12
del gt_lookup_12

print(
    "No-match S1:",
    sum(
        len(v) == 0
        for v in true_by_s1.values()
    )
)

print(
    "Matched S1:",
    sum(
        len(v) > 0
        for v in true_by_s1.values()
    )
)

print(
    "True pairs:",
    sum(
        len(v)
        for v in true_by_s1.values()
    )
)

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def evaluate_threshold_12(
    threshold
):

    accepted = best_all[
        best_all["score"] >= threshold
    ]

    # best_all already contains exactly one S1 candidate
    # per target because Cell 11 applied target exclusivity.

    predicted_by_s1 = {}

    for row in accepted.itertuples(
        index=False
    ):

        predicted_by_s1.setdefault(
            row.s1_id,
            set()
        ).add(
            row.query_id
        )

    f05_values = []

    tp_total = 0
    fp_total = 0
    fn_total = 0

    for sid, true_targets in true_by_s1.items():

        predicted_targets = (
            predicted_by_s1.get(
                sid,
                set()
            )
        )

        tp = len(
            true_targets
            &
            predicted_targets
        )

        fp = len(
            predicted_targets
            -
            true_targets
        )

        fn = len(
            true_targets
            -
            predicted_targets
        )

        # Empty prediction on a true no-match entity = 1.0
        if (
            tp + fp
            == 0
        ):

            precision = (
                1.0
                if len(true_targets) == 0
                else 0.0
            )

        else:

            precision = (
                tp
                /
                (tp + fp)
            )

        if (
            tp + fn
            == 0
        ):

            recall = 1.0

        else:

            recall = (
                tp
                /
                (tp + fn)
            )

        if (
            precision > 0
            and
            recall > 0
        ):

            f05 = (
                1.25
                * precision
                * recall
                /
                (
                    0.25 * precision
                    +
                    recall
                )
            )

        else:

            f05 = 0.0

        f05_values.append(
            f05
        )

        tp_total += tp
        fp_total += fp
        fn_total += fn

    return {
        "threshold": threshold,

        "macro_f05": float(
            np.mean(
                f05_values
            )
        ),

        "precision": (
            tp_total
            /
            (tp_total + fp_total)
            if tp_total + fp_total > 0
            else 0.0
        ),

        "recall": (
            tp_total
            /
            (tp_total + fn_total)
            if tp_total + fn_total > 0
            else 0.0
        ),

        "tp": tp_total,
        "fp": fp_total,
        "fn": fn_total
    }


# ------------------------------------------------------------
# Fine threshold grid
# ------------------------------------------------------------

print(
    "\nRunning fine threshold sweep..."
)

thresholds = np.arange(
    0.550,
    0.701,
    0.001
)

results12 = []

for i, threshold in enumerate(
    thresholds,
    start=1
):

    results12.append(
        evaluate_threshold_12(
            float(threshold)
        )
    )

    if i % 25 == 0:
        print(
            f"{i}/{len(thresholds)} thresholds"
        )

results12_df = pd.DataFrame(
    results12
)

# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

best12 = (
    results12_df
    .loc[
        results12_df["macro_f05"].idxmax()
    ]
)

print(
    "\n" + "=" * 100
)
print(
    "BEST FINE-GRAINED THRESHOLD"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best12['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best12['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best12['precision']:.6f}"
)

print(
    "Recall:",
    f"{best12['recall']:.6f}"
)

print(
    "TP:",
    int(best12["tp"])
)

print(
    "FP:",
    int(best12["fp"])
)

print(
    "FN:",
    int(best12["fn"])
)

print(
    "\nTop 20:"
)

print(
    results12_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Save threshold search
# ------------------------------------------------------------

threshold_results_path = os.path.join(
    MODEL_DIR,
    "full_s1_fine_threshold_results.csv"
)

results12_df.to_csv(
    threshold_results_path,
    index=False
)

best_threshold_path = os.path.join(
    MODEL_DIR,
    "blocked_tfidf_rare_reranker_full_s1_best_threshold.txt"
)

with open(
    best_threshold_path,
    "w"
) as f:

    f.write(
        str(
            best12["threshold"]
        )
    )

print(
    "\nSaved:",
    threshold_results_path
)

print(
    "Saved best threshold:",
    best_threshold_path
)

print(
    "\nImprovement over 0.988468:",
    f"{best12['macro_f05'] - 0.988468:+.6f}"
)

print(
    "\nCELL 12B COMPLETE."
)

CELL 12B — RESTORE FULL VALIDATION + FINE THRESHOLD SEARCH

Loading saved validation checkpoint...
Rows: 69,231
Columns: ['query_id', 's1_id', 'score']

Rebuilding representative 20k S1 set...
Validation S1: 20,000

Loading ground truth...
No-match S1: 1131
Matched S1: 18869
True pairs: 69231

Running fine threshold sweep...
25/151 thresholds
50/151 thresholds
75/151 thresholds
100/151 thresholds
125/151 thresholds
150/151 thresholds

BEST FINE-GRAINED THRESHOLD
Threshold: 0.628
Macro F0.5: 0.988477
Precision: 0.994811
Recall: 0.977496
TP: 67673
FP: 353
FN: 1558

Top 20:
 threshold  macro_f05  precision   recall    tp  fp   fn
     0.628   0.988477   0.994811 0.977496 67673 353 1558
     0.637   0.988475   0.994868 0.977337 67662 349 1569
     0.619   0.988472   0.994724 0.977669 67685 359 1546
     0.617   0.988471   0.994695 0.977727 67689 361 1542
     0.638   0.988470   0.994868 0.977322 67661 349 1570
     0.639   0.988470   0.994883 0.977293 67659 348 1572
     0.627   0.988470  

In [ ]:
# CELL 13 — ANALYZE REMAINING FP/FN AT BEST THRESHOLD

import os
import re
import numpy as np
import pandas as pd

from rapidfuzz import fuzz

print("=" * 100)
print("CELL 13 — REMAINING ERROR ANALYSIS")
print("=" * 100)

BEST_THRESHOLD = 0.628

# ------------------------------------------------------------
# Load validation prediction checkpoint
# ------------------------------------------------------------

checkpoint_path = (
    MODEL_DIR
    /
    "full_validation_target_exclusive_top1.parquet"
)

if not checkpoint_path.exists():

    raise FileNotFoundError(
        checkpoint_path
    )

pred = pd.read_parquet(
    checkpoint_path
)

print(
    "Prediction rows:",
    f"{len(pred):,}"
)

# ------------------------------------------------------------
# Rebuild validation S1 + GT if necessary
# ------------------------------------------------------------

if "VAL_S1" not in globals():

    print(
        "\nLoading representative S1..."
    )

    train_s1_path = (
        BASE
        /
        "DATASET"
        /
        "train"
        /
        "train_source1.tsv"
    )

    full_s1_13 = pd.read_csv(
        train_s1_path,
        sep="\t",
        dtype=str
    )

    VAL_S1 = (
        full_s1_13
        .sample(
            n=20_000,
            random_state=42
        )
        .copy()
    )

    del full_s1_13

# ------------------------------------------------------------
# Load GT
# ------------------------------------------------------------

gt_path_13 = (
    BASE
    /
    "DATASET"
    /
    "train"
    /
    "train_ground_truth.tsv"
)

gt13 = pd.read_csv(
    gt_path_13,
    sep="\t",
    dtype=str
)

true_by_s1_13 = {}

target_to_true_13 = {}

for row in gt13.itertuples(
    index=False
):

    value = row.matched_entity_ids

    if pd.isna(value):

        targets_here = set()

    else:

        targets_here = set(
            x.strip()
            for x in str(value).split(",")
            if x.strip()
        )

    true_by_s1_13[
        row.source1_entity_id
    ] = targets_here

    for target_id in targets_here:

        target_to_true_13[
            target_id
        ] = row.source1_entity_id

del gt13

# ------------------------------------------------------------
# Accepted predictions
# ------------------------------------------------------------

accepted = pred[
    pred["score"] >= BEST_THRESHOLD
].copy()

print(
    "\nAccepted predictions:",
    f"{len(accepted):,}"
)

# ------------------------------------------------------------
# Ground-truth labeling
# ------------------------------------------------------------

accepted["true_s1"] = (
    accepted["query_id"]
    .map(target_to_true_13)
)

accepted["is_correct"] = (
    accepted["s1_id"]
    ==
    accepted["true_s1"]
)

fp = accepted[
    ~accepted["is_correct"]
].copy()

tp = accepted[
    accepted["is_correct"]
].copy()

print(
    "TP predictions:",
    f"{len(tp):,}"
)

print(
    "FP predictions:",
    f"{len(fp):,}"
)

# ------------------------------------------------------------
# False-negative true target assignments
# ------------------------------------------------------------

predicted_target_to_s1 = dict(
    zip(
        pred["query_id"],
        pred["s1_id"]
    )
)

fn_rows = []

for target_id, true_s1 in target_to_true_13.items():

    predicted_s1 = (
        predicted_target_to_s1.get(
            target_id
        )
    )

    if predicted_s1 != true_s1:

        # Score of the selected wrong candidate
        selected = pred[
            pred["query_id"]
            ==
            target_id
        ]

        if len(selected):

            score = float(
                selected.iloc[0]["score"]
            )

        else:

            score = np.nan

        fn_rows.append({
            "query_id":
                target_id,

            "true_s1":
                true_s1,

            "predicted_s1":
                predicted_s1,

            "selected_score":
                score
        })

fn = pd.DataFrame(
    fn_rows
)

print(
    "\nTrue target assignments missed:",
    f"{len(fn):,}"
)

# ------------------------------------------------------------
# Score distributions
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "SCORE DISTRIBUTIONS"
)
print(
    "=" * 100
)

for name, data in [
    ("TRUE POSITIVES", tp["score"]),
    ("FALSE POSITIVES", fp["score"]),
    ("MISSED TRUE TARGETS", fn["selected_score"])
]:

    values = (
        data
        .dropna()
        .to_numpy()
    )

    print(
        "\n" + name
    )

    if len(values):

        for q in [
            0,
            1,
            5,
            10,
            25,
            50,
            75,
            90,
            95,
            99,
            100
        ]:

            print(
                f"P{q:3d}:",
                f"{np.percentile(values,q):.6f}"
            )

    else:

        print(
            "No values."
        )

# ------------------------------------------------------------
# False-positive score bands
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FALSE-POSITIVE SCORE BANDS"
)
print(
    "=" * 100
)

for lo, hi in [
    (0.628, 0.65),
    (0.65, 0.70),
    (0.70, 0.75),
    (0.75, 0.80),
    (0.80, 0.85),
    (0.85, 0.90),
    (0.90, 0.95),
    (0.95, 1.01)
]:

    count = int(
        (
            (fp["score"] >= lo)
            &
            (fp["score"] < hi)
        ).sum()
    )

    print(
        f"{lo:.3f}-{hi:.3f}:",
        count
    )

# ------------------------------------------------------------
# Inspect the highest-scoring false positives
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "TOP FALSE POSITIVES"
)
print(
    "=" * 100
)

# Build raw-record lookups
s1_raw_13 = (
    VAL_S1
    .set_index("entity_id")
)

# Recover target raw records from validation pool
if "VAL_TARGET_POOL" in globals():

    target_raw_13 = (
        VAL_TARGET_POOL
        .set_index("entity_id")
    )

else:

    target_raw_13 = None

top_fp = (
    fp
    .sort_values(
        "score",
        ascending=False
    )
    .head(30)
)

display_cols = [
    "query_id",
    "s1_id",
    "true_s1",
    "score"
]

print(
    top_fp[
        display_cols
    ].to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Inspect FP evidence
# ------------------------------------------------------------

fp_evidence = []

for row in top_fp.itertuples(
    index=False
):

    if (
        target_raw_13 is None
        or row.query_id not in target_raw_13.index
        or row.s1_id not in s1_raw_13.index
    ):

        continue

    q = target_raw_13.loc[
        row.query_id
    ]

    s = s1_raw_13.loc[
        row.s1_id
    ]

    q_name = str(
        q.get(
            "name_n",
            ""
        )
    )

    q_addr = str(
        q.get(
            "addr_n",
            ""
        )
    )

    s_name = str(
        s.get(
            "name_n",
            ""
        )
    )

    s_addr = str(
        s.get(
            "addr_n",
            ""
        )
    )

    q_nums = set(
        re.findall(
            r"\d+",
            q_addr
        )
    )

    s_nums = set(
        re.findall(
            r"\d+",
            s_addr
        )
    )

    fp_evidence.append({

        "query_id":
            row.query_id,

        "pred_s1":
            row.s1_id,

        "score":
            row.score,

        "name_ratio":
            fuzz.ratio(
                q_name,
                s_name
            ) / 100,

        "addr_ratio":
            fuzz.ratio(
                q_addr,
                s_addr
            ) / 100,

        "name_partial":
            fuzz.partial_ratio(
                q_name,
                s_name
            ) / 100,

        "addr_partial":
            fuzz.partial_ratio(
                q_addr,
                s_addr
            ) / 100,

        "number_overlap":
            len(
                q_nums
                &
                s_nums
            ),

        "name":
            q.get(
                "business_name",
                ""
            ),

        "pred_name":
            s.get(
                "business_name",
                ""
            ),

        "address":
            q.get(
                "business_address",
                ""
            ),

        "pred_address":
            s.get(
                "business_address",
                ""
            )
    })

fp_evidence_df = pd.DataFrame(
    fp_evidence
)

print(
    "\nDetailed FP evidence:"
)

if len(fp_evidence_df):

    print(
        fp_evidence_df.to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# Save error-analysis artifacts
# ------------------------------------------------------------

fp_path = (
    MODEL_DIR
    /
    "full_validation_false_positives.csv"
)

fn_path = (
    MODEL_DIR
    /
    "full_validation_false_negatives.csv"
)

fp.to_csv(
    fp_path,
    index=False
)

fn.to_csv(
    fn_path,
    index=False
)

print(
    "\nSaved FP analysis:",
    fp_path
)

print(
    "Saved FN analysis:",
    fn_path
)

print(
    "\nCELL 13 COMPLETE."
)

CELL 13 — REMAINING ERROR ANALYSIS
Prediction rows: 69,231


TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
# CELL 13B — FIX MODEL_DIR PATH TYPE

from pathlib import Path

print("=" * 100)
print("CELL 13B — FIX MODEL_DIR PATH TYPE")
print("=" * 100)

# Convert the existing string path into a pathlib Path
MODEL_DIR = Path(
    MODEL_DIR
)

checkpoint_path = (
    MODEL_DIR
    /
    "full_validation_target_exclusive_top1.parquet"
)

results_path = (
    MODEL_DIR
    /
    "full_s1_validation_results.csv"
)

print(
    "MODEL_DIR:",
    MODEL_DIR
)

print(
    "Checkpoint exists:",
    checkpoint_path.exists()
)

print(
    "Results file exists:",
    results_path.exists()
)

print(
    "\nCheckpoint:",
    checkpoint_path
)

print(
    "\nCELL 13B COMPLETE."
)

print(
    "\nNow rerun CELL 13 unchanged."
)

CELL 13B — FIX MODEL_DIR PATH TYPE
MODEL_DIR: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS
Checkpoint exists: True
Results file exists: True

Checkpoint: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_target_exclusive_top1.parquet

CELL 13B COMPLETE.

Now rerun CELL 13 unchanged.


In [ ]:
# CELL 13B — FIX PATH HANDLING FOR REMAINING ERROR ANALYSIS

import os
import pandas as pd
import numpy as np

print("=" * 100)
print("CELL 13B — FIX PATH HANDLING")
print("=" * 100)

# ------------------------------------------------------------
# Use a plain string path consistently
# ------------------------------------------------------------

BASE = (
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

TRAIN_DIR = os.path.join(
    BASE,
    "DATASET",
    "train"
)

GT_PATH = os.path.join(
    TRAIN_DIR,
    "train_ground_truth.tsv"
)

S1_PATH = os.path.join(
    TRAIN_DIR,
    "train_source1.tsv"
)

print(
    "BASE:",
    BASE
)

print(
    "GT_PATH:",
    GT_PATH
)

print(
    "S1_PATH:",
    S1_PATH
)

# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

for path in [
    GT_PATH,
    S1_PATH
]:

    print(
        os.path.basename(path),
        "->",
        os.path.exists(path)
    )

    if not os.path.exists(path):
        raise FileNotFoundError(
            path
        )

# ------------------------------------------------------------
# Check prediction object from Cell 13
# ------------------------------------------------------------

prediction_objects = [
    "best_all",
    "prediction_df",
    "predictions",
    "best_target"
]

found = []

for name in prediction_objects:

    if name in globals():

        obj = globals()[name]

        if isinstance(
            obj,
            pd.DataFrame
        ):

            found.append(
                (
                    name,
                    obj
                )
            )

print(
    "\nPrediction dataframes found:",
    [
        (name, len(df))
        for name, df in found
    ]
)

# ------------------------------------------------------------
# Reload GT for remaining error analysis
# ------------------------------------------------------------

gt13 = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str
)

print(
    "\nGT rows:",
    f"{len(gt13):,}"
)

# ------------------------------------------------------------
# Representative S1
# ------------------------------------------------------------

s1_path_exists = os.path.exists(
    S1_PATH
)

if s1_path_exists:

    s1_13 = pd.read_csv(
        S1_PATH,
        sep="\t",
        dtype=str
    )

    s1_13 = (
        s1_13
        .sample(
            n=20_000,
            random_state=42
        )
        .copy()
    )

    print(
        "Representative S1:",
        f"{len(s1_13):,}"
    )

# ------------------------------------------------------------
# Finish
# ------------------------------------------------------------

print(
    "\nPATH FIX COMPLETE."
)

print(
    "You can now rerun CELL 13 unchanged."
)

print(
    "\nCELL 13B COMPLETE."
)

CELL 13B — FIX PATH HANDLING
BASE: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026
GT_PATH: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/train/train_ground_truth.tsv
S1_PATH: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/DATASET/train/train_source1.tsv
train_ground_truth.tsv -> True
train_source1.tsv -> True

Prediction dataframes found: [('best_all', 69231)]

GT rows: 2,206,821
Representative S1: 20,000

PATH FIX COMPLETE.
You can now rerun CELL 13 unchanged.

CELL 13B COMPLETE.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import scipy.sparse as sp
from collections import defaultdict

# Define the base directory for saving artifacts
BASE_DIR = "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
MODEL_DIR = os.path.join(BASE_DIR, "MODELS")
FINAL_ARTIFACTS_DIR = os.path.join(MODEL_DIR, "FINAL_SAVED_ARTIFACTS")

os.makedirs(FINAL_ARTIFACTS_DIR, exist_ok=True)
print(f"Created directory: {FINAL_ARTIFACTS_DIR}")

# --- 1. Save the final model and its best threshold ---
# Look up the model under whichever active variable name it has in memory
if 'model9' in globals():
    active_model = model9
elif 'ranking_model' in globals():
    active_model = ranking_model
elif 'model' in globals():
    active_model = model
else:
    raise NameError("No verified LightGBM model found in globals ('model9', 'ranking_model', or 'model').")

if hasattr(active_model, 'booster_'):
    final_model = active_model.booster_
else:
    final_model = active_model

if 'best12' in globals():
    final_threshold = float(best12['threshold'])
elif 'best15' in globals():
    final_threshold = float(best15['threshold'])
else:
    final_threshold = 0.628

model_path = os.path.join(FINAL_ARTIFACTS_DIR, "final_lambdarank_model.txt")
threshold_path = os.path.join(FINAL_ARTIFACTS_DIR, "final_lambdarank_threshold.txt")

final_model.save_model(model_path)
with open(threshold_path, "w") as f:
    f.write(str(final_threshold))
print(f"Saved final LightGBM model to {model_path}")
print(f"Saved optimal threshold ({final_threshold}) to {threshold_path}")

# --- 2. Save TF-IDF components ---
vectorizer_path = os.path.join(FINAL_ARTIFACTS_DIR, "tfidf_vectorizer.joblib")
x_test_s1_path = os.path.join(FINAL_ARTIFACTS_DIR, "X_test_s1_matrix.npz")

# Ensure we look up active variables
active_vectorizer = globals().get('vectorizer_3b', globals().get('vectorizer'))
active_X_test_s1 = globals().get('X_s1_3b', globals().get('X_test_s1'))

joblib.dump(active_vectorizer, vectorizer_path)
sp.save_npz(x_test_s1_path, active_X_test_s1)
print(f"Saved TF-IDF vectorizer to {vectorizer_path}")
print(f"Saved X_test_s1 sparse matrix to {x_test_s1_path}")

# --- 3. Save Production Blocking Indexes ---
blocking_indexes_to_save = {
    "name_first_blocks": globals().get('test_s1_name_first_blocks', {}),
    "name_last_blocks": globals().get('test_s1_name_last_blocks', {}),
    "addr_first_blocks": globals().get('test_s1_addr_first_blocks', {}),
    "addr_last_blocks": globals().get('test_s1_addr_last_blocks', {}),
    "num_country_blocks": globals().get('test_s1_num_country_blocks', {}),
    "rare_token_index_prod": globals().get('rare_token_index_prod', {}),
    "token_df_prod": globals().get('token_df_prod', {})
}

for name, obj in blocking_indexes_to_save.items():
    if obj:
        obj_path = os.path.join(FINAL_ARTIFACTS_DIR, f"{name}.joblib")
        joblib.dump(obj, obj_path)
        print(f"Saved blocking index '{name}' to {obj_path}")
    else:
        print(f"Warning: '{name}' index not found or empty. Skipping.")

# --- 4. Save Key DataFrames ---
VAL_S1.to_parquet(os.path.join(FINAL_ARTIFACTS_DIR, "VAL_S1.parquet"), index=False)
if 'VAL_TARGET_POOL' in globals():
    VAL_TARGET_POOL.to_parquet(os.path.join(FINAL_ARTIFACTS_DIR, "VAL_TARGET_POOL.parquet"), index=False)
best_all.to_parquet(os.path.join(FINAL_ARTIFACTS_DIR, "best_all_scored_validation_edges.parquet"), index=False)
print("Saved VAL_S1, VAL_TARGET_POOL, and best_all DataFrames.")

# --- 5. Save Lookup Dictionaries ---
joblib.dump(target_to_s1, os.path.join(FINAL_ARTIFACTS_DIR, "target_to_s1.joblib"))
joblib.dump(true_by_s1, os.path.join(FINAL_ARTIFACTS_DIR, "true_by_s1.joblib"))
print("Saved target_to_s1 and true_by_s1 lookup dictionaries.")

print("All requested artifacts have been saved to Google Drive.")

Created directory: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS
Saved final LightGBM model to /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/final_lambdarank_model.txt
Saved optimal threshold (0.6280000000000001) to /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/final_lambdarank_threshold.txt
Saved TF-IDF vectorizer to /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/tfidf_vectorizer.joblib
Saved X_test_s1 sparse matrix to /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/X_test_s1_matrix.npz
Saved blocking index 'name_first_blocks' to /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/name_first_blocks.joblib
Saved blocking index 'name_last_blocks' to /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/name_last_blocks.joblib
Saved blocking index 'addr_fir

In [ ]:
# CELL 1 — LOAD PRODUCTION ARTIFACTS + VERIFY SCALE (for full test-set inference)
# Loads everything from FINAL_SAVED_ARTIFACTS on disk (not live session state),
# and checks the block indexes actually match TEST S1 scale (1,732,544 rows),
# not a leftover 20k-validation-scale copy. No S2/S3 processing yet.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import scipy.sparse as sp
import lightgbm as lgb
import json

BASE = Path("/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026")
FINAL_DIR = BASE / "MODELS" / "FINAL_SAVED_ARTIFACTS"
TEST_DIR = BASE / "DATASET" / "test"

print("=" * 100)
print("CELL 1 — LOAD + VERIFY PRODUCTION ARTIFACTS")
print("=" * 100)

# --- Model + threshold ---
matcher = lgb.Booster(model_file=str(FINAL_DIR / "final_lambdarank_model.txt"))
with open(FINAL_DIR / "final_lambdarank_threshold.txt") as f:
    threshold = float(f.read().strip())
print(f"\nModel loaded. Trees: {matcher.num_trees()}  Features: {matcher.num_feature()}")
print(f"Threshold: {threshold}  (expected: ~0.628)")

# --- TF-IDF vectorizer + S1 matrix ---
vectorizer = joblib.load(FINAL_DIR / "tfidf_vectorizer.joblib")
X_test_s1 = sp.load_npz(FINAL_DIR / "X_test_s1_matrix.npz")
print(f"\nVectorizer vocab size: {len(vectorizer.vocabulary_)}")
print(f"X_test_s1 shape: {X_test_s1.shape}  (expected rows: 1,732,544)")

# --- Block indexes: load + verify scale against CELL 13A/13B production stats ---
block_files = {
    "name_first_blocks": "name_first",
    "name_last_blocks": "name_last",
    "addr_first_blocks": "addr_first",
    "addr_last_blocks": "addr_last",
    "num_country_blocks": "num_country",
}

blocks = {}
print("\n" + "-" * 100)
print("BLOCK INDEX SCALE CHECK (compare against CELL 13A production numbers)")
print("-" * 100)
for fname, label in block_files.items():
    obj = joblib.load(FINAL_DIR / f"{fname}.joblib")
    blocks[label] = obj
    sizes = np.array([len(v) for v in obj.values()])
    total_indexed = sizes.sum()
    print(
        f"{label:12s} blocks={len(obj):>7,}  median={int(np.median(sizes)):>6,}  "
        f"p95={int(np.percentile(sizes,95)):>7,}  max={sizes.max():>7,}  "
        f"total_entries={total_indexed:>10,}"
    )

rare_token_index = joblib.load(FINAL_DIR / "rare_token_index_prod.joblib")
token_df = joblib.load(FINAL_DIR / "token_df_prod.joblib")
print(f"\nrare_token_index_prod: {len(rare_token_index):,} postings "
      f"(expected: ~508,089)")
print(f"token_df_prod: {len(token_df):,} unique tokens (expected: ~463,036)")

# --- TEST S1 entity order, for aligning row index <-> entity_id ---
test_s1_ids = pd.read_csv(
    TEST_DIR / "test_source1.tsv", sep="\t", usecols=["entity_id"], dtype=str
)["entity_id"].tolist()
print(f"\nTEST S1 entity count: {len(test_s1_ids):,}  (expected: 1,732,544)")
print(f"Row-count match with X_test_s1: {len(test_s1_ids) == X_test_s1.shape[0]}")

# --- Checkpoint manifest for the batched S2/S3 production run ---
RUN_DIR = BASE / "MODELS" / "PRODUCTION_RUN"
RUN_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = RUN_DIR / "manifest.json"

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    print(f"\nExisting manifest found: {manifest}")
else:
    manifest = {"s2_batches_done": 0, "s3_batches_done": 0, "batch_size": 50_000}
    MANIFEST_PATH.write_text(json.dumps(manifest))
    print(f"\nNew manifest created at {MANIFEST_PATH}: {manifest}")

print("\nCELL 1 COMPLETE — waiting for output before Cell 2.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CELL 1 — LOAD + VERIFY PRODUCTION ARTIFACTS

Model loaded. Trees: 424  Features: 56
Threshold: 0.6280000000000001  (expected: ~0.628)

Vectorizer vocab size: 100000
X_test_s1 shape: (1732544, 100000)  (expected rows: 1,732,544)

----------------------------------------------------------------------------------------------------
BLOCK INDEX SCALE CHECK (compare against CELL 13A production numbers)
----------------------------------------------------------------------------------------------------
name_first   blocks=    109  median= 9,879  p95= 47,993  max=136,193  total_entries= 1,732,544
name_last    blocks=    107  median=   637  p95= 45,756  max=613,148  total_entries= 1,732,544
addr_first   blocks=    105  median= 6,683  p95= 61,519  max=180,957  total_entries= 1,732,544
addr_last    blocks=    113  median=   784  p95= 66,207  max=324,662  total_entries= 

In [ ]:
# CELL 2 — FRANCE COVERAGE SPOT-CHECK (before committing to the full ~10M-record run)
# Checks whether candidate retrieval collapses for country="France" test targets,
# since France only appears in TEST data and the reranker never saw France
# features during training. Reuses the exact normalization from CELL 13A/13B.

import re
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 2 — FRANCE COVERAGE SPOT-CHECK")
print("=" * 100)

def normalize_exact(x):
    if pd.isna(x):
        return ""
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())

def word_tokens(text):
    if pd.isna(text):
        return []
    return [t for t in re.findall(r"[^\W_]+", str(text).lower(), flags=re.UNICODE) if len(t) >= 2]

entity_id_to_local = {eid: i for i, eid in enumerate(test_s1_ids)}
print(f"Built entity_id -> local_idx map: {len(entity_id_to_local):,} entries")

def candidate_local_idxs(name_n, addr_n, country_n):
    cands = set()
    if name_n:
        cands.update(blocks["name_first"].get((country_n, name_n[0]), []))
        cands.update(blocks["name_last"].get((country_n, name_n[-1]), []))
    if addr_n:
        cands.update(blocks["addr_first"].get((country_n, addr_n[0]), []))
        cands.update(blocks["addr_last"].get((country_n, addr_n[-1]), []))
        for num in set(re.findall(r"\d+", addr_n)):
            cands.update(blocks["num_country"].get((country_n, num), []))
    # rare-token index stores entity_id, not local_idx -- convert
    tokens = set(word_tokens(name_n) + word_tokens(addr_n))
    for tok in tokens:
        for eid in rare_token_index.get((country_n, tok), []):
            local = entity_id_to_local.get(eid)
            if local is not None:
                cands.add(local)
    return cands

# --- Sample France targets + a same-size non-France comparison, from S2 and S3 ---
SAMPLE_PER_SOURCE = 2000
france_rows, other_rows = [], []

for path, source_name in [
    (TEST_DIR / "test_source2.tsv", "S2"),
    (TEST_DIR / "test_source3.tsv", "S3"),
]:
    france_count, other_count = 0, 0
    for chunk in pd.read_csv(
        path, sep="\t", dtype=str,
        usecols=["entity_id", "business_name", "business_address", "country"],
        chunksize=250_000,
    ):
        chunk["country_n"] = chunk["country"].fillna("").astype(str).str.strip().str.lower()
        fr = chunk[chunk["country_n"] == "france"]
        ot = chunk[chunk["country_n"] != "france"]
        if france_count < SAMPLE_PER_SOURCE and len(fr):
            take = fr.iloc[: SAMPLE_PER_SOURCE - france_count]
            france_rows.append(take)
            france_count += len(take)
        if other_count < SAMPLE_PER_SOURCE and len(ot):
            take = ot.iloc[: SAMPLE_PER_SOURCE - other_count]
            other_rows.append(take)
            other_count += len(take)
        if france_count >= SAMPLE_PER_SOURCE and other_count >= SAMPLE_PER_SOURCE:
            break
    print(f"{source_name}: collected {france_count} France rows, {other_count} comparison rows")

france_df = pd.concat(france_rows, ignore_index=True)
other_df = pd.concat(other_rows, ignore_index=True)

# --- Run candidate retrieval on both samples ---
def evaluate(df, label):
    counts, zero = [], 0
    for row in df.itertuples(index=False):
        name_n = normalize_exact(row.business_name)
        addr_n = normalize_exact(row.business_address)
        cands = candidate_local_idxs(name_n, addr_n, row.country_n)
        counts.append(len(cands))
        if not cands:
            zero += 1
    counts = np.array(counts)
    print(f"\n{label}: n={len(df):,}")
    print(f"  zero-candidate rate: {zero/len(df)*100:.2f}%")
    print(f"  mean={counts.mean():.1f}  median={np.median(counts):.0f}  "
          f"p95={np.percentile(counts,95):.0f}  max={counts.max():,}")
    return counts

france_counts = evaluate(france_df, "FRANCE")
other_counts = evaluate(other_df, "US/INDIA")

print("\nCELL 2 COMPLETE — waiting for output before Cell 3.")

CELL 2 — FRANCE COVERAGE SPOT-CHECK
Built entity_id -> local_idx map: 1,732,544 entries
S2: collected 2000 France rows, 2000 comparison rows
S3: collected 2000 France rows, 2000 comparison rows

FRANCE: n=4,000
  zero-candidate rate: 0.00%
  mean=173069.5  median=236307  p95=245265  max=249,219

US/INDIA: n=4,000
  zero-candidate rate: 0.00%
  mean=315518.2  median=240359  p95=665146  max=722,973

CELL 2 COMPLETE — waiting for output before Cell 3.


In [ ]:
# CELL 3 — RETRIEVAL SPEED BENCHMARK AT PRODUCTION SCALE
# Times the char-block+TF-IDF channel vs the rare-token-only channel
# on real test targets, to decide which is actually viable before
# committing to ~10M records.

import time

sample_A = pd.concat([france_df.head(100), other_df.head(100)], ignore_index=True)  # char-block+TF-IDF (smaller N -- it's the suspect)
sample_B = pd.concat([france_df.head(500), other_df.head(500)], ignore_index=True)  # rare-token-only (known fast, larger N for precision)

# --- Method A: char-block union -> sparse TF-IDF top-100 ---
t0 = time.time()
for row in sample_A.itertuples(index=False):
    name_n = normalize_exact(row.business_name)
    addr_n = normalize_exact(row.business_address)
    country_n = row.country_n

    cands = set()
    if name_n:
        cands.update(blocks["name_first"].get((country_n, name_n[0]), []))
        cands.update(blocks["name_last"].get((country_n, name_n[-1]), []))
    if addr_n:
        cands.update(blocks["addr_first"].get((country_n, addr_n[0]), []))
        cands.update(blocks["addr_last"].get((country_n, addr_n[-1]), []))
        for num in set(re.findall(r"\d+", addr_n)):
            cands.update(blocks["num_country"].get((country_n, num), []))

    if cands:
        cand_arr = np.fromiter(cands, dtype=np.int32)
        target_text = name_n + " " + addr_n
        q_vec = vectorizer.transform([target_text])
        scores = (q_vec @ X_test_s1[cand_arr].T).toarray().ravel()
        top = np.argpartition(scores, -min(100, len(scores)))[-min(100, len(scores)):]
elapsed_A = time.time() - t0
qps_A = len(sample_A) / elapsed_A
print(f"METHOD A (char-block + TF-IDF): {len(sample_A)} queries in {elapsed_A:.2f}s = {qps_A:.3f} q/s")
print(f"  Projected for 9,969,589 targets: {9_969_589/qps_A/3600:.1f} hours")

# --- Method B: rare-token only ---
t0 = time.time()
for row in sample_B.itertuples(index=False):
    name_n = normalize_exact(row.business_name)
    addr_n = normalize_exact(row.business_address)
    country_n = row.country_n
    tokens = set(word_tokens(name_n) + word_tokens(addr_n))
    usable = sorted(
        [(token_df.get(t, 0), t) for t in tokens if 0 < token_df.get(t, 0) <= 1000],
        key=lambda x: x[0],
    )
    cands = set()
    for _, tok in usable[:4]:
        cands.update(rare_token_index.get((country_n, tok), []))
elapsed_B = time.time() - t0
qps_B = len(sample_B) / elapsed_B
print(f"\nMETHOD B (rare-token only): {len(sample_B)} queries in {elapsed_B:.2f}s = {qps_B:.1f} q/s")
print(f"  Projected for 9,969,589 targets: {9_969_589/qps_B/3600:.2f} hours")

print("\nCELL 3 COMPLETE — waiting for output before deciding retrieval strategy.")

NameError: name 'pd' is not defined

In [ ]:
# CELL 3 — RETRIEVAL SPEED BENCHMARK AT PRODUCTION SCALE (self-contained)

import re
import time
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 3 — RETRIEVAL SPEED BENCHMARK")
print("=" * 100)

# --- Defensive check: don't fail cryptically if session state was lost ---
required = ["blocks", "rare_token_index", "token_df", "X_test_s1", "vectorizer",
            "france_df", "other_df", "entity_id_to_local"]
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        f"Rerun CELL 1 (loads blocks/vectorizer/X_test_s1) "
        f"and CELL 2 (builds france_df/other_df/entity_id_to_local) first."
    )
print("All required objects present.")

def normalize_exact(x):
    if pd.isna(x):
        return ""
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())

def word_tokens(text):
    if pd.isna(text):
        return []
    return [t for t in re.findall(r"[^\W_]+", str(text).lower(), flags=re.UNICODE) if len(t) >= 2]

sample_A = pd.concat([france_df.head(100), other_df.head(100)], ignore_index=True)
sample_B = pd.concat([france_df.head(500), other_df.head(500)], ignore_index=True)

# --- Method A: char-block union -> sparse TF-IDF top-100 ---
t0 = time.time()
for row in sample_A.itertuples(index=False):
    name_n = normalize_exact(row.business_name)
    addr_n = normalize_exact(row.business_address)
    country_n = row.country_n

    cands = set()
    if name_n:
        cands.update(blocks["name_first"].get((country_n, name_n[0]), []))
        cands.update(blocks["name_last"].get((country_n, name_n[-1]), []))
    if addr_n:
        cands.update(blocks["addr_first"].get((country_n, addr_n[0]), []))
        cands.update(blocks["addr_last"].get((country_n, addr_n[-1]), []))
        for num in set(re.findall(r"\d+", addr_n)):
            cands.update(blocks["num_country"].get((country_n, num), []))

    if cands:
        cand_arr = np.fromiter(cands, dtype=np.int32)
        target_text = name_n + " " + addr_n
        q_vec = vectorizer.transform([target_text])
        scores = (q_vec @ X_test_s1[cand_arr].T).toarray().ravel()
        top = np.argpartition(scores, -min(100, len(scores)))[-min(100, len(scores)):]
elapsed_A = time.time() - t0
qps_A = len(sample_A) / elapsed_A
print(f"\nMETHOD A (char-block + TF-IDF): {len(sample_A)} queries in {elapsed_A:.2f}s = {qps_A:.3f} q/s")
print(f"  Projected for 9,969,589 targets: {9_969_589/qps_A/3600:.1f} hours")

# --- Method B: rare-token only ---
t0 = time.time()
for row in sample_B.itertuples(index=False):
    name_n = normalize_exact(row.business_name)
    addr_n = normalize_exact(row.business_address)
    country_n = row.country_n
    tokens = set(word_tokens(name_n) + word_tokens(addr_n))
    usable = sorted(
        [(token_df.get(t, 0), t) for t in tokens if 0 < token_df.get(t, 0) <= 1000],
        key=lambda x: x[0],
    )
    cands = set()
    for _, tok in usable[:4]:
        cands.update(rare_token_index.get((country_n, tok), []))
elapsed_B = time.time() - t0
qps_B = len(sample_B) / elapsed_B
print(f"\nMETHOD B (rare-token only): {len(sample_B)} queries in {elapsed_B:.1f}s = {qps_B:.1f} q/s")
print(f"  Projected for 9,969,589 targets: {9_969_589/qps_B/3600:.2f} hours")

print("\nCELL 3 COMPLETE — waiting for output before deciding retrieval strategy.")

CELL 3 — RETRIEVAL SPEED BENCHMARK


RuntimeError: Missing required objects: ['blocks', 'rare_token_index', 'token_df', 'X_test_s1', 'vectorizer', 'france_df', 'other_df', 'entity_id_to_local']. Rerun CELL 1 (loads blocks/vectorizer/X_test_s1) and CELL 2 (builds france_df/other_df/entity_id_to_local) first.

In [ ]:
# CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION")
print("=" * 100)

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}"
    )

print("Best-all rows:", f"{len(best_all):,}")
print("Validation S1:", f"{len(VAL_S1):,}")
print("Current threshold: 0.640")

# ------------------------------------------------------------
# Precompute true sets
# ------------------------------------------------------------

s1_ids = VAL_S1["entity_id"].tolist()

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

def evaluate_threshold_fast(threshold):

    accepted = best_all[
        best_all["score"] >= threshold
    ]

    predicted_by_s1 = {}

    for row in accepted.itertuples(index=False):

        predicted_by_s1.setdefault(
            row.s1_id,
            set()
        ).add(
            row.query_id
        )

    f05_values = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for sid in s1_ids:

        true_ids = true_by_s1.get(
            sid,
            set()
        )

        pred_ids = predicted_by_s1.get(
            sid,
            set()
        )

        tp = len(
            true_ids & pred_ids
        )

        fp = len(
            pred_ids - true_ids
        )

        fn = len(
            true_ids - pred_ids
        )

        if tp + fp > 0:
            precision = tp / (tp + fp)
        else:
            precision = (
                1.0
                if len(true_ids) == 0
                else 0.0
            )

        if tp + fn > 0:
            recall = tp / (tp + fn)
        else:
            recall = 1.0

        if precision > 0 and recall > 0:
            f05 = (
                1.25
                * precision
                * recall
                /
                (
                    0.25 * precision
                    + recall
                )
            )
        else:
            f05 = 0.0

        f05_values.append(f05)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    return {
        "threshold": threshold,
        "macro_f05": float(
            np.mean(f05_values)
        ),
        "precision": (
            total_tp / (total_tp + total_fp)
            if total_tp + total_fp > 0
            else 0.0
        ),
        "recall": (
            total_tp / (total_tp + total_fn)
            if total_tp + total_fn > 0
            else 0.0
        ),
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn
    }


# ------------------------------------------------------------
# Fine sweep
# ------------------------------------------------------------

thresholds = np.arange(
    0.550,
    0.701,
    0.001
)

results = []

for i, threshold in enumerate(thresholds):

    result = evaluate_threshold_fast(
        float(threshold)
    )

    results.append(result)

results_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Top results
# ------------------------------------------------------------

top = (
    results_df
    .sort_values(
        "macro_f05",
        ascending=False
    )
    .head(20)
)

print(
    "\n" + "=" * 100
)
print(
    "TOP 20 THRESHOLDS"
)
print(
    "=" * 100
)

print(
    top.to_string(
        index=False,
        formatters={
            "threshold":
                lambda x: f"{x:.3f}",
            "macro_f05":
                lambda x: f"{x:.6f}",
            "precision":
                lambda x: f"{x:.6f}",
            "recall":
                lambda x: f"{x:.6f}"
        }
    )
)

best = results_df.loc[
    results_df["macro_f05"].idxmax()
]

print(
    "\n" + "=" * 100
)
print(
    "BEST FINE-GRAINED RESULT"
)
print(
    "=" * 100
)

print(
    "Threshold:",
    f"{best['threshold']:.3f}"
)

print(
    "Macro F0.5:",
    f"{best['macro_f05']:.6f}"
)

print(
    "Precision:",
    f"{best['precision']:.6f}"
)

print(
    "Recall:",
    f"{best['recall']:.6f}"
)

print(
    "TP:",
    int(best["tp"])
)

print(
    "FP:",
    int(best["fp"])
)

print(
    "FN:",
    int(best["fn"])
)

print(
    "\nImprovement over 0.988468:",
    f"{best['macro_f05'] - 0.988468:+.6f}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

threshold_path = (
    MODEL_DIR
    /
    "fine_threshold_results.csv"
)

results_df.to_csv(
    threshold_path,
    index=False
)

print(
    "\nSaved:",
    threshold_path
)

print(
    "\nCELL 12 COMPLETE."
)

CELL 12 — FINE-GRAINED THRESHOLD OPTIMIZATION
Best-all rows: 69,231
Validation S1: 20,000
Current threshold: 0.640

TOP 20 THRESHOLDS
threshold macro_f05 precision   recall    tp  fp   fn
    0.628  0.988477  0.994811 0.977496 67673 353 1558
    0.637  0.988475  0.994868 0.977337 67662 349 1569
    0.619  0.988472  0.994724 0.977669 67685 359 1546
    0.617  0.988471  0.994695 0.977727 67689 361 1542
    0.638  0.988470  0.994868 0.977322 67661 349 1570
    0.639  0.988470  0.994883 0.977293 67659 348 1572
    0.627  0.988470  0.994796 0.977496 67673 354 1558
    0.618  0.988469  0.994709 0.977683 67686 360 1545
    0.640  0.988468  0.994883 0.977279 67658 348 1573
    0.621  0.988467  0.994724 0.977655 67684 359 1547
    0.620  0.988467  0.994724 0.977655 67684 359 1547
    0.629  0.988467  0.994825 0.977452 67670 352 1561
    0.642  0.988465  0.994897 0.977250 67656 347 1575
    0.615  0.988465  0.994681 0.977756 67691 362 1540
    0.636  0.988464  0.994854 0.977351 67663 350 1568
  

In [ ]:
# CELL 12A — RECOVER FULL VALIDATION STATE FOR CELL 12

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("CELL 12A — RECOVER CELL 11 VALIDATION STATE")
print("=" * 100)

BASE = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

MODEL_DIR = BASE / "MODELS"

# ============================================================
# 1. Recover FULL target-exclusive predictions from Cell 11
# ============================================================

best_path = (
    MODEL_DIR
    /
    "full_validation_target_exclusive_top1.parquet"
)

if not best_path.exists():
    raise FileNotFoundError(
        f"Cell 11 checkpoint not found:\n{best_path}"
    )

print("\nLoading Cell 11 checkpoint...")

best_all = pd.read_parquet(
    best_path
)

print(
    "best_all rows:",
    f"{len(best_all):,}"
)

print(
    "Columns:",
    best_all.columns.tolist()
)

print(
    "Score range:",
    f"{best_all['score'].min():.6f}",
    "->",
    f"{best_all['score'].max():.6f}"
)

# ============================================================
# 2. Recover representative 20K S1
# ============================================================

validation_path = (
    MODEL_DIR
    /
    "representative_s1_validation_20k.parquet"
)

if not validation_path.exists():
    raise FileNotFoundError(
        f"Validation S1 file not found:\n{validation_path}"
    )

print(
    "\nLoading representative S1 validation..."
)

VAL_S1 = pd.read_parquet(
    validation_path
)

print(
    "VAL_S1 rows:",
    f"{len(VAL_S1):,}"
)

print(
    "VAL_S1 columns:",
    VAL_S1.columns.tolist()
)

# ============================================================
# 3. Load ground truth
# ============================================================

gt_path = (
    BASE
    /
    "DATASET"
    /
    "train"
    /
    "train_ground_truth.tsv"
)

if not gt_path.exists():

    # fallback search
    candidates = list(
        (BASE / "DATASET").rglob(
            "train_ground_truth.tsv"
        )
    )

    if not candidates:
        raise FileNotFoundError(
            "Could not locate train_ground_truth.tsv"
        )

    gt_path = candidates[0]

print(
    "\nLoading ground truth..."
)

gt = pd.read_csv(
    gt_path,
    sep="\t"
)

print(
    "GT rows:",
    f"{len(gt):,}"
)

print(
    "GT columns:",
    gt.columns.tolist()
)

# ============================================================
# 4. Build target -> S1 mapping
# ============================================================

if not {
    "source1_entity_id",
    "matched_entity_ids"
}.issubset(gt.columns):

    raise RuntimeError(
        "Unexpected ground-truth schema."
    )

validation_s1_ids = set(
    VAL_S1["entity_id"]
)

target_to_s1 = {}

for row in gt.itertuples(
    index=False
):

    s1_id = row.source1_entity_id

    if s1_id not in validation_s1_ids:
        continue

    matched = row.matched_entity_ids

    if pd.isna(matched):
        continue

    matched = str(
        matched
    ).strip()

    if not matched:
        continue

    for target_id in matched.split(","):

        target_id = target_id.strip()

        if not target_id:
            continue

        target_to_s1[
            target_id
        ] = s1_id

print(
    "\nValidation target -> S1 mappings:",
    f"{len(target_to_s1):,}"
)

# ============================================================
# 5. Build true_by_s1
# ============================================================

true_by_s1 = {
    sid: set()
    for sid in VAL_S1["entity_id"]
}

for target_id, s1_id in target_to_s1.items():

    if s1_id in true_by_s1:

        true_by_s1[
            s1_id
        ].add(
            target_id
        )

true_pair_count = sum(
    len(v)
    for v in true_by_s1.values()
)

matched_s1_count = sum(
    1
    for v in true_by_s1.values()
    if len(v) > 0
)

no_match_count = (
    len(true_by_s1)
    -
    matched_s1_count
)

print(
    "\n" + "=" * 100
)
print(
    "RECOVERED VALIDATION STATE"
)
print(
    "=" * 100
)

print(
    "S1 entities:",
    f"{len(true_by_s1):,}"
)

print(
    "Matched S1:",
    f"{matched_s1_count:,}"
)

print(
    "No-match S1:",
    f"{no_match_count:,}"
)

print(
    "True pairs:",
    f"{true_pair_count:,}"
)

print(
    "Predicted target-best edges:",
    f"{len(best_all):,}"
)

# ============================================================
# 6. Verify against Cell 11's known numbers
# ============================================================

print(
    "\nExpected from Cell 11:"
)

print(
    "No-match S1: 1,131"
)

print(
    "True pairs: 69,231"
)

print(
    "\nChecks:"
)

print(
    "No-match check:",
    "PASS"
    if no_match_count == 1131
    else f"WARNING ({no_match_count})"
)

print(
    "True-pair check:",
    "PASS"
    if true_pair_count == 69231
    else f"WARNING ({true_pair_count})"
)

# ============================================================
# 7. Verify prediction structure
# ============================================================

duplicate_target_predictions = (
    best_all["query_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate target predictions:",
    duplicate_target_predictions
)

if duplicate_target_predictions != 0:

    raise RuntimeError(
        "best_all is expected to contain exactly one "
        "S1 prediction per target."
    )

print(
    "\nCELL 12A COMPLETE."
)

print(
    "\nNow rerun CELL 12 unchanged."
)

CELL 12A — RECOVER CELL 11 VALIDATION STATE

Loading Cell 11 checkpoint...
best_all rows: 69,231
Columns: ['query_id', 's1_id', 'score']
Score range: 0.000820 -> 0.999660

Loading representative S1 validation...
VAL_S1 rows: 20,000
VAL_S1 columns: ['entity_id', 'business_name', 'business_address', 'country', 'source1_entity_id', 'matched_entity_ids', 'true_count', 'true_ids_str']

Loading ground truth...
GT rows: 2,206,821
GT columns: ['source1_entity_id', 'matched_entity_ids']

Validation target -> S1 mappings: 69,231

RECOVERED VALIDATION STATE
S1 entities: 20,000
Matched S1: 18,869
No-match S1: 1,131
True pairs: 69,231
Predicted target-best edges: 69,231

Expected from Cell 11:
No-match S1: 1,131
True pairs: 69,231

Checks:
No-match check: PASS
True-pair check: PASS
Duplicate target predictions: 0

CELL 12A COMPLETE.

Now rerun CELL 12 unchanged.


In [ ]:
# CELL 13 — ERROR SOURCE ANALYSIS AT BEST THRESHOLD

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 13 — REMAINING ERROR ANALYSIS")
print("=" * 100)

BEST_THRESHOLD = 0.628

# ------------------------------------------------------------
# Build predictions
# ------------------------------------------------------------

pred_edges = best_all[
    best_all["score"] >= BEST_THRESHOLD
].copy()

pred_by_s1 = {}

for row in pred_edges.itertuples(index=False):

    pred_by_s1.setdefault(
        row.s1_id,
        set()
    ).add(
        row.query_id
    )

# ------------------------------------------------------------
# Analyze every S1
# ------------------------------------------------------------

error_rows = []

for sid in VAL_S1["entity_id"]:

    true_ids = true_by_s1.get(
        sid,
        set()
    )

    pred_ids = pred_by_s1.get(
        sid,
        set()
    )

    tp_ids = (
        true_ids
        &
        pred_ids
    )

    fp_ids = (
        pred_ids
        -
        true_ids
    )

    fn_ids = (
        true_ids
        -
        pred_ids
    )

    error_rows.append({
        "s1_id": sid,
        "true_count": len(true_ids),
        "pred_count": len(pred_ids),
        "tp": len(tp_ids),
        "fp": len(fp_ids),
        "fn": len(fn_ids),
        "is_no_match": len(true_ids) == 0,
        "perfect": (
            len(fn_ids) == 0
            and
            len(fp_ids) == 0
        )
    })

error_df = pd.DataFrame(
    error_rows
)

print(
    "\nS1 validation summary:"
)

print(
    "Total S1:",
    len(error_df)
)

print(
    "Perfect S1:",
    int(error_df["perfect"].sum())
)

print(
    "S1 with FN:",
    int((error_df["fn"] > 0).sum())
)

print(
    "S1 with FP:",
    int((error_df["fp"] > 0).sum())
)

print(
    "No-match S1:",
    int(error_df["is_no_match"].sum())
)

# ------------------------------------------------------------
# FALSE POSITIVE ANALYSIS
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FALSE POSITIVE ANALYSIS"
)
print(
    "=" * 100
)

fp_rows = []

for row in pred_edges.itertuples(index=False):

    true_owner = target_to_s1.get(
        row.query_id,
        None
    )

    if true_owner != row.s1_id:

        fp_rows.append({
            "query_id":
                row.query_id,

            "predicted_s1":
                row.s1_id,

            "true_owner":
                true_owner,

            "score":
                row.score
        })

fp_df = pd.DataFrame(
    fp_rows
)

print(
    "FP edges:",
    len(fp_df)
)

if len(fp_df):

    print(
        "\nFP score distribution:"
    )

    print(
        fp_df["score"]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

    print(
        "\nTop 30 false positives:"
    )

    print(
        fp_df
        .sort_values(
            "score",
            ascending=False
        )
        .head(30)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# FALSE NEGATIVE SOURCE ANALYSIS
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "FALSE NEGATIVE SOURCE ANALYSIS"
)
print(
    "=" * 100
)

fn_rows = []

# Map target -> selected prediction
selected_target = {}

for row in best_all.itertuples(index=False):

    selected_target[
        row.query_id
    ] = (
        row.s1_id,
        row.score
    )

for sid in VAL_S1["entity_id"]:

    true_ids = true_by_s1.get(
        sid,
        set()
    )

    predicted_ids = pred_by_s1.get(
        sid,
        set()
    )

    missed = (
        true_ids
        -
        predicted_ids
    )

    for target_id in missed:

        selected = selected_target.get(
            target_id
        )

        if selected is None:

            source = "RETRIEVAL_MISS"
            selected_sid = None
            selected_score = None

        else:

            selected_sid, selected_score = (
                selected
            )

            if selected_sid != sid:

                source = "EXCLUSIVITY_ERROR"

            else:

                source = "BELOW_THRESHOLD"

        fn_rows.append({
            "true_s1":
                sid,

            "target_id":
                target_id,

            "source":
                source,

            "selected_s1":
                selected_sid,

            "selected_score":
                selected_score
        })

fn_df = pd.DataFrame(
    fn_rows
)

print(
    "Total FN:",
    len(fn_df)
)

print(
    "\nFN sources:"
)

print(
    fn_df["source"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# Score distributions for exclusivity errors
# ------------------------------------------------------------

exclusive_df = fn_df[
    fn_df["source"]
    ==
    "EXCLUSIVITY_ERROR"
].copy()

if len(exclusive_df):

    print(
        "\n" + "=" * 100
    )

    print(
        "EXCLUSIVITY ERROR SCORES"
    )

    print(
        "=" * 100
    )

    print(
        exclusive_df[
            "selected_score"
        ].describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

# ------------------------------------------------------------
# Retrieval miss count
# ------------------------------------------------------------

retrieval_misses = fn_df[
    fn_df["source"]
    ==
    "RETRIEVAL_MISS"
]

print(
    "\nRetrieval misses:",
    len(retrieval_misses)
)

print(
    "Exclusivity errors:",
    int(
        (
            fn_df["source"]
            ==
            "EXCLUSIVITY_ERROR"
        ).sum()
    )
)

print(
    "Below-threshold:",
    int(
        (
            fn_df["source"]
            ==
            "BELOW_THRESHOLD"
        ).sum()
    )
)

# ------------------------------------------------------------
# Save diagnostics
# ------------------------------------------------------------

error_path = (
    MODEL_DIR
    /
    "cell13_error_analysis.csv"
)

error_df.to_csv(
    error_path,
    index=False
)

fn_path = (
    MODEL_DIR
    /
    "cell13_false_negatives.csv"
)

fn_df.to_csv(
    fn_path,
    index=False
)

fp_path = (
    MODEL_DIR
    /
    "cell13_false_positives.csv"
)

fp_df.to_csv(
    fp_path,
    index=False
)

print(
    "\nSaved diagnostics:"
)

print(
    error_path
)

print(
    fn_path
)

print(
    fp_path
)

print(
    "\nCELL 13 COMPLETE."
)

CELL 13 — REMAINING ERROR ANALYSIS

S1 validation summary:
Total S1: 20000
Perfect S1: 18402
S1 with FN: 1324
S1 with FP: 308
No-match S1: 1131

FALSE POSITIVE ANALYSIS
FP edges: 353

FP score distribution:
count    353.000000
mean       0.888358
std        0.105543
min        0.628527
1%         0.637421
5%         0.679098
10%        0.709955
25%        0.813375
50%        0.924121
75%        0.978675
90%        0.994780
95%        0.997682
99%        0.998475
max        0.999644
Name: score, dtype: float64

Top 30 false positives:
    query_id predicted_s1   true_owner    score
S3-797521910 S1-979690967 S1-383911575 0.999644
S2-640825269 S1-491584669 S1-111270002 0.999479
S3-791170196 S1-597550527 S1-330267378 0.999391
S2-692390669 S1-289217668 S1-613731683 0.998502
S3-564841594 S1-550886545 S1-438115387 0.998449
S2-786265004 S1-360579271 S1-321361130 0.998398
S3-727062069 S1-123636921 S1-437766158 0.998378
S3-181916057 S1-387795523 S1-797677711 0.998322
S3-712542961 S1-617334573  S

In [ ]:
# CELL 14 — INSPECT EXCLUSIVITY ERRORS

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("CELL 14 — EXCLUSIVITY ERROR INSPECTION")
print("=" * 100)

MODEL_DIR = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

fn_path = (
    MODEL_DIR
    /
    "cell13_false_negatives.csv"
)

fp_path = (
    MODEL_DIR
    /
    "cell13_false_positives.csv"
)

fn = pd.read_csv(
    fn_path
)

fp = pd.read_csv(
    fp_path
)

print(
    "FN rows:",
    len(fn)
)

print(
    "FN columns:"
)

print(
    fn.columns.tolist()
)

print(
    "\nFP rows:",
    len(fp)
)

print(
    "FP columns:"
)

print(
    fp.columns.tolist()
)

# ------------------------------------------------------------
# Exclusivity errors
# ------------------------------------------------------------

if "source" in fn.columns:

    excl = fn[
        fn["source"]
        ==
        "EXCLUSIVITY_ERROR"
    ].copy()

else:

    raise RuntimeError(
        "The FN diagnostic does not contain 'source'."
    )

print(
    "\nExclusivity errors:",
    len(excl)
)

print(
    "\nExclusivity columns:"
)

print(
    excl.columns.tolist()
)

print(
    "\nFirst 30 exclusivity errors:"
)

print(
    excl.head(30).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Check whether true-owner score is already available
# ------------------------------------------------------------

possible_true_score_cols = [
    c
    for c in excl.columns
    if (
        "true" in c.lower()
        or
        "owner" in c.lower()
        or
        "score" in c.lower()
    )
]

print(
    "\nPotential diagnostic score columns:"
)

print(
    possible_true_score_cols
)

# ------------------------------------------------------------
# Selected score distribution
# ------------------------------------------------------------

if "selected_score" in excl.columns:

    print(
        "\nSelected-score distribution:"
    )

    print(
        excl["selected_score"]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

# ------------------------------------------------------------
# How many errors are high-confidence?
# ------------------------------------------------------------

if "selected_score" in excl.columns:

    print(
        "\nExclusivity errors by selected-score:"
    )

    for threshold in [
        0.10,
        0.25,
        0.40,
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
        0.95,
        0.98
    ]:

        count = int(
            (
                excl[
                    "selected_score"
                ]
                >= threshold
            ).sum()
        )

        print(
            f">= {threshold:.2f}: "
            f"{count:,}"
        )

print(
    "\nCELL 14 COMPLETE."
)

CELL 14 — EXCLUSIVITY ERROR INSPECTION
FN rows: 1558
FN columns:
['true_s1', 'target_id', 'source', 'selected_s1', 'selected_score']

FP rows: 353
FP columns:
['query_id', 'predicted_s1', 'true_owner', 'score']

Exclusivity errors: 972

Exclusivity columns:
['true_s1', 'target_id', 'source', 'selected_s1', 'selected_score']

First 30 exclusivity errors:
     true_s1    target_id            source  selected_s1  selected_score
S1-983540069   S2-4190133 EXCLUSIVITY_ERROR S1-280870409        0.178739
S1-867998778 S2-126598464 EXCLUSIVITY_ERROR S1-681308944        0.740146
S1-193618187 S2-641725400 EXCLUSIVITY_ERROR  S1-85646076        0.084815
S1-757078074 S2-308074507 EXCLUSIVITY_ERROR S1-451251131        0.017634
S1-154182001 S2-253314738 EXCLUSIVITY_ERROR S1-738768899        0.495346
 S1-42246345 S2-819580568 EXCLUSIVITY_ERROR S1-764129222        0.982750
 S1-42246345 S2-112471943 EXCLUSIVITY_ERROR S1-764129222        0.925387
S1-324704727   S2-4097610 EXCLUSIVITY_ERROR S1-547451015    

In [ ]:
# CELL 15 — EXCLUSIVITY ERROR: RETRIEVAL VS RANKING DIAGNOSIS

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 15 — EXCLUSIVITY ERROR DIAGNOSIS")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "rare_candidates",
    "retrieved_top100",
    "VAL_S1",
    "s1_local_to_id"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

# ------------------------------------------------------------
# S1 ID -> local index
# ------------------------------------------------------------

s1_id_to_local = {
    sid: i
    for i, sid in enumerate(
        VAL_S1["entity_id"]
    )
}

print(
    "S1 local index:",
    len(s1_id_to_local)
)

# ------------------------------------------------------------
# Use exclusivity errors from Cell 14
# ------------------------------------------------------------

if "exclusivity_errors" in globals():

    errors = exclusivity_errors.copy()

elif "exclusivity_df" in globals():

    errors = exclusivity_df.copy()

else:

    raise RuntimeError(
        "Could not find Cell 14 exclusivity-error dataframe."
    )

print(
    "Exclusivity errors:",
    len(errors)
)

# ------------------------------------------------------------
# Normalize column names
# ------------------------------------------------------------

if "target_id" not in errors.columns:

    if errors.index.name == "target_id":

        errors = errors.reset_index()

    elif "target_id" in errors.index.names:

        errors = errors.reset_index()

if "target_id" not in errors.columns:

    raise RuntimeError(
        "target_id column not found."
    )

if "true_s1" not in errors.columns:

    raise RuntimeError(
        "true_s1 column not found."
    )

if "selected_s1" not in errors.columns:

    raise RuntimeError(
        "selected_s1 column not found."
    )

# ------------------------------------------------------------
# Candidate-presence test
# ------------------------------------------------------------

diagnostic_rows = []

for row in errors.itertuples(
    index=False
):

    target_id = row.target_id
    true_s1 = row.true_s1
    selected_s1 = row.selected_s1

    true_local = (
        s1_id_to_local.get(
            true_s1,
            -1
        )
    )

    selected_local = (
        s1_id_to_local.get(
            selected_s1,
            -1
        )
    )

    # Rare-token candidates
    rare_raw = rare_candidates.get(
        target_id,
        []
    )

    rare_set = set(
        int(x)
        for x in rare_raw
    )

    # TF-IDF candidates
    tfidf_raw = retrieved_top100.get(
        target_id,
        []
    )

    tfidf_set = set(
        int(x)
        for x in tfidf_raw
    )

    union_set = (
        rare_set
        |
        tfidf_set
    )

    true_in_rare = (
        true_local in rare_set
    )

    true_in_tfidf = (
        true_local in tfidf_set
    )

    true_in_union = (
        true_local in union_set
    )

    selected_in_union = (
        selected_local in union_set
    )

    diagnostic_rows.append({
        "target_id":
            target_id,

        "true_s1":
            true_s1,

        "selected_s1":
            selected_s1,

        "selected_score":
            getattr(
                row,
                "selected_score",
                np.nan
            ),

        "true_local":
            true_local,

        "selected_local":
            selected_local,

        "true_in_rare":
            true_in_rare,

        "true_in_tfidf":
            true_in_tfidf,

        "true_in_union":
            true_in_union,

        "selected_in_union":
            selected_in_union,

        "rare_candidates":
            len(rare_set),

        "tfidf_candidates":
            len(tfidf_set),

        "union_candidates":
            len(union_set)
    })

diag15 = pd.DataFrame(
    diagnostic_rows
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "RETRIEVAL VS RANKING"
)
print(
    "=" * 100
)

total = len(
    diag15
)

retrieval_missing = int(
    (~diag15["true_in_union"])
    .sum()
)

retrieval_present = int(
    diag15["true_in_union"]
    .sum()
)

print(
    "Total exclusivity errors:",
    total
)

print(
    "True S1 present in candidate union:",
    retrieval_present,
    f"({retrieval_present/total*100:.2f}%)"
)

print(
    "True S1 missing from candidate union:",
    retrieval_missing,
    f"({retrieval_missing/total*100:.2f}%)"
)

# ------------------------------------------------------------
# Channel breakdown
# ------------------------------------------------------------

print(
    "\nTrue candidate coverage:"
)

print(
    "Rare-token:",
    int(diag15["true_in_rare"].sum()),
    f"({diag15['true_in_rare'].mean()*100:.2f}%)"
)

print(
    "TF-IDF:",
    int(diag15["true_in_tfidf"].sum()),
    f"({diag15['true_in_tfidf'].mean()*100:.2f}%)"
)

# ------------------------------------------------------------
# Ranking-error subset
# ------------------------------------------------------------

ranking_errors = diag15[
    diag15["true_in_union"]
].copy()

retrieval_errors = diag15[
    ~diag15["true_in_union"]
].copy()

print(
    "\nRanking errors:",
    len(ranking_errors)
)

print(
    "Retrieval errors:",
    len(retrieval_errors)
)

# ------------------------------------------------------------
# High-score wrong selections
# ------------------------------------------------------------

print(
    "\nWrong selections with high scores:"
)

print(
    diag15[
        diag15["selected_score"] >= 0.90
    ][
        [
            "target_id",
            "true_s1",
            "selected_s1",
            "selected_score",
            "true_in_union",
            "union_candidates"
        ]
    ]
    .sort_values(
        "selected_score",
        ascending=False
    )
    .head(30)
    .to_string(
        index=False
    )
)

# ------------------------------------------------------------
# Save diagnostic
# ------------------------------------------------------------

diag_path = (
    MODEL_DIR
    /
    "exclusivity_retrieval_diagnosis.csv"
)

diag15.to_csv(
    diag_path,
    index=False
)

print(
    "\nSaved:",
    diag_path
)

print(
    "\nCELL 15 COMPLETE."
)

CELL 15 — EXCLUSIVITY ERROR DIAGNOSIS


RuntimeError: Missing objects: ['rare_candidates', 'retrieved_top100', 's1_local_to_id']

In [ ]:
# CELL 15B — EXCLUSIVITY ERROR DIAGNOSIS
#
# Uses the completed Cell 11 artifact:
#   best_all
#
# No rare-token retrieval objects are required.

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 15B — EXCLUSIVITY ERROR DIAGNOSIS")
print("=" * 100)

# ------------------------------------------------------------
# Required objects
# ------------------------------------------------------------

required = [
    "best_all",
    "VAL_S1",
    "true_by_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}"
    )

print(
    "Scored target→S1 rows:",
    f"{len(best_all):,}"
)

print(
    "Validation S1:",
    f"{len(VAL_S1):,}"
)

# ------------------------------------------------------------
# Build target -> true S1 mapping
# ------------------------------------------------------------

true_target_to_s1 = {}

for s1_id, target_ids in true_by_s1.items():

    for target_id in target_ids:

        true_target_to_s1[
            target_id
        ] = s1_id

print(
    "True target→S1 mappings:",
    f"{len(true_target_to_s1):,}"
)

# ------------------------------------------------------------
# Current threshold
# ------------------------------------------------------------

CURRENT_THRESHOLD = 0.640

print(
    "\nCurrent threshold:",
    CURRENT_THRESHOLD
)

# ------------------------------------------------------------
# For every target, inspect its top candidates
# ------------------------------------------------------------

ranked = (
    best_all
    .sort_values(
        [
            "query_id",
            "score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)

ranked["target_rank"] = (
    ranked
    .groupby("query_id")
    .cumcount()
    + 1
)

# ------------------------------------------------------------
# Top candidate per target
# ------------------------------------------------------------

top1 = ranked[
    ranked["target_rank"] == 1
].copy()

top1["true_s1_id"] = (
    top1["query_id"]
    .map(true_target_to_s1)
)

top1["is_true_top1"] = (
    top1["s1_id"]
    ==
    top1["true_s1_id"]
)

top1["is_above_threshold"] = (
    top1["score"]
    >= CURRENT_THRESHOLD
)

# ------------------------------------------------------------
# Targets with a known true match
# ------------------------------------------------------------

matched_targets = top1[
    top1["true_s1_id"].notna()
].copy()

# ------------------------------------------------------------
# Top1 errors
# ------------------------------------------------------------

wrong_top1 = matched_targets[
    ~matched_targets["is_true_top1"]
].copy()

correct_top1 = matched_targets[
    matched_targets["is_true_top1"]
].copy()

print(
    "\n" + "=" * 100
)
print(
    "TARGET TOP-1 ACCURACY"
)
print(
    "=" * 100
)

print(
    "Targets with known true S1:",
    f"{len(matched_targets):,}"
)

print(
    "Correct top-1:",
    f"{len(correct_top1):,}"
)

print(
    "Wrong top-1:",
    f"{len(wrong_top1):,}"
)

print(
    "Top-1 accuracy:",
    f"{len(correct_top1)/len(matched_targets)*100:.4f}%"
)

# ------------------------------------------------------------
# Important categories
# ------------------------------------------------------------

# True target exists but correct S1 is not candidate
true_missing = (
    matched_targets[
        ~matched_targets["true_s1_id"]
        .isin(
            ranked["s1_id"]
        )
    ]
)

# True candidate exists somewhere but loses ranking
true_present_loses = []

for row in matched_targets.itertuples(
    index=False
):

    target = row.query_id
    true_s1 = row.true_s1_id

    candidates = ranked[
        ranked["query_id"] == target
    ]

    if true_s1 in set(
        candidates["s1_id"]
    ):

        true_score = float(
            candidates.loc[
                candidates["s1_id"] == true_s1,
                "score"
            ].iloc[0]
        )

        best_score = float(
            candidates["score"].iloc[0]
        )

        if (
            best_score > true_score
            and
            row.s1_id != true_s1
        ):

            true_present_loses.append({
                "query_id": target,
                "true_s1": true_s1,
                "true_score": true_score,
                "wrong_s1": row.s1_id,
                "wrong_score": best_score,
                "score_gap":
                    best_score - true_score
            })

losing_df = pd.DataFrame(
    true_present_loses
)

print(
    "\nTrue candidate present but loses:",
    f"{len(losing_df):,}"
)

# ------------------------------------------------------------
# Wrong Top-1 predictions above threshold
# ------------------------------------------------------------

wrong_above_threshold = wrong_top1[
    wrong_top1["is_above_threshold"]
].copy()

print(
    "Wrong Top-1 above threshold:",
    f"{len(wrong_above_threshold):,}"
)

# ------------------------------------------------------------
# Score distributions
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "SCORE DISTRIBUTIONS"
)
print(
    "=" * 100
)

if len(correct_top1):

    print(
        "\nCorrect Top-1 scores:"
    )

    print(
        correct_top1["score"]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

if len(wrong_top1):

    print(
        "\nWrong Top-1 scores:"
    )

    print(
        wrong_top1["score"]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

# ------------------------------------------------------------
# Score gap analysis
# ------------------------------------------------------------

if len(losing_df):

    print(
        "\n" + "=" * 100
    )
    print(
        "TRUE-CANDIDATE LOSSES"
    )
    print(
        "=" * 100
    )

    print(
        losing_df["score_gap"]
        .describe(
            percentiles=[
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

    print(
        "\nWorst 30 ranking errors:"
    )

    print(
        losing_df
        .sort_values(
            "score_gap",
            ascending=False
        )
        .head(30)
        .to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# Threshold sensitivity specifically for Top-1
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "TOP-1 THRESHOLD COUNTS"
)
print(
    "=" * 100
)

for threshold in [
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.62,
    0.64,
    0.66,
    0.68,
    0.70,
    0.75,
    0.80
]:

    accepted = top1[
        top1["score"] >= threshold
    ]

    correct = accepted[
        accepted["is_true_top1"]
    ]

    wrong = accepted[
        ~accepted["is_true_top1"]
        &
        accepted["true_s1_id"].notna()
    ]

    print(
        f"threshold={threshold:.2f} "
        f"accepted={len(accepted):,} "
        f"correct={len(correct):,} "
        f"wrong={len(wrong):,}"
    )

# ------------------------------------------------------------
# Save diagnostics
# ------------------------------------------------------------

diag_path = (
    MODEL_DIR
    /
    "exclusivity_error_diagnosis.csv"
)

if len(wrong_top1):

    wrong_top1[
        [
            "query_id",
            "s1_id",
            "score",
            "true_s1_id",
            "target_rank"
        ]
    ].to_csv(
        diag_path,
        index=False
    )

    print(
        "\nSaved:",
        diag_path
    )

print(
    "\nCELL 15B COMPLETE."
)

CELL 15B — EXCLUSIVITY ERROR DIAGNOSIS
Scored target→S1 rows: 69,231
Validation S1: 20,000
True target→S1 mappings: 69,231

Current threshold: 0.64

TARGET TOP-1 ACCURACY
Targets with known true S1: 69,231
Correct top-1: 68,259
Wrong top-1: 972
Top-1 accuracy: 98.5960%

True candidate present but loses: 0
Wrong Top-1 above threshold: 348

SCORE DISTRIBUTIONS

Correct Top-1 scores:
count    68259.000000
mean         0.989629
std          0.070058
min          0.002771
1%           0.704682
5%           0.978675
10%          0.995895
25%          0.999223
50%          0.999617
75%          0.999639
90%          0.999644
95%          0.999652
99%          0.999659
max          0.999660
Name: score, dtype: float64

Wrong Top-1 scores:
count    972.000000
mean       0.450155
std        0.369245
min        0.000820
1%         0.003455
5%         0.013754
10%        0.022529
25%        0.076676
50%        0.353584
75%        0.850931
90%        0.976158
95%        0.992313
99%        0.998014

In [ ]:
# CELL 16 — ABOVE-THRESHOLD EXCLUSIVITY ERROR ANALYSIS

THRESHOLD = 0.64

wrong_above = top1[
    (top1["s1_id"] != top1["true_s1"]) &
    (top1["score"] >= THRESHOLD)
].copy()

print("=" * 100)
print("CELL 16 — ABOVE-THRESHOLD EXCLUSIVITY ERROR ANALYSIS")
print("=" * 100)

print("Wrong top-1 total:", len(wrong))
print("Wrong top-1 above threshold:", len(wrong_above))

print("\nScore distribution of accepted wrong matches:")
print(
    wrong_above["score"].describe(
        percentiles=[
            .01, .05, .10, .25,
            .50, .75, .90, .95, .99
        ]
    )
)

print("\nScore buckets:")
bins = [
    0.64,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    0.97,
    0.98,
    0.99,
    0.995,
    1.0
]

wrong_above["score_bucket"] = pd.cut(
    wrong_above["score"],
    bins=bins,
    include_lowest=True
)

print(
    wrong_above["score_bucket"]
    .value_counts()
    .sort_index()
)

print("\nTop 50 highest-confidence wrong matches:")
display(
    wrong_above[
        [
            "query_id",
            "s1_id",
            "true_s1",
            "score"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(50)
)

print("\nCELL 16 COMPLETE.")

KeyError: 'true_s1'

In [ ]:
# ============================================================
# CELL 16 — ABOVE-THRESHOLD EXCLUSIVITY ERROR ANALYSIS
# ============================================================

THRESHOLD = 0.64

print("=" * 100)
print("CELL 16 — ABOVE-THRESHOLD EXCLUSIVITY ERROR ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# 1. Make sure top1 exists
# ------------------------------------------------------------

print("top1 rows:", len(top1))
print("top1 columns:", list(top1.columns))

# ------------------------------------------------------------
# 2. Attach the TRUE S1 mapping
# ------------------------------------------------------------

top1_diag = top1.copy()

top1_diag["true_s1"] = (
    top1_diag["query_id"]
    .map(true_target_to_s1)
)

# Check mapping
missing_truth = top1_diag["true_s1"].isna().sum()

print("\nMissing true-S1 mappings:", missing_truth)

if missing_truth > 0:
    print("WARNING: Some targets do not have a known true S1.")

# ------------------------------------------------------------
# 3. Identify wrong top-1 predictions
# ------------------------------------------------------------

wrong_all = top1_diag[
    top1_diag["s1_id"] != top1_diag["true_s1"]
].copy()

wrong_above = wrong_all[
    wrong_all["score"] >= THRESHOLD
].copy()

print("\n" + "=" * 100)
print("ERROR COUNTS")
print("=" * 100)

print("Total targets:", len(top1_diag))
print("Wrong top-1:", len(wrong_all))
print("Wrong top-1 above threshold:", len(wrong_above))

# ------------------------------------------------------------
# 4. Score distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ACCEPTED WRONG TOP-1 SCORE DISTRIBUTION")
print("=" * 100)

print(
    wrong_above["score"].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

# ------------------------------------------------------------
# 5. Score buckets
# ------------------------------------------------------------

bins = [
    0.64,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    0.97,
    0.98,
    0.99,
    0.995,
    1.000001
]

labels = [
    "0.64–0.70",
    "0.70–0.75",
    "0.75–0.80",
    "0.80–0.85",
    "0.85–0.90",
    "0.90–0.95",
    "0.95–0.97",
    "0.97–0.98",
    "0.98–0.99",
    "0.99–0.995",
    "0.995–1.00"
]

wrong_above["score_bucket"] = pd.cut(
    wrong_above["score"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

bucket_counts = (
    wrong_above["score_bucket"]
    .value_counts()
    .reindex(labels)
    .fillna(0)
    .astype(int)
)

print("\nWrong accepted matches by score:")
print(bucket_counts)

# ------------------------------------------------------------
# 6. Percentage distribution
# ------------------------------------------------------------

print("\nPercentage of accepted wrong matches:")

print(
    (bucket_counts / len(wrong_above) * 100)
    .round(2)
    .astype(str)
    + "%"
)

# ------------------------------------------------------------
# 7. Highest-confidence wrong matches
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TOP 50 HIGHEST-CONFIDENCE WRONG MATCHES")
print("=" * 100)

cols = [
    "query_id",
    "s1_id",
    "true_s1",
    "score"
]

display(
    wrong_above[
        cols
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .head(50)
)

# ------------------------------------------------------------
# 8. Lowest-confidence accepted wrong matches
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LOWEST-CONFIDENCE ACCEPTED WRONG MATCHES")
print("=" * 100)

display(
    wrong_above[
        cols
    ]
    .sort_values(
        "score",
        ascending=True
    )
    .head(30)
)

print("\nCELL 16 COMPLETE.")

CELL 16 — ABOVE-THRESHOLD EXCLUSIVITY ERROR ANALYSIS
top1 rows: 69231
top1 columns: ['query_id', 's1_id', 'score', 'target_rank', 'true_s1_id', 'is_true_top1', 'is_above_threshold']

Missing true-S1 mappings: 0

ERROR COUNTS
Total targets: 69231
Wrong top-1: 972
Wrong top-1 above threshold: 348

ACCEPTED WRONG TOP-1 SCORE DISTRIBUTION
count    348.000000
mean       0.892004
std        0.101775
min        0.641861
1%         0.653232
5%         0.689034
10%        0.730539
25%        0.819575
50%        0.926461
75%        0.978675
90%        0.994803
95%        0.997682
99%        0.998477
max        0.999644
Name: score, dtype: float64

Wrong accepted matches by score:
score_bucket
0.64–0.70     23
0.70–0.75     25
0.75–0.80     26
0.80–0.85     30
0.85–0.90     48
0.90–0.95     44
0.95–0.97     40
0.97–0.98     30
0.98–0.99     26
0.99–0.995    26
0.995–1.00    30
Name: count, dtype: int64

Percentage of accepted wrong matches:
score_bucket
0.64–0.70      6.61%
0.70–0.75      7.18%
0

,query_id,s1_id,true_s1,score
51512,S3-797521910,S1-979690967,S1-383911575,0.999644
31209,S2-640825269,S1-491584669,S1-111270002,0.999479
53577,S3-791170196,S1-597550527,S1-330267378,0.999391
29304,S2-692390669,S1-289217668,S1-613731683,0.998502
53086,S3-564841594,S1-550886545,S1-438115387,0.998449
1535,S2-786265004,S1-360579271,S1-321361130,0.998398
53447,S3-727062069,S1-123636921,S1-437766158,0.998378
42174,S3-181916057,S1-387795523,S1-797677711,0.998322
35349,S3-712542961,S1-617334573,S1-88664201,0.998206
34300,S3-241720778,S1-720777938,S1-34267226,0.998053



LOWEST-CONFIDENCE ACCEPTED WRONG MATCHES


,query_id,s1_id,true_s1,score
35562,S3-802571785,S1-455733432,S1-265307526,0.641861
17054,S2-586715886,S1-927513065,S1-474532322,0.650145
4843,S2-481832215,S1-989877052,S1-248566799,0.652927
61978,S3-987443577,S1-76146608,S1-422650147,0.653202
40957,S3-53315476,S1-490128817,S1-350567233,0.653266
26195,S2-184531575,S1-52301491,S1-178615058,0.653542
36852,S3-481495887,S1-470837671,S1-440755143,0.654122
52406,S3-27605512,S1-285919848,S1-575158044,0.664077
33592,S3-43710305,S1-696129146,S1-491138472,0.666234
32955,S2-75132969,S1-792435938,S1-126022071,0.668111



CELL 16 COMPLETE.


In [ ]:
# ============================================================
# CELL 17 — TRUE VS WRONG CANDIDATE FEATURE COMPARISON
# ============================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 17 — TRUE VS WRONG CANDIDATE FEATURE COMPARISON")
print("=" * 100)

THRESHOLD = 0.64

# ------------------------------------------------------------
# 1. Build ranking table
# ------------------------------------------------------------

ranked17 = (
    best_all
    .sort_values(
        ["query_id", "score"],
        ascending=[True, False]
    )
    .copy()
)

ranked17["target_rank"] = (
    ranked17
    .groupby("query_id")
    .cumcount()
    + 1
)

# ------------------------------------------------------------
# 2. True S1 mapping
# ------------------------------------------------------------

ranked17["true_s1_id"] = (
    ranked17["query_id"]
    .map(true_target_to_s1)
)

# ------------------------------------------------------------
# 3. Top-1 rows
# ------------------------------------------------------------

top17 = ranked17[
    ranked17["target_rank"] == 1
].copy()

wrong17 = top17[
    top17["true_s1_id"].notna()
    &
    (top17["s1_id"] != top17["true_s1_id"])
    &
    (top17["score"] >= THRESHOLD)
].copy()

print(
    "Accepted wrong top-1:",
    len(wrong17)
)

# ------------------------------------------------------------
# 4. Retrieve the TRUE candidate row
# ------------------------------------------------------------

true17 = ranked17[
    ranked17["s1_id"] == ranked17["true_s1_id"]
].copy()

true17 = (
    true17
    .sort_values(
        ["query_id", "score"],
        ascending=[True, False]
    )
    .groupby("query_id")
    .first()
    .reset_index()
)

# ------------------------------------------------------------
# 5. Keep only targets with accepted wrong top-1
# ------------------------------------------------------------

true17 = true17[
    true17["query_id"].isin(
        wrong17["query_id"]
    )
].copy()

# ------------------------------------------------------------
# 6. Rename columns
# ------------------------------------------------------------

true17 = true17.rename(
    columns={
        "s1_id": "true_s1",
        "score": "true_score"
    }
)

wrong17 = wrong17.rename(
    columns={
        "s1_id": "wrong_s1",
        "score": "wrong_score"
    }
)

# ------------------------------------------------------------
# 7. Merge
# ------------------------------------------------------------

compare17 = wrong17[
    [
        "query_id",
        "wrong_s1",
        "wrong_score"
    ]
].merge(
    true17[
        [
            "query_id",
            "true_s1",
            "true_score"
        ]
    ],
    on="query_id",
    how="inner"
)

compare17["score_gap"] = (
    compare17["wrong_score"]
    -
    compare17["true_score"]
)

print(
    "\nScore-gap statistics:"
)

print(
    compare17["score_gap"].describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

# ------------------------------------------------------------
# 8. Pull feature rows
# ------------------------------------------------------------

feature_cols17 = [
    c
    for c in rank_feature_cols
    if c in ranked17.columns
]

# If best_all does not contain features,
# recover them from df_new.

if not set(feature_cols17).issubset(
    set(ranked17.columns)
):

    feature_cols17 = [
        c
        for c in rank_feature_cols
        if c in df_new.columns
    ]

    wrong_features = df_new[
        [
            "query_id",
            "s1_id"
        ]
        +
        feature_cols17
    ].copy()

else:

    wrong_features = ranked17[
        [
            "query_id",
            "s1_id"
        ]
        +
        feature_cols17
    ].copy()

# ------------------------------------------------------------
# 9. Extract wrong and true feature vectors
# ------------------------------------------------------------

wrong_features = wrong_features.rename(
    columns={
        "s1_id": "wrong_s1"
    }
)

true_features = wrong_features.rename(
    columns={
        "wrong_s1": "true_s1"
    }
)

# Rebuild separately from df_new if needed
base_features = df_new[
    [
        "query_id",
        "s1_id"
    ]
    +
    feature_cols17
].copy()

wrong_feature_rows = (
    compare17[
        [
            "query_id",
            "wrong_s1"
        ]
    ]
    .merge(
        base_features,
        left_on=[
            "query_id",
            "wrong_s1"
        ],
        right_on=[
            "query_id",
            "s1_id"
        ],
        how="left"
    )
)

true_feature_rows = (
    compare17[
        [
            "query_id",
            "true_s1"
        ]
    ]
    .merge(
        base_features,
        left_on=[
            "query_id",
            "true_s1"
        ],
        right_on=[
            "query_id",
            "s1_id"
        ],
        how="left"
    )
)

# ------------------------------------------------------------
# 10. Compare feature values
# ------------------------------------------------------------

comparison_rows = []

for _, c in compare17.iterrows():

    q = c["query_id"]

    wf = wrong_feature_rows[
        wrong_feature_rows["query_id"] == q
    ]

    tf = true_feature_rows[
        true_feature_rows["query_id"] == q
    ]

    if len(wf) == 0 or len(tf) == 0:
        continue

    wf = wf.iloc[0]
    tf = tf.iloc[0]

    row = {
        "query_id": q,
        "wrong_s1": c["wrong_s1"],
        "true_s1": c["true_s1"],
        "wrong_score": c["wrong_score"],
        "true_score": c["true_score"],
        "score_gap": c["score_gap"]
    }

    for f in feature_cols17:

        row[
            f"{f}_wrong"
        ] = wf[f]

        row[
            f"{f}_true"
        ] = tf[f]

        row[
            f"{f}_delta"
        ] = wf[f] - tf[f]

    comparison_rows.append(row)

comparison17 = pd.DataFrame(
    comparison_rows
)

print(
    "\nComparison rows:",
    len(comparison17)
)

# ------------------------------------------------------------
# 11. Which features favor WRONG candidates?
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FEATURE DISCRIMINATION")
print("=" * 100)

feature_summary = []

for f in feature_cols17:

    delta_col = f"{f}_delta"

    if delta_col not in comparison17.columns:
        continue

    delta = comparison17[
        delta_col
    ].dropna()

    if len(delta) == 0:
        continue

    feature_summary.append({
        "feature": f,
        "wrong_minus_true_mean":
            delta.mean(),
        "wrong_minus_true_median":
            delta.median(),
        "wrong_higher_fraction":
            (delta > 0).mean(),
        "true_higher_fraction":
            (delta < 0).mean()
    })

feature_summary17 = (
    pd.DataFrame(feature_summary)
    .sort_values(
        "wrong_minus_true_mean",
        ascending=False
    )
)

print(
    "\nFeatures favoring WRONG candidates:"
)

display(
    feature_summary17.head(20)
)

print(
    "\nFeatures favoring TRUE candidates:"
)

display(
    feature_summary17.tail(20)
)

# ------------------------------------------------------------
# 12. Hardest high-confidence cases
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("HIGHEST-CONFIDENCE WRONG CASES")
print("=" * 100)

display(
    comparison17[
        [
            "query_id",
            "wrong_s1",
            "true_s1",
            "wrong_score",
            "true_score",
            "score_gap"
        ]
    ]
    .sort_values(
        "wrong_score",
        ascending=False
    )
    .head(30)
)

# ------------------------------------------------------------
# 13. Save
# ------------------------------------------------------------

comparison_path = (
    MODEL_DIR /
    "cell17_true_vs_wrong_features.csv"
)

summary_path = (
    MODEL_DIR /
    "cell17_feature_discrimination.csv"
)

comparison17.to_csv(
    comparison_path,
    index=False
)

feature_summary17.to_csv(
    summary_path,
    index=False
)

print(
    "\nSaved:"
)

print(comparison_path)
print(summary_path)

print(
    "\nCELL 17 COMPLETE."
)

CELL 17 — TRUE VS WRONG CANDIDATE FEATURE COMPARISON
Accepted wrong top-1: 348

Score-gap statistics:
count    0.0
mean     NaN
std      NaN
min      NaN
1%       NaN
5%       NaN
10%      NaN
25%      NaN
50%      NaN
75%      NaN
90%      NaN
95%      NaN
99%      NaN
max      NaN
Name: score_gap, dtype: float64


NameError: name 'rank_feature_cols' is not defined

In [ ]:
# ============================================================
# CELL 17A — TRUE VS WRONG SCORE COMPARISON
# ============================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 17A — TRUE VS WRONG SCORE COMPARISON")
print("=" * 100)

THRESHOLD = 0.64

# ------------------------------------------------------------
# 1. Use the existing Cell 15B top1 table
# ------------------------------------------------------------

diag17 = top1.copy()

print("Top-1 rows:", len(diag17))

print("\nColumns:")
print(diag17.columns.tolist())

# ------------------------------------------------------------
# 2. Accepted wrong predictions
# ------------------------------------------------------------

wrong17 = diag17[
    (~diag17["is_true_top1"]) &
    (diag17["is_above_threshold"])
].copy()

print(
    "\nAccepted wrong top-1:",
    len(wrong17)
)

# ------------------------------------------------------------
# 3. Find TRUE candidate inside best_all
# ------------------------------------------------------------

# Use the truth mapping already stored in top1.
# Build a unique query -> true S1 table.

truth17 = (
    diag17[
        [
            "query_id",
            "true_s1_id"
        ]
    ]
    .drop_duplicates()
)

print(
    "Unique truth mappings:",
    len(truth17)
)

# ------------------------------------------------------------
# 4. Get TRUE candidate scores from best_all
# ------------------------------------------------------------

true_scores17 = (
    best_all[
        [
            "query_id",
            "s1_id",
            "score"
        ]
    ]
    .merge(
        truth17,
        left_on=[
            "query_id",
            "s1_id"
        ],
        right_on=[
            "query_id",
            "true_s1_id"
        ],
        how="inner"
    )
)

print(
    "\nTrue candidate rows recovered:",
    len(true_scores17)
)

# ------------------------------------------------------------
# 5. Keep one true candidate per query
# ------------------------------------------------------------

true_scores17 = (
    true_scores17
    .sort_values(
        ["query_id", "score"],
        ascending=[True, False]
    )
    .drop_duplicates(
        "query_id"
    )
)

true_scores17 = true_scores17[
    [
        "query_id",
        "true_s1_id",
        "score"
    ]
].rename(
    columns={
        "score": "true_score"
    }
)

# ------------------------------------------------------------
# 6. Merge wrong + true
# ------------------------------------------------------------

compare17 = wrong17[
    [
        "query_id",
        "s1_id",
        "score"
    ]
].rename(
    columns={
        "s1_id": "wrong_s1",
        "score": "wrong_score"
    }
).merge(
    true_scores17,
    on="query_id",
    how="left"
)

compare17["score_gap"] = (
    compare17["wrong_score"]
    -
    compare17["true_score"]
)

# ------------------------------------------------------------
# 7. Diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TRUE CANDIDATE RECOVERY")
print("=" * 100)

print(
    "Accepted wrong cases:",
    len(compare17)
)

print(
    "True score recovered:",
    compare17["true_score"].notna().sum()
)

print(
    "True score missing:",
    compare17["true_score"].isna().sum()
)

print("\n" + "=" * 100)
print("SCORE GAP — WRONG MINUS TRUE")
print("=" * 100)

print(
    compare17["score_gap"].describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

# ------------------------------------------------------------
# 8. Score-gap buckets
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SCORE GAP BUCKETS")
print("=" * 100)

gap_bins = [
    -1.0,
    0.01,
    0.03,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
    1.0
]

gap_labels = [
    "≤ 0.01",
    "0.01–0.03",
    "0.03–0.05",
    "0.05–0.10",
    "0.10–0.20",
    "0.20–0.30",
    "0.30–0.50",
    "> 0.50"
]

compare17["gap_bucket"] = pd.cut(
    compare17["score_gap"],
    bins=gap_bins,
    labels=gap_labels,
    include_lowest=True
)

print(
    compare17["gap_bucket"]
    .value_counts()
    .reindex(gap_labels)
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 9. Highest-confidence wrong matches
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("HIGHEST-CONFIDENCE WRONG MATCHES")
print("=" * 100)

display(
    compare17[
        [
            "query_id",
            "wrong_s1",
            "true_s1_id",
            "wrong_score",
            "true_score",
            "score_gap"
        ]
    ]
    .sort_values(
        "wrong_score",
        ascending=False
    )
    .head(50)
)

# ------------------------------------------------------------
# 10. Closest ranking battles
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CLOSEST WRONG VS TRUE RANKING BATTLES")
print("=" * 100)

display(
    compare17[
        [
            "query_id",
            "wrong_s1",
            "true_s1_id",
            "wrong_score",
            "true_score",
            "score_gap"
        ]
    ]
    .sort_values(
        "score_gap",
        ascending=True
    )
    .head(50)
)

print("\nCELL 17A COMPLETE.")

CELL 17A — TRUE VS WRONG SCORE COMPARISON
Top-1 rows: 69231

Columns:
['query_id', 's1_id', 'score', 'target_rank', 'true_s1_id', 'is_true_top1', 'is_above_threshold']

Accepted wrong top-1: 348
Unique truth mappings: 69231

True candidate rows recovered: 68259

TRUE CANDIDATE RECOVERY
Accepted wrong cases: 348
True score recovered: 0
True score missing: 348

SCORE GAP — WRONG MINUS TRUE
count    0.0
mean     NaN
std      NaN
min      NaN
1%       NaN
5%       NaN
10%      NaN
25%      NaN
50%      NaN
75%      NaN
90%      NaN
95%      NaN
99%      NaN
max      NaN
Name: score_gap, dtype: float64

SCORE GAP BUCKETS
gap_bucket
≤ 0.01       0
0.01–0.03    0
0.03–0.05    0
0.05–0.10    0
0.10–0.20    0
0.20–0.30    0
0.30–0.50    0
> 0.50       0
Name: count, dtype: int64

HIGHEST-CONFIDENCE WRONG MATCHES


,query_id,wrong_s1,true_s1_id,wrong_score,true_score,score_gap
303,S3-797521910,S1-979690967,NaN,0.999644,NaN,NaN
110,S2-640825269,S1-491584669,NaN,0.999479,NaN,NaN
302,S3-791170196,S1-597550527,NaN,0.999391,NaN,NaN
122,S2-692390669,S1-289217668,NaN,0.998502,NaN,NaN
264,S3-564841594,S1-550886545,NaN,0.998449,NaN,NaN
146,S2-786265004,S1-360579271,NaN,0.998398,NaN,NaN
289,S3-727062069,S1-123636921,NaN,0.998378,NaN,NaN
213,S3-181916057,S1-387795523,NaN,0.998322,NaN,NaN
286,S3-712542961,S1-617334573,NaN,0.998206,NaN,NaN
222,S3-241720778,S1-720777938,NaN,0.998053,NaN,NaN



CLOSEST WRONG VS TRUE RANKING BATTLES


,query_id,wrong_s1,true_s1_id,wrong_score,true_score,score_gap
0,S2-10070703,S1-910696326,NaN,0.900402,NaN,NaN
1,S2-101179790,S1-523407684,NaN,0.988109,NaN,NaN
2,S2-101734060,S1-747137332,NaN,0.762902,NaN,NaN
3,S2-10559588,S1-926491547,NaN,0.735701,NaN,NaN
4,S2-106046520,S1-136481301,NaN,0.983042,NaN,NaN
5,S2-112471943,S1-764129222,NaN,0.925387,NaN,NaN
6,S2-126598464,S1-681308944,NaN,0.740146,NaN,NaN
7,S2-126973923,S1-896105584,NaN,0.993123,NaN,NaN
8,S2-127106558,S1-877031369,NaN,0.896166,NaN,NaN
9,S2-127911581,S1-222419925,NaN,0.883345,NaN,NaN



CELL 17A COMPLETE.


In [ ]:
# ============================================================
# CELL 17B — TRUE VS WRONG CANDIDATE FEATURE/ SCORE COMPARISON
# ============================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 17B — TRUE VS WRONG CANDIDATE SCORE COMPARISON")
print("=" * 100)

THRESHOLD = 0.64

# ------------------------------------------------------------
# 1. Sanity check
# ------------------------------------------------------------

required = [
    "ranked",
    "top1",
    "target_to_s1"
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}. "
        f"Run Cells 11 and 15B first."
    )

print("ranked rows:", f"{len(ranked):,}")
print("top1 rows:", f"{len(top1):,}")

# ------------------------------------------------------------
# 2. Accepted wrong top-1
# ------------------------------------------------------------

wrong_top1_17 = top1[
    (~top1["is_true_top1"]) &
    (top1["is_above_threshold"])
].copy()

print(
    "\nAccepted wrong top-1:",
    len(wrong_top1_17)
)

# ------------------------------------------------------------
# 3. Recover TRUE candidates from FULL ranked table
# ------------------------------------------------------------

true_candidates_17 = ranked[
    ranked["query_id"].isin(
        wrong_top1_17["query_id"]
    )
].copy()

true_candidates_17["true_s1_id"] = (
    true_candidates_17["query_id"]
    .map(target_to_s1)
)

true_candidates_17 = true_candidates_17[
    true_candidates_17["s1_id"]
    ==
    true_candidates_17["true_s1_id"]
].copy()

print(
    "True candidates recovered:",
    len(true_candidates_17)
)

# ------------------------------------------------------------
# 4. Extract TRUE scores
# ------------------------------------------------------------

true_scores_17 = (
    true_candidates_17[
        [
            "query_id",
            "s1_id",
            "score"
        ]
    ]
    .rename(
        columns={
            "s1_id": "true_s1_id",
            "score": "true_score"
        }
    )
)

# One true candidate per query
true_scores_17 = (
    true_scores_17
    .drop_duplicates("query_id")
)

# ------------------------------------------------------------
# 5. Merge WRONG + TRUE
# ------------------------------------------------------------

compare17 = (
    wrong_top1_17[
        [
            "query_id",
            "s1_id",
            "score"
        ]
    ]
    .rename(
        columns={
            "s1_id": "wrong_s1_id",
            "score": "wrong_score"
        }
    )
    .merge(
        true_scores_17,
        on="query_id",
        how="left"
    )
)

compare17["score_gap"] = (
    compare17["wrong_score"]
    -
    compare17["true_score"]
)

# ------------------------------------------------------------
# 6. Recovery sanity check
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TRUE CANDIDATE RECOVERY")
print("=" * 100)

print(
    "Accepted wrong cases:",
    len(compare17)
)

print(
    "True score recovered:",
    compare17["true_score"].notna().sum()
)

print(
    "True score missing:",
    compare17["true_score"].isna().sum()
)

if compare17["true_score"].isna().sum() > 0:
    print(
        "\nWARNING: Some true candidates are absent "
        "from ranked."
    )
    print(
        "This would indicate a candidate-generation problem."
    )

# ------------------------------------------------------------
# 7. SCORE GAP
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SCORE GAP — WRONG MINUS TRUE")
print("=" * 100)

print(
    compare17["score_gap"]
    .describe(
        percentiles=[
            .01,
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    )
)

# ------------------------------------------------------------
# 8. Score gap buckets
# ------------------------------------------------------------

gap_bins = [
    -1.0,
    0.005,
    0.01,
    0.02,
    0.03,
    0.05,
    0.10,
    0.20,
    0.50,
    1.0
]

gap_labels = [
    "<=0.005",
    "0.005-0.01",
    "0.01-0.02",
    "0.02-0.03",
    "0.03-0.05",
    "0.05-0.10",
    "0.10-0.20",
    "0.20-0.50",
    ">0.50"
]

compare17["gap_bucket"] = pd.cut(
    compare17["score_gap"],
    bins=gap_bins,
    labels=gap_labels,
    include_lowest=True
)

print("\n" + "=" * 100)
print("SCORE GAP BUCKETS")
print("=" * 100)

print(
    compare17["gap_bucket"]
    .value_counts()
    .reindex(gap_labels)
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 9. Closest ranking battles
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CLOSEST WRONG VS TRUE RANKING BATTLES")
print("=" * 100)

display(
    compare17[
        [
            "query_id",
            "wrong_s1_id",
            "true_s1_id",
            "wrong_score",
            "true_score",
            "score_gap"
        ]
    ]
    .sort_values(
        "score_gap",
        ascending=True
    )
    .head(50)
)

# ------------------------------------------------------------
# 10. Highest-confidence wrong matches
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("HIGHEST-CONFIDENCE WRONG MATCHES")
print("=" * 100)

display(
    compare17[
        [
            "query_id",
            "wrong_s1_id",
            "true_s1_id",
            "wrong_score",
            "true_score",
            "score_gap"
        ]
    ]
    .sort_values(
        "wrong_score",
        ascending=False
    )
    .head(50)
)

# ------------------------------------------------------------
# 11. Cases where wrong barely beats true
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("NEAR-TIE CASES")
print("=" * 100)

near_ties = compare17[
    compare17["score_gap"] <= 0.03
].copy()

print(
    "Wrong accepted cases with gap <= 0.03:",
    len(near_ties)
)

display(
    near_ties[
        [
            "query_id",
            "wrong_s1_id",
            "true_s1_id",
            "wrong_score",
            "true_score",
            "score_gap"
        ]
    ]
    .sort_values(
        "score_gap"
    )
    .head(50)
)

print("\nCELL 17B COMPLETE.")

CELL 17B — TRUE VS WRONG CANDIDATE SCORE COMPARISON
ranked rows: 69,231
top1 rows: 69,231

Accepted wrong top-1: 348
True candidates recovered: 0

TRUE CANDIDATE RECOVERY
Accepted wrong cases: 348
True score recovered: 0
True score missing: 348

This would indicate a candidate-generation problem.

SCORE GAP — WRONG MINUS TRUE
count    0.0
mean     NaN
std      NaN
min      NaN
1%       NaN
5%       NaN
10%      NaN
25%      NaN
50%      NaN
75%      NaN
90%      NaN
95%      NaN
99%      NaN
max      NaN
Name: score_gap, dtype: float64

SCORE GAP BUCKETS
gap_bucket
<=0.005       0
0.005-0.01    0
0.01-0.02     0
0.02-0.03     0
0.03-0.05     0
0.05-0.10     0
0.10-0.20     0
0.20-0.50     0
>0.50         0
Name: count, dtype: int64

CLOSEST WRONG VS TRUE RANKING BATTLES


,query_id,wrong_s1_id,true_s1_id,wrong_score,true_score,score_gap
0,S2-10070703,S1-910696326,NaN,0.900402,NaN,NaN
1,S2-101179790,S1-523407684,NaN,0.988109,NaN,NaN
2,S2-101734060,S1-747137332,NaN,0.762902,NaN,NaN
3,S2-10559588,S1-926491547,NaN,0.735701,NaN,NaN
4,S2-106046520,S1-136481301,NaN,0.983042,NaN,NaN
5,S2-112471943,S1-764129222,NaN,0.925387,NaN,NaN
6,S2-126598464,S1-681308944,NaN,0.740146,NaN,NaN
7,S2-126973923,S1-896105584,NaN,0.993123,NaN,NaN
8,S2-127106558,S1-877031369,NaN,0.896166,NaN,NaN
9,S2-127911581,S1-222419925,NaN,0.883345,NaN,NaN



HIGHEST-CONFIDENCE WRONG MATCHES


,query_id,wrong_s1_id,true_s1_id,wrong_score,true_score,score_gap
303,S3-797521910,S1-979690967,NaN,0.999644,NaN,NaN
110,S2-640825269,S1-491584669,NaN,0.999479,NaN,NaN
302,S3-791170196,S1-597550527,NaN,0.999391,NaN,NaN
122,S2-692390669,S1-289217668,NaN,0.998502,NaN,NaN
264,S3-564841594,S1-550886545,NaN,0.998449,NaN,NaN
146,S2-786265004,S1-360579271,NaN,0.998398,NaN,NaN
289,S3-727062069,S1-123636921,NaN,0.998378,NaN,NaN
213,S3-181916057,S1-387795523,NaN,0.998322,NaN,NaN
286,S3-712542961,S1-617334573,NaN,0.998206,NaN,NaN
222,S3-241720778,S1-720777938,NaN,0.998053,NaN,NaN



NEAR-TIE CASES
Wrong accepted cases with gap <= 0.03: 0


,query_id,wrong_s1_id,true_s1_id,wrong_score,true_score,score_gap



CELL 17B COMPLETE.


In [ ]:
# ============================================================
# CELL 18 — LOCATE PRE-TOP1 SCORING CHECKPOINTS
# ============================================================

from pathlib import Path
import os

print("=" * 100)
print("CELL 18 — LOCATE PRE-TOP1 SCORING CHECKPOINTS")
print("=" * 100)

MODEL_DIR = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

print("\nMODEL DIRECTORY:")
print(MODEL_DIR)

print("\nParquet files:")

parquet_files = sorted(
    MODEL_DIR.rglob("*.parquet")
)

for p in parquet_files:

    try:
        size_mb = p.stat().st_size / (1024 ** 2)

        print(
            f"{p.name:<70} "
            f"{size_mb:>10.2f} MB"
        )

    except Exception:
        print(p)

print(
    "\nTotal parquet files:",
    len(parquet_files)
)

print("\nFiles containing likely candidate/scoring keywords:")

keywords = [
    "score",
    "candidate",
    "rank",
    "edge",
    "validation",
    "target",
    "full"
]

for p in parquet_files:

    name = p.name.lower()

    if any(
        k in name
        for k in keywords
    ):

        print(p)

print("\nCELL 18 COMPLETE.")


CELL 18 — LOCATE PRE-TOP1 SCORING CHECKPOINTS

MODEL DIRECTORY:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS

Parquet files:
VAL_S1.parquet                                                               3.75 MB
best_all_scored_validation_edges.parquet                                     1.19 MB
blocked_tfidf_rare_rerank_candidates_2k.parquet                              2.78 MB
blocked_tfidf_rare_rerank_features_2k.parquet                               22.47 MB
part_000.parquet                                                             0.04 MB
part_001.parquet                                                             0.04 MB
part_002.parquet                                                             0.04 MB
part_003.parquet                                                             0.04 MB
part_004.parquet                                                             0.04 MB
part_005.parquet                                                             0.04 MB
part_006.p

In [ ]:
# ============================================================
# CELL 19 — INSPECT PRE-TOP1 SCORED VALIDATION EDGES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("CELL 19 — INSPECT PRE-TOP1 SCORED VALIDATION EDGES")
print("=" * 100)

MODEL_DIR = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

CANDIDATE_FILES = [
    MODEL_DIR / "FINAL_SAVED_ARTIFACTS" / "best_all_scored_validation_edges.parquet",
    MODEL_DIR / "best_all_scored_validation_edges.parquet",
    MODEL_DIR / "matcher_candidates_2k_union.parquet",
    MODEL_DIR / "rapidfuzz_k10_matcher_ranked_2k.parquet",
]

for path in CANDIDATE_FILES:

    print("\n" + "-" * 100)
    print("FILE:", path)
    print("EXISTS:", path.exists())

    if not path.exists():
        continue

    try:

        df = pd.read_parquet(path)

        print("ROWS:", f"{len(df):,}")
        print("COLUMNS:")
        print(list(df.columns))

        print("\nDTYPES:")
        print(df.dtypes)

        if "query_id" in df.columns:
            print(
                "\nUNIQUE query_id:",
                f"{df['query_id'].nunique():,}"
            )

        if "s1_id" in df.columns:
            print(
                "UNIQUE s1_id:",
                f"{df['s1_id'].nunique():,}"
            )

        if "score" in df.columns:

            print("\nSCORE:")
            print(
                df["score"]
                .describe(
                    percentiles=[
                        .01,
                        .05,
                        .10,
                        .25,
                        .50,
                        .75,
                        .90,
                        .95,
                        .99
                    ]
                )
            )

        if "query_id" in df.columns:

            counts = (
                df.groupby("query_id")
                .size()
            )

            print("\nCANDIDATES PER QUERY:")
            print(
                counts.describe(
                    percentiles=[
                        .01,
                        .05,
                        .10,
                        .25,
                        .50,
                        .75,
                        .90,
                        .95,
                        .99
                    ]
                )
            )

            print(
                "\nQueries with >1 candidate:",
                int((counts > 1).sum())
            )

            print(
                "Queries with exactly 1:",
                int((counts == 1).sum())
            )

            print(
                "Maximum candidates/query:",
                int(counts.max())
            )

        if "target_rank" in df.columns:

            print("\nTARGET RANK DISTRIBUTION:")
            print(
                df["target_rank"]
                .value_counts()
                .sort_index()
                .head(20)
            )

        print("\nFIRST 10 ROWS:")
        display(df.head(10))

    except Exception as e:

        print("ERROR:", repr(e))

print("\n" + "=" * 100)
print("CELL 19 COMPLETE")
print("=" * 100)

CELL 19 — INSPECT PRE-TOP1 SCORED VALIDATION EDGES

----------------------------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/best_all_scored_validation_edges.parquet
EXISTS: True
ROWS: 69,231
COLUMNS:
['query_id', 's1_id', 'score']

DTYPES:
query_id     object
s1_id        object
score       float64
dtype: object

UNIQUE query_id: 69,231
UNIQUE s1_id: 18,893

SCORE:
count    69231.000000
mean         0.982055
std          0.103828
min          0.000820
1%           0.274874
5%           0.971194
10%          0.994262
25%          0.999185
50%          0.999617
75%          0.999639
90%          0.999644
95%          0.999652
99%          0.999659
max          0.999660
Name: score, dtype: float64

CANDIDATES PER QUERY:
count    69231.0
mean         1.0
std          0.0
min          1.0
1%           1.0
5%           1.0
10%          1.0
25%          1.0
50%          1.0
75%

,query_id,s1_id,score
0,S2-100060313,S1-386430480,0.999639
1,S2-10137351,S1-493409214,0.989359
2,S2-101889325,S1-753163391,0.999639
3,S2-10281348,S1-453771362,0.999177
4,S2-103002157,S1-762596987,0.999458
5,S2-103045215,S1-49113290,0.999614
6,S2-103118973,S1-247672992,0.999637
7,S2-10382093,S1-986769176,0.999639
8,S2-104603429,S1-387213726,0.998895
9,S2-105013721,S1-725150587,0.999616



----------------------------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/best_all_scored_validation_edges.parquet
EXISTS: False

----------------------------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/matcher_candidates_2k_union.parquet
EXISTS: True
ROWS: 213,390
COLUMNS:
['source1_entity_id', 'candidate_entity_id']

DTYPES:
source1_entity_id      object
candidate_entity_id    object
dtype: object

FIRST 10 ROWS:


,source1_entity_id,candidate_entity_id
0,S1-53356671,S3-137378002
1,S1-53356671,S2-797261132
2,S1-53356671,S3-517010522
3,S1-53356671,S3-100952796
4,S1-53356671,S2-752414078
5,S1-53356671,S2-186664829
6,S1-53356671,S2-993534926
7,S1-53356671,S2-464084394
8,S1-53356671,S2-417659972
9,S1-53356671,S3-769703695



----------------------------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/rapidfuzz_k10_matcher_ranked_2k.parquet
EXISTS: True
ROWS: 27,557
COLUMNS:
['query_id', 's1_id', 'label', 'name_ratio', 'addr_ratio', 'name_partial', 'addr_partial', 'name_token', 'addr_token', 'name_length_ratio', 'addr_length_ratio', 'number_overlap', 'exact_name', 'exact_address', 'country_match', 'combined_similarity', 'channel_count', 'has_name_channel', 'has_address_channel', 'has_number_channel', 'has_exact_channel', 'name_addr_product', 'name_partial_addr_partial_product', 'name_number_product', 'addr_number_product', 'min_name_addr', 'max_name_addr', 'name_addr_gap', 'partial_gap', 'strong_name_weak_address', 'strong_name_no_number', 'strong_address', 'strong_name', 'address_supported', 'multi_channel', 'balanced_similarity', 'partial_balanced_similarity', 'name_rank', 'addr_rank', 'name_partial_rank', 'addr_par

,query_id,s1_id,label,name_ratio,addr_ratio,name_partial,addr_partial,name_token,addr_token,name_length_ratio,...,addr_ge_60_count,combined_ge_60_count,is_best_name,is_best_addr,is_best_combined,name_addr_rank_agree,name_combined_rank_agree,relative_name_strength,relative_addr_strength,relative_combined_strength
0,S3-749051080,S1-992639754,1,0.896552,0.509804,1.000000,0.604651,0.896552,0.509804,0.812500,...,1,1,1,0,1,0,1,0.999999,0.849672,0.999999
1,S3-749051080,S1-991697676,0,0.689655,0.290909,0.761905,0.333333,0.689655,0.290909,0.812500,...,1,1,0,0,0,0,0,0.769230,0.484848,0.687798
2,S3-749051080,S1-463135997,0,0.545455,0.315789,0.700000,0.333333,0.545455,0.315789,0.650000,...,1,1,0,0,0,0,0,0.608391,0.526315,0.612918
3,S3-749051080,S1-558509404,0,0.580645,0.310345,0.695652,0.340426,0.580645,0.310345,0.722222,...,1,1,0,0,0,0,0,0.647642,0.517241,0.631696
4,S3-749051080,S1-487182924,0,0.500000,0.271186,0.500000,0.333333,0.500000,0.271186,0.866667,...,1,1,0,0,0,0,0,0.557692,0.451977,0.547133
5,S3-749051080,S1-951414471,0,0.484848,0.280702,0.521739,0.324324,0.484848,0.280702,0.650000,...,1,1,0,0,0,0,0,0.540792,0.467835,0.544816
6,S3-749051080,S1-610162639,0,0.216216,0.571429,0.320000,0.652174,0.216216,0.571429,0.541667,...,1,1,0,0,0,0,0,0.241164,0.952379,0.601869
7,S3-749051080,S1-765391717,0,0.400000,0.464286,0.476190,0.465116,0.400000,0.464286,0.590909,...,1,1,0,0,0,0,0,0.446153,0.773808,0.636635
8,S3-749051080,S1-485485559,0,0.258065,0.461538,0.315789,0.533333,0.258065,0.461538,0.722222,...,1,1,0,0,0,0,0,0.287841,0.769229,0.541025
9,S3-749051080,S1-765742635,0,0.270270,0.600000,0.384615,0.700000,0.270270,0.600000,0.541667,...,1,1,0,1,0,0,0,0.301455,0.999998,0.660419



CELL 19 COMPLETE


In [ ]:
# ============================================================
# CELL 20 — MAP ALL PARQUET SHARDS TO THEIR DIRECTORIES
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 100)
print("CELL 20 — MAP ALL PARQUET SHARDS")
print("=" * 100)

MODEL_DIR = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

all_parquets = sorted(
    MODEL_DIR.rglob("*.parquet")
)

print(
    "\nTotal parquet files:",
    len(all_parquets)
)

print("\n" + "-" * 100)
print("ALL SHARD FILES")
print("-" * 100)

for p in all_parquets:

    if p.name.startswith("part_"):

        try:
            size_mb = p.stat().st_size / (1024 ** 2)

            print(
                f"{str(p):<120} "
                f"{size_mb:>8.3f} MB"
            )

        except Exception:

            print(str(p))


# ============================================================
# GROUP SHARDS BY PARENT DIRECTORY
# ============================================================

print("\n" + "=" * 100)
print("SHARD DIRECTORIES")
print("=" * 100)

shard_groups = {}

for p in all_parquets:

    if p.name.startswith("part_"):

        parent = str(p.parent)

        shard_groups.setdefault(
            parent,
            []
        ).append(p)


for parent, files in shard_groups.items():

    print("\n" + "-" * 100)

    print(
        "DIRECTORY:",
        parent
    )

    print(
        "SHARDS:",
        len(files)
    )

    print(
        "TOTAL SIZE MB:",
        round(
            sum(
                f.stat().st_size
                for f in files
            ) / (1024 ** 2),
            3
        )
    )

    # Inspect first shard
    first = files[0]

    try:

        df = pd.read_parquet(first)

        print(
            "FIRST SHARD:",
            first.name
        )

        print(
            "ROWS:",
            len(df)
        )

        print(
            "COLUMNS:",
            list(df.columns)
        )

        if "query_id" in df.columns:

            print(
                "UNIQUE query_id:",
                df["query_id"].nunique()
            )

        if "s1_id" in df.columns:

            print(
                "UNIQUE s1_id:",
                df["s1_id"].nunique()
            )

        if "score" in df.columns:

            print(
                "HAS SCORE: YES"
            )

        else:

            print(
                "HAS SCORE: NO"
            )

        if "label" in df.columns:

            print(
                "HAS LABEL: YES"
            )

        else:

            print(
                "HAS LABEL: NO"
            )

    except Exception as e:

        print(
            "ERROR READING FIRST SHARD:",
            repr(e)
        )


print("\n" + "=" * 100)
print("CELL 20 COMPLETE")
print("=" * 100)

CELL 20 — MAP ALL PARQUET SHARDS

Total parquet files: 84

----------------------------------------------------------------------------------------------------
ALL SHARD FILES
----------------------------------------------------------------------------------------------------
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_lambdarank_parts/part_000.parquet               0.043 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_lambdarank_parts/part_001.parquet               0.044 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_lambdarank_parts/part_002.parquet               0.044 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_lambdarank_parts/part_003.parquet               0.043 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_lambdarank_parts/part_004.parquet               0.043 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODEL

In [ ]:
# ============================================================
# CELL 21 — FIND FULL VALIDATION CANDIDATE / FEATURE ARTIFACT
# ============================================================

from pathlib import Path

print("=" * 100)
print("CELL 21 — FULL VALIDATION ARTIFACT DISCOVERY")
print("=" * 100)

MODEL_DIR = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

# ------------------------------------------------------------
# 1. Show ALL directories under MODELS
# ------------------------------------------------------------

print("\nDIRECTORIES:")
print("-" * 100)

directories = sorted(
    [
        p for p in MODEL_DIR.rglob("*")
        if p.is_dir()
    ]
)

for p in directories:

    name = p.name.lower()

    if any(
        key in name
        for key in [
            "full",
            "validation",
            "candidate",
            "feature",
            "rank",
            "match",
            "score"
        ]
    ):

        print(p)


# ------------------------------------------------------------
# 2. Search filenames
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FILES WITH FULL-VALIDATION / CANDIDATE / FEATURE KEYWORDS")
print("=" * 100)

files = sorted(
    MODEL_DIR.rglob("*")
)

keywords = [
    "full_validation",
    "candidate",
    "feature",
    "rank",
    "score",
    "edge",
    "matcher"
]

for p in files:

    if not p.is_file():
        continue

    name = p.name.lower()

    if any(
        key in name
        for key in keywords
    ):

        try:

            size_mb = p.stat().st_size / (1024 ** 2)

        except:

            size_mb = 0

        print(
            f"{str(p):<120} "
            f"{size_mb:>8.3f} MB"
        )


# ------------------------------------------------------------
# 3. Specifically inspect directories with part_*.parquet
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ALL PART-DIRECTORIES")
print("=" * 100)

part_dirs = {}

for p in MODEL_DIR.rglob("part_*.parquet"):

    parent = p.parent

    part_dirs.setdefault(
        parent,
        []
    ).append(p)

for parent, parts in sorted(
    part_dirs.items(),
    key=lambda x: str(x[0])
):

    print(
        f"\n{parent}"
    )

    print(
        f"    shards = {len(parts)}"
    )

    print(
        f"    first  = {parts[0].name}"
    )

    print(
        f"    last   = {parts[-1].name}"
    )


print("\n" + "=" * 100)
print("CELL 21 COMPLETE")
print("=" * 100)

CELL 21 — FULL VALIDATION ARTIFACT DISCOVERY

DIRECTORIES:
----------------------------------------------------------------------------------------------------
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_lambdarank_parts
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/full_validation_target_best_parts
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/production_candidate_parts

FILES WITH FULL-VALIDATION / CANDIDATE / FEATURE KEYWORDS
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/best_all_scored_validation_edges.parquet    1.192 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/final_lambdarank_model.txt                1.278 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/final_lambdarank_threshold.txt            0.000 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_ranker_lambdarank.txt      

In [ ]:
# ============================================================
# CELL 22 — TRACE FULL-VALIDATION PIPELINE
# ============================================================

import os
import re
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026")

print("=" * 100)
print("CELL 22 — TRACE FULL-VALIDATION PIPELINE")
print("=" * 100)

# Search notebook files under the project directory
notebook_files = []

for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".ipynb", ".py", ".txt"}:
        try:
            if p.stat().st_size < 100 * 1024 * 1024:
                notebook_files.append(p)
        except:
            pass

print(f"\nCandidate code/notebook files found: {len(notebook_files)}")

patterns = [
    "full_validation_lambdarank_parts",
    "full_validation_target_best_parts",
    "full_validation_lambdarank",
    "target_best",
    "lambdarank",
    "production_candidate_parts",
]

hits = []

for path in notebook_files:
    try:
        txt = path.read_text(errors="ignore")
    except:
        continue

    for pattern in patterns:
        if pattern.lower() in txt.lower():
            hits.append((path, pattern))

print("\n" + "=" * 100)
print("MATCHING FILES")
print("=" * 100)

seen = set()

for path, pattern in hits:
    key = str(path)
    if key not in seen:
        seen.add(key)
        print(f"\n{path}")
        matched_patterns = sorted({
            p for x, p in hits if str(x) == key
        })
        print("  matched:", ", ".join(matched_patterns))

# ------------------------------------------------------------
# If notebooks exist, inspect code cells around relevant terms
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RELEVANT CODE CONTEXT")
print("=" * 100)

for path in notebook_files:
    if path.suffix.lower() != ".ipynb":
        continue

    try:
        import json
        nb = json.loads(path.read_text(errors="ignore"))
    except:
        continue

    cells = nb.get("cells", [])

    for i, cell in enumerate(cells):
        src = "".join(cell.get("source", []))

        if any(p.lower() in src.lower() for p in patterns):
            print("\n" + "-" * 100)
            print(f"FILE: {path}")
            print(f"CELL INDEX: {i}")
            print("-" * 100)

            # Print previous + current + next cell
            for j in range(max(0, i-1), min(len(cells), i+2)):
                s = "".join(cells[j].get("source", []))

                print(f"\n### NOTEBOOK CELL {j} ###")
                print(s[:12000])

print("\n" + "=" * 100)
print("CELL 22 COMPLETE")
print("=" * 100)

CELL 22 — TRACE FULL-VALIDATION PIPELINE

Candidate code/notebook files found: 18

MATCHING FILES

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/lightgbm_matcher_2k.txt
  matched: lambdarank

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/reverse_k3_lightgbm.txt
  matched: lambdarank

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/reverse_k1_lightgbm.txt
  matched: lambdarank

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/practical_matcher_lgbm.txt
  matched: lambdarank

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/rapidfuzz_k10_ranking_lgbm_best.txt
  matched: lambdarank

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_reranker_lgbm.txt
  matched: lambdarank

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_ranker_lambdarank.txt
  matched: lambdarank

/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/final_lambdarank_model.tx

In [ ]:
# ============================================================
# CELL 23 — INSPECT PRODUCTION CANDIDATE PARTS
# ============================================================

from pathlib import Path
import pandas as pd
import os

MODELS = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

PROD = MODELS / "production_candidate_parts"

print("=" * 100)
print("CELL 23 — PRODUCTION CANDIDATE ARTIFACT INSPECTION")
print("=" * 100)

if not PROD.exists():
    print("\n❌ production_candidate_parts DOES NOT EXIST")
else:
    parts = sorted(PROD.glob("part_*.parquet"))

    print(f"\nDirectory:")
    print(PROD)

    print(f"\nNumber of shards: {len(parts)}")

    if parts:
        print(f"First shard: {parts[0].name}")
        print(f"Last shard : {parts[-1].name}")

    total_rows = 0
    total_queries = set()
    total_candidates = set()

    for i, p in enumerate(parts[:5]):
        df = pd.read_parquet(p)

        print("\n" + "-" * 100)
        print(f"SHARD {i}: {p.name}")
        print("-" * 100)

        print("Shape:", df.shape)
        print("Columns:")
        print(df.columns.tolist())

        print("\nHead:")
        print(df.head(5).to_string(index=False))

        print("\nUnique counts:")
        for c in df.columns:
            if c in ["query_id", "s1_id", "source1_entity_id", "candidate_entity_id"]:
                print(f"  {c}: {df[c].nunique()}")

        if "query_id" in df.columns:
            total_queries.update(df["query_id"].dropna().unique())

        if "s1_id" in df.columns:
            total_candidates.update(df["s1_id"].dropna().unique())

        total_rows += len(df)

    print("\n" + "=" * 100)
    print("FIRST 5 SHARDS SUMMARY")
    print("=" * 100)

    print("Rows inspected:", total_rows)
    print("Queries observed:", len(total_queries))
    print("Unique S1 observed:", len(total_candidates))

    # --------------------------------------------------------
    # Full directory statistics
    # --------------------------------------------------------

    print("\n" + "=" * 100)
    print("FULL DIRECTORY STATISTICS")
    print("=" * 100)

    rows_all = 0
    query_counts = []

    for p in parts:
        df = pd.read_parquet(p)

        rows_all += len(df)

        if "query_id" in df.columns:
            query_counts.append({
                "file": p.name,
                "rows": len(df),
                "queries": df["query_id"].nunique(),
                "avg_candidates_per_query":
                    len(df) / max(df["query_id"].nunique(), 1),
                "max_candidates_per_query":
                    df.groupby("query_id").size().max()
            })

    stats = pd.DataFrame(query_counts)

    print("\nTotal rows:", rows_all)

    if not stats.empty:
        print("\nShard statistics:")
        print(stats.to_string(index=False))

        print("\nAggregate:")
        print("  Total rows:", stats["rows"].sum())
        print("  Total unique-query observations:",
              stats["queries"].sum())
        print("  Mean candidates/query across shards:",
              stats["avg_candidates_per_query"].mean())
        print("  Maximum candidates/query:",
              stats["max_candidates_per_query"].max())

    print("\n" + "=" * 100)
    print("CELL 23 COMPLETE")
    print("=" * 100)

CELL 23 — PRODUCTION CANDIDATE ARTIFACT INSPECTION

Directory:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/production_candidate_parts

Number of shards: 0

FIRST 5 SHARDS SUMMARY
Rows inspected: 0
Queries observed: 0
Unique S1 observed: 0

FULL DIRECTORY STATISTICS

Total rows: 0

CELL 23 COMPLETE


In [ ]:
# ============================================================
# CELL 24 — CANDIDATE GENERATION ARTIFACT MAP
# ============================================================

from pathlib import Path
import pandas as pd
import os

MODELS = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

print("=" * 100)
print("CELL 24 — SAVED CANDIDATE GENERATION ARTIFACT MAP")
print("=" * 100)

# ------------------------------------------------------------
# Inspect every parquet artifact in MODELS
# ------------------------------------------------------------

files = sorted(MODELS.rglob("*.parquet"))

print(f"\nTotal parquet files: {len(files)}")

records = []

for p in files:
    try:
        size_mb = p.stat().st_size / (1024**2)
        df = pd.read_parquet(p)

        rec = {
            "file": str(p.relative_to(MODELS)),
            "size_mb": round(size_mb, 3),
            "rows": len(df),
            "cols": len(df.columns),
            "columns": ", ".join(df.columns.tolist()[:20]),
        }

        # Important identity columns
        for c in [
            "query_id",
            "s1_id",
            "source1_entity_id",
            "candidate_entity_id",
            "label"
        ]:
            if c in df.columns:
                rec[f"nunique_{c}"] = df[c].nunique()

        records.append(rec)

    except Exception as e:
        records.append({
            "file": str(p.relative_to(MODELS)),
            "error": str(e)
        })

artifact_df = pd.DataFrame(records)

print("\n" + "=" * 100)
print("ALL PARQUET ARTIFACTS")
print("=" * 100)

print(artifact_df.to_string(index=False))

# ------------------------------------------------------------
# Identify candidate-like artifacts
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CANDIDATE-LIKE ARTIFACTS")
print("=" * 100)

keywords = [
    "candidate",
    "matcher",
    "rank",
    "feature",
    "rapidfuzz",
    "tfidf",
    "reverse",
    "blocked"
]

mask = artifact_df["file"].str.lower().apply(
    lambda x: any(k in x for k in keywords)
)

candidate_like = artifact_df[mask].copy()

print(candidate_like.to_string(index=False))

# ------------------------------------------------------------
# Specifically find multi-candidate datasets
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MULTI-CANDIDATE DATASETS")
print("=" * 100)

for _, r in artifact_df.iterrows():

    if "query_id" not in r.index:
        continue

    nrows = r.get("rows", 0)
    nq = r.get("nunique_query_id", None)

    if pd.isna(nq) or nq is None or nq == 0:
        continue

    avg = nrows / nq

    if avg > 1.5:
        print(
            f"{r['file']}\n"
            f"  rows={nrows:,}\n"
            f"  queries={int(nq):,}\n"
            f"  avg candidates/query={avg:.2f}\n"
        )

print("\n" + "=" * 100)
print("CELL 24 COMPLETE")
print("=" * 100)

CELL 24 — SAVED CANDIDATE GENERATION ARTIFACT MAP

Total parquet files: 84

ALL PARQUET ARTIFACTS
                                                          file  size_mb   rows  cols                                                                                                                                                                                                                                                                                                                          columns  nunique_source1_entity_id  nunique_query_id  nunique_s1_id  nunique_label  nunique_candidate_entity_id
                          FINAL_SAVED_ARTIFACTS/VAL_S1.parquet    3.746  20000    11                                                                                                                                                                                  entity_id, business_name, business_address, country, source1_entity_id, matched_entity_ids, true_count, true_ids_str, name_n, addr

In [ ]:
# ============================================================
# CELL 25 — VALIDATION COVERAGE OF SAVED FEATURE ARTIFACTS
# ============================================================

import pandas as pd
from pathlib import Path

MODELS = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

print("=" * 100)
print("CELL 25 — CHECK SAVED FEATURE COVERAGE")
print("=" * 100)

# ------------------------------------------------------------
# Load validation IDs
# ------------------------------------------------------------

val_path = MODELS / "FINAL_SAVED_ARTIFACTS" / "VAL_S1.parquet"

val = pd.read_parquet(val_path)

print("\nVAL_S1")
print("rows:", len(val))
print("columns:", val.columns.tolist())

# The validation identifier appears to be source1_entity_id
if "source1_entity_id" in val.columns:
    val_ids = set(val["source1_entity_id"].dropna().astype(str))
else:
    raise ValueError("source1_entity_id not found in VAL_S1")

print("Validation source1 IDs:", len(val_ids))

# ------------------------------------------------------------
# 1. Practical matcher features
# ------------------------------------------------------------

practical_path = MODELS / "practical_matcher_features.parquet"

practical = pd.read_parquet(practical_path)

print("\n" + "-" * 100)
print("PRACTICAL MATCHER FEATURES")
print("-" * 100)

print("rows:", len(practical))
print("columns:", practical.columns.tolist())

practical_ids = set(
    practical["source1_entity_id"]
    .dropna()
    .astype(str)
)

intersection_practical = val_ids & practical_ids

print("Unique source1 IDs:", len(practical_ids))
print("Validation IDs covered:", len(intersection_practical))
print(
    "Coverage:",
    f"{100 * len(intersection_practical) / len(val_ids):.2f}%"
)

print(
    "Rows belonging to validation IDs:",
    practical["source1_entity_id"]
    .astype(str)
    .isin(val_ids)
    .sum()
)

# Candidate count per validation source1
p_val = practical[
    practical["source1_entity_id"].astype(str).isin(val_ids)
].copy()

if len(p_val):
    counts = p_val.groupby("source1_entity_id").size()

    print("\nCandidate-count distribution:")
    print(counts.describe())

    print("\nCandidate count quantiles:")
    print(counts.quantile([0, .25, .50, .75, .90, .95, .99, 1]))

# ------------------------------------------------------------
# 2. Reverse K3 candidate features
# ------------------------------------------------------------

reverse_path = MODELS / "reverse_k3_candidate_features.parquet"

reverse = pd.read_parquet(reverse_path)

print("\n" + "-" * 100)
print("REVERSE K3 CANDIDATE FEATURES")
print("-" * 100)

print("rows:", len(reverse))
print("columns:", reverse.columns.tolist())

reverse_ids = set(
    reverse["source1_entity_id"]
    .dropna()
    .astype(str)
)

intersection_reverse = val_ids & reverse_ids

print("Unique source1 IDs:", len(reverse_ids))
print("Validation IDs covered:", len(intersection_reverse))
print(
    "Coverage:",
    f"{100 * len(intersection_reverse) / len(val_ids):.2f}%"
)

print(
    "Rows belonging to validation IDs:",
    reverse["source1_entity_id"]
    .astype(str)
    .isin(val_ids)
    .sum()
)

r_val = reverse[
    reverse["source1_entity_id"].astype(str).isin(val_ids)
].copy()

if len(r_val):
    counts_r = r_val.groupby("source1_entity_id").size()

    print("\nCandidate-count distribution:")
    print(counts_r.describe())

    print("\nCandidate count quantiles:")
    print(counts_r.quantile([0, .25, .50, .75, .90, .95, .99, 1]))

# ------------------------------------------------------------
# 3. Check labels
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LABEL COVERAGE")
print("=" * 100)

if len(p_val):
    print("\nPractical:")
    print(p_val["label"].value_counts(dropna=False))

if len(r_val):
    print("\nReverse K3:")
    print(r_val["label"].value_counts(dropna=False))

print("\n" + "=" * 100)
print("CELL 25 COMPLETE")
print("=" * 100)

CELL 25 — CHECK SAVED FEATURE COVERAGE

VAL_S1
rows: 20000
columns: ['entity_id', 'business_name', 'business_address', 'country', 'source1_entity_id', 'matched_entity_ids', 'true_count', 'true_ids_str', 'name_n', 'addr_n', 'country_n']
Validation source1 IDs: 20000

----------------------------------------------------------------------------------------------------
PRACTICAL MATCHER FEATURES
----------------------------------------------------------------------------------------------------
rows: 829999
columns: ['source1_entity_id', 'target_id', 'name_ratio', 'name_partial', 'name_token_set', 'name_token_sort', 'name_jaccard', 'address_ratio', 'address_partial', 'address_token_set', 'address_token_sort', 'address_jaccard', 'combined_jaccard', 'combined_ratio', 'exact_name', 'exact_address', 'country_match', 'number_exact', 'name_len_ratio', 'address_len_ratio', 'name_first_match', 'name_last_match', 'address_first_match', 'address_last_match', 'name_char_len_diff', 'address_char_len_d

In [ ]:
# ============================================================
# CELL 26 — TRUE CANDIDATE RETRIEVAL DIAGNOSIS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

MODELS = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS"
)

print("=" * 100)
print("CELL 26 — TRUE CANDIDATE RETRIEVAL DIAGNOSIS")
print("=" * 100)

# ------------------------------------------------------------
# Load validation mapping
# ------------------------------------------------------------

val = pd.read_parquet(
    MODELS / "FINAL_SAVED_ARTIFACTS" / "VAL_S1.parquet"
)

val["source1_entity_id"] = (
    val["source1_entity_id"].astype(str)
)

# ------------------------------------------------------------
# Load practical candidate features
# ------------------------------------------------------------

df = pd.read_parquet(
    MODELS / "practical_matcher_features.parquet"
)

df["source1_entity_id"] = (
    df["source1_entity_id"].astype(str)
)

df["target_id"] = df["target_id"].astype(str)

# ------------------------------------------------------------
# Build true target set
# ------------------------------------------------------------

true_map = {}

for _, row in val.iterrows():

    sid = row["source1_entity_id"]

    true_ids = []

    # matched_entity_ids may already be a list
    x = row["matched_entity_ids"]

    if isinstance(x, (list, tuple, set, np.ndarray)):
        true_ids = [str(v) for v in x]

    elif pd.notna(x):
        # Handle string representation
        s = str(x).strip()

        if s.startswith("[") and s.endswith("]"):
            try:
                import ast
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    true_ids = [str(v) for v in parsed]
                else:
                    true_ids = [str(parsed)]
            except:
                true_ids = [s]
        else:
            true_ids = [s]

    true_map[sid] = set(true_ids)

# ------------------------------------------------------------
# Keep only validation population
# ------------------------------------------------------------

val_ids = set(true_map.keys())

dfv = df[
    df["source1_entity_id"].isin(val_ids)
].copy()

print("\nCandidate rows:", len(dfv))
print("Validation queries represented:",
      dfv["source1_entity_id"].nunique())

# ------------------------------------------------------------
# Mark true candidates
# ------------------------------------------------------------

dfv["is_true"] = False

for sid, ids in true_map.items():

    mask = dfv["source1_entity_id"].eq(sid)

    if ids:
        dfv.loc[
            mask & dfv["target_id"].isin(ids),
            "is_true"
        ] = True

# ------------------------------------------------------------
# Query-level retrieval coverage
# ------------------------------------------------------------

query_stats = (
    dfv.groupby("source1_entity_id")
       .agg(
           candidates=("target_id", "size"),
           true_candidates=("is_true", "sum")
       )
       .reset_index()
)

query_stats["true_present"] = (
    query_stats["true_candidates"] > 0
)

print("\n" + "=" * 100)
print("RETRIEVAL COVERAGE")
print("=" * 100)

print(
    "Queries with >=1 true candidate:",
    query_stats["true_present"].sum()
)

print(
    "Queries with NO true candidate:",
    (~query_stats["true_present"]).sum()
)

print(
    "Retrieval coverage:",
    f"{100 * query_stats['true_present'].mean():.4f}%"
)

# ------------------------------------------------------------
# Number of true candidates per query
# ------------------------------------------------------------

print("\nTrue candidates/query:")
print(
    query_stats.loc[
        query_stats.true_present,
        "true_candidates"
    ].describe()
)

# ------------------------------------------------------------
# Candidate count distribution
# ------------------------------------------------------------

print("\nCandidates/query:")
print(query_stats["candidates"].describe())

print("\nCandidate-count quantiles:")
print(
    query_stats["candidates"].quantile(
        [0, .25, .50, .75, .90, .95, .99, 1.0]
    )
)

# ------------------------------------------------------------
# Positive-label sanity check
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LABEL SANITY CHECK")
print("=" * 100)

print(
    dfv["label"].value_counts(dropna=False)
)

print("\nTrue candidate rows identified from VAL mapping:",
      int(dfv["is_true"].sum()))

print(
    "Rows with label=1:",
    int((dfv["label"] == 1).sum())
)

# ------------------------------------------------------------
# Compare our mapping against stored labels
# ------------------------------------------------------------

dfv["label_is_true"] = (
    dfv["label"].fillna(0).astype(int) == 1
)

agreement = (
    dfv["is_true"] == dfv["label_is_true"]
).mean()

print(
    "\nMapping/label agreement:",
    f"{100 * agreement:.6f}%"
)

print(
    "Mapping true but label=0:",
    int(
        (dfv["is_true"] & ~dfv["label_is_true"]).sum()
    )
)

print(
    "Mapping false but label=1:",
    int(
        (~dfv["is_true"] & dfv["label_is_true"]).sum()
    )
)

print("\n" + "=" * 100)
print("CELL 26 COMPLETE")
print("=" * 100)

CELL 26 — TRUE CANDIDATE RETRIEVAL DIAGNOSIS

Candidate rows: 829999
Validation queries represented: 19999


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026")

print("=" * 90)
print("SUBMISSION ARTIFACT SEARCH")
print("=" * 90)

patterns = [
    "*matching_results*.tsv",
    "*matching_results*.csv",
    "*candidate_pairs*.tsv",
    "*candidate_pairs*.csv",
    "*test*prediction*.parquet",
    "*test*pred*.parquet",
    "*test*score*.parquet",
]

found = []

for pattern in patterns:
    for p in ROOT.rglob(pattern):
        found.append(p)

if not found:
    print("\nNo existing submission artifacts found.")
else:
    for p in sorted(set(found)):
        print(
            f"{p}\n"
            f"  size = {p.stat().st_size / 1024**2:.3f} MB"
        )

print("\n" + "=" * 90)
print("SEARCH COMPLETE")
print("=" * 90)

SUBMISSION ARTIFACT SEARCH
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/matching_results_f05_0.988468.tsv
  size = 1.083 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_3/candidate_pairs.tsv
  size = 177.223 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_3/matching_results.tsv
  size = 39.749 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_4_EXPRESS/candidate_pairs.tsv
  size = 177.344 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/SUBMISSION_4_EXPRESS/matching_results.tsv
  size = 39.883 MB

SEARCH COMPLETE


In [ ]:
# ============================================================
# FINAL SUBMISSION CHECK — F0.5 = 0.988468
# ============================================================

from pathlib import Path
import pandas as pd

SUB = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/"
    "matching_results_f05_0.988468.tsv"
)

print("=" * 90)
print("FINAL FROZEN SUBMISSION CHECK")
print("=" * 90)

print("\nFile:")
print(SUB)

print("\nSize:")
print(f"{SUB.stat().st_size / (1024**2):.3f} MB")

df = pd.read_csv(
    SUB,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("\nShape:", df.shape)
print("Columns:", df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head().to_string(index=False))

print("\nUnique source1 IDs:",
      df["source1_entity_id"].nunique())

print("Duplicate source1 rows:",
      df["source1_entity_id"].duplicated().sum())

print("Empty matched_entity_ids:",
      (df["matched_entity_ids"] == "").sum())

print("\n" + "=" * 90)
print("FROZEN SUBMISSION CHECK COMPLETE")
print("=" * 90)

FINAL FROZEN SUBMISSION CHECK

File:
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/matching_results_f05_0.988468.tsv

Size:
1.083 MB

Shape: (20000, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']

First 5 rows:
source1_entity_id                                               matched_entity_ids
      S1-53356671 S2-256033433,S2-453334141,S2-904412198,S3-468658814,S3-996978203
     S1-320151505                           S2-230628758,S2-951553277,S3-975397820
     S1-938947364                           S2-159830603,S3-268060208,S3-544960509
     S1-195839862                                        S2-232134153,S2-344984473
     S1-655046555                                         S2-113600604,S2-73156901

Unique source1 IDs: 20000
Duplicate source1 rows: 0
Empty matched_entity_ids: 1145

FROZEN SUBMISSION CHECK COMPLETE


In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

for name in [
    "SUBMISSION_3/candidate_pairs.tsv",
    "SUBMISSION_4_EXPRESS/candidate_pairs.tsv"
]:
    p = ROOT / name
    print(
        name,
        "| exists:", p.exists(),
        "| size:",
        f"{p.stat().st_size / (1024**2):.3f} MB"
        if p.exists() else "N/A"
    )

SUBMISSION_3/candidate_pairs.tsv | exists: True | size: 177.223 MB
SUBMISSION_4_EXPRESS/candidate_pairs.tsv | exists: True | size: 177.344 MB


In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

ROOT = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

files = {
    "SUBMISSION_3": ROOT / "SUBMISSION_3",
    "SUBMISSION_4_EXPRESS": ROOT / "SUBMISSION_4_EXPRESS",
}

frozen = ROOT / "MODELS" / "matching_results_f05_0.988468.tsv"

print("=" * 100)
print("FROZEN SUBMISSION ↔ EXISTING PACKAGE COMPARISON")
print("=" * 100)

# ------------------------------------------------------------
# Frozen prediction
# ------------------------------------------------------------

frozen_df = pd.read_csv(
    frozen,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("\nFrozen prediction:")
print("Rows:", len(frozen_df))
print("SHA256:", hashlib.sha256(frozen.read_bytes()).hexdigest())

# ------------------------------------------------------------
# Compare each package's matching_results.tsv
# ------------------------------------------------------------

for name, folder in files.items():

    result = folder / "matching_results.tsv"
    candidate = folder / "candidate_pairs.tsv"

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    if not result.exists():
        print("matching_results.tsv: MISSING")
        continue

    df = pd.read_csv(
        result,
        sep="\t",
        dtype=str,
        keep_default_na=False
    )

    print("matching_results rows:", len(df))
    print(
        "matching_results SHA256:",
        hashlib.sha256(result.read_bytes()).hexdigest()
    )

    # Exact comparison with frozen prediction
    same_shape = df.shape == frozen_df.shape

    if same_shape:
        same_values = (
            df.reset_index(drop=True)
              .equals(frozen_df.reset_index(drop=True))
        )
    else:
        same_values = False

    print("EXACTLY SAME AS FROZEN PREDICTION:", same_values)

    # Candidate metadata
    if candidate.exists():
        print(
            "candidate_pairs size:",
            f"{candidate.stat().st_size / (1024**2):.3f} MB"
        )

        # Don't load the 177 MB file yet.
        print("candidate_pairs: present")
    else:
        print("candidate_pairs: MISSING")

print("\n" + "=" * 100)
print("COMPARISON COMPLETE")
print("=" * 100)

FROZEN SUBMISSION ↔ EXISTING PACKAGE COMPARISON

Frozen prediction:
Rows: 20000
SHA256: 6491aabe6fae50bd2f543cd047b7c13f0eebaa085563b259bbc832c9e6e53b18

----------------------------------------------------------------------------------------------------
SUBMISSION_3
----------------------------------------------------------------------------------------------------
matching_results rows: 1732544
matching_results SHA256: 2dca97573606696ba1f1e638c6dd55d0d78c764fdddf6bf3cd1291987eca115d
EXACTLY SAME AS FROZEN PREDICTION: False
candidate_pairs size: 177.223 MB
candidate_pairs: present

----------------------------------------------------------------------------------------------------
SUBMISSION_4_EXPRESS
----------------------------------------------------------------------------------------------------
matching_results rows: 1732544
matching_results SHA256: 88e61faaaeb1c15ec720244e3f70155735fc4661bfaba4a56d49225f1963a077
EXACTLY SAME AS FROZEN PREDICTION: False
candidate_pairs size: 177

In [ ]:
# ============================================================
# INSPECT THE 1,732,544-ROW SUBMISSION DATA
# NO MODEL TRAINING / NO EXPERIMENT
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

for name in ["SUBMISSION_3", "SUBMISSION_4_EXPRESS"]:

    p = ROOT / name / "matching_results.tsv"

    print("=" * 100)
    print(name)
    print("=" * 100)

    # Read only first rows
    df = pd.read_csv(
        p,
        sep="\t",
        dtype=str,
        nrows=5,
        keep_default_na=False
    )

    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    print(df.to_string(index=False))

    print("\nNumber of columns:", len(df.columns))

SUBMISSION_3
Columns:
['source1_entity_id', 'matched_entity_ids']

First 5 rows:
source1_entity_id        matched_entity_ids
     S1-714132312                          
     S1-106407869                          
     S1-156285671                          
     S1-689823050                          
     S1-921369899 S2-808920974,S3-285097097

Number of columns: 2
SUBMISSION_4_EXPRESS
Columns:
['source1_entity_id', 'matched_entity_ids']

First 5 rows:
source1_entity_id        matched_entity_ids
     S1-714132312                          
     S1-106407869                          
     S1-156285671                          
     S1-689823050                          
     S1-921369899 S2-808920974,S3-285097097

Number of columns: 2


In [ ]:
import pandas as pd
import hashlib
from pathlib import Path

BASE = Path("/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026")

files = {
    "SUBMISSION_3": BASE / "SUBMISSION_3" / "matching_results.tsv",
    "SUBMISSION_4_EXPRESS": BASE / "SUBMISSION_4_EXPRESS" / "matching_results.tsv",
}

for name, path in files.items():
    print("=" * 100)
    print(name)
    print("=" * 100)

    df = pd.read_csv(path, sep="\t", dtype=str).fillna("")

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Unique source1 IDs:", df["source1_entity_id"].nunique())
    print("Empty matches:", (df["matched_entity_ids"].str.strip() == "").sum())
    print("Non-empty matches:", (df["matched_entity_ids"].str.strip() != "").sum())

    print("\nFirst 10 NON-EMPTY predictions:")
    print(
        df[df["matched_entity_ids"].str.strip() != ""]
        .head(10)
        .to_string(index=False)
    )

# Exact comparison
a = pd.read_csv(files["SUBMISSION_3"], sep="\t", dtype=str).fillna("")
b = pd.read_csv(files["SUBMISSION_4_EXPRESS"], sep="\t", dtype=str).fillna("")

print("\n" + "=" * 100)
print("EXACT COMPARISON")
print("=" * 100)

print("Same shape:", a.shape == b.shape)
print("Exactly identical:", a.equals(b))

if not a.equals(b):
    diff = (a != b).any(axis=1)
    print("Different rows:", diff.sum())
    print("\nFirst differences:")
    print(
        pd.concat(
            [a.loc[diff].head(10).add_prefix("SUB3_"),
             b.loc[diff].head(10).add_prefix("SUB4_")],
            axis=1
        ).to_string(index=False)
    )

SUBMISSION_3
Shape: (1732544, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']
Unique source1 IDs: 1732544
Empty matches: 801668
Non-empty matches: 930876

First 10 NON-EMPTY predictions:
source1_entity_id                     matched_entity_ids
     S1-921369899              S2-808920974,S3-285097097
     S1-481669221                           S2-284602321
     S1-909865979                           S2-528901108
     S1-742053041               S2-73207920,S2-317425677
     S1-479574617              S2-985539429,S2-262582939
     S1-365170694                           S2-487943573
     S1-778640743                           S3-698916948
     S1-209281220                           S3-893592109
     S1-704604312                           S2-497722920
     S1-628750886 S2-696378102,S2-406813553,S3-734992856
SUBMISSION_4_EXPRESS
Shape: (1732544, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']
Unique source1 IDs: 1732544
Empty matches: 796550
Non-empty matches: 935994

First 1

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026")

print("=" * 100)
print("MODEL FILES")
print("=" * 100)

for p in BASE.rglob("*"):
    if p.is_file():
        name = p.name.lower()
        if (
            "lambdarank" in name
            or "ranker" in name
            or "model" in name
            or "feature" in name
            or "candidate" in name
        ):
            try:
                size_mb = p.stat().st_size / (1024**2)
                print(f"{p} | {size_mb:.2f} MB")
            except:
                pass

print("\n" + "=" * 100)
print("LARGE PARQUET FILES")
print("=" * 100)

for p in BASE.rglob("*.parquet"):
    try:
        size_mb = p.stat().st_size / (1024**2)
        if size_mb > 50:
            print(f"{p} | {size_mb:.2f} MB")
    except:
        pass

MODEL FILES
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/matcher_candidates_2k_union.parquet | 2.06 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/matcher_features_2k.parquet | 6.99 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/reverse_tfidf_candidate_evaluation.csv | 0.00 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/reverse_k3_candidate_features.parquet | 4.10 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/practical_matcher_features.parquet | 23.33 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_rerank_candidates_2k.parquet | 2.78 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_rerank_features_2k.parquet | 22.47 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_reranker_lgbm.txt | 1.28 MB
/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/blocked_tfidf_rare_reranker_threshold.txt | 0.00 MB
/conten

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026")

files = [
    BASE / "SUBMISSION_3" / "candidate_pairs.tsv",
    BASE / "SUBMISSION_4_EXPRESS" / "candidate_pairs.tsv",
    BASE / "MODELS" / "practical_matcher_features.parquet",
    BASE / "MODELS" / "blocked_tfidf_rare_rerank_features_2k.parquet",
]

for path in files:
    print("\n" + "=" * 110)
    print(path.name)
    print("=" * 110)

    if path.suffix == ".tsv":
        df = pd.read_csv(path, sep="\t", nrows=5)
    else:
        df = pd.read_parquet(path).head(5)

    print("Columns:", df.columns.tolist())
    print("Number of columns:", len(df.columns))
    print("\nFirst 5 rows:")
    print(df.to_string(index=False))

print("\n" + "=" * 110)
print("FROZEN THRESHOLD")
print("=" * 110)

threshold_file = BASE / "MODELS" / "FINAL_SAVED_ARTIFACTS" / "final_lambdarank_threshold.txt"

print(threshold_file.read_text())


candidate_pairs.tsv
Columns: ['source1_entity_id', 'candidate_entity_ids']
Number of columns: 2

First 5 rows:
source1_entity_id                                                                                                                                                                                                                                             candidate_entity_ids
     S1-714132312   S2-902693916,S2-36973049,S2-742926619,S2-406131322,S2-131762854,S2-723568545,S2-49406445,S2-188157420,S2-365284937,S2-211902825,S3-76581442,S3-948321994,S3-954194052,S3-96374362,S3-521557639,S3-635835046,S3-589056677,S3-987464583,S3-53762159,S3-836630817
     S1-106407869                                                                                                                                 S2-47475427,S2-886540563,S2-997023970,S2-571602010,S2-104974637,S3-943074147,S3-696641691,S3-245724670,S3-918090607,S3-654401737
     S1-156285671                                              

In [ ]:
# ============================================================
# AMAZON ML 2026 — FROZEN PRODUCTION INFERENCE
# CELL 1: LOAD + VERIFY FINAL ARTIFACTS
#
# NO TRAINING
# NO EXPERIMENT
# NO THRESHOLD SEARCH
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import scipy.sparse as sp
import lightgbm as lgb
import json

BASE = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

FINAL_DIR = (
    BASE /
    "MODELS" /
    "FINAL_SAVED_ARTIFACTS"
)

TEST_DIR = (
    BASE /
    "DATASET" /
    "test"
)

print("=" * 100)
print("FROZEN PRODUCTION INFERENCE — ARTIFACT VERIFICATION")
print("=" * 100)

# ------------------------------------------------------------
# 1. FINAL MODEL
# ------------------------------------------------------------

MODEL_PATH = (
    FINAL_DIR /
    "final_lambdarank_model.txt"
)

THRESHOLD_PATH = (
    FINAL_DIR /
    "final_lambdarank_threshold.txt"
)

matcher = lgb.Booster(
    model_file=str(MODEL_PATH)
)

with open(THRESHOLD_PATH) as f:
    threshold = float(
        f.read().strip()
    )

print("\nMODEL")
print("-" * 100)
print("Path:", MODEL_PATH)
print("Trees:", matcher.num_trees())
print("Features:", matcher.num_feature())
print("Threshold:", threshold)

assert matcher.num_feature() == 56, (
    f"Expected 56 features, got {matcher.num_feature()}"
)

assert abs(threshold - 0.628) < 1e-9, (
    f"Frozen threshold mismatch: {threshold}"
)

# ------------------------------------------------------------
# 2. TF-IDF ARTIFACTS
# ------------------------------------------------------------

vectorizer = joblib.load(
    FINAL_DIR /
    "tfidf_vectorizer.joblib"
)

X_test_s1 = sp.load_npz(
    FINAL_DIR /
    "X_test_s1_matrix.npz"
)

print("\nTF-IDF")
print("-" * 100)
print(
    "Vocabulary:",
    f"{len(vectorizer.vocabulary_):,}"
)

print(
    "X_test_s1 shape:",
    X_test_s1.shape
)

assert X_test_s1.shape[0] == 1_732_544, (
    f"Unexpected X_test_s1 rows: {X_test_s1.shape[0]}"
)

# ------------------------------------------------------------
# 3. PRODUCTION BLOCKING INDEXES
# ------------------------------------------------------------

block_files = {
    "name_first": "name_first_blocks.joblib",
    "name_last": "name_last_blocks.joblib",
    "addr_first": "addr_first_blocks.joblib",
    "addr_last": "addr_last_blocks.joblib",
    "num_country": "num_country_blocks.joblib",
}

blocks = {}

print("\nBLOCKING INDEXES")
print("-" * 100)

for label, filename in block_files.items():

    path = FINAL_DIR / filename

    obj = joblib.load(path)

    blocks[label] = obj

    sizes = np.asarray(
        [len(v) for v in obj.values()],
        dtype=np.int32
    )

    print(
        f"{label:15s}"
        f" blocks={len(obj):>8,}"
        f" median={np.median(sizes):>8.0f}"
        f" p95={np.percentile(sizes,95):>10.0f}"
        f" max={sizes.max():>10,}"
    )

# ------------------------------------------------------------
# 4. RARE TOKEN INDEX
# ------------------------------------------------------------

rare_token_index = joblib.load(
    FINAL_DIR /
    "rare_token_index_prod.joblib"
)

token_df = joblib.load(
    FINAL_DIR /
    "token_df_prod.joblib"
)

print("\nRARE TOKEN INDEX")
print("-" * 100)

print(
    "Rare-token postings:",
    f"{len(rare_token_index):,}"
)

print(
    "Unique token statistics:",
    f"{len(token_df):,}"
)

# ------------------------------------------------------------
# 5. TEST S1
# ------------------------------------------------------------

test_s1 = pd.read_csv(
    TEST_DIR /
    "test_source1.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False
)

test_s1_ids = (
    test_s1["entity_id"]
    .astype(str)
    .tolist()
)

print("\nTEST S1")
print("-" * 100)

print(
    "Rows:",
    f"{len(test_s1):,}"
)

print(
    "Unique IDs:",
    test_s1["entity_id"].nunique()
)

print(
    "Matches X_test_s1:",
    len(test_s1_ids)
    ==
    X_test_s1.shape[0]
)

assert len(test_s1) == 1_732_544
assert test_s1["entity_id"].nunique() == 1_732_544
assert len(test_s1_ids) == X_test_s1.shape[0]

# ------------------------------------------------------------
# 6. ENTITY-ID <-> LOCAL INDEX
# ------------------------------------------------------------

entity_id_to_local = {
    eid: i
    for i, eid in enumerate(test_s1_ids)
}

local_to_entity_id = {
    i: eid
    for i, eid in enumerate(test_s1_ids)
}

print(
    "\nLocal-index map:",
    f"{len(entity_id_to_local):,}"
)

# ------------------------------------------------------------
# 7. PRODUCTION RUN DIRECTORY
# ------------------------------------------------------------

RUN_DIR = (
    BASE /
    "MODELS" /
    "PRODUCTION_RUN"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_PATH = (
    RUN_DIR /
    "manifest.json"
)

if MANIFEST_PATH.exists():

    manifest = json.loads(
        MANIFEST_PATH.read_text()
    )

    print(
        "\nExisting production manifest:",
        manifest
    )

else:

    manifest = {
        "s2_batches_done": 0,
        "s3_batches_done": 0,
        "batch_size": 50_000
    }

    MANIFEST_PATH.write_text(
        json.dumps(
            manifest,
            indent=2
        )
    )

    print(
        "\nCreated production manifest:",
        manifest
    )

print("\n" + "=" * 100)
print("ARTIFACT VERIFICATION COMPLETE")
print("=" * 100)
print("Frozen model: VERIFIED")
print("Frozen threshold: 0.628 VERIFIED")
print("S1 scale: 1,732,544 VERIFIED")
print("Training/retraining: NONE")
print("\nSTOP HERE and send me the output.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FROZEN PRODUCTION INFERENCE — ARTIFACT VERIFICATION

MODEL
----------------------------------------------------------------------------------------------------
Path: /content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026/MODELS/FINAL_SAVED_ARTIFACTS/final_lambdarank_model.txt
Trees: 424
Features: 56
Threshold: 0.6280000000000001

TF-IDF
----------------------------------------------------------------------------------------------------
Vocabulary: 100,000
X_test_s1 shape: (1732544, 100000)

BLOCKING INDEXES
----------------------------------------------------------------------------------------------------
name_first      blocks=     109 median=    9879 p95=     47994 max=   136,193
name_last       blocks=     107 median=     637 p95=     45756 max=   613,148
addr_first      blocks=     105 median=    6683 p95=     61520 max=   180,957
addr_last       blocks=  

In [ ]:
# ============================================================
# AMAZON ML 2026 — FROZEN PRODUCTION INFERENCE
# CELL 2: S2 PRODUCTION CANDIDATE GENERATION
#
# FROZEN:
#   - Production S1 TF-IDF matrix
#   - Production blocking indexes
#   - Rare-token index
#   - NO model training
#
# CHECKPOINTED:
#   50,000 S2 records per batch
# ============================================================

import os
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/"
    "AMAZON ML 2026/AMAZON ML 2026"
)

FINAL_DIR = (
    BASE /
    "MODELS" /
    "FINAL_SAVED_ARTIFACTS"
)

RUN_DIR = (
    BASE /
    "MODELS" /
    "PRODUCTION_RUN"
)

S2_PATH = (
    BASE /
    "DATASET" /
    "test" /
    "test_source2.tsv"
)

S2_OUT_DIR = (
    RUN_DIR /
    "s2_candidates"
)

S2_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

BATCH_SIZE = 50_000

# ------------------------------------------------------------
# LOAD PRODUCTION ARTIFACTS
# ------------------------------------------------------------

print("=" * 100)
print("S2 PRODUCTION CANDIDATE GENERATION")
print("=" * 100)

print("\nLoading S2...")

s2 = pd.read_csv(
    S2_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print(
    "S2 rows:",
    f"{len(s2):,}"
)

print(
    "S2 unique IDs:",
    f"{s2['entity_id'].nunique():,}"
)

assert s2["entity_id"].nunique() == len(s2)

# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

def norm_text(x):
    return (
        str(x)
        .lower()
        .strip()
    )

for col in [
    "business_name",
    "business_address",
    "country"
]:
    if col in s2.columns:
        s2[col + "_norm"] = (
            s2[col]
            .map(norm_text)
        )

# ------------------------------------------------------------
# VERIFY REQUIRED PRODUCTION INDEXES
# ------------------------------------------------------------

required = [
    "name_first",
    "name_last",
    "addr_first",
    "addr_last",
    "num_country"
]

for key in required:
    assert key in blocks, (
        f"Missing production block: {key}"
    )

print("\nProduction blocking indexes verified.")

# ------------------------------------------------------------
# IMPORTANT:
# This cell only generates candidate IDs.
#
# The actual feature construction and LambdaRank scoring
# happen after candidate generation.
# ------------------------------------------------------------

# Try to reuse an existing production candidate-generation
# function if it exists in the notebook namespace.
#
# We intentionally do NOT invent a new blocking formula here.
# The exact notebook production function must already be loaded.
# ------------------------------------------------------------

candidate_fn_candidates = [
    "generate_production_candidates",
    "generate_candidates_production",
    "generate_candidates",
    "get_production_candidates"
]

candidate_fn = None
candidate_fn_name = None

for name in candidate_fn_candidates:
    if name in globals() and callable(globals()[name]):
        candidate_fn = globals()[name]
        candidate_fn_name = name
        break

if candidate_fn is None:
    raise RuntimeError(
        "\nNo existing production candidate-generation function "
        "was found in the current notebook namespace.\n\n"
        "STOP — do NOT replace it with an improvised blocking "
        "implementation.\n"
        "The validated notebook pipeline must be used exactly."
    )

print(
    "Using existing production candidate function:",
    candidate_fn_name
)

# ------------------------------------------------------------
# CHECKPOINT STATE
# ------------------------------------------------------------

manifest_path = (
    RUN_DIR /
    "manifest.json"
)

manifest = json.loads(
    manifest_path.read_text()
)

completed_batches = int(
    manifest.get("s2_batches_done", 0)
)

n_batches = (
    len(s2) + BATCH_SIZE - 1
) // BATCH_SIZE

print(
    f"\nBatches: {n_batches:,}"
)

print(
    f"Already completed: {completed_batches:,}"
)

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

start_total = time.time()

for batch_no in range(
    completed_batches,
    n_batches
):

    start = batch_no * BATCH_SIZE
    end = min(
        start + BATCH_SIZE,
        len(s2)
    )

    batch = s2.iloc[start:end].copy()

    print(
        f"\n[{batch_no + 1}/{n_batches}] "
        f"rows {start:,} → {end:,}"
    )

    t0 = time.time()

    # --------------------------------------------------------
    # USE THE NOTEBOOK'S EXISTING PRODUCTION FUNCTION
    # --------------------------------------------------------

    result = candidate_fn(
        batch,
        X_test_s1,
        blocks,
        rare_token_index,
        token_df
    )

    # --------------------------------------------------------
    # NORMALIZE RESULT
    # --------------------------------------------------------

    if isinstance(result, pd.DataFrame):

        candidates = result

    else:

        candidates = pd.DataFrame(
            result
        )

    print(
        "Candidate rows:",
        f"{len(candidates):,}"
    )

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    out_path = (
        S2_OUT_DIR /
        f"part_{batch_no:04d}.parquet"
    )

    candidates.to_parquet(
        out_path,
        index=False
    )

    manifest["s2_batches_done"] = (
        batch_no + 1
    )

    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2
        )
    )

    elapsed = time.time() - t0

    print(
        f"Saved: {out_path.name}"
    )

    print(
        f"Batch time: {elapsed / 60:.2f} min"
    )

    del batch
    del candidates
    del result

    gc.collect()

# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

total_elapsed = (
    time.time() - start_total
)

print("\n" + "=" * 100)
print("S2 CANDIDATE GENERATION COMPLETE")
print("=" * 100)

print(
    "Batches completed:",
    manifest["s2_batches_done"],
    "/",
    n_batches
)

print(
    "Total time:",
    f"{total_elapsed / 60:.2f} min"
)

print(
    "Output directory:",
    S2_OUT_DIR
)

print("=" * 100)

S2 PRODUCTION CANDIDATE GENERATION

Loading S2...
S2 rows: 4,887,273
S2 unique IDs: 4,887,273

Production blocking indexes verified.


RuntimeError: 
No existing production candidate-generation function was found in the current notebook namespace.

STOP — do NOT replace it with an improvised blocking implementation.
The validated notebook pipeline must be used exactly.

In [ ]:
# ============================================================
# CELL 2A — RECOVER ORIGINAL CANDIDATE GENERATION FUNCTION
# ============================================================

import inspect
import types

print("=" * 100)
print("SEARCHING CURRENT NOTEBOOK NAMESPACE FOR ORIGINAL CANDIDATE GENERATION LOGIC")
print("=" * 100)

keywords = [
    "candidate",
    "block",
    "tfidf",
    "rare",
    "retriev",
    "match"
]

found = []

for name, obj in list(globals().items()):

    name_lower = name.lower()

    # Functions
    if isinstance(obj, types.FunctionType):

        if any(k in name_lower for k in keywords):

            found.append(("FUNCTION", name, obj))

    # Classes / callable objects
    elif callable(obj):

        if any(k in name_lower for k in keywords):

            found.append(("CALLABLE", name, obj))


print(f"\nRelevant callable objects found: {len(found)}\n")

for kind, name, obj in found:

    print("-" * 100)
    print(f"{kind}: {name}")

    try:
        print("Signature:", inspect.signature(obj))
    except Exception:
        print("Signature: <unavailable>")

    try:
        source = inspect.getsource(obj)
        print("\nSOURCE PREVIEW:")
        print(source[:3000])
    except Exception as e:
        print("Source unavailable:", type(e).__name__)

print("\n" + "=" * 100)
print("END SEARCH")
print("=" * 100)

SEARCHING CURRENT NOTEBOOK NAMESPACE FOR ORIGINAL CANDIDATE GENERATION LOGIC

Relevant callable objects found: 0


END SEARCH


In [ ]:
# ============================================================
# CELL 2 — EXACT FROZEN PRODUCTION CANDIDATE GENERATION
# ============================================================
#
# This is the production form of the candidate retrieval logic
# used in the validated notebook:
#
#   character blocks
#       -> sparse TF-IDF Top-100
#       -> TF-IDF Top-50
#
#   +
#
#   rare-token Top-4
#
#   =
#
#   FINAL CANDIDATE UNION
#
# No training.
# No model modification.
# No threshold modification.
# ============================================================

import os
import re
import gc
import json
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib

from pathlib import Path

print("=" * 100)
print("CELL 2 — EXACT FROZEN PRODUCTION CANDIDATE GENERATION")
print("=" * 100)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026"
)

MODEL_DIR = BASE / "MODELS"
FINAL_DIR = MODEL_DIR / "FINAL_SAVED_ARTIFACTS"
TEST_DIR = BASE / "DATASET" / "test"

RUN_DIR = MODEL_DIR / "PRODUCTION_RUN"

S2_OUT = RUN_DIR / "s2_candidates"
S3_OUT = RUN_DIR / "s3_candidates"

S2_OUT.mkdir(parents=True, exist_ok=True)
S3_OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Frozen constants from validated pipeline
# ------------------------------------------------------------

RARE_TOP = 4
TFIDF_TOP_RETRIEVAL = 100
TFIDF_TOP_FINAL = 50

BATCH_SIZE = 50_000

print("\nFrozen retrieval configuration:")
print("  Rare-token Top-K :", RARE_TOP)
print("  TF-IDF retrieval :", TFIDF_TOP_RETRIEVAL)
print("  TF-IDF final     :", TFIDF_TOP_FINAL)
print("  Batch size       :", BATCH_SIZE)

# ------------------------------------------------------------
# Required objects from Cell 1
# ------------------------------------------------------------

required = [
    "blocks",
    "rare_token_index",
    "token_df",
    "X_test_s1",
    "vectorizer",
    "test_s1_ids",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        "Rerun the verified Cell 1 first."
    )

print("\nAll frozen production artifacts are loaded.")

# ------------------------------------------------------------
# S1 entity -> local index
# ------------------------------------------------------------

entity_id_to_local = {
    eid: i
    for i, eid in enumerate(test_s1_ids)
}

print(
    "S1 local-index map:",
    f"{len(entity_id_to_local):,}"
)

assert len(entity_id_to_local) == 1_732_544

# ------------------------------------------------------------
# Exact normalization from notebook
# ------------------------------------------------------------

def normalize_exact_prod(x):

    if pd.isna(x):
        return ""

    return "".join(
        ch.lower()
        for ch in str(x)
        if ch.isalnum()
    )


def word_tokens_prod(text):

    if pd.isna(text):
        return []

    return [
        token
        for token in re.findall(
            r"[^\W_]+",
            str(text).lower(),
            flags=re.UNICODE
        )
        if len(token) >= 2
    ]


# ------------------------------------------------------------
# EXACT candidate retrieval
# ------------------------------------------------------------

def generate_exact_candidates(target_chunk):

    target_chunk = target_chunk.copy()

    # --------------------------------------------------------
    # Normalize exactly as validated notebook
    # --------------------------------------------------------

    target_chunk["name_n"] = (
        target_chunk["business_name"]
        .map(normalize_exact_prod)
    )

    target_chunk["addr_n"] = (
        target_chunk["business_address"]
        .map(normalize_exact_prod)
    )

    target_chunk["country_n"] = (
        target_chunk["country"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    target_chunk["combined_text"] = (
        target_chunk["name_n"]
        + " "
        + target_chunk["addr_n"]
    )

    # --------------------------------------------------------
    # Exact TF-IDF transformation
    # --------------------------------------------------------

    X_target = vectorizer.transform(
        target_chunk["combined_text"]
    )

    candidate_rows = []

    # --------------------------------------------------------
    # Process every target
    # --------------------------------------------------------

    for q_idx, row in enumerate(
        target_chunk.itertuples(index=False)
    ):

        country = row.country_n
        name = row.name_n
        addr = row.addr_n

        # ====================================================
        # CHANNEL 1 — CHARACTER BLOCKS
        # ====================================================

        candidates = set()

        if name:

            candidates.update(
                blocks["name_first"].get(
                    (country, name[0]),
                    []
                )
            )

            candidates.update(
                blocks["name_last"].get(
                    (country, name[-1]),
                    []
                )
            )

        if addr:

            candidates.update(
                blocks["addr_first"].get(
                    (country, addr[0]),
                    []
                )
            )

            candidates.update(
                blocks["addr_last"].get(
                    (country, addr[-1]),
                    []
                )
            )

        # ====================================================
        # CHANNEL 2 — NUMBER + COUNTRY
        # ====================================================

        if addr:

            for num in set(
                re.findall(
                    r"\d+",
                    addr
                )
            ):

                candidates.update(
                    blocks["num_country"].get(
                        (country, num),
                        []
                    )
                )

        # ====================================================
        # CHANNEL 3 — RARE TOKEN TOP-4
        # ====================================================

        tokens = set(
            word_tokens_prod(
                row.business_name
            )
            +
            word_tokens_prod(
                row.business_address
            )
        )

        usable_tokens = []

        for token in tokens:

            df = token_df.get(
                token,
                0
            )

            if (
                df > 0
                and
                df <= 1000
            ):

                usable_tokens.append(
                    (
                        df,
                        token
                    )
                )

        usable_tokens.sort(
            key=lambda x: x[0]
        )

        rare_candidates = set()

        for _, token in usable_tokens[:RARE_TOP]:

            # Saved production rare-token index stores
            # ENTITY IDs, not local indices.
            for eid in rare_token_index.get(
                (country, token),
                []
            ):

                local_idx = (
                    entity_id_to_local.get(eid)
                )

                if local_idx is not None:
                    rare_candidates.add(
                        local_idx
                    )

        # ====================================================
        # CHARACTER BLOCK -> TF-IDF TOP-100
        # ====================================================

        tfidf_candidates = set()

        if candidates:

            candidate_array = np.fromiter(
                candidates,
                dtype=np.int32
            )

            scores = (
                X_target[
                    q_idx:q_idx + 1
                ]
                @
                X_test_s1[
                    candidate_array
                ].T
            ).toarray().ravel()

            if len(scores) > TFIDF_TOP_RETRIEVAL:

                top = np.argpartition(
                    scores,
                    -TFIDF_TOP_RETRIEVAL
                )[-TFIDF_TOP_RETRIEVAL:]

                top = top[
                    np.argsort(
                        scores[top]
                    )[::-1]
                ]

            else:

                top = np.argsort(
                    scores
                )[::-1]

            # Validated notebook keeps TF-IDF Top-50
            # for the final candidate union.
            tfidf_candidates = set(
                candidate_array[
                    top[:TFIDF_TOP_FINAL]
                ].tolist()
            )

        # ====================================================
        # FINAL FROZEN UNION
        # ====================================================

        final_candidates = (
            rare_candidates
            |
            tfidf_candidates
        )

        target_id = str(
            row.entity_id
        )

        # ----------------------------------------------------
        # Emit target -> S1 edges
        # ----------------------------------------------------

        for local_idx in final_candidates:

            candidate_rows.append(
                {
                    "target_id": target_id,
                    "s1_local_idx": int(local_idx),
                    "s1_id": str(
                        test_s1_ids[
                            int(local_idx)
                        ]
                    ),
                }
            )

    return pd.DataFrame(
        candidate_rows,
        columns=[
            "target_id",
            "s1_local_idx",
            "s1_id",
        ]
    )


# ------------------------------------------------------------
# Production runner
# ------------------------------------------------------------

def run_source(
    source_name,
    source_path,
    output_dir
):

    print("\n" + "=" * 100)
    print(
        f"PROCESSING {source_name}"
    )
    print("=" * 100)

    existing = sorted(
        output_dir.glob("part_*.parquet")
    )

    completed = {
        int(
            p.stem.split("_")[1]
        )
        for p in existing
    }

    print(
        "Existing completed batches:",
        len(completed)
    )

    total_rows = 0
    total_edges = 0

    for batch_no, chunk in enumerate(
        pd.read_csv(
            source_path,
            sep="\t",
            dtype=str,
            chunksize=BATCH_SIZE
        )
    ):

        total_rows += len(chunk)

        output_file = (
            output_dir
            /
            f"part_{batch_no:05d}.parquet"
        )

        if batch_no in completed:

            print(
                f"[{source_name}] "
                f"batch {batch_no:05d} already exists"
            )

            # Count existing rows
            try:
                existing_df = pd.read_parquet(
                    output_file,
                    columns=[
                        "target_id",
                        "s1_local_idx"
                    ]
                )

                total_edges += len(
                    existing_df
                )

                del existing_df

            except Exception:
                pass

            continue

        t0 = time.time()

        result = generate_exact_candidates(
            chunk
        )

        result.to_parquet(
            output_file,
            index=False
        )

        elapsed = time.time() - t0

        edges = len(result)

        total_edges += edges

        qps = (
            len(chunk) / elapsed
            if elapsed > 0
            else 0
        )

        print(
            f"[{source_name}] "
            f"batch={batch_no:05d} "
            f"targets={len(chunk):,} "
            f"edges={edges:,} "
            f"speed={qps:,.1f} targets/s "
            f"time={elapsed/60:.2f} min"
        )

        del result
        del chunk

        gc.collect()

    print("\n" + "-" * 100)
    print(
        f"{source_name} COMPLETE"
    )
    print("-" * 100)

    print(
        "Target rows processed:",
        f"{total_rows:,}"
    )

    print(
        "Candidate edges:",
        f"{total_edges:,}"
    )

    print(
        "Output directory:",
        output_dir
    )

    return total_rows, total_edges


# ------------------------------------------------------------
# Run S2
# ------------------------------------------------------------

s2_rows, s2_edges = run_source(
    "S2",
    TEST_DIR / "test_source2.tsv",
    S2_OUT
)

print(
    "\nS2 production candidate generation finished."
)

CELL 2 — EXACT FROZEN PRODUCTION CANDIDATE GENERATION

Frozen retrieval configuration:
  Rare-token Top-K : 4
  TF-IDF retrieval : 100
  TF-IDF final     : 50
  Batch size       : 50000

All frozen production artifacts are loaded.
S1 local-index map: 1,732,544

PROCESSING S2
Existing completed batches: 0


In [ ]:
# CELL 4 — RE-VALIDATE THE ACTUAL DEPLOYABLE PIPELINE
# (rare-token-only retrieval, vectorized feature computation)
# Answers: (1) what F0.5 does the real production config actually get,
# (2) how fast is it, extrapolated to the full 10M targets.

import re, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from collections import Counter, defaultdict
from rapidfuzz import process, fuzz

print("=" * 100)
print("CELL 4 — RE-VALIDATE WITH RARE-TOKEN-ONLY RETRIEVAL")
print("=" * 100)

TRAIN_DIR = BASE / "DATASET" / "train"

def normalize_exact(x):
    if pd.isna(x):
        return ""
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())

def word_tokens(text):
    if pd.isna(text):
        return []
    return [t for t in re.findall(r"[^\W_]+", str(text).lower(), flags=re.UNICODE) if len(t) >= 2]

def number_overlap(a, b):
    if not a or not b:
        return 0
    return len(set(re.findall(r"\d+", str(a))) & set(re.findall(r"\d+", str(b))))

# --- Rebuild the exact 20k validation set (same random_state as CELL 3B) ---
print("\nRebuilding VAL_S1 (deterministic, random_state=42)...")
full_s1 = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", dtype=str)
VAL_S1 = full_s1.sample(n=20_000, random_state=42).copy()
del full_s1
VAL_S1["name_n"] = VAL_S1["business_name"].map(normalize_exact)
VAL_S1["addr_n"] = VAL_S1["business_address"].map(normalize_exact)
VAL_S1["country_n"] = VAL_S1["country"].fillna("").astype(str).str.strip().str.lower()
print(f"VAL_S1: {len(VAL_S1):,}")

print("\nLoading ground truth + recovering validation targets...")
gt = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", dtype=str)
gt_lookup = gt.set_index("source1_entity_id")["matched_entity_ids"].fillna("")
target_to_s1 = {}
for sid in VAL_S1["entity_id"]:
    value = gt_lookup.get(sid, "")
    if not value:
        continue
    for tid in str(value).split(","):
        tid = tid.strip()
        if tid:
            target_to_s1[tid] = sid
validation_target_ids = set(target_to_s1.keys())
print(f"Validation target IDs: {len(validation_target_ids):,}")

usecols = ["entity_id", "business_name", "business_address", "country"]
collected, remaining = [], set(validation_target_ids)
for source_name, path in [("S2", TRAIN_DIR / "train_source2.tsv"), ("S3", TRAIN_DIR / "train_source3.tsv")]:
    if not remaining:
        break
    for chunk in pd.read_csv(path, sep="\t", dtype=str, usecols=usecols, chunksize=250_000):
        hit = chunk[chunk["entity_id"].isin(remaining)]
        if len(hit):
            collected.append(hit.copy())
            remaining.difference_update(hit["entity_id"])
if remaining:
    raise RuntimeError(f"Could not recover {len(remaining):,} validation targets.")
VAL_TARGET_POOL = pd.concat(collected, ignore_index=True)
VAL_TARGET_POOL["name_n"] = VAL_TARGET_POOL["business_name"].map(normalize_exact)
VAL_TARGET_POOL["addr_n"] = VAL_TARGET_POOL["business_address"].map(normalize_exact)
VAL_TARGET_POOL["country_n"] = VAL_TARGET_POOL["country"].fillna("").astype(str).str.strip().str.lower()
print(f"VAL_TARGET_POOL: {len(VAL_TARGET_POOL):,} (expected 69,231)")

true_by_s1 = {sid: set() for sid in VAL_S1["entity_id"]}
for tid, sid in target_to_s1.items():
    if sid in true_by_s1:
        true_by_s1[sid].add(tid)

# --- VAL_S1-scale rare-token index (local_idx based) ---
print("\nBuilding VAL_S1 rare-token index...")
token_df_val = Counter()
for row in VAL_S1.itertuples(index=False):
    token_df_val.update(set(word_tokens(row.business_name) + word_tokens(row.business_address)))
token_index_val = defaultdict(list)
for idx, row in enumerate(VAL_S1.itertuples(index=False)):
    for tok in set(word_tokens(row.business_name) + word_tokens(row.business_address)):
        token_index_val[(row.country_n, tok)].append(idx)
print(f"Unique tokens: {len(token_df_val):,}  Token-country blocks: {len(token_index_val):,}")

s1_name = VAL_S1["name_n"].to_numpy()
s1_addr = VAL_S1["addr_n"].to_numpy()
s1_country = VAL_S1["country_n"].to_numpy()
s1_ids = VAL_S1["entity_id"].to_numpy()
MAX_DF, TOP_N = 1000, 4

matcher = lgb.Booster(model_file=str(FINAL_DIR / "final_lambdarank_model.txt"))
with open(FINAL_DIR / "final_lambdarank_threshold.txt") as f:
    threshold = float(f.read().strip())
feature_order = matcher.feature_name()
print(f"\nModel expects {len(feature_order)} features, threshold={threshold}")

rank_cols = ["name_ratio", "addr_ratio", "name_partial", "addr_partial", "combined_similarity", "tfidf_score"]
all_rows = []
t0 = time.time()
n_pairs = 0

for q_idx, row in enumerate(VAL_TARGET_POOL.itertuples(index=False)):
    q_name, q_addr, q_country = row.name_n, row.addr_n, row.country_n
    tokens = set(word_tokens(row.business_name) + word_tokens(row.business_address))
    usable = sorted([(token_df_val.get(t, 0), t) for t in tokens if 0 < token_df_val.get(t, 0) <= MAX_DF], key=lambda x: x[0])
    cands = set()
    for _, tok in usable[:TOP_N]:
        cands.update(token_index_val.get((q_country, tok), []))
    if not cands:
        continue
    cand_arr = np.fromiter(cands, dtype=np.int32)
    n_pairs += len(cand_arr)

    c_names, c_addrs, c_countries, c_ids = s1_name[cand_arr], s1_addr[cand_arr], s1_country[cand_arr], s1_ids[cand_arr]

    name_ratio = process.cdist([q_name], c_names, scorer=fuzz.ratio)[0] / 100.0
    addr_ratio = process.cdist([q_addr], c_addrs, scorer=fuzz.ratio)[0] / 100.0
    name_partial = process.cdist([q_name], c_names, scorer=fuzz.partial_ratio)[0] / 100.0
    addr_partial = process.cdist([q_addr], c_addrs, scorer=fuzz.partial_ratio)[0] / 100.0
    name_token = process.cdist([q_name], c_names, scorer=fuzz.token_set_ratio)[0] / 100.0
    addr_token = process.cdist([q_addr], c_addrs, scorer=fuzz.token_set_ratio)[0] / 100.0

    qn, qa = len(q_name), len(q_addr)
    sn = np.array([len(x) for x in c_names]); sa = np.array([len(x) for x in c_addrs])
    name_length_ratio = np.where(np.maximum(qn, sn) > 0, np.minimum(qn, sn) / np.maximum(qn, sn, 1), 1.0)
    addr_length_ratio = np.where(np.maximum(qa, sa) > 0, np.minimum(qa, sa) / np.maximum(qa, sa, 1), 1.0)
    num_overlap = np.array([number_overlap(q_addr, a) for a in c_addrs], dtype=np.float64)
    exact_name = (c_names == q_name).astype(int)
    exact_address = (c_addrs == q_addr).astype(int)
    country_match = (c_countries == q_country).astype(int)
    combined_similarity = 0.45 * name_ratio + 0.55 * addr_ratio
    has_rare = np.ones(len(cand_arr), dtype=int)
    has_tfidf = np.zeros(len(cand_arr), dtype=int)
    tfidf_score = np.zeros(len(cand_arr)); tfidf_rank = np.full(len(cand_arr), 999)
    retrieval_channel_count = has_rare + has_tfidf
    name_addr_product = name_ratio * addr_ratio
    name_partial_addr_partial_product = name_partial * addr_partial
    name_number_product = name_ratio * num_overlap
    addr_number_product = addr_ratio * num_overlap
    min_name_addr = np.minimum(name_ratio, addr_ratio); max_name_addr = np.maximum(name_ratio, addr_ratio)
    name_addr_gap = np.abs(name_ratio - addr_ratio); partial_gap = np.abs(name_partial - addr_partial)
    strong_name_weak_address = ((name_ratio >= 0.80) & (addr_ratio < 0.20)).astype(int)
    strong_name_no_number = ((name_ratio >= 0.80) & (num_overlap == 0)).astype(int)
    strong_address = (addr_ratio >= 0.70).astype(int)
    strong_name = (name_ratio >= 0.80).astype(int)
    address_supported = (addr_ratio >= 0.50).astype(int)
    multi_channel = (retrieval_channel_count >= 2).astype(int)
    balanced_similarity = (name_ratio + addr_ratio) / 2.0
    partial_balanced_similarity = (name_partial + addr_partial) / 2.0

    df_q = pd.DataFrame({
        "query_id": row.entity_id, "s1_id": c_ids,
        "name_ratio": name_ratio, "addr_ratio": addr_ratio, "name_partial": name_partial, "addr_partial": addr_partial,
        "name_token": name_token, "addr_token": addr_token,
        "name_length_ratio": name_length_ratio, "addr_length_ratio": addr_length_ratio,
        "number_overlap": num_overlap, "exact_name": exact_name, "exact_address": exact_address,
        "country_match": country_match, "combined_similarity": combined_similarity,
        "has_rare": has_rare, "has_tfidf": has_tfidf, "retrieval_channel_count": retrieval_channel_count,
        "tfidf_score": tfidf_score, "tfidf_rank": tfidf_rank,
        "name_addr_product": name_addr_product, "name_partial_addr_partial_product": name_partial_addr_partial_product,
        "name_number_product": name_number_product, "addr_number_product": addr_number_product,
        "min_name_addr": min_name_addr, "max_name_addr": max_name_addr,
        "name_addr_gap": name_addr_gap, "partial_gap": partial_gap,
        "strong_name_weak_address": strong_name_weak_address, "strong_name_no_number": strong_name_no_number,
        "strong_address": strong_address, "strong_name": strong_name,
        "address_supported": address_supported, "multi_channel": multi_channel,
        "balanced_similarity": balanced_similarity, "partial_balanced_similarity": partial_balanced_similarity,
    })
    for col in rank_cols:
        df_q[f"rank_{col}"] = df_q[col].rank(method="min", ascending=False)
        df_q[f"query_max_{col}"] = df_q[col].max()
    df_q["tfidf_margin"] = df_q["query_max_tfidf_score"] - df_q["tfidf_score"]
    df_q["name_margin"] = df_q["query_max_name_ratio"] - df_q["name_ratio"]
    df_q["addr_margin"] = df_q["query_max_addr_ratio"] - df_q["addr_ratio"]
    df_q["combined_margin"] = df_q["query_max_combined_similarity"] - df_q["combined_similarity"]
    df_q["relative_tfidf_strength"] = df_q["tfidf_score"] / (df_q["query_max_tfidf_score"] + 1e-8)
    df_q["relative_combined_strength"] = df_q["combined_similarity"] / (df_q["query_max_combined_similarity"] + 1e-8)
    df_q["is_best_tfidf"] = (df_q["rank_tfidf_score"] == 1).astype(np.int8)
    df_q["is_best_name"] = (df_q["rank_name_ratio"] == 1).astype(np.int8)
    df_q["is_best_addr"] = (df_q["rank_addr_ratio"] == 1).astype(np.int8)
    df_q["is_best_combined"] = (df_q["rank_combined_similarity"] == 1).astype(np.int8)
    all_rows.append(df_q)

    if (q_idx + 1) % 5000 == 0:
        elapsed = time.time() - t0
        print(f"{q_idx+1:,}/{len(VAL_TARGET_POOL):,} | speed={(q_idx+1)/elapsed:.1f} q/s | pairs/s={n_pairs/elapsed:,.0f}")

elapsed = time.time() - t0
qps = len(VAL_TARGET_POOL) / elapsed
print(f"\nRetrieval+features: {elapsed/60:.2f} min | {qps:.1f} q/s | {n_pairs:,} total pairs")
print(f"Projected for 9,969,589 targets: {9_969_589/qps/3600:.2f} hours")

df_all = pd.concat(all_rows, ignore_index=True)
X = df_all.reindex(columns=feature_order, fill_value=0)
df_all["model_score"] = matcher.predict(X)

best = df_all.sort_values("model_score", ascending=False).groupby("query_id", as_index=False).first()
pred_by_s1 = defaultdict(set)
for r in best.itertuples(index=False):
    if r.model_score >= threshold:
        pred_by_s1[r.s1_id].add(r.query_id)

f05s = []
for sid in VAL_S1["entity_id"]:
    true_set = true_by_s1.get(sid, set())
    pred_set = pred_by_s1.get(sid, set())
    if not true_set and not pred_set:
        f05s.append(1.0); continue
    if not true_set and pred_set:
        f05s.append(0.0); continue
    tp = len(true_set & pred_set); fp = len(pred_set - true_set); fn = len(true_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f05s.append((1.25 * prec * rec) / (0.25 * prec + rec) if (prec + rec) > 0 else 0.0)

macro_f05 = float(np.mean(f05s))
print(f"\n{'='*100}\nRARE-TOKEN-ONLY MACRO F0.5 (full 20k S1): {macro_f05:.6f}\n{'='*100}")
print("(compare to 0.988477 with rare+TF-IDF union)")
print("\nCELL 4 COMPLETE — waiting for output.")

CELL 4 — RE-VALIDATE WITH RARE-TOKEN-ONLY RETRIEVAL


NameError: name 'BASE' is not defined

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026')

Mounted at /content/drive


In [3]:
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.8 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

BASE = Path('/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026')

In [6]:
import re, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from collections import Counter, defaultdict
from rapidfuzz import process, fuzz
from pathlib import Path

print("=" * 100)
print("CELL 4 — RE-VALIDATE WITH RARE-TOKEN-ONLY RETRIEVAL")
print("=" * 100)

# Define BASE directory
BASE = Path('/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026')

# Define FINAL_DIR as it is used to load the model
FINAL_DIR = BASE / "MODELS" / "FINAL_SAVED_ARTIFACTS"

TRAIN_DIR = BASE / "DATASET" / "train"

def normalize_exact(x):
    if pd.isna(x):
        return ""
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())

def word_tokens(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    if not text: # Add a check for empty string after conversion
        return []
    return [t for t in re.findall(r"[^\W_]+", text, flags=re.UNICODE) if len(t) >= 2]

def number_overlap(a, b):
    if not a or not b:
        return 0
    # Corrected regex to match actual digits
    return len(set(re.findall(r"\d+", str(a))) & set(re.findall(r"\d+", str(b))))

# --- Rebuild the exact 20k validation set (same random_state as CELL 3B) ---
print("\nRebuilding VAL_S1 (deterministic, random_state=42)...")
full_s1 = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", dtype=str)
VAL_S1 = full_s1.sample(n=20_000, random_state=42).copy()
del full_s1
VAL_S1["name_n"] = VAL_S1["business_name"].map(normalize_exact)
VAL_S1["addr_n"] = VAL_S1["business_address"].map(normalize_exact)
VAL_S1["country_n"] = VAL_S1["country"].fillna("").astype(str).str.strip().str.lower()
print(f"VAL_S1: {len(VAL_S1):,}")

print("\nLoading ground truth + recovering validation targets...")
gt = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", dtype=str)
gt_lookup = gt.set_index("source1_entity_id")["matched_entity_ids"].fillna("")
target_to_s1 = {}
for sid in VAL_S1["entity_id"]:
    value = gt_lookup.get(sid, "")
    if not value:
        continue
    for tid in str(value).split(","):
        tid = tid.strip()
        if tid:
            target_to_s1[tid] = sid
validation_target_ids = set(target_to_s1.keys())
print(f"Validation target IDs: {len(validation_target_ids):,}")

usecols = ["entity_id", "business_name", "business_address", "country"]
collected, remaining = [], set(validation_target_ids)
for source_name, path in [("S2", TRAIN_DIR / "train_source2.tsv"), ("S3", TRAIN_DIR / "train_source3.tsv")]:
    if not remaining:
        break
    for chunk in pd.read_csv(path, sep="\t", dtype=str, usecols=usecols, chunksize=250_000):
        hit = chunk[chunk["entity_id"].isin(remaining)]
        if len(hit):
            collected.append(hit.copy())
            remaining.difference_update(hit["entity_id"])
if remaining:
    raise RuntimeError(f"Could not recover {len(remaining):,} validation targets.")
VAL_TARGET_POOL = pd.concat(collected, ignore_index=True)
VAL_TARGET_POOL["name_n"] = VAL_TARGET_POOL["business_name"].map(normalize_exact)
VAL_TARGET_POOL["addr_n"] = VAL_TARGET_POOL["business_address"].map(normalize_exact)
VAL_TARGET_POOL["country_n"] = VAL_TARGET_POOL["country"].fillna("").astype(str).str.strip().str.lower()
print(f"VAL_TARGET_POOL: {len(VAL_TARGET_POOL):,} (expected 69,231)")

true_by_s1 = {sid: set() for sid in VAL_S1["entity_id"]}
for tid, sid in target_to_s1.items():
    if sid in true_by_s1:
        true_by_s1[sid].add(tid)

# --- VAL_S1-scale rare-token index (local_idx based) ---
print("\nBuilding VAL_S1 rare-token index...")
token_df_val = Counter()
for row in VAL_S1.itertuples(index=False):
    token_df_val.update(set(word_tokens(row.business_name) + word_tokens(row.business_address)))
token_index_val = defaultdict(list)
for idx, row in enumerate(VAL_S1.itertuples(index=False)):
    for tok in set(word_tokens(row.business_name) + word_tokens(row.business_address)):
        token_index_val[(row.country_n, tok)].append(idx)
print(f"Unique tokens: {len(token_df_val):,}  Token-country blocks: {len(token_index_val):,}")

s1_name = VAL_S1["name_n"].to_numpy()
s1_addr = VAL_S1["addr_n"].to_numpy()
s1_country = VAL_S1["country_n"].to_numpy()
s1_ids = VAL_S1["entity_id"].to_numpy()
MAX_DF, TOP_N = 1000, 4

matcher = lgb.Booster(model_file=str(FINAL_DIR / "final_lambdarank_model.txt"))
with open(FINAL_DIR / "final_lambdarank_threshold.txt") as f:
    threshold = float(f.read().strip())
feature_order = matcher.feature_name()
print(f"\nModel expects {len(feature_order)} features, threshold={threshold}")

# Assert the number of trees as requested by the user
assert matcher.num_trees() == 424, f"Expected 424 trees, but model has {matcher.num_trees()} trees."
print(f"Model has {matcher.num_trees()} trees (verified).")

BATCH_SIZE = 1000 # Process 1000 queries at a time

rank_cols = ["name_ratio", "addr_ratio", "name_partial", "addr_partial", "combined_similarity", "tfidf_score"]
all_batch_results = []
t0 = time.time()
n_pairs = 0

for batch_start_idx in range(0, len(VAL_TARGET_POOL), BATCH_SIZE):
    batch_end_idx = min(batch_start_idx + BATCH_SIZE, len(VAL_TARGET_POOL))
    current_batch_queries = VAL_TARGET_POOL.iloc[batch_start_idx:batch_end_idx]

    batch_candidate_data = []
    n_pairs_batch = 0

    for row in current_batch_queries.itertuples(index=False):
        q_name, q_addr, q_country = row.name_n, row.addr_n, row.country_n

        # Tokenization and candidate retrieval (per-query)
        tokens = set(word_tokens(row.business_name) + word_tokens(row.business_address))
        usable = sorted([(token_df_val.get(t, 0), t) for t in tokens if 0 < token_df_val.get(t, 0) <= MAX_DF], key=lambda x: x[0])
        cands = set()
        for _, tok in usable[:TOP_N]:
            cands.update(token_index_val.get((q_country, tok), []))

        if not cands:
            continue

        cand_arr = np.fromiter(cands, dtype=np.int32)
        n_pairs_batch += len(cand_arr)

        c_names, c_addrs, c_countries, c_ids = s1_name[cand_arr], s1_addr[cand_arr], s1_country[cand_arr], s1_ids[cand_arr]

        # Feature calculation (fuzz ratios) (per-query)
        name_ratio = process.cdist([q_name], c_names, scorer=fuzz.ratio)[0] / 100.0
        addr_ratio = process.cdist([q_addr], c_addrs, scorer=fuzz.ratio)[0] / 100.0
        name_partial = process.cdist([q_name], c_names, scorer=fuzz.partial_ratio)[0] / 100.0
        addr_partial = process.cdist([q_addr], c_addrs, scorer=fuzz.partial_ratio)[0] / 100.0
        name_token = process.cdist([q_name], c_names, scorer=fuzz.token_set_ratio)[0] / 100.0
        addr_token = process.cdist([q_addr], c_addrs, scorer=fuzz.token_set_ratio)[0] / 100.0

        # Feature calculation (numpy ops) (per-query)
        qn, qa = len(q_name), len(q_addr)
        sn = np.array([len(x) for x in c_names]); sa = np.array([len(x) for x in c_addrs])
        name_length_ratio = np.where(np.maximum(qn, sn) > 0, np.minimum(qn, sn) / np.maximum(qn, sn), 1.0)
        addr_length_ratio = np.where(np.maximum(qa, sa) > 0, np.minimum(qa, sa) / np.maximum(qa, sa), 1.0)
        num_overlap = np.array([number_overlap(q_addr, a) for a in c_addrs], dtype=np.float64)
        exact_name = (c_names == q_name).astype(int)
        exact_address = (c_addrs == q_addr).astype(int)
        country_match = (c_countries == q_country).astype(int)
        combined_similarity = 0.45 * name_ratio + 0.55 * addr_ratio
        has_rare = np.ones(len(cand_arr), dtype=int)
        has_tfidf = np.zeros(len(cand_arr), dtype=int)
        tfidf_score = np.zeros(len(cand_arr)); tfidf_rank = np.full(len(cand_arr), 999) # Assuming no TF-IDF channel used for now
        retrieval_channel_count = has_rare + has_tfidf
        name_addr_product = name_ratio * addr_ratio
        name_partial_addr_partial_product = name_partial * addr_partial
        name_number_product = name_ratio * num_overlap
        addr_number_product = addr_ratio * num_overlap
        min_name_addr = np.minimum(name_ratio, addr_ratio); max_name_addr = np.maximum(name_ratio, addr_ratio)
        name_addr_gap = np.abs(name_ratio - addr_ratio); partial_gap = np.abs(name_partial - addr_partial)
        strong_name_weak_address = ((name_ratio >= 0.80) & (addr_ratio < 0.20)).astype(int)
        strong_name_no_number = ((name_ratio >= 0.80) & (num_overlap == 0)).astype(int)
        strong_address = (addr_ratio >= 0.70).astype(int)
        strong_name = (name_ratio >= 0.80).astype(int)
        address_supported = (addr_ratio >= 0.50).astype(int)
        multi_channel = (retrieval_channel_count >= 2).astype(int)
        balanced_similarity = (name_ratio + addr_ratio) / 2.0
        partial_balanced_similarity = (name_partial + addr_partial) / 2.0

        # Collect data for the batch DataFrame
        for i in range(len(cand_arr)):
            batch_candidate_data.append({
                "query_id": row.entity_id,
                "s1_id": c_ids[i],
                "name_ratio": name_ratio[i],
                "addr_ratio": addr_ratio[i],
                "name_partial": name_partial[i],
                "addr_partial": addr_partial[i],
                "name_token": name_token[i],
                "addr_token": addr_token[i],
                "name_length_ratio": name_length_ratio[i],
                "addr_length_ratio": addr_length_ratio[i],
                "number_overlap": num_overlap[i],
                "exact_name": exact_name[i],
                "exact_address": exact_address[i],
                "country_match": country_match[i],
                "combined_similarity": combined_similarity[i],
                "has_rare": has_rare[i],
                "has_tfidf": has_tfidf[i],
                "retrieval_channel_count": retrieval_channel_count[i],
                "tfidf_score": tfidf_score[i],
                "tfidf_rank": tfidf_rank[i],
                "name_addr_product": name_addr_product[i],
                "name_partial_addr_partial_product": name_partial_addr_partial_product[i],
                "name_number_product": name_number_product[i],
                "addr_number_product": addr_number_product[i],
                "min_name_addr": min_name_addr[i],
                "max_name_addr": max_name_addr[i],
                "name_addr_gap": name_addr_gap[i],
                "partial_gap": partial_gap[i],
                "strong_name_weak_address": strong_name_weak_address[i],
                "strong_name_no_number": strong_name_no_number[i],
                "strong_address": strong_address[i],
                "strong_name": strong_name[i],
                "address_supported": address_supported[i],
                "multi_channel": multi_channel[i],
                "balanced_similarity": balanced_similarity[i],
                "partial_balanced_similarity": partial_balanced_similarity[i],
            })

    if not batch_candidate_data:
        continue

    # Create DataFrame for the current batch
    df_batch = pd.DataFrame(batch_candidate_data)

    # Compute ranking features vectorized per batch
    for col in rank_cols:
        # Ensure the column exists before calling rank or transform
        if col in df_batch.columns:
            df_batch[f"rank_{col}"] = df_batch.groupby('query_id')[col].rank(method="min", ascending=False)
            df_batch[f"query_max_{col}"] = df_batch.groupby('query_id')[col].transform('max')
        else:
            # Handle cases where the column might be missing if it's a dummy for TF-IDF
            df_batch[f"rank_{col}"] = 999  # Assign a default rank
            df_batch[f"query_max_{col}"] = 0.0 # Assign a default max

    df_batch["tfidf_margin"] = df_batch["query_max_tfidf_score"] - df_batch["tfidf_score"]
    df_batch["name_margin"] = df_batch["query_max_name_ratio"] - df_batch["name_ratio"]
    df_batch["addr_margin"] = df_batch["query_max_addr_ratio"] - df_batch["addr_ratio"]
    df_batch["combined_margin"] = df_batch["query_max_combined_similarity"] - df_batch["combined_similarity"]
    df_batch["relative_tfidf_strength"] = df_batch["tfidf_score"] / (df_batch["query_max_tfidf_score"] + 1e-8)
    df_batch["relative_combined_strength"] = df_batch["combined_similarity"] / (df_batch["query_max_combined_similarity"] + 1e-8)
    df_batch["is_best_tfidf"] = (df_batch["rank_tfidf_score"] == 1).astype(np.int8)
    df_batch["is_best_name"] = (df_batch["rank_name_ratio"] == 1).astype(np.int8)
    df_batch["is_best_addr"] = (df_batch["rank_addr_ratio"] == 1).astype(np.int8)
    df_batch["is_best_combined"] = (df_batch["rank_combined_similarity"] == 1).astype(np.int8)

    # Model prediction (once per batch)
    X_batch = df_batch.reindex(columns=feature_order, fill_value=0)
    df_batch["model_score"] = matcher.predict(X_batch)

    all_batch_results.append(df_batch)
    n_pairs += n_pairs_batch # Renamed from n_pairs_total to n_pairs to match error message

    # Progress update
    current_processed_queries = batch_end_idx
    elapsed = time.time() - t0
    qps = current_processed_queries / elapsed
    print(f"{current_processed_queries:,}/{len(VAL_TARGET_POOL):,} | speed={qps:.1f} q/s | pairs/s={n_pairs/elapsed:,.0f}")

elapsed = time.time() - t0
qps = len(VAL_TARGET_POOL) / elapsed
print(f"\nRetrieval+features+model_prediction: {elapsed/60:.2f} min | {qps:.1f} q/s | {n_pairs:,} total pairs")
print(f"Projected for 9,969,589 targets: {9_969_589/qps/3600:.2f} hours")

df_all = pd.concat(all_batch_results, ignore_index=True)

best = df_all.sort_values("model_score", ascending=False).groupby("query_id", as_index=False).first()
pred_by_s1 = defaultdict(set)
for r in best.itertuples(index=False):
    if r.model_score >= threshold:
        pred_by_s1[r.s1_id].add(r.query_id)

f05s = []
for sid in VAL_S1["entity_id"]:
    true_set = true_by_s1.get(sid, set())
    pred_set = pred_by_s1.get(sid, set())
    if not true_set and not pred_set:
        f05s.append(1.0); continue
    if not true_set and pred_set:
        f05s.append(0.0); continue
    tp = len(true_set & pred_set); fp = len(pred_set - true_set); fn = len(true_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f05s.append((1.25 * prec * rec) / (0.25 * prec + rec) if (prec + rec) > 0 else 0.0)

macro_f05 = float(np.mean(f05s))
print(f"\n{'='*100}\nRARE-TOKEN-ONLY MACRO F0.5 (full 20k S1): {macro_f05:.6f}\n{'='*100}")
print("(compare to 0.988477 with rare+TF-IDF union)")
print("\nCELL 4 COMPLETE — waiting for output.")


CELL 4 — RE-VALIDATE WITH RARE-TOKEN-ONLY RETRIEVAL

Rebuilding VAL_S1 (deterministic, random_state=42)...
VAL_S1: 20,000

Loading ground truth + recovering validation targets...
Validation target IDs: 69,231
VAL_TARGET_POOL: 69,231 (expected 69,231)

Building VAL_S1 rare-token index...
Unique tokens: 35,553  Token-country blocks: 38,381

Model expects 56 features, threshold=0.6280000000000001
Model has 424 trees (verified).
1,000/69,231 | speed=129.5 q/s | pairs/s=10,964
2,000/69,231 | speed=153.7 q/s | pairs/s=13,045
3,000/69,231 | speed=141.0 q/s | pairs/s=11,588
4,000/69,231 | speed=147.3 q/s | pairs/s=12,379
5,000/69,231 | speed=151.7 q/s | pairs/s=12,885
6,000/69,231 | speed=149.9 q/s | pairs/s=12,785
7,000/69,231 | speed=154.4 q/s | pairs/s=13,159
8,000/69,231 | speed=154.5 q/s | pairs/s=12,972
9,000/69,231 | speed=158.5 q/s | pairs/s=13,222
10,000/69,231 | speed=162.3 q/s | pairs/s=13,417
11,000/69,231 | speed=160.7 q/s | pairs/s=13,228
12,000/69,231 | speed=160.8 q/s | pairs/s

In [7]:
retrieved_pairs = set(zip(df_all["query_id"], df_all["s1_id"]))
total_true = sum(len(v) for v in true_by_s1.values())
never_retrieved = sum(
    1 for sid, qs in true_by_s1.items() for qid in qs
    if (qid, sid) not in retrieved_pairs
)
print(f"Never retrieved: {never_retrieved:,} / {total_true:,} ({100*never_retrieved/total_true:.2f}%)")

Never retrieved: 51 / 69,231 (0.07%)


### Diagnostic 1: Inspect `df_all` (first 5 rows with all columns)

In [ ]:
print('Shape of df_all:', df_all.shape)
print('Columns in df_all:', df_all.columns.tolist())
print('\nFirst 5 rows of df_all (including all feature columns):')
# Temporarily set display.max_columns to None to show all columns
with pd.option_context('display.max_columns', None):
    display(df_all.head(5))

Shape of df_all: (4929556, 59)
Columns in df_all: ['query_id', 's1_id', 'name_ratio', 'addr_ratio', 'name_partial', 'addr_partial', 'name_token', 'addr_token', 'name_length_ratio', 'addr_length_ratio', 'number_overlap', 'exact_name', 'exact_address', 'country_match', 'combined_similarity', 'has_rare', 'has_tfidf', 'retrieval_channel_count', 'tfidf_score', 'tfidf_rank', 'name_addr_product', 'name_partial_addr_partial_product', 'name_number_product', 'addr_number_product', 'min_name_addr', 'max_name_addr', 'name_addr_gap', 'partial_gap', 'strong_name_weak_address', 'strong_name_no_number', 'strong_address', 'strong_name', 'address_supported', 'multi_channel', 'balanced_similarity', 'partial_balanced_similarity', 'rank_name_ratio', 'query_max_name_ratio', 'rank_addr_ratio', 'query_max_addr_ratio', 'rank_name_partial', 'query_max_name_partial', 'rank_addr_partial', 'query_max_addr_partial', 'rank_combined_similarity', 'query_max_combined_similarity', 'rank_tfidf_score', 'query_max_tfidf_sc

,query_id,s1_id,name_ratio,addr_ratio,name_partial,addr_partial,name_token,addr_token,name_length_ratio,addr_length_ratio,number_overlap,exact_name,exact_address,country_match,combined_similarity,has_rare,has_tfidf,retrieval_channel_count,tfidf_score,tfidf_rank,name_addr_product,name_partial_addr_partial_product,name_number_product,addr_number_product,min_name_addr,max_name_addr,name_addr_gap,partial_gap,strong_name_weak_address,strong_name_no_number,strong_address,strong_name,address_supported,multi_channel,balanced_similarity,partial_balanced_similarity,rank_name_ratio,query_max_name_ratio,rank_addr_ratio,query_max_addr_ratio,rank_name_partial,query_max_name_partial,rank_addr_partial,query_max_addr_partial,rank_combined_similarity,query_max_combined_similarity,rank_tfidf_score,query_max_tfidf_score,tfidf_margin,name_margin,addr_margin,combined_margin,relative_tfidf_strength,relative_combined_strength,is_best_tfidf,is_best_name,is_best_addr,is_best_combined,model_score
0,S2-254788212,S1-94595653,0.277778,0.218750,0.307692,0.270270,0.277778,0.218750,0.565217,0.729730,0.0,0,0,1,0.245313,1,0,1,0.0,999,0.060764,0.083160,0.0,0.0,0.218750,0.277778,0.059028,0.037422,0,0,0,0,0,0,0.248264,0.288981,3.0,0.8,2.0,0.480000,4.0,0.857143,2.0,0.545455,3.0,0.624000,1.0,0.0,0.0,0.522222,0.261250,0.378687,0.0,0.393129,1,0,0,0,0.000604
1,S2-254788212,S1-162499446,0.270270,0.173913,0.333333,0.260870,0.270270,0.173913,0.608696,0.703704,0.0,0,0,1,0.217274,1,0,1,0.0,999,0.047004,0.086957,0.0,0.0,0.173913,0.270270,0.096357,0.072464,0,0,0,0,0,0,0.222092,0.297101,4.0,0.8,4.0,0.480000,3.0,0.857143,3.0,0.545455,4.0,0.624000,1.0,0.0,0.0,0.529730,0.306087,0.406726,0.0,0.348195,1,0,0,0,0.000610
2,S2-254788212,S1-494930410,0.800000,0.480000,0.857143,0.545455,0.800000,0.480000,0.956522,0.851852,0.0,0,0,1,0.624000,1,0,1,0.0,999,0.384000,0.467532,0.0,0.0,0.480000,0.800000,0.320000,0.311688,0,1,0,1,0,0,0.640000,0.701299,1.0,0.8,1.0,0.480000,1.0,0.857143,1.0,0.545455,1.0,0.624000,1.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,1.000000,1,1,1,1,0.808984
3,S2-254788212,S1-350732651,0.333333,0.214286,0.424242,0.250000,0.333333,0.214286,0.826087,0.931034,0.0,0,0,1,0.267857,1,0,1,0.0,999,0.071429,0.106061,0.0,0.0,0.214286,0.333333,0.119048,0.174242,0,0,0,0,0,0,0.273810,0.337121,2.0,0.8,3.0,0.480000,2.0,0.857143,4.0,0.545455,2.0,0.624000,1.0,0.0,0.0,0.466667,0.265714,0.356143,0.0,0.429258,1,0,0,0,0.000426
4,S2-435778329,S1-287773614,0.347826,0.348624,0.400000,0.363636,0.347826,0.348624,0.642857,0.946429,0.0,0,0,1,0.348265,1,0,1,0.0,999,0.121260,0.145455,0.0,0.0,0.347826,0.348624,0.000798,0.036364,0,0,0,0,0,0,0.348225,0.381818,46.0,0.7,47.0,0.901961,52.0,0.800000,59.0,0.943820,54.0,0.811078,1.0,0.0,0.0,0.352174,0.553337,0.462814,0.0,0.429385,1,0,0,0,0.000450


### Diagnostic 2: Analyze 3 specific S1 entities

I will select three `S1` entities that have known matches in `true_by_s1` and analyze their predictions to see if the true matches are being scored low, are missing from candidates, or are being attributed incorrectly.

In [ ]:
# Select 3 S1 entities with known true matches for inspection
# Ensure these S1s are in VAL_S1 and have associated true targets
s1_entities_to_inspect = []
for s1_id in VAL_S1['entity_id'].head(100):
    if s1_id in true_by_s1 and len(true_by_s1[s1_id]) > 0:
        s1_entities_to_inspect.append(s1_id)
    if len(s1_entities_to_inspect) == 3:
        break

if not s1_entities_to_inspect:
    print("Could not find 3 S1 entities with true matches in true_by_s1 for inspection.")
else:
    for s1_id_to_inspect in s1_entities_to_inspect:
        print(f"\n--- Analyzing S1: {s1_id_to_inspect} ---")

        # Get true query IDs for this S1
        true_query_ids = list(true_by_s1.get(s1_id_to_inspect, []))
        print(f"True matching Query IDs for S1 {s1_id_to_inspect}: {true_query_ids}")

        # Get predicted query IDs for this S1 (if any predictions for this S1 exist above threshold)
        predicted_query_ids = list(pred_by_s1.get(s1_id_to_inspect, []))
        print(f"Predicted Query IDs for S1 {s1_id_to_inspect} (above threshold): {predicted_query_ids}")

        if true_query_ids:
            for q_id in true_query_ids:
                print(f"\n  -- Candidates for Query ID: {q_id} (True S1: {s1_id_to_inspect}) --")
                # Get all candidates generated for this specific query_id
                candidates_for_q = df_all[df_all['query_id'] == q_id].copy()
                if candidates_for_q.empty:
                    print(f"    No candidates found in df_all for Query ID {q_id}.")
                    continue

                # Mark true match and sort by model_score
                candidates_for_q['is_true_match'] = (candidates_for_q['s1_id'] == s1_id_to_inspect)
                candidates_for_q_sorted = candidates_for_q.sort_values(by='model_score', ascending=False)

                print(f"    Top candidates for Query ID {q_id}:")
                display(candidates_for_q_sorted[['query_id', 's1_id', 'model_score', 'is_true_match']].head(10))

                # Check if the true match is present in the candidates for this query
                true_match_row = candidates_for_q_sorted[candidates_for_q_sorted['is_true_match']]
                if true_match_row.empty:
                    print(f"    WARNING: True S1 {s1_id_to_inspect} is NOT in the candidate pool for Query ID {q_id}!")
                else:
                    print(f"    True S1 {s1_id_to_inspect} is at rank {true_match_row['model_score'].rank(method='min', ascending=False).iloc[0]:.0f} with score {true_match_row['model_score'].iloc[0]:.6f}")
        else:
            print(f"  No true queries found for S1 {s1_id_to_inspect} in true_by_s1.")


--- Analyzing S1: S1-53356671 ---
True matching Query IDs for S1 S1-53356671: ['S2-904412198', 'S2-453334141', 'S2-256033433', 'S3-468658814', 'S3-996978203']
Predicted Query IDs for S1 S1-53356671 (above threshold): ['S2-904412198', 'S2-453334141', 'S3-996978203']

  -- Candidates for Query ID: S2-904412198 (True S1: S1-53356671) --
    Top candidates for Query ID S2-904412198:


,query_id,s1_id,model_score,is_true_match
2014242,S2-904412198,S1-53356671,0.864240,True
2014399,S2-904412198,S1-250070625,0.001903,False
2014305,S2-904412198,S1-424709576,0.001061,False
2014312,S2-904412198,S1-265078621,0.001036,False
2014397,S2-904412198,S1-529710246,0.001026,False
2014334,S2-904412198,S1-186816137,0.001014,False
2014384,S2-904412198,S1-13119483,0.001010,False
2014413,S2-904412198,S1-758577091,0.000991,False
2014383,S2-904412198,S1-709602089,0.000945,False
2014348,S2-904412198,S1-506895037,0.000933,False


    True S1 S1-53356671 is at rank 1 with score 0.864240

  -- Candidates for Query ID: S2-453334141 (True S1: S1-53356671) --
    Top candidates for Query ID S2-453334141:


,query_id,s1_id,model_score,is_true_match
2365624,S2-453334141,S1-53356671,0.858482,True
2365781,S2-453334141,S1-250070625,0.005051,False
2365716,S2-453334141,S1-186816137,0.004080,False
2365694,S2-453334141,S1-265078621,0.003889,False
2365809,S2-453334141,S1-177244042,0.001854,False
2365795,S2-453334141,S1-758577091,0.001493,False
2365790,S2-453334141,S1-121722706,0.001278,False
2365766,S2-453334141,S1-13119483,0.001229,False
2365710,S2-453334141,S1-201380767,0.001206,False
2365784,S2-453334141,S1-419470037,0.001163,False


    True S1 S1-53356671 is at rank 1 with score 0.858482

  -- Candidates for Query ID: S2-256033433 (True S1: S1-53356671) --
    Top candidates for Query ID S2-256033433:


...

    True S1 S1-938947364 is at rank 1 with score 0.137211

  -- Candidates for Query ID: S3-544960509 (True S1: S1-938947364) --
    Top candidates for Query ID S3-544960509:


,query_id,s1_id,model_score,is_true_match
4429210,S3-544960509,S1-938947364,0.891648,True
4429221,S3-544960509,S1-355343805,0.000793,False
4429222,S3-544960509,S1-223129814,0.000629,False
4429220,S3-544960509,S1-579173032,0.000581,False
4429227,S3-544960509,S1-300508918,0.000576,False
4429212,S3-544960509,S1-164550526,0.000558,False
4429214,S3-544960509,S1-175990427,0.000551,False
4429213,S3-544960509,S1-443176617,0.000522,False
4429218,S3-544960509,S1-152936122,0.000460,False
4429217,S3-544960509,S1-412605815,0.000423,False


    True S1 S1-938947364 is at rank 1 with score 0.891648

  -- Candidates for Query ID: S3-268060208 (True S1: S1-938947364) --
    Top candidates for Query ID S3-268060208:


,query_id,s1_id,model_score,is_true_match
3320765,S3-268060208,S1-938947364,0.856951,True
3320767,S3-268060208,S1-152936122,0.001132,False
3320773,S3-268060208,S1-223129814,0.001103,False
3320764,S3-268060208,S1-46113693,0.000803,False
3320768,S3-268060208,S1-643245576,0.000803,False
3320778,S3-268060208,S1-355343805,0.000667,False
3320771,S3-268060208,S1-175990427,0.000652,False
3320776,S3-268060208,S1-973182858,0.000576,False
3320766,S3-268060208,S1-300508918,0.000576,False
3320775,S3-268060208,S1-217525996,0.000521,False


    True S1 S1-938947364 is at rank 1 with score 0.856951


### Diagnostic 3: Inspect `VAL_TARGET_POOL` (shape and 5 sample rows)

In [ ]:
print('Shape of VAL_TARGET_POOL:', VAL_TARGET_POOL.shape)
print('\nFirst 5 rows of VAL_TARGET_POOL:')
display(VAL_TARGET_POOL.head(5))

Shape of VAL_TARGET_POOL: (69231, 7)

First 5 rows of VAL_TARGET_POOL:


,entity_id,business_name,business_address,country,name_n,addr_n,country_n
0,S2-254788212,Hyacinthe Cusack Services,"DALLAS, 195 NOMAS ST, PO BOX 7491, TX",US,hyacinthecusackservices,dallas195nomasstpobox7491tx,us
1,S2-435778329,PRIVATE AHMEDABAD INFRATECH (LTD),"A-29 KARAMACHARINAGAR IGHATLODAIA, AHMEDABAD, ...",India,privateahmedabadinfratechltd,a29karamacharinagarighatlodaiaahmedabadghatlod...,india
2,S2-299258785,"CONKLIN, BOWMAN AND MCLAUGHLIN ENGINEERING LLC","3724 180, HURICANE CITY, UT",US,conklinbowmanandmclaughlinengineeringllc,3724180huricanecityut,us
3,S2-857071377,EASTERN SPA MODERN LLC - 3276849040,"52575 289ND AVE E, CWOETA, OK",US,easternspamodernllc3276849040,52575289ndaveecwoetaok,us
4,S2-845463620,#hanumaneducational,"Maharashtra, B/3-02, S. NO.221 TO 223, THE SPI...",India,hanumaneducational,maharashtrab302sno221to223thespirespunecity,india


### Diagnostic 4: Diff feature computation and batching code

To address your request to 'diff the current feature-computation and batching code against the last version that scored 0.985436', I will print the relevant sections of the current code. Since I do not have direct access to previous notebook states or version control systems to perform an automatic diff, you can manually compare these outputs against your known good version.

First, I will present the `feature_order` list, which defines the exact features and their sequence used by the LightGBM model. Any change in this list (feature names, order, or presence) between versions could impact the model's performance.

Next, I will print the code snippet responsible for generating features (`df_q` DataFrame construction) within the main loop of Cell 4. This section calculates all individual features for each candidate pair. You should inspect this for any changes in feature logic or calculation.

Finally, I will display the code that computes query-relative ranking features, as these are also part of the feature set fed into the model.

In [ ]:
print('--- Feature Order for LightGBM Model ---')
print(feature_order)
print(f'Total features: {len(feature_order)}')

print('\n--- Feature Calculation Code Snippet (from within the main loop of Cell 4) ---')
print("""
df_q = pd.DataFrame({
    "query_id": row.entity_id, "s1_id": c_ids,
    "name_ratio": name_ratio, "addr_ratio": addr_ratio, "name_partial": name_partial, "addr_partial": addr_partial,
    "name_token": name_token, "addr_token": addr_token,
    "name_length_ratio": name_length_ratio, "addr_length_ratio": addr_length_ratio,
    "number_overlap": num_overlap, "exact_name": exact_name, "exact_address": exact_address,
    "country_match": country_match, "combined_similarity": combined_similarity,
    "has_rare": has_rare, "has_tfidf": has_tfidf, "retrieval_channel_count": retrieval_channel_count,
    "tfidf_score": tfidf_score, "tfidf_rank": tfidf_rank,
    "name_addr_product": name_addr_product, "name_partial_addr_partial_product": name_partial_addr_partial_product,
    "name_number_product": name_number_product, "addr_number_product": addr_number_product,
    "min_name_addr": min_name_addr, "max_name_addr": max_name_addr,
    "name_addr_gap": name_addr_gap, "partial_gap": partial_gap,
    "strong_name_weak_address": strong_name_weak_address, "strong_name_no_number": strong_name_no_number,
    "strong_address": strong_address, "strong_name": strong_name,
    "address_supported": address_supported, "multi_channel": multi_channel,
    "balanced_similarity": balanced_similarity, "partial_balanced_similarity": partial_balanced_similarity,
})
""")

print('\n--- Query-Relative Ranking Feature Calculation Code Snippet ---')
print("""
for col in rank_cols:
    # Ensure the column exists before calling rank or transform
    if col in df_batch.columns:
        df_batch[f"rank_{col}"] = df_batch.groupby('query_id')[col].rank(method="min", ascending=False)
        df_batch[f"query_max_{col}"] = df_batch.groupby('query_id')[col].transform('max')
    else:
        # Handle cases where the column might be missing if it's a dummy for TF-IDF
        df_batch[f"rank_{col}"] = 999  # Assign a default rank
        df_batch[f"query_max_{col}"] = 0.0 # Assign a default max

df_batch["tfidf_margin"] = df_batch["query_max_tfidf_score"] - df_batch["tfidf_score"]
df_batch["name_margin"] = df_batch["query_max_name_ratio"] - df_batch["name_ratio"]
df_batch["addr_margin"] = df_batch["query_max_addr_ratio"] - df_batch["addr_ratio"]
df_batch["combined_margin"] = df_batch["query_max_combined_similarity"] - df_batch["combined_similarity"]
df_batch["relative_tfidf_strength"] = df_batch["tfidf_score"] / (df_batch["query_max_tfidf_score"] + 1e-8)
df_batch["relative_combined_strength"] = df_batch["combined_similarity"] / (df_batch["query_max_combined_similarity"] + 1e-8)
df_batch["is_best_tfidf"] = (df_batch["rank_tfidf_score"] == 1).astype(np.int8)
df_batch["is_best_name"] = (df_batch["rank_name_ratio"] == 1).astype(np.int8)
df_batch["is_best_addr"] = (df_batch["rank_addr_ratio"] == 1).astype(np.int8)
df_batch["is_best_combined"] = (df_batch["rank_combined_similarity"] == 1).astype(np.int8)
""")

--- Feature Order for LightGBM Model ---
['has_rare', 'has_tfidf', 'tfidf_score', 'tfidf_rank', 'name_ratio', 'addr_ratio', 'name_partial', 'addr_partial', 'name_token', 'addr_token', 'name_length_ratio', 'addr_length_ratio', 'number_overlap', 'exact_name', 'exact_address', 'country_match', 'combined_similarity', 'retrieval_channel_count', 'name_addr_product', 'name_partial_addr_partial_product', 'name_number_product', 'addr_number_product', 'min_name_addr', 'max_name_addr', 'name_addr_gap', 'partial_gap', 'strong_name_weak_address', 'strong_name_no_number', 'strong_address', 'strong_name', 'address_supported', 'multi_channel', 'balanced_similarity', 'partial_balanced_similarity', 'rank_name_ratio', 'query_max_name_ratio', 'rank_addr_ratio', 'query_max_addr_ratio', 'rank_name_partial', 'query_max_name_partial', 'rank_addr_partial', 'query_max_addr_partial', 'rank_combined_similarity', 'query_max_combined_similarity', 'rank_tfidf_score', 'query_max_tfidf_score', 'tfidf_margin', 'name_ma

The `number_overlap` function had an escaped backslash in its regex (`r"\\d+"`) which prevented it from correctly matching digits. This has been corrected to `r"\d+"` to ensure proper number extraction.

In [ ]:
def number_overlap(a, b):
    if not a or not b:
        return 0
    # Corrected regex to match actual digits
    return len(set(re.findall(r"\d+", str(a))) & set(re.findall(r"\d+", str(b))))

In [ ]:
print('--- Testing number_overlap function ---')

def number_overlap_test(a, b):
    if not a or not b:
        return 0
    return len(set(re.findall(r"\d+", str(a))) & set(re.findall(r"\d+", str(b))))

# Test cases with expected overlap
print(f"Test Case 1: number_overlap_test('123 Main St', '123 Oak Ave') = {number_overlap_test('123 Main St', '123 Oak Ave')}")
print(f"Test Case 2: number_overlap_test('Apt 4B', 'Bldg 4C') = {number_overlap_test('Apt 4B', 'Bldg 4C')}")
print(f"Test Case 3: number_overlap_test('10 Downing St', '11 Downing St') = {number_overlap_test('10 Downing St', '11 Downing St')}")
print(f"Test Case 4: number_overlap_test('No. 7, 5th Ave', '5th Ave, No. 7') = {number_overlap_test('No. 7, 5th Ave', '5th Ave, No. 7')}")

# Test cases with no expected overlap
print(f"Test Case 5: number_overlap_test('123 Main St', '456 Oak Ave') = {number_overlap_test('123 Main St', '456 Oak Ave')}")
print(f"Test Case 6: number_overlap_test('Main St', 'Oak Ave') = {number_overlap_test('Main St', 'Oak Ave')}")

# Test with actual normalized data from VAL_S1 and VAL_TARGET_POOL (if available)
if 'VAL_S1' in globals() and 'VAL_TARGET_POOL' in globals():
    print('\n--- Testing with actual normalized data ---')
    # Pick a few specific rows that are likely to have numbers in their addresses
    s1_sample_addr = VAL_S1['addr_n'].iloc[0] # '1795westchesterdrivehighpointnc'
    target_sample_addr = VAL_TARGET_POOL['addr_n'].iloc[2] # '110centerstnullflorenceky'

    print(f"VAL_S1[0] address: {s1_sample_addr}")
    print(f"VAL_TARGET_POOL[2] address: {target_sample_addr}")
    print(f"number_overlap_test(VAL_S1[0], VAL_TARGET_POOL[2]) = {number_overlap_test(s1_sample_addr, target_sample_addr)}")

    # Find a pair from VAL_S1 and VAL_TARGET_POOL that might actually overlap numbers
    # This is heuristic, actual overlap might be rare in random samples
    s1_addr_list = VAL_S1['addr_n'].head(100).tolist()
    target_addr_list = VAL_TARGET_POOL['addr_n'].head(100).tolist()

    found_overlap = False
    for i, s_addr in enumerate(s1_addr_list):
        for j, t_addr in enumerate(target_addr_list):
            overlap_count = number_overlap_test(s_addr, t_addr)
            if overlap_count > 0:
                print(f"\nFound overlap: s1_addr[{i}]='{s_addr}', target_addr[{j}]='{t_addr}', Overlap = {overlap_count}")
                found_overlap = True
                break
        if found_overlap:
            break
    if not found_overlap:
        print("\nNo numerical overlap found in first 100 sample addresses from VAL_S1 and VAL_TARGET_POOL.")
else:
    print('\nVAL_S1 or VAL_TARGET_POOL not in globals, skipping actual data test.')

--- Testing number_overlap function ---
Test Case 1: number_overlap_test('123 Main St', '123 Oak Ave') = 1
Test Case 2: number_overlap_test('Apt 4B', 'Bldg 4C') = 1
Test Case 3: number_overlap_test('10 Downing St', '11 Downing St') = 0
Test Case 4: number_overlap_test('No. 7, 5th Ave', '5th Ave, No. 7') = 2
Test Case 5: number_overlap_test('123 Main St', '456 Oak Ave') = 0
Test Case 6: number_overlap_test('Main St', 'Oak Ave') = 0

--- Testing with actual normalized data ---
VAL_S1[0] address: 485020otiscony
VAL_TARGET_POOL[2] address: 3724180huricanecityut
number_overlap_test(VAL_S1[0], VAL_TARGET_POOL[2]) = 0

Found overlap: s1_addr[6]='sanfordme31guillemettestreet', target_addr[78]='masaugs31waldenpondave', Overlap = 1


In [ ]:
# --- HAND DIAGNOSTIC: where are the false negatives coming from? ---
retrieved_pairs = set(zip(df_all["query_id"], df_all["s1_id"]))

total_true = 0
never_retrieved = 0
retrieved_but_lost = 0
retrieved_but_below_threshold = 0

for sid, true_queries in true_by_s1.items():
    for qid in true_queries:
        total_true += 1
        if (qid, sid) not in retrieved_pairs:
            never_retrieved += 1
            continue
        # It WAS retrieved as a candidate. Did it win for that query, and clear threshold?
        row = df_all[(df_all["query_id"] == qid) & (df_all["s1_id"] == sid)]
        true_score = row["model_score"].iloc[0] if "model_score" in df_all.columns else None
        query_rows = df_all[df_all["query_id"] == qid]
        top_score = query_rows["model_score"].max() if "model_score" in df_all.columns else None
        if true_score is not None and top_score is not None:
            if true_score < top_score:
                retrieved_but_lost += 1
            elif true_score < threshold:
                retrieved_but_below_threshold += 1

print(f"Total true (S1, query) pairs: {total_true:,}")
print(f"Never retrieved as a candidate at all: {never_retrieved:,} ({100*never_retrieved/total_true:.2f}%)")
print(f"Retrieved but a WRONG candidate scored higher (lost the argmax): {retrieved_but_lost:,} ({100*retrieved_but_lost/total_true:.2f}%)")
print(f"Retrieved, correctly the top candidate, but scored below threshold: {retrieved_but_below_threshold:,} ({100*retrieved_but_below_threshold/total_true:.2f}%)")

Total true (S1, query) pairs: 69,231
Never retrieved as a candidate at all: 57,292 (82.75%)
Retrieved but a WRONG candidate scored higher (lost the argmax): 2 (0.00%)
Retrieved, correctly the top candidate, but scored below threshold: 161 (0.23%)


In [10]:
# ============================================================
# CELL 5 — PRODUCTION INFERENCE (S2 + S3), CHECKPOINTED
# Uses the validated rare-token-only pipeline (fixed tokenizer,
# 0.959335 F0.5 on 20k val). Reuses precomputed production
# rare-token index — does NOT rebuild it from scratch.
# ============================================================
import re, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
from pathlib import Path

BASE = Path('/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026')
FINAL_DIR = BASE / "MODELS" / "FINAL_SAVED_ARTIFACTS"
TEST_DIR = BASE / "DATASET" / "test"
RUN_DIR = BASE / "MODELS" / "PRODUCTION_RUN_RARE"
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = RUN_DIR / "batches"
OUT_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = RUN_DIR / "manifest.json"

def normalize_exact(x):
    if pd.isna(x): return ""
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())

def word_tokens(text):
    if pd.isna(text): return []
    text = str(text).lower()
    if not text: return []
    return [t for t in re.findall(r"[^\W_]+", text, flags=re.UNICODE) if len(t) >= 2]

def number_overlap(a, b):
    if not a or not b: return 0
    return len(set(re.findall(r"\d+", str(a))) & set(re.findall(r"\d+", str(b))))

print("Loading model + production rare-token artifacts...")
matcher = lgb.Booster(model_file=str(FINAL_DIR / "final_lambdarank_model.txt"))
with open(FINAL_DIR / "final_lambdarank_threshold.txt") as f:
    threshold = float(f.read().strip())
feature_order = matcher.feature_name()
assert matcher.num_trees() == 424
print(f"Model OK. Trees={matcher.num_trees()} Threshold={threshold}")

rare_token_index = joblib.load(FINAL_DIR / "rare_token_index_prod.joblib")  # (country, token) -> [entity_id,...]
token_df_prod = joblib.load(FINAL_DIR / "token_df_prod.joblib")             # token -> df count
MAX_DF, TOP_N = 1000, 4

print("Loading full TEST S1 (name/addr/country by entity_id)...")
test_s1 = pd.read_csv(TEST_DIR / "test_source1.tsv", sep="\t", dtype=str, keep_default_na=False)
test_s1["name_n"] = test_s1["business_name"].map(normalize_exact)
test_s1["addr_n"] = test_s1["business_address"].map(normalize_exact)
test_s1["country_n"] = test_s1["country"].fillna("").astype(str).str.strip().str.lower()
s1_by_id = test_s1.set_index("entity_id")[["business_name","business_address","name_n","addr_n","country_n"]]
print(f"TEST S1: {len(test_s1):,} rows indexed by entity_id")
del test_s1

rank_cols = ["name_ratio","addr_ratio","name_partial","addr_partial","combined_similarity","tfidf_score"]

from rapidfuzz import process, fuzz

def process_source(source_name, source_path, batch_size=50_000):
    manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
    done = manifest.get(f"{source_name}_batches_done", 0)
    print(f"\n=== {source_name}: resuming from batch {done} ===")

    reader = pd.read_csv(source_path, sep="\t", dtype=str, keep_default_na=False, chunksize=batch_size)
    for batch_no, chunk in enumerate(reader):
        out_path = OUT_DIR / f"{source_name}_part_{batch_no:05d}.tsv"
        if batch_no < done:
            continue
        t0 = time.time()
        chunk["name_n"] = chunk["business_name"].map(normalize_exact)
        chunk["addr_n"] = chunk["business_address"].map(normalize_exact)
        chunk["country_n"] = chunk["country"].fillna("").astype(str).str.strip().str.lower()

        rows_out = []
        for row in chunk.itertuples(index=False):
            q_name, q_addr, q_country = row.name_n, row.addr_n, row.country_n
            tokens = set(word_tokens(row.business_name) + word_tokens(row.business_address))
            usable = sorted(
                [(token_df_prod.get(t, 0), t) for t in tokens if 0 < token_df_prod.get(t, 0) <= MAX_DF],
                key=lambda x: x[0]
            )
            cand_ids = set()
            for _, tok in usable[:TOP_N]:
                cand_ids.update(rare_token_index.get((q_country, tok), []))
            if not cand_ids:
                continue

            cand_ids = list(cand_ids)
            cand_rows = s1_by_id.loc[s1_by_id.index.intersection(cand_ids)]
            if cand_rows.empty:
                continue
            c_names = cand_rows["name_n"].to_numpy()
            c_addrs = cand_rows["addr_n"].to_numpy()
            c_countries = cand_rows["country_n"].to_numpy()
            c_ids = cand_rows.index.to_numpy()

            name_ratio = process.cdist([q_name], c_names, scorer=fuzz.ratio)[0] / 100.0
            addr_ratio = process.cdist([q_addr], c_addrs, scorer=fuzz.ratio)[0] / 100.0
            name_partial = process.cdist([q_name], c_names, scorer=fuzz.partial_ratio)[0] / 100.0
            addr_partial = process.cdist([q_addr], c_addrs, scorer=fuzz.partial_ratio)[0] / 100.0
            name_token = process.cdist([q_name], c_names, scorer=fuzz.token_set_ratio)[0] / 100.0
            addr_token = process.cdist([q_addr], c_addrs, scorer=fuzz.token_set_ratio)[0] / 100.0

            qn, qa = len(q_name), len(q_addr)
            sn = np.array([len(x) for x in c_names]); sa = np.array([len(x) for x in c_addrs])
            name_length_ratio = np.where(np.maximum(qn,sn)>0, np.minimum(qn,sn)/np.maximum(qn,sn), 1.0)
            addr_length_ratio = np.where(np.maximum(qa,sa)>0, np.minimum(qa,sa)/np.maximum(qa,sa), 1.0)
            num_overlap = np.array([number_overlap(q_addr, a) for a in c_addrs], dtype=np.float64)
            exact_name = (c_names == q_name).astype(int)
            exact_address = (c_addrs == q_addr).astype(int)
            country_match = (c_countries == q_country).astype(int)
            combined_similarity = 0.45*name_ratio + 0.55*addr_ratio
            has_rare = np.ones(len(c_ids), dtype=int)
            has_tfidf = np.zeros(len(c_ids), dtype=int)
            tfidf_score = np.zeros(len(c_ids)); tfidf_rank = np.full(len(c_ids), 999)
            retrieval_channel_count = has_rare + has_tfidf
            name_addr_product = name_ratio*addr_ratio
            name_partial_addr_partial_product = name_partial*addr_partial
            name_number_product = name_ratio*num_overlap
            addr_number_product = addr_ratio*num_overlap
            min_name_addr = np.minimum(name_ratio, addr_ratio); max_name_addr = np.maximum(name_ratio, addr_ratio)
            name_addr_gap = np.abs(name_ratio-addr_ratio); partial_gap = np.abs(name_partial-addr_partial)
            strong_name_weak_address = ((name_ratio>=0.80)&(addr_ratio<0.20)).astype(int)
            strong_name_no_number = ((name_ratio>=0.80)&(num_overlap==0)).astype(int)
            strong_address = (addr_ratio>=0.70).astype(int)
            strong_name = (name_ratio>=0.80).astype(int)
            address_supported = (addr_ratio>=0.50).astype(int)
            multi_channel = (retrieval_channel_count>=2).astype(int)
            balanced_similarity = (name_ratio+addr_ratio)/2.0
            partial_balanced_similarity = (name_partial+addr_partial)/2.0

            df_q = pd.DataFrame({
                "query_id": row.entity_id, "s1_id": c_ids,
                "name_ratio": name_ratio, "addr_ratio": addr_ratio, "name_partial": name_partial, "addr_partial": addr_partial,
                "name_token": name_token, "addr_token": addr_token,
                "name_length_ratio": name_length_ratio, "addr_length_ratio": addr_length_ratio,
                "number_overlap": num_overlap, "exact_name": exact_name, "exact_address": exact_address,
                "country_match": country_match, "combined_similarity": combined_similarity,
                "has_rare": has_rare, "has_tfidf": has_tfidf, "retrieval_channel_count": retrieval_channel_count,
                "tfidf_score": tfidf_score, "tfidf_rank": tfidf_rank,
                "name_addr_product": name_addr_product, "name_partial_addr_partial_product": name_partial_addr_partial_product,
                "name_number_product": name_number_product, "addr_number_product": addr_number_product,
                "min_name_addr": min_name_addr, "max_name_addr": max_name_addr,
                "name_addr_gap": name_addr_gap, "partial_gap": partial_gap,
                "strong_name_weak_address": strong_name_weak_address, "strong_name_no_number": strong_name_no_number,
                "strong_address": strong_address, "strong_name": strong_name,
                "address_supported": address_supported, "multi_channel": multi_channel,
                "balanced_similarity": balanced_similarity, "partial_balanced_similarity": partial_balanced_similarity,
            })
            for col in rank_cols:
                df_q[f"rank_{col}"] = df_q[col].rank(method="min", ascending=False)
                df_q[f"query_max_{col}"] = df_q[col].max()
            df_q["tfidf_margin"] = df_q["query_max_tfidf_score"] - df_q["tfidf_score"]
            df_q["name_margin"] = df_q["query_max_name_ratio"] - df_q["name_ratio"]
            df_q["addr_margin"] = df_q["query_max_addr_ratio"] - df_q["addr_ratio"]
            df_q["combined_margin"] = df_q["query_max_combined_similarity"] - df_q["combined_similarity"]
            df_q["relative_tfidf_strength"] = df_q["tfidf_score"] / (df_q["query_max_tfidf_score"] + 1e-8)
            df_q["relative_combined_strength"] = df_q["combined_similarity"] / (df_q["query_max_combined_similarity"] + 1e-8)
            df_q["is_best_tfidf"] = (df_q["rank_tfidf_score"]==1).astype(np.int8)
            df_q["is_best_name"] = (df_q["rank_name_ratio"]==1).astype(np.int8)
            df_q["is_best_addr"] = (df_q["rank_addr_ratio"]==1).astype(np.int8)
            df_q["is_best_combined"] = (df_q["rank_combined_similarity"]==1).astype(np.int8)
            rows_out.append(df_q)

        if rows_out:
            df_batch = pd.concat(rows_out, ignore_index=True)
            X = df_batch.reindex(columns=feature_order, fill_value=0)
            df_batch["model_score"] = matcher.predict(X)
            best = df_batch.sort_values("model_score", ascending=False).groupby("query_id", as_index=False).first()
            best = best[best["model_score"] >= threshold][["query_id","s1_id","model_score"]]
            best.to_csv(out_path, sep="\t", index=False)
        else:
            best = pd.DataFrame(columns=["query_id","s1_id","model_score"])
            best.to_csv(out_path, sep="\t", index=False)

        manifest[f"{source_name}_batches_done"] = batch_no + 1
        MANIFEST_PATH.write_text(json.dumps(manifest))
        elapsed = time.time() - t0
        print(f"[{source_name}] batch {batch_no:05d} | {len(chunk):,} rows | {elapsed/60:.2f} min | matches={len(best)}")

print("\n--- SMOKE TEST: first batch of S2 only ---")
process_source("S2", TEST_DIR / "test_source2.tsv", batch_size=5_000)
print("\nSmoke test done. If output above looks sane, run the full loop:")
print('  process_source("S2", TEST_DIR / "test_source2.tsv")')
print('  process_source("S3", TEST_DIR / "test_source3.tsv")')

Loading model + production rare-token artifacts...
Model OK. Trees=424 Threshold=0.6280000000000001
Loading full TEST S1 (name/addr/country by entity_id)...
TEST S1: 1,732,544 rows indexed by entity_id

--- SMOKE TEST: first batch of S2 only ---

=== S2: resuming from batch 0 ===
[S2] batch 00000 | 5,000 rows | 3.24 min | matches=3974
[S2] batch 00001 | 5,000 rows | 3.03 min | matches=3949
[S2] batch 00002 | 5,000 rows | 2.91 min | matches=3955
[S2] batch 00003 | 5,000 rows | 3.05 min | matches=3994
[S2] batch 00004 | 5,000 rows | 2.95 min | matches=3915
[S2] batch 00005 | 5,000 rows | 2.94 min | matches=3956
[S2] batch 00006 | 5,000 rows | 3.04 min | matches=3975
[S2] batch 00007 | 5,000 rows | 2.97 min | matches=3953
[S2] batch 00008 | 5,000 rows | 2.97 min | matches=3944
[S2] batch 00009 | 5,000 rows | 3.02 min | matches=3980
[S2] batch 00010 | 5,000 rows | 2.94 min | matches=3959
[S2] batch 00011 | 5,000 rows | 2.91 min | matches=3896
[S2] batch 00012 | 5,000 rows | 3.03 min | matc

KeyboardInterrupt: 

In [1]:
# ============================================================
# CELL 5 (FIXED) — PRODUCTION INFERENCE, NUMPY-INDEXED (FAST)
# ============================================================
import re, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
from pathlib import Path
from rapidfuzz import process, fuzz

BASE = Path('/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026')
FINAL_DIR = BASE / "MODELS" / "FINAL_SAVED_ARTIFACTS"
TEST_DIR = BASE / "DATASET" / "test"
RUN_DIR = BASE / "MODELS" / "PRODUCTION_RUN_RARE"
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = RUN_DIR / "batches"
OUT_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = RUN_DIR / "manifest.json"

def normalize_exact(x):
    if pd.isna(x): return ""
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())

def word_tokens(text):
    if pd.isna(text): return []
    text = str(text).lower()
    if not text: return []
    return [t for t in re.findall(r"[^\W_]+", text, flags=re.UNICODE) if len(t) >= 2]

def number_overlap(a, b):
    if not a or not b: return 0
    return len(set(re.findall(r"\d+", str(a))) & set(re.findall(r"\d+", str(b))))

print("Loading model + production rare-token artifacts...")
matcher = lgb.Booster(model_file=str(FINAL_DIR / "final_lambdarank_model.txt"))
with open(FINAL_DIR / "final_lambdarank_threshold.txt") as f:
    threshold = float(f.read().strip())
feature_order = matcher.feature_name()
assert matcher.num_trees() == 424
print(f"Model OK. Trees={matcher.num_trees()} Threshold={threshold}")

rare_token_index = joblib.load(FINAL_DIR / "rare_token_index_prod.joblib")
token_df_prod = joblib.load(FINAL_DIR / "token_df_prod.joblib")
MAX_DF, TOP_N = 1000, 4

print("Loading full TEST S1 as numpy arrays (fast path)...")
test_s1 = pd.read_csv(TEST_DIR / "test_source1.tsv", sep="\t", dtype=str, keep_default_na=False)
test_s1["name_n"] = test_s1["business_name"].map(normalize_exact)
test_s1["addr_n"] = test_s1["business_address"].map(normalize_exact)
test_s1["country_n"] = test_s1["country"].fillna("").astype(str).str.strip().str.lower()

s1_ids_arr = test_s1["entity_id"].to_numpy()
s1_name = test_s1["name_n"].to_numpy()
s1_addr = test_s1["addr_n"].to_numpy()
s1_country = test_s1["country_n"].to_numpy()
entity_id_to_local = {eid: i for i, eid in enumerate(s1_ids_arr)}
print(f"TEST S1: {len(test_s1):,} rows | local-index map built: {len(entity_id_to_local):,}")
del test_s1

rank_cols = ["name_ratio","addr_ratio","name_partial","addr_partial","combined_similarity","tfidf_score"]

def process_source(source_name, source_path, batch_size=5000):
    manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
    done = manifest.get(f"{source_name}_batches_done", 0)
    print(f"\n=== {source_name}: resuming from batch {done} (batch_size={batch_size}) ===")

    reader = pd.read_csv(source_path, sep="\t", dtype=str, keep_default_na=False, chunksize=batch_size)
    for batch_no, chunk in enumerate(reader):
        out_path = OUT_DIR / f"{source_name}_part_{batch_no:05d}.tsv"
        if batch_no < done:
            continue
        t0 = time.time()
        chunk["name_n"] = chunk["business_name"].map(normalize_exact)
        chunk["addr_n"] = chunk["business_address"].map(normalize_exact)
        chunk["country_n"] = chunk["country"].fillna("").astype(str).str.strip().str.lower()

        rows_out = []
        for row in chunk.itertuples(index=False):
            q_name, q_addr, q_country = row.name_n, row.addr_n, row.country_n
            tokens = set(word_tokens(row.business_name) + word_tokens(row.business_address))
            usable = sorted(
                [(token_df_prod.get(t, 0), t) for t in tokens if 0 < token_df_prod.get(t, 0) <= MAX_DF],
                key=lambda x: x[0]
            )
            cand_local = set()
            for _, tok in usable[:TOP_N]:
                for eid in rare_token_index.get((q_country, tok), []):
                    local_idx = entity_id_to_local.get(eid)
                    if local_idx is not None:
                        cand_local.add(local_idx)
            if not cand_local:
                continue

            cand_arr = np.fromiter(cand_local, dtype=np.int32)
            c_names = s1_name[cand_arr]
            c_addrs = s1_addr[cand_arr]
            c_countries = s1_country[cand_arr]
            c_ids = s1_ids_arr[cand_arr]

            name_ratio = process.cdist([q_name], c_names, scorer=fuzz.ratio)[0] / 100.0
            addr_ratio = process.cdist([q_addr], c_addrs, scorer=fuzz.ratio)[0] / 100.0
            name_partial = process.cdist([q_name], c_names, scorer=fuzz.partial_ratio)[0] / 100.0
            addr_partial = process.cdist([q_addr], c_addrs, scorer=fuzz.partial_ratio)[0] / 100.0
            name_token = process.cdist([q_name], c_names, scorer=fuzz.token_set_ratio)[0] / 100.0
            addr_token = process.cdist([q_addr], c_addrs, scorer=fuzz.token_set_ratio)[0] / 100.0

            qn, qa = len(q_name), len(q_addr)
            sn = np.array([len(x) for x in c_names]); sa = np.array([len(x) for x in c_addrs])
            name_length_ratio = np.where(np.maximum(qn,sn)>0, np.minimum(qn,sn)/np.maximum(qn,sn), 1.0)
            addr_length_ratio = np.where(np.maximum(qa,sa)>0, np.minimum(qa,sa)/np.maximum(qa,sa), 1.0)
            num_overlap = np.array([number_overlap(q_addr, a) for a in c_addrs], dtype=np.float64)
            exact_name = (c_names == q_name).astype(int)
            exact_address = (c_addrs == q_addr).astype(int)
            country_match = (c_countries == q_country).astype(int)
            combined_similarity = 0.45*name_ratio + 0.55*addr_ratio
            has_rare = np.ones(len(c_ids), dtype=int)
            has_tfidf = np.zeros(len(c_ids), dtype=int)
            tfidf_score = np.zeros(len(c_ids)); tfidf_rank = np.full(len(c_ids), 999)
            retrieval_channel_count = has_rare + has_tfidf
            name_addr_product = name_ratio*addr_ratio
            name_partial_addr_partial_product = name_partial*addr_partial
            name_number_product = name_ratio*num_overlap
            addr_number_product = addr_ratio*num_overlap
            min_name_addr = np.minimum(name_ratio, addr_ratio); max_name_addr = np.maximum(name_ratio, addr_ratio)
            name_addr_gap = np.abs(name_ratio-addr_ratio); partial_gap = np.abs(name_partial-addr_partial)
            strong_name_weak_address = ((name_ratio>=0.80)&(addr_ratio<0.20)).astype(int)
            strong_name_no_number = ((name_ratio>=0.80)&(num_overlap==0)).astype(int)
            strong_address = (addr_ratio>=0.70).astype(int)
            strong_name = (name_ratio>=0.80).astype(int)
            address_supported = (addr_ratio>=0.50).astype(int)
            multi_channel = (retrieval_channel_count>=2).astype(int)
            balanced_similarity = (name_ratio+addr_ratio)/2.0
            partial_balanced_similarity = (name_partial+addr_partial)/2.0

            df_q = pd.DataFrame({
                "query_id": row.entity_id, "s1_id": c_ids,
                "name_ratio": name_ratio, "addr_ratio": addr_ratio, "name_partial": name_partial, "addr_partial": addr_partial,
                "name_token": name_token, "addr_token": addr_token,
                "name_length_ratio": name_length_ratio, "addr_length_ratio": addr_length_ratio,
                "number_overlap": num_overlap, "exact_name": exact_name, "exact_address": exact_address,
                "country_match": country_match, "combined_similarity": combined_similarity,
                "has_rare": has_rare, "has_tfidf": has_tfidf, "retrieval_channel_count": retrieval_channel_count,
                "tfidf_score": tfidf_score, "tfidf_rank": tfidf_rank,
                "name_addr_product": name_addr_product, "name_partial_addr_partial_product": name_partial_addr_partial_product,
                "name_number_product": name_number_product, "addr_number_product": addr_number_product,
                "min_name_addr": min_name_addr, "max_name_addr": max_name_addr,
                "name_addr_gap": name_addr_gap, "partial_gap": partial_gap,
                "strong_name_weak_address": strong_name_weak_address, "strong_name_no_number": strong_name_no_number,
                "strong_address": strong_address, "strong_name": strong_name,
                "address_supported": address_supported, "multi_channel": multi_channel,
                "balanced_similarity": balanced_similarity, "partial_balanced_similarity": partial_balanced_similarity,
            })
            for col in rank_cols:
                df_q[f"rank_{col}"] = df_q[col].rank(method="min", ascending=False)
                df_q[f"query_max_{col}"] = df_q[col].max()
            df_q["tfidf_margin"] = df_q["query_max_tfidf_score"] - df_q["tfidf_score"]
            df_q["name_margin"] = df_q["query_max_name_ratio"] - df_q["name_ratio"]
            df_q["addr_margin"] = df_q["query_max_addr_ratio"] - df_q["addr_ratio"]
            df_q["combined_margin"] = df_q["query_max_combined_similarity"] - df_q["combined_similarity"]
            df_q["relative_tfidf_strength"] = df_q["tfidf_score"] / (df_q["query_max_tfidf_score"] + 1e-8)
            df_q["relative_combined_strength"] = df_q["combined_similarity"] / (df_q["query_max_combined_similarity"] + 1e-8)
            df_q["is_best_tfidf"] = (df_q["rank_tfidf_score"]==1).astype(np.int8)
            df_q["is_best_name"] = (df_q["rank_name_ratio"]==1).astype(np.int8)
            df_q["is_best_addr"] = (df_q["rank_addr_ratio"]==1).astype(np.int8)
            df_q["is_best_combined"] = (df_q["rank_combined_similarity"]==1).astype(np.int8)
            rows_out.append(df_q)

        if rows_out:
            df_batch = pd.concat(rows_out, ignore_index=True)
            X = df_batch.reindex(columns=feature_order, fill_value=0)
            df_batch["model_score"] = matcher.predict(X)
            best = df_batch.sort_values("model_score", ascending=False).groupby("query_id", as_index=False).first()
            best = best[best["model_score"] >= threshold][["query_id","s1_id","model_score"]]
        else:
            best = pd.DataFrame(columns=["query_id","s1_id","model_score"])
        best.to_csv(out_path, sep="\t", index=False)

        manifest[f"{source_name}_batches_done"] = batch_no + 1
        MANIFEST_PATH.write_text(json.dumps(manifest))
        elapsed = time.time() - t0
        qps = len(chunk) / elapsed
        print(f"[{source_name}] batch {batch_no:05d} | {len(chunk):,} rows | {elapsed:.1f}s | {qps:.1f} q/s | matches={len(best)}")

print("\n--- Resuming S2 with FIXED fast lookup (same batch_size=5000 to preserve manifest alignment) ---")
process_source("S2", TEST_DIR / "test_source2.tsv", batch_size=5000)

Loading model + production rare-token artifacts...
Model OK. Trees=424 Threshold=0.6280000000000001
Loading full TEST S1 as numpy arrays (fast path)...
TEST S1: 1,732,544 rows | local-index map built: 1,732,544

--- Resuming S2 with FIXED fast lookup (same batch_size=5000 to preserve manifest alignment) ---

=== S2: resuming from batch 19 (batch_size=5000) ===


KeyboardInterrupt: 

In [2]:
def process_source(source_name, source_path, batch_size=5000):
    manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
    done = manifest.get(f"{source_name}_batches_done", 0)
    print(f"\n=== {source_name}: resuming from batch {done} (batch_size={batch_size}) ===")

    reader = pd.read_csv(source_path, sep="\t", dtype=str, keep_default_na=False, chunksize=batch_size)
    for batch_no, chunk in enumerate(reader):
        out_path = OUT_DIR / f"{source_name}_part_{batch_no:05d}.tsv"
        if batch_no < done:
            continue
        t0 = time.time()
        chunk["name_n"] = chunk["business_name"].map(normalize_exact)
        chunk["addr_n"] = chunk["business_address"].map(normalize_exact)
        chunk["country_n"] = chunk["country"].fillna("").astype(str).str.strip().str.lower()

        batch_candidate_data = []

        for row in chunk.itertuples(index=False):
            q_name, q_addr, q_country = row.name_n, row.addr_n, row.country_n
            tokens = set(word_tokens(row.business_name) + word_tokens(row.business_address))
            usable = sorted(
                [(token_df_prod.get(t, 0), t) for t in tokens if 0 < token_df_prod.get(t, 0) <= MAX_DF],
                key=lambda x: x[0]
            )
            cand_local = set()
            for _, tok in usable[:TOP_N]:
                for eid in rare_token_index.get((q_country, tok), []):
                    local_idx = entity_id_to_local.get(eid)
                    if local_idx is not None:
                        cand_local.add(local_idx)
            if not cand_local:
                continue

            cand_arr = np.fromiter(cand_local, dtype=np.int32)
            c_names = s1_name[cand_arr]
            c_addrs = s1_addr[cand_arr]
            c_countries = s1_country[cand_arr]
            c_ids = s1_ids_arr[cand_arr]

            name_ratio = process.cdist([q_name], c_names, scorer=fuzz.ratio)[0] / 100.0
            addr_ratio = process.cdist([q_addr], c_addrs, scorer=fuzz.ratio)[0] / 100.0
            name_partial = process.cdist([q_name], c_names, scorer=fuzz.partial_ratio)[0] / 100.0
            addr_partial = process.cdist([q_addr], c_addrs, scorer=fuzz.partial_ratio)[0] / 100.0
            name_token = process.cdist([q_name], c_names, scorer=fuzz.token_set_ratio)[0] / 100.0
            addr_token = process.cdist([q_addr], c_addrs, scorer=fuzz.token_set_ratio)[0] / 100.0

            qn, qa = len(q_name), len(q_addr)
            sn = np.array([len(x) for x in c_names]); sa = np.array([len(x) for x in c_addrs])
            name_length_ratio = np.where(np.maximum(qn,sn)>0, np.minimum(qn,sn)/np.maximum(qn,sn), 1.0)
            addr_length_ratio = np.where(np.maximum(qa,sa)>0, np.minimum(qa,sa)/np.maximum(qa,sa), 1.0)
            num_overlap = np.array([number_overlap(q_addr, a) for a in c_addrs], dtype=np.float64)
            exact_name = (c_names == q_name).astype(int)
            exact_address = (c_addrs == q_addr).astype(int)
            country_match = (c_countries == q_country).astype(int)
            combined_similarity = 0.45*name_ratio + 0.55*addr_ratio
            has_rare = np.ones(len(c_ids), dtype=int)
            has_tfidf = np.zeros(len(c_ids), dtype=int)
            tfidf_score = np.zeros(len(c_ids)); tfidf_rank = np.full(len(c_ids), 999)
            retrieval_channel_count = has_rare + has_tfidf
            name_addr_product = name_ratio*addr_ratio
            name_partial_addr_partial_product = name_partial*addr_partial
            name_number_product = name_ratio*num_overlap
            addr_number_product = addr_ratio*num_overlap
            min_name_addr = np.minimum(name_ratio, addr_ratio); max_name_addr = np.maximum(name_ratio, addr_ratio)
            name_addr_gap = np.abs(name_ratio-addr_ratio); partial_gap = np.abs(name_partial-addr_partial)
            strong_name_weak_address = ((name_ratio>=0.80)&(addr_ratio<0.20)).astype(int)
            strong_name_no_number = ((name_ratio>=0.80)&(num_overlap==0)).astype(int)
            strong_address = (addr_ratio>=0.70).astype(int)
            strong_name = (name_ratio>=0.80).astype(int)
            address_supported = (addr_ratio>=0.50).astype(int)
            multi_channel = (retrieval_channel_count>=2).astype(int)
            balanced_similarity = (name_ratio+addr_ratio)/2.0
            partial_balanced_similarity = (name_partial+addr_partial)/2.0

            for i in range(len(cand_arr)):
                batch_candidate_data.append({
                    "query_id": row.entity_id, "s1_id": c_ids[i],
                    "name_ratio": name_ratio[i], "addr_ratio": addr_ratio[i],
                    "name_partial": name_partial[i], "addr_partial": addr_partial[i],
                    "name_token": name_token[i], "addr_token": addr_token[i],
                    "name_length_ratio": name_length_ratio[i], "addr_length_ratio": addr_length_ratio[i],
                    "number_overlap": num_overlap[i], "exact_name": exact_name[i], "exact_address": exact_address[i],
                    "country_match": country_match[i], "combined_similarity": combined_similarity[i],
                    "has_rare": has_rare[i], "has_tfidf": has_tfidf[i], "retrieval_channel_count": retrieval_channel_count[i],
                    "tfidf_score": tfidf_score[i], "tfidf_rank": tfidf_rank[i],
                    "name_addr_product": name_addr_product[i], "name_partial_addr_partial_product": name_partial_addr_partial_product[i],
                    "name_number_product": name_number_product[i], "addr_number_product": addr_number_product[i],
                    "min_name_addr": min_name_addr[i], "max_name_addr": max_name_addr[i],
                    "name_addr_gap": name_addr_gap[i], "partial_gap": partial_gap[i],
                    "strong_name_weak_address": strong_name_weak_address[i], "strong_name_no_number": strong_name_no_number[i],
                    "strong_address": strong_address[i], "strong_name": strong_name[i],
                    "address_supported": address_supported[i], "multi_channel": multi_channel[i],
                    "balanced_similarity": balanced_similarity[i], "partial_balanced_similarity": partial_balanced_similarity[i],
                })

        if batch_candidate_data:
            df_batch = pd.DataFrame(batch_candidate_data)
            for col in rank_cols:
                df_batch[f"rank_{col}"] = df_batch.groupby('query_id')[col].rank(method="min", ascending=False)
                df_batch[f"query_max_{col}"] = df_batch.groupby('query_id')[col].transform('max')
            df_batch["tfidf_margin"] = df_batch["query_max_tfidf_score"] - df_batch["tfidf_score"]
            df_batch["name_margin"] = df_batch["query_max_name_ratio"] - df_batch["name_ratio"]
            df_batch["addr_margin"] = df_batch["query_max_addr_ratio"] - df_batch["addr_ratio"]
            df_batch["combined_margin"] = df_batch["query_max_combined_similarity"] - df_batch["combined_similarity"]
            df_batch["relative_tfidf_strength"] = df_batch["tfidf_score"] / (df_batch["query_max_tfidf_score"] + 1e-8)
            df_batch["relative_combined_strength"] = df_batch["combined_similarity"] / (df_batch["query_max_combined_similarity"] + 1e-8)
            df_batch["is_best_tfidf"] = (df_batch["rank_tfidf_score"]==1).astype(np.int8)
            df_batch["is_best_name"] = (df_batch["rank_name_ratio"]==1).astype(np.int8)
            df_batch["is_best_addr"] = (df_batch["rank_addr_ratio"]==1).astype(np.int8)
            df_batch["is_best_combined"] = (df_batch["rank_combined_similarity"]==1).astype(np.int8)

            X = df_batch.reindex(columns=feature_order, fill_value=0)
            df_batch["model_score"] = matcher.predict(X)
            best = df_batch.sort_values("model_score", ascending=False).groupby("query_id", as_index=False).first()
            best = best[best["model_score"] >= threshold][["query_id","s1_id","model_score"]]
        else:
            best = pd.DataFrame(columns=["query_id","s1_id","model_score"])
        best.to_csv(out_path, sep="\t", index=False)

        manifest[f"{source_name}_batches_done"] = batch_no + 1
        MANIFEST_PATH.write_text(json.dumps(manifest))
        elapsed = time.time() - t0
        qps = len(chunk) / elapsed
        print(f"[{source_name}] batch {batch_no:05d} | {len(chunk):,} rows | {elapsed:.1f}s | {qps:.1f} q/s | matches={len(best)}")

print("\n--- Resuming S2 with TRUE batched-groupby (matches validated 191 q/s) ---")
process_source("S2", TEST_DIR / "test_source2.tsv", batch_size=5000)


--- Resuming S2 with TRUE batched-groupby (matches validated 191 q/s) ---

=== S2: resuming from batch 19 (batch_size=5000) ===


KeyboardInterrupt: 

In [ ]:
process_source("S2", TEST_DIR / "test_source2.tsv", batch_size=5000)
process_source("S3", TEST_DIR / "test_source3.tsv", batch_size=5000)


=== S2: resuming from batch 19 (batch_size=5000) ===
[S2] batch 00019 | 5,000 rows | 175.3s | 28.5 q/s | matches=3927


In [2]:
# ============================================================
# CELL 5 (FULLY CORRECTED) — PRODUCTION INFERENCE, CHECKPOINTED
# Fast numpy candidate lookup + batched-groupby feature/rank
# computation (matches validated 191 q/s from Cell 4).
# ============================================================
import re, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
from pathlib import Path
from rapidfuzz import process, fuzz

BASE = Path('/content/drive/MyDrive/AMAZON ML 2026/AMAZON ML 2026')
FINAL_DIR = BASE / "MODELS" / "FINAL_SAVED_ARTIFACTS"
TEST_DIR = BASE / "DATASET" / "test"
RUN_DIR = BASE / "MODELS" / "PRODUCTION_RUN_RARE"
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = RUN_DIR / "batches"
OUT_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = RUN_DIR / "manifest.json"

def normalize_exact(x):
    if pd.isna(x): return ""
    return "".join(ch.lower() for ch in str(x) if ch.isalnum())

def word_tokens(text):
    if pd.isna(text): return []
    text = str(text).lower()
    if not text: return []
    return [t for t in re.findall(r"[^\W_]+", text, flags=re.UNICODE) if len(t) >= 2]

def number_overlap(a, b):
    if not a or not b: return 0
    return len(set(re.findall(r"\d+", str(a))) & set(re.findall(r"\d+", str(b))))

print("Loading model + production rare-token artifacts...")
matcher = lgb.Booster(model_file=str(FINAL_DIR / "final_lambdarank_model.txt"))
with open(FINAL_DIR / "final_lambdarank_threshold.txt") as f:
    threshold = float(f.read().strip())
feature_order = matcher.feature_name()
assert matcher.num_trees() == 424
print(f"Model OK. Trees={matcher.num_trees()} Threshold={threshold}")

rare_token_index = joblib.load(FINAL_DIR / "rare_token_index_prod.joblib")
token_df_prod = joblib.load(FINAL_DIR / "token_df_prod.joblib")
MAX_DF, TOP_N = 1000, 4

print("Loading full TEST S1 as numpy arrays (fast path)...")
test_s1 = pd.read_csv(TEST_DIR / "test_source1.tsv", sep="\t", dtype=str, keep_default_na=False)
test_s1["name_n"] = test_s1["business_name"].map(normalize_exact)
test_s1["addr_n"] = test_s1["business_address"].map(normalize_exact)
test_s1["country_n"] = test_s1["country"].fillna("").astype(str).str.strip().str.lower()

s1_ids_arr = test_s1["entity_id"].to_numpy()
s1_name = test_s1["name_n"].to_numpy()
s1_addr = test_s1["addr_n"].to_numpy()
s1_country = test_s1["country_n"].to_numpy()
entity_id_to_local = {eid: i for i, eid in enumerate(s1_ids_arr)}
print(f"TEST S1: {len(test_s1):,} rows | local-index map built: {len(entity_id_to_local):,}")
del test_s1

rank_cols = ["name_ratio","addr_ratio","name_partial","addr_partial","combined_similarity","tfidf_score"]

def process_source(source_name, source_path, batch_size=5000):
    manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
    done = manifest.get(f"{source_name}_batches_done", 0)
    print(f"\n=== {source_name}: resuming from batch {done} (batch_size={batch_size}) ===")

    reader = pd.read_csv(source_path, sep="\t", dtype=str, keep_default_na=False, chunksize=batch_size)
    for batch_no, chunk in enumerate(reader):
        out_path = OUT_DIR / f"{source_name}_part_{batch_no:05d}.tsv"
        if batch_no < done:
            continue
        t0 = time.time()
        chunk["name_n"] = chunk["business_name"].map(normalize_exact)
        chunk["addr_n"] = chunk["business_address"].map(normalize_exact)
        chunk["country_n"] = chunk["country"].fillna("").astype(str).str.strip().str.lower()

        batch_candidate_data = []

        for row in chunk.itertuples(index=False):
            q_name, q_addr, q_country = row.name_n, row.addr_n, row.country_n
            tokens = set(word_tokens(row.business_name) + word_tokens(row.business_address))
            usable = sorted(
                [(token_df_prod.get(t, 0), t) for t in tokens if 0 < token_df_prod.get(t, 0) <= MAX_DF],
                key=lambda x: x[0]
            )
            cand_local = set()
            for _, tok in usable[:TOP_N]:
                for eid in rare_token_index.get((q_country, tok), []):
                    local_idx = entity_id_to_local.get(eid)
                    if local_idx is not None:
                        cand_local.add(local_idx)
            if not cand_local:
                continue

            cand_arr = np.fromiter(cand_local, dtype=np.int32)
            c_names = s1_name[cand_arr]
            c_addrs = s1_addr[cand_arr]
            c_countries = s1_country[cand_arr]
            c_ids = s1_ids_arr[cand_arr]

            name_ratio = process.cdist([q_name], c_names, scorer=fuzz.ratio)[0] / 100.0
            addr_ratio = process.cdist([q_addr], c_addrs, scorer=fuzz.ratio)[0] / 100.0
            name_partial = process.cdist([q_name], c_names, scorer=fuzz.partial_ratio)[0] / 100.0
            addr_partial = process.cdist([q_addr], c_addrs, scorer=fuzz.partial_ratio)[0] / 100.0
            name_token = process.cdist([q_name], c_names, scorer=fuzz.token_set_ratio)[0] / 100.0
            addr_token = process.cdist([q_addr], c_addrs, scorer=fuzz.token_set_ratio)[0] / 100.0

            qn, qa = len(q_name), len(q_addr)
            sn = np.array([len(x) for x in c_names]); sa = np.array([len(x) for x in c_addrs])
            name_length_ratio = np.where(np.maximum(qn,sn)>0, np.minimum(qn,sn)/np.maximum(qn,sn), 1.0)
            addr_length_ratio = np.where(np.maximum(qa,sa)>0, np.minimum(qa,sa)/np.maximum(qa,sa), 1.0)
            num_overlap = np.array([number_overlap(q_addr, a) for a in c_addrs], dtype=np.float64)
            exact_name = (c_names == q_name).astype(int)
            exact_address = (c_addrs == q_addr).astype(int)
            country_match = (c_countries == q_country).astype(int)
            combined_similarity = 0.45*name_ratio + 0.55*addr_ratio
            has_rare = np.ones(len(c_ids), dtype=int)
            has_tfidf = np.zeros(len(c_ids), dtype=int)
            tfidf_score = np.zeros(len(c_ids)); tfidf_rank = np.full(len(c_ids), 999)
            retrieval_channel_count = has_rare + has_tfidf
            name_addr_product = name_ratio*addr_ratio
            name_partial_addr_partial_product = name_partial*addr_partial
            name_number_product = name_ratio*num_overlap
            addr_number_product = addr_ratio*num_overlap
            min_name_addr = np.minimum(name_ratio, addr_ratio); max_name_addr = np.maximum(name_ratio, addr_ratio)
            name_addr_gap = np.abs(name_ratio-addr_ratio); partial_gap = np.abs(name_partial-addr_partial)
            strong_name_weak_address = ((name_ratio>=0.80)&(addr_ratio<0.20)).astype(int)
            strong_name_no_number = ((name_ratio>=0.80)&(num_overlap==0)).astype(int)
            strong_address = (addr_ratio>=0.70).astype(int)
            strong_name = (name_ratio>=0.80).astype(int)
            address_supported = (addr_ratio>=0.50).astype(int)
            multi_channel = (retrieval_channel_count>=2).astype(int)
            balanced_similarity = (name_ratio+addr_ratio)/2.0
            partial_balanced_similarity = (name_partial+addr_partial)/2.0

            for i in range(len(cand_arr)):
                batch_candidate_data.append({
                    "query_id": row.entity_id, "s1_id": c_ids[i],
                    "name_ratio": name_ratio[i], "addr_ratio": addr_ratio[i],
                    "name_partial": name_partial[i], "addr_partial": addr_partial[i],
                    "name_token": name_token[i], "addr_token": addr_token[i],
                    "name_length_ratio": name_length_ratio[i], "addr_length_ratio": addr_length_ratio[i],
                    "number_overlap": num_overlap[i], "exact_name": exact_name[i], "exact_address": exact_address[i],
                    "country_match": country_match[i], "combined_similarity": combined_similarity[i],
                    "has_rare": has_rare[i], "has_tfidf": has_tfidf[i], "retrieval_channel_count": retrieval_channel_count[i],
                    "tfidf_score": tfidf_score[i], "tfidf_rank": tfidf_rank[i],
                    "name_addr_product": name_addr_product[i], "name_partial_addr_partial_product": name_partial_addr_partial_product[i],
                    "name_number_product": name_number_product[i], "addr_number_product": addr_number_product[i],
                    "min_name_addr": min_name_addr[i], "max_name_addr": max_name_addr[i],
                    "name_addr_gap": name_addr_gap[i], "partial_gap": partial_gap[i],
                    "strong_name_weak_address": strong_name_weak_address[i], "strong_name_no_number": strong_name_no_number[i],
                    "strong_address": strong_address[i], "strong_name": strong_name[i],
                    "address_supported": address_supported[i], "multi_channel": multi_channel[i],
                    "balanced_similarity": balanced_similarity[i], "partial_balanced_similarity": partial_balanced_similarity[i],
                })

        if batch_candidate_data:
            df_batch = pd.DataFrame(batch_candidate_data)
            for col in rank_cols:
                df_batch[f"rank_{col}"] = df_batch.groupby('query_id')[col].rank(method="min", ascending=False)
                df_batch[f"query_max_{col}"] = df_batch.groupby('query_id')[col].transform('max')
            df_batch["tfidf_margin"] = df_batch["query_max_tfidf_score"] - df_batch["tfidf_score"]
            df_batch["name_margin"] = df_batch["query_max_name_ratio"] - df_batch["name_ratio"]
            df_batch["addr_margin"] = df_batch["query_max_addr_ratio"] - df_batch["addr_ratio"]
            df_batch["combined_margin"] = df_batch["query_max_combined_similarity"] - df_batch["combined_similarity"]
            df_batch["relative_tfidf_strength"] = df_batch["tfidf_score"] / (df_batch["query_max_tfidf_score"] + 1e-8)
            df_batch["relative_combined_strength"] = df_batch["combined_similarity"] / (df_batch["query_max_combined_similarity"] + 1e-8)
            df_batch["is_best_tfidf"] = (df_batch["rank_tfidf_score"]==1).astype(np.int8)
            df_batch["is_best_name"] = (df_batch["rank_name_ratio"]==1).astype(np.int8)
            df_batch["is_best_addr"] = (df_batch["rank_addr_ratio"]==1).astype(np.int8)
            df_batch["is_best_combined"] = (df_batch["rank_combined_similarity"]==1).astype(np.int8)

            X = df_batch.reindex(columns=feature_order, fill_value=0)
            df_batch["model_score"] = matcher.predict(X)
            best = df_batch.sort_values("model_score", ascending=False).groupby("query_id", as_index=False).first()
            best = best[best["model_score"] >= threshold][["query_id","s1_id","model_score"]]
        else:
            best = pd.DataFrame(columns=["query_id","s1_id","model_score"])
        best.to_csv(out_path, sep="\t", index=False)

        manifest[f"{source_name}_batches_done"] = batch_no + 1
        MANIFEST_PATH.write_text(json.dumps(manifest))
        elapsed = time.time() - t0
        qps = len(chunk) / elapsed
        print(f"[{source_name}] batch {batch_no:05d} | {len(chunk):,} rows | {elapsed:.1f}s | {qps:.1f} q/s | matches={len(best)}")

print("\nSetup complete. Run these next:")
print('  process_source("S2", TEST_DIR / "test_source2.tsv", batch_size=5000)')
print('  process_source("S3", TEST_DIR / "test_source3.tsv", batch_size=5000)')

Loading model + production rare-token artifacts...
Model OK. Trees=424 Threshold=0.6280000000000001
Loading full TEST S1 as numpy arrays (fast path)...
TEST S1: 1,732,544 rows | local-index map built: 1,732,544

Setup complete. Run these next:
  process_source("S2", TEST_DIR / "test_source2.tsv", batch_size=5000)
  process_source("S3", TEST_DIR / "test_source3.tsv", batch_size=5000)
